# Linear layer with fused bias and ReLU epilogue

`torch.relu(x @ w.T + b)` launches a matmul, then a bias add, then a ReLU, writing and re-reading the `(M, N)` output twice in between. Fusing the bias and activation into the matmul's **epilogue** (the code after the K loop, while the tile is still in registers) removes both round trips.

Write `linear_relu(x, w, b)`.

## Inputs
- `x`: `float32` CUDA tensor `(M, K)`, the activations.
- `w`: `float32` CUDA tensor `(N, K)`, in `nn.Linear` layout (one row per output feature). Do **not** call `w.t().contiguous()`; read it through its strides.
- `b`: `float32` CUDA tensor `(N,)`.
- All sizes $\ge 1$, not necessarily multiples of the tile sizes.

## Output
- `float32` tensor `(M, N)`: $\max(0,\ x w^\top + b)$.

## Requirements
- 2-D grid over `(BM, BN)` output tiles, K loop with `tl.dot(..., input_precision="ieee")` into a `float32` accumulator, block sizes at least 16.
- Load `w` directly as `(BK, BN)` tiles: element `(k, n)` of the tile is `w[n, k]`.
- Add the bias (broadcast over rows) and apply ReLU **before** the single store.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def linear_relu(x, w, b):
    """x: (M, K), w: (N, K) nn.Linear layout, b: (N,), float32 CUDA. Return relu(x @ w.T + b), (M, N)."""
    raise NotImplementedError


@triton.jit
def _linear_relu_kernel(x_ptr, w_ptr, b_ptr, out_ptr, M, N, K,
                        stride_xm, stride_xk, stride_wn, stride_wk, stride_om, stride_on,
                        BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, -1.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0], [2.0, 0.0], [0.0, 3.0]], "dtype": "float32"}, {"__nd__": [0.5, -1.0, 1.0], "dtype": "float32"}], "e": {"__nd__": [[0.5, 1.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 2.0], [-3.0, 0.5]], "dtype": "float32"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float32"}, {"__nd__": [0.0, 0.0], "dtype": "float32"}], "e": {"__nd__": [[1.0, 2.0], [0.0, 0.5]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.2442295104265213]], "dtype": "float32"}, {"__nd__": [[0.6781783103942871]], "dtype": "float32"}, {"__nd__": [-0.5855293869972229], "dtype": "float32"}], "e": {"__nd__": [[0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.908673107624054, -1.9918382167816162, 0.9716230034828186, 0.016657300293445587, 0.2057313472032547, -0.7835948467254639, 1.226498007774353, 0.9432005882263184, -0.12182388454675674, -0.5558634400367737, -0.3566913902759552, -0.7986255288124084, 0.02656414918601513, -0.6161677241325378, 0.5855425000190735, -0.35946857929229736, -0.35395705699920654, 1.5372530221939087, -0.7257195115089417, -1.567915916442871, 0.36621519923210144, -2.0778684616088867, 0.2643786370754242, 0.9817201495170593, 0.07011502981185913, 0.009275074116885662, -2.0506093502044678, 0.555303692817688, 0.3962925672531128, -1.1719977855682373, 1.9131706953048706, -1.2240161895751953], [-1.6792904138565063, -1.8110578060150146, 0.025970907881855965, 0.337871253490448, 0.4409739077091217, -0.3373599648475647, -0.8126749992370605, 0.8427645564079285, -0.897585391998291, -1.209840178489685, 1.401315689086914, 0.17757150530815125, 0.47128990292549133, 0.6258407235145569, -0.5717971324920654, -0.5328626036643982, 0.6405816674232483, -0.6171512007713318, 0.33602067828178406, -0.08945955336093903, -1.1942715644836426, 0.9663907885551453, -1.2893619537353516, 0.2329634428024292, 0.06698467582464218, 1.2666112184524536, -1.119625210762024, -0.5431966781616211, -0.10030654817819595, 0.18084706366062164, 0.8684571385383606, 0.311098575592041], [0.024860581383109093, 1.028337836265564, 2.0960636138916016, -1.355525016784668, -0.005746586713939905, 0.7596399188041687, 0.08574265241622925, 1.4346377849578857, -0.33417242765426636, -1.0140036344528198, 0.8705107569694519, -2.772899627685547, -0.47575879096984863, 2.366079568862915, 0.02624358981847763, 0.35776424407958984, -0.9516060948371887, -0.023186860606074333, -0.142198845744133, 1.8583364486694336, -1.0100823640823364, -2.761763572692871, 0.406651109457016, 1.5500158071517944, -0.6886655688285828, -1.8235293626785278, -1.072821855545044, 0.43356963992118835, -0.5622586011886597, 0.04237709566950798, -2.3575165271759033, -0.5203630328178406], [0.7882662415504456, -0.2646603584289551, -1.0240422487258911, 1.0519297122955322, 0.9869784712791443, -0.465405136346817, -0.7418329119682312, 1.2780617475509644, -0.5499284863471985, -0.16336235404014587, 1.8009670972824097, 0.18686023354530334, -0.4954204857349396, -1.9764130115509033, 0.03540647402405739, -0.08659043163061142, -0.6238414645195007, 0.9255470633506775, -1.29141366481781, -1.582876443862915, -0.7881645560264587, -0.05441165715456009, -1.5299396514892578, -1.2086704969406128, -0.878113329410553, -0.09118537604808807, -0.7701676487922668, 0.906456470489502, -2.288282632827759, 0.32059523463249207, -0.6827162504196167, 0.37315794825553894], [1.8834346532821655, -0.1632070392370224, 0.10310136526823044, 0.7419676184654236, -1.4207983016967773, 0.5683597326278687, 0.7223551869392395, -0.2285628765821457, 0.561270534992218, 0.03691384196281433, -0.854572594165802, -1.4058407545089722, 0.8899444341659546, 1.6896194219589233, -0.8768210411071777, 0.27832821011543274, -1.3032004833221436, -0.5589523315429688, -0.5835057497024536, -1.226510763168335, -0.16090881824493408, 0.9061158299446106, 0.47554928064346313, -0.6005617380142212, 1.339725375175476, 0.13721147179603577, 1.7215131521224976, -1.3855865001678467, 2.939544439315796, -0.2535961866378784, -1.2589082717895508, -1.1424063444137573], [0.8577532768249512, 0.8306096792221069, 0.7384529709815979, 0.535173237323761, -1.2004013061523438, -0.21633176505565643, -2.3846635818481445, -0.9456061720848083, 0.600105881690979, -1.9474297761917114, 2.308201789855957, 1.2014718055725098, -1.209220290184021, 0.6629740595817566, 0.49550899863243103, -0.19802534580230713, 0.5313516855239868, -0.32428279519081116, -0.07539493590593338, 0.49510011076927185, 0.42963549494743347, 0.29148489236831665, 0.312614381313324, 0.9380107522010803, 0.7631500363349915, -1.2063119411468506, -0.7742786407470703, 1.448045253753662, 1.5511633157730103, 0.5475131869316101, -0.282124787569046, 0.8151403069496155], [0.677524209022522, -1.550122618675232, 0.06604824960231781, -1.401731252670288, -0.46045035123825073, 0.03156856819987297, 1.7203665971755981, -2.116495132446289, -0.3564068377017975, 0.6824882626533508, 0.6673679947853088, 0.8509935140609741, -0.9936638474464417, -0.12922403216362, -0.4625622630119324, 1.310268759727478, -0.13779179751873016, 0.018062083050608635, 0.1910465806722641, -1.3094490766525269, 0.5175983905792236, -1.105385184288025, 0.18315619230270386, -0.725476861000061, 1.1982847452163696, 0.0378996767103672, 1.9309453964233398, -2.234001636505127, 1.3723385334014893, 0.8599913716316223, 0.2925330400466919, 2.207212209701538], [-1.428061842918396, 0.2125828117132187, 1.2472155094146729, 0.6071975231170654, -0.3785315454006195, 0.6479513645172119, -0.4152216613292694, -0.34074708819389343, -0.6620199680328369, 1.0542597770690918, 0.3121424615383148, 2.516436815261841, 0.8469467759132385, -1.4834136962890625, 0.8025632500648499, -1.0888036489486694, -0.2761707603931427, 0.1411350667476654, -1.0903810262680054, -0.5054087042808533, -0.6738417148590088, 0.14091387391090393, -0.07351671904325485, -0.10417802631855011, 1.706321358680725, 0.5424520969390869, 0.4329860806465149, 1.0972990989685059, 1.8622812032699585, 1.3982452154159546, -1.0550459623336792, 0.528129518032074], [0.7738372087478638, 0.963693380355835, -0.14110112190246582, 0.07199166715145111, -0.8861479163169861, -0.3685474693775177, -1.1664094924926758, 1.2740064859390259, -0.5194709300994873, -0.9326830506324768, 0.29533764719963074, 0.10678128898143768, -0.6862568855285645, 0.3766423761844635, -0.6261906623840332, 0.5099687576293945, -0.027005810290575027, 0.21739985048770905, -0.38105854392051697, -0.6694474220275879, 0.2550680637359619, -1.571166753768921, 1.3847401142120361, -0.4556908905506134, 2.0766775608062744, 0.6717970371246338, -0.09297751635313034, -0.6028946042060852, 0.5192105770111084, 0.4627750813961029, 0.43113261461257935, 0.8396090269088745], [0.12075815349817276, 1.1450384855270386, -0.7671528458595276, -0.2034383863210678, 0.6195155382156372, -0.7800310850143433, 0.04339888319373131, 1.5349925756454468, 0.10570438951253891, -0.4695192873477936, 1.1293766498565674, -1.1359763145446777, -0.3315819501876831, 1.065853476524353, -0.7700591683387756, 0.2577850818634033, -0.06984575092792511, 1.567361831665039, -1.4218859672546387, -0.5414602756500244, -0.32494816184043884, 0.37377190589904785, 0.5519407987594604, 0.4234330952167511, 1.4191385507583618, 0.6719740033149719, -1.280900478363037, -1.4450773000717163, -0.5374815464019775, -0.5256240367889404, -0.43677374720573425, 1.048702359199524], [-0.6312126517295837, 1.1090986728668213, 1.1486974954605103, 0.03117666207253933, -0.5212129354476929, 1.2880710363388062, -0.8457046151161194, 0.47892680764198303, 0.2725573182106018, 1.2974969148635864, 0.3588549792766571, 0.02115074172616005, 1.0551698207855225, -0.38008996844291687, 0.5624021291732788, 0.13273724913597107, 0.03486168012022972, -1.4824353456497192, -0.004171815700829029, -1.010840892791748, 0.5300000905990601, 2.3697516918182373, -0.22607606649398804, 0.916414737701416, 1.008702278137207, -1.1519744396209717, 0.7081195712089539, -0.3151938021183014, -0.4398360848426819, 0.8204535841941833, -0.5559180378913879, -0.5085629820823669], [0.758003294467926, 1.2898341417312622, 0.0665305033326149, 0.667269229888916, -1.4184764623641968, -0.10151807218790054, 0.44692373275756836, -0.8430501222610474, 0.6445439457893372, 0.28212910890579224, 0.2858833968639374, 1.185017704963684, 1.2655065059661865, -1.2153466939926147, -0.655250608921051, 0.5383467674255371, -0.30107632279396057, 1.990139126777649, 1.1058590412139893, -0.8206589221954346, -0.756020188331604, -1.265444040298462, 0.3714088797569275, -0.6052727699279785, -0.675859272480011, -1.26534104347229, 0.8897148370742798, -0.7469561696052551, -0.9058302640914917, -1.8864752054214478, 0.4047553837299347, 0.23043541610240936], [0.731235921382904, 1.435508370399475, 0.20713375508785248, -0.34434810280799866, -1.603782296180725, 0.10765611380338669, 0.352532297372818, 0.2570239305496216, -1.9473342895507812, 1.2354575395584106, 0.621164858341217, -0.3898916244506836, -0.2841491401195526, -1.9202481508255005, 0.3231218159198761, -0.34349390864372253, -1.1304130554199219, 1.9742324352264404, 1.5448740720748901, 0.723875880241394, -0.021106572821736336, -1.2412753105163574, -0.019198864698410034, 0.4698638916015625, -0.10832037776708603, -0.0352642647922039, 0.18669651448726654, -1.1436364650726318, 0.8206506967544556, 1.0070949792861938, -0.4301830530166626, -0.694354236125946], [2.12083101272583, 1.1412556171417236, -2.3199520111083984, -0.10212771594524384, -1.0199848413467407, -0.911765992641449, -0.6898592114448547, 0.042914681136608124, -0.13190171122550964, -1.2948063611984253, -1.086269736289978, -2.0601816177368164, -0.09036444127559662, 0.2683982849121094, 0.861137866973877, 0.9811378121376038, 0.0984790101647377, -0.4038579761981964, 2.140232801437378, -0.9266030192375183, -1.356798768043518, -0.4616711437702179, 0.19456183910369873, 1.350053071975708, -0.4729847311973572, -1.6206237077713013, 0.7332177758216858, -0.451732337474823, -1.069022297859192, -0.9127473831176758, 0.13095340132713318, -1.767643690109253], [1.4744679927825928, -0.16959546506404877, 0.044059619307518005, 1.0464228391647339, -0.7506640553474426, 0.058580346405506134, -0.08871757984161377, 0.47698917984962463, 2.4423515796661377, -1.1017593145370483, -0.07236749678850174, 2.1126856803894043, -0.9033886790275574, 1.6119085550308228, 0.07725691050291061, 1.0282176733016968, 0.5597047805786133, 0.31701040267944336, -0.8924376964569092, -0.14918915927410126, 0.6837552189826965, -0.38634979724884033, -1.0866663455963135, -0.9823412299156189, 1.4133989810943604, 1.5955826044082642, -0.7194187641143799, -0.02219247817993164, 1.2412447929382324, 0.2922365963459015, 0.06600586324930191, 0.6171085834503174], [-1.1087758541107178, -0.6849214434623718, 0.7650294303894043, 0.20128902792930603, 0.46228116750717163, 0.09147476404905319, -1.2907781600952148, 0.42384764552116394, 1.2987734079360962, 0.5165961384773254, 0.7332479357719421, -0.9379262328147888, -0.2493397742509842, -0.45382627844810486, -0.5050371885299683, -0.6189478635787964, 0.5931559205055237, 0.5535312294960022, 0.8461155891418457, 1.803438425064087, 0.17577272653579712, 0.4692700505256653, 0.18175314366817474, -1.6122817993164062, -1.0580105781555176, 0.47070568799972534, -0.437141478061676, -0.04911904036998749, -1.4225701093673706, 0.2894658148288727, -0.2948756814002991, 0.18841050565242767], [0.1938774734735489, -1.6234675645828247, 1.1037007570266724, -1.2645443677902222, -0.2975785732269287, -0.4607681930065155, 1.0341354608535767, 0.11002510786056519, 0.9641250967979431, 0.20614498853683472, -0.02163729816675186, -0.8039284348487854, -0.46862906217575073, 0.03982388600707054, 0.25987812876701355, -0.6401951909065247, 1.4107394218444824, 0.3045172691345215, -1.1063040494918823, 1.4902573823928833, -1.6819958686828613, 0.3978745937347412, -0.5411322712898254, 1.5766156911849976, 0.8385195732116699, -1.1624234914779663, 1.0088200569152832, -2.014866590499878, 0.8383592367172241, -0.04215892404317856, -0.9069400429725647, 0.9850627779960632], [0.7793011665344238, 0.1972033828496933, 0.2903929650783539, 0.4076736569404602, -0.34988799691200256, 0.5646336078643799, -1.8578473329544067, -3.3783059120178223, 0.46763932704925537, 0.39219948649406433, -0.6645877361297607, -1.0772368907928467, 1.322703242301941, -0.5116183161735535, -0.6024770140647888, -0.9670376181602478, 0.7994547486305237, -2.210301160812378, -0.1884174942970276, -0.7147330641746521, -0.14374808967113495, 0.6686262488365173, -0.48656851053237915, -0.2759510576725006, -1.8693516254425049, -0.5481179356575012, -1.0563409328460693, 0.23091112077236176, -0.23590552806854248, -0.4410960078239441, 0.15613523125648499, -0.9003161787986755], [-1.1357147693634033, -2.591874837875366, -0.07696814835071564, -0.008487158454954624, -0.06346952170133591, -0.5905216932296753, 1.0355345010757446, -0.7569940090179443, 1.7747714519500732, -0.2984006702899933, -0.8344867825508118, -0.407957524061203, -0.08879907429218292, 0.36422494053840637, 1.8591938018798828, 0.12196828424930573, 0.6957732439041138, 0.9022302627563477, 0.42240002751350403, 1.902679681777954, -0.35032951831817627, -0.6405642628669739, -1.233908772468567, -0.28897297382354736, -2.118762731552124, 0.6052508354187012, -0.7145331501960754, 0.6296747326850891, 0.34052079916000366, -1.216822862625122, 2.0829808712005615, -0.7847947478294373], [-0.8643216490745544, -0.42819884419441223, 0.4590015709400177, -1.0916907787322998, 0.7887330055236816, 0.1224244087934494, -0.19870929419994354, -1.0624504089355469, 0.08647492527961731, -0.2852915823459625, -2.0446507930755615, 0.2932725250720978, -0.9549773931503296, -0.3366837799549103, -0.9000543355941772, -0.37373995780944824, -0.4826197028160095, -0.40410372614860535, -0.479557603597641, -0.7038170099258423, 0.9813553690910339, 0.4153318703174591, 0.5026451349258423, 1.279034972190857, -0.5740797519683838, 0.3987925946712494, -0.5776610970497131, -0.8862967491149902, 1.2717164754867554, -0.8980426788330078, 2.436704158782959, 2.339419364929199], [-2.199031352996826, 0.8807365894317627, -0.8794530034065247, 0.5338550209999084, 0.15598846971988678, -2.5713038444519043, 0.07825625687837601, 1.2898757457733154, 0.2219495177268982, 1.8640059232711792, -0.7450062036514282, 1.8824149370193481, -0.033344198018312454, -1.3305531740188599, 0.5357947945594788, 1.7007088661193848, 0.17738373577594757, 0.21971648931503296, 0.10367688536643982, -2.190293550491333, 2.0310795307159424, 1.9033862352371216, 1.498549461364746, 0.4311140775680542, 0.10192485898733139, -0.31525400280952454, 1.584736943244934, 0.8267670273780823, -1.1574472188949585, -0.4626406729221344, 0.3557780086994171, -0.5482146143913269], [0.7054611444473267, 2.020714044570923, 2.0070271492004395, 0.7037056684494019, -0.8866956233978271, 0.987679123878479, -1.2178329229354858, -0.48253437876701355, -0.6776924133300781, -0.2215011566877365, 0.3336980938911438, 1.0897315740585327, 0.32397225499153137, -1.635119915008545, 0.6351867914199829, 0.8161022067070007, 0.8525441288948059, -0.6633188128471375, -0.15879206359386444, -0.4328099489212036, 0.8844480514526367, -1.860754132270813, -0.4644651710987091, 1.3608434200286865, -0.4677206873893738, 1.3676420450210571, 0.8259490728378296, -1.8927260637283325, -0.868949830532074, 1.9368979930877686, -1.7168594598770142, -0.6096845269203186], [0.10494133830070496, 0.2627459168434143, 1.648580551147461, -0.9849259257316589, 0.1271544098854065, 0.7644791603088379, -0.3164345324039459, -0.9095356464385986, -0.4679838716983795, -1.2229214906692505, -0.018975842744112015, 1.0835822820663452, -0.5243836641311646, 0.6659331321716309, -2.0335288047790527, -1.8794047832489014, 1.242360234260559, -0.18613077700138092, 0.4193349778652191, -1.5038294792175293, -0.9705826044082642, 1.1677765846252441, -0.05951584875583649, 0.6271851062774658, 0.010502126067876816, 1.1600195169448853, -0.5721317529678345, -2.3246750831604004, -0.9782713651657104, 0.5362254977226257, 0.2481641173362732, -0.28374889492988586], [1.16843843460083, -0.6239627003669739, -1.1876120567321777, 0.8602170348167419, -0.7138726711273193, 0.21955932676792145, -1.475966453552246, -1.5150413513183594, -0.6786030530929565, 2.2108426094055176, 0.46076419949531555, -0.29339560866355896, -0.4249100089073181, 0.22392286360263824, 0.12868984043598175, 0.7124330401420593, 2.0381369590759277, -0.3344275951385498, 2.654749870300293, -1.936463475227356, 0.22066399455070496, -0.7347033619880676, 0.5239610075950623, -1.346581220626831, -0.8339415788650513, 0.03002861887216568, -1.3794866800308228, 3.1478211879730225, 1.1685909032821655, 0.8281846046447754, 0.3949577212333679, 0.5087741613388062], [-0.14671339094638824, 0.25146356225013733, -0.2381734997034073, -1.0584455728530884, -0.18893569707870483, 1.460396647453308, 0.3301696181297302, 1.643898606300354, 1.0217448472976685, 2.294506072998047, 0.23353053629398346, 0.17782965302467346, -0.4072949290275574, 1.2374941110610962, -0.4195460081100464, 0.6632556915283203, -1.706338882446289, -0.45663002133369446, 1.457572340965271, 0.5353385210037231, 1.3713423013687134, -0.5067242383956909, -0.21308961510658264, -1.6842752695083618, 0.000999265699647367, 0.8409709930419922, -0.21846815943717957, 0.10541406273841858, -0.9476772546768188, -0.8482616543769836, 0.01648731715977192, 0.7897587418556213], [-0.3577535152435303, 0.7168538570404053, 1.171687126159668, -0.4730895459651947, -1.154298186302185, -2.25225830078125, 0.2514287531375885, -0.7300543189048767, 0.09110548347234726, -0.6710216999053955, -0.2477521300315857, 0.38657689094543457, -0.36342892050743103, 1.028835415840149, -0.7900721430778503, -0.6366562843322754, -0.4094008207321167, 0.43344324827194214, -1.0980838537216187, 1.152174949645996, -1.7927912473678589, -0.1751430332660675, 0.3466140329837799, -1.5092802047729492, 0.3662741780281067, 0.8289434313774109, 0.23396100103855133, -0.06858494132757187, -0.08980286866426468, -0.020719369873404503, 0.9538746476173401, -0.2638140022754669], [-0.49304813146591187, 0.870059072971344, 0.5930554866790771, 1.0547723770141602, -1.1376819610595703, 1.1171047687530518, -0.17293024063110352, -0.9266826510429382, 0.7678662538528442, 0.3330967426300049, -1.0838396549224854, -0.2353426218032837, 0.6391165852546692, -1.7724186182022095, 0.5454650521278381, -0.2092706859111786, 0.5150064826011658, -2.22721266746521, 1.5982489585876465, -0.6126388311386108, -0.584618330001831, -0.24001820385456085, 0.5491673350334167, -0.14234885573387146, 1.5541894435882568, -0.23548419773578644, 1.644727349281311, -0.20320041477680206, 0.006399756763130426, -1.5919462442398071, -0.8051860332489014, 0.2948168218135834], [-1.453619360923767, 1.6895970106124878, 0.21479833126068115, -0.3569778501987457, -0.08947034925222397, 1.6742488145828247, 0.7819204926490784, -1.8999671936035156, -0.4186273217201233, 0.9186208248138428, -0.8511638641357422, 0.3575587272644043, -2.2730953693389893, -0.5886784195899963, 0.11027547717094421, 0.7261242270469666, -0.4828501343727112, 0.9266637563705444, -1.489090919494629, -0.18690955638885498, 0.18432985246181488, -1.5722856521606445, 1.2926499843597412, 0.29175281524658203, -1.1074904203414917, 0.5533058643341064, -0.8372839689254761, -0.6520130634307861, -1.3715142011642456, -0.36173543334007263, -2.199115753173828, 0.21564257144927979], [1.8521581888198853, -0.44631585478782654, -2.3055787086486816, -0.48022982478141785, -0.06982550770044327, 0.1518249362707138, -0.06626269221305847, 1.522088646888733, -0.18877293169498444, -0.3969428539276123, 0.34881457686424255, 0.44453155994415283, -0.21391457319259644, -0.2442125678062439, -0.6516838073730469, 0.7290862202644348, -1.5552717447280884, 0.11668923497200012, 0.8875584602355957, -0.056738756597042084, -1.7426186800003052, -0.1648644208908081, 0.7268940806388855, 0.22723934054374695, -1.0843671560287476, 2.2854573726654053, 0.9306540489196777, 0.1524798721075058, 2.7728710174560547, -0.2926527261734009, -1.5122406482696533, -0.16717128455638885], [0.06748698651790619, -1.573412537574768, 0.6914598941802979, 0.9852930903434753, -1.4233707189559937, -0.6515166759490967, -1.2258801460266113, -1.3056389093399048, 0.4402550458908081, 0.09019701927900314, -0.22140759229660034, 0.3080846965312958, -0.3887486159801483, -0.6527918577194214, -0.3998890519142151, -0.2497093230485916, 0.017887363210320473, 0.5110506415367126, -0.7338200807571411, -0.2937448024749756, -0.5769650936126709, -0.6267738938331604, 0.789901614189148, -0.7096683979034424, -0.15572354197502136, -1.0225986242294312, 0.7031776905059814, 1.1010713577270508, 0.16963614523410797, 1.9366044998168945, -2.1088109016418457, -0.6942316889762878], [0.6552905440330505, -1.5589923858642578, -0.1686600297689438, 0.07132601737976074, -0.2491832971572876, -0.04553522542119026, 0.8494147062301636, 1.0831520557403564, -1.4115222692489624, -1.3226014375686646, 0.4439689517021179, -1.1769673824310303, 1.9106268882751465, -0.10321437567472458, 0.2304975390434265, -0.17655205726623535, -0.3678123652935028, -0.7324563264846802, 1.6880239248275757, 0.3646531403064728, -2.0175230503082275, -0.6366455554962158, 0.5305564403533936, 0.419695645570755, 1.22826087474823, 0.26875659823417664, -0.25447624921798706, -0.17487160861492157, 0.2914971113204956, 0.5440022945404053, -0.6400141716003418, -0.3258984684944153], [0.5696122646331787, 0.14384368062019348, -0.10494919121265411, 0.8393099308013916, -0.7411941885948181, 0.604608952999115, 1.2234406471252441, -0.007221451960504055, -0.09019596129655838, -1.0571491718292236, -0.36685487627983093, -0.6615107655525208, 0.9998350143432617, -0.2603878378868103, -0.2846689820289612, 1.0808554887771606, -0.07705812901258469, 0.5517496466636658, -0.3593486249446869, 0.38675323128700256, -0.17594198882579803, 1.28606379032135, 0.23185373842716217, -1.4037396907806396, -0.4870957136154175, -1.2634989023208618, 0.5394064784049988, 0.7863485217094421, 0.2571902573108673, -0.505214273929596, -1.1388351917266846, -0.04624580964446068], [-0.6666211485862732, 0.9299550652503967, -0.3900010883808136, 0.4569445848464966, -0.4797729551792145, 1.0938323736190796, -1.015821933746338, -0.35487329959869385, 0.5801128149032593, 1.6761776208877563, -1.4313257932662964, -1.2713825702667236, 0.47024187445640564, 1.5795180797576904, 0.840117335319519, 0.7164403200149536, -1.0669502019882202, 0.2619483172893524, 0.5110509991645813, 0.2127469778060913, 0.3043553829193115, 0.7538708448410034, -2.4350624084472656, -2.037213087081909, -1.1896467208862305, 0.08640604466199875, -0.38939180970191956, -1.333541750907898, 1.6200014352798462, -0.20765337347984314, 0.1964084506034851, 0.934014618396759], [2.1299667358398438, -0.05937260016798973, 0.18538175523281097, -1.24529230594635, -0.047750163823366165, -1.2606549263000488, -0.4534507095813751, 0.762035608291626, 0.35773128271102905, 0.9473527669906616, 0.25127583742141724, 1.165534496307373, -0.9823053479194641, 1.2165658473968506, -0.5720015168190002, -0.37748414278030396, -0.3130272328853607, 1.8318284749984741, -0.755315899848938, 0.48123809695243835, 0.4184362590312958, 0.09712006151676178, -0.7905883193016052, -1.146176815032959, -1.1521797180175781, 1.0871083736419678, -0.7912207841873169, -1.2125718593597412, 0.9539628028869629, 0.6130418181419373, -0.03760168328881264, -0.5327630639076233], [-2.7411961555480957, -0.7875716090202332, -0.1810922771692276, -1.084431767463684, -1.083017349243164, 2.04459285736084, -1.7179542779922485, 0.28130656480789185, -0.8170990347862244, 0.17296364903450012, -1.234825611114502, 1.1758352518081665, -0.3581206798553467, 1.6865730285644531, 1.3451310396194458, -0.9340949058532715, -1.0324592590332031, -1.3350958824157715, -1.0199147462844849, 1.2080159187316895, 1.6068663597106934, -0.615419328212738, 1.0706443786621094, -0.07173783332109451, -0.2932702600955963, -0.16687385737895966, 0.7743040323257446, -2.032183885574341, -0.3864991068840027, -0.6338786482810974, 0.8036869168281555, 0.7340179085731506], [1.002187967300415, 0.6327388286590576, 0.35001102089881897, -0.8287439346313477, 0.2566145360469818, -0.37635499238967896, -0.530590832233429, -2.234694242477417, 1.0808992385864258, -0.9784522652626038, -0.03659346327185631, 1.8987884521484375, 0.9077801704406738, -0.47357040643692017, 2.512911319732666, 1.0508511066436768, -0.8404424786567688, -0.45568498969078064, 1.050739049911499, 0.8514031767845154, 0.41578516364097595, -1.2057063579559326, -0.08963317424058914, 0.8604403138160706, 1.9161856174468994, 0.17284442484378815, -0.18890149891376495, -0.2092764526605606, 0.5231194496154785, -1.811607003211975, -0.39841651916503906, -0.11844164878129959], [0.3667477071285248, -0.06092188134789467, -0.24173563718795776, -0.4971436858177185, 0.08136333525180817, 0.5345335006713867, 0.391979843378067, 0.999647855758667, -0.4141857326030731, 0.6225354671478271, -1.5199633836746216, -0.32161158323287964, 1.4356969594955444, -0.8804827928543091, 0.7157102227210999, -0.6216526627540588, 0.1391766220331192, -1.5311734676361084, 1.7764099836349487, 1.5255804061889648, -0.706987738609314, -1.0094743967056274, -1.6018497943878174, 0.2450639307498932, 0.22730638086795807, 0.6028292775154114, 0.4679868817329407, -0.9569804668426514, 1.1062860488891602, 1.598558783531189, -0.28017476201057434, -0.5585685968399048], [-0.19109167158603668, -0.36005961894989014, 0.9042751789093018, 0.9369707107543945, 0.6514820456504822, -0.28015440702438354, 0.07834327220916748, 1.049814224243164, 0.6575080752372742, 0.2429865449666977, 0.3784550726413727, 1.1759876012802124, -0.33912429213523865, -0.9916170239448547, -0.31889867782592773, 1.9262373447418213, -0.7675665616989136, 1.2540245056152344, -1.0712262392044067, 0.06275126338005066, 0.697287380695343, -0.3471737504005432, -0.6067963242530823, 0.4418797194957733, -1.0103721618652344, 1.117099404335022, -0.5605607032775879, 0.5190281271934509, -0.5593774318695068, -0.09251299500465393, -0.11283052712678909, 1.009926199913025], [0.6140658259391785, -1.1257480382919312, -1.4179562330245972, -1.1540793180465698, -0.8143371343612671, 1.0715068578720093, 0.18190917372703552, 1.115204930305481, -0.7715412974357605, -0.2642531394958496, -0.4275836944580078, -1.5311267375946045, 1.220084547996521, -1.0733147859573364, -1.22972571849823, 0.3084525465965271, 1.0433807373046875, 0.8173388838768005, 0.5239142179489136, 1.5340180397033691, -0.5870999097824097, -1.6699663400650024, 0.5605949759483337, -0.34195762872695923, 0.6398231983184814, 1.923579216003418, 1.0850703716278076, 0.35340601205825806, -0.5409907698631287, 1.1261489391326904, 1.090585708618164, 0.8371157646179199], [0.2894197702407837, -0.8702405095100403, 0.3080299198627472, 1.3082659244537354, -0.6576446890830994, 0.6728256344795227, 2.053058624267578, 0.7262754440307617, 0.6604263186454773, 0.2220163345336914, 0.8038765788078308, 3.143601179122925, -0.6768108606338501, -0.6280800700187683, 0.3788105845451355, -0.9434313178062439, -0.7660344243049622, -0.9312745928764343, 0.05502048879861832, -1.999068021774292, -0.6613089442253113, -1.1296374797821045, -0.39749976992607117, 0.4709216356277466, 0.46664902567863464, 0.1046510562300682, 0.5559921264648438, -0.2958945631980896, -0.0766935870051384, -0.7573899030685425, 0.1519746333360672, -0.33655300736427307], [-1.6676329374313354, 0.024073228240013123, 2.2479469776153564, -1.5662908554077148, 0.06504174321889877, -1.7903180122375488, 1.5590095520019531, 1.8695151805877686, 1.0769224166870117, 1.4847363233566284, -0.1794830709695816, -0.6948310136795044, -1.360391616821289, -0.0598268136382103, 0.8952800035476685, 0.43946871161460876, -0.2359990030527115, 1.6867742538452148, -1.286850094795227, -0.8348576426506042, 0.9359789490699768, -1.9089468717575073, -1.288960337638855, -0.13802941143512726, 0.6014868021011353, 0.6076739430427551, 1.186743140220642, 0.5269624590873718, -0.9505404829978943, -0.4072766900062561, -0.46601948142051697, 0.8081745505332947], [0.6970392465591431, -0.45002180337905884, -1.682005524635315, 0.17772947251796722, -1.5120810270309448, -1.081551432609558, 0.14565496146678925, 0.23952630162239075, -0.7710174918174744, 0.0004152687615714967, 0.3242308795452118, 1.734399676322937, 0.725453794002533, -0.7530349493026733, -0.47365880012512207, 0.29167360067367554, 0.8036800026893616, -0.03423430398106575, -0.1256529986858368, 1.4152891635894775, 0.01728626899421215, -2.0023460388183594, 2.734375, -0.3510293960571289, -0.48603320121765137, 0.3447023034095764, 0.09951484948396683, 0.8184103965759277, 0.08005054295063019, 0.6535492539405823, -0.7844510078430176, -2.3427200317382812], [0.36547955870628357, -1.0579177141189575, -1.5841273069381714, 0.1409728080034256, 0.1150657907128334, 0.4726623594760895, 1.0480252504348755, -0.19827570021152496, -1.6257824897766113, 0.7642096281051636, 0.5986250638961792, -1.0807859897613525, 1.8725799322128296, -0.8254682421684265, 0.9981333017349243, 0.6906164884567261, -1.0161014795303345, -0.49197471141815186, 0.6007152795791626, 1.038454532623291, -0.7079176902770996, -0.654640793800354, -1.2848137617111206, 1.2795542478561401, 0.8853291869163513, -0.12275205552577972, -1.2942718267440796, -0.17929747700691223, -0.012974015437066555, 0.469721257686615, -1.8189817667007446, 0.7000770568847656], [-0.2871089577674866, 0.9712028503417969, 0.5474992990493774, -1.031348466873169, -0.009285307489335537, -0.4970257580280304, 0.1772327721118927, -0.8323125243186951, -0.4707986116409302, -0.497012197971344, -1.8114532232284546, 0.4477362632751465, 1.8671958446502686, -0.42883002758026123, 0.41875317692756653, -0.412577748298645, -2.4837822914123535, -0.982201337814331, 1.6044433116912842, 2.1900386810302734, -0.011112086474895477, -0.8410353064537048, 0.7919454574584961, -0.9389298558235168, 0.282083123922348, 0.008200567215681076, -0.7790672183036804, 0.591848611831665, -0.06500758975744247, 1.74161696434021, 1.1260188817977905, 0.0769542008638382], [-1.2938629388809204, -0.9852269291877747, 1.0564039945602417, 0.7993819713592529, -1.130022644996643, 0.05069808289408684, -0.7718801498413086, -1.142060399055481, -1.7130285501480103, -0.14094994962215424, -0.05556024983525276, -0.40130454301834106, -1.2555162906646729, 0.2783273756504059, -0.5765014290809631, 0.6811889410018921, 0.21242542564868927, -0.4576770067214966, 0.9001460075378418, -0.7517944574356079, -0.7011725902557373, 0.052004843950271606, 0.733307957649231, -0.18944232165813446, 0.11169779300689697, 0.26562535762786865, -0.3254048526287079, 0.2542407214641571, 0.21793636679649353, -1.6762840747833252, 1.2997426986694336, 1.4477193355560303], [0.15167829394340515, 1.036028265953064, 0.14433619379997253, 0.7980797290802002, 1.007669448852539, -0.4008141756057739, 0.4804076552391052, 1.394811987876892, 1.1051307916641235, 1.5506151914596558, 1.4766461849212646, 0.625903844833374, -0.9039197564125061, 0.6454834342002869, 0.9528501629829407, 0.8927079439163208, -0.5464520454406738, 0.7883429527282715, -0.5127334594726562, 0.7657971978187561, 1.9650752544403076, -0.9122742414474487, -0.3643856644630432, 0.10562719404697418, 1.3270678520202637, 3.3114569187164307, 0.5177169442176819, 0.13980633020401, -0.12409958243370056, -1.8295069932937622, 0.021509971469640732, 0.17506073415279388], [1.3979058265686035, -2.0522682666778564, -0.06322520226240158, 0.2361951470375061, -1.2310580015182495, 1.2511909008026123, 0.17881815135478973, 0.6114664673805237, -0.14587721228599548, -0.3285788595676422, -1.6153221130371094, 0.7051966190338135, 1.0416021347045898, 0.606738269329071, -0.80910724401474, -0.6462461948394775, -1.2618541717529297, -0.22538240253925323, 1.4076850414276123, 0.656622052192688, -0.6982907056808472, -1.4735631942749023, 0.2552814781665802, 0.46125394105911255, -0.2316388189792633, -0.35861125588417053, -0.5389755368232727, -0.23281057178974152, -0.9777384400367737, 0.23361219465732574, 0.41275113821029663, 0.25503629446029663], [0.12184222787618637, 1.2971380949020386, -0.27642112970352173, 0.01693001203238964, 1.7628130912780762, -1.42096745967865, -2.025644540786743, 0.5033683776855469, -0.22326326370239258, 1.9080990552902222, 0.9625605344772339, 0.7559243440628052, 0.8119369745254517, -1.023525357246399, 0.09828009456396103, 2.0503222942352295, -1.2520359754562378, -0.04215852543711662, -1.0427297353744507, 0.6009312868118286, 0.9133834838867188, -0.9339809417724609, -0.013897934928536415, 1.3488556146621704, 0.08120652288198471, -1.042816162109375, 0.4502711892127991, 1.9537723064422607, -1.3823059797286987, -0.4478707015514374, -0.29507049918174744, 0.5026881694793701], [0.2905030846595764, 0.5801788568496704, 1.8543510437011719, -0.0509415902197361, -0.8422914743423462, 0.08469048887491226, 1.5020229816436768, -0.17866235971450806, 0.6710541844367981, 0.1395939737558365, -0.18618996441364288, -0.4114069640636444, 0.9045621156692505, -0.5107026696205139, -0.20526300370693207, -1.6933530569076538, -0.22958128154277802, -0.19640138745307922, 0.12270977348089218, -0.5559691786766052, -1.3171268701553345, 0.325920432806015, 0.5325255393981934, -0.7197888493537903, 0.19748152792453766, -0.5040237307548523, -0.2257051020860672, 0.08935236185789108, -0.597078800201416, -1.0573238134384155, 1.8590950965881348, -0.2135387361049652], [0.17362193763256073, 1.2603613138198853, 1.0671712160110474, -1.0088822841644287, -0.5333748459815979, 2.899656057357788, 0.0556776188313961, 0.7961845993995667, 0.2705245614051819, -0.25015178322792053, 0.028733626008033752, 0.2477695196866989, 0.31767404079437256, -2.6210319995880127, -0.5402088761329651, -0.8138109445571899, -1.8788855075836182, -0.9384562373161316, -2.238485336303711, 1.751550316810608, -0.5329470634460449, -0.5390416383743286, 0.7090004086494446, -0.2764870226383209, 0.6183055639266968, -0.3803424835205078, -0.12847883999347687, -0.389636367559433, 0.8341671824455261, -0.5920498967170715, -2.5512855052948, 0.02223348058760166], [0.9358929991722107, -0.7647896409034729, -1.1072521209716797, -0.654647946357727, -2.370574951171875, -1.0811351537704468, -0.8057559728622437, 0.6574650406837463, -0.5151503086090088, 0.673833429813385, -1.4029215574264526, 2.436474323272705, -0.9175528883934021, -1.0873767137527466, 1.2707935571670532, -1.1640582084655762, 0.23150621354579926, 2.7829198837280273, -0.5738005638122559, -1.979904055595398, 0.0006796267116442323, -0.7222022414207458, -0.9820485711097717, 0.46652308106422424, 0.9698838591575623, 0.36547017097473145, 0.30183205008506775, -1.5649594068527222, -0.5811253190040588, 0.7574899196624756, -1.1699634790420532, 0.39740341901779175], [-0.9758617877960205, 0.8444153070449829, -0.9685763716697693, 0.7163301110267639, 0.7142573595046997, -0.021645618602633476, -0.23142056167125702, -1.31273353099823, -1.4173353910446167, 2.069187879562378, -0.8976691961288452, -1.990270972251892, 0.23294131457805634, -0.5856021046638489, 0.0330188050866127, -0.08590234071016312, -1.2851403951644897, -2.112945556640625, -0.1264415830373764, 1.113085389137268, -0.4545076787471771, 0.1540745496749878, 0.684877872467041, -0.670017421245575, -0.8595895171165466, -0.6488455533981323, 0.795432448387146, 0.4195735454559326, 0.1855829656124115, -1.337180256843567, 1.819072961807251, -0.008578948676586151], [-0.323930025100708, 0.6021824479103088, 0.6022330522537231, -0.18132975697517395, 0.5940319299697876, 1.885651707649231, 0.5189850926399231, -0.28496047854423523, -1.1213089227676392, -1.1348845958709717, -0.8555221557617188, -0.6803025007247925, -1.3080581426620483, -0.35632088780403137, -0.2137693166732788, -1.1093392372131348, -1.785986304283142, 0.8511269092559814, 0.6736550331115723, -0.3883598744869232, -0.027605554088950157, 0.08335631340742111, 0.6510563492774963, 0.5001433491706848, -0.19782446324825287, -0.5073934197425842, -0.8223716020584106, 0.47605615854263306, 1.7376171350479126, 0.040731970220804214, 0.7399045825004578, 1.0854475498199463], [-0.0985318124294281, -1.8981887102127075, -0.70579993724823, 0.9681376218795776, -0.22964715957641602, 1.0818997621536255, -0.8947927951812744, 0.5130959749221802, -0.7854371070861816, -0.7328507900238037, 0.6438304781913757, 0.5994018912315369, -1.410705327987671, 0.7093774080276489, 0.0632651224732399, 0.6584917306900024, 0.738660454750061, 0.20736855268478394, 0.9887383580207825, -0.445903480052948, 0.919333815574646, -2.1842517852783203, -0.1621151715517044, -0.6361609101295471, 0.03965306282043457, 1.4057631492614746, 1.3715736865997314, 1.012753963470459, 0.11530225723981857, -1.867620587348938, 0.313242644071579, 0.23034264147281647], [-0.5019346475601196, -0.5036650896072388, 1.7020224332809448, 1.1489324569702148, -0.45315125584602356, 0.8480469584465027, 0.4839458763599396, -0.0828801691532135, -0.7200599908828735, 0.10396820306777954, -0.2179717719554901, -1.8807092905044556, -0.05545950308442116, 1.662727952003479, -1.2112542390823364, 0.54664146900177, 0.8051274418830872, -0.5513562560081482, 0.6532658338546753, 2.0610129833221436, 0.43223485350608826, 0.16035574674606323, 0.02901693992316723, 1.035402774810791, 0.44927164912223816, 0.5798695087432861, -0.6123720407485962, -0.4245051443576813, 0.8962380886077881, 2.243875741958618, 1.2580459117889404, 0.8025161027908325], [-0.7788418531417847, 0.5712229609489441, -0.5723757147789001, 1.2931101322174072, -0.46465587615966797, -2.1083858013153076, 2.6083738803863525, 0.4208097755908966, -0.17801812291145325, 0.5035929679870605, -1.663711428642273, -0.3514253795146942, -0.9500145316123962, -0.9414988160133362, -0.7746259570121765, 0.7555281519889832, 0.18111886084079742, -0.04334257170557976, -0.634570837020874, -0.45858585834503174, -0.48602038621902466, 0.32697442173957825, 0.6366895437240601, 0.08036958426237106, -0.08800765872001648, 0.8227361440658569, 0.9359179735183716, -0.3659855127334595, -0.22542783617973328, 0.7403736710548401, 1.4327598810195923, 0.9502383470535278], [0.6405982375144958, 0.4018610715866089, -1.0755445957183838, -0.870851993560791, 0.2100689858198166, -0.30515292286872864, 0.4763672947883606, 0.41723617911338806, 0.9106704592704773, -0.9193323850631714, 0.22111470997333527, -1.6228983402252197, -0.3409558832645416, 1.0135600566864014, -0.9339138865470886, -0.1708303689956665, -1.1424144506454468, 0.25002753734588623, 2.059204339981079, -1.278580904006958, -0.37311995029449463, 0.6428597569465637, 1.097033143043518, 0.4333668053150177, 0.860946536064148, -0.8435613512992859, 0.1889108419418335, -0.24722367525100708, -0.9309055209159851, -0.12174363434314728, -0.7757580280303955, -1.0415000915527344], [0.1795307695865631, 1.9098678827285767, -1.2969396114349365, -1.188086986541748, 0.2560608386993408, -1.5740690231323242, -0.776984453201294, 0.5060439109802246, -0.6327071189880371, -0.031788043677806854, 1.6280767917633057, 0.5996643304824829, -1.4068962335586548, -0.6045019030570984, 1.7364249229431152, -0.0988890677690506, -1.3343875408172607, 0.4987689256668091, 0.5199673175811768, 1.8422080278396606, 0.44523632526397705, 0.8361654281616211, 1.315782904624939, -1.5836577415466309, -0.12083756178617477, -0.24094875156879425, -0.09776324033737183, -0.18558931350708008, -0.44434159994125366, 0.9146496653556824, -0.48644721508026123, 1.33339262008667], [-0.9755883812904358, -0.6207608580589294, -2.2281360626220703, -1.5187296867370605, 1.9137474298477173, -1.2233901023864746, 0.09434939920902252, 0.6786699891090393, 0.41596147418022156, 1.404193639755249, 2.3749778270721436, -0.22571375966072083, -0.3092792332172394, 1.0512317419052124, -0.6697331070899963, -1.6725389957427979, 0.3120315968990326, -0.6803784966468811, -0.4807385206222534, -0.6530231237411499, -0.2627277076244354, 0.26283398270606995, -0.9474620819091797, 3.0128843784332275, -2.3060543537139893, 1.3263728618621826, -0.3130759000778198, -0.7319697141647339, -0.24014262855052948, 0.9835206866264343, -0.39583009481430054, -0.13507424294948578], [-0.2517492473125458, -0.8794248700141907, 1.0997487306594849, 0.651420533657074, 0.1972641944885254, 1.015114188194275, 0.9309386014938354, 0.30038851499557495, -0.23630814254283905, -0.014394636265933514, 0.546262264251709, 1.198743224143982, 0.01411642599850893, 0.7208502292633057, 0.6192249059677124, -1.4954839944839478, -1.3131000995635986, 1.3769301176071167, -0.3535457253456116, -0.9026524424552917, 1.4736583232879639, -1.374699592590332, 1.2379906177520752, 0.6817824840545654, 0.27637502551078796, -0.9239507913589478, 0.7604207992553711, -0.7637712359428406, 0.5414149165153503, 0.19363848865032196, 0.21567308902740479, -0.9467437863349915], [-1.3197602033615112, -0.24859115481376648, 0.585982620716095, -0.38235315680503845, -1.480721354484558, 0.40444716811180115, 0.10507018864154816, 0.06334815174341202, -0.09098640084266663, -1.2945778369903564, -0.7153469920158386, -0.1428593248128891, 0.7672686576843262, 0.23737043142318726, 0.17421232163906097, 2.108505964279175, 0.2945019006729126, -0.7812998294830322, 0.2036971151828766, -2.6293728351593018, -0.06626589596271515, -0.20468856394290924, -1.5715144872665405, -1.548919439315796, 0.7305697202682495, -0.2933427393436432, -0.5545394420623779, 0.5720877647399902, 0.27420759201049805, 0.7531883120536804, -0.5877879858016968, 0.1217571571469307], [2.063701629638672, 0.4105721712112427, -0.3201073706150055, 0.7919560670852661, 0.6576456427574158, 1.932621717453003, -0.18827177584171295, 0.27151069045066833, 0.9059984683990479, 0.4475719928741455, -0.32109102606773376, 1.288493275642395, -2.061694383621216, 0.09892162680625916, 0.3196912705898285, -1.2511711120605469, 0.42985570430755615, -0.3189542293548584, -0.7415277361869812, -0.4048323631286621, 1.7648594379425049, 0.5349887013435364, -2.1136443614959717, 0.4237426221370697, 0.0849573165178299, -0.5270269513130188, -0.16258902847766876, -0.9413419961929321, 0.3946157991886139, -0.5684219598770142, -0.2913525104522705, -1.0342423915863037], [1.48045015335083, -2.4244725704193115, -0.33001548051834106, -3.0707545280456543, 0.2278633415699005, -1.1062321662902832, 0.38319987058639526, -0.752495288848877, 1.8849728107452393, 1.577014446258545, -0.7819765210151672, 1.2504900693893433, 0.5954405665397644, 0.25975608825683594, -0.9100755453109741, -0.28053075075149536, 0.35988759994506836, 1.2136739492416382, -0.904145359992981, 0.21202833950519562, -1.4094182252883911, 0.8166524171829224, 0.49519413709640503, 0.3439541161060333, 0.34301674365997314, -0.5495489239692688, 0.5354313850402832, 0.33488574624061584, -0.4630175232887268, -0.6146739721298218, 1.0672497749328613, -0.5316687822341919], [0.603783905506134, -0.6768771409988403, -0.6223391890525818, 0.6659144759178162, 1.933975338935852, 0.2255580872297287, -0.0003736283106263727, 0.6971838474273682, -0.824311375617981, 0.19231005012989044, -0.48667678236961365, -1.4583369493484497, 1.6900255680084229, -0.3317872881889343, -0.6776583790779114, -0.38429683446884155, -0.972730278968811, -0.30465167760849, -2.754810333251953, 0.27774402499198914, -1.6691739559173584, -1.9573252201080322, 1.2630406618118286, 0.639996349811554, -1.3157823085784912, 1.1386723518371582, 0.4534473419189453, -0.11816810816526413, -0.1380831003189087, -0.9846125245094299, -0.44223469495773315, 2.2417736053466797]], "dtype": "float32"}, {"__nd__": [[-0.1174071729183197, -0.7260189652442932, 0.512484610080719, 0.2425568401813507, -1.0288972854614258, -0.3836694359779358, -1.1577022075653076, 1.8116151094436646, 0.5266350507736206, -1.0100841522216797, -0.0008798708440735936, 1.4947775602340698, 0.2932656407356262, 0.019138848409056664, 0.31178948283195496, -1.0427522659301758, 0.5177802443504333, -1.2464913129806519, 0.37592813372612, 1.4297685623168945, -0.14137770235538483, 0.5914096236228943, -0.6484798789024353, 0.38835608959198, -1.0002875328063965, -2.517662525177002, 0.14329290390014648, 0.6270604133605957, -0.34171730279922485, -0.7090991735458374, -0.11973840743303299, 0.4629136621952057], [1.492537498474121, 0.914040207862854, 0.6353827714920044, -0.815731942653656, -0.6195490956306458, -2.5107154846191406, -0.16914093494415283, 0.2041570246219635, 1.7355424165725708, -2.956120014190674, -0.356123685836792, -1.4309616088867188, -0.14149630069732666, -0.7460205554962158, 2.2726869583129883, 1.3434231281280518, 0.19062213599681854, 0.10856206715106964, -2.335340738296509, 0.014155663549900055, -0.4291015565395355, 0.6987317800521851, -0.7399562001228333, -1.1106297969818115, 1.1255687475204468, -0.6847163438796997, 0.2146216481924057, 0.5907929539680481, 1.9473180770874023, 0.04034636542201042, 0.279243528842926, 0.9329707026481628], [1.2131389379501343, -0.9172232151031494, 0.1534716933965683, 1.419039249420166, -1.4793747663497925, 0.605097234249115, -3.372201681137085, -0.06050660461187363, -0.5318929553031921, 0.5245169401168823, 0.2558560371398926, -0.08195459842681885, 0.9311040639877319, -0.6008831858634949, -1.065094232559204, -1.679147481918335, 0.1008974239230156, 2.1747307777404785, 0.6654133200645447, 0.1640549898147583, -2.0524895191192627, 1.0659563541412354, 1.439664363861084, -0.569469690322876, -1.1395004987716675, -1.54189932346344, -1.5904353857040405, 0.27806976437568665, 0.933819055557251, 0.3371547758579254, 1.571147084236145, -0.032639361917972565], [-0.3450344502925873, -0.34926941990852356, -0.10403183102607727, 2.2029545307159424, 0.6001700758934021, 0.7934267520904541, 1.5599089860916138, -0.7766572833061218, 2.3159432411193848, -0.9565203189849854, 0.7783269882202148, 1.4886703491210938, 0.20930735766887665, 0.5771493911743164, -0.3895280361175537, -0.468260794878006, 0.22356414794921875, -0.29651209712028503, -0.09094254672527313, 0.29402288794517517, 0.6558542847633362, 0.16763527691364288, -1.127573847770691, -0.12021544575691223, -0.6962503790855408, 0.2968413531780243, 0.6948553919792175, 0.4897501468658447, 0.16446956992149353, 0.42986050248146057, -0.6421554684638977, 1.5994127988815308], [1.1546862125396729, -0.7333735227584839, -0.14421667158603668, 0.4516441226005554, 0.43247470259666443, 0.01454511284828186, -1.118424892425537, 1.037530779838562, 0.4923875629901886, -0.28862541913986206, -0.3761971890926361, -0.9987441301345825, -1.6147007942199707, 0.08137713372707367, 1.5650451183319092, 0.9675899147987366, 0.23018743097782135, 0.9681344628334045, 0.010865440592169762, -1.462202548980713, -1.9968518018722534, 1.0740485191345215, -1.071441888809204, 0.7966464161872864, -0.09624770283699036, 0.4497266113758087, -0.06145329028367996, 1.5988531112670898, 0.16374056041240692, -0.27689501643180847, 0.8930727243423462, 1.1413745880126953], [0.3198983371257782, -0.5495325922966003, -1.9724299907684326, 0.43066439032554626, -0.5313632488250732, -0.39421677589416504, 0.4477643668651581, 0.4370711147785187, 1.9067838191986084, -0.04782847315073013, -1.1533068418502808, 0.892316997051239, 1.6098051071166992, 0.20448037981987, -1.7738330364227295, -0.991780698299408, 0.36294689774513245, 2.069300889968872, 0.8814547061920166, 0.980876088142395, 0.30400627851486206, -2.0948550701141357, 0.21537980437278748, -0.3271009624004364, -1.848780870437622, 0.4659860134124756, 0.3477707803249359, -0.2509656846523285, 0.061824582517147064, 0.9639939665794373, -1.6849439144134521, -0.11794660240411758], [-1.2377122640609741, 0.5689893364906311, -1.9197989702224731, 0.4056538939476013, -0.3145858943462372, -0.6154211759567261, 0.8289555311203003, 1.2121623754501343, 0.9903049468994141, -0.008260749280452728, 1.9953409433364868, -0.29041776061058044, 0.8470393419265747, -0.18383313715457916, -1.3628575801849365, 0.37001335620880127, -0.4687908887863159, 0.4763306677341461, 1.36182701587677, 1.0836504697799683, 1.0731663703918457, 1.7803263664245605, -0.5941675305366516, -0.5130202174186707, 1.1262099742889404, 0.8437495231628418, -0.7126931548118591, -0.8167803287506104, 0.11483883112668991, 1.5636658668518066, -0.942152202129364, -0.6880605220794678], [-0.6534768342971802, -1.9231277704238892, 1.4240410327911377, 0.8328628540039062, -0.8708368539810181, 0.3641338646411896, 0.5804203152656555, 0.01660618558526039, -1.2924420833587646, 1.4723724126815796, -0.0953567624092102, 1.0870440006256104, -1.2566499710083008, -1.973584532737732, 0.9761897325515747, -1.593888521194458, 0.17227965593338013, -0.914367139339447, 0.43176737427711487, -1.3054109811782837, 0.5540335178375244, 2.3186092376708984, -0.004675767384469509, 1.4096293449401855, 1.5259778499603271, -0.43229109048843384, 0.7316336035728455, -0.21277648210525513, -0.4099053740501404, 0.7052336931228638, 0.25711408257484436, 0.6130722761154175], [-0.35723766684532166, 0.4995110332965851, 0.9793586134910583, 0.15271161496639252, 0.6539870500564575, -1.994149923324585, 0.4202614724636078, -2.4509668350219727, -0.6712583899497986, 0.19517432153224945, -0.3635671138763428, -1.354923963546753, 0.44298839569091797, 1.0325688123703003, -0.4711601138114929, 0.7000911831855774, 1.5908888578414917, -0.6340575814247131, -0.7886511087417603, 1.7851818799972534, 1.5086448192596436, 0.665388286113739, -1.6586480140686035, 2.339482545852661, 0.07037436962127686, 0.7056158781051636, -0.05245472863316536, -0.3523883521556854, -1.4113218784332275, -1.166754126548767, -0.8407763242721558, -0.8146672248840332], [-1.3549805879592896, -0.2824314534664154, 0.8362248539924622, -0.3035052418708801, 0.019012214615941048, 1.7199175357818604, -2.4807517528533936, 0.2400110363960266, 0.8510888814926147, 0.12607169151306152, -2.0592458248138428, -0.7877221703529358, 0.10737801343202591, -0.5280165076255798, 0.42891380190849304, -0.42836788296699524, -1.1087181568145752, -0.8033351302146912, -1.256826639175415, -0.15545617043972015, 0.6902275085449219, -0.3654692769050598, 0.3535454571247101, -0.5411552786827087, 0.03230403736233711, 0.06443886458873749, -1.7139360904693604, 0.9754146337509155, 1.2807507514953613, -0.8269845247268677, -0.6927925944328308, -0.4667195677757263], [-1.4565274715423584, -1.1107934713363647, -0.5484502911567688, -1.6385847330093384, -0.9284619688987732, 0.8495269417762756, -0.29672080278396606, -0.6702131032943726, -0.2498384267091751, 1.5919668674468994, 0.7027508020401001, -1.7407790422439575, 0.31771159172058105, 0.2956083416938782, -1.5683419704437256, 1.0177040100097656, -0.11340484768152237, -1.1344027519226074, -0.12493927031755447, -1.2832821607589722, 0.5392302870750427, -0.5620331168174744, -0.7919986844062805, -0.8623457551002502, 0.11696288734674454, 0.22496585547924042, -0.9501521587371826, -2.247824192047119, -0.7563572525978088, -0.46231770515441895, 0.11420907080173492, -0.3580271303653717], [0.1806459128856659, 0.9140375256538391, -0.7228456735610962, -0.44656386971473694, -0.5187138319015503, -0.6237432360649109, 0.2946748733520508, 0.279754102230072, 1.5002641677856445, 0.46971553564071655, -0.1721535623073578, -1.407845139503479, 1.0607447624206543, -0.38648197054862976, 0.00557310413569212, 2.5054421424865723, -0.3810384273529053, 0.5746882557868958, -0.2864328920841217, 0.9924854636192322, 0.6244928240776062, 1.0457514524459839, -1.1487370729446411, 0.4858097732067108, 0.24464890360832214, 0.4652954339981079, -1.6184803247451782, 0.17256277799606323, 0.2810826301574707, 0.7908728122711182, 1.0062453746795654, -1.0570932626724243], [-1.562087893486023, 1.8201487064361572, -0.6931436657905579, 1.3194046020507812, 1.4841907024383545, 0.870130181312561, 1.2130922079086304, -0.263516366481781, 0.11550579220056534, 0.8024688959121704, -0.5478391051292419, -0.728914201259613, -0.4019140303134918, -1.8426551818847656, 0.6284084320068359, -0.0736376941204071, 1.5165587663650513, 0.4319636821746826, 1.7117670774459839, 0.9945627450942993, 0.5235228538513184, 1.257203221321106, -1.3338117599487305, 1.0519498586654663, -0.053877949714660645, -1.1401032209396362, -1.1796841621398926, 0.8252250552177429, 0.24627304077148438, -0.4583003520965576, -2.182103157043457, -0.0786273330450058], [0.5299658179283142, -1.088641881942749, -0.593416690826416, -0.9805434942245483, 1.071114420890808, 0.013923315331339836, -0.4206967055797577, -1.1562581062316895, 1.8437564373016357, -0.6674844026565552, 0.8548171520233154, 0.9693689942359924, -0.3228408098220825, 0.5985015034675598, -0.042205072939395905, -0.9270237684249878, -0.8023006319999695, -0.12301229685544968, 1.2086089849472046, -0.3411623239517212, 0.11648403108119965, 0.9293791651725769, 1.075040340423584, -1.2132229804992676, 2.6900477409362793, 0.9885206818580627, -0.31084662675857544, 0.4161643981933594, 1.36458420753479, -0.7373444437980652, -0.24722658097743988, -0.5487863421440125], [-1.391600489616394, -0.37223705649375916, -0.21340709924697876, -1.1884592771530151, -0.08254599571228027, 0.07921494543552399, -0.33978471159935, 0.638598620891571, -1.148289442062378, 0.5302699208259583, -3.4146010875701904, 2.213549852371216, -0.006282752845436335, -0.5223023891448975, 0.613158643245697, -0.17130057513713837, -0.23610319197177887, -0.02970270812511444, 0.5951933264732361, -1.1547974348068237, 0.7221261262893677, -0.4584510922431946, 0.5792325139045715, -0.8639489412307739, -1.1414570808410645, -0.12732847034931183, -1.0861876010894775, -0.6228938102722168, 0.774242639541626, 0.7061089873313904, -0.5870435833930969, -0.2600880265235901], [0.3948134481906891, -0.8043360710144043, -1.721771001815796, -1.0255920886993408, 1.0545434951782227, 1.685206413269043, 0.37823721766471863, -1.2988964319229126, 0.343633234500885, 0.39718833565711975, 0.4402455687522888, 1.7408138513565063, 1.068454384803772, 1.2773597240447998, -0.6811556220054626, -0.2884873151779175, 0.045089974999427795, 0.8383808732032776, -0.00540901767089963, 0.7567059397697449, 1.0592386722564697, 0.42068344354629517, 1.8892688751220703, -0.1908905953168869, 0.7208955883979797, -0.7891550660133362, 0.04826989024877548, 0.8830084204673767, 1.654820203781128, 0.2890174686908722, 1.3547840118408203, 1.8239775896072388], [0.028471186757087708, 1.3120726346969604, 1.5307602882385254, -0.1652388572692871, 0.7940577864646912, -1.8031699657440186, -1.9294830560684204, -1.1882010698318481, 0.32546466588974, 0.7068927884101868, -1.7856370210647583, -0.22335165739059448, -0.15001027286052704, -1.2551169395446777, -0.3254949152469635, 1.01942777633667, 0.0072982762940227985, 0.9961373209953308, 0.8305241465568542, 0.5843574404716492, -0.10900437086820602, 0.3306380808353424, -1.0286991596221924, 1.2616808414459229, 1.827742338180542, -1.229021668434143, -1.131657600402832, -0.025952234864234924, -1.296075701713562, 0.5682414174079895, -1.250844120979309, 2.3362679481506348], [1.9568922519683838, 0.7127273082733154, -0.48790210485458374, 1.2151899337768555, -0.4057399332523346, -0.9357940554618835, -1.976340889930725, -0.8780418038368225, 1.5489648580551147, 1.473878264427185, 0.6073119044303894, -0.023409534245729446, -1.4817901849746704, -0.6081677079200745, -0.5080576539039612, -0.183860644698143, -0.9356178641319275, 0.7116814255714417, -0.7089414596557617, -0.5953912138938904, 0.5972751379013062, 0.450065016746521, 1.1047415733337402, -0.7846498489379883, 0.8645503520965576, -1.706593632698059, -0.27082115411758423, -1.0793342590332031, -0.17642918229103088, 1.3198299407958984, -2.293954610824585, 1.3138456344604492], [1.2299036979675293, -0.46217939257621765, -0.8617010116577148, 0.28052598237991333, -1.914171814918518, 1.3384184837341309, 1.8456364870071411, -0.8038567304611206, -0.5901079177856445, 0.60094153881073, -0.8267751336097717, -0.7011429071426392, 0.7415373921394348, -0.04340618848800659, 0.6711041331291199, 1.4745081663131714, -0.8926685452461243, 0.2059335857629776, -1.188637137413025, 0.9988672137260437, -0.1370135098695755, 0.18217962980270386, 1.4103082418441772, -1.5053422451019287, 0.349790096282959, -1.1276271343231201, 0.49193382263183594, -1.8929722309112549, -0.4812205731868744, -0.4585909843444824, 1.204024076461792, 0.3938722312450409], [-1.2208985090255737, -0.7114760875701904, 1.0712265968322754, -0.3458375632762909, 0.18404850363731384, -0.5092481970787048, 0.8676202297210693, 1.0643715858459473, 2.2201836109161377, -0.6752159595489502, -0.6693100333213806, 0.1429334580898285, -0.4022858738899231, -1.5984495878219604, 0.9046782851219177, 1.0954015254974365, -1.4716031551361084, -0.49255526065826416, 1.1505857706069946, -2.1962316036224365, 0.12997525930404663, 0.17865175008773804, -0.6230910420417786, 0.32316264510154724, 0.4455488920211792, -1.4585926532745361, 1.0281049013137817, -0.22513549029827118, 1.5000308752059937, -1.4190788269042969, -0.49915045499801636, -0.7944359183311462], [-0.608564555644989, 2.1869897842407227, -0.44572943449020386, -0.024972157552838326, -0.1507539004087448, -0.5284356474876404, -0.6613674163818359, 0.6666103601455688, -0.7713857293128967, 0.3674376606941223, 0.6312143206596375, -0.2501382529735565, 0.284389853477478, -0.8042814135551453, -0.16355131566524506, -1.6749871969223022, -0.8956055045127869, 0.23842817544937134, 1.390770673751831, 0.508458137512207, 0.4379848837852478, 0.7478218674659729, -0.8263955116271973, -0.2969847321510315, -0.43197691440582275, -1.7633248567581177, -1.3926578760147095, 0.36356809735298157, 1.1647403240203857, 0.3330341577529907, -1.1288020610809326, 0.9742048382759094], [-0.5254908800125122, -1.360451340675354, 0.6258444786071777, 0.689318060874939, 1.8130348920822144, -0.40619781613349915, 1.2990658283233643, -0.1817300021648407, 0.5500503778457642, 1.6409528255462646, 0.4220932424068451, 0.893473744392395, 1.285361647605896, 1.2236013412475586, -0.5694753527641296, 1.6066114902496338, -0.12689922749996185, -0.6416653990745544, 0.058810193091630936, 1.8760113716125488, -0.605036199092865, 0.8808242678642273, -0.27369794249534607, 2.291926622390747, -1.412774682044983, -1.3474057912826538, -1.478650450706482, -0.21274064481258392, 0.37971043586730957, -1.3758878707885742, -2.074284791946411, -0.5276649594306946], [1.1614001989364624, -1.9453049898147583, -0.9252808094024658, 0.9773887991905212, -0.2302972674369812, 0.06258609145879745, -1.3971476554870605, 0.2885485887527466, 0.30560991168022156, 0.85307776927948, 0.8386404514312744, -0.11010227352380753, 0.646999180316925, -1.2723175287246704, -0.4661218822002411, 0.7065449953079224, -1.8558778762817383, -1.2658281326293945, -0.6894575357437134, 0.537558376789093, 0.4122369885444641, -1.0960458517074585, 0.13681192696094513, 1.2420083284378052, 1.267538070678711, -1.807474970817566, -0.4575881063938141, 0.9594555497169495, 1.1544233560562134, -0.38945016264915466, -0.3277783691883087, 0.8089464902877808], [1.6098483800888062, -0.47774195671081543, -1.4978307485580444, -0.38354605436325073, 0.7638183832168579, -2.4034130573272705, -1.001987099647522, 0.3864414393901825, -0.8312963247299194, -0.2926638424396515, 0.22297099232673645, -0.028577718883752823, -0.30771100521087646, -0.6689304709434509, 1.0575565099716187, 1.2470372915267944, 1.2086414098739624, 1.1211223602294922, 1.087046504020691, -0.7613910436630249, -0.03377808630466461, 1.981001615524292, -1.3360159397125244, -0.9277607202529907, -0.1407051682472229, -0.5788097977638245, 1.2124241590499878, -0.4250902235507965, -0.6285891532897949, -0.9728075265884399, -0.23894259333610535, -0.4281105399131775], [-0.24107560515403748, -0.16030149161815643, -0.08215199410915375, 0.447004497051239, -0.4169739782810211, 1.460188627243042, -0.7876601815223694, 0.9855449199676514, 0.4764266610145569, 0.024210108444094658, -0.42515233159065247, -0.3257073163986206, 0.0959508940577507, 1.736158013343811, 1.289656400680542, -1.0980027914047241, 0.8471526503562927, -0.43389928340911865, 0.03398748114705086, 1.1700702905654907, -0.3161333501338959, 0.4410599172115326, 0.802453339099884, -0.3695604205131531, -0.9946157932281494, 0.24425756931304932, -0.10311255604028702, -1.0345814228057861, 1.5143386125564575, 2.0428617000579834, 0.9355481266975403, -1.2214384078979492], [1.3501155376434326, -0.11441682279109955, -0.6915003061294556, -1.0282243490219116, 0.5577948093414307, 1.1607369184494019, -0.22475062310695648, 1.7155224084854126, -1.4347254037857056, -1.3554435968399048, -0.05446266382932663, 1.3776901960372925, -0.16808664798736572, -0.48977676033973694, 0.4403970241546631, 0.5685186386108398, -0.838917076587677, -0.9270663857460022, -0.3321954607963562, 0.7683335542678833, -0.8090445399284363, 0.6378259062767029, 0.4492020010948181, 1.1603245735168457, -0.18525587022304535, 0.6367202401161194, -0.3761940002441406, -0.8947442770004272, -0.16141781210899353, 2.0122101306915283, -1.3755675554275513, -0.20052595436573029], [0.06026191636919975, 0.27003413438796997, 1.2238578796386719, 0.19808128476142883, 0.08538033813238144, 0.4717133045196533, 0.13498634099960327, 0.8327644467353821, 0.5988262891769409, -0.0024257893674075603, 1.1533371210098267, 2.148421287536621, -0.40586531162261963, -0.2813536524772644, -0.3601542115211487, -1.5457427501678467, 0.5248485207557678, -0.7946674823760986, -0.3321060240268707, -1.500943660736084, -1.632761001586914, -1.641952395439148, -0.061613745987415314, 2.2850072383880615, 0.2547157406806946, -0.14230608940124512, -0.4997938573360443, -0.039255283772945404, -0.3479369878768921, 1.1522536277770996, 0.7447113394737244, 0.728671133518219], [0.9169086217880249, -1.7370985746383667, -0.4436385929584503, 0.9895817041397095, -0.35495588183403015, -1.057057499885559, -0.28448957204818726, -0.4379104971885681, 0.9007928967475891, 0.5157066583633423, 0.07360053807497025, -0.43670812249183655, 0.9389755129814148, -0.016742656007409096, -0.19729477167129517, 0.006388638634234667, -0.6566819548606873, 0.807517409324646, 0.8558369278907776, -1.0106736421585083, 1.0027267932891846, 0.4819526672363281, 0.2419138103723526, 0.23996864259243011, -0.5046717524528503, 0.06129029765725136, -1.2184901237487793, 0.16893312335014343, 0.09628714621067047, -0.8382494449615479, -0.061206888407468796, -1.0857665538787842], [0.20561353862285614, 0.23539279401302338, 1.3835607767105103, 0.8621496558189392, -0.02493215724825859, -1.431376576423645, 1.9802268743515015, 1.2146682739257812, -0.26030993461608887, -0.21098101139068604, -1.04141104221344, -0.8037106990814209, -1.089449405670166, -0.989885151386261, -1.9048893451690674, 0.7401859164237976, 0.8235988020896912, 1.489993929862976, -0.10204321146011353, -1.2141332626342773, 1.0107378959655762, 1.328525424003601, -1.3695521354675293, -1.6377289295196533, -1.44759202003479, -0.3058657944202423, -1.346130609512329, 1.3176168203353882, -0.5824900269508362, -0.7175639867782593, 1.317630410194397, -0.8911647200584412], [0.6924837827682495, -0.5762951970100403, -0.30647504329681396, -2.0569958686828613, -0.6744077205657959, 0.7041735053062439, 0.5926944017410278, 0.030656985938549042, 0.7139174342155457, 1.2677780389785767, -0.5432898998260498, 1.6360228061676025, -0.7101138234138489, -1.8005084991455078, -0.5038612484931946, -0.3852059543132782, -0.0888838991522789, 0.39472126960754395, 0.2410985231399536, -1.8261222839355469, -0.047814372926950455, -1.402146816253662, 0.4094945192337036, 0.294501394033432, 1.3030859231948853, -1.3479104042053223, 1.9076199531555176, 0.5908364653587341, -1.3154605627059937, 0.019028419628739357, 0.674091637134552, 1.229552984237671], [0.692688524723053, -0.11228221654891968, 2.0264573097229004, -0.06124739721417427, -1.7088042497634888, -0.5741324424743652, 1.0553669929504395, 0.228774756193161, -0.152324378490448, -0.6777658462524414, 0.22405044734477997, -0.23569022119045258, -1.2004197835922241, -0.6825817227363586, 0.018861226737499237, 1.0986967086791992, 0.9243276119232178, 0.7693374156951904, 0.43080613017082214, -0.2172195464372635, 0.9284957647323608, -0.6785355806350708, -1.0628644227981567, -0.30515819787979126, 1.13266122341156, -0.9127494692802429, -0.6107641458511353, 1.326339602470398, 0.39155229926109314, 0.6873496174812317, -0.23019744455814362, 1.867303490638733], [0.4036393463611603, 1.0758861303329468, -0.08759108930826187, -0.7754300236701965, -0.27432072162628174, -0.1613035798072815, 0.3390141427516937, -1.4832384586334229, -1.2437317371368408, 1.3315125703811646, -0.4029058814048767, -0.6717578768730164, -1.6507772207260132, 1.3912447690963745, 0.49045759439468384, -0.14356212317943573, 0.15814447402954102, 2.3571550846099854, 0.2613580524921417, -1.4132527112960815, -0.33649352192878723, 0.6668097376823425, 0.4391292631626129, -2.0607736110687256, 0.3617302179336548, -0.23899459838867188, -0.5667098760604858, -0.2165510505437851, -0.10032968968153, -0.7964543104171753, -0.74657142162323, 0.7810240983963013], [-0.11671240627765656, 0.43040069937705994, -0.1610218733549118, -2.450606346130371, 0.9790446758270264, -0.2675422132015228, -0.3362276256084442, 0.4659014642238617, -0.8171595931053162, -0.09080005437135696, -0.2461802214384079, 0.46899864077568054, -1.4184244871139526, -0.7408665418624878, 1.0866841077804565, 3.324822425842285, -1.5948364734649658, -2.4576375484466553, 1.996519684791565, -0.3965177834033966, -0.3379327058792114, 0.4161224961280823, 0.9218489527702332, -0.9391533732414246, -1.5119208097457886, -1.4824941158294678, -0.809302568435669, -0.9599273800849915, -0.4666331112384796, -0.2659858167171478, 0.40260642766952515, 0.11951899528503418], [-1.1831316947937012, 0.39096856117248535, -1.2457600831985474, 0.821755588054657, 1.351973295211792, 2.4492599964141846, 0.9767924547195435, 1.4820129871368408, 0.8777884840965271, 0.908790647983551, 0.7898701429367065, -0.6987850069999695, -0.6591277718544006, -1.4859869480133057, -0.6305503249168396, 0.6418905854225159, 2.4827234745025635, 0.7643387317657471, 0.5302248001098633, 0.19114050269126892, 1.3012921810150146, 0.0009915201226249337, -0.3727360963821411, -0.2377447634935379, -0.9607122540473938, -0.11098037660121918, 0.4950704574584961, 0.3882019519805908, 0.14355966448783875, -0.44636300206184387, -0.04328734800219536, 0.4330662786960602], [-0.4396131634712219, -0.908872127532959, 0.2714633047580719, 1.0691255331039429, 0.2351430207490921, 1.335197925567627, -0.07409690320491791, 1.1247544288635254, 1.7361935377120972, -0.3437836766242981, -0.39676433801651, 0.1672729104757309, -1.468887209892273, -1.5791817903518677, 1.142048716545105, -0.030267640948295593, 1.2691949605941772, -0.973904013633728, -1.4671989679336548, 0.9290511012077332, 0.855695903301239, -0.9721899032592773, 0.4758716821670532, 0.8641979098320007, 0.7674891352653503, -0.08393925428390503, -0.22862885892391205, -1.1662819385528564, 0.477661669254303, 0.48221060633659363, -0.7732380032539368, 0.311954140663147], [-0.840694010257721, -0.37037014961242676, 1.747397541999817, -0.7333382964134216, -0.19993163645267487, 0.4772131145000458, -0.7891556024551392, 0.8228501081466675, -1.4819082021713257, -0.3633899986743927, 0.6621002554893494, 1.1861696243286133, 1.5122811794281006, 0.43673059344291687, 0.3900165855884552, 1.218416690826416, 0.4301789104938507, 1.5042184591293335, -0.5470299124717712, -1.0884920358657837, -0.5400812029838562, 0.7584265470504761, -0.9219916462898254, -0.2539668381214142, 0.14441247284412384, -0.7912785410881042, -1.2868623733520508, -1.3245983123779297, 0.1593126356601715, -0.5969408750534058, 0.123625248670578, -0.5965524315834045], [0.2292361706495285, -0.08547006547451019, 0.6431173086166382, 1.5639007091522217, -0.9151041507720947, -1.2018399238586426, 0.8034415245056152, 0.3475048840045929, 2.0585336685180664, -1.3627606630325317, -1.357447862625122, 1.1020448207855225, -0.018453354015946388, 0.2131832391023636, -0.702642023563385, -0.5337320566177368, 0.36844417452812195, 0.3783895671367645, 0.12960584461688995, 1.1244378089904785, -0.5320705771446228, 0.2155250757932663, 0.5132915377616882, -0.4131941497325897, -1.5183652639389038, 1.0581589937210083, 2.085097551345825, 2.7054014205932617, 0.3984420597553253, -0.578033447265625, 0.7682441473007202, 1.53707754611969], [-0.06658654659986496, 0.3697216212749481, -0.7407073974609375, 1.09095299243927, -0.018374290317296982, -1.060088872909546, 0.7841867804527283, 0.24699535965919495, 1.0773526430130005, -0.22910048067569733, 0.4762900769710541, 0.04644589498639107, -0.6316073536872864, -0.923588752746582, 0.12664355337619781, -0.473506361246109, -0.5447471737861633, 0.5391983389854431, -1.45032799243927, 0.5763815641403198, -0.5108956694602966, 2.4912078380584717, 1.107999563217163, 0.865048348903656, 0.06716913729906082, -0.8269757032394409, -0.112896628677845, 0.3328773081302643, 0.9779290556907654, 0.20005536079406738, -0.31866171956062317, -0.4141435921192169], [0.05165773630142212, -0.9690824747085571, -0.25712716579437256, 1.2494475841522217, -0.3619311451911926, 0.6024962067604065, -0.663706362247467, 1.918864130973816, -0.375395804643631, -0.20043858885765076, -0.5566104054450989, -1.1597480773925781, 1.8372305631637573, -1.3471801280975342, 1.6279277801513672, -0.3735215365886688, -0.0020479492377489805, 1.445783019065857, 0.22818270325660706, -1.5721381902694702, 0.14431506395339966, 0.6178327798843384, -0.8542377352714539, -0.7674024701118469, -0.6928079724311829, -1.1109364032745361, -0.9412893056869507, 1.0414228439331055, -0.6336174011230469, 1.7832196950912476, 0.1164749339222908, 0.5824148654937744], [-0.8258072137832642, -0.9894587993621826, -1.0930570363998413, -0.4751368761062622, 1.2013461589813232, 0.5019079446792603, 1.2895922660827637, 0.12808702886104584, -0.7703496217727661, 0.4914487302303314, 0.44149050116539, -0.1538679450750351, 0.042010288685560226, 0.9120524525642395, 2.3103415966033936, -1.389280080795288, 0.22163577377796173, -1.8149926662445068, 0.7109354734420776, 1.8437296152114868, -0.4654514789581299, 0.3711051344871521, 0.22218234837055206, -0.9704591035842896, -0.4503892660140991, -0.6120918393135071, -0.4110134243965149, -1.4292588233947754, 0.495570570230484, 0.04556998610496521, 0.8063605427742004, 1.3324062824249268], [0.6664594411849976, -1.3845864534378052, 0.509743869304657, 1.2778215408325195, 1.7921112775802612, 0.6883166432380676, 0.05552484840154648, -1.7629677057266235, 1.1072829961776733, 0.3191913366317749, -1.672788143157959, -1.1969099044799805, -1.0112553834915161, 0.495701402425766, -1.0562556982040405, 0.11516103893518448, 0.2899964451789856, 0.3490143418312073, 0.11659225821495056, -0.531718373298645, 0.8012540340423584, -0.8981603384017944, -0.3072303533554077, -1.0956753492355347, 1.1925681829452515, -1.216274619102478, 1.1729379892349243, 1.0095300674438477, 0.7302752733230591, 0.10172460228204727, 0.19752353429794312, -0.6274089217185974], [-2.1226043701171875, 0.7297385334968567, 0.9358780980110168, 0.17618809640407562, -0.26652100682258606, -1.9779821634292603, 1.012554407119751, -0.9403730034828186, 0.11617803573608398, -0.6535825133323669, -0.7686537504196167, 1.2369117736816406, -0.4141245484352112, 0.26712319254875183, 0.5576423406600952, -1.373674988746643, -0.5340499877929688, 0.08596312999725342, -0.39366415143013, -0.5850144028663635, -1.112839937210083, -0.39434921741485596, -0.4185526371002197, 0.3561578094959259, 1.225054144859314, 1.4667744636535645, 3.2520503997802734, 1.5236479043960571, -0.03467786684632301, 1.3837127685546875, 1.6454834938049316, 0.7173197269439697], [-0.795571506023407, 0.9179543256759644, 0.1006559506058693, -2.622042417526245, 0.4302118122577667, 0.5310279130935669, 0.18148577213287354, 0.20656870305538177, 0.09259973466396332, 0.0018473694799467921, 2.013352632522583, -1.617969036102295, 0.26653483510017395, 0.3015773892402649, 0.14583663642406464, 0.7065932154655457, 0.6311918497085571, -0.7611745595932007, -0.8208752274513245, -0.10983602702617645, -0.44319626688957214, 0.8191020488739014, -0.702162504196167, -0.35190269351005554, -0.05548727139830589, 0.002526806900277734, 0.678747296333313, -1.435931921005249, -0.27029260993003845, -1.1810475587844849, 0.9409763216972351, -0.8228422999382019], [0.8012350797653198, -0.0530875027179718, 0.8826577663421631, -0.03018239513039589, 0.8201279640197754, 0.6831642985343933, -0.35256826877593994, -0.06726089119911194, -0.08006101846694946, 1.3634083271026611, 1.7713035345077515, 0.6080744862556458, 0.23186534643173218, -0.70697021484375, -0.8697310090065002, 0.06219610944390297, 1.2751719951629639, -1.7849225997924805, 0.9607082605361938, 0.8127485513687134, 0.4913087785243988, -0.36139246821403503, -0.40731656551361084, -0.6065886616706848, -2.8893115520477295, 0.2411145120859146, 1.6411136388778687, 0.3252207636833191, -0.8758596181869507, 0.007680414244532585, 1.5297918319702148, -1.0987414121627808], [-0.8060535192489624, 0.3884452283382416, 0.7173294425010681, 0.16411742568016052, -1.3773685693740845, -0.5574047565460205, -0.7266663312911987, -1.2632272243499756, -1.6597566604614258, -0.48566582798957825, -0.8523231148719788, -0.5934658050537109, -1.485136866569519, 0.10103131830692291, 0.02548234537243843, -1.0858758687973022, -0.47576987743377686, -0.585755467414856, 1.348418116569519, 0.32276684045791626, 0.7920063138008118, 0.9429702162742615, 0.9775813817977905, 0.4234539270401001, -2.551961660385132, 0.013238082639873028, 0.2946242094039917, -0.8929789066314697, 0.9413708448410034, -0.2197040468454361, -0.35968470573425293, 2.1703014373779297], [-0.2995295226573944, 0.6301031708717346, 2.25947904586792, 0.9533759951591492, -0.8415756821632385, -1.628100872039795, -1.7335338592529297, 0.4498254954814911, 0.4586934447288513, -0.3914260268211365, 0.21908794343471527, -0.14956241846084595, -0.08799992501735687, 1.4235821962356567, -0.6079670786857605, -1.0544031858444214, 0.30300799012184143, 2.043882369995117, -0.8254686594009399, -0.17147691547870636, -0.4376447796821594, 0.4419404864311218, -0.12455913424491882, 0.35005736351013184, -0.8371225595474243, -0.146714985370636, -1.672374963760376, -2.0838918685913086, -0.37964749336242676, 0.233140766620636, -0.7520666718482971, 1.2593110799789429], [-0.23283232748508453, -0.5098824501037598, 0.552471399307251, 0.2143196016550064, -1.01484215259552, -1.1658931970596313, 3.1427340507507324, 1.1069194078445435, -0.315388560295105, 0.7242894768714905, 0.2617880702018738, 1.835159182548523, 0.22400076687335968, 0.3648439347743988, 1.9272898435592651, -1.361310601234436, 1.282662272453308, -1.5592681169509888, 1.1003706455230713, -1.8281196355819702, -0.3461977243423462, -0.12275756150484085, 0.329403281211853, 0.11222830414772034, -0.3423284590244293, -2.069561004638672, 1.51456618309021, -0.29412826895713806, 0.1149267703294754, -1.0384145975112915, 0.8641067147254944, -1.289754033088684], [-0.22008466720581055, -0.9859964847564697, 0.42834657430648804, -0.16833068430423737, 0.5405852794647217, -0.17561188340187073, -1.1331591606140137, 1.6094409227371216, -0.13346512615680695, 0.34973934292793274, -0.05583793297410011, -1.0401406288146973, 0.910414457321167, -0.6551429629325867, 0.6672707200050354, 0.14539502561092377, 0.33148500323295593, -0.7302101254463196, -2.19152569770813, 0.26604267954826355, -0.25348836183547974, -1.0558037757873535, -0.36029791831970215, 1.8229461908340454, 0.6147601008415222, 0.4785288870334625, 0.2703964114189148, -0.20156016945838928, -0.3408593535423279, 0.010323505848646164, -1.332814335823059, 0.3403851091861725]], "dtype": "float32"}, {"__nd__": [-0.00149571115616709, -0.08299796283245087, -0.7703281044960022, 1.1607931852340698, 1.0321760177612305, 0.9414279460906982, 0.2790106534957886, -0.38623619079589844, -0.5385637879371643, 0.6653020977973938, 1.2267175912857056, -0.20923040807247162, -1.2915549278259277, 0.45499253273010254, 0.3038255274295807, 0.10518201440572739, -1.4664390087127686, -0.0932297632098198, 1.126584768295288, -0.4285418391227722, -0.31813427805900574, -0.2676350772380829, 0.07302457839250565, -0.9759670495986938, -0.7402195930480957, -0.10009443759918213, 1.1112606525421143, 1.2327817678451538, -0.2485022097826004, -1.3538960218429565, -1.3365286588668823, -0.31896722316741943, -1.7238479852676392, 0.45852574706077576, 0.7408455014228821, 0.48871299624443054, -0.379784494638443, -0.3346948027610779, 1.3549211025238037, 1.0116755962371826, 0.11804523319005966, 0.2073068916797638, -0.2702596187591553, -0.31209054589271545, -1.733587622642517, -1.1406807899475098, 0.5488818883895874, -1.7018669843673706], "dtype": "float32"}], "e": {"__nd__": [[0.0, 2.1285736560821533, 1.0423935651779175, 0.0, 5.442904472351074, 0.0, 0.0, 0.8024419546127319, 0.0, 6.699124336242676, 3.4946539402008057, 1.953109622001648, 0.0, 0.0, 4.640079498291016, 0.0, 0.0, 0.0, 0.0, 9.23827838897705, 0.0, 0.0, 2.915709972381592, 0.0, 0.0, 0.0, 6.910502910614014, 10.174721717834473, 12.733207702636719, 0.2836054265499115, 3.0788309574127197, 0.0, 0.0, 0.0, 3.4120354652404785, 7.462117671966553, 0.0, 0.0, 8.637826919555664, 0.0, 2.9990453720092773, 0.8115595579147339, 0.0, 0.0, 0.0, 0.7779257893562317, 8.120113372802734, 2.6874771118164062], [4.2456488609313965, 0.0, 4.269219398498535, 2.9151105880737305, 5.1233344078063965, 0.0, 7.039480686187744, 3.7334861755371094, 0.7615585327148438, 0.0, 6.650959014892578, 0.0, 0.0, 2.275602102279663, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.5123047828674316, 0.0, 0.5339813232421875, 2.7374112606048584, 3.065312147140503, 6.574446678161621, 1.593970775604248, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.21044921875, 0.0, 0.0, 4.490991592407227, 7.362361907958984, 0.0, 2.1550331115722656, 4.0068535804748535, 0.7459192276000977, 0.0, 5.113205909729004, 0.0, 1.4892579317092896], [5.041065692901611, 1.5423426628112793, 0.0, 0.0, 0.0, 1.6291593313217163, 0.0, 0.0, 5.989804267883301, 6.303943634033203, 2.5554540157318115, 0.0, 0.19764162600040436, 0.0, 0.0, 0.0, 2.1121866703033447, 0.0, 0.0, 0.0, 4.968686103820801, 12.165299415588379, 2.670696973800659, 0.0, 2.6162679195404053, 6.0401177406311035, 4.255731105804443, 0.0, 0.0, 0.0, 3.9878151416778564, 0.0, 3.222682476043701, 0.0, 2.3561418056488037, 2.478029727935791, 0.0, 0.0, 0.0, 0.831226110458374, 0.0, 0.0, 5.860363483428955, 0.0, 0.0, 8.326523780822754, 0.0, 8.934036254882812], [0.9887823462486267, 1.3231154680252075, 4.31396484375, 2.695685386657715, 10.995560646057129, 0.8652965426445007, 2.5284652709960938, 0.24041223526000977, 0.0, 0.0, 0.20968398451805115, 0.0, 0.0, 0.0, 0.0, 0.0, 0.17172300815582275, 8.721607208251953, 0.0, 0.0, 3.469298839569092, 0.0, 4.0095601081848145, 10.12100887298584, 0.0, 4.031403541564941, 5.417357444763184, 3.586848020553589, 10.108516693115234, 1.945439338684082, 0.0, 1.1089481115341187, 0.0, 5.319990634918213, 0.0, 3.6931965351104736, 0.0, 1.7458715438842773, 16.557693481445312, 0.0, 0.0, 0.0, 0.0, 2.5485599040985107, 0.0, 1.6144957542419434, 0.0, 2.8362815380096436], [0.0, 8.814409255981445, 0.0, 1.103960633277893, 0.0, 2.0927188396453857, 2.506331205368042, 0.0, 0.0, 3.1804916858673096, 5.138622760772705, 1.1435776948928833, 0.0, 9.363390922546387, 0.0, 1.5717631578445435, 0.0, 7.391574382781982, 13.22497844696045, 7.769412517547607, 0.0, 0.0, 4.610374927520752, 0.0, 5.300815105438232, 0.0, 0.0, 5.992182731628418, 0.0, 0.0, 0.0, 3.5076091289520264, 0.0, 0.0, 0.0, 0.0, 0.0, 3.8186469078063965, 0.0, 0.0, 9.749693870544434, 0.0, 0.5945267677307129, 0.0, 0.0, 0.0, 3.0919392108917236, 0.0], [10.460498809814453, 13.373992919921875, 10.06875228881836, 6.1561689376831055, 5.570693492889404, 0.0, 0.0, 0.0, 0.0, 2.9179859161376953, 0.0, 0.0, 0.0, 8.255804061889648, 0.0, 6.155669689178467, 4.3592658042907715, 11.866466522216797, 0.0, 0.0, 7.194223403930664, 0.0, 8.97239875793457, 0.0, 1.7874397039413452, 1.6770591735839844, 9.056208610534668, 0.0, 0.0, 0.0, 8.937704086303711, 0.0, 0.0, 0.0, 4.478180408477783, 1.030923843383789, 3.605637311935425, 5.247701644897461, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.6155037879943848, 6.995429039001465, 0.0, 0.0], [0.0, 1.4863590002059937, 0.0, 5.000497817993164, 0.0, 0.0, 0.0, 7.369784832000732, 0.0, 0.0, 11.6287260055542, 0.0, 0.0, 6.980528831481934, 1.752698302268982, 12.026025772094727, 0.0, 6.742738723754883, 13.780482292175293, 4.303877353668213, 0.0, 0.0, 2.448585033416748, 0.0, 0.0, 0.0, 2.823995351791382, 0.0, 0.0, 15.305655479431152, 7.752734661102295, 7.850925445556641, 4.271237373352051, 0.0, 1.8183798789978027, 0.0, 0.0, 0.0, 0.0, 5.199323654174805, 7.236485004425049, 7.513820648193359, 2.0358033180236816, 0.0, 2.6136701107025146, 0.0, 5.9084272384643555, 0.0], [0.0, 0.0, 2.4889936447143555, 5.421693801879883, 0.0, 0.0, 0.0, 14.781768798828125, 0.0, 7.370825290679932, 0.0, 0.0, 1.4413670301437378, 4.987833499908447, 7.734472751617432, 4.832797050476074, 1.9616318941116333, 3.131856679916382, 0.0, 1.9256664514541626, 4.650928974151611, 0.0, 4.681326866149902, 0.0, 0.0, 3.0986075401306152, 11.656972885131836, 0.0, 0.0, 4.250032424926758, 2.0069847106933594, 0.0, 0.0, 0.0, 5.653417587280273, 5.813904285430908, 0.8032894134521484, 3.4921631813049316, 5.206395149230957, 0.0, 0.0, 14.741507530212402, 0.0, 0.0, 0.0, 0.0, 3.0138213634490967, 2.470085382461548], [0.0, 7.019678592681885, 1.1387689113616943, 0.0, 1.5371211767196655, 0.0, 0.7188028693199158, 0.0, 0.0, 0.9399030804634094, 1.0110307931900024, 0.0, 0.0, 4.115987777709961, 0.0, 1.4747908115386963, 0.32928192615509033, 7.3044352531433105, 2.6572139263153076, 0.0, 0.0, 0.0, 3.4544053077697754, 0.0, 0.0, 3.4907257556915283, 5.554821014404297, 0.0, 0.0, 2.3550329208374023, 4.534552574157715, 1.9496402740478516, 0.0, 0.0, 3.7480294704437256, 0.8943462371826172, 0.0, 0.0, 0.0, 0.0, 0.0, 1.935172438621521, 0.0, 0.0, 0.0, 3.486797332763672, 0.0, 0.4506154954433441], [0.0, 6.134592056274414, 0.0, 0.0, 4.587091445922852, 0.0, 8.805612564086914, 0.0, 0.8061132431030273, 0.0, 4.852094650268555, 4.258389949798584, 0.0, 1.8815906047821045, 0.0, 0.0, 2.8927414417266846, 7.032918930053711, 1.074469804763794, 0.0, 1.1713944673538208, 0.0, 0.0, 1.163909912109375, 0.0, 2.196486234664917, 1.8937692642211914, 0.3201444745063782, 1.1724218130111694, 0.0, 0.0, 6.862158298492432, 0.0, 0.8696154356002808, 0.7930717468261719, 5.218165874481201, 0.0, 4.3531904220581055, 0.0, 0.0, 0.0, 0.0, 6.265750885009766, 0.0, 0.0, 11.807004928588867, 0.0, 3.5260040760040283], [4.39868688583374, 0.0, 0.0, 0.0, 0.0, 0.0, 4.616073131561279, 13.990450859069824, 0.7127479910850525, 4.771547317504883, 4.714529514312744, 2.387167453765869, 5.488391399383545, 0.0, 0.0, 0.0, 3.7353994846343994, 4.791351318359375, 0.19403821229934692, 5.735518455505371, 4.436811447143555, 2.1271603107452393, 1.8346500396728516, 0.0, 2.3396966457366943, 2.8004403114318848, 3.9825093746185303, 0.0, 0.0, 0.9171766638755798, 0.0, 0.0, 0.9695588946342468, 1.9323108196258545, 3.053330183029175, 6.184090614318848, 0.0, 2.9984593391418457, 6.608315944671631, 0.0, 0.0, 0.0, 5.474327564239502, 2.0552518367767334, 0.0, 0.0, 5.8891777992248535, 2.0310704708099365], [0.745165228843689, 0.0, 8.21389102935791, 2.550750732421875, 0.0, 11.067765235900879, 0.0, 0.0, 0.0, 0.0, 0.4815308153629303, 0.0, 0.0, 0.0, 0.0, 0.10944484919309616, 1.570773720741272, 4.7246551513671875, 10.515244483947754, 4.84568452835083, 0.9954178929328918, 0.0, 0.0, 3.184114933013916, 0.0, 0.0, 2.635808229446411, 5.171509265899658, 3.944486379623413, 10.327022552490234, 0.6787264347076416, 5.243060111999512, 0.0, 1.460762619972229, 0.0, 4.564823150634766, 5.356270790100098, 0.0, 1.6878243684768677, 0.0, 0.0, 0.0, 0.0, 3.0186116695404053, 0.0, 1.9240390062332153, 8.042656898498535, 0.0], [0.0, 0.0, 6.339770793914795, 0.0, 0.0, 5.537355422973633, 4.752101421356201, 2.4530022144317627, 0.0, 0.0, 0.9244982600212097, 1.4284228086471558, 3.312744617462158, 0.0, 3.316974401473999, 0.0, 1.5306731462478638, 4.532298564910889, 5.934988975524902, 0.0, 10.319611549377441, 0.0, 0.18178904056549072, 0.0, 0.0, 4.508042335510254, 1.0468353033065796, 0.08067760616540909, 0.0, 2.3587164878845215, 3.3066890239715576, 5.078085422515869, 0.0, 0.0, 0.0, 2.083045244216919, 0.0, 0.0, 4.2574615478515625, 0.0, 0.0, 0.0, 0.0, 0.0, 4.080508708953857, 0.0980755165219307, 0.21997308731079102, 0.0], [2.271528482437134, 6.119204998016357, 2.918848991394043, 0.0, 9.350892066955566, 2.6317882537841797, 0.0, 0.0, 1.6824619770050049, 0.0, 0.5066714882850647, 4.534389495849609, 0.07442935556173325, 0.0, 0.0, 0.0, 0.0, 0.8988433480262756, 6.166220664978027, 4.566162586212158, 0.0, 0.0, 0.7938665151596069, 14.502943992614746, 0.0, 1.9496251344680786, 0.0, 6.315074443817139, 0.0, 0.0, 0.0, 0.8812850713729858, 10.489444732666016, 0.0, 0.0, 0.0, 0.0, 0.0, 3.678922414779663, 0.0, 0.0, 0.0, 2.531491279602051, 0.0, 0.0, 0.0, 7.111780166625977, 0.0], [0.0, 13.45681381225586, 0.0, 14.176339149475098, 6.460620403289795, 5.9024176597595215, 3.5744335651397705, 0.0, 0.0, 1.3291008472442627, 0.0, 4.156555652618408, 0.0, 11.521463394165039, 0.0, 5.058906555175781, 0.0, 5.895604133605957, 0.0, 0.5321499705314636, 0.0, 0.0, 0.5476540923118591, 0.0, 2.5981638431549072, 0.38444462418556213, 4.780421257019043, 2.4702672958374023, 0.7828444242477417, 0.0, 7.109001636505127, 0.0, 0.0, 0.0, 8.845681190490723, 0.6392915844917297, 9.812311172485352, 0.0, 0.0, 0.0, 4.23278284072876, 2.1076602935791016, 0.0, 0.0, 0.0, 3.8332602977752686, 0.0, 0.0], [4.189548969268799, 0.0, 6.979715824127197, 3.548738956451416, 0.0, 6.864908218383789, 7.295643329620361, 0.0, 0.0040415446273982525, 4.2634358406066895, 4.506359100341797, 0.7461198568344116, 2.7937917709350586, 0.7033400535583496, 0.0, 0.0, 1.786767840385437, 3.2399981021881104, 0.0, 0.0, 2.454564332962036, 1.0182205438613892, 0.0, 0.0, 3.3692760467529297, 0.0, 0.0, 2.4950172901153564, 3.384342908859253, 0.0, 0.0, 0.0, 0.0, 8.258768081665039, 3.318990468978882, 0.0, 3.063985824584961, 0.0, 4.842347145080566, 4.164753437042236, 0.3880559504032135, 0.0, 1.2843152284622192, 8.260701179504395, 0.6262897849082947, 6.164095401763916, 0.0, 0.0], [5.387106418609619, 9.25150203704834, 0.0, 1.900464415550232, 2.938663959503174, 0.42815279960632324, 0.0, 6.494513034820557, 6.730531692504883, 0.0, 1.9889953136444092, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0602803230285645, 1.8389579057693481, 5.200116157531738, 3.3379604816436768, 0.0, 9.582618713378906, 1.6016665697097778, 0.0, 1.8139156103134155, 1.7859258651733398, 5.686970233917236, 0.0, 0.0, 2.1559407711029053, 2.9886209964752197, 0.0, 0.0, 0.0, 10.889143943786621, 2.2826457023620605, 0.0, 5.125669002532959, 0.0, 8.385560035705566, 0.43357330560684204, 1.5833468437194824, 6.156510829925537, 0.0, 0.0, 6.229405403137207, 5.7776970863342285, 7.9131364822387695], [1.526343822479248, 0.0, 11.746549606323242, 0.9006527066230774, 0.0, 0.0, 0.0, 0.0, 9.742055892944336, 10.853423118591309, 9.203438758850098, 0.28283950686454773, 0.8017264604568481, 0.0, 0.0, 0.0, 0.8270806670188904, 3.343445062637329, 0.3760954737663269, 0.0, 1.2764019966125488, 2.077852725982666, 1.9086707830429077, 0.0, 1.1204519271850586, 0.0, 0.0, 7.619329452514648, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.619734764099121, 0.0, 5.990091323852539, 0.0, 1.237934947013855, 10.622048377990723, 5.496884346008301, 0.0, 0.0, 0.0], [4.249402046203613, 4.067100524902344, 0.0, 6.597878932952881, 6.7749152183532715, 7.705191612243652, 0.0, 0.0, 5.010082244873047, 2.727536916732788, 0.0, 6.588559150695801, 0.0, 0.0, 1.228217601776123, 0.10569856315851212, 0.0, 0.0, 3.15291166305542, 3.488053321838379, 0.0, 7.894948959350586, 0.0, 0.07689454406499863, 5.765716075897217, 0.0, 0.0, 8.052343368530273, 7.169305801391602, 0.0, 0.0, 0.0, 0.0, 3.6689891815185547, 3.066800832748413, 0.0, 11.936925888061523, 0.0, 2.764301300048828, 11.084394454956055, 3.9542527198791504, 2.7192368507385254, 0.02265699952840805, 4.922915935516357, 0.0, 0.0, 5.6002984046936035, 0.0], [0.0, 0.36616748571395874, 0.0, 1.2153425216674805, 3.241986036300659, 0.0, 0.0, 5.463812828063965, 0.6630120277404785, 8.309508323669434, 4.574840545654297, 0.0, 0.0, 1.5556141138076782, 9.726659774780273, 10.481131553649902, 4.079000473022461, 0.0, 1.4523166418075562, 3.1544418334960938, 0.0, 0.0, 0.05281872674822807, 0.0, 0.0, 0.0, 4.264537811279297, 0.0, 2.0435683727264404, 3.8997597694396973, 0.0, 0.0, 3.773775100708008, 0.0, 2.9854040145874023, 0.0, 3.389688730239868, 0.0, 0.0, 4.46626091003418, 3.1177656650543213, 5.626540660858154, 0.0, 0.0, 12.090800285339355, 1.393659234046936, 0.0, 0.0], [0.3991347849369049, 0.0, 0.0, 0.0, 0.7842734456062317, 0.0, 6.3309173583984375, 12.65084171295166, 0.22869034111499786, 0.0, 0.0, 3.2365574836730957, 4.67825984954834, 0.0, 11.377246856689453, 0.0, 3.509965658187866, 3.202803373336792, 0.0, 13.268306732177734, 0.0, 0.2755853235721588, 0.0, 12.139023780822754, 0.0, 0.0, 0.0, 5.63198184967041, 10.365914344787598, 8.67673110961914, 0.0, 0.20322763919830322, 7.458799362182617, 6.636601448059082, 0.0, 1.3894927501678467, 4.113420009613037, 8.130743980407715, 4.763308525085449, 0.0, 0.0, 11.376977920532227, 0.0, 0.5126788020133972, 0.0, 0.0, 15.262520790100098, 0.0], [0.0, 0.0, 0.0, 1.3968063592910767, 0.0, 2.320035696029663, 0.0, 2.9244699478149414, 5.596467971801758, 0.0, 1.110784888267517, 0.0, 0.10476628690958023, 0.0, 3.3711190223693848, 0.0, 5.983989238739014, 5.850323677062988, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.47537270188331604, 10.803031921386719, 11.863517761230469, 0.0, 0.0, 0.0, 2.12929105758667, 0.0, 0.0, 0.087053082883358, 10.586383819580078, 4.239046573638916, 0.0, 0.0, 0.49055784940719604, 0.0, 0.0, 0.4248630702495575, 0.0, 7.518896579742432, 0.8812474608421326, 4.733733177185059, 0.0, 6.368834018707275], [0.0, 0.0, 3.9913363456726074, 0.8791689872741699, 0.0, 0.0, 0.0, 6.717475891113281, 2.790043830871582, 0.0, 7.23140287399292, 0.0, 0.0, 3.869961977005005, 2.88320255279541, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.8421965837478638, 5.025169372558594, 13.447198867797852, 0.0, 0.6169286966323853, 0.0, 0.0, 2.6046905517578125, 0.0, 0.0, 0.0, 8.162919044494629, 0.0, 0.0, 0.0, 0.0, 0.0, 2.672623634338379, 4.455541610717773, 2.0456879138946533, 3.9512102603912354, 11.527276039123535, 0.7406191229820251, 0.0], [0.0, 0.0, 15.474664688110352, 0.0, 10.163324356079102, 2.653733253479004, 0.0, 0.14974075555801392, 0.0, 2.2543344497680664, 3.0606424808502197, 0.0, 2.7731621265411377, 4.161101341247559, 4.242079734802246, 8.3613920211792, 0.0, 5.911620140075684, 0.0, 0.0, 3.632821798324585, 0.0, 5.281203269958496, 4.2781877517700195, 0.3618493378162384, 0.0, 0.0, 9.906709671020508, 3.863384962081909, 1.443652629852295, 5.87274694442749, 10.120967864990234, 1.8074476718902588, 6.801569938659668, 0.0, 0.0, 0.7770289182662964, 0.0, 10.145695686340332, 0.0, 8.926605224609375, 0.0, 0.0, 8.506182670593262, 2.332393169403076, 0.0, 0.7381215691566467, 0.0], [0.0, 0.0, 0.0, 2.1169397830963135, 0.0, 4.092377662658691, 7.204372406005859, 0.0, 0.0, 4.326094627380371, 11.539738655090332, 2.714841365814209, 0.0, 4.408123016357422, 3.152726888656616, 4.671706676483154, 0.0, 0.4423333406448364, 5.248305797576904, 0.6468570232391357, 1.432712435722351, 0.7069523930549622, 0.5988059043884277, 0.0, 0.0, 0.0, 0.0, 1.385940670967102, 0.007670330815017223, 4.00394868850708, 0.0, 4.019464015960693, 8.354394912719727, 7.375732421875, 0.0, 0.0, 0.0, 0.0, 0.0, 5.642588138580322, 0.0, 0.0, 2.936582088470459, 5.296200275421143, 0.0, 0.0, 0.0, 0.0], [0.0, 9.972107887268066, 2.18709659576416, 0.0, 0.0, 1.214503526687622, 0.0, 0.0, 2.112140655517578, 0.0, 0.0, 0.0, 0.0, 2.751816511154175, 0.0, 0.0, 0.0, 0.0, 3.5734848976135254, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.8235195279121399, 0.0, 3.744292736053467, 0.0, 0.0, 4.703268527984619, 0.0, 0.0, 0.0, 0.676033079624176, 9.99825382232666, 1.903017282485962, 0.0, 0.0, 0.0, 15.25513744354248, 1.2920501232147217, 0.0, 0.0, 8.238698959350586, 0.14040139317512512, 0.0], [2.309842586517334, 0.0, 0.0, 2.825505256652832, 0.0, 0.0, 0.0, 9.27376937866211, 0.0, 5.833527088165283, 2.318186044692993, 0.0, 6.8465704917907715, 3.789606809616089, 0.3073621988296509, 0.0, 3.174278736114502, 1.123540997505188, 4.889683723449707, 11.14773941040039, 0.0, 0.0, 3.42891526222229, 0.0, 0.0, 0.0, 1.7681360244750977, 0.0, 0.0, 6.860082149505615, 0.0, 0.0, 0.0, 2.7022128105163574, 6.552374362945557, 0.0, 2.8720602989196777, 0.0, 0.0, 0.6910818815231323, 4.496352672576904, 4.6952314376831055, 0.0, 0.7793764472007751, 1.367119550704956, 0.0, 9.665548324584961, 0.0], [0.0, 0.0, 0.0, 0.5292848348617554, 0.0, 2.844257116317749, 0.0, 0.0, 3.5681259632110596, 7.418327331542969, 6.7667717933654785, 0.0, 7.24042272567749, 0.0, 8.861337661743164, 0.0, 3.651796579360962, 7.038299560546875, 5.74331521987915, 0.0, 0.0, 1.7713158130645752, 0.0, 0.0, 0.0, 1.0424445867538452, 1.7531818151474, 0.0, 0.0, 2.841329336166382, 0.0, 11.895181655883789, 2.989053249359131, 5.379938125610352, 7.310769081115723, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.39025354385376, 1.5226560831069946, 0.0, 0.0], [0.0, 0.7795847654342651, 1.9299472570419312, 0.0, 7.5615739822387695, 12.538518905639648, 5.62725830078125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.9339375495910645, 2.0830044746398926, 3.3749871253967285, 0.0, 0.0, 0.0, 2.470654010772705, 0.0, 0.823043704032898, 5.98292350769043, 3.557467222213745, 1.5312936305999756, 13.988218307495117, 0.0, 3.5653281211853027, 0.0, 0.0, 0.0, 0.0, 3.023571014404297, 0.0, 0.0, 0.0, 5.362033367156982, 2.991225004196167, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.726172685623169, 2.3867433071136475, 9.887040138244629, 3.0313832759857178, 0.0, 9.305837631225586, 0.0, 4.012678146362305, 0.0, 4.945522785186768, 0.0, 0.0, 0.0, 0.87528395652771, 2.8469831943511963, 0.0, 2.5349037647247314, 14.957087516784668, 0.0, 0.0, 0.0, 0.0, 7.039863586425781, 0.0, 0.0023361151106655598, 0.0, 1.9877451658248901, 4.6568827629089355, 0.0, 2.1840250492095947, 3.62007474899292, 0.2756563723087311, 0.0, 0.0, 3.982835054397583, 0.0, 4.613918304443359, 3.1439943313598633, 6.414629936218262, 0.0, 8.392568588256836, 3.821772575378418, 0.0, 0.0, 0.0, 3.0726327896118164, 0.0, 0.34518107771873474], [0.8794189691543579, 0.0, 2.643099546432495, 0.0, 1.3400686979293823, 2.368525266647339, 4.489722728729248, 0.0, 0.0, 0.0, 0.9297460317611694, 0.0, 0.0, 3.091820478439331, 0.0, 0.0, 0.0, 0.0, 3.599308729171753, 0.0, 0.0, 1.9840683937072754, 5.542945384979248, 0.0, 1.3397600650787354, 9.232060432434082, 2.8630924224853516, 2.7176401615142822, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3981049060821533, 0.0, 0.0, 7.33560037612915, 7.043327331542969, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.6910223960876465, 3.1879043579101562], [2.5865886211395264, 8.051713943481445, 1.358010172843933, 5.098137378692627, 0.44297468662261963, 3.706974506378174, 2.7413125038146973, 0.0, 0.0, 0.0, 0.0, 2.5735085010528564, 3.7093095779418945, 0.0, 0.0, 0.9305495023727417, 0.0, 1.1364330053329468, 11.362913131713867, 2.484363317489624, 1.3511031866073608, 3.010236978530884, 0.2895531952381134, 2.822331428527832, 0.0, 0.0, 0.0, 2.4756174087524414, 7.711685657501221, 0.0, 2.455066204071045, 2.344270706176758, 0.0, 3.6638131141662598, 0.0, 1.926048994064331, 6.230532646179199, 5.170382499694824, 7.250053405761719, 0.0, 3.1554653644561768, 0.0, 0.0, 0.0, 0.0, 0.0, 1.168068766593933, 0.0], [0.0, 1.7076901197433472, 2.536839246749878, 2.956641674041748, 3.906665802001953, 2.1853394508361816, 2.920468807220459, 0.0, 0.0, 10.553586959838867, 11.021768569946289, 8.206711769104004, 3.0252065658569336, 0.0, 6.406909942626953, 0.0, 0.574068546295166, 2.525190591812134, 7.852892875671387, 0.7303289175033569, 7.3985185623168945, 0.1564202755689621, 0.0, 0.09898354858160019, 8.159148216247559, 0.0, 0.0, 2.064440965652466, 0.8294491171836853, 0.0, 0.0, 9.355992317199707, 2.3276093006134033, 2.8436288833618164, 0.0, 3.962933301925659, 0.0, 0.0, 6.662229061126709, 8.177871704101562, 3.392941951751709, 0.0, 2.9897408485412598, 0.0, 5.561020851135254, 4.173521518707275, 0.0, 0.0], [0.0, 3.193417549133301, 4.694547176361084, 0.0, 1.5203216075897217, 9.902976036071777, 3.0160701274871826, 0.0, 0.0, 0.0, 0.8418301939964294, 3.424250364303589, 0.0, 3.287963628768921, 4.173018932342529, 1.2923539876937866, 0.0, 7.870296478271484, 0.0, 0.0, 0.3342418670654297, 0.0, 0.0, 5.200263023376465, 4.110108375549316, 3.501157760620117, 0.0, 4.725337505340576, 6.085001468658447, 0.0, 0.0, 7.770850658416748, 0.0, 0.0, 0.0, 4.462127208709717, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3897947072982788, 0.0, 10.415603637695312, 0.0, 0.0], [7.3743367195129395, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.458164215087891, 0.0, 14.45896053314209, 10.733414649963379, 0.0, 0.0, 0.0, 14.486405372619629, 7.786245822906494, 0.0, 0.0, 9.751123428344727, 0.0, 0.0, 0.0, 0.0, 0.0, 11.493925094604492, 3.1729631423950195, 0.0, 0.0, 0.0, 0.15470388531684875, 0.0, 0.0, 4.60995626449585, 0.0, 9.136388778686523, 3.900331735610962, 0.0, 0.0, 0.0, 15.484387397766113, 0.0, 2.073824405670166, 3.455505847930908, 0.0, 10.349813461303711, 0.12735483050346375, 1.3690376281738281, 2.161956310272217], [0.14089804887771606, 10.757434844970703, 0.0, 2.2057130336761475, 0.0, 0.0, 0.0, 0.0, 5.574253082275391, 2.9291536808013916, 0.0, 1.654892921447754, 0.0, 13.785224914550781, 0.0, 4.6554388999938965, 8.306605339050293, 0.093811996281147, 2.1484172344207764, 8.20778751373291, 0.0, 3.5473647117614746, 6.668403625488281, 1.1840587854385376, 0.0, 0.10013240575790405, 0.9210397601127625, 2.7254929542541504, 0.0, 2.070437431335449, 0.08908882737159729, 0.0, 5.239406108856201, 0.0, 3.0616769790649414, 1.411467432975769, 0.0, 0.0, 0.0, 1.6153899431228638, 0.0, 1.8177709579467773, 0.0, 0.0, 0.0, 0.0, 1.7294100522994995, 0.0], [2.2246711254119873, 0.0, 0.0, 0.0, 0.0, 7.667860984802246, 4.088311672210693, 0.6830466985702515, 0.0, 1.2726448774337769, 1.572447419166565, 2.744364023208618, 4.4106764793396, 0.0, 7.446930885314941, 0.0, 0.0, 0.0, 0.7978693842887878, 0.59566730260849, 3.64306640625, 0.5577145218849182, 1.0884532928466797, 0.0, 8.43924331665039, 8.849933624267578, 2.0394363403320312, 0.0, 0.0, 0.0, 0.0, 0.0, 0.34338581562042236, 0.7320241928100586, 4.379345417022705, 0.0, 0.0, 0.0, 5.070287704467773, 10.648297309875488, 0.0, 0.8704896569252014, 0.0, 3.853489875793457, 0.0, 0.0, 3.9171791076660156, 2.852226972579956], [0.0, 0.8805131316184998, 0.0, 7.770968914031982, 6.036744594573975, 3.826058864593506, 1.1779581308364868, 0.0, 0.0, 1.30368173122406, 0.0, 4.9605560302734375, 0.0, 0.0, 0.6094533801078796, 0.0, 3.6750400066375732, 1.3369576930999756, 0.0, 2.3714842796325684, 0.0, 6.393645286560059, 3.246584892272949, 0.0, 0.0, 1.5819084644317627, 3.937458038330078, 3.107952356338501, 8.847319602966309, 0.0, 4.123326778411865, 0.0, 0.0, 6.12289571762085, 5.305492877960205, 4.308157920837402, 6.049161434173584, 1.0376189947128296, 4.622772693634033, 0.0, 0.0, 0.0, 0.0, 1.1485154628753662, 0.0, 3.8259057998657227, 0.0, 2.8373498916625977], [0.0, 0.0, 2.3325963020324707, 0.0, 0.7006638050079346, 12.630804061889648, 3.97442626953125, 0.0, 0.0, 0.0, 5.355246543884277, 2.7841458320617676, 0.0, 0.0, 0.0, 5.335392475128174, 0.0, 0.0, 8.571086883544922, 0.0, 0.0, 0.0, 0.9433547258377075, 0.0, 1.590639352798462, 4.7237372398376465, 0.0, 0.0, 0.0, 5.453257083892822, 0.012914630584418774, 0.0, 0.0, 7.516787528991699, 1.1581720113754272, 0.0, 3.6531641483306885, 0.0, 5.572875022888184, 1.387521743774414, 0.0, 1.410888910293579, 0.5666925311088562, 2.749180316925049, 0.0, 0.0, 0.0, 3.1444787979125977], [3.488011598587036, 0.0, 0.0, 11.659337997436523, 0.4721684157848358, 1.2721269130706787, 0.0, 12.925040245056152, 0.0, 0.0, 0.0, 0.0, 0.0, 4.212585926055908, 3.1689634323120117, 0.0, 0.0, 0.0, 0.9731938242912292, 11.61513900756836, 0.0, 0.2907494604587555, 3.600125551223755, 0.0, 0.0, 1.637913465499878, 18.73712158203125, 2.132786989212036, 0.0, 12.04587459564209, 0.0, 0.0, 0.0, 0.14562979340553284, 6.709623336791992, 0.9620014429092407, 3.848330020904541, 0.12510982155799866, 0.0, 0.7587388157844543, 0.0, 8.576930046081543, 0.0, 2.060164213180542, 0.0, 0.0, 22.450481414794922, 0.0], [0.0, 8.21557331085205, 0.0, 0.0, 4.722404956817627, 0.008729270659387112, 0.0, 1.5828944444656372, 3.3220791816711426, 0.09853137284517288, 3.959444284439087, 0.9484894871711731, 0.0, 0.0, 1.4393314123153687, 0.0, 5.587447166442871, 0.0, 0.0, 11.78407096862793, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.037796974182129, 0.0, 10.823128700256348, 10.178956985473633, 11.6214599609375, 5.390414237976074, 0.0, 3.8445427417755127, 4.997046947479248, 2.373244524002075, 3.349133253097534, 0.0, 1.4923498630523682, 0.0, 1.435305118560791, 14.853336334228516, 4.425617694854736, 0.0, 0.0, 4.361728668212891, 6.730782985687256, 7.400422096252441], [1.596229910850525, 0.0, 4.405272006988525, 0.0, 0.0, 16.997314453125, 3.2647206783294678, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.44767072796821594, 5.863646984100342, 3.310514211654663, 0.0, 0.17348842322826385, 5.4598541259765625, 0.0, 0.0, 0.04024645686149597, 4.573793888092041, 0.0, 2.392396926879883, 6.2410664558410645, 0.6985435485839844, 4.341748237609863, 0.0, 0.1649288833141327, 0.0, 0.0, 0.0, 0.0, 1.7941434383392334, 0.0, 3.7262394428253174, 1.326788067817688, 0.0, 0.0, 0.0, 0.0, 0.0, 4.72836971282959, 0.0, 0.0, 5.829357624053955, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 2.516385078430176, 7.444145679473877, 0.9635019302368164, 4.041767120361328, 0.0, 6.246160984039307, 5.20502233505249, 8.972298622131348, 0.0, 0.0, 0.0, 3.3160030841827393, 0.0, 6.165568828582764, 0.0, 5.301736354827881, 12.267361640930176, 12.92187213897705, 0.0, 0.0, 8.882613182067871, 0.0, 3.2754623889923096, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6261118650436401, 1.174582600593567, 0.0, 0.0, 8.446800231933594, 9.368118286132812, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.110021114349365], [0.8132021427154541, 0.1309518814086914, 0.7133734822273254, 0.0, 0.0, 4.603507995605469, 1.649843454360962, 0.0, 0.0, 5.610218048095703, 0.0, 4.372700214385986, 0.0, 2.732499599456787, 10.155498504638672, 5.296726703643799, 5.231879234313965, 0.0, 6.02944278717041, 0.0, 6.575834274291992, 0.0, 0.9398969411849976, 0.0, 2.9636991024017334, 4.493830680847168, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.310791015625, 0.0, 0.0, 0.0, 2.640887975692749, 0.0, 2.0669608116149902, 8.109284400939941, 0.0, 7.98600959777832, 0.0, 0.0, 2.296813726425171, 0.0, 0.0, 0.0], [0.0, 0.0, 7.068968296051025, 0.0, 6.167929172515869, 0.0, 0.0, 7.0155744552612305, 0.0, 3.0700998306274414, 7.1204705238342285, 0.0, 0.0, 0.0, 0.8168790936470032, 0.0, 0.5144136548042297, 0.0, 3.1741487979888916, 0.0, 0.0, 0.0, 0.6236118674278259, 0.0, 0.0, 0.0, 0.7697480916976929, 1.2864975929260254, 3.003645658493042, 0.0, 4.03366231918335, 6.249957084655762, 3.4235713481903076, 0.0, 0.0, 3.7648861408233643, 2.132138967514038, 0.0, 0.0, 1.6472214460372925, 1.6890989542007446, 3.657780170440674, 0.0, 0.0, 11.59492015838623, 1.8651297092437744, 0.6479136943817139, 0.0], [0.0, 0.0, 0.0, 6.390517234802246, 2.251476764678955, 0.0, 6.939739227294922, 0.0, 5.161502361297607, 0.0, 0.0, 4.5658063888549805, 0.0, 6.620608329772949, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6188466548919678, 0.0, 0.0, 0.0, 0.0, 0.0, 1.7119539976119995, 0.0, 0.0, 0.0, 0.3753115236759186, 0.0, 8.123392105102539, 6.1783833503723145, 0.0, 2.8523147106170654, 0.0, 0.0, 0.0, 0.0, 3.7118496894836426, 2.0151724815368652, 0.9296443462371826, 0.0, 0.0, 0.0, 2.5935401916503906], [6.570703029632568, 0.0, 8.867546081542969, 1.3853111267089844, 0.0, 12.668397903442383, 0.0, 0.0, 0.0, 1.806158423423767, 1.8801000118255615, 0.0, 0.0, 0.0, 6.19145393371582, 3.855889320373535, 0.0, 0.0, 8.779823303222656, 0.0, 0.0, 1.831313967704773, 7.505661487579346, 0.0, 2.3243143558502197, 5.929534912109375, 5.926339149475098, 5.602737903594971, 0.0, 4.903189182281494, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.332730770111084, 0.0, 5.6425981521606445, 2.8625667095184326, 1.4415806531906128, 0.0, 0.0, 0.0, 0.0, 0.0, 2.155630588531494, 0.0], [2.703371524810791, 0.0, 0.0, 0.0, 2.6348209381103516, 0.0, 0.0, 0.0, 6.274152755737305, 2.412780284881592, 0.0, 5.617084980010986, 2.1525931358337402, 0.0, 0.0, 0.3234279155731201, 14.387099266052246, 8.610407829284668, 0.0, 0.0, 3.853048324584961, 9.530989646911621, 15.188817024230957, 4.111597061157227, 0.0, 0.19140173494815826, 2.0342600345611572, 1.2252122163772583, 0.0, 4.215709209442139, 0.0, 0.0, 6.1941447257995605, 1.4046181440353394, 0.0, 0.04454588145017624, 0.0, 0.44761770963668823, 1.9054139852523804, 0.0, 0.0, 0.0, 0.0, 5.396332263946533, 0.0, 0.0, 0.0, 8.398706436157227], [1.6662708520889282, 1.563020944595337, 6.136031627655029, 0.04783359915018082, 0.0, 0.0, 0.0, 3.92706561088562, 0.0, 0.0, 0.5335589647293091, 0.0, 0.0, 1.0717638731002808, 0.0, 0.0, 0.0, 0.0, 7.255035877227783, 3.1330628395080566, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.554507255554199, 1.7666290998458862, 7.872922897338867, 3.5935893058776855, 0.0, 1.2764900922775269, 0.0, 0.0, 0.0, 1.8158364295959473, 5.461647033691406, 0.9090309739112854, 2.273503065109253, 2.0967748165130615, 0.0, 6.417019844055176, 3.22001576423645, 2.316978693008423, 0.0, 0.8817418217658997, 12.681674003601074, 0.0], [3.2441608905792236, 2.500438928604126, 0.0, 0.2822062373161316, 0.0, 0.5758777856826782, 0.0, 0.0, 0.0, 16.540311813354492, 0.8484948873519897, 0.0, 3.040989398956299, 0.3053232431411743, 1.5827041864395142, 0.0, 0.0, 6.366929054260254, 7.610840320587158, 2.98439884185791, 6.442444324493408, 1.1932026147842407, 9.877318382263184, 0.0, 0.0, 12.371506690979004, 3.7389566898345947, 0.0, 0.0, 2.6827704906463623, 0.0, 0.0, 0.0, 2.1766602993011475, 13.723794937133789, 0.3415231704711914, 0.0, 4.30722713470459, 0.0, 0.0, 0.0, 0.0, 2.367640495300293, 0.0, 0.0, 0.0, 0.0, 7.947880268096924], [0.0, 1.5778549909591675, 4.162528991699219, 0.0, 5.919225692749023, 11.191596031188965, 0.0, 11.811452865600586, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 15.117937088012695, 0.0, 6.080652236938477, 12.506417274475098, 1.7104241847991943, 1.2922810316085815, 0.0, 0.0, 0.0, 9.55844783782959, 0.0, 5.286716938018799, 8.511560440063477, 3.7852256298065186, 0.0, 15.657370567321777, 4.226840019226074, 9.115893363952637, 0.0, 0.0, 4.022599220275879, 6.962386608123779, 0.0, 0.0, 9.772225379943848, 0.0, 0.0, 5.946785926818848, 0.0, 0.0, 0.0, 8.440566062927246, 6.231963157653809, 1.5766302347183228], [0.0, 0.0, 1.6303956508636475, 0.0, 0.0, 0.0, 0.0, 0.0, 3.9874894618988037, 6.219648361206055, 7.155148506164551, 0.32290220260620117, 2.1945853233337402, 0.0, 0.0, 0.9755554795265198, 0.0, 0.0, 10.892558097839355, 0.0, 3.034369468688965, 1.3624240159988403, 5.194198131561279, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5784652233123779, 7.679268836975098, 1.086702585220337, 0.0, 0.0, 0.0, 0.010651237331330776, 0.0, 11.698724746704102, 4.480192184448242, 1.7366300821304321, 4.495731353759766, 8.446596145629883, 6.6905717849731445, 0.0, 0.6336216926574707, 0.0], [0.0, 0.0, 5.638230800628662, 0.004584379959851503, 4.516967296600342, 0.0, 0.0, 2.8450851440429688, 0.0, 8.450241088867188, 0.0, 0.0, 2.7704977989196777, 2.192399501800537, 5.391164302825928, 7.212283134460449, 0.0, 0.0, 1.3099629878997803, 2.6023900508880615, 7.6624369621276855, 0.0, 0.23081345856189728, 0.0, 0.9640815854072571, 2.3025996685028076, 2.989144802093506, 0.0, 1.518315076828003, 0.0, 1.8764219284057617, 5.090746879577637, 1.095825433731079, 1.0778577327728271, 0.0, 0.0, 0.0, 1.1451839208602905, 3.226457118988037, 3.592926025390625, 3.77764892578125, 0.99916672706604, 0.0, 0.0, 9.514542579650879, 0.0, 0.0, 0.0], [0.10408226400613785, 0.0, 0.0, 4.647604465484619, 6.115428447723389, 3.3358137607574463, 0.0, 0.0, 0.0, 0.9678249359130859, 2.1051442623138428, 0.0, 0.0, 4.155891418457031, 0.0, 2.432936668395996, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.202030897140503, 1.59113609790802, 0.0, 0.0, 0.11141344904899597, 2.5639617443084717, 0.0, 2.3913705348968506, 1.2703686952590942, 0.0, 0.0, 8.377989768981934, 3.679828405380249, 0.0, 6.083482265472412, 0.0, 0.0, 0.0, 6.690446853637695, 0.9936556220054626, 0.0, 5.382875442504883, 0.0, 0.0, 1.373663306236267, 0.0], [0.0, 0.0, 1.772863745689392, 3.3682503700256348, 0.0, 0.0, 4.967734336853027, 1.4813743829727173, 8.382902145385742, 0.0, 4.045224189758301, 5.5208330154418945, 0.0, 0.0, 0.0, 2.875425100326538, 0.784225583076477, 0.0, 3.7043817043304443, 0.0, 0.0, 3.7507355213165283, 0.0, 0.0, 11.14822769165039, 0.23196843266487122, 1.1311966180801392, 0.0, 0.0, 0.0, 5.465547561645508, 0.0, 0.0, 1.8581703901290894, 3.9048268795013428, 0.0, 0.35723960399627686, 0.0, 0.0, 6.002956867218018, 4.227107524871826, 0.7965563535690308, 0.0, 1.1933295726776123, 5.007829666137695, 4.3121256828308105, 0.0, 0.0], [0.0, 4.0550384521484375, 0.0, 3.0644052028656006, 2.613129138946533, 2.0207204818725586, 2.142444372177124, 5.116476535797119, 1.1704984903335571, 0.0, 0.0, 3.167872190475464, 0.0, 0.0, 2.6579391956329346, 0.0, 0.0, 0.0, 8.355875968933105, 1.699561357498169, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9501600861549377, 0.0, 13.131879806518555, 2.2806832790374756, 2.2121739387512207, 0.654784083366394, 0.0, 2.3948206901550293, 1.170609474182129, 0.0, 11.1085205078125, 6.589502334594727, 0.0, 0.11947929114103317, 0.31483274698257446, 15.92365550994873, 0.0, 0.0, 1.9845707416534424, 0.0, 8.112698554992676, 0.0], [0.0, 0.0, 0.0, 0.0, 2.0899908542633057, 3.33402681350708, 8.423816680908203, 0.0, 0.0, 0.0, 4.516607761383057, 1.3150802850723267, 0.0, 9.13175106048584, 0.0, 0.0, 0.0, 4.7293219566345215, 0.21662236750125885, 6.564695835113525, 1.8740814924240112, 0.0, 0.0, 3.785982131958008, 0.0, 0.8762678503990173, 0.0, 6.328816890716553, 0.0, 1.1974680423736572, 0.0, 4.17180871963501, 4.563678741455078, 0.0, 0.0, 0.0, 0.0, 1.6894193887710571, 0.06333405524492264, 0.0, 2.7067580223083496, 0.0, 3.2401654720306396, 0.0, 0.0, 0.0, 2.078727960586548, 0.0], [1.8648362159729004, 8.0935697555542, 0.0, 0.0, 0.0, 0.0, 9.339898109436035, 0.0, 0.0, 0.0, 0.0, 0.243191197514534, 0.0, 4.009814262390137, 0.5390645861625671, 3.4558844566345215, 4.1111602783203125, 12.968177795410156, 2.5148959159851074, 0.0, 12.68143081665039, 0.0, 0.0, 7.696677207946777, 0.0, 7.542220592498779, 0.0, 0.0, 0.0, 0.0, 0.0, 7.622243404388428, 12.600834846496582, 0.0, 0.0, 0.0, 0.0, 5.784897804260254, 0.04468632861971855, 10.383857727050781, 0.0, 0.4103488326072693, 1.398722767829895, 0.0, 5.977290153503418, 0.6088047027587891, 0.0, 0.0], [0.0, 0.0, 0.0, 3.662872314453125, 1.887843370437622, 7.258304119110107, 9.579733848571777, 0.0, 7.905553817749023, 0.0, 8.701899528503418, 0.11874695122241974, 0.35719385743141174, 0.0, 0.0, 0.18112102150917053, 0.0, 0.0, 0.0, 0.0, 2.142935037612915, 12.766973495483398, 0.0, 0.04095916077494621, 2.124706745147705, 6.008831977844238, 12.194673538208008, 3.0551884174346924, 0.0, 0.0, 0.0, 0.0, 0.0, 5.536437034606934, 0.0, 0.0, 0.0, 2.9897663593292236, 0.0, 8.329253196716309, 0.0, 0.0, 8.897850036621094, 8.233102798461914, 0.0, 1.540115237236023, 1.2411860227584839, 6.494675159454346], [0.0, 0.0, 0.0, 4.10267448425293, 0.0, 4.073468208312988, 0.0, 6.458726406097412, 0.0, 0.9935699701309204, 0.0, 0.0, 0.0, 2.9647412300109863, 4.2745161056518555, 6.6314167976379395, 0.0, 0.2726149559020996, 3.6820247173309326, 4.990070819854736, 0.0, 0.0, 2.1385109424591064, 0.0, 3.2590279579162598, 0.0, 8.527944564819336, 3.8277602195739746, 0.0, 4.54369592666626, 0.0, 0.0, 0.0, 0.0, 4.833049774169922, 4.217574596405029, 0.0, 0.0, 2.1570146083831787, 1.0829548835754395, 4.489745616912842, 3.805933952331543, 0.0, 0.0, 0.0, 0.6563212275505066, 11.615222930908203, 0.0], [0.0, 9.355277061462402, 0.0, 1.473645567893982, 4.298467636108398, 0.0, 0.9486613869667053, 0.0, 0.0, 6.382765769958496, 10.724740982055664, 3.7815439701080322, 0.0, 0.0, 7.974704742431641, 0.0, 0.0, 0.0, 1.1649619340896606, 10.696924209594727, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5873275399208069, 6.509660720825195, 0.14773564040660858, 9.143845558166504, 1.8217402696609497, 5.361165523529053, 0.0, 0.0, 10.483193397521973, 0.0, 0.0, 10.728463172912598, 0.0, 1.9506757259368896, 2.284820556640625, 1.350691556930542, 0.0, 0.0, 0.0, 2.9385132789611816, 0.0], [3.348146438598633, 0.0, 0.0, 8.214576721191406, 3.1680502891540527, 0.0, 0.0, 5.893457412719727, 0.012696249410510063, 3.796785354614258, 0.0, 0.0, 5.286221027374268, 3.045402765274048, 0.7097906470298767, 0.0, 0.0, 7.602560043334961, 0.0, 1.6600961685180664, 0.4142175614833832, 0.0, 1.2521750926971436, 1.238830327987671, 2.993222951889038, 3.0568621158599854, 4.97064208984375, 1.9195882081985474, 2.1421539783477783, 1.0031486749649048, 0.0, 0.0, 0.0, 8.066452026367188, 11.291903495788574, 0.0, 0.0, 1.095199704170227, 0.0, 0.23939619958400726, 7.2704315185546875, 0.0, 0.0, 3.518362522125244, 0.0, 0.0, 3.3102359771728516, 0.0], [1.1909065246582031, 1.6915827989578247, 5.090652942657471, 0.0, 3.9716808795928955, 7.43742561340332, 0.0, 0.1740516573190689, 0.0, 0.0, 3.0535855293273926, 2.3332645893096924, 0.0, 10.560056686401367, 0.9103378653526306, 12.085709571838379, 0.0, 0.26134923100471497, 5.58430814743042, 0.9999423623085022, 0.0, 6.863731384277344, 2.328538417816162, 5.462152004241943, 0.0, 0.0, 2.4798874855041504, 6.976773738861084, 0.0, 14.387727737426758, 0.0, 1.757678508758545, 0.0, 0.0, 0.0, 1.4134571552276611, 4.168590068817139, 2.9300696849823, 0.0, 0.0, 4.327141761779785, 2.417163610458374, 3.2574574947357178, 0.9338800311088562, 0.0, 0.0, 5.225123882293701, 0.0], [0.0, 0.0, 0.3471689820289612, 1.697574496269226, 3.437840700149536, 7.313679218292236, 0.0, 0.0, 0.0, 5.054795265197754, 0.8324772119522095, 0.0, 0.0, 0.0, 0.0, 2.4304990768432617, 0.0, 0.0, 5.8354339599609375, 0.0, 0.0, 5.3825483322143555, 9.112720489501953, 0.0, 0.0, 4.118463039398193, 4.296041965484619, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5211674571037292, 3.3247931003570557, 0.0, 4.1130547523498535, 0.0, 2.7920548915863037, 4.54401159286499, 0.0, 0.0, 0.0, 0.0, 0.0, 0.29640430212020874, 0.0, 14.035836219787598]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0606515407562256, 2.5576534271240234, 0.5936883687973022, 0.23176610469818115, 0.9613128900527954, -1.2646461725234985, 1.3731149435043335, -0.5974401235580444, 0.04319300875067711, -0.48249000310897827, -0.723338782787323, -1.6154905557632446, -0.4448525607585907, 0.0941752940416336, -0.7764905095100403, -0.6517080068588257, 0.28686538338661194, -1.3865681886672974, -0.314679354429245, 1.3083964586257935, -0.8168781399726868, 1.793724775314331, -1.4119395017623901, -2.0673182010650635, 0.47149744629859924, 0.2804511487483978, 1.826268196105957, -1.8303433656692505, -0.2568034529685974, -0.2286052405834198, 1.6477737426757812, 0.9806665778160095, -0.565863847732544, 1.7276746034622192, -0.6895554661750793, 0.4894106984138489, -0.06973259150981903, -0.5303156971931458, -0.28159576654434204, -0.19735269248485565, -0.93824702501297, -0.465378075838089, 1.4331464767456055, -0.6645650863647461, 0.1424781233072281, 0.9919764995574951, -0.33521607518196106, 0.8405066728591919, 0.9105698466300964, -0.24149088561534882, 1.0878980159759521, 1.0261839628219604, 0.6229356527328491, 0.9619866013526917, -0.1912129670381546, 1.0500245094299316, 1.9443608522415161, -0.3185634911060333, 0.694871187210083, 0.250342458486557, -1.4697904586791992, 1.5217257738113403, -0.9101579189300537, 2.2377209663391113, -1.776975154876709, 0.1301470696926117, 1.3702024221420288, 0.5167412161827087, 0.04761958122253418, 0.8226932287216187], [0.6766005754470825, -1.3098571300506592, -2.846752405166626, 0.4138610363006592, -0.16285815834999084, -0.13656887412071228, 0.6744697093963623, 1.5701885223388672, 0.8276308178901672, -1.1841734647750854, 0.17045025527477264, 0.41270220279693604, -0.02480938844382763, -1.0953466892242432, 0.8723534941673279, -0.5818805694580078, -0.42556679248809814, 0.4063534438610077, 0.8614922761917114, -0.44499441981315613, 0.35574114322662354, 0.1363406926393509, 0.8392705917358398, -1.0249587297439575, -1.309643268585205, -0.7727394700050354, -0.08644112944602966, -0.16311143338680267, -1.2428535223007202, 1.0090736150741577, -1.6394296884536743, -1.4189156293869019, -1.0813531875610352, 0.7659176588058472, -1.0619311332702637, -0.18089689314365387, -1.6689541339874268, 0.049271490424871445, -1.234287142753601, 1.1323381662368774, 1.0178899765014648, 1.7857866287231445, -0.2223079949617386, -0.9803049564361572, -1.3473232984542847, 1.8815927505493164, 0.6492519378662109, -1.006654143333435, -0.3897116780281067, 0.3477683365345001, -0.4269031882286072, 1.3919873237609863, 1.3802881240844727, -1.664423942565918, 1.504850149154663, 0.7551701068878174, -0.9095001816749573, -0.3005005419254303, -1.4669389724731445, -0.0653160959482193, -0.8970653414726257, -0.590920090675354, -0.44845685362815857, -0.7380058765411377, 0.6357821822166443, -1.7968214750289917, 0.7404730916023254, 0.10997184365987778, -0.5550875663757324, -1.3289177417755127], [-2.7437000274658203, 1.31037175655365, -1.5708177089691162, 0.536404013633728, -1.029588222503662, 0.24101005494594574, -1.5153942108154297, 0.42844194173812866, -1.2839261293411255, 0.7207021713256836, 0.34690162539482117, -3.0786311626434326, -1.1212058067321777, 2.6370038986206055, 0.759689450263977, 1.4537869691848755, -0.560975193977356, -1.3321748971939087, -1.3779819011688232, -0.4166877865791321, -0.7535454034805298, 0.12598948180675507, -1.2669087648391724, 0.4214051365852356, -1.5818852186203003, -1.0159876346588135, 0.3872793912887573, 1.230928659439087, 1.2138593196868896, 0.18687234818935394, 0.5525246858596802, -1.505287528038025, 0.8349557518959045, 0.5081072449684143, -0.32631945610046387, 0.9151943922042847, 0.2603875398635864, -0.004322839435189962, -0.23115284740924835, -0.14769117534160614, 0.21976973116397858, -2.2074577808380127, 0.5996572971343994, 1.221523642539978, -1.285631775856018, -1.179508924484253, -0.10757448524236679, 1.3674356937408447, -0.1401415467262268, -0.5698738098144531, -0.924095094203949, 2.452349901199341, 1.3226443529129028, -0.45866021513938904, 2.2187695503234863, 0.27882903814315796, -1.5700469017028809, -1.008694052696228, -0.04398486018180847, -1.7461326122283936, -1.3915176391601562, -0.21592789888381958, 0.33216962218284607, -0.3461923599243164, 1.342668890953064, -0.5344446301460266, -0.26197707653045654, -0.18372277915477753, -1.0659645795822144, 0.3478214740753174], [-0.33288663625717163, 0.9879616498947144, -1.4483212232589722, -0.48416733741760254, -1.6870009899139404, -1.3497209548950195, -0.4537155032157898, -0.9057151675224304, 0.9759641885757446, 1.288335919380188, -0.7641544342041016, -0.18925826251506805, 0.04738175496459007, 0.09115267544984818, -0.3539312481880188, 1.573405385017395, -0.44093769788742065, 0.5888463258743286, -0.6542934775352478, 0.303667813539505, 0.9316606521606445, -0.20623435080051422, 1.188779354095459, -0.285518079996109, 0.17886221408843994, 0.10033684223890305, 1.0622148513793945, -1.0435256958007812, -0.8286218643188477, -0.9062567949295044, -0.7520270943641663, -0.7903410196304321, -1.2380585670471191, -0.3879297375679016, 0.7930052876472473, 0.26854485273361206, 1.8287311792373657, -0.8009293079376221, -1.3627558946609497, -0.931877076625824, -0.6562860608100891, -0.21938882768154144, 0.04003724455833435, -0.7366697192192078, 2.2742550373077393, 1.2828543186187744, -0.24042721092700958, -0.9052936434745789, -0.5787122249603271, 0.848113477230072, 0.2773800194263458, -1.0771453380584717, -0.7058082222938538, -1.0650317668914795, 0.4965279698371887, 1.5909719467163086, 1.271020531654358, -0.5126186609268188, 0.29263898730278015, -0.9666117429733276, 0.3870837390422821, 0.5842396020889282, -2.18204665184021, -0.3916947841644287, -0.616879940032959, -0.8418192863464355, -0.8784971237182617, 2.1999588012695312, 0.135885089635849, -0.52181077003479], [-0.7831057906150818, -0.4775305986404419, -0.8836738467216492, 0.007034923881292343, -1.2939465045928955, -0.5138521790504456, 0.6478519439697266, 0.7927893400192261, -2.0770630836486816, 0.6224237680435181, -0.7305393218994141, 1.1006228923797607, -1.1090340614318848, -0.3578868508338928, 0.07790444791316986, 0.810039758682251, 0.8326632380485535, -0.8142804503440857, -0.8075596690177917, 1.0685213804244995, -0.6375274062156677, 1.7281121015548706, 0.6206420660018921, -1.1257976293563843, 0.8783038854598999, 0.287154883146286, -0.15611472725868225, -0.40108275413513184, 0.7605195641517639, -0.8415133357048035, -0.7743569612503052, -0.6364250183105469, -0.5199012160301208, 0.18260450661182404, 0.283248633146286, 0.7404756546020508, 2.0879714488983154, 0.6901607513427734, -0.45397377014160156, 0.1904764175415039, -0.02110900916159153, -1.2772133350372314, 0.32089993357658386, -1.1770026683807373, 0.19955706596374512, -0.298505574464798, -1.5423771142959595, 0.7496199011802673, 0.5204272270202637, 1.0692514181137085, -0.8045253753662109, -0.13570882380008698, 0.5083265900611877, 2.2248902320861816, 0.025404246523976326, -0.45330414175987244, -0.7948739528656006, 1.3723230361938477, 1.1682469844818115, -0.2565823793411255, 0.34177398681640625, 1.5878407955169678, 0.5785824656486511, -0.7034671902656555, -0.2700929045677185, 0.26389577984809875, 1.9471285343170166, -1.6089991331100464, 0.49996379017829895, 0.06392203271389008], [1.9967392683029175, 0.7585861682891846, -0.08193934708833694, -0.047330453991889954, -0.10414358973503113, 0.43727219104766846, 0.6535855531692505, -0.9515607953071594, -0.5527638792991638, 0.3468623459339142, 0.18839557468891144, -0.41417551040649414, -1.954186201095581, -0.8662412166595459, -0.37480321526527405, -0.4461090862751007, 1.099717617034912, -0.4511880576610565, -0.5392776727676392, 0.8030468225479126, 0.7588285803794861, -1.2788640260696411, 1.0438908338546753, 0.8585583567619324, -0.6350833177566528, -1.2371069192886353, 0.7282446026802063, -0.22408005595207214, -0.6741774678230286, -0.3325444459915161, -1.4000725746154785, 0.2080165296792984, 0.7646178603172302, 0.22004272043704987, 0.3572116792201996, -0.374392032623291, 0.37066376209259033, 0.9842724800109863, -1.350522756576538, -1.330812931060791, 0.1442011296749115, -0.22650845348834991, -0.9396080374717712, 0.882142961025238, 0.30342555046081543, -0.5404994487762451, 1.2938567399978638, -0.953309953212738, -1.9751800298690796, -1.3072799444198608, -0.5603880286216736, 0.12300649285316467, 0.5247988700866699, -0.663158118724823, 0.6474241614341736, 0.500895082950592, -0.750475287437439, 1.7637686729431152, 1.5677958726882935, -0.12196266651153564, 0.8282620310783386, -0.28548169136047363, -0.09227566421031952, -0.44458630681037903, 0.3333042860031128, -1.3369358777999878, 0.4989360570907593, 0.08656199276447296, 0.8661974668502808, 1.1016981601715088], [0.26456838846206665, 0.29045528173446655, -0.6088231205940247, -1.4495049715042114, 0.30986958742141724, 1.0243996381759644, 0.7278415560722351, -0.29304713010787964, -0.9596682190895081, -0.09346601366996765, 0.5215076804161072, -0.2669944763183594, 0.7033055424690247, 0.4303682744503021, -1.1716605424880981, -0.7632526159286499, -0.2428540587425232, 1.8977373838424683, 0.3566698729991913, 1.0319864749908447, -0.6609389781951904, -1.0770847797393799, -0.1966952681541443, 0.42041030526161194, 0.3781966269016266, -0.18645894527435303, 1.265936255455017, 0.5193565487861633, -0.6631333231925964, 2.4494285583496094, 0.7541011571884155, -0.1772412210702896, 0.19609034061431885, 0.12517808377742767, -1.8362140655517578, -0.5612788796424866, 0.5746454000473022, 0.36497896909713745, 0.6278236508369446, 0.29558658599853516, 2.175036668777466, -0.26174241304397583, -0.26277825236320496, -0.27151602506637573, -0.44672638177871704, 3.009488821029663, -0.20310798287391663, 0.005535218399018049, 1.2879786491394043, -0.4730186462402344, 1.8828054666519165, 0.4192592203617096, -0.34576416015625, 0.4864572584629059, 0.2331356257200241, 0.7235337495803833, -0.589115560054779, -0.33881640434265137, 2.073570728302002, 0.4759497344493866, -0.5745871663093567, -0.28814685344696045, 0.5119843482971191, -0.7400906085968018, 0.7328019738197327, 0.3071412146091461, -1.6304585933685303, -0.7575989961624146, 0.010888793505728245, -0.37451642751693726], [0.10044369846582413, -1.1287260055541992, -0.7157589793205261, 0.9314684271812439, -0.47101160883903503, -1.0490566492080688, -0.6686654090881348, -0.015675021335482597, 0.10725662857294083, -0.10617447644472122, -0.4371272027492523, 0.5558120012283325, 0.5248823165893555, -1.3714330196380615, 2.3806798458099365, 1.0506153106689453, 0.003237508237361908, 0.2947576940059662, 0.17720213532447815, 1.190466284751892, -0.23861746490001678, -0.5652115345001221, 0.003687373362481594, -1.480046272277832, -3.295769214630127, -0.05512268841266632, 0.8293001055717468, -1.9438589811325073, -0.9678758382797241, -0.7274370193481445, 0.4269048273563385, 1.014647364616394, 0.4055432975292206, -0.16195110976696014, 0.2519766688346863, 0.8575495481491089, 0.5044434070587158, -0.4989509880542755, -1.6685028076171875, 1.8573006391525269, -2.0999653339385986, 1.7090868949890137, 0.005425895098596811, 0.6264461874961853, 0.48125773668289185, 0.33947157859802246, 0.5400753021240234, -0.7090471982955933, 0.4377082586288452, -1.7366231679916382, 1.0358339548110962, -1.7547075748443604, -0.7353577017784119, 0.6081898212432861, 1.6877659559249878, 0.5757653117179871, -2.139949083328247, 0.27264663577079773, -0.5738770365715027, 0.058575376868247986, 1.8181025981903076, 0.4194963872432709, -0.25049418210983276, -1.5636801719665527, -1.3268548250198364, -0.1525784134864807, 0.5436217188835144, 0.13034476339817047, -0.9024050831794739, 0.8209434151649475], [-1.028293251991272, -0.17258381843566895, 1.314777135848999, -0.916950523853302, -0.23824745416641235, 0.3932700455188751, -2.0720479488372803, 0.13195247948169708, 0.17758844792842865, -1.7912038564682007, 0.24554239213466644, -1.7364027500152588, 1.4715677499771118, 0.1017184630036354, -1.2450425624847412, -2.0058584213256836, 1.8494433164596558, 0.0594472698867321, 1.7426447868347168, -1.6666988134384155, 0.30144819617271423, 0.5157173275947571, -0.3013840615749359, -1.2973294258117676, 0.08598963916301727, -0.3010564148426056, 1.249077320098877, -0.41762104630470276, -0.05413046479225159, 0.12494909763336182, 1.494951605796814, 0.6314712166786194, 0.6983479261398315, -1.272891640663147, 2.05590558052063, -1.3556667566299438, 0.12085409462451935, -1.3446789979934692, 1.0429636240005493, -0.8421379923820496, -1.5313876867294312, -0.37525415420532227, 0.6390547156333923, 0.599068820476532, -2.1799256801605225, -1.6927670240402222, 0.03858204558491707, 1.4275822639465332, 0.6794961094856262, -1.3148348331451416, -1.7312465906143188, -0.6493779420852661, -0.947174072265625, 0.9862803220748901, 0.5639150142669678, -1.6171741485595703, -0.2590656280517578, -0.5023130774497986, -0.6170729994773865, 1.1494427919387817, 0.3069354295730591, 1.0521560907363892, -0.2629881501197815, 1.4400379657745361, -2.420506477355957, -1.333400845527649, -0.8282564282417297, -0.6763657927513123, -0.3958495557308197, 0.47622403502464294], [0.2977296710014343, 0.8845996856689453, -0.22982466220855713, -0.5711629986763, -0.9509753584861755, 0.7314020991325378, 0.906326413154602, 0.8198972940444946, -1.53125, 0.2973887026309967, 0.444720059633255, 0.6513028144836426, -0.282105952501297, 1.224040150642395, 0.7099939584732056, 0.1729699671268463, -0.8147888779640198, -2.8668148517608643, 1.2151966094970703, -0.05977020785212517, 0.424811452627182, 0.4891236424446106, 0.021624485030770302, 0.27397263050079346, -0.6536562442779541, 1.378318190574646, 1.9055218696594238, 0.8436110019683838, -0.7134314775466919, -2.022616147994995, 1.1829750537872314, -0.36619624495506287, -1.0917067527770996, 0.15883281826972961, 1.0099719762802124, 0.4659252464771271, 0.5253422856330872, 1.9445207118988037, 0.11558312922716141, -0.06746084243059158, 0.45149052143096924, -0.4495212435722351, 0.5275041460990906, 2.1252059936523438, 0.22619199752807617, 0.4121904969215393, -0.8949384689331055, -1.2836666107177734, 0.789975106716156, -1.8110146522521973, 0.9150922894477844, 0.289620578289032, 1.079598069190979, -2.008939027786255, 0.8305591344833374, 1.349385142326355, -0.3183170258998871, -0.3892036974430084, -0.6451265215873718, 0.906947135925293, 2.4831385612487793, -0.06913984566926956, 0.7993813157081604, -0.24610166251659393, -1.075441598892212, -0.052560869604349136, -0.6714430451393127, -0.5281216502189636, -0.880347490310669, 0.48960596323013306], [0.8048507571220398, -0.2540739178657532, 2.668933153152466, -0.8646929264068604, -1.806589126586914, -0.3488054871559143, -0.30695468187332153, -1.1679801940917969, 1.7690303325653076, 0.401259183883667, 0.9117491245269775, -1.688319444656372, 1.6824196577072144, -1.5604419708251953, 0.5475510358810425, -0.9336534142494202, 0.4732241928577423, 0.1480012834072113, 1.3016164302825928, -0.19537034630775452, -0.701996386051178, -0.2347279042005539, -1.0743803977966309, 0.9792268872261047, -0.053422827273607254, 0.6611739993095398, 1.2378363609313965, -1.2887190580368042, 0.7157343626022339, -0.6714146137237549, -0.3798992335796356, 0.054530974477529526, -1.0680127143859863, -1.541237711906433, -0.1570086032152176, -0.06945505738258362, 0.9996194243431091, -0.44716039299964905, -1.0277764797210693, -0.02368008904159069, -0.9985285401344299, 1.2555253505706787, -0.14896070957183838, 0.5253012776374817, -0.5569172501564026, 0.22710584104061127, 0.30681973695755005, -0.002471586922183633, 0.7101332545280457, 0.1171276867389679, 0.045329801738262177, 0.38618314266204834, -1.177029013633728, 0.9953610301017761, -0.5528466105461121, 0.3345785439014435, 0.7721243500709534, 1.3806145191192627, 0.29179736971855164, 0.9458077549934387, -0.30587664246559143, 0.24020089209079742, 0.298042356967926, 1.3135591745376587, 1.018600583076477, 1.4192895889282227, 0.21063409745693207, -2.4832777976989746, -2.114861011505127, -1.044941782951355], [0.1400420218706131, -1.818866491317749, 0.3683120906352997, 1.5002107620239258, -0.8989444375038147, -0.7540261149406433, 0.008055571466684341, -0.35378366708755493, 0.2774817645549774, 1.2596131563186646, -0.9872041344642639, 1.2292453050613403, 1.3287187814712524, -0.029694069176912308, 1.4052722454071045, -1.4843558073043823, 0.36941006779670715, -1.3627640008926392, -0.21483980119228363, -0.2190539538860321, 0.46310240030288696, 0.902694046497345, -0.32371222972869873, 0.4291200637817383, -0.026277001947164536, 0.567237377166748, 0.10287483036518097, 0.016469331458210945, 2.3889048099517822, -0.5127806067466736, 0.14166180789470673, -1.3978614807128906, -0.15723596513271332, -1.823734998703003, 2.34643816947937, 1.2178514003753662, -1.7296991348266602, 0.7827566862106323, 0.8953877091407776, -0.8813378810882568, 0.8296411037445068, -0.5303566455841064, -0.8559873700141907, -0.6296629905700684, 1.7904301881790161, 0.3852081894874573, 1.305118441581726, 0.7974644303321838, -0.39951804280281067, -1.4220772981643677, -0.6272244453430176, -0.17209912836551666, -0.43903836607933044, -0.865237295627594, -0.754679799079895, -0.8277823328971863, -0.07407204061746597, 1.2874184846878052, 0.13874107599258423, 0.09541744738817215, -0.36458879709243774, 0.4659332036972046, 0.6347085237503052, -0.23468276858329773, 0.06841301172971725, 1.1426023244857788, 0.273902952671051, 0.0450923815369606, 0.4060218036174774, 0.6639407277107239], [0.020756753161549568, -0.5453267693519592, -0.1175660565495491, 1.0641175508499146, 0.270844429731369, -1.7971216440200806, 0.8388926982879639, -0.28660115599632263, -0.5757779479026794, 1.0187835693359375, 0.9028865694999695, -0.32491981983184814, -1.4345810413360596, -1.8419729471206665, -0.5413538217544556, -0.5939244627952576, -1.5266536474227905, 1.3645800352096558, -0.5905935764312744, 0.9320970773696899, -1.7222392559051514, -0.3352765738964081, 0.24352659285068512, -0.27815157175064087, -0.38987311720848083, 3.0295891761779785, 1.449951171875, 1.9486624002456665, -0.6610803008079529, 0.8646615147590637, -0.5544203519821167, 0.9360107779502869, -2.221456527709961, -0.4501345455646515, -0.6854619979858398, -1.7468839883804321, -0.8816969394683838, 0.6799121499061584, 0.7309144139289856, 1.0366626977920532, -0.47670137882232666, -0.17591235041618347, 1.4515008926391602, -1.6099399328231812, 0.45358264446258545, 0.088653065264225, 1.948545217514038, -0.054731860756874084, -0.7863906025886536, 0.16897667944431305, -0.8513244986534119, 0.6870133876800537, 0.6839594841003418, -0.9461444616317749, -1.0562703609466553, 0.3966616988182068, -0.1558489054441452, 0.8670638799667358, 0.6775516867637634, 1.094224452972412, 0.4986821413040161, 0.24874001741409302, -0.4587031304836273, 0.41744837164878845, 0.531101405620575, -2.006584882736206, -0.35371965169906616, 1.0582783222198486, 0.9876677989959717, 1.887813925743103], [-1.6990636587142944, 0.4094935357570648, 0.017067287117242813, 0.718380868434906, 0.03011786751449108, 0.4652250409126282, -0.39870965480804443, 0.13172587752342224, 1.2104445695877075, -1.9845688343048096, -0.062316592782735825, -1.7657806873321533, 0.8565233945846558, -1.5329124927520752, 0.9165382385253906, 1.1753443479537964, -0.7111254334449768, -1.1877548694610596, 0.3626474440097809, -0.1598074734210968, 0.9655517339706421, 0.0028370972722768784, -0.10157725214958191, -0.2258012741804123, -0.6740546226501465, -0.14458265900611877, -1.6982629299163818, -0.14161108434200287, 2.44404935836792, -1.228301763534546, -0.8516052961349487, -0.7993439435958862, 0.887934684753418, 1.4494805335998535, 1.070798397064209, -2.058955669403076, 0.6267463564872742, 0.9828438758850098, -0.07016847282648087, -0.3996613919734955, -0.2958819568157196, 0.8109724521636963, 0.47927039861679077, 0.10003058612346649, -1.7651749849319458, 0.5525930523872375, 0.2202625870704651, -0.9781579375267029, -0.021375050768256187, 0.45349574089050293, -2.331696033477783, 0.38427257537841797, 0.3301506042480469, 1.2269679307937622, 0.42238348722457886, 0.5742087364196777, -0.12139823287725449, -0.04541191831231117, -0.9237596392631531, -0.9846172332763672, -0.14359869062900543, -0.6484354138374329, -0.3834053874015808, 0.23376907408237457, -0.4310104250907898, 0.803276002407074, 0.13635092973709106, 1.0502601861953735, -2.337235689163208, 0.6480607986450195], [1.766179084777832, -0.8772888779640198, -1.4276155233383179, 0.362989217042923, -0.6120008230209351, 0.5841700434684753, -2.27023983001709, 0.22120264172554016, -0.1400967836380005, 0.9023288488388062, -0.6452611088752747, -1.1834102869033813, 0.35213032364845276, -0.25983983278274536, -0.5353584885597229, 0.08827506005764008, 1.1107596158981323, 1.5128626823425293, 0.9363526701927185, -0.34232306480407715, 1.1021125316619873, 1.0650349855422974, -0.9573784470558167, -1.0168894529342651, 0.5109562873840332, -0.8345186114311218, 0.7424753904342651, -2.417048454284668, -1.3328391313552856, -0.7149707674980164, 0.832019567489624, -0.1639900952577591, 0.7766457200050354, 0.7446886301040649, 0.5734858512878418, -1.1859502792358398, 0.6217820644378662, 0.31649038195610046, 1.0344045162200928, -0.6880480051040649, -0.5486257076263428, -2.5515286922454834, 1.717107892036438, -0.9544403553009033, 0.7081795334815979, 1.0805094242095947, -0.5469435453414917, 0.05753772705793381, -0.12631157040596008, -0.1325514316558838, 0.372061550617218, -0.25061875581741333, -0.49222275614738464, 1.090345859527588, -1.248462438583374, 0.07062731683254242, -0.7251725196838379, -0.47729140520095825, 0.7756333351135254, 0.5348201394081116, -0.11603987962007523, -1.9648833274841309, -0.05389576032757759, -1.2367141246795654, 0.07997669279575348, -0.5738682746887207, -1.8259661197662354, -0.2663849890232086, 0.3500731289386749, 2.546190023422241], [-1.927659273147583, 1.5187828540802002, -0.6002634763717651, 0.612022340297699, 0.3935507535934448, -0.7570977210998535, 1.3013241291046143, -0.1804710477590561, -0.8195360898971558, -0.3625156581401825, -1.3696460723876953, 0.5730927586555481, -0.1792006641626358, -0.8721498847007751, 0.749072253704071, -0.6081551909446716, 1.4213107824325562, 0.31778573989868164, -1.1705899238586426, -0.670435905456543, -0.35665521025657654, 0.28560465574264526, -1.9553569555282593, 0.6799071431159973, 0.24840275943279266, -0.21884021162986755, -2.4115564823150635, -1.000793218612671, 1.296696424484253, -0.8551842570304871, 0.0902177020907402, -0.6061618328094482, 1.2034251689910889, 0.4780307114124298, -0.9263650178909302, -0.9951076507568359, 0.039827197790145874, 0.10201428085565567, -1.1705735921859741, -0.3913050591945648, -0.020144717767834663, -0.010176488198339939, 1.3683828115463257, -0.23891885578632355, -0.5455825924873352, 2.302652597427368, -1.497185468673706, 0.21644525229930878, 0.6183366775512695, -1.8973382711410522, -0.4736148416996002, -0.645180344581604, 1.0373066663742065, -0.42715397477149963, -0.20932713150978088, -0.8180968165397644, -1.1882227659225464, 0.5505572557449341, -0.8866912722587585, -0.518264889717102, -0.5343424081802368, -0.6004974246025085, -0.7256961464881897, -0.4233247637748718, 1.2914292812347412, 1.274060606956482, -0.11952126771211624, 1.1617661714553833, 0.15783433616161346, 0.3229760229587555], [-0.8872747421264648, 0.14234161376953125, -0.656665563583374, -0.10508794337511063, 0.5076656341552734, 0.1379452794790268, 0.13727763295173645, -0.37159645557403564, -1.2862989902496338, 0.4263131618499756, 0.9655224680900574, 1.001399278640747, -0.29918402433395386, 0.11305677890777588, -0.9879078269004822, -1.0481469631195068, 0.8757948875427246, 1.1149826049804688, -0.30456215143203735, -1.6157233715057373, 0.9904114007949829, 0.3156073987483978, 0.9376819133758545, -0.12395328283309937, -0.29679933190345764, -0.16809585690498352, -0.25903868675231934, 0.5420526266098022, 0.3135264813899994, 0.6696634292602539, -0.5055762529373169, -1.0375467538833618, 0.8990753293037415, -1.9437739849090576, -0.3523358404636383, 0.6408737301826477, -0.15813606977462769, -0.6700598001480103, -0.15972980856895447, 0.4165484309196472, -1.6618343591690063, 0.037381403148174286, 0.4263726770877838, 0.4460028111934662, -0.31510797142982483, -1.3178409337997437, -1.1437902450561523, 0.1273200511932373, 1.4714162349700928, -0.07158553600311279, 1.0279709100723267, -0.6806556582450867, -0.3740999698638916, -1.028122067451477, 0.7716821432113647, -0.24206148087978363, 0.8936293125152588, -1.1117541790008545, 2.037771463394165, -0.4589604139328003, 0.5378116965293884, -0.37047502398490906, 0.8067787885665894, 0.2467285692691803, -0.5970600843429565, -1.6802501678466797, 1.6057454347610474, -1.0001565217971802, 1.8949716091156006, 0.7844845652580261], [1.412642002105713, 0.4323138892650604, -1.0136127471923828, -0.46532005071640015, -0.2927437722682953, -0.4634919762611389, -0.47297099232673645, -0.514862596988678, 2.590759754180908, 1.1593314409255981, -0.043055977672338486, -1.0220365524291992, -0.2682186961174011, -0.10024308413267136, 1.228835940361023, -0.48505786061286926, -0.43804696202278137, 1.194122552871704, 0.047537654638290405, 1.1940336227416992, 0.1588478684425354, 0.16556264460086823, 0.40699654817581177, 1.115973711013794, -0.08862757682800293, 1.1825873851776123, -0.5574589371681213, 0.11078144609928131, -0.8948537707328796, 1.223521113395691, -0.8932482600212097, -0.061093639582395554, 0.47873881459236145, 0.5905675292015076, 0.30361559987068176, 0.8718217611312866, 1.4198664426803589, -1.1326278448104858, 0.5897471904754639, 0.3185807466506958, -0.40714454650878906, 0.12899081408977509, -1.4885947704315186, -1.5614500045776367, 0.5671162009239197, -0.1604197472333908, 0.07687254995107651, -1.3791725635528564, 1.1944228410720825, 0.4502173960208893, -0.7788052558898926, -0.9289998412132263, -1.490780234336853, -0.07675470411777496, -0.22150428593158722, -1.3869972229003906, -0.5936502814292908, 1.3132662773132324, -1.7967853546142578, 0.0489063523709774, -0.3748683035373688, 0.5495164394378662, -1.967995047569275, -0.510731041431427, 0.6518135070800781, 1.585691213607788, -0.3587576448917389, -1.1587001085281372, 0.07372362911701202, -1.1241885423660278], [1.148049235343933, 0.4892863929271698, -0.6769551634788513, 0.6208882331848145, -2.1853277683258057, -0.8482995629310608, 0.5573829412460327, -0.43218183517456055, -0.14325731992721558, 1.8135517835617065, -0.06172670051455498, 0.5246851444244385, -1.2995096445083618, 0.0815972313284874, 1.489070177078247, -0.21428430080413818, 0.08621513843536377, -0.22828857600688934, -0.6860117316246033, 0.06098994240164757, 1.473714828491211, -0.4352668821811676, 0.6278106570243835, 0.481219619512558, 1.519814372062683, -0.6708230972290039, -1.9269952774047852, 1.61884343624115, 0.7010557651519775, 0.04829752817749977, 0.05108214542269707, 0.5984488129615784, 0.16497766971588135, 1.3746235370635986, 0.007998635061085224, -0.6105923056602478, 0.31270134449005127, 0.665889322757721, -1.9245364665985107, -0.6723674535751343, -0.7415323853492737, 0.09046594053506851, -0.2292492836713791, 1.8544195890426636, -0.11724445223808289, -0.9580698013305664, -1.0154954195022583, 0.19036991894245148, 0.5243765711784363, 0.729207456111908, 0.7622227072715759, 0.06713984906673431, -0.35908859968185425, 0.7511100172996521, -0.24990203976631165, 1.1447452306747437, 0.8529034852981567, -1.3325574398040771, -1.01656174659729, 0.2220243662595749, 0.7906979322433472, 0.08817827701568604, -0.3483777642250061, 0.5685049891471863, -0.2172098010778427, -0.6942442655563354, 0.462864488363266, 0.40380966663360596, -0.36701247096061707, 1.3138498067855835], [-1.3732874393463135, -0.06982727348804474, 0.16169334948062897, -0.8395776152610779, -0.5026694536209106, 0.5304539203643799, 0.1091453954577446, 0.11741924285888672, -0.18532994389533997, -1.053495168685913, 0.47395408153533936, 0.3820961117744446, -1.1498544216156006, -1.0007776021957397, 0.6031934022903442, -1.4082657098770142, 1.4900000095367432, 0.8936585783958435, -1.6169490814208984, 0.3766958713531494, 0.5150327086448669, -1.404788851737976, 0.901321291923523, -1.076248288154602, -0.9109501838684082, 1.6338865756988525, -0.1302366703748703, 0.5903844833374023, -0.5886701345443726, 0.06442000716924667, 0.6368133425712585, -1.0481207370758057, 1.390089511871338, 2.119530439376831, -0.25097164511680603, 0.2903115153312683, 1.3091225624084473, 1.0225822925567627, -0.463665246963501, 0.739169716835022, -1.7256965637207031, 1.2079979181289673, -0.7974160313606262, 0.17081032693386078, 1.8971906900405884, 0.7607680559158325, 1.6609718799591064, -0.7566083073616028, -0.9039631485939026, -0.34518900513648987, 0.08278297632932663, 2.353580951690674, -0.2688300609588623, 0.24491246044635773, 1.2430437803268433, -1.39768385887146, -0.4724476635456085, -2.0183444023132324, -0.6606695652008057, -0.27727171778678894, 1.7210626602172852, -0.23991791903972626, -0.5152246356010437, -2.164743661880493, -0.17797894775867462, 0.7191734910011292, 0.5280633568763733, -0.7114253640174866, 2.340379238128662, -2.022453546524048], [0.14539362490177155, 0.7949970960617065, -0.742160975933075, -0.4706689119338989, -0.6592567563056946, -0.4459013342857361, 0.4416622519493103, 1.4082090854644775, 1.5629754066467285, 0.8319268226623535, 0.0033601776231080294, 0.22644871473312378, 2.4811625480651855, -0.06824847310781479, -0.21888074278831482, -0.014142360538244247, 0.42731934785842896, -0.057969268411397934, 0.7268288135528564, 0.13029685616493225, -0.8736979961395264, -1.3425068855285645, 0.3570639193058014, -0.549075722694397, 1.64255690574646, -0.15185004472732544, -0.16312190890312195, 0.6708922386169434, 0.4221731424331665, -1.0912789106369019, -1.1975713968276978, -0.6786945462226868, -0.5240144729614258, -0.447345495223999, -0.3641330599784851, -1.4921152591705322, -1.1114150285720825, -0.1555139273405075, -0.9399256110191345, -0.15901796519756317, 0.9799933433532715, 0.14556029438972473, -0.9241716861724854, 2.4888370037078857, -0.6213151216506958, -0.43265923857688904, -0.08021917194128036, -0.9179575443267822, 0.3735296130180359, 1.4132174253463745, -0.5631515383720398, -0.18477022647857666, 0.0804358720779419, -2.5907065868377686, -0.7751012444496155, 0.29575231671333313, -0.3593384921550751, -1.3627686500549316, -0.37435948848724365, -1.254188895225525, 1.2139477729797363, -0.6992148160934448, 0.46659234166145325, -0.4735007882118225, 0.07513206452131271, 0.8140284419059753, -0.2939894497394562, 0.5789493322372437, -0.782784104347229, 0.15249833464622498], [1.008744239807129, 0.24326074123382568, 0.08932718634605408, -1.7020175457000732, 0.6781213283538818, 3.0529391765594482, 0.7209813594818115, 0.29828816652297974, 1.0518394708633423, 0.9759426116943359, 0.8131321668624878, 0.3106447458267212, -1.31430983543396, 0.440621942281723, 0.2591869831085205, 0.9461684823036194, 0.12112139910459518, -0.7911463975906372, 0.12048212438821793, -1.0385264158248901, -0.6189882755279541, 0.9750323295593262, -2.781665086746216, -0.046280380338430405, -0.3789069950580597, 0.46711266040802, 2.04742169380188, 0.5702471733093262, 0.6749873161315918, 0.7508204579353333, -1.590298056602478, 0.916262149810791, 0.20061838626861572, -1.021690011024475, 2.253232002258301, 0.08104957640171051, -1.4098381996154785, -0.40205496549606323, -0.0199320949614048, 0.9787507057189941, -0.28520455956459045, 0.6873759627342224, -0.7908949255943298, 0.09838490933179855, -0.37040871381759644, -1.989209771156311, 0.8572334051132202, -0.9443160891532898, -1.2661759853363037, 0.40241682529449463, 0.3884083330631256, 0.3400447964668274, 0.17615695297718048, 0.9994267225265503, -1.7178070545196533, 0.2403467744588852, 0.6667269468307495, -2.547429323196411, -0.15608611702919006, -1.259332299232483, -0.950622022151947, -0.3345218598842621, 0.4991978108882904, 1.3454006910324097, -1.943068504333496, -2.3446216583251953, 1.7601008415222168, -0.6237662434577942, -0.1776050180196762, 1.002681851387024], [0.6608684659004211, -1.1171839237213135, -0.03674117475748062, 0.008874761871993542, 0.18457280099391937, 0.7849209904670715, -0.952974796295166, 0.47047534584999084, -1.8631961345672607, -1.2497787475585938, 0.3656747341156006, 1.417778730392456, 0.7628600001335144, -1.1006356477737427, 0.7569746971130371, -0.024664830416440964, 1.333667278289795, -0.25715190172195435, 0.9048314094543457, -3.053837776184082, -0.3094763159751892, -0.21573002636432648, -0.9201335906982422, -1.7889734506607056, 0.2589896023273468, -0.3370118737220764, 1.138496994972229, -1.493532419204712, 0.5944072008132935, -1.8466285467147827, -1.481894612312317, 1.3413804769515991, -0.8827516436576843, -0.7754274010658264, -0.7571775913238525, 2.477277994155884, -0.3827666640281677, -0.016209140419960022, 1.5988038778305054, -2.328502655029297, -0.9723830223083496, -0.18704889714717865, 1.4852066040039062, 1.2812944650650024, 0.8495708107948303, 0.06170455738902092, -1.9031866788864136, 0.10456612706184387, -0.5868911147117615, -0.48610976338386536, 0.7690855264663696, 2.7425708770751953, -0.20601637661457062, -1.4799292087554932, 0.5394333600997925, 0.4109336733818054, -0.2712780833244324, -1.104327917098999, -0.6315057873725891, -0.32945671677589417, 1.5494564771652222, 0.05926307663321495, -0.26013267040252686, -0.9865021109580994, 0.8990790843963623, -0.10379333794116974, -1.070400595664978, 1.1281261444091797, -0.04878148436546326, 0.3311368525028229], [0.5834237337112427, 0.7528724074363708, -0.10024280846118927, 0.5507588386535645, 1.3069144487380981, 1.4512429237365723, -1.3138161897659302, -0.0017303070053458214, 1.1808068752288818, 2.5168392658233643, 0.23614761233329773, -0.10642464458942413, -1.6777005195617676, 0.39184245467185974, 0.8192826509475708, 1.3481853008270264, 0.5424103140830994, 1.7539284229278564, 0.7027630805969238, 1.4278655052185059, 1.5702075958251953, -0.02796429954469204, 0.5854743719100952, -2.952899217605591, -0.035598274320364, -0.8199120163917542, -0.8824045062065125, -0.534594714641571, 0.9414483308792114, 0.20575769245624542, -1.9576753377914429, -0.31591320037841797, -0.5915126800537109, 0.5867993831634521, 0.116376593708992, 0.3156636655330658, -0.7196157574653625, -1.8124966621398926, -1.0043433904647827, -0.29195278882980347, 0.24887949228286743, 1.7618212699890137, -1.4434841871261597, -1.2655634880065918, 2.280231237411499, -0.7540982365608215, 0.8870624303817749, -0.8739179968833923, -0.49144670367240906, -0.6268806457519531, -0.07532363384962082, 0.06175791844725609, -0.21086090803146362, -0.24476441740989685, -1.0779145956039429, 0.39470377564430237, 2.178539276123047, -0.30330440402030945, -1.0367823839187622, 1.1432102918624878, 0.47244665026664734, -1.5849716663360596, -0.3478725850582123, 0.7436974048614502, -0.8641024231910706, -0.4792928695678711, 0.2882968485355377, -0.411892294883728, -0.3053654730319977, 1.3048309087753296], [-0.9156036376953125, -0.7241633534431458, -0.8722369074821472, 0.665138304233551, -0.5029081106185913, 0.9397671222686768, 0.4916267693042755, 0.027359208092093468, 0.6745944619178772, -1.9034901857376099, -0.8984330296516418, 1.6163344383239746, 1.5505902767181396, -1.9253681898117065, 0.9552143216133118, -1.2087782621383667, -0.050404418259859085, -1.053632378578186, 0.15285466611385345, -0.2128714621067047, 1.0841742753982544, 0.4778183102607727, -0.7120028734207153, 0.3234029710292816, -0.6298501491546631, 0.041704121977090836, 0.9925374984741211, 0.4209223687648773, 2.029745101928711, 0.4334787428379059, 0.24368952214717865, 1.5624239444732666, 0.5915908813476562, 0.07718739658594131, -0.006733645219355822, -2.048368215560913, -0.653236448764801, 1.449020504951477, 1.248685359954834, 0.836581289768219, -1.3575518131256104, 2.682206869125366, -2.9245784282684326, 2.174513816833496, -0.5010995268821716, 1.9281198978424072, -0.7840511798858643, 0.03719040006399155, 1.0807650089263916, 0.7061882019042969, 0.4495294392108917, -0.6595670580863953, -0.581513524055481, -0.2958185076713562, 0.43396440148353577, 0.6123432517051697, 0.3513343334197998, 1.0317115783691406, -1.2930569648742676, -1.0425148010253906, -0.3480595350265503, -0.17630381882190704, -1.2962596416473389, -0.6028597354888916, -1.209187626838684, -1.2521946430206299, 0.7227441072463989, 0.030737973749637604, -0.010500299744307995, 0.8198574781417847], [0.5977253317832947, 0.21628877520561218, -3.302590847015381, -0.4062308073043823, 1.0918447971343994, -0.3749905228614807, -0.24646799266338348, 1.0869247913360596, 0.5057415962219238, 1.2654331922531128, -0.9215114116668701, 1.0537184476852417, 0.5489962100982666, -1.5772054195404053, 0.7154155373573303, 0.387008398771286, -1.3552672863006592, -1.0432926416397095, -1.335417628288269, 2.84037446975708, -1.003533124923706, 1.0113643407821655, -0.11857771128416061, -0.9241695404052734, -0.4103175103664398, -0.4925841987133026, -0.6305696964263916, -0.03356090188026428, -1.3405097723007202, -0.8206245303153992, -0.6593015789985657, 1.2485065460205078, -1.406893253326416, 1.2764472961425781, 0.6620164513587952, -0.09889595210552216, -1.535399079322815, -0.9953275918960571, 0.1897565871477127, 0.3234192728996277, -0.7570043802261353, -0.7624805569648743, -2.536628246307373, -0.6860742568969727, 0.9156803488731384, -0.851216197013855, 1.1070557832717896, 0.6743570566177368, 1.1216784715652466, 1.1929881572723389, -0.38197022676467896, -2.066523313522339, 0.10532694309949875, -0.5933589935302734, 0.536988377571106, -1.5994946956634521, 2.0330650806427, -2.340437650680542, 0.7266569137573242, 0.31128406524658203, 0.10097257047891617, -1.0775783061981201, -1.21083664894104, -1.1625382900238037, 0.49917006492614746, 0.6281677484512329, -1.4696288108825684, 0.0663641095161438, 0.3000698685646057, -0.6457305550575256], [-1.2971290349960327, -1.113785982131958, -0.3121607005596161, -2.3655760288238525, -0.3244122266769409, -1.8852283954620361, 1.075279951095581, 0.7384932041168213, 0.3719545006752014, -0.8150249123573303, 0.2892911434173584, 0.5455862879753113, 2.477499485015869, 0.3466283977031708, 1.7160592079162598, 0.1841677576303482, -0.15821950137615204, 0.03131064772605896, 0.3206709325313568, -0.6725526452064514, -0.4621709883213043, 0.8448137044906616, 1.1915370225906372, -0.24676208198070526, -0.8249143958091736, -0.6934171319007874, 0.049353957176208496, -1.5838062763214111, -2.0822696685791016, 0.9425659775733948, -0.2920393645763397, -0.7504709959030151, 0.2726196050643921, -1.4902037382125854, 0.6492486000061035, -0.6463470458984375, -1.5689830780029297, 0.5992093086242676, 0.191869854927063, 0.3386460840702057, -0.3640209436416626, -0.18661390244960785, -0.6580480337142944, 0.9681567549705505, -0.42957696318626404, -0.32862788438796997, 0.7989210486412048, 0.936000645160675, -0.7907072901725769, 0.2011018842458725, -0.4359177350997925, 1.8569117784500122, -0.4886379837989807, -0.655193030834198, -1.3536269664764404, -0.42807596921920776, 1.1505770683288574, -0.23662018775939941, 0.49874716997146606, -1.7765554189682007, 0.03484748676419258, 0.1747111827135086, 3.0464398860931396, 0.49858126044273376, 1.411745309829712, -0.3700374960899353, -0.4112017750740051, -0.07581751048564911, -0.24282774329185486, 0.9098692536354065], [-0.416450560092926, 0.6571426391601562, 0.27714747190475464, -0.8811939358711243, 0.4722136855125427, -0.6594135165214539, 0.3176756203174591, -1.3817635774612427, -0.6866694092750549, -1.7391893863677979, 0.22854068875312805, 2.7953312397003174, -0.5794811248779297, -1.5331469774246216, 0.7799816131591797, -0.1963047832250595, 0.6550517082214355, -1.6806368827819824, 0.8140290379524231, 1.0440560579299927, -1.1638041734695435, 0.3626295030117035, -1.708100438117981, 1.6162885427474976, -1.3901355266571045, -1.3834396600723267, -0.2338220328092575, 0.5515264272689819, -0.8435161113739014, 0.9503119587898254, 1.2472447156906128, -1.9738825559616089, 1.3133571147918701, -0.6663789749145508, 1.4831870794296265, 1.3101494312286377, -0.8517029881477356, -1.4649641513824463, 0.48697131872177124, 0.7000783681869507, -0.7984591126441956, -1.1166908740997314, 0.6777129173278809, 0.25167974829673767, -0.18758492171764374, -0.30448946356773376, -0.40153375267982483, -0.724795937538147, -0.7051705718040466, -0.39086443185806274, 1.2240785360336304, 0.8297163248062134, 1.3912091255187988, 1.0993114709854126, 0.0869361013174057, -0.508853018283844, 1.857661485671997, 1.9205604791641235, 1.254485845565796, 1.044826865196228, 1.4975358247756958, -0.9874070286750793, 0.7807462811470032, -0.14889171719551086, -1.1911215782165527, -0.039925962686538696, -0.5876899361610413, -1.8952369689941406, -0.6954630613327026, 2.14860200881958], [-0.3101080656051636, 1.8861838579177856, -0.266057550907135, -0.4709007441997528, -1.249831199645996, -0.9660161733627319, -0.1257476806640625, -0.3725881576538086, -1.2483640909194946, 0.9787702560424805, 1.2475130558013916, -0.7607910633087158, -0.023516492918133736, 1.590779185295105, 0.8173912167549133, -0.8347796201705933, -0.5850117206573486, -0.7338497638702393, 0.6907241344451904, 1.3019349575042725, -1.4246153831481934, 2.356954574584961, -1.139370322227478, -0.610906720161438, -1.8887619972229004, -0.9042600393295288, -0.39338332414627075, 0.7043688893318176, -1.172554850578308, -0.3520359992980957, -1.6395999193191528, 0.2773906886577606, -0.6711809635162354, -0.19685758650302887, -0.07295786589384079, 1.37315833568573, -1.0960352420806885, 0.42733749747276306, -2.448354482650757, 0.43717318773269653, 0.6724143028259277, -0.3889244496822357, -0.6685081124305725, -1.8544973134994507, 0.6566479206085205, -0.8047401309013367, 1.4186466932296753, -0.20454105734825134, -0.8050122261047363, 0.4355953633785248, -1.646392583847046, -1.2891347408294678, -1.2958564758300781, -0.9728417992591858, 0.34288960695266724, -0.21623340249061584, 1.4867016077041626, 0.9011992812156677, -0.40606728196144104, 1.0594135522842407, 2.342710494995117, -2.2434494495391846, 0.11449334025382996, -2.5908172130584717, 0.6959967613220215, -0.6689755320549011, -0.4844905734062195, -0.18217170238494873, 2.06886625289917, -0.6296361088752747], [-1.0432188510894775, 1.6568307876586914, -0.11952224373817444, 0.3150942921638489, 0.8264550566673279, -2.316338539123535, -2.4632554054260254, 1.5155481100082397, -1.3913826942443848, 1.7403078079223633, 0.568118155002594, -0.36938953399658203, -0.020594678819179535, -0.5950168371200562, 0.3664458394050598, 0.5609181523323059, 1.5269287824630737, 0.09869138151407242, -0.9532755017280579, -1.2624998092651367, -1.4092543125152588, 0.6358963251113892, -0.009628115221858025, -2.259800672531128, -0.4425058364868164, -0.15592330694198608, -0.4542592167854309, 1.7117522954940796, -0.6385511755943298, 0.8908399343490601, 0.04569365084171295, 0.2612133324146271, 0.35141128301620483, -0.5418741106987, 0.8867648839950562, -0.42968088388442993, -0.8162844777107239, -2.3566160202026367, 0.6816470623016357, -1.4126321077346802, -0.467521071434021, 0.7106391191482544, -0.8878952860832214, -0.4563385546207428, -0.32748934626579285, -0.11891370266675949, 1.7381932735443115, 0.6970881819725037, -1.8798907995224, -2.086355686187744, 0.5363555550575256, -1.4165856838226318, -0.30250832438468933, -0.24937085807323456, -0.1950993835926056, 0.7064480781555176, 0.7615858316421509, -0.17994914948940277, -1.042110800743103, -2.146193265914917, -0.9120081067085266, 1.3690474033355713, -0.5039676427841187, -1.0187644958496094, -0.06524748355150223, 0.4731426239013672, 0.10777163505554199, -0.059898994863033295, -2.3322436809539795, -0.2914672791957855], [0.850554347038269, 1.6832239627838135, -1.651706337928772, -1.6917976140975952, -0.9312698841094971, 1.3867777585983276, 0.5408008098602295, -0.8464835286140442, -0.3767572045326233, -0.4798908829689026, -1.2530806064605713, -1.283464789390564, 0.7666533589363098, -0.023450246080756187, -0.5052387714385986, 0.7316541075706482, 0.2644244432449341, -1.1922601461410522, -0.0035511814057826996, -0.23216032981872559, 1.413641095161438, 1.7017405033111572, -0.03475435450673103, -1.7415759563446045, -1.5209804773330688, -0.3982805013656616, 0.5572363138198853, 0.23948067426681519, 0.3987700939178467, -2.089789390563965, -0.6189022660255432, -0.7103429436683655, -0.09583663195371628, -0.9459060430526733, 0.5265063643455505, -1.0672096014022827, -1.0258785486221313, 0.44047826528549194, 0.708665668964386, -0.12801262736320496, -0.576271653175354, 0.5130496025085449, -1.2565640211105347, 1.0556395053863525, 0.5809636116027832, 1.026016354560852, -1.1825077533721924, 1.7813555002212524, -1.661355972290039, -0.3621346652507782, -1.161985158920288, 1.0018315315246582, 1.0767066478729248, -1.2590950727462769, 1.3194656372070312, 0.6105888485908508, 0.5400267243385315, -1.1562749147415161, 1.0412007570266724, 0.6204010248184204, -1.5224136114120483, 2.282700777053833, -0.3110467195510864, 0.2865832448005676, -0.8456307053565979, 1.1837917566299438, 0.9548560380935669, -1.404337763786316, -0.4184286594390869, -0.29738283157348633], [0.49856120347976685, 0.9077929854393005, -1.6711167097091675, 0.42445361614227295, 0.6137335300445557, -0.4653436541557312, -0.34243345260620117, 2.255784034729004, -1.6014817953109741, 0.7445325255393982, 0.8618454933166504, 0.8915815353393555, -0.5118446946144104, -0.24762040376663208, 0.32499977946281433, -1.3309675455093384, -0.9854186177253723, 0.11589314788579941, -0.16723321378231049, -0.8565110564231873, -0.027308648452162743, 0.34806644916534424, -2.072512626647949, 0.5221235156059265, -0.44729113578796387, 2.1752865314483643, 0.29216164350509644, -0.0707818940281868, 0.6207095980644226, 0.6419399380683899, -0.1363535076379776, -0.6094299554824829, 0.6553137898445129, 0.03161326050758362, 0.7629393339157104, -0.05202634632587433, 0.29245394468307495, -0.11007563024759293, -0.20861543715000153, -1.2047522068023682, -0.9280334115028381, 0.3193027973175049, 0.6423349380493164, -0.8871339559555054, 0.16575157642364502, 1.3817135095596313, -0.9672300219535828, -0.3963676989078522, 0.5860434770584106, 0.6908977627754211, 0.027652068063616753, -0.8721703290939331, -0.465523898601532, -0.014993005432188511, 1.8182032108306885, 0.6399084329605103, -0.10626819729804993, 0.8479230999946594, -1.7167437076568604, 1.496601939201355, 1.0299299955368042, -0.3732738792896271, -0.6091238260269165, -1.143938660621643, -0.9491655230522156, 0.08568543195724487, -2.4955406188964844, 2.172661542892456, 0.7548016905784607, 2.507655143737793], [-0.051654573529958725, 1.2483959197998047, 0.15410642325878143, -0.006835675332695246, -0.18984659016132355, -0.9846289157867432, 0.8834548592567444, 0.6145676970481873, 0.7935166358947754, 0.14777952432632446, 0.1422121822834015, -0.4137348234653473, 1.554351806640625, -0.470398485660553, -0.7931687831878662, 0.777372419834137, 0.04747876897454262, -0.3642924726009369, -0.012768752872943878, -0.34746912121772766, 0.7230272889137268, -1.1273106336593628, -0.22982719540596008, 0.14594337344169617, 0.9956153631210327, -2.0641422271728516, 0.06326496601104736, 0.5480599403381348, -1.3804774284362793, -0.07410101592540741, -0.20273487269878387, -1.5872433185577393, -0.4095759093761444, -0.007798075210303068, -0.15721061825752258, 0.34508374333381653, -0.2924726605415344, -1.7939435243606567, -0.8172586560249329, 0.10486447066068649, -0.3197212219238281, -0.0645233765244484, -0.6021197438240051, -0.9111853837966919, 0.1534099280834198, 1.0512384176254272, -0.8973721861839294, -0.12240029126405716, 0.1775951236486435, 0.558060884475708, -0.9268584251403809, 1.1280893087387085, -0.4597414433956146, -0.881271481513977, -0.5394056439399719, 0.5405983924865723, 0.7827800512313843, 2.0482895374298096, 1.0532183647155762, 0.2910907566547394, -0.7524689435958862, -0.7933026552200317, 0.7845556139945984, -0.5144237875938416, -0.21217083930969238, -0.6965071558952332, -1.737204909324646, -0.20798011124134064, -0.2678524851799011, -0.2233528345823288], [-0.6493821740150452, 1.0244187116622925, 0.48942437767982483, -2.0250282287597656, -1.2156668901443481, -2.6449530124664307, 1.090261459350586, 0.7145497798919678, -0.25281548500061035, 0.13593824207782745, -0.09771200269460678, -1.5365368127822876, 0.622536301612854, 0.8906843662261963, 0.16400957107543945, 0.5747228264808655, 0.470492959022522, 1.5732922554016113, -1.5005602836608887, -1.8874800205230713, 0.44675949215888977, 1.2368745803833008, -1.1654233932495117, 0.628187358379364, 0.8079712390899658, -1.7780356407165527, -0.5777923464775085, -0.7888555526733398, 0.1921653151512146, 0.07455876469612122, 1.4518678188323975, -0.6679198145866394, 0.8974360227584839, 0.8267962336540222, -0.450190931558609, 1.3295023441314697, 0.11216100305318832, -0.7914392948150635, 0.7285072803497314, -1.3895405530929565, 0.26555415987968445, -1.3007397651672363, 0.6370399594306946, -0.21166770160198212, -0.9561774134635925, -0.7405630946159363, 1.0065524578094482, -2.013115882873535, 1.6563663482666016, -0.7486078143119812, -0.394687682390213, 1.1200988292694092, 1.0312732458114624, -0.649863064289093, 0.022625451907515526, 0.52479487657547, 0.2388319969177246, -1.1279476881027222, -0.008176809176802635, -1.522718071937561, -0.9318139553070068, 0.4138861298561096, -1.30390465259552, -0.6772108674049377, 1.365472435951233, 0.5551645159721375, -0.5030056834220886, -0.10989242047071457, 0.5271936058998108, 1.2933107614517212], [-0.9437060356140137, -0.2530275583267212, 0.15916380286216736, 0.7850599884986877, -0.3280784785747528, 1.343253254890442, -0.002932343864813447, -0.5469697117805481, 1.502589464187622, 2.6438100337982178, -1.883339285850525, 1.4069559574127197, 0.06258431077003479, 0.7678619027137756, 0.8607608079910278, 0.07934503257274628, 2.625760316848755, -0.0002028285089181736, 0.2687516510486603, 0.5070169568061829, 2.4185619354248047, 0.605722963809967, 0.2389930933713913, -1.580820083618164, 0.042893823236227036, 1.029858946800232, 0.7749303579330444, 0.7590883374214172, -2.41180682182312, 0.48119640350341797, -1.1542593240737915, -0.7971442937850952, -0.1710178107023239, 0.2169729471206665, -0.16342297196388245, 0.4198220372200012, 0.42003363370895386, 0.23876003921031952, -1.2702624797821045, 0.5595719218254089, 1.6151783466339111, -0.0038308631628751755, -0.949158787727356, -0.11396069079637527, 0.40154558420181274, 1.7051196098327637, -1.5495632886886597, -0.6847859621047974, -0.5977287888526917, -0.09497714042663574, -0.9245327711105347, -0.6890000104904175, 0.6782225966453552, -0.9635002017021179, 0.03196791931986809, 0.718752920627594, 0.12732811272144318, 2.0510663986206055, -0.1798439919948578, 1.0810261964797974, 0.11888733506202698, 0.8177593350410461, 0.22961130738258362, 0.5652208924293518, -1.3473702669143677, -0.9019660353660583, 0.14717359840869904, 0.832571268081665, 0.26858454942703247, 1.4398281574249268], [-0.310320645570755, -0.67080157995224, 0.49793094396591187, 0.9684374332427979, -0.37565168738365173, -0.2916737496852875, -0.43726134300231934, 0.12273908406496048, -0.3067246675491333, 0.10200675576925278, -0.5680094361305237, 0.6946368217468262, 0.8047148585319519, 0.3375164270401001, -0.7518507242202759, -1.2310702800750732, -0.1683158576488495, 0.5245546698570251, -0.15897239744663239, -1.5327918529510498, -0.37014421820640564, -0.6507990956306458, 1.2489689588546753, 1.8865272998809814, -0.42080944776535034, 0.3300701677799225, 0.8106151223182678, -2.24666166305542, 1.1922085285186768, -0.33866041898727417, 0.4140881597995758, -1.7466731071472168, -0.8349800705909729, 0.13129249215126038, 0.3716821074485779, -0.9564524292945862, -0.8190258145332336, 0.0165045578032732, 0.402656227350235, 0.9171570539474487, -0.20455098152160645, 0.2950029969215393, -0.3495695888996124, -0.6613456010818481, 0.08606306463479996, -2.269517421722412, 0.034034084528684616, 0.42946964502334595, 0.8695943355560303, 0.17326079308986664, 1.1803218126296997, -0.9611225128173828, 0.7192819714546204, 1.0801435708999634, 0.10413884371519089, -0.8986505270004272, -1.1952152252197266, -1.5289645195007324, -1.4981244802474976, 1.4330695867538452, 0.3640185296535492, 0.3399791121482849, -1.1090902090072632, 0.2714288532733917, 0.5904386043548584, -1.279956579208374, 0.8038923740386963, 1.5412403345108032, -0.6769434809684753, 0.2563264071941376], [0.07509829103946686, 0.3980633020401001, -0.141377791762352, 0.908894956111908, -0.08307117968797684, 0.28007203340530396, -0.6185204386711121, -0.7844841480255127, -0.27498874068260193, -0.9811447262763977, 0.025332843884825706, -1.5020184516906738, 1.8166391849517822, 1.1572216749191284, 0.9086259603500366, -0.5887162685394287, -2.202432155609131, -1.2212363481521606, -0.565498411655426, -2.028876543045044, -0.74628084897995, -0.1656492054462433, 0.05437291041016579, 0.3359147608280182, -0.4366992712020874, 0.20288896560668945, 0.2696230709552765, 1.824399471282959, -0.5993373990058899, 0.3672563135623932, 1.1358845233917236, -0.6998850703239441, -0.3893590569496155, 2.6480257511138916, -0.18861690163612366, -0.27608177065849304, -1.5209243297576904, 1.1906721591949463, -1.261229395866394, -0.7907392978668213, -0.356232225894928, 0.1291000247001648, -1.0169609785079956, -0.3201335668563843, -1.0193085670471191, 0.2838490903377533, -1.2697395086288452, 0.20080167055130005, 0.7294086813926697, -1.7208342552185059, -0.4393704831600189, 1.3220685720443726, 1.227860689163208, -1.5575333833694458, 0.12268941849470139, -1.0060830116271973, 1.2891629934310913, 1.064253568649292, -1.0594596862792969, 1.7084366083145142, -0.04367375746369362, -0.8120640516281128, 0.49646642804145813, 1.087751030921936, -0.018345380201935768, 2.6242713928222656, -2.3809916973114014, 0.51077800989151, 0.8920814990997314, -1.8632019758224487]], "dtype": "float32"}, {"__nd__": [[-0.8571405410766602, -0.3683617413043976, -2.2493770122528076, -3.1101534366607666, 0.6432065963745117, -0.55499666929245, 1.327027678489685, -0.9015619158744812, 1.7670252323150635, 0.18563425540924072, 2.5107715129852295, -0.09308605641126633, -0.7228333950042725, 0.6379318237304688, 0.9043153524398804, 0.7227282524108887, 1.5516036748886108, -0.4032585322856903, 0.8884559869766235, 2.0132391452789307, -0.8808065056800842, 0.018685176968574524, 0.6257436275482178, 0.9458417892456055, 1.2775620222091675, 0.615402340888977, 1.0384026765823364, -0.935739278793335, -0.7807280421257019, 0.092917300760746, -1.0726348161697388, 0.5759370923042297, -0.5560475587844849, -0.11130183190107346, 0.37636834383010864, 2.2038025856018066, 0.03462838754057884, 0.5823489427566528, 0.38440704345703125, 0.5107390880584717, -0.33214524388313293, -0.3745829463005066, -0.8750027418136597, -0.9965816140174866, 0.1402319371700287, 0.17166250944137573, -0.8639963269233704, 1.3639168739318848, -0.35721513628959656, -0.56209796667099, -0.8811677098274231, 0.49180281162261963, 0.12351641803979874, 0.846642255783081, -0.6199420690536499, -1.0842225551605225, 0.4648198187351227, 0.25753992795944214, 0.5927041172981262, -1.6299115419387817, 2.020113945007324, 1.5916115045547485, -0.4612886607646942, -0.380691260099411, -0.016951631754636765, 0.2719156742095947, -0.07770437002182007, 0.15272952616214752, -0.17779362201690674, 0.8207201957702637], [1.3340920209884644, 0.5336781144142151, -0.3369405269622803, 0.1980707347393036, 1.7854129076004028, -0.22282716631889343, 0.9911348223686218, -0.16364112496376038, -1.082817554473877, 0.6155694127082825, 0.9878480434417725, 1.857823371887207, 1.6332967281341553, 2.7946078777313232, 1.9357801675796509, 0.45863866806030273, 0.010292491875588894, -0.27395087480545044, -0.6151056289672852, 1.604621410369873, -0.8931154608726501, -0.7363333106040955, 0.5374143123626709, 0.8405728936195374, -2.0206384658813477, 0.045643601566553116, -1.0404778718948364, 0.42332887649536133, -0.5622595548629761, 0.8609504103660583, -0.7100648880004883, 1.104145884513855, 1.0229500532150269, 0.2734234929084778, -1.559774398803711, -1.614787220954895, -0.9406268000602722, -0.5435051321983337, -1.2663543224334717, 2.0437424182891846, -0.08305910974740982, -1.1615396738052368, -0.20414388179779053, 0.22414155304431915, -0.1544206440448761, -1.3763835430145264, -0.6402629613876343, -1.1001373529434204, -0.44176334142684937, -1.013709545135498, 0.3333251476287842, -1.0331642627716064, 1.338681697845459, 0.041237153112888336, 2.3313229084014893, 1.6469916105270386, 0.521631121635437, 1.324154257774353, 1.6937536001205444, -1.4098234176635742, -0.2279970645904541, -0.5584660172462463, -0.27754664421081543, 0.25240838527679443, -1.2432398796081543, 0.039320483803749084, 0.4772918224334717, -0.6701040267944336, -1.5601637363433838, 0.36820217967033386], [-0.33243170380592346, -0.6728309988975525, -1.8278037309646606, -0.6129709482192993, -0.2028767168521881, 0.28270912170410156, 0.5720961093902588, -0.9763504862785339, -0.3935870826244354, -0.7465530633926392, 0.10260356962680817, -0.6270830631256104, -0.11611463874578476, -0.9347810745239258, 0.46969014406204224, 2.1221134662628174, -0.656118631362915, -2.2192254066467285, -0.5082473754882812, 0.01962776854634285, -0.9168676137924194, -0.3579869270324707, 0.5659375786781311, -0.5502930879592896, -0.6757728457450867, 1.5674941539764404, 0.10067518055438995, -1.791608452796936, -0.2374849021434784, -0.6201767325401306, 1.292475700378418, -1.0412763357162476, 0.625945508480072, 2.196176528930664, -0.2139790952205658, 0.3919484317302704, 0.5400525331497192, 0.9301244616508484, -0.2215913087129593, -0.10082294046878815, -1.8501198291778564, 0.5000888109207153, 0.36553382873535156, -2.2099874019622803, -0.05683102831244469, 0.44214195013046265, -1.5610491037368774, 0.3221435546875, 1.1150540113449097, 1.714201807975769, 0.27386054396629333, 0.427804172039032, 0.5082412362098694, 1.759882926940918, 2.9004764556884766, -0.7718384265899658, -1.6975560188293457, -1.9317036867141724, -0.3841209411621094, 1.3116581439971924, -0.6490358710289001, 0.20463989675045013, -0.6900838017463684, -0.34465914964675903, -0.21531988680362701, 0.2500964105129242, -0.7418954968452454, 0.8841222524642944, 0.006882956251502037, 1.4533586502075195], [-0.9861264824867249, -1.249516248703003, 0.13993798196315765, 0.6365470290184021, 0.8095239400863647, 1.152329683303833, 1.4340749979019165, -0.6508913636207581, -1.1864508390426636, -0.3654937148094177, -0.1507946252822876, 0.8608245253562927, -1.2450840473175049, -0.8927059769630432, -0.38202613592147827, -0.708280622959137, -0.7192956209182739, -1.415570616722107, 1.0434592962265015, 0.5471269488334656, 0.9923493266105652, -0.2086395025253296, 0.931908905506134, -1.3660972118377686, 0.7068082094192505, -0.3365704417228699, -0.05991841480135918, 0.40209871530532837, 1.8300024271011353, -2.1460351943969727, 0.1548311561346054, 1.7315527200698853, 0.5385605096817017, 1.2062458992004395, -0.7680298686027527, -0.27098211646080017, 1.6298540830612183, -0.6369825005531311, 1.6463168859481812, 0.0029962819535285234, 1.0984317064285278, 0.22899776697158813, -0.2905566990375519, 0.08587788045406342, 0.30194368958473206, 0.8976543545722961, -0.020654497668147087, -0.004938961006700993, 1.4216785430908203, -0.7720365524291992, 0.28828027844429016, 1.3622326850891113, -0.017460597679018974, -0.6068228483200073, 1.1735377311706543, 0.876183271408081, 0.618122398853302, 0.022081127390265465, -0.63487309217453, 0.5685470700263977, -0.7768811583518982, -0.42855459451675415, 0.00704676378518343, -0.17011557519435883, -0.4815651774406433, -1.1503502130508423, -0.1256202608346939, 0.15474635362625122, -1.2899430990219116, 0.06988652050495148], [-0.2329074889421463, -0.036469340324401855, -0.9009217023849487, -0.6750026941299438, 1.3511477708816528, -0.9193712472915649, -0.48258551955223083, 1.1231834888458252, -0.9338412880897522, -0.6515620350837708, 0.6499268412590027, 0.773311197757721, -0.5195835828781128, 0.11719478666782379, 0.9193043112754822, 0.04344428330659866, -1.8239871263504028, 0.339802622795105, 0.9426324963569641, 0.385701060295105, -0.35901862382888794, 0.022776884958148003, -0.34949612617492676, -0.33901718258857727, 0.10661947727203369, 0.2638690173625946, -0.9969668388366699, 0.09758434444665909, -0.0025416912976652384, -1.002272129058838, 1.4944912195205688, 0.0830511599779129, 0.8664290904998779, 0.0024714916944503784, 1.4409674406051636, 0.6508946418762207, 1.18509840965271, -0.024566255509853363, 0.9144356846809387, -0.9701168537139893, 0.32827043533325195, -1.083577036857605, 0.5978246331214905, 0.9470736980438232, 2.0313494205474854, 0.2542903423309326, 1.045515537261963, -1.4515302181243896, 0.0294093769043684, 0.18911850452423096, -0.9625914096832275, -0.7229345440864563, 0.651670515537262, 0.5159866213798523, 1.07156240940094, 0.5685282945632935, 0.7326558828353882, 0.760735273361206, 1.5465943813323975, 1.3595448732376099, 0.5595533847808838, -0.9072167873382568, -0.5148715376853943, 1.052158236503601, -0.6620727181434631, 0.01080833375453949, -0.06973619014024734, -0.7205185890197754, 1.002869963645935, 0.05526924878358841], [0.2615765631198883, -0.4866672158241272, 0.24980637431144714, -0.5826014876365662, -1.2859071493148804, -0.4195399284362793, 0.5926623940467834, 1.4062719345092773, -1.5123107433319092, -1.6574735641479492, -0.7211925387382507, -0.0061328657902777195, 0.13395287096500397, -1.0916800498962402, 0.14121660590171814, -2.1955275535583496, 0.2734695076942444, 1.1726762056350708, -1.730522871017456, -0.5513784289360046, -0.6103554964065552, 1.336389183998108, 0.16094066202640533, -0.8935160636901855, 0.20170806348323822, 1.0508261919021606, 0.26501089334487915, -0.9486764073371887, 0.6176599860191345, 0.675728976726532, -0.16526775062084198, -0.33341366052627563, 1.0600067377090454, -0.7066901922225952, 0.41956284642219543, -0.010582447983324528, -0.10704962909221649, 1.4770931005477905, 0.46347275376319885, 0.6352806687355042, 0.45451581478118896, -0.5590769052505493, 0.1050659790635109, 1.2143861055374146, -0.10433448851108551, -0.15397131443023682, -0.8116874694824219, -2.0432024002075195, -1.2680732011795044, -0.25865694880485535, 0.8587332963943481, 0.9608488082885742, 0.5468617677688599, -0.4905630350112915, -1.8559848070144653, -0.05346497520804405, -0.5203754305839539, -1.508646011352539, -0.23100975155830383, 1.4237200021743774, -0.0021602471824735403, -0.18762020766735077, 1.1328871250152588, 0.5359842777252197, -1.353079080581665, -1.0657985210418701, 1.0051627159118652, -1.512756109237671, 0.4409688115119934, -0.20729504525661469], [-0.0031810072250664234, 1.0959011316299438, -0.2958556115627289, 0.609617292881012, -1.1384471654891968, -0.726478099822998, 1.3008663654327393, -0.49405625462532043, 2.0088491439819336, -0.6450119614601135, -2.2915093898773193, -1.3845620155334473, -0.5785183906555176, -0.056175533682107925, -2.1216635704040527, 0.42443716526031494, -0.09772885590791702, -0.5653775334358215, 1.8348186016082764, 0.13176903128623962, -0.04257003217935562, -2.1282105445861816, 1.9351527690887451, -0.4545278251171112, 0.4194806218147278, -0.15130412578582764, -0.21560856699943542, 0.48897409439086914, -1.369717001914978, 1.4586623907089233, 0.6876012086868286, 0.20515912771224976, 1.5866155624389648, -1.1472266912460327, -0.0834481492638588, 1.0292890071868896, 0.6628395915031433, 0.7297863960266113, -1.3303413391113281, -1.8616409301757812, 0.3886983394622803, 0.011129070073366165, -0.6896387338638306, 0.8254024982452393, -0.5294515490531921, 0.07662565261125565, 0.5105022192001343, 1.4999916553497314, 0.857570469379425, 0.6181437969207764, -0.34131503105163574, 1.0661191940307617, -2.4110376834869385, -1.113207221031189, -0.047107718884944916, 0.27092498540878296, -0.5679925084114075, 1.345246434211731, 0.3341391384601593, 0.5376935005187988, -0.05355779081583023, 1.916526198387146, 0.9013684988021851, 0.28723862767219543, 2.9741227626800537, -0.4212148189544678, 0.40471136569976807, -0.7996025085449219, -1.5012080669403076, -0.2816006541252136], [0.794671893119812, -1.6939948797225952, 0.4832860231399536, -1.0529919862747192, 0.38371142745018005, -1.2784456014633179, 0.4901684820652008, -2.0026662349700928, 0.7441406846046448, -0.07320816814899445, -1.1205106973648071, -0.5991027355194092, 0.5972432494163513, 0.018801407888531685, -1.4984958171844482, 0.08221110701560974, -0.6990966796875, -1.183659553527832, 0.5460423827171326, 0.33063656091690063, -2.3249659538269043, 1.7356762886047363, 0.5797150731086731, 0.129365012049675, -1.1003549098968506, -0.09491850435733795, -0.30516359210014343, 0.9536889791488647, 1.2716619968414307, 1.4596821069717407, 1.5584979057312012, 0.3636757731437683, 0.01702883094549179, -0.7777265310287476, -0.8254966139793396, -1.9606043100357056, -0.4710310697555542, -0.5676445960998535, -0.8768438696861267, -1.166938304901123, 0.1380566954612732, 0.5694388151168823, 0.5191088318824768, 1.3585695028305054, -1.7716623544692993, 1.1965157985687256, 1.1021922826766968, 0.12332939356565475, -0.4562727212905884, -2.0442519187927246, 0.1453721523284912, 0.39236751198768616, -0.6940842866897583, 0.04444205015897751, -0.0529487207531929, -0.7453556656837463, 2.0728416442871094, -2.7195794582366943, -1.3926670551300049, 0.935433030128479, 0.5106262564659119, -0.8493965864181519, 2.365178108215332, -1.109399437904358, 0.2314014434814453, -0.9129252433776855, -0.2767149806022644, 0.261813759803772, -0.7307666540145874, -0.18328864872455597], [0.2637890875339508, 0.6947709321975708, -0.650030791759491, -0.0495627261698246, 1.7225288152694702, 2.298802137374878, -2.3802976608276367, -0.6820511221885681, 0.25638172030448914, 0.2490958273410797, -1.9228243827819824, -0.8239014148712158, 0.7167131304740906, 3.2657458782196045, 1.0743175745010376, 0.992229163646698, -0.5798612833023071, -0.9309375286102295, 0.16479410231113434, 1.5195233821868896, -1.0727169513702393, -0.3691738247871399, -0.48445549607276917, -0.2253091186285019, 0.4621289074420929, 0.04162735864520073, -1.002456545829773, -0.773220956325531, -0.16319505870342255, -0.5375954508781433, 0.5017535090446472, 1.0131354331970215, 2.4104695320129395, 0.5000672340393066, -1.5897142887115479, 1.0952800512313843, -1.1863113641738892, -0.6373858451843262, 0.8091775178909302, -0.2207738608121872, -0.41130518913269043, -2.2149252891540527, -0.4894750118255615, -0.4478989541530609, -0.7294209003448486, -0.3899639844894409, -1.1872572898864746, -1.4171388149261475, -0.758120059967041, 0.4645496606826782, -2.1011109352111816, 0.6158577799797058, 0.5824797749519348, 1.1805264949798584, -0.5835021734237671, -0.05076941102743149, 1.2061909437179565, -0.8506802916526794, -1.6964951753616333, -0.014775999821722507, -0.4980224370956421, 0.13658586144447327, -0.9755467772483826, -0.5152084827423096, 0.3631933927536011, 0.0828484371304512, -0.14639289677143097, -2.2320985794067383, -1.014204502105713, -1.1980968713760376], [2.6921353340148926, -0.27364376187324524, -0.7255210280418396, 0.43824318051338196, -1.683575987815857, 0.28334298729896545, 1.3457292318344116, 0.9718425273895264, -0.7029590606689453, 1.8138573169708252, 1.7134243249893188, -1.9720189571380615, 0.0016799228033050895, -0.5788780450820923, -0.8600907921791077, -0.6420004963874817, -0.9122701287269592, 1.234433889389038, 0.07742773741483688, 0.13421468436717987, 1.4129496812820435, -0.3445596396923065, 0.22963844239711761, -0.9766693115234375, 0.24904973804950714, 2.197327136993408, 0.5724201202392578, -0.5398250222206116, -0.4663202166557312, -0.23544850945472717, 0.7117171883583069, -0.7217600345611572, 0.18170619010925293, 0.010472232475876808, 0.9797576665878296, -0.45343178510665894, 1.6405175924301147, 0.5110697746276855, -1.0340447425842285, -1.118800401687622, -0.7408480048179626, -0.6474724411964417, -0.9057098031044006, 0.9176476001739502, 0.8085467219352722, -0.14376941323280334, 0.3812638223171234, -0.00020983740978408605, 0.9300366044044495, -0.019700894132256508, -0.936846137046814, 1.1052151918411255, 0.24265024065971375, 1.0265369415283203, -1.4515423774719238, 0.645563006401062, 1.4243874549865723, -0.989615261554718, 0.15496210753917694, -0.5457296967506409, 1.4721858501434326, -1.1934534311294556, -0.9925718903541565, 0.2664087414741516, -0.9793201684951782, -0.7025161385536194, 1.1915932893753052, -1.601926326751709, -0.4109054207801819, 1.8861627578735352], [1.0628398656845093, -1.3697346448898315, 0.3699960708618164, 0.45246216654777527, 0.9661010503768921, 0.24005699157714844, -0.19406136870384216, -0.3033582866191864, 0.8953250646591187, -1.3294044733047485, 0.4177280366420746, -2.208542585372925, -0.8160519003868103, 0.12739774584770203, 1.425226092338562, -1.5997439622879028, 0.026175128296017647, 0.1334695667028427, 0.15008804202079773, -0.3193928599357605, 1.0803266763687134, -0.5986247062683105, 0.6817699074745178, -0.3735791742801666, -0.39860087633132935, 0.8701329827308655, 0.3469662368297577, -0.14563007652759552, 1.037266492843628, 0.11792951077222824, -0.9985672831535339, -1.009114384651184, -0.22369149327278137, 0.5250216126441956, -1.5448309183120728, -0.12265219539403915, 0.030970368534326553, -0.10526944696903229, 1.1930903196334839, -1.125120759010315, 1.5685577392578125, 0.6181604266166687, 0.24790610373020172, 1.1727416515350342, -0.684319794178009, -0.5492177605628967, -0.34073197841644287, -1.5653105974197388, -0.32468393445014954, -0.5025843381881714, -0.7173975110054016, -1.7715775966644287, -0.5717772245407104, -0.5592949390411377, -0.4581872224807739, -0.12429323047399521, -0.7270760536193848, -0.37966132164001465, -0.019446542486548424, 0.3704140782356262, 1.4898958206176758, 0.7176023721694946, 1.0401628017425537, 1.3352315425872803, -0.42937421798706055, -1.1167930364608765, -1.0996211767196655, 0.5709444880485535, 0.5406415462493896, 1.363118052482605], [0.030799228698015213, -1.5634934902191162, 0.3970395624637604, 0.05759197100996971, 0.32175660133361816, 0.2523515820503235, -0.20989754796028137, 0.678878128528595, -0.020526722073554993, 0.522081196308136, -0.6847794055938721, -1.5565273761749268, 0.8954436182975769, -1.5000402927398682, 0.8241360187530518, -0.8461757302284241, -0.04987715557217598, -1.464902400970459, -1.5673692226409912, -0.20772789418697357, 0.0556974932551384, -1.8023072481155396, -0.43691155314445496, -1.126330852508545, -2.7634198665618896, -0.8640654683113098, -0.3108064830303192, 0.26999372243881226, -1.0000892877578735, -0.8330262899398804, 0.8903170228004456, 1.7766739130020142, -1.7937374114990234, 0.6120229959487915, 0.15903091430664062, 0.9086666107177734, -0.5850419402122498, 0.4686700105667114, -0.4664839804172516, -0.11336977034807205, 0.00031620223307982087, 0.0034061812330037355, -0.4724594056606293, -1.2995136976242065, 0.84581059217453, 1.285802960395813, -1.230915904045105, -1.3024635314941406, 1.873799443244934, -1.3025184869766235, 0.21457579731941223, -1.346419334411621, 1.4612720012664795, 0.3063717484474182, -1.3314974308013916, 1.3459439277648926, 1.6184614896774292, 0.19682897627353668, 1.154363989830017, -0.0240484531968832, -0.5184795260429382, -0.8044363260269165, -0.3359692096710205, 2.120347738265991, -0.0785721018910408, 2.1474053859710693, -2.7641944885253906, -0.41393139958381653, -2.8574013710021973, -1.1413911581039429], [1.3464950323104858, 0.33802810311317444, 0.08348707109689713, 0.14180073142051697, -0.5707988142967224, -1.1298582553863525, 1.3951526880264282, 0.1446598917245865, -1.687770962715149, -0.9476631283760071, -0.9979718923568726, 0.7243666648864746, 0.5538193583488464, 0.23516453802585602, -0.18895645439624786, -0.8155350089073181, -0.8408425450325012, -0.21071752905845642, 0.39810407161712646, -0.1142343208193779, 0.06720273196697235, -0.6857346892356873, 0.4686170518398285, 0.7671589255332947, -1.482757806777954, 0.5463289618492126, 0.714851975440979, -1.9592877626419067, -1.6986923217773438, 0.7021442651748657, 0.28057757019996643, 0.7411226034164429, 0.36981627345085144, -0.6869109869003296, -0.6397427320480347, -1.3644438982009888, 0.1187211126089096, -0.17624659836292267, 1.3568472862243652, -1.3893177509307861, 0.5321850180625916, 1.096045970916748, -0.5042386651039124, -0.38338702917099, -0.8772920966148376, -0.9037691950798035, 0.19756273925304413, -2.7266132831573486, -1.8475197553634644, 2.126420497894287, -0.9197064638137817, -0.8574692010879517, -1.0521759986877441, 1.134427547454834, -0.3060855269432068, 1.0209773778915405, 1.1525065898895264, -2.242464780807495, 0.07393144071102142, -0.403047114610672, -0.8702646493911743, 0.16952532529830933, -0.0754387155175209, 0.5173395276069641, -2.0889878273010254, 0.29156777262687683, 0.8032830357551575, -0.019201282411813736, -1.0242266654968262, 0.0601656548678875], [0.6200699806213379, 1.354433298110962, -0.9899015426635742, -0.15836267173290253, -0.24456243216991425, -0.5152182579040527, -0.49254918098449707, 0.5741260051727295, 1.852135419845581, -0.6694885492324829, 0.06834075599908829, -0.046945635229349136, -0.25490638613700867, -1.917226791381836, -1.15633225440979, -0.18515248596668243, 0.6907395720481873, 0.7615931034088135, -0.5019304156303406, 2.176117420196533, -0.6158872246742249, 0.8158695101737976, -0.888400137424469, -0.017836980521678925, -0.6114625930786133, 0.772961437702179, 0.43264269828796387, -0.3749217987060547, 1.3679821491241455, 0.5047932267189026, 0.032169193029403687, 1.5553580522537231, -0.34170594811439514, 1.8370946645736694, -1.1947327852249146, 0.1534438580274582, 1.1343120336532593, -1.186240792274475, 1.1846859455108643, -1.8040497303009033, 0.5157070159912109, 0.6013340353965759, 0.518248975276947, -0.15572191774845123, 0.044361889362335205, -0.22074632346630096, -1.692512035369873, 0.5061233639717102, 0.5379152894020081, -0.007332373410463333, -0.4965779185295105, 1.0164930820465088, 0.8596020936965942, 1.8921762704849243, -0.5117799043655396, -1.5586596727371216, -0.6504849791526794, -0.056472692638635635, 0.4459199607372284, -0.5348396897315979, -0.10347826778888702, 1.204482078552246, -0.7705656290054321, -0.056509193032979965, 0.7468891739845276, 0.760338544845581, 0.2812921106815338, -0.6176493763923645, 0.9533157348632812, -0.3179340064525604], [-1.2704596519470215, -0.03538898378610611, -1.7753044366836548, 1.693306803703308, 0.06882685422897339, 2.5860466957092285, -0.7681459188461304, -1.6559948921203613, -0.297824889421463, 0.5018019676208496, 1.1594560146331787, -0.37066394090652466, -0.49895134568214417, 0.8831310272216797, -0.7710071206092834, 0.15826933085918427, -0.4610120952129364, 0.6351568698883057, 0.5883492231369019, 0.4884583055973053, -0.18950459361076355, -2.132002353668213, 0.26050248742103577, 0.11690076440572739, 1.0326918363571167, 0.8297448754310608, 0.1118621826171875, 0.12585106492042542, 1.4426262378692627, 0.7007337808609009, 0.04052663594484329, 0.771317720413208, 0.01365606114268303, -0.14819006621837616, 0.6049183011054993, 0.2812672555446625, 1.8529871702194214, -0.9656842350959778, -1.5007795095443726, 1.0048527717590332, -0.4193272888660431, -0.03471959009766579, -0.19232012331485748, 0.018111590296030045, -1.439591884613037, -0.35855185985565186, 0.0847596675157547, 0.3559610843658447, 0.2217695116996765, 0.46605440974235535, 0.5448307991027832, -1.3864448070526123, 1.2799447774887085, -2.0429015159606934, 0.4220474064350128, -1.0933343172073364, -0.24848899245262146, -1.1553549766540527, 0.5152639150619507, -1.116151213645935, -0.5707465410232544, 0.6866588592529297, -1.5166617631912231, 0.5325042605400085, -0.34893620014190674, -0.4949268400669098, -1.14829421043396, 0.9484717845916748, 1.033052921295166, 0.47299548983573914], [-1.734916090965271, 0.21152454614639282, -0.4168086051940918, -0.6380833983421326, 0.9712266325950623, -0.6325545310974121, -0.009756742976605892, 0.052589643746614456, -0.20359280705451965, 1.5047141313552856, 0.07934094220399857, 1.2468199729919434, -2.2829177379608154, -1.2147011756896973, -0.5597510933876038, 0.9411425590515137, 0.34944674372673035, 0.47238942980766296, 1.0912500619888306, -0.6259028315544128, 1.5566397905349731, -0.7216721773147583, 1.4389532804489136, 1.699727177619934, 0.12699630856513977, -0.9935365319252014, 0.48595505952835083, -1.9047907590866089, -1.0271410942077637, 1.32833993434906, 0.3970901072025299, -0.46242234110832214, -0.3267369866371155, 0.24625352025032043, -1.077258825302124, 0.28427425026893616, 0.4687544107437134, -0.16080939769744873, -1.1361397504806519, -1.9721776247024536, 0.5118255615234375, 0.2815760374069214, 0.5273334383964539, 0.4356797933578491, -0.009268699213862419, -0.6582273244857788, -1.1307936906814575, -0.822199285030365, -0.4134184718132019, -1.2190115451812744, 0.7201541662216187, 0.6252344846725464, 1.0834981203079224, 0.29206401109695435, -0.20837686955928802, 0.4422558844089508, 0.14707554876804352, 0.5565720200538635, -0.3908960223197937, -0.7889446020126343, -0.03122340701520443, -1.1237037181854248, 0.46254464983940125, 1.0345171689987183, 0.4083998501300812, -0.3691678047180176, 0.3278341293334961, -1.1316759586334229, 0.4490339457988739, 0.7708394527435303], [0.16706210374832153, 1.215351939201355, -0.029603533446788788, -1.6154485940933228, -1.036468505859375, 0.24682489037513733, 0.24951063096523285, 0.22266541421413422, 1.0333689451217651, 0.30153709650039673, -0.30261534452438354, 0.38606947660446167, 0.4715295732021332, 0.4503501355648041, -0.14246386289596558, 0.4895637035369873, 0.42638930678367615, 0.9338672757148743, 1.0083880424499512, -0.32771992683410645, -0.8364623785018921, 0.43854647874832153, -0.06584983319044113, -1.1827229261398315, 0.9528862237930298, -0.07478807866573334, 0.642806351184845, 2.0060837268829346, 0.4290037751197815, 0.023261787369847298, 0.2831163704395294, -1.173699975013733, 0.020874449983239174, -0.2887563407421112, 0.1298540234565735, -1.242069125175476, 0.007437718566507101, 0.4143950045108795, -1.4389487504959106, -0.5601195693016052, 0.12007176131010056, -0.8326181173324585, 0.5338985323905945, 0.8230530619621277, -0.5921337604522705, -1.1215758323669434, 0.095308817923069, -0.6493937373161316, -0.6979823112487793, 0.3403180241584778, 0.6667659878730774, 2.8089375495910645, 1.0696128606796265, -0.7595387697219849, -0.07849861681461334, 0.1867814064025879, 0.5167560577392578, 0.8287913799285889, -0.5047172904014587, 0.562182605266571, -0.9921565651893616, 1.5302797555923462, 1.2871296405792236, -0.023652784526348114, 0.4528365433216095, 0.7291001081466675, 1.0295780897140503, 2.086017370223999, -0.46002233028411865, -0.3761695325374603], [1.1727728843688965, -1.7486076354980469, 0.3268463611602783, -0.9927005171775818, 1.6584243774414062, -0.006028777454048395, -1.1513065099716187, -1.4120293855667114, -1.3875293731689453, -0.994541347026825, 0.006782864686101675, 0.1981063336133957, -0.11952829360961914, 0.6621193289756775, -0.8362443447113037, -0.3897642195224762, -0.3555520474910736, -0.4738425612449646, -0.11774252355098724, 0.19027294218540192, -2.0884106159210205, 0.25022706389427185, -0.2021178901195526, -1.2695826292037964, 1.0212236642837524, -0.3355437219142914, -0.32979142665863037, -0.5731181502342224, 0.07543494552373886, 0.03885117918252945, -0.0882900133728981, 0.9547404050827026, -0.10212858766317368, -1.9490338563919067, -0.4423217475414276, -1.2140971422195435, -0.20176100730895996, 1.8661201000213623, -1.075400471687317, -0.04222387075424194, -0.787118673324585, -0.3470574617385864, 0.2563706934452057, 0.6870332956314087, -0.2367004007101059, 0.12651976943016052, 0.7597976922988892, -0.12298724800348282, 1.9336130619049072, 1.5458134412765503, 1.440853238105774, 0.23725906014442444, 1.078911542892456, 0.25513285398483276, 0.8649300336837769, 2.0233147144317627, -0.5338375568389893, -0.5875112414360046, 0.9239002466201782, 1.4154505729675293, -1.9957892894744873, 0.043913379311561584, -0.8902871608734131, -0.2765849232673645, -1.7723805904388428, 0.4259832799434662, 1.185483455657959, 1.473586082458496, 0.24472661316394806, 0.44075700640678406], [-1.2747660875320435, 1.406562089920044, 1.2259553670883179, -0.5451957583427429, -1.4632256031036377, 0.9340304136276245, 0.08796077221632004, -0.2574270963668823, 1.0839734077453613, -0.63044273853302, 0.06090070307254791, 1.1283589601516724, 0.19961228966712952, 0.8915780782699585, 0.8484931588172913, 1.0845314264297485, 0.4327888488769531, 0.4378786087036133, 0.2015315145254135, -1.2375409603118896, 2.3051259517669678, 1.9568593502044678, -1.3228850364685059, 0.2159847915172577, -0.2672962546348572, -1.7399007081985474, -0.422714501619339, -2.288527011871338, 2.67518949508667, 0.6887146830558777, 0.1455542892217636, 0.24216695129871368, 1.101943016052246, 0.8886591792106628, 0.23697078227996826, 1.0950686931610107, -1.8461666107177734, -2.4438157081604004, -0.20318163931369781, -0.5675692558288574, -0.557577908039093, 0.778339684009552, -0.6397757530212402, 0.2992329001426697, 0.8955854177474976, 0.32563090324401855, 0.36440545320510864, 0.9106916189193726, -0.028256874531507492, -1.2476704120635986, -0.6539366245269775, 1.4500845670700073, 0.8162475228309631, 1.2601393461227417, 1.1065778732299805, 1.206062912940979, 1.2809820175170898, 0.9322572946548462, 0.30152812600135803, -1.074425458908081, 1.3337502479553223, 0.12051530927419662, 0.08875353634357452, -1.0242570638656616, 2.0066277980804443, -0.4804028272628784, -1.258397102355957, 0.86598801612854, 0.649144172668457, -0.551755428314209]], "dtype": "float32"}, {"__nd__": [1.1338187456130981, -1.2532843351364136, -0.3652786612510681, -0.9231684803962708, 0.803765594959259, -1.4165500402450562, 1.2510185241699219, 0.8251039981842041, -1.4176794290542603, 1.841830849647522, -0.10966306924819946, -0.7974951863288879, 0.13745731115341187, 0.6439968347549438, -0.6462289690971375, 1.43292236328125, -0.9513416886329651, -1.2949386835098267, -0.3569820821285248], "dtype": "float32"}], "e": {"__nd__": [[4.791030406951904, 0.0, 14.647356033325195, 6.814721584320068, 2.2006123065948486, 1.4865983724594116, 0.0, 4.154424667358398, 2.757983684539795, 12.236180305480957, 0.0, 10.674356460571289, 9.165033340454102, 20.464462280273438, 0.0, 0.0, 4.564984321594238, 16.580217361450195, 0.11030195653438568], [0.0, 3.6040456295013428, 7.166036128997803, 2.729153633117676, 0.0, 4.400463581085205, 2.450493812561035, 3.381084680557251, 0.0, 0.0, 3.414837121963501, 0.0, 3.64442777633667, 0.0, 0.0, 5.015572547912598, 3.6010639667510986, 0.0, 0.0], [0.0, 7.201076030731201, 14.524604797363281, 0.0, 0.0, 0.0, 5.924560546875, 0.0, 22.19718360900879, 0.0, 0.0, 0.0, 0.0, 0.0, 13.584482192993164, 2.819243907928467, 10.439811706542969, 0.0, 12.8819580078125], [10.11590576171875, 0.0, 7.721652984619141, 0.0, 4.0163187980651855, 0.0, 4.589879512786865, 0.0, 0.0, 11.960269927978516, 0.0, 2.652698516845703, 6.307911396026611, 0.0, 10.632469177246094, 6.412042140960693, 0.3008463680744171, 0.0, 4.113990783691406], [15.905149459838867, 0.0, 15.264986038208008, 2.6001508235931396, 5.91597318649292, 6.781607627868652, 0.0, 0.0, 0.0, 8.44444465637207, 0.0, 0.0, 0.0, 14.822896003723145, 0.0, 3.4269211292266846, 0.0, 0.5047788619995117, 0.0], [5.3284687995910645, 9.903270721435547, 0.0, 0.0, 3.1957664489746094, 0.0, 12.712882995605469, 0.0, 0.0, 13.567255020141602, 9.724142074584961, 0.0, 1.9867225885391235, 0.0, 0.0, 16.55484390258789, 0.0, 0.0, 0.0], [3.6061723232269287, 4.790197372436523, 0.0, 2.804849147796631, 0.0, 1.154811143875122, 10.42859172821045, 9.07348346710205, 0.0, 0.9020370244979858, 0.0, 5.737429618835449, 0.0, 6.971949577331543, 4.001257419586182, 4.3354926109313965, 0.0, 7.3436713218688965, 0.0], [2.9668052196502686, 17.695545196533203, 20.078907012939453, 0.0, 2.7831273078918457, 0.0, 0.0, 0.0, 0.0, 1.4521907567977905, 0.2345433533191681, 12.178352355957031, 2.606086492538452, 0.0, 0.0, 0.0, 0.0, 0.7628767490386963, 0.47235730290412903], [0.0, 0.0, 0.0, 0.0, 0.42522579431533813, 6.6055755615234375, 3.708599090576172, 14.333157539367676, 1.0137666463851929, 5.192303657531738, 12.064082145690918, 0.0, 4.548847675323486, 1.9730857610702515, 3.2530107498168945, 0.0, 0.0, 1.6245685815811157, 0.7607576251029968], [2.602980613708496, 6.06890869140625, 5.045167922973633, 13.466093063354492, 13.135407447814941, 7.5395708084106445, 0.0, 3.8773915767669678, 0.0, 15.021565437316895, 4.863708019256592, 9.011795043945312, 0.0, 0.0, 0.0, 3.3566792011260986, 4.676700592041016, 0.0, 0.0], [2.6622045040130615, 0.0, 0.0, 0.0, 0.0, 0.0, 16.682655334472656, 8.391386032104492, 0.0, 13.492854118347168, 0.0, 17.4428653717041, 1.5682165622711182, 6.342231273651123, 0.0, 0.0, 2.258561134338379, 0.0, 2.365898847579956], [0.0556771419942379, 0.0, 0.0, 0.0, 3.5631611347198486, 2.844379186630249, 0.0, 1.289017677307129, 0.0, 0.0, 0.9890296459197998, 1.9040228128433228, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.566174507141113], [3.0865378379821777, 0.0, 0.0, 0.0, 3.435433864593506, 3.7986104488372803, 0.0, 5.830220699310303, 0.0, 14.008687019348145, 6.840001583099365, 0.0, 0.2651767134666443, 2.6634700298309326, 3.1020498275756836, 0.0, 1.170398473739624, 4.503133773803711, 0.0], [0.0, 0.0, 17.44677734375, 10.806389808654785, 0.6449791193008423, 0.0, 0.0, 4.169402122497559, 5.808907508850098, 1.4985182285308838, 8.824949264526367, 7.792230129241943, 4.067587375640869, 4.210918426513672, 0.0, 0.0, 1.393802285194397, 0.0, 9.766569137573242], [0.0, 0.0, 10.574090003967285, 0.0, 7.110625267028809, 3.2228217124938965, 0.0, 0.0, 7.214972972869873, 22.757631301879883, 2.231724262237549, 1.3003944158554077, 0.0, 4.557572841644287, 0.0, 5.254054069519043, 0.0, 1.2340043783187866, 0.0], [0.0, 0.0, 6.77654504776001, 1.2909812927246094, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.976839542388916, 0.0, 9.729486465454102, 0.18607157468795776, 7.676555633544922, 0.0, 0.0, 13.586568832397461], [0.0, 5.919119358062744, 0.0, 0.0, 2.1346595287323, 9.829265594482422, 0.0, 0.0, 0.0, 8.816278457641602, 1.5143097639083862, 0.0, 0.0, 0.0, 11.286656379699707, 13.088263511657715, 0.0, 4.347738265991211, 0.0], [16.139623641967773, 0.0, 1.1551871299743652, 0.0, 0.0, 0.0, 11.001760482788086, 0.0, 13.52013874053955, 6.661433219909668, 1.801324725151062, 0.5995391011238098, 0.0, 16.957313537597656, 1.9598051309585571, 0.0, 0.0, 0.0, 0.0], [0.0, 3.160238027572632, 0.5624933838844299, 2.53644061088562, 0.0, 0.0, 5.755428791046143, 2.400705099105835, 0.0, 22.727331161499023, 1.101245403289795, 0.0, 0.2149307280778885, 0.0, 2.716432571411133, 8.552762985229492, 5.93936824798584, 0.0, 0.6402795314788818], [5.954844951629639, 0.0, 15.81863021850586, 0.9335311651229858, 3.7132365703582764, 14.048932075500488, 0.0, 0.0, 0.0, 10.367406845092773, 0.0, 0.0, 0.0, 8.058119773864746, 0.0, 0.0, 0.0, 0.0, 2.6736133098602295], [1.1683868169784546, 2.0041587352752686, 0.0, 0.0, 0.0, 0.0, 9.137872695922852, 6.31561803817749, 0.0, 7.078792095184326, 5.02230167388916, 0.0, 2.2369601726531982, 0.0, 3.9332709312438965, 0.0, 16.074560165405273, 0.0, 0.0], [10.785183906555176, 2.458620548248291, 0.0, 0.0, 0.0, 12.722397804260254, 0.0, 5.2056884765625, 6.46431827545166, 15.51888370513916, 0.0, 0.0, 14.06954288482666, 0.0, 4.964888095855713, 0.0, 5.728187561035156, 3.758864641189575, 1.665992021560669], [2.910161256790161, 0.0, 4.600362300872803, 13.433032035827637, 0.0, 8.341354370117188, 0.0, 0.0, 0.0, 0.0, 5.7719902992248535, 1.2044569253921509, 0.0, 2.3372292518615723, 0.0, 4.41701078414917, 1.6329094171524048, 3.480372428894043, 17.211803436279297], [1.2784831523895264, 0.5808467268943787, 0.0, 4.864307403564453, 3.4352378845214844, 0.0, 0.0, 0.0, 9.848787307739258, 17.967350006103516, 6.831912517547607, 3.8453192710876465, 0.0, 0.0, 3.41646671295166, 13.293828010559082, 0.0, 0.0, 14.165247917175293], [0.0, 0.0, 1.2613525390625, 18.040607452392578, 0.0, 5.913139343261719, 0.0, 4.3451972007751465, 0.0, 0.0, 1.8333966732025146, 0.0, 7.638484477996826, 2.3175203800201416, 0.44306403398513794, 0.0, 0.0, 4.832172870635986, 8.821744918823242], [13.110570907592773, 7.635995388031006, 13.931795120239258, 0.0, 10.48529052734375, 0.0, 0.0, 3.3982365131378174, 12.01710033416748, 0.9022878408432007, 0.0, 20.217304229736328, 1.9046783447265625, 10.397188186645508, 3.2645797729492188, 0.0, 0.0, 0.0, 0.0], [19.77196502685547, 3.0990447998046875, 0.0, 0.0, 0.0, 10.438193321228027, 12.586564064025879, 17.8338623046875, 0.0, 0.0, 0.0, 0.0, 9.369311332702637, 0.0, 0.0, 6.328508377075195, 10.704344749450684, 0.0, 9.446884155273438], [11.555500984191895, 17.31352996826172, 0.3702957034111023, 0.0, 19.141136169433594, 4.492494583129883, 0.0, 7.406427383422852, 0.0, 0.0, 0.0, 0.9547536969184875, 0.47164130210876465, 0.0, 0.0, 13.542901039123535, 0.0, 0.0, 11.225193977355957], [9.982598304748535, 8.329656600952148, 0.0, 0.0, 6.95023250579834, 0.0, 0.0, 6.6796345710754395, 4.010920524597168, 0.8171846866607666, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.85720157623291], [0.0, 5.065155506134033, 0.0, 0.0, 0.0, 0.0, 0.0, 9.042506217956543, 0.2631402015686035, 0.0, 0.0, 7.44746208190918, 2.073132276535034, 0.0, 0.0, 0.0, 2.5060009956359863, 0.0, 2.5671396255493164], [2.167044162750244, 0.0, 8.23128604888916, 3.3320910930633545, 0.0, 3.8973796367645264, 4.870172500610352, 1.970846176147461, 6.614879131317139, 5.679359436035156, 0.0, 1.2125011682510376, 4.191586017608643, 2.9838197231292725, 0.0, 0.0, 11.902986526489258, 2.5394537448883057, 6.668725967407227], [0.0, 0.0, 20.346303939819336, 0.0, 14.774068832397461, 0.0, 0.0, 0.0, 0.0, 10.09801197052002, 5.289924621582031, 0.0, 2.029989004135132, 0.0, 3.6719534397125244, 0.0, 0.0, 0.841739296913147, 0.5396296381950378], [0.26038870215415955, 0.0, 0.0, 0.0, 0.0, 0.0, 15.302725791931152, 0.0, 0.0, 0.0, 0.0, 9.165620803833008, 0.0, 0.0, 0.0, 5.566701412200928, 4.7220234870910645, 0.0, 8.859207153320312], [0.8368610739707947, 0.0, 0.0, 0.0, 4.445540904998779, 3.8646774291992188, 0.0, 0.0, 2.4022912979125977, 6.515738010406494, 0.0, 0.0, 1.5474581718444824, 6.1365885734558105, 0.0, 10.294682502746582, 7.653277397155762, 0.0, 23.621503829956055], [5.921319961547852, 5.180976867675781, 0.0, 0.0, 0.0, 0.0, 5.138876914978027, 0.0, 0.0, 1.4403724670410156, 6.620254993438721, 2.377730369567871, 0.0, 0.0, 1.2578182220458984, 10.527999877929688, 6.513326168060303, 0.0, 0.0], [0.0, 0.0, 13.622575759887695, 0.0, 0.0, 5.031983852386475, 0.0, 0.8354725241661072, 0.0, 0.0, 4.514101505279541, 0.0, 8.088142395019531, 0.0, 4.366733551025391, 3.2189197540283203, 0.0, 2.9086530208587646, 0.0], [0.0, 0.0, 8.184503555297852, 0.0, 0.0, 0.0, 0.0, 12.997997283935547, 9.880755424499512, 0.0, 2.1057145595550537, 28.565092086791992, 0.0, 0.0, 0.019373303279280663, 0.0, 9.416112899780273, 0.0, 0.345360666513443]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.014662863686680794, 0.4175163507461548, -0.47254133224487305, 1.1717206239700317, -2.03790545463562], [0.2726537585258484, -0.8326484560966492, -1.683976411819458, 1.3722188472747803, -1.0120162963867188], [0.3363596200942993, -1.3248378038406372, 0.7428870797157288, -1.3586195707321167, 1.0474891662597656], [-0.3624807894229889, 0.721045970916748, -0.19197720289230347, 0.41261109709739685, 0.8457456827163696], [0.31408318877220154, -1.1239941120147705, -2.694974422454834, -0.9711863398551941, 0.7310958504676819], [-0.21472015976905823, -0.3571987450122833, -0.8791906833648682, 2.4687840938568115, 0.5070970058441162], [-0.5165259838104248, -1.705353021621704, 0.8019644618034363, -0.6133522391319275, 0.36831825971603394], [-1.480400800704956, -2.975383996963501, 0.2141239494085312, 0.9701715111732483, -0.07198984920978546]], "dtype": "float32"}, {"__nd__": [[-0.1901674121618271, 0.019341522827744484, -0.73227858543396, -0.13010857999324799, 0.8901511430740356], [-0.36686405539512634, 0.0805702805519104, 0.3584160804748535, -0.23186390101909637, -1.603018045425415], [0.8382386565208435, 0.12754635512828827, -0.13550995290279388, -0.08252903074026108, 0.498956561088562], [-0.1260247528553009, -0.41860559582710266, -0.6546623706817627, 1.3834620714187622, 0.3555644750595093], [-0.934567928314209, 0.20286819338798523, 1.2218457460403442, 2.9867265224456787, -1.2328954935073853], [-1.992349624633789, 0.9935787916183472, 0.21046604216098785, -1.3410046100616455, 1.1005375385284424], [-0.19110481441020966, 1.1051228046417236, 0.866243839263916, 0.7046096324920654, -0.7035282254219055], [-0.3794960081577301, 2.7205142974853516, 1.2427775859832764, -0.9552038311958313, -0.8145894408226013], [0.012258310802280903, -0.7961945533752441, 0.679166853427887, -0.2535170018672943, 0.47573792934417725], [1.1594302654266357, 0.7747335433959961, -1.4546436071395874, 0.02994178980588913, 0.6232971549034119], [0.3692494332790375, -0.1944017857313156, 0.44872140884399414, -0.992795467376709, -0.6264527440071106], [-0.9051035642623901, -1.0206351280212402, 1.02928626537323, -0.37329185009002686, 1.3993052244186401], [-1.5531095266342163, -0.42908135056495667, 0.45904093980789185, -1.0182645320892334, 2.21090030670166], [-0.04184996336698532, 0.9177617430686951, 0.09956077486276627, 1.2440063953399658, 0.9701336026191711], [-0.034726135432720184, 0.4017692506313324, 1.391345739364624, 0.37041160464286804, 2.038484573364258], [0.5431233048439026, -0.4066201150417328, 0.9770691394805908, 0.023939695209264755, -1.180504322052002], [-0.034190382808446884, 0.4026266634464264, -0.20333589613437653, 0.33445221185684204, -0.09790559113025665], [0.0632065013051033, -0.33088287711143494, -1.4778441190719604, 1.7865478992462158, 1.2778100967407227], [0.2840654253959656, 0.7117384076118469, -0.3044082224369049, -0.2452136129140854, 0.6215062141418457], [1.530266523361206, -0.7947506904602051, -0.15566907823085785, 0.5882652997970581, -0.42643243074417114], [-0.6008760929107666, 1.2957336902618408, 0.2129836082458496, 0.10695231705904007, -0.25064516067504883], [2.100264549255371, -0.9578306078910828, 0.04924624785780907, -1.794437289237976, 1.9639668464660645], [-1.0147472620010376, -1.715080976486206, 0.13267038762569427, 1.4461150169372559, 0.7380008101463318], [1.3515243530273438, 0.3624078631401062, -1.3183858394622803, 0.5292319655418396, -0.07192036509513855], [-1.286358118057251, -0.07880625128746033, 0.16889499127864838, -0.04081498086452484, 0.07999122142791748], [1.3079174757003784, 1.9861749410629272, -1.454317331314087, -0.7388665080070496, 0.4900761842727661], [-0.7538750171661377, 0.003479280509054661, -0.07561089098453522, 1.2312918901443481, 0.029579712077975273], [-0.5882896184921265, -0.7991794347763062, -1.2908446788787842, -0.587807834148407, 1.5422337055206299], [-0.2700795829296112, -0.056636642664670944, 1.108934998512268, -0.9647698998451233, 0.7102833390235901], [-0.09938207268714905, -0.3220949172973633, 1.0878870487213135, 0.013123667798936367, -1.8747104406356812], [2.7044756412506104, 1.2124645709991455, 0.32799291610717773, -1.9800326824188232, -0.3271312713623047], [0.961029589176178, -0.47612640261650085, -1.034881591796875, 0.9252437353134155, 0.5334323048591614], [-0.925055742263794, 1.3628193140029907, 1.0290659666061401, 1.1113897562026978, -2.3375585079193115], [1.3611639738082886, -1.2883752584457397, 0.2703469693660736, -0.3629129230976105, 0.3352561593055725], [0.4905310273170471, 0.31582170724868774, 1.0815761089324951, -0.14986294507980347, 0.3369094133377075], [0.12525689601898193, -1.595971941947937, -2.1346311569213867, 1.179987907409668, 0.7880222797393799], [0.04842185601592064, -0.4189276099205017, -0.07239387929439545, 0.35440272092819214, -1.2989017963409424], [-1.056222915649414, 0.8483350872993469, -0.6941163539886475, 0.13222353160381317, -0.11988785862922668], [0.19737546145915985, 1.248407006263733, -0.6765572428703308, -0.4717290997505188, -0.7316868901252747], [-0.5606998205184937, -1.8451184034347534, 0.44590967893600464, -1.319211483001709, 1.0308510065078735], [-1.849887490272522, -1.6535087823867798, 0.17941293120384216, -0.5047076940536499, 0.28887447714805603], [-0.2586183249950409, -1.141504168510437, -1.501128911972046, -0.7490028738975525, -1.1618584394454956], [2.966550350189209, 0.4691931903362274, -0.21757638454437256, 0.6787170171737671, 0.33630603551864624], [1.3855230808258057, -1.056166172027588, 0.5728036165237427, 0.6011050939559937, 0.07835597544908524], [-1.3823236227035522, 1.159148097038269, -0.41502711176872253, 0.9481869339942932, -1.0376259088516235], [-0.6327806115150452, -0.1179463192820549, -0.6666686534881592, -1.421181321144104, 0.8162292242050171], [1.0668206214904785, 0.5991711020469666, -0.4231509268283844, -1.5460225343704224, 0.5201340913772583], [2.4894518852233887, -0.4430431127548218, -0.1498260200023651, 0.6452715396881104, 1.279638648033142], [-0.8943226337432861, -0.7780291438102722, -1.2030318975448608, 0.610541820526123, -0.49446743726730347], [-1.109129548072815, 1.327819585800171, -0.48847436904907227, -1.7606055736541748, -2.7591564655303955], [-0.7930060625076294, 0.4107772409915924, -0.9192050695419312, 0.6587496995925903, -0.950856626033783], [0.634576141834259, -0.15944130718708038, 1.0206636190414429, 0.3222755491733551, 0.27040764689445496], [0.07477595657110214, -1.2647324800491333, 1.053015947341919, -1.0547000169754028, 0.085238017141819], [-0.35428255796432495, 1.600579857826233, 1.8089923858642578, 1.2597585916519165, 0.8344480395317078], [1.5682209730148315, 0.7847478985786438, 1.4228394031524658, 0.12030711770057678, -1.299996018409729], [2.9547998905181885, 1.0760890245437622, 0.8148531913757324, -0.4314906597137451, -0.0566253624856472], [0.36750420928001404, 0.32023218274116516, -2.2842400074005127, -0.24466043710708618, -2.1397616863250732], [0.011338992044329643, -0.01958124339580536, -1.59336519241333, -1.4475845098495483, 0.771648645401001], [0.5507149696350098, -0.5658243298530579, -1.4240453243255615, 0.8088994026184082, 0.2822074294090271], [-0.7137455344200134, -0.24888980388641357, -0.08560775965452194, -0.8555506467819214, -1.8143690824508667], [1.6936289072036743, -0.7584318518638611, 0.47851449251174927, 0.3385125994682312, -0.9751561284065247], [0.8812839388847351, 1.0619534254074097, 0.8851302862167358, 1.1034797430038452, -0.7233819365501404], [0.3407568633556366, 0.17338821291923523, -0.7862218618392944, 0.2143126279115677, -0.5928519368171692], [-0.11315985769033432, 1.5730544328689575, 0.5502153038978577, -1.1587021350860596, -0.1686154156923294], [-0.24105657637119293, 1.5259689092636108, 0.4107159972190857, 0.5934235453605652, -0.2237825095653534], [0.04024292528629303, -0.9362071752548218, 1.3685234785079956, -1.4152113199234009, 0.38822799921035767], [0.02530279941856861, 0.31170105934143066, -0.6779190897941589, 1.1316583156585693, -0.7576392889022827], [0.9606608152389526, 0.12526269257068634, 0.5653107762336731, 2.5543644428253174, -1.0543750524520874], [0.010072615928947926, -0.5011117458343506, -1.3902150392532349, -1.3788831233978271, 0.5553614497184753], [-0.38315311074256897, -0.5916526913642883, 0.597589910030365, -0.5384775400161743, -0.7654372453689575], [-2.827213764190674, -1.3321325778961182, 0.45702022314071655, -1.3520123958587646, -0.28835275769233704], [0.780219554901123, 0.8809154629707336, -0.10122181475162506, 2.1131505966186523, 0.47568586468696594], [-0.5100060105323792, -0.26437991857528687, -1.0205413103103638, -0.16867601871490479, -2.1017377376556396], [-1.9068562984466553, -0.30986788868904114, -0.5652992725372314, -0.4705077111721039, -0.849460244178772], [-0.26084551215171814, 1.6353085041046143, -0.33207303285598755, -0.14973877370357513, -1.45354163646698], [-0.6901825666427612, 1.6282401084899902, -2.2246015071868896, -0.7581581473350525, -1.0423474311828613], [0.9350568652153015, 0.6476342678070068, 0.5584990382194519, 0.9307117462158203, -1.1012730598449707], [-1.0950877666473389, 0.894675612449646, -1.908389925956726, -0.498008668422699, -0.2958827316761017], [0.5331414937973022, 1.199005365371704, 0.323144793510437, 0.4959733188152313, -0.24877342581748962], [-1.6695451736450195, 0.034695807844400406, 1.069810390472412, 0.5264514088630676, 2.3193881511688232], [-0.760780394077301, 0.598913848400116, -0.6134651303291321, 1.9565227031707764, 0.49095892906188965], [0.4471420347690582, 0.884818971157074, 1.2388370037078857, 1.3853176832199097, -0.3089362680912018], [-0.5633097290992737, -0.24141600728034973, -0.48637303709983826, 1.7607004642486572, -0.6261120438575745], [-0.9240188002586365, 0.4896903932094574, 0.18849028646945953, -0.23305320739746094, 1.1435039043426514], [-1.1005077362060547, -1.0324043035507202, -1.635581135749817, 0.2722618877887726, 1.0527998208999634], [-0.8735296130180359, -0.015051743015646935, -1.109633207321167, -0.675270676612854, -1.9384729862213135], [0.1180596724152565, -0.1384911835193634, -0.9763278961181641, -0.3691615164279938, 0.2866517901420593], [-1.2982276678085327, -0.1438150256872177, 0.1207079067826271, -2.1847903728485107, 0.3285885453224182], [-0.2330043911933899, 1.0056254863739014, 0.6499727964401245, 0.7578533887863159, -0.7727317810058594], [1.0906920433044434, -1.4843759536743164, 1.4239190816879272, -0.3167247176170349, -2.4101953506469727], [0.3706267774105072, 0.3615834712982178, 0.40808531641960144, -0.5185343027114868, 0.4439728558063507], [0.18769948184490204, -0.047066930681467056, -0.2448548823595047, 0.998113214969635, 0.24826933443546295], [-0.6442291140556335, -0.6133899092674255, 0.6449149250984192, -0.2552606761455536, -0.31303149461746216], [0.99753737449646, -0.7569056749343872, -0.7921451330184937, 0.32195839285850525, 0.009763810783624649], [-0.19342385232448578, 0.9222729206085205, -0.5598133206367493, -1.3899848461151123, -0.09293326735496521], [-0.45587673783302307, 1.1837456226348877, 0.6472418904304504, -0.08010193705558777, -0.5001158714294434], [0.45330655574798584, -0.4251157343387604, -0.0792265236377716, 0.41233861446380615, 0.10027105361223221], [-0.2503334879875183, -1.1477957963943481, 1.1265079975128174, 0.6282532811164856, -1.04957914352417], [-0.049571432173252106, -0.9806080460548401, 0.6767899394035339, -0.6558874249458313, 0.04139396548271179], [-0.33729541301727295, 1.8648396730422974, 0.34243032336235046, 1.1082829236984253, -0.11039934307336807]], "dtype": "float32"}, {"__nd__": [-100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0], "dtype": "float32"}], "e": {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.31064364314079285, 0.8469423651695251, -0.43056029081344604, -0.40638861060142517, 2.306795120239258, -0.6372950673103333, -1.794058918952942, -0.416342169046402, -0.7688634395599365, 0.6959202885627747, -0.43107151985168457, -1.6417224407196045, -2.4994428157806396, 1.4921423196792603, 0.2692846357822418, 0.7151943445205688, -0.06243659928441048, 3.559403657913208, 0.27780869603157043, 0.09982628375291824, 0.4003342092037201, -0.0293294508010149, 0.4348661005496979, 0.8896337747573853, -1.3128323554992676, -0.5236961841583252, -1.6516139507293701, -0.6793867945671082, -0.49695271253585815, 1.0025464296340942, 2.4470248222351074, -0.4334809184074402, 0.12407714128494263, 1.475980520248413, 0.18854336440563202, 0.44831883907318115, -0.40034762024879456, -1.3969004154205322, -1.1053417921066284, -2.0127782821655273, -0.3680292069911957, -0.8840600848197937, 0.7452577948570251, 0.5918183326721191, 0.09469752013683319, -0.33801960945129395, 1.8753620386123657, 3.365507125854492, 0.3941285312175751, -1.5149927139282227, -0.8750150203704834, 1.0388915538787842, 0.19377848505973816, -0.18074309825897217, -0.1126571074128151, 0.9165748953819275, -0.6513800024986267, 1.212146282196045, 0.08892642706632614, -0.9618440270423889, 1.7159123420715332, -0.5312870144844055, -1.859845519065857, -0.11002053320407867, 1.8996731042861938, -0.8825649619102478, -0.770906925201416, 0.3920004367828369, -0.7344492673873901, 0.7928892374038696, -0.8595315217971802, -0.6580559015274048, -0.4626668691635132, 0.5009202361106873, 3.3997490406036377, -0.7170273065567017, -0.7201502323150635, 0.2996130585670471, -0.2492833137512207, 0.8801514506340027, 0.004762361291795969, -0.8798431754112244, -1.1238080263137817, -0.9264653325080872, 1.170718789100647, 1.5911920070648193, 0.15216852724552155, -1.600921869277954, -1.3596771955490112, -0.055329278111457825, 1.4667073488235474, -1.6584069728851318, 0.6022246479988098, 0.12620298564434052, 2.588472843170166, 0.496472030878067, -1.7753782272338867, -1.0546730756759644, 0.8154352903366089, 0.5154814124107361, -0.8070752620697021, 0.9365688562393188, 0.1403196156024933, 0.7991434335708618, 0.13018612563610077, 0.5659862756729126, 0.8249983191490173, 1.0176706314086914, -0.15163621306419373, 2.750659227371216, -1.4861054420471191, 0.8985216617584229, 0.00016058157780207694, 0.025141535326838493, -0.7635264992713928, 1.6140050888061523, -0.47375941276550293, 0.968548595905304, -1.3837199211120605, -0.058009061962366104, 0.6896144151687622, -1.1216456890106201, 0.6096636652946472, 1.7007787227630615, -1.9618158340454102, -1.1672425270080566, -0.1638408601284027, 0.4681715667247772, -1.4125975370407104], [-0.7915250658988953, -2.0095174312591553, 0.7103850841522217, -0.9756954312324524, 0.29801735281944275, -1.4291465282440186, 1.4877957105636597, -0.42994534969329834, -0.9247596263885498, 0.5093398094177246, 0.6087518334388733, 1.324663519859314, -0.3556765019893646, 1.477041244506836, -0.48754096031188965, -0.5404707789421082, -1.432969331741333, -1.015769362449646, -1.6684484481811523, -1.645949125289917, -0.17200906574726105, 1.0433480739593506, -0.6457549929618835, 1.0186887979507446, 0.8329452276229858, 1.82656729221344, 0.6938459873199463, 0.13093432784080505, -0.15040616691112518, 0.6396892070770264, -0.2127465307712555, -0.45164424180984497, 0.0232897587120533, -1.2238972187042236, -1.3391656875610352, -0.04957522824406624, 0.2574997544288635, 1.6625967025756836, 0.695094883441925, -0.3990224599838257, 0.6035330295562744, -0.38481253385543823, 0.9172912240028381, -1.4272178411483765, 1.0395176410675049, -0.23011566698551178, -2.339773416519165, -0.37482306361198425, 0.1374194473028183, -1.3763912916183472, -0.4438447058200836, 1.0802295207977295, 0.9896339178085327, -0.025822415947914124, 2.2660434246063232, 0.5521202683448792, 0.07489395141601562, 1.913260579109192, 0.5068822503089905, 1.3498841524124146, -0.8054533004760742, -0.09523577988147736, 0.4225580394268036, -0.49929356575012207, 0.20222744345664978, 1.1820123195648193, 1.2092036008834839, 0.07797478884458542, 2.7672317028045654, 0.1793200671672821, -1.7622036933898926, -0.5718463659286499, 0.15682245790958405, -0.9825018644332886, -0.8260040879249573, -0.8408406972885132, -0.06219824403524399, -1.9354721307754517, 0.20432575047016144, 0.173646941781044, -0.8245469927787781, -0.28410235047340393, -0.6928467154502869, -1.6857507228851318, 0.6959257125854492, -0.6390137672424316, 0.40077564120292664, -0.2411399483680725, -0.20408332347869873, 0.27730220556259155, -0.5361298322677612, -0.5572994351387024, -0.1672421246767044, 0.3543950021266937, -0.8617879748344421, 0.477084219455719, 0.28798389434814453, 0.5510630011558533, -0.08948147296905518, 0.44579294323921204, 0.3136175870895386, 0.9987030029296875, 2.3533968925476074, 1.3386362791061401, 0.4032301604747772, 0.4571707248687744, 0.6636412739753723, 0.8600956797599792, 0.5370272994041443, -0.4920728802680969, 1.2912547588348389, -0.5652713179588318, 0.1304892748594284, -2.3175477981567383, 0.8152157664299011, -0.6031823754310608, 0.06627125293016434, -0.10551100969314575, 0.209551602602005, -0.21099059283733368, 1.124311923980713, 1.6352850198745728, -1.508965015411377, -1.4433318376541138, -0.22262240946292877, 1.0917855501174927, 0.9835261702537537, 0.3823811113834381, 0.8016808032989502], [0.16362783312797546, 0.5892923474311829, -1.563684105873108, -1.6578599214553833, 0.04026345908641815, -0.019069895148277283, 1.766893982887268, 1.5380522012710571, -0.42555952072143555, -1.311774492263794, 0.8198143243789673, -0.27683424949645996, 0.45006659626960754, 0.12279373407363892, 0.27920466661453247, 0.35228875279426575, 0.24540361762046814, -0.7936218976974487, 0.09095698595046997, 0.5830639600753784, 0.25601840019226074, 0.2881692945957184, 1.6731162071228027, -0.6348820328712463, 0.3801405429840088, -0.8815636038780212, 0.10126396268606186, 1.7266225814819336, -0.8060517907142639, 1.033276081085205, 0.4348944127559662, 0.15047802031040192, 0.7284645438194275, -0.1062958613038063, 0.7191751599311829, -0.07219560444355011, -2.0430004596710205, -0.9159256219863892, 0.2306438386440277, 0.09072251617908478, -1.097442626953125, -1.1773884296417236, 0.07377104461193085, -0.5627134442329407, -1.5224459171295166, 0.19484680891036987, 0.5034576058387756, 2.6079695224761963, -0.466583251953125, -1.4776736497879028, 0.8657993674278259, 0.4624165892601013, -1.7187831401824951, -0.46057912707328796, 1.248187780380249, -0.5223813056945801, 1.3346047401428223, 0.7731084823608398, 1.8257759809494019, 1.0924466848373413, 0.346342533826828, -0.2923644781112671, -0.20740312337875366, 1.0956448316574097, 0.4555233120918274, -0.8791144490242004, -0.6743967533111572, -1.9371063709259033, 0.306442528963089, 0.8233665227890015, -0.1208699494600296, 0.06531873345375061, 0.5832768678665161, -1.5358607769012451, -0.5724644064903259, 0.5431138873100281, 0.3424375057220459, -0.4772055149078369, -0.9634964466094971, -0.037308089435100555, -0.4162507951259613, -0.8895332217216492, -0.4288047254085541, -0.37399065494537354, -0.729526162147522, -0.27459871768951416, -0.7010944485664368, -0.627838134765625, 0.8562508225440979, -0.214819997549057, -0.6961075663566589, -1.3298181295394897, 0.10092686861753464, 0.815816342830658, -0.9563029408454895, -0.32160380482673645, 0.5900401473045349, 0.5482795834541321, -0.5209495425224304, -1.5232068300247192, -1.7790207862854004, 0.5476048588752747, -0.4928554892539978, 0.16958393156528473, 1.5632197856903076, 1.566256046295166, -0.5610514283180237, -0.6283817887306213, 0.7937073707580566, 1.1396057605743408, -0.8890631794929504, -0.04718857631087303, 1.0946071147918701, 1.0697391033172607, 0.6516625881195068, 0.8736416697502136, -0.6932874917984009, 0.8405865430831909, 1.5320520401000977, 0.9493924379348755, 0.32520678639411926, -1.5106523036956787, 0.5403228998184204, 0.24336117506027222, -0.10551289469003677, -0.1685355305671692, 2.1977953910827637, -2.049964666366577, -0.21163532137870789], [1.4444652795791626, -0.4739629328250885, -0.429007351398468, 2.043936252593994, -0.2047446370124817, 1.2385910749435425, -0.5134220123291016, 0.43846216797828674, 0.7640644311904907, -1.1031898260116577, 0.7902467846870422, 0.30638352036476135, 1.066014289855957, 0.11400223523378372, 0.20100471377372742, 1.2084219455718994, 0.3088187575340271, -0.9591802954673767, -0.8735172152519226, -1.0550954341888428, 0.5955957174301147, 1.5170093774795532, 0.12148544192314148, 0.36458471417427063, -0.64412921667099, -1.8757528066635132, 1.5209499597549438, 1.615325689315796, -1.2910404205322266, 0.08366983383893967, 0.5973527431488037, -0.33871904015541077, -0.10239152610301971, 0.3897593915462494, -0.4794500768184662, 0.6229235529899597, -0.18457555770874023, 0.9009553790092468, -1.975593090057373, -0.5353918671607971, 1.025732159614563, 0.04373987019062042, -0.26549121737480164, -1.335382342338562, 1.2033711671829224, 0.8113714456558228, 0.6834104657173157, -2.092724561691284, 0.24895375967025757, -1.2696226835250854, -0.20763680338859558, 0.9511311054229736, -0.5839378237724304, -1.1000314950942993, 0.41546863317489624, -1.7871935367584229, -0.0044119274243712425, 0.3244117796421051, 0.22942224144935608, 0.44448527693748474, -1.170246958732605, -0.8696664571762085, 0.9870551824569702, 0.7933963537216187, -0.2523045241832733, -0.45970582962036133, 0.6821111440658569, -0.5642683506011963, 0.38179776072502136, -1.280430555343628, -0.8369239568710327, 1.1520217657089233, 0.47861140966415405, -0.5742380023002625, -0.8614830374717712, 0.4628504812717438, -0.5469251275062561, -0.8592374324798584, -0.284079909324646, -1.4980714321136475, 0.3060491979122162, 0.19926545023918152, 1.1441564559936523, -0.07711296528577805, -1.9147658348083496, -0.30426788330078125, -1.2115440368652344, -0.482957124710083, 0.4913884699344635, -0.7688484787940979, -1.5640629529953003, -0.14588497579097748, -0.16314220428466797, 0.2586025297641754, 0.1642833650112152, -0.3515359163284302, -0.4729629456996918, -0.2732924818992615, -0.12184210866689682, 1.524849534034729, 0.6182081699371338, -0.7001476287841797, -0.46381235122680664, -0.9567686319351196, -0.941798210144043, -1.2536991834640503, 0.8332827687263489, -0.016601163893938065, -2.531036376953125, 0.6808294653892517, -1.2654327154159546, 0.5020086765289307, 1.174286127090454, -0.4072849452495575, -0.8915256857872009, -0.05545264109969139, 0.8988516926765442, -0.05056561902165413, 1.743350863456726, -1.392147421836853, -1.0918110609054565, 0.10454808920621872, -0.7917678356170654, -1.1646113395690918, 0.47030580043792725, -0.11377047002315521, -0.3836197257041931, 0.7094312310218811, 0.37312576174736023], [-0.16101765632629395, -0.9591503143310547, -0.2783590853214264, -0.34211769700050354, 0.9437093138694763, 0.4388527572154999, 1.1014440059661865, 0.4041917026042938, 0.5447516441345215, 1.534129023551941, 1.808341145515442, -2.2326197624206543, -1.7434093952178955, 1.3977526426315308, 0.26351335644721985, 1.864525318145752, 0.30168360471725464, 0.7391488552093506, 1.0986591577529907, 0.5179809927940369, 1.059126377105713, 0.05123402550816536, -0.006015122402459383, 0.7492715120315552, 0.18515515327453613, -0.4932039976119995, -0.8993231058120728, 0.763008713722229, 2.1448535919189453, 1.1104415655136108, 0.8992915749549866, -0.6947358250617981, -1.1384174823760986, 0.6479111313819885, 0.7158501744270325, -1.1918107271194458, -0.8351474404335022, -0.37947624921798706, -0.6551890969276428, 1.6616394519805908, 0.5066813230514526, 0.13051635026931763, -1.44867742061615, 1.5755703449249268, -0.28175845742225647, -0.720068633556366, -1.161837100982666, 0.9750630855560303, -0.2831689715385437, 0.12327954918146133, -0.8137116432189941, -0.2148042470216751, 0.39359357953071594, 0.30600154399871826, -1.057709813117981, 0.762313187122345, -0.5912198424339294, -1.1382110118865967, 0.6866050958633423, 0.3583376407623291, 1.8838427066802979, 0.5524652600288391, -0.43637406826019287, 0.8426041603088379, -0.7841304540634155, -0.1419460028409958, -0.7079370021820068, -0.29715409874916077, 1.0025110244750977, -0.7898154258728027, -0.058163441717624664, 1.3920531272888184, 0.04134052246809006, -0.6313126087188721, -0.15954355895519257, -0.9988166689872742, 0.12358070909976959, -1.6478184461593628, 0.23420487344264984, 0.3111851215362549, -0.5122223496437073, -0.77987140417099, -0.4948936104774475, 0.21463872492313385, -1.2305717468261719, 0.6087364554405212, 0.3096882104873657, 0.5812689661979675, 0.30515000224113464, 0.609296441078186, 0.4450588822364807, -0.6687511205673218, -0.9458813667297363, -1.445487380027771, 0.20025435090065002, -0.25792327523231506, -0.23976925015449524, -0.9541367292404175, -1.0362712144851685, 1.1482490301132202, -0.34203770756721497, -0.8034796714782715, 0.8667973279953003, -0.8531026840209961, -0.5701481699943542, -1.0390987396240234, -1.8037278652191162, 1.6094962358474731, 1.4796308279037476, -0.17836293578147888, -0.310528427362442, -0.019384261220693588, -0.18862546980381012, 1.007836103439331, -2.623664617538452, 0.2547628879547119, 0.24184457957744598, -0.5187360048294067, -3.0908737182617188, -2.066441297531128, 0.5214141607284546, -1.3653829097747803, 1.4882938861846924, 1.7441723346710205, -0.7656140923500061, 0.03883623331785202, -1.0130151510238647, -0.27290642261505127, -1.441893219947815], [0.05270984396338463, -0.9057878255844116, -1.9021199941635132, -0.28093844652175903, 1.4308851957321167, 0.8278784155845642, -0.7158030271530151, -0.14896516501903534, 0.7846697568893433, 0.3325900733470917, -0.026194648817181587, -0.06169721111655235, 0.5781630873680115, 0.1965368539094925, -0.5491927266120911, 1.259843349456787, -1.073050856590271, 0.08470180630683899, -0.2723783254623413, 1.1412445306777954, 1.1760739088058472, 1.8510335683822632, 0.6422930955886841, -0.161297008395195, -2.7479379177093506, -0.36192217469215393, 0.37856948375701904, -1.2702151536941528, -1.8848637342453003, 0.5950563549995422, 0.720513105392456, -1.370576024055481, 0.9910932779312134, -1.2108756303787231, -0.0481051430106163, -0.30289939045906067, 0.44180387258529663, 0.5597632527351379, 0.18471628427505493, -1.2074875831604004, -0.6930239796638489, -0.873751163482666, 0.6316980719566345, 1.2052220106124878, 1.4205678701400757, 0.6134393215179443, -0.1482229232788086, 1.1038891077041626, -2.006484031677246, -0.5596016645431519, -0.22769179940223694, -1.8043681383132935, -0.020922280848026276, -0.4644666612148285, 0.5042363405227661, -0.015657935291528702, -1.6296906471252441, 0.25318643450737, -0.8436301946640015, -0.42030349373817444, -1.2205102443695068, -0.006064925342798233, -0.6994168162345886, -1.444257140159607, -0.6716792583465576, -0.8643935918807983, 0.28967350721359253, 0.16091351211071014, 1.030548095703125, -2.1327476501464844, -1.1149345636367798, 0.5253978371620178, -0.877197265625, 0.046342771500349045, 0.1317569613456726, 0.08486998081207275, 0.5688846707344055, 0.30279675126075745, -2.574932098388672, 0.6859785914421082, -1.3811466693878174, 0.12273117899894714, 0.41284841299057007, 0.015848202630877495, -0.2839784622192383, -1.410197138786316, 1.741545557975769, -0.18866701424121857, -0.30303671956062317, -0.03448309749364853, 1.1077030897140503, 1.3950777053833008, -1.3400373458862305, -0.4218917489051819, 2.1835432052612305, 0.298954576253891, -0.4847554564476013, -0.5793541669845581, 0.7787415385246277, 0.2559678852558136, -0.3193829655647278, -0.8240723609924316, -0.5806556344032288, 0.8002396821975708, 0.6669257879257202, 0.26335978507995605, -1.2217674255371094, 1.2077345848083496, -0.20399555563926697, -0.054695021361112595, 0.8865405321121216, 0.1851092278957367, 2.0868241786956787, 0.4870871901512146, 0.005486214999109507, -0.27509692311286926, 0.6801033616065979, 0.431092232465744, -0.31382665038108826, 0.4586109220981598, -2.0989277362823486, 1.0381004810333252, 1.6201088428497314, 2.252619981765747, 1.5328129529953003, -0.639611005783081, 1.049902319908142, 0.03440352901816368, -0.011972676031291485], [-0.09160315990447998, -0.02641390450298786, -0.29336223006248474, -1.0172909498214722, -0.8728454113006592, -0.7496278285980225, -0.20266243815422058, 0.007570404093712568, 1.1685619354248047, -0.09353257715702057, -1.161676049232483, -0.7360880374908447, -0.32639437913894653, -0.137652188539505, 1.5836181640625, -1.0010137557983398, 0.7285125851631165, 0.731101930141449, -0.3553120195865631, -1.5902589559555054, 0.5107611417770386, 0.6565040946006775, -0.9053970575332642, -0.4185870289802551, -0.8189364075660706, 1.4752917289733887, -0.4888245463371277, -0.6292035579681396, 1.1727956533432007, -0.8854550719261169, -2.2999322414398193, 0.3156591057777405, -0.1665714830160141, 1.1756205558776855, 1.01071298122406, 1.3977375030517578, -0.768723726272583, 0.18721631169319153, -0.27755239605903625, 1.1780221462249756, 0.06788965314626694, 0.4100538194179535, 0.7900305390357971, 1.1856416463851929, 2.0739214420318604, 0.6560336351394653, -0.37253838777542114, 0.313401997089386, 0.18217141926288605, -0.9463661909103394, 0.7235215902328491, -1.022956371307373, -1.6577273607254028, -0.23799099028110504, -2.3161537647247314, 1.6068644523620605, 0.8345121145248413, 0.7745622396469116, -0.4503386318683624, 0.04207777976989746, -1.2677078247070312, -0.1633329838514328, -0.7923353910446167, -0.7035039067268372, 1.3091990947723389, 0.7200970649719238, -0.6374142169952393, 2.2951183319091797, -1.9347914457321167, 1.7358146905899048, -0.44346603751182556, -1.6305075883865356, -0.4502358138561249, 0.602358341217041, 1.1905946731567383, 0.05732845142483711, -0.5322251915931702, 1.7797257900238037, -0.7585711479187012, -0.43337365984916687, 0.23646342754364014, 0.26775819063186646, -0.986047625541687, 1.4918456077575684, 0.5125306844711304, -0.046430960297584534, -0.7553852796554565, -1.0749433040618896, 0.13989409804344177, 1.8155983686447144, -0.3355991244316101, 1.7471387386322021, 0.8815993070602417, -0.7403600811958313, 0.22807593643665314, 1.0034773349761963, -0.16783519089221954, -0.4658934772014618, 0.9156695008277893, -0.4736388921737671, -1.1505001783370972, -1.2783135175704956, 1.8732844591140747, 1.4807871580123901, 0.1400509923696518, 0.0314006544649601, -0.4517614543437958, -2.2185444831848145, -3.01294207572937, -0.1317249834537506, 0.20220987498760223, -0.7215763926506042, -1.5870490074157715, 0.09047678112983704, -1.0355616807937622, 0.0656602531671524, -1.2503198385238647, -1.0903851985931396, 0.09132631123065948, 0.12462583929300308, 0.15750116109848022, 0.02904338575899601, -0.6354572772979736, -0.344939261674881, -1.1293648481369019, 0.21288982033729553, 1.5111644268035889, 0.2741084098815918, -0.6724392175674438], [-1.267743706703186, 0.47901028394699097, 1.5976481437683105, -0.046025410294532776, 0.09430892020463943, -0.05350582301616669, -1.0922013521194458, 0.5331526398658752, -2.5938100814819336, -0.42497217655181885, -0.048549093306064606, -1.5327032804489136, -1.748592495918274, -0.35787972807884216, 1.0767804384231567, 0.15713104605674744, 0.09222745895385742, 2.5546722412109375, -0.39178773760795593, 0.6595900654792786, 0.2243921458721161, 1.1818100214004517, -0.9800791144371033, -1.1723564863204956, 0.44804760813713074, -0.7547260522842407, 0.42808088660240173, -0.07006852328777313, 1.0530623197555542, -0.32617881894111633, -1.0736240148544312, -1.7317992448806763, 0.22552578151226044, -2.0942234992980957, 0.9265079498291016, -1.3805327415466309, -0.6287225484848022, -0.05717951059341431, -0.29124704003334045, 1.0716530084609985, -0.6502063274383545, 1.006774663925171, -0.3710634708404541, 0.3523585796356201, 0.7908596396446228, -0.5478803515434265, -0.5089295506477356, -0.6407663226127625, 1.6992555856704712, 1.3711297512054443, -0.23723940551280975, 0.5202592611312866, -0.28358298540115356, 2.214524030685425, -1.1599538326263428, -0.9792754054069519, -0.9506280422210693, 0.9287295341491699, -0.5401473641395569, 1.5366599559783936, 0.7585563659667969, 0.765192449092865, -1.8911198377609253, -0.9674468636512756, 1.1832730770111084, 1.7323755025863647, -0.10684019327163696, 0.9196873307228088, -0.6424628496170044, 0.8289053440093994, -0.9765757322311401, -0.4623450040817261, -1.0376272201538086, 1.5363521575927734, -0.22165487706661224, -1.681525468826294, -2.0604183673858643, 1.0033519268035889, -0.36733824014663696, -0.9696717858314514, -0.03393556550145149, 0.23558400571346283, 0.02321617491543293, 0.4388481676578522, 1.6089503765106201, -0.4924243092536926, 0.7306508421897888, -0.8530875444412231, 0.854094922542572, -1.1414693593978882, -1.5410619974136353, -0.144075408577919, -0.4177747666835785, -0.30141380429267883, -0.6147212982177734, 0.1582341194152832, -1.3527638912200928, 1.0622707605361938, 0.2655864357948303, 1.4848644733428955, -0.737260103225708, 0.306664377450943, 1.12923264503479, -0.2523273229598999, 1.1506747007369995, 1.470665693283081, 1.824296236038208, -1.5145093202590942, -0.1458434909582138, -0.7288767695426941, 1.5343354940414429, -0.5715506076812744, 0.9166170358657837, 0.8531360626220703, 1.0113548040390015, 0.7092071175575256, -0.35998523235321045, 0.6739071011543274, 0.03912072256207466, -0.7620700597763062, 1.742496132850647, 1.2365230321884155, 0.09825011342763901, 0.16095682978630066, 0.9784925580024719, -2.0045642852783203, 0.22150567173957825, -0.23508380353450775, 1.1552708148956299], [0.32655447721481323, 1.9942246675491333, -0.74750816822052, -0.1813131421804428, 0.9614636898040771, 1.9003214836120605, -0.931131899356842, -0.21845154464244843, -1.4525907039642334, -1.0007610321044922, 0.8055108785629272, 1.1815216541290283, -0.2169918268918991, -1.305553913116455, 0.22311140596866608, -1.119593858718872, 0.4234446883201599, 0.4421575665473938, 0.7550405263900757, 1.2363265752792358, 1.6013833284378052, -1.1592599153518677, -0.5802470445632935, -0.42839863896369934, -0.4824855923652649, -1.8629788160324097, -0.03984140604734421, 1.3215235471725464, 0.04912479594349861, 0.23577216267585754, 0.16652977466583252, 0.8591116070747375, -0.628075897693634, -0.31429994106292725, -0.03736420348286629, -0.7193482518196106, -0.37235796451568604, 0.06318626552820206, 0.09410841017961502, 1.4922456741333008, 0.6596398949623108, 0.24325701594352722, -0.6139737367630005, 1.3430345058441162, 0.8741334080696106, 0.6911394000053406, -2.108874797821045, -0.2622644603252411, -0.8399468660354614, 1.0362985134124756, -0.5429021716117859, -0.6240472793579102, -0.5292083024978638, 2.676812171936035, 0.5186809301376343, -0.6284509301185608, -0.9383818507194519, 1.5516613721847534, -0.43055662512779236, -0.0345868356525898, -0.5025750398635864, 2.4527812004089355, 0.416524738073349, 0.5741568803787231, -0.11308792233467102, -1.22285795211792, 0.24347251653671265, 0.3168427050113678, -0.05917872488498688, 0.21284595131874084, -0.1464659869670868, -1.6495617628097534, -1.5951306819915771, 0.010489481501281261, -0.5850309729576111, -0.412569522857666, -0.4968028664588928, -0.2769958972930908, 1.9325565099716187, -1.1619839668273926, -0.25028446316719055, 1.0641504526138306, 2.72322940826416, -0.08587104082107544, 0.5872906446456909, -0.6150094270706177, -0.6398616433143616, 0.388586163520813, -0.7848191857337952, 0.8851293921470642, -0.16343654692173004, -0.9800827503204346, 0.07782281190156937, 0.8647306561470032, -0.3047906160354614, -0.1730877161026001, -0.510057806968689, 0.36314618587493896, 0.16035406291484833, 0.827095627784729, -1.7177611589431763, -1.5697510242462158, -0.3267281651496887, 0.37434303760528564, 0.23780608177185059, -1.5549498796463013, 0.4875214993953705, -0.24602161347866058, 0.970682680606842, -0.4861427843570709, 0.3567904531955719, -0.20713011920452118, -1.3181233406066895, 0.47356751561164856, -0.33508798480033875, -1.1624294519424438, 2.1497480869293213, 1.008597493171692, 0.3345148265361786, 0.8269380927085876, 0.26765549182891846, 0.5169912576675415, -0.7327116131782532, 1.2460652589797974, -1.2686783075332642, -1.128807783126831, 0.7219661474227905, -0.10340981930494308, 0.46552300453186035], [0.04366084188222885, -0.16787059605121613, 1.0090227127075195, -1.2648388147354126, 0.3045453727245331, -0.30827322602272034, 1.3529126644134521, 0.1094396784901619, 0.5973683595657349, -0.33613651990890503, 0.7363181710243225, 0.3172914981842041, -1.8720266819000244, 0.743316650390625, -0.5092736482620239, -1.0913912057876587, 1.5567344427108765, 1.5415669679641724, 0.3694649934768677, 0.9748710989952087, -0.28742510080337524, 1.2535502910614014, 0.19547279179096222, -1.6675196886062622, -1.6620060205459595, -0.18987087905406952, -1.4554282426834106, -1.128130555152893, 1.5370596647262573, -0.33052900433540344, 0.566165566444397, 0.06454836577177048, 0.5923842191696167, 0.04515442997217178, -2.5300393104553223, 1.172745704650879, -1.1124807596206665, -0.41323983669281006, -1.1561319828033447, 1.889059066772461, -2.9769299030303955, 1.371649146080017, 0.6673197746276855, -0.4016701877117157, -0.41594523191452026, 1.432234525680542, 2.3055813312530518, -0.6412191987037659, 2.384216785430908, -0.08653295040130615, -0.646641194820404, 1.9629902839660645, 0.4513404071331024, -0.40423327684402466, 0.7229015231132507, -0.5742050409317017, 1.4315621852874756, 0.19302619993686676, -0.3853929042816162, 1.5462239980697632, 0.68232262134552, -1.0487476587295532, -1.1127407550811768, -0.4267989695072174, 0.4394879639148712, 1.0900330543518066, -0.9599704742431641, 0.8644773960113525, -0.8818691372871399, 1.5737733840942383, -1.1130315065383911, 0.09362193197011948, 2.364539623260498, -1.8232299089431763, -1.4846826791763306, 0.8288204669952393, -0.11812406033277512, 0.57257479429245, 0.12037643790245056, -1.064711570739746, -0.4307539463043213, -0.06249189004302025, 0.059062592685222626, 0.8855121731758118, -0.001908626756630838, -1.066550612449646, -0.4884459376335144, -0.46245819330215454, -0.49176377058029175, 0.20975428819656372, -0.6616619229316711, 1.739251732826233, 0.2764657139778137, -0.9887328743934631, -0.6073716878890991, 0.701349139213562, 0.07246001809835434, -1.6520813703536987, -1.9613122940063477, 0.38344210386276245, -0.8720821738243103, -1.0867030620574951, -0.2811082899570465, -0.8991952538490295, 1.208235740661621, -0.37970805168151855, 0.3759572505950928, 0.6950081586837769, -0.4089229106903076, -1.65616774559021, -0.1630106270313263, -1.1567553281784058, 0.08760784566402435, 0.31494012475013733, -0.04982417821884155, 1.0644171237945557, -0.02916560508310795, 0.5805395245552063, -1.4905849695205688, -1.1334233283996582, 2.3558335304260254, 0.7130371332168579, -0.9739477038383484, 1.0359309911727905, -0.3826614022254944, 1.6701128482818604, -1.285728096961975, -0.8260677456855774, 0.7209993004798889], [0.5716715455055237, 1.6115374565124512, 0.8855198621749878, -0.46786367893218994, 0.08641367405653, 1.3137282133102417, 0.16196253895759583, -0.6969941854476929, -0.4344523251056671, 0.4070097506046295, 1.3805785179138184, -1.0153812170028687, 1.6812102794647217, 1.6771810054779053, -1.3310474157333374, 1.281875491142273, -0.228814497590065, -2.2778384685516357, -1.0930458307266235, 0.19817368686199188, -1.1944975852966309, 1.170332908630371, 0.33067309856414795, -0.0870436578989029, -1.4014025926589966, 0.5395044684410095, -0.43045756220817566, -0.388746052980423, -0.05197534337639809, 1.027353286743164, -0.022908000275492668, -0.15758174657821655, -0.20333673059940338, -1.2050169706344604, 0.7987976670265198, -1.5582082271575928, -0.5258148908615112, 1.1990461349487305, 0.5127248167991638, 0.02820007875561714, -0.9120117425918579, -1.1751916408538818, -1.144684910774231, -0.10062786936759949, -1.83018958568573, 1.9550127983093262, -0.19927525520324707, -1.5840258598327637, 1.7271952629089355, -1.7478525638580322, 0.843171238899231, 1.6200578212738037, -0.789029598236084, -0.8659589290618896, -0.6174516081809998, -0.1414540410041809, -0.3349616825580597, 0.3138301968574524, -1.3213481903076172, -1.3558257818222046, -1.10274076461792, 1.3094651699066162, -0.6603732109069824, -0.6803109645843506, 1.0358150005340576, -0.2546343505382538, -0.4226529598236084, 0.33096760511398315, -0.09130795300006866, -0.7246098518371582, -0.2540428340435028, -1.3821648359298706, 2.1694164276123047, -0.4921540319919586, -0.5836857557296753, -2.3808488845825195, 0.191087543964386, 0.5030202865600586, 1.1620806455612183, 0.2169046550989151, -0.3604101538658142, 0.286122590303421, 0.6638453602790833, 1.0572291612625122, -0.6920652985572815, 0.18581001460552216, -0.5439477562904358, 0.6422935724258423, -0.5202441811561584, 0.04707708954811096, 1.1071921586990356, 0.21814733743667603, 1.5472700595855713, -0.3490374684333801, 0.7349749803543091, 1.1667823791503906, -0.4254071116447449, -0.40533092617988586, 1.1951969861984253, 1.039515495300293, -1.5802083015441895, -0.294354110956192, 0.8816443681716919, 1.065981388092041, 1.2549257278442383, 0.9190014004707336, -0.756442666053772, 0.8031924366950989, 1.0445245504379272, -1.5921505689620972, -0.45798295736312866, 0.051208607852458954, -0.6961139440536499, 0.39973095059394836, 1.5767704248428345, 0.7752203345298767, 0.8062241077423096, 0.39146214723587036, 1.797130823135376, 0.08217160403728485, 0.7133626341819763, 1.1525804996490479, -0.4723868668079376, -0.2780863642692566, 0.31834933161735535, 0.6673014760017395, -1.0749341249465942, -1.2270102500915527, 0.7260611057281494], [0.6704457402229309, 0.7965896129608154, -0.770643413066864, 0.44498249888420105, -0.9675679802894592, -0.5423160791397095, -0.6642957329750061, -0.7165587544441223, -2.7576558589935303, -1.241502285003662, 0.09658642113208771, 0.49789026379585266, 0.7042917013168335, -0.03810815140604973, -1.260129690170288, 0.33756253123283386, 0.7848760485649109, 1.5458667278289795, 0.3310655951499939, -0.3534662425518036, -0.5700988173484802, -0.7955714464187622, -0.6189670562744141, -0.3479272723197937, 0.43186137080192566, 0.8591586351394653, -1.603567123413086, -0.6240260601043701, -0.5880801677703857, -1.7277425527572632, 0.48061567544937134, 0.44917547702789307, -0.3172122538089752, 0.18762359023094177, -0.24318155646324158, -0.273612380027771, 1.0405206680297852, 0.6342014074325562, -0.03879834711551666, -0.4857446551322937, 0.6175280213356018, -0.1881844848394394, -1.0472369194030762, -1.2447963953018188, -1.6182669401168823, -1.0002273321151733, -0.3674705922603607, -0.7808390855789185, -0.29803794622421265, 1.2755119800567627, -0.48166778683662415, -0.9885533452033997, -0.46285751461982727, 0.3743025064468384, -0.2643257677555084, 0.20118600130081177, 0.9792584776878357, 0.31745535135269165, -0.5183410048484802, -0.6663176417350769, 0.2150845229625702, 1.6876769065856934, -0.3243491053581238, 0.7032454013824463, -1.5091718435287476, -0.7961637377738953, 0.8426432013511658, -1.1771459579467773, 0.4057978391647339, -0.6623398661613464, 2.7302045822143555, -1.7566317319869995, 0.959797203540802, 0.35837534070014954, 0.1571800708770752, 0.782008171081543, -0.4298987090587616, 1.6174050569534302, -0.9303157925605774, 1.9527621269226074, -1.1250005960464478, -0.4851517677307129, 0.36749714612960815, -0.21738390624523163, -0.36368998885154724, 0.01878085918724537, -0.8235637545585632, 0.6432792544364929, -0.7925462126731873, 0.2975994050502777, 0.8981698751449585, -0.5145794153213501, 0.6931838989257812, -0.9411779642105103, 0.29163628816604614, -0.259564071893692, -0.4757257103919983, -0.3822624981403351, -0.07869995385408401, 0.9862639904022217, -1.9862565994262695, 0.10526158660650253, 0.5174415111541748, 0.06817291676998138, 0.8088889122009277, 0.7329105734825134, 1.1174298524856567, 0.3914833962917328, 0.25270169973373413, 1.4724801778793335, 0.4717462658882141, 0.27700701355934143, 1.0848157405853271, 1.6458849906921387, 1.4439454078674316, -0.16365379095077515, 1.130602478981018, -1.1330431699752808, 0.1720307469367981, -0.8026392459869385, 1.8535698652267456, -0.8484272360801697, -0.12409641593694687, 0.4391074776649475, -0.7557663321495056, -0.12785199284553528, 0.34481310844421387, 2.8735787868499756, 1.383322834968567], [1.0595579147338867, -0.5513688921928406, -0.9053094387054443, -1.2645270824432373, 0.4733947217464447, -0.9422634840011597, 1.8169995546340942, 1.2837004661560059, -0.2027367204427719, -1.2454609870910645, 1.1678920984268188, 0.17131295800209045, 1.0842039585113525, -1.3202972412109375, -0.4591691792011261, 1.055878758430481, -1.659847617149353, -1.572301983833313, -0.8346555829048157, 1.1901119947433472, 0.09549420326948166, 1.0395079851150513, -0.9990096092224121, 0.6483890414237976, 1.1537179946899414, 0.1476733237504959, 0.8119512796401978, -1.6389544010162354, -0.5262966752052307, 0.11243611574172974, -0.0736195296049118, -0.23285184800624847, 0.4753086268901825, 0.4546911120414734, -0.36400264501571655, 0.23629745841026306, 2.3328969478607178, -0.9785005450248718, 0.5330775380134583, -1.2056655883789062, -0.9371376037597656, 0.4252980947494507, 0.21077603101730347, -0.22820326685905457, -0.1586354672908783, -1.944150447845459, -0.9158155918121338, -2.465614080429077, -0.0022500050254166126, 0.9318292140960693, 0.7128738760948181, 0.004716889467090368, 1.6977148056030273, 0.13915084302425385, 0.9833136796951294, 0.41203218698501587, -0.10152401030063629, 1.2622358798980713, -0.3956565856933594, 0.4578843116760254, 0.5459064841270447, 0.6663852334022522, -1.0496618747711182, -0.7533761858940125, -0.052400559186935425, -0.8530617952346802, -1.051021695137024, -0.1259528398513794, 0.5085874199867249, -0.5436857342720032, 0.4764828383922577, -0.16713374853134155, -1.0018858909606934, 0.8430088758468628, -0.16999760270118713, -0.04355067387223244, -1.1943720579147339, 0.8386644124984741, -1.723666787147522, -1.10196053981781, 1.6324961185455322, 0.42855390906333923, -1.4102001190185547, 2.316188335418701, -1.361033320426941, -0.013623049482703209, 0.5230115056037903, -0.1330949068069458, 1.4062576293945312, -0.3681076467037201, 2.1201369762420654, 1.3871989250183105, -0.9757794141769409, 0.5080941915512085, 1.123360276222229, -1.178954839706421, -1.6669390201568604, -1.9288967847824097, 0.40904057025909424, 0.10195273905992508, -1.1651098728179932, 0.2049998641014099, -0.2908613979816437, -0.099448062479496, -0.06933151930570602, 0.17859117686748505, -0.578671395778656, 1.6370315551757812, 1.1914047002792358, -0.6008322238922119, 0.06128118559718132, -0.499505877494812, 0.041644204407930374, 0.4995495080947876, 0.6457935571670532, -1.8270421028137207, 0.33979272842407227, -0.15736792981624603, 0.28240805864334106, 1.6514081954956055, -0.012655364349484444, 0.1597798466682434, 0.4625214636325836, -0.9269840121269226, -0.7253603935241699, -0.03766340762376785, -1.1941922903060913, 0.36760860681533813, 1.7161141633987427], [-1.4914084672927856, 2.2190918922424316, -1.6886940002441406, 0.19524680078029633, 2.002250909805298, -0.4814043641090393, -0.7413701415061951, 1.3200956583023071, -0.626854419708252, 0.15435883402824402, 0.7345340251922607, -0.2901013493537903, 0.8527316451072693, 0.2001139372587204, -0.7559700608253479, -0.8718256950378418, -1.1363850831985474, 2.4185423851013184, 0.2622969448566437, -0.4275461733341217, 0.40871644020080566, -1.605786919593811, -2.4100182056427, 0.24128401279449463, -0.34205198287963867, 0.6083024740219116, -1.969237208366394, 0.22364282608032227, -0.8305425047874451, 0.37590888142585754, -0.04785564914345741, -1.2941011190414429, -1.7753829956054688, -0.7818466424942017, -0.2976493239402771, 0.7706463932991028, 1.3556158542633057, 0.910349428653717, 0.21223361790180206, 2.032160520553589, 0.20918162167072296, 1.5818594694137573, 1.0742454528808594, -0.67270827293396, -0.978244423866272, -1.1032434701919556, -1.0030044317245483, -1.062638521194458, -0.4412570297718048, -0.3777564764022827, -0.21998454630374908, 0.16026821732521057, 1.1630446910858154, -1.8149889707565308, 0.7954919338226318, 0.7540551424026489, -0.15354903042316437, -0.5125629901885986, -0.8701397776603699, 1.156345248222351, -1.723162293434143, 1.0040849447250366, -1.066643476486206, -0.6521787047386169, 0.7042604088783264, -1.5826103687286377, -0.5343400835990906, -0.6507138609886169, 0.133505180478096, 0.30648618936538696, -1.8688586950302124, 1.7014565467834473, -1.3120917081832886, -1.5466843843460083, -0.908250093460083, 0.5469051003456116, -0.8873326182365417, 1.7520004510879517, 0.972287654876709, 1.7581101655960083, 0.5012967586517334, -2.108192205429077, -0.16675563156604767, 1.2276058197021484, -0.13627532124519348, -0.6906803250312805, 0.09118151664733887, -0.746497631072998, 0.09739069640636444, -0.45989271998405457, -0.47448545694351196, 1.542492151260376, -0.400799959897995, -0.8181443214416504, 0.07876446843147278, -0.23920203745365143, -0.3612847626209259, 0.5021969676017761, -1.30010187625885, 1.6131470203399658, -0.8181880116462708, -0.518965482711792, 0.12111090868711472, 0.07937078922986984, 0.6378609538078308, 0.9820817112922668, 1.0735961198806763, 0.4681450128555298, 3.1114697456359863, -0.8426927328109741, 0.9995133280754089, 0.2706475555896759, -0.5074455142021179, 0.47129112482070923, -0.12378066778182983, -0.11668745428323746, -1.5091549158096313, -0.8588955402374268, -1.1742441654205322, 0.015008102171123028, -1.1614350080490112, 0.43834587931632996, 0.1755165010690689, -0.1592041701078415, 0.16916733980178833, 0.2585785984992981, -0.11268189549446106, 1.3640894889831543, -0.1795053631067276], [-2.280597686767578, -0.03828377649188042, -0.5569093227386475, -0.7418959140777588, -0.20725710690021515, 1.8818986415863037, -0.6935842633247375, 0.47342514991760254, -0.8358938097953796, 0.6415505409240723, -0.8852922916412354, -1.764596700668335, 0.9972943067550659, -1.4695576429367065, 0.17621175944805145, 0.7619379162788391, -0.1483622044324875, 0.14120905101299286, -0.2540368139743805, 0.1962926834821701, 1.9446378946304321, -0.254642516374588, 0.11060608178377151, -0.4472481608390808, -0.9930338859558105, -0.5170482993125916, 0.6676107048988342, -0.15127785503864288, 1.3258576393127441, -1.808701753616333, 0.7121748328208923, -0.09455689042806625, -2.1021792888641357, -0.3337937593460083, 0.4043838381767273, 0.6696761250495911, 0.6409825682640076, -0.1750253140926361, 0.2486533522605896, 0.7136827707290649, -0.8061583638191223, -0.38960811495780945, 1.7024121284484863, 0.6033782958984375, -1.1134861707687378, 0.08423024415969849, 1.404435396194458, 1.0964515209197998, -0.40956947207450867, -0.17100219428539276, 1.6457726955413818, 1.620408296585083, -0.21404850482940674, -1.1026078462600708, 0.2900373339653015, 0.38709384202957153, -0.06341566890478134, -0.11289837956428528, 2.620340585708618, 0.22019539773464203, 0.10846647620201111, 0.9333215951919556, 0.7475919723510742, 1.0973621606826782, 0.6188109517097473, -2.662095069885254, -1.1009762287139893, 0.06851392239332199, 0.10129031538963318, -0.7649351358413696, -0.09323551505804062, -0.027907850220799446, 2.0316152572631836, -0.3585875332355499, 0.6254087686538696, -1.9365187883377075, 1.3001335859298706, -1.3709006309509277, 1.7762444019317627, 1.5553016662597656, 0.2929828464984894, 0.02780299074947834, -0.03325716778635979, 0.2515656650066376, -0.5646596550941467, 0.3184870183467865, 0.3097674548625946, 0.009696561843156815, 0.31594741344451904, 1.113247275352478, 2.634852409362793, -1.3783488273620605, 0.20848962664604187, 0.5568764209747314, 0.2899248003959656, 0.5844407081604004, -1.5767368078231812, -1.2454346418380737, 0.014191468246281147, -0.07340489327907562, 1.7280775308609009, -0.46949437260627747, -0.07748647034168243, 1.2577844858169556, -1.3094439506530762, 0.7936649322509766, -1.5325658321380615, 0.015863087028265, 0.19877028465270996, -1.0894355773925781, 1.5092253684997559, 0.37852197885513306, -0.4033450186252594, 0.129375159740448, 1.0745304822921753, -1.9399359226226807, 0.8928310871124268, -0.10991501063108444, 0.44957587122917175, 0.35295116901397705, -2.0067551136016846, 0.8064026832580566, 0.26619428396224976, 2.6747279167175293, 1.1187548637390137, 0.5360696911811829, 0.3805575966835022, 0.09275646507740021, 0.3922836482524872], [0.6859764456748962, -1.6356360912322998, -1.3283917903900146, 0.5937805771827698, -0.3694102466106415, -0.9476404786109924, -0.45930472016334534, 0.38468557596206665, -1.433212161064148, 0.7399313449859619, -0.4738653302192688, 0.43064889311790466, 0.2646055519580841, 0.8357477188110352, 0.3876359164714813, 0.4896656274795532, 0.43880873918533325, -1.0159783363342285, 0.014199459925293922, 0.31063851714134216, 1.3382428884506226, 0.12004353106021881, 1.8863680362701416, 0.32068467140197754, -0.34572166204452515, 0.16673439741134644, -1.5472484827041626, -1.3316127061843872, -0.2437572330236435, 1.3983854055404663, -1.0164328813552856, 2.9982311725616455, -0.36150243878364563, 1.6029052734375, 0.33507147431373596, 0.39092499017715454, -0.1587822437286377, 0.5276636481285095, 0.7110834717750549, -0.14426518976688385, -0.44647249579429626, -0.3110283613204956, 0.42831239104270935, 0.8263929486274719, -1.609010100364685, 2.1726086139678955, -1.1960264444351196, -0.7003853917121887, -1.177328109741211, -0.5519654154777527, 2.609959602355957, -2.3367230892181396, -0.477478563785553, -0.39419859647750854, -0.7218453884124756, 1.6288732290267944, 0.14446304738521576, -0.2554275095462799, -0.7743983268737793, 0.9282258152961731, -0.3809627294540405, 1.3000308275222778, -0.6241772770881653, 0.45194801688194275, -0.12612849473953247, 0.8483948111534119, -0.4163266122341156, 2.445342540740967, 0.3349169194698334, 0.06405913829803467, 0.6720679402351379, 1.049630880355835, -0.13135723769664764, -0.7423328757286072, 0.12588277459144592, -0.5021966099739075, 0.25809556245803833, 0.48424455523490906, -0.3073883056640625, 0.02387385629117489, 2.1397323608398438, 1.3703017234802246, -0.8457038402557373, 0.8168581128120422, 0.1412535309791565, 0.06951170414686203, -1.055024266242981, 0.23268382251262665, 0.14064092934131622, 1.6508699655532837, -1.1432245969772339, 1.7097910642623901, -2.0253355503082275, 1.7526732683181763, -0.558742105960846, -0.132359117269516, -2.402399778366089, 1.2930145263671875, 0.47297921776771545, 0.6850882768630981, 0.6087445616722107, -1.0162413120269775, -0.6477484703063965, -1.6980829238891602, -0.024795344099402428, -0.41327306628227234, 0.024925122037529945, -0.8176509737968445, -0.23283813893795013, -0.7375186085700989, -0.10123144835233688, -0.1084851399064064, -0.12296371906995773, -1.4713982343673706, -0.006595600862056017, 0.1769457459449768, 0.8622053861618042, 0.834862470626831, 0.28552544116973877, -0.4735528528690338, 0.8589565753936768, 0.022894149646162987, -2.3624424934387207, -1.0199118852615356, -0.5678367614746094, 0.13466189801692963, 0.4848407208919525, 1.0411653518676758, 0.658018946647644], [0.2206142097711563, -0.5158799290657043, 0.31723156571388245, -0.28798359632492065, -1.6407755613327026, 0.434762179851532, -0.8258621692657471, 1.6536145210266113, -1.3140361309051514, 0.07347692549228668, 1.0875067710876465, 0.6200503706932068, 0.0769088938832283, -0.9709969162940979, -0.024046912789344788, -1.3397753238677979, -0.2584874629974365, -0.28011274337768555, -0.6358147859573364, 0.07412219047546387, -0.8128689527511597, -1.4843376874923706, -1.1578844785690308, 1.0052813291549683, 0.28973421454429626, 1.4329744577407837, 0.8598473072052002, -1.7237900495529175, 1.3525879383087158, 0.38461482524871826, -1.3027610778808594, 0.22167588770389557, -1.6248441934585571, 0.9456143379211426, 1.0635147094726562, -0.5005723834037781, -0.48280736804008484, 0.6657769680023193, 0.07961080223321915, -0.12957395613193512, 0.45052722096443176, 0.38638919591903687, -1.3406139612197876, 0.4687190651893616, -1.356432557106018, -2.3704757690429688, -0.5774306058883667, 0.4238835871219635, 0.26690760254859924, -0.8002704381942749, 0.10110315680503845, -1.3717570304870605, -1.4027273654937744, 0.5177035927772522, 2.200063705444336, 0.7107305526733398, 0.8307897448539734, 0.4131225645542145, 2.461879253387451, -1.318013310432434, 2.132199764251709, 1.109653115272522, -0.48851943016052246, -0.2190571427345276, -1.9208290576934814, 0.42520058155059814, 1.0468493700027466, 0.1470881700515747, 1.0641891956329346, 0.20664651691913605, 0.16329334676265717, -0.07675210386514664, -0.3033429980278015, 1.914912462234497, -0.05589791014790535, -2.2378454208374023, -0.3280552625656128, 1.1936640739440918, 0.5656910538673401, -0.5848731994628906, -0.6377524733543396, -0.05773250013589859, -0.3398025333881378, 1.7815223932266235, 0.7330098748207092, 0.5488507151603699, 0.5025376081466675, -1.1226423978805542, 0.3270157277584076, -1.7499812841415405, 2.3211684226989746, 2.132889986038208, 1.9540382623672485, -1.0986974239349365, -1.1572681665420532, 0.691045880317688, 1.2572658061981201, -1.2630434036254883, -0.9852065443992615, -0.40322795510292053, -0.2966547906398773, 1.1895922422409058, 1.2297288179397583, -0.6027683615684509, -0.7642052173614502, 0.42232874035835266, 1.0654455423355103, -0.4842740595340729, -2.118381977081299, -0.1042473316192627, 1.2192068099975586, -1.4553165435791016, -1.0715125799179077, 0.9301067590713501, -0.45074209570884705, -0.27612245082855225, 1.0113369226455688, 0.5657174587249756, 0.41921743750572205, -1.0350466966629028, -0.11455775052309036, -2.196521520614624, -0.23365764319896698, 1.379077672958374, -0.6021904945373535, 0.7349372506141663, -0.9759260416030884, -1.3641458749771118, 1.274784803390503], [1.3775562047958374, -1.049681544303894, -1.8634443283081055, 0.13465052843093872, 0.5566733479499817, 0.39201173186302185, -0.3147616982460022, -0.6579665541648865, 0.7591637969017029, -1.4832326173782349, 0.48803478479385376, -1.6486862897872925, 0.31426185369491577, -0.7945801615715027, 0.47629472613334656, -1.7111927270889282, 0.5375776886940002, 0.09208377450704575, -0.8395015001296997, -0.889228105545044, -1.1280466318130493, -0.16709771752357483, -1.371465802192688, 0.6875517964363098, -0.8555070757865906, -0.29289665818214417, 0.042462121695280075, 0.32307934761047363, -0.6018555164337158, 0.7588670253753662, -1.398837924003601, -0.37554696202278137, -1.0859163999557495, 0.4574846029281616, -0.05221102386713028, -0.03443758934736252, 0.19597280025482178, -0.23088887333869934, -0.5851321220397949, 0.4283648133277893, -1.7361029386520386, -0.38025930523872375, 1.506205439567566, -0.8265096545219421, 1.6380752325057983, -1.2646158933639526, -0.11355459690093994, 0.6919890642166138, -1.9316037893295288, 0.3749828040599823, -0.6192420721054077, -0.35427379608154297, 2.169809341430664, -0.11643275618553162, -0.12709955871105194, -0.19392599165439606, -1.2915229797363281, -0.8169499635696411, -1.1894495487213135, 0.457356333732605, -0.9194473624229431, 0.5428904294967651, 0.2642703950405121, -0.46501174569129944, 0.5726203322410583, 1.7147233486175537, 0.35244423151016235, -0.30262041091918945, -1.0748811960220337, -2.1038930416107178, -0.4272499084472656, -2.2584002017974854, 0.5734944939613342, -1.412505865097046, -0.8730936646461487, -0.8003171682357788, 0.04743823781609535, 1.2959014177322388, -0.6931935548782349, -0.8754889369010925, 0.546383798122406, -0.9741616249084473, 0.4686066210269928, 0.17889012396335602, 0.9205098748207092, 1.993161916732788, -1.2004643678665161, -0.8746027946472168, 0.19584491848945618, -0.2382379174232483, -1.3426353931427002, 0.8843130469322205, 0.2775072455406189, -1.1595855951309204, 0.7991214990615845, -2.890702247619629, 0.0059800175949931145, 0.7497924566268921, 1.9588205814361572, 0.8608970046043396, -1.3870335817337036, -0.19204774498939514, 0.24396836757659912, 2.398686647415161, 0.6174834966659546, -0.3109460473060608, 0.886118471622467, -0.9014471173286438, -0.7900338768959045, 0.5145263075828552, -0.12794315814971924, 0.6225265264511108, -1.11067533493042, 1.0566426515579224, -2.176894187927246, 0.15204331278800964, -1.4334681034088135, -0.2618347406387329, 0.5770086646080017, -2.271267890930176, 1.1037367582321167, 0.18511426448822021, 1.2235989570617676, -1.0399818420410156, 0.5400487780570984, -0.8108500838279724, -0.8037251830101013, -1.733021855354309, -1.8366090059280396], [-0.7424052357673645, 1.0153611898422241, 0.3137340545654297, -0.5065883994102478, -0.8961093425750732, -2.2703030109405518, 0.6718885898590088, -0.43057966232299805, -1.3023568391799927, -0.5442684888839722, 0.34932318329811096, -1.4764529466629028, -0.2579145133495331, 0.1811702996492386, 1.2114408016204834, -0.16718702018260956, -1.596254825592041, 0.37634986639022827, -1.3328105211257935, -0.7586924433708191, 0.7120977640151978, 0.47970518469810486, -0.13577011227607727, -0.4785804748535156, -0.8631119132041931, -0.6253948211669922, -0.9332327842712402, 0.5521699786186218, 1.8024405241012573, -0.824410617351532, 1.1610839366912842, 1.0200997591018677, -0.2535567581653595, 0.20849189162254333, 0.27038949728012085, 0.8058378100395203, 0.49844467639923096, -0.8297175168991089, -1.8203246593475342, 0.6345095634460449, 1.011055827140808, -1.563380241394043, -0.4167690575122833, -1.3447498083114624, 0.4270894527435303, 0.37062788009643555, 0.10635362565517426, -0.10039783269166946, 0.14626795053482056, 0.5372408032417297, 0.8194078803062439, -0.223359614610672, -0.3767574727535248, -0.4154265820980072, 0.838757336139679, -1.6946513652801514, -0.3524729013442993, -0.7909308075904846, -1.3284333944320679, 1.1288691759109497, 0.6093228459358215, 0.5003388524055481, 1.0729100704193115, 1.064636468887329, -0.9901944994926453, 0.7549862265586853, -0.3839641809463501, 1.5610334873199463, 0.3391110599040985, -0.23688651621341705, 1.160914421081543, -0.7259711027145386, -0.47536781430244446, 1.679033637046814, -0.8290286660194397, -2.3749561309814453, 1.0871707201004028, 0.7227883338928223, -1.305296540260315, 1.2214702367782593, 0.2848038077354431, -0.0834457129240036, -0.8313454389572144, 0.8142379522323608, -0.06074443832039833, -1.4683314561843872, -1.6197422742843628, 1.0421642065048218, -1.4209798574447632, -0.03174568712711334, 1.415548324584961, 0.2863909900188446, -0.7537227869033813, -0.16445599496364594, 0.4562029540538788, -1.3898576498031616, 0.6813018321990967, 0.9833115339279175, 0.8300604224205017, 0.07527586072683334, -1.4731886386871338, -0.24829241633415222, 0.37337952852249146, 0.3055741488933563, 1.5505428314208984, -0.5042561292648315, 0.6072478294372559, -0.5818975567817688, -2.909804344177246, 1.8317689895629883, -0.42336055636405945, -0.7777281403541565, 0.07856568694114685, -0.6061673164367676, 1.2062726020812988, -0.3351849615573883, -0.4203459620475769, 1.0903035402297974, 0.6278899312019348, 0.5887719392776489, 1.141409158706665, 1.2614136934280396, 0.3586280643939972, -0.4698861241340637, -0.40629667043685913, 2.184968948364258, -0.8248834013938904, -0.061556924134492874, -0.9809048175811768], [-1.8594452142715454, 1.6660234928131104, 0.308073490858078, -2.1319427490234375, -1.2023547887802124, -0.3096538186073303, 0.6498215794563293, 0.7114132046699524, 1.0221222639083862, 0.09768831729888916, -1.739863395690918, 0.1882835030555725, -2.0998101234436035, 0.9948850274085999, 0.8490638732910156, 0.7349055409431458, -0.26328930258750916, -0.6220487952232361, -0.507503867149353, -0.08149483799934387, -0.8472532033920288, -0.2352491319179535, -1.6419020891189575, 0.41110163927078247, 0.3498005270957947, 0.7551933526992798, -0.30097338557243347, 0.13673868775367737, -1.0377938747406006, -0.593580961227417, -1.0666882991790771, 0.02564912848174572, 0.019836651161313057, -2.002610206604004, 1.1102031469345093, -0.9433388710021973, 0.259438693523407, 1.25676429271698, 1.6228471994400024, -0.7953783869743347, -0.4062079191207886, -0.04395267367362976, -0.4542362093925476, -1.7116773128509521, 1.8365432024002075, -1.0773122310638428, -0.7053984999656677, 0.6098453998565674, -0.17348705232143402, 1.1106981039047241, -1.905574917793274, 0.07487756013870239, -0.3190426230430603, -1.1646778583526611, 1.3209749460220337, 0.8886579275131226, 0.9464061856269836, 0.6991266012191772, 0.9561672210693359, 1.3323408365249634, 0.10183374583721161, 0.628821611404419, -0.5631968975067139, 0.6281212568283081, -1.2925602197647095, -2.1634488105773926, 0.6865056753158569, 0.23614342510700226, -0.836219310760498, 0.961754560470581, 1.8241137266159058, 2.057445764541626, 0.6009582877159119, 0.3677864074707031, -0.11116980761289597, -1.7898786067962646, 0.573884129524231, -0.6007176637649536, -2.214724540710449, -0.031781889498233795, 0.4473145604133606, 0.1278582513332367, 0.0565347857773304, -0.8216981291770935, 0.29088717699050903, -0.46056464314460754, 0.41340410709381104, -1.5502026081085205, -0.41161447763442993, 0.3544432520866394, 0.8339892029762268, 0.7895671725273132, -1.4010528326034546, -0.626349925994873, -1.5271782875061035, 1.448662281036377, 1.2500907182693481, 0.9314931631088257, 0.8052171468734741, 2.0261147022247314, 0.778701663017273, -0.9204223155975342, 0.4718996286392212, -0.5171781182289124, -0.2262553572654724, -0.0335492268204689, -0.7035310864448547, 1.1704156398773193, 0.7449080348014832, -0.6575453877449036, 0.5187481045722961, 1.3094909191131592, -0.8056625127792358, -1.160005807876587, 1.370469570159912, 1.6589528322219849, -0.15681475400924683, -1.1767853498458862, 1.1155039072036743, -0.3926861584186554, 1.508567214012146, -0.9579750895500183, 0.1379423439502716, 0.06777577847242355, 0.8704912662506104, -0.4111347198486328, 0.4801301062107086, 0.7511969208717346, 0.8593785762786865], [-0.6157179474830627, -1.3390697240829468, -0.4424298107624054, 0.43178707361221313, 0.7196685671806335, -2.198876142501831, 0.6743506789207458, 1.4542319774627686, 0.24289101362228394, 0.5377458930015564, 0.3826726973056793, -0.40198060870170593, -0.08035027235746384, 0.13308008015155792, -0.9840318560600281, 0.2537803649902344, -0.7165108919143677, 2.5888400077819824, -1.2417594194412231, 0.5575389266014099, -1.0232657194137573, -0.747913658618927, -0.022839676588773727, 1.1104267835617065, 1.660607099533081, -1.2656819820404053, 1.3405786752700806, -0.34503674507141113, -0.9320172667503357, 0.3689691424369812, -0.8798510432243347, 1.201137900352478, 1.5179072618484497, 1.9859282970428467, 0.2872728109359741, 0.5509761571884155, 0.3254143297672272, -0.2123286873102188, -0.4549729824066162, -0.493007630109787, 0.6894108653068542, -0.8731988072395325, -0.048338424414396286, -0.341448038816452, -1.1629127264022827, 0.17958442866802216, 2.664623975753784, 0.629631519317627, 0.013630855828523636, -1.1129419803619385, -0.5971365571022034, 1.6577577590942383, -2.08288836479187, -0.2715229094028473, 1.2142298221588135, 0.17619307339191437, 0.08711902052164078, -0.8914769887924194, 1.390188217163086, -0.164578378200531, 0.4055860638618469, -1.7022565603256226, 0.8908567428588867, 1.5114868879318237, 0.9310857653617859, -0.32132241129875183, 0.293844074010849, 0.25063493847846985, 0.09648144990205765, 1.44264817237854, -0.1801026612520218, -1.1959508657455444, -0.7231581211090088, 1.0881832838058472, 1.2240471839904785, -1.762284755706787, -0.4169405698776245, 0.973884642124176, -0.2579290270805359, 2.42476224899292, -0.7141335606575012, -1.4384796619415283, 0.41785386204719543, -0.38802370429039, 0.32326042652130127, -0.05853671953082085, 0.5365267992019653, -1.1361933946609497, -0.5363131165504456, -1.424880027770996, -0.3060292899608612, -1.1438158750534058, 0.8507658839225769, 2.436049699783325, 0.15467716753482819, -0.007083583157509565, -0.6235296130180359, 1.1952073574066162, 1.049456000328064, 1.272214651107788, 0.6467065811157227, -0.47513309121131897, 1.1601251363754272, -0.43067237734794617, -0.11929155886173248, 1.3981480598449707, 1.1528702974319458, 1.3353585004806519, 0.4873826205730438, 0.07400642335414886, -0.06341878324747086, -0.0973103865981102, 0.12063032388687134, 0.2809735834598541, -0.8227423429489136, -0.43038085103034973, 0.6633119583129883, 0.904864490032196, -0.29444363713264465, -0.3821706473827362, 0.19138893485069275, -0.46213024854660034, 0.09606238454580307, -1.5972554683685303, 0.43488937616348267, -1.0093576908111572, -0.19015857577323914, -1.613738775253296, -0.9193139672279358], [0.7205218076705933, -0.3785409927368164, 1.4279873371124268, -0.25713905692100525, 0.39973708987236023, 0.16243936121463776, -0.18370497226715088, -1.3254048824310303, -2.5834810733795166, 0.9878934025764465, -1.811732530593872, 0.3962774872779846, 0.3314787447452545, 0.986085832118988, 0.22678503394126892, 0.010926523245871067, -0.1992831528186798, -0.6874139308929443, 0.38314205408096313, 0.9703080058097839, 0.03836223483085632, 0.8138575553894043, -1.1932233572006226, -0.20678335428237915, -0.42275717854499817, -0.5010624527931213, 1.0788366794586182, 0.28328606486320496, 1.2813841104507446, 0.3547169268131256, 0.15217868983745575, 0.4745236337184906, -1.2863001823425293, 0.17327505350112915, -0.2676548957824707, -0.8531719446182251, 0.13630947470664978, -0.8030596971511841, 1.0379236936569214, -1.2643481492996216, 1.9718501567840576, -0.4212278127670288, 1.4202555418014526, 1.0826342105865479, 1.5510666370391846, -0.04668295755982399, 0.18319599330425262, 1.2748825550079346, 0.6836440563201904, 0.18129439651966095, 0.31125110387802124, 0.9909974336624146, 1.184836983680725, 0.5722315311431885, 0.14228150248527527, -0.5871931314468384, -1.7034718990325928, 1.4333173036575317, 0.39343637228012085, -0.8692650198936462, -0.5867803692817688, -1.0494327545166016, 0.8314911723136902, 0.5319674015045166, -0.32428818941116333, 0.9726906418800354, 0.5983718633651733, -0.1561301201581955, 0.014695318415760994, 0.07966460287570953, 0.06071673333644867, 0.4544840455055237, -0.3938736617565155, 1.3380540609359741, 0.3206923305988312, 1.8672778606414795, 0.024631677195429802, -0.24485190212726593, -0.6483426690101624, -2.260300874710083, 0.3621427118778229, 2.310924768447876, 0.06249551475048065, -0.7408434152603149, 0.2590017020702362, 1.1346099376678467, 0.6699944138526917, -1.2223255634307861, -0.6891074776649475, -1.1783397197723389, 1.2450546026229858, 0.7892829179763794, 0.6199104189872742, -0.2433360368013382, -1.8792551755905151, 1.450191617012024, 0.6964420080184937, 1.0040432214736938, -0.8408258557319641, -0.30405548214912415, 1.3215068578720093, -0.3858891427516937, 1.2657967805862427, -0.6319383978843689, 0.018916111439466476, 0.8905983567237854, 0.18623864650726318, -0.06637456268072128, -0.7913936972618103, 0.7136115431785583, 0.2781680226325989, 0.27085453271865845, 0.6888017058372498, 0.11905907839536667, -3.0745303630828857, -2.328108310699463, 0.44536077976226807, 0.4640844166278839, -0.30712834000587463, -0.19144272804260254, 0.13131991028785706, 1.5784868001937866, -1.9691963195800781, 0.05139424651861191, -0.345858633518219, -1.3917698860168457, -0.9410361647605896, -0.2948001027107239, 1.2343169450759888], [1.1315014362335205, 2.263108491897583, 1.8024158477783203, -0.16820472478866577, 1.0007961988449097, 0.1823517382144928, -0.6100382804870605, -0.32632681727409363, 1.7116649150848389, -0.5867152810096741, 1.9308652877807617, 0.15985187888145447, 0.12133803218603134, -0.2793629467487335, -2.28348970413208, -0.32771825790405273, -0.02819034457206726, -1.2528502941131592, 1.1464170217514038, 0.29536816477775574, -0.46429842710494995, 0.22366951406002045, -0.5001140236854553, -1.294908881187439, 0.005102612543851137, -0.3297017514705658, -0.8052543997764587, -2.0167691707611084, 2.2920572757720947, -0.5626713037490845, -0.6820939779281616, 1.147011160850525, -1.1071521043777466, -0.7345042824745178, 0.4903391897678375, -0.2762582302093506, -1.4295661449432373, 1.4131487607955933, -0.3187265992164612, 0.9961447715759277, 1.1658791303634644, 0.28237050771713257, -0.4793703258037567, -0.7542099952697754, -0.9855241775512695, 1.078294277191162, 0.750458300113678, 2.0520215034484863, 0.4853506088256836, 1.1843938827514648, -0.5164905786514282, 1.6129789352416992, 0.45867449045181274, -0.3318920135498047, 0.8408499360084534, -0.11641628295183182, 1.8349249362945557, -1.7744232416152954, -1.1892101764678955, 1.0373865365982056, -2.4785797595977783, 0.4975653886795044, -0.597991943359375, 0.04213330149650574, 0.3352500796318054, -2.221148729324341, 1.8681508302688599, -1.325348138809204, 2.2550737857818604, 0.41804972290992737, 1.1103770732879639, -0.8775811791419983, -0.5970245003700256, 0.013257171958684921, 0.014766017906367779, 0.5402299165725708, 0.7643624544143677, 0.2875271439552307, 1.5449169874191284, 0.10743926465511322, -1.7377101182937622, 0.5414198637008667, -1.9379956722259521, 0.8989757299423218, -1.2229151725769043, -0.12733785808086395, -0.5487610697746277, 0.2742486596107483, 0.6132894158363342, 0.4014560282230377, 2.273737907409668, 0.5111023783683777, 0.515318751335144, 0.4434962272644043, -1.8378456830978394, -1.465551495552063, -1.009286880493164, -1.2453179359436035, 1.3196643590927124, -0.6238869428634644, 0.3615182936191559, -0.12899616360664368, 0.8938256502151489, 1.2201520204544067, 0.4757601022720337, -1.7906992435455322, -0.960332453250885, 1.1060811281204224, -0.33267152309417725, 0.20064640045166016, -0.17499661445617676, 1.2326933145523071, 2.101386785507202, -0.8616140484809875, 0.21699251234531403, 1.0540837049484253, 1.3314024209976196, 0.5569623708724976, -1.23063063621521, -1.2589863538742065, 0.9014477729797363, -1.1025954484939575, -0.05807480588555336, 1.0405408143997192, 0.5489693284034729, 1.0232352018356323, 0.42870163917541504, -0.5407267808914185, 1.336896538734436], [-0.8296371102333069, 0.7495778203010559, 0.06437630206346512, -0.9240651726722717, 1.0566284656524658, 0.8728609085083008, 1.1232922077178955, -1.8288183212280273, -0.1876305192708969, 0.49113109707832336, -1.1390910148620605, -0.30133527517318726, 0.5415480732917786, 1.5782833099365234, -1.8295249938964844, -1.4906259775161743, 0.08172418922185898, 1.6939173936843872, -0.6768988370895386, 0.5676279664039612, -0.2956492006778717, -1.144999384880066, -0.04635481536388397, 0.8431594967842102, 0.7267357110977173, 1.9867565631866455, -1.4813638925552368, 0.44473469257354736, 1.7704497575759888, 1.5544476509094238, 0.3105276823043823, -0.39685383439064026, -2.678487777709961, -0.12452801316976547, -1.0405775308609009, 1.0441464185714722, 0.13537786900997162, 0.8098307251930237, 2.2133851051330566, 0.1496930718421936, -2.074533462524414, -0.0036198783200234175, -0.5569137334823608, -0.011717458255589008, -0.7167035937309265, -0.5845535397529602, -0.054106611758470535, -1.0507245063781738, 1.40091872215271, 1.5336651802062988, -0.9768730998039246, -0.1987004578113556, 0.2540731430053711, 0.06319405883550644, 1.8869907855987549, 0.5486142635345459, -0.0607440359890461, -2.0425539016723633, -0.7896507978439331, -1.1897634267807007, 1.3706214427947998, -0.7562623620033264, 0.660933256149292, 0.5815737247467041, 0.28926071524620056, -1.2180343866348267, 0.9167212247848511, 0.05750489607453346, -1.2137300968170166, -0.6790667772293091, -0.45896726846694946, 0.06533334404230118, -0.5554065704345703, -0.651221752166748, -0.025426218286156654, 0.8449317812919617, -0.9503381848335266, -0.40964624285697937, 0.3435248136520386, -0.44063684344291687, 2.092254638671875, 0.22983144223690033, -0.21718424558639526, -0.35337576270103455, 0.6933544278144836, 0.11426924914121628, 0.4229939579963684, -1.1249303817749023, 0.533674955368042, -0.23056507110595703, -0.8342134952545166, -3.0224030017852783, -1.6514766216278076, -0.026499349623918533, 1.4957226514816284, -0.5880084037780762, 0.4843951463699341, -0.31597667932510376, 0.6013088226318359, -1.2242313623428345, -1.7613451480865479, -1.473849892616272, -0.3685917854309082, -2.1033782958984375, -1.472005844116211, 1.9832789897918701, 0.8731534481048584, -0.24733629822731018, -0.7881821393966675, 0.2675682008266449, 0.7901749014854431, -0.6934666633605957, -1.9213368892669678, -0.2938421666622162, 0.19501179456710815, -0.1394580453634262, 1.018325924873352, 0.8495991826057434, 1.1280628442764282, 0.6021094918251038, 0.18163864314556122, 0.4185332655906677, -1.198055624961853, -0.10672194510698318, -1.1709589958190918, -1.0699471235275269, 0.7802553772926331, -0.46283379197120667, -0.2580181062221527], [0.5502229928970337, -1.239665150642395, 0.006701576989144087, 0.3226240873336792, 2.3584554195404053, -1.4412634372711182, -2.481273889541626, 0.6969704031944275, 0.925039529800415, -0.9764087796211243, 0.7590687274932861, 0.03515965864062309, -1.3430122137069702, 0.7751604318618774, 0.5651026368141174, -0.4498409926891327, -0.048202961683273315, 0.8393270373344421, 1.2109979391098022, -0.9403294920921326, 0.5479963421821594, 0.042801134288311005, -0.47301414608955383, 0.1433241218328476, 0.007263361942023039, 1.1463409662246704, 0.012712552212178707, 0.21747632324695587, -1.2909961938858032, -0.7157436609268188, -0.7004877328872681, -1.5570443868637085, 0.13042199611663818, 0.6949207782745361, 0.18432427942752838, -0.2933633327484131, 0.13249237835407257, -1.2323565483093262, -1.7291688919067383, -0.8524002432823181, -0.11536838114261627, -1.2180900573730469, -0.3347215950489044, -0.23700477182865143, 0.2901458144187927, -0.1591225415468216, 0.5440722107887268, -0.5210726857185364, 0.5506285429000854, -0.8426150679588318, 1.2828047275543213, 0.8466190695762634, 1.2224303483963013, 0.5329251289367676, -0.46311044692993164, 0.936166524887085, -0.16133444011211395, -1.839482307434082, -0.2813543677330017, 1.0637638568878174, -0.5538159012794495, -1.2548694610595703, 1.1320602893829346, -0.2709023058414459, -0.6270576119422913, -0.9180918335914612, 1.3556691408157349, -0.430818110704422, 2.011023998260498, -0.9352414608001709, 2.290300130844116, 0.13797208666801453, -0.048659391701221466, 0.7240861654281616, 1.559413194656372, -0.8940134048461914, -1.3922579288482666, 0.5216001868247986, -0.06493628770112991, 0.8136110901832581, -0.6036784052848816, 0.5080231428146362, -1.9696828126907349, -0.14681008458137512, -0.5662452578544617, 2.1436338424682617, 0.4354902505874634, 0.5280830264091492, 0.26434949040412903, 0.4894919991493225, 0.6124239563941956, 0.6159877181053162, -1.305585503578186, -0.5887175798416138, -0.7602764964103699, 0.5982530117034912, 2.182041883468628, 0.40356436371803284, -1.3640419244766235, -0.9377874732017517, 1.022394061088562, 0.5748649835586548, -1.232906460762024, 1.309067726135254, -0.5643961429595947, -0.3178240954875946, -0.3654806315898895, -2.0756711959838867, 1.37213933467865, -0.4055747985839844, 0.413864403963089, -1.3596409559249878, 1.0257166624069214, 2.545715093612671, -2.3037831783294678, -1.8105791807174683, -0.01751558668911457, -0.9914426803588867, 1.4244898557662964, -1.6660118103027344, 1.2744163274765015, 0.7114629745483398, -1.2624589204788208, -0.4794888198375702, 0.0023266675416380167, 1.5542793273925781, 1.077333688735962, -0.08568015694618225, 0.09890583902597427], [-0.35378023982048035, 0.7978200912475586, 2.0348119735717773, -0.17407597601413727, -1.0041344165802002, -0.5671373605728149, 0.329953134059906, -0.9850727319717407, -0.6288931369781494, 2.5092830657958984, -1.1260584592819214, 1.4212132692337036, -0.056013960391283035, 0.2235042005777359, 0.5162089467048645, 0.5791441798210144, -0.11632441729307175, -0.3351205885410309, 0.2510684132575989, 0.1339026540517807, -0.8977187275886536, 0.6162213087081909, -0.4114479422569275, -0.5909806489944458, -1.1058343648910522, -1.9990602731704712, -2.7939131259918213, -1.7558006048202515, 0.32314804196357727, 0.27075833082199097, 0.7613979578018188, -1.5000710487365723, 0.27661845088005066, 0.6483784317970276, -1.3860232830047607, -0.49275869131088257, 0.8530286550521851, -1.3235375881195068, -0.4317818880081177, -1.3635578155517578, -1.500156044960022, -0.6308984756469727, -2.5752947330474854, -0.8540583848953247, -1.1432617902755737, 0.7834799885749817, 0.20629175007343292, -2.8616561889648438, 0.6524993777275085, 0.8771111369132996, -1.6637266874313354, 0.26759523153305054, -1.514666199684143, 1.2316147089004517, 0.33549705147743225, -0.14228221774101257, -0.2608153820037842, -1.2520980834960938, -0.993762731552124, -0.03147820010781288, -1.9244434833526611, -0.8408313393592834, 0.4220626652240753, -0.6121742725372314, -0.9404640793800354, -0.8055843114852905, 0.26357752084732056, -0.2782275676727295, 0.959071159362793, 1.143559217453003, -1.2402207851409912, -0.606677234172821, -0.7358958721160889, -0.7241560220718384, 0.4565155506134033, -0.8783923387527466, -2.119051933288574, -1.0150216817855835, 2.275648355484009, 1.0633022785186768, -1.8499270677566528, 1.9317514896392822, 1.1215808391571045, -0.7580822110176086, -0.32169029116630554, 1.9668114185333252, -0.17559915781021118, 0.1851235330104828, 0.004267041105777025, 0.2323496788740158, -2.011096715927124, 0.7782054543495178, 0.4882737398147583, -1.2815442085266113, 0.35367223620414734, 1.8415348529815674, -0.9905837774276733, -0.1561371237039566, 0.4834812581539154, 0.2860916256904602, -2.486480712890625, 0.908255934715271, 1.8988360166549683, -1.2192062139511108, 0.5867913365364075, 1.090585470199585, 0.8607800006866455, -0.5574841499328613, -0.8022616505622864, -1.0204910039901733, 0.40883731842041016, 0.9864931106567383, 0.12100635468959808, 1.5365190505981445, -0.8234146237373352, 1.3061449527740479, 0.0973304882645607, 0.17920151352882385, 0.41963088512420654, -0.5367982387542725, -0.8710606694221497, -0.3806324303150177, 2.249100685119629, 0.9703137874603271, 0.031414952129125595, -0.7363954782485962, 0.8414919972419739, 1.3381531238555908, -2.0332272052764893], [0.10651186853647232, -0.9106408357620239, -0.5070840716362, 1.2131372690200806, 0.8654528260231018, 0.9759734272956848, -0.4775862693786621, -0.665168285369873, -0.05959172919392586, -0.3133235573768616, 1.5074306726455688, 1.0570268630981445, 1.823309302330017, 0.5004193186759949, -0.3357725739479065, -0.5702285766601562, -0.6303291916847229, 0.22048141062259674, -0.5471484065055847, -0.38430821895599365, 0.16044728457927704, 2.1104886531829834, 0.7268950939178467, -0.060420144349336624, 0.2064228057861328, -1.0156348943710327, -1.2888545989990234, -1.2391045093536377, 0.9541375041007996, -0.9491848349571228, 0.347243994474411, -0.7107869982719421, 0.9144579768180847, 0.5527699589729309, -0.9858359098434448, -0.6962385177612305, 0.24767936766147614, -0.677038311958313, -0.6919165849685669, -1.0815962553024292, 1.51465904712677, -2.0509045124053955, -0.08248791843652725, -1.3143757581710815, -0.6178757548332214, -1.487313985824585, 0.7223041653633118, 1.1007643938064575, -0.08233435451984406, 0.461930513381958, -0.09513986110687256, -0.9900382161140442, -0.014347722753882408, 0.5271696448326111, -1.452122688293457, -2.343636989593506, 0.3006839156150818, 0.9108262658119202, -0.6898109912872314, -2.3867685794830322, -0.27739477157592773, -2.4944889545440674, 0.03206435590982437, -0.8538042306900024, -0.36693814396858215, 2.501295328140259, -1.6640138626098633, -0.41612717509269714, -2.289689540863037, -0.7915056943893433, -1.6842671632766724, 0.4506461024284363, -2.375121593475342, -0.3612341284751892, -0.29715341329574585, 2.1795449256896973, -0.5534592866897583, -1.0441665649414062, -0.29561418294906616, 1.4040123224258423, -1.5305520296096802, 0.048061978071928024, 0.4290023446083069, 0.31019455194473267, -0.3894115090370178, -0.8331838846206665, -1.5898866653442383, -0.9661175012588501, 0.8754521608352661, 1.6482151746749878, 0.5272332429885864, -0.5770922899246216, -0.3585241436958313, 0.9501441717147827, 0.2568458914756775, 0.12884773313999176, 0.06653985381126404, -0.8375177979469299, -2.4254796504974365, 0.059796471148729324, -0.5869592428207397, 1.6675716638565063, 0.17967157065868378, -0.055594924837350845, -0.33063220977783203, -0.006298485212028027, -1.0856319665908813, -0.33184492588043213, 0.14971868693828583, 0.3805387616157532, 1.4152181148529053, 0.5816946625709534, -0.21296940743923187, 1.0985738039016724, -0.3463374674320221, -0.3079737722873688, 0.3261291980743408, -0.7861438393592834, 1.0634773969650269, -0.6545050144195557, -0.0013585276901721954, -0.4128868877887726, 1.1888879537582397, 0.18167045712471008, -0.06922696530818939, 1.9356802701950073, 1.0484850406646729, -0.6053622364997864, 0.3949715495109558], [-1.085772156715393, -0.4141280949115753, -1.2021441459655762, -0.6716819405555725, -1.1016329526901245, -0.08653293550014496, 0.9098379611968994, -0.4784708023071289, 0.3501473367214203, -0.28575995564460754, 0.8111833333969116, -0.048366304486989975, 1.3484264612197876, -1.2284969091415405, 0.7486830949783325, -0.32085397839546204, -0.026659000664949417, -0.657265841960907, -0.9544476866722107, 0.16309089958667755, 1.159460425376892, 0.6842795014381409, -0.4736920893192291, -0.1046544760465622, -0.9353628158569336, -0.5947003364562988, -0.127152219414711, 0.5181236267089844, -0.252073734998703, -0.7760715484619141, 0.3852458894252777, 0.4253135919570923, -0.6856482625007629, -0.8141794204711914, 0.9804893136024475, -0.2729153335094452, 0.04510580375790596, -0.34405165910720825, -0.6276362538337708, 0.4167582094669342, -0.07562098652124405, 0.48378416895866394, -1.2821662425994873, -1.6520739793777466, -0.12156689912080765, 1.2010877132415771, -0.14718396961688995, -0.2589525878429413, -0.5715749263763428, -0.35260966420173645, 0.9068032503128052, -0.4573615491390228, 0.45470714569091797, -0.7444782257080078, -0.622178852558136, 0.3515237271785736, -0.5142890214920044, -0.0031795946415513754, -0.589629590511322, -0.7168009281158447, -0.5404428243637085, -1.1251381635665894, -0.8045541644096375, -0.8795820474624634, -0.5041012763977051, -0.5336707830429077, -0.9934307336807251, -0.8867098093032837, 1.1737351417541504, -0.17186786234378815, 0.04719412326812744, 0.3003968596458435, -1.3372427225112915, -0.2340584695339203, 2.700252056121826, -1.357688307762146, -1.2661559581756592, -0.5282111763954163, 0.924665093421936, -0.1580520123243332, 0.47265419363975525, 1.0269182920455933, -1.541509985923767, 1.8329370021820068, 0.09462874382734299, -2.8999955654144287, 0.37089765071868896, -0.022052323445677757, 1.0547646284103394, -0.084694504737854, -1.419904112815857, -0.18097054958343506, -0.4706054925918579, 0.2842855453491211, 0.1425919532775879, -1.47564697265625, 1.0850239992141724, 0.24810536205768585, 0.6603123545646667, -0.7266576886177063, -1.107230305671692, 0.43612179160118103, -0.8815745711326599, -0.7408697009086609, -1.557567834854126, 1.7087726593017578, 1.7936452627182007, -0.8675910234451294, 0.27440109848976135, -0.3621276319026947, 0.19652186334133148, -1.2362664937973022, 0.9177681803703308, 0.20347601175308228, 0.06411992758512497, 0.4188658893108368, -0.9850490689277649, 0.33552688360214233, -0.8004998564720154, -0.12779968976974487, -0.10955502092838287, 0.7275434732437134, 1.6246269941329956, -0.47076234221458435, -0.08261413872241974, 0.8926643133163452, 1.473305344581604, 0.2538585662841797, 0.7261401414871216], [1.2700997591018677, -0.15460413694381714, -1.2529606819152832, -0.10550899058580399, -0.9632699489593506, 0.39804625511169434, 0.46901094913482666, -0.6026937961578369, -0.38061243295669556, 1.2993991374969482, -0.4278814196586609, -2.272866725921631, 1.1723451614379883, -0.5293684005737305, -1.1678980588912964, -1.0722073316574097, -0.4564139246940613, 0.5449516773223877, 0.938126266002655, 0.7432382702827454, 0.3240075707435608, -0.28827163577079773, 1.6550524234771729, -0.8986006379127502, -0.3539098799228668, -0.8967990279197693, -0.7316504120826721, -1.4786618947982788, -0.15394015610218048, 0.03867271542549133, -0.9785521626472473, 0.3579961359500885, -0.24555018544197083, 0.4883633852005005, 0.881958544254303, 0.9339410066604614, 0.28791090846061707, 0.11648304015398026, 1.0060142278671265, -0.11352023482322693, -1.179079532623291, -0.38390153646469116, -1.5973317623138428, -0.6976699233055115, -0.20445643365383148, -0.24160440266132355, -1.943163514137268, -0.27684009075164795, -0.20370085537433624, -0.07149653881788254, 0.891510546207428, 0.7870270013809204, -0.561855673789978, -0.06150477007031441, 0.2565552890300751, 2.3972396850585938, 0.22018538415431976, 1.263190507888794, 0.5634796023368835, 0.5557958483695984, 1.0677696466445923, -0.8551065325737, 1.2828187942504883, 0.7282806038856506, 0.5315945148468018, -0.7526755332946777, 1.5902522802352905, 1.258919596672058, 1.8227659463882446, 0.23267050087451935, 0.1475231945514679, 3.101992130279541, -2.278604745864868, 0.7216992378234863, -2.3043692111968994, 1.3166224956512451, 0.5336883068084717, 0.42941907048225403, 0.7254627346992493, 0.45508360862731934, -1.9801244735717773, -1.1592715978622437, 0.80155348777771, 0.047888435423374176, -0.3554428517818451, 0.9655774235725403, -0.526541531085968, -0.6655592918395996, 1.0124520063400269, -0.0703720897436142, -2.7765281200408936, 2.3607869148254395, -0.5232967138290405, -1.571334958076477, 1.4183231592178345, 0.9532020092010498, -0.42637866735458374, 0.5507919788360596, 0.007422259077429771, -1.2295691967010498, 1.2617833614349365, -0.5398383140563965, 0.3709096312522888, -0.13858520984649658, 2.3151304721832275, -0.37378445267677307, 0.7775942087173462, -1.3482520580291748, 1.7053858041763306, 0.12955766916275024, -0.9221960306167603, 1.4893300533294678, -0.21041423082351685, -1.4375158548355103, -0.30928656458854675, 0.0680144652724266, -0.7626726627349854, -0.3780650198459625, 1.682176113128662, 1.708795428276062, -0.7754226326942444, 0.25704362988471985, 0.09146355092525482, -0.2720949351787567, -0.4194457232952118, -0.04721739888191223, 0.966400146484375, 0.6575457453727722, 0.1699189841747284], [-0.5395210981369019, -0.05372699350118637, 1.166847825050354, -0.23548570275306702, -1.0575876235961914, -0.10670580714941025, -0.1688210368156433, 0.14600065350532532, -1.7938473224639893, 0.7049537301063538, -2.483478546142578, 0.8573561310768127, 1.5417547225952148, -0.7971748113632202, 0.08730725944042206, -1.456630825996399, 1.117682695388794, 2.440751791000366, 0.6196703314781189, 0.7404330372810364, -0.6793614625930786, 1.8314862251281738, -1.05238938331604, 0.8206520080566406, -1.241659164428711, -1.8814828395843506, -0.3626719117164612, 0.9674810171127319, 2.2509219646453857, -2.667224168777466, -0.1415291130542755, 0.4419809579849243, -0.7577776908874512, -0.0043773772194981575, 0.09973554313182831, 0.10830798745155334, 0.07371960580348969, -1.4625449180603027, 0.11215343326330185, -1.5352725982666016, 0.887550950050354, -1.5965943336486816, -1.1648485660552979, -0.5486238598823547, -1.0810039043426514, -0.5894953012466431, 2.407071113586426, -0.012963112443685532, -0.12989121675491333, 0.04711494594812393, 0.211606964468956, -1.2201110124588013, -1.9877108335494995, -0.310764342546463, -0.5002818703651428, 1.3723939657211304, -0.9256719946861267, -0.0026602514553815126, -0.9862207174301147, 1.02906334400177, -0.5996097922325134, -0.7513487339019775, 0.6367974281311035, -1.5066598653793335, -0.639203667640686, 0.7739095091819763, 1.8009779453277588, -1.4685308933258057, -0.7339358925819397, 0.06438642740249634, 1.258225917816162, -0.13913670182228088, 0.05431092530488968, 1.2938554286956787, 0.46250277757644653, -0.07406007498502731, -0.667364239692688, 0.06856046617031097, 0.7179285287857056, -0.4668520390987396, 0.6382873058319092, -0.43504637479782104, 0.04915938898921013, -0.7507537007331848, 1.9438918828964233, -1.1900101900100708, 0.884737491607666, -1.103108286857605, 3.15678334236145, 0.40388059616088867, -0.6430411338806152, 0.14628778398036957, 0.9232845902442932, 0.02365773357450962, -0.5513547658920288, -1.3818365335464478, 0.6091631054878235, 0.9801838994026184, -0.2484768182039261, -0.468933641910553, 0.5831835865974426, -0.7495503425598145, -0.4469643235206604, -1.0685755014419556, -0.4467773139476776, 0.7747675776481628, 0.6434618234634399, -0.48365309834480286, -0.37008237838745117, 0.6929054856300354, 0.37053436040878296, -0.13935799896717072, 3.4127936363220215, -1.1601476669311523, 0.6378829479217529, -0.8125985264778137, 0.692548930644989, -1.1612211465835571, -0.9496686458587646, 0.8083319664001465, -0.5664513111114502, -0.19416719675064087, -1.5706301927566528, 0.6309528946876526, -0.5067675709724426, 0.37032583355903625, 0.6936783790588379, 2.255967140197754, 0.48461827635765076], [0.2570015490055084, 0.8922932147979736, -1.5545307397842407, 0.4768455922603607, -0.7457257509231567, -2.243720769882202, -0.05665643885731697, -0.8732407689094543, -0.3583886921405792, -0.47762516140937805, 0.012106615118682384, 0.9587364196777344, -2.3394341468811035, 0.21168309450149536, -1.7719203233718872, -0.11368650197982788, -0.04959932342171669, -0.23029989004135132, 0.45480915904045105, -0.2246592789888382, -0.8308831453323364, 1.0685847997665405, -1.2367619276046753, 0.22714433073997498, 0.9937096238136292, -1.6651767492294312, 1.715138554573059, -2.1665191650390625, 0.17979495227336884, 2.0486397743225098, -0.014222382567822933, -0.3841385245323181, -1.097773790359497, 0.5870062112808228, -0.002356449142098427, 0.2449307143688202, -1.8138434886932373, 2.0486388206481934, 1.4071427583694458, 1.6989165544509888, -0.5175551176071167, 0.8835088014602661, -1.5325928926467896, 0.7769739031791687, -0.7059540152549744, 0.15565623342990875, -1.2475566864013672, -0.042590606957674026, 0.47425588965415955, -1.282279372215271, -0.14657779037952423, 0.5490896701812744, -0.010912065394222736, 0.9430986642837524, 2.605720043182373, 0.11499588191509247, 0.8621492385864258, -1.085788607597351, 0.8180932998657227, -0.3271397352218628, -2.3244643211364746, -1.0376893281936646, 0.26625391840934753, -1.4197683334350586, -0.11315761506557465, 1.9456385374069214, -0.8848373293876648, 0.6985006928443909, 0.7333857417106628, 1.4545842409133911, 1.479698657989502, 0.32912248373031616, -0.1200963631272316, -2.0666091442108154, -0.7460307478904724, 0.6247702240943909, -0.32861778140068054, -0.5539244413375854, 0.5884624719619751, 0.032561808824539185, -0.46218055486679077, 0.4473821222782135, -1.8161429166793823, 0.8760985732078552, -0.39071235060691833, 0.7032285928726196, -0.45777490735054016, 0.7370107769966125, -0.601649820804596, -0.2768310606479645, -0.4655480682849884, 1.6815062761306763, -0.2464362233877182, 1.805799961090088, 0.3692253828048706, 0.777880072593689, 0.2485073059797287, 0.3183228075504303, -0.8491170406341553, -0.786361038684845, -1.4243929386138916, 1.5674209594726562, 1.0821529626846313, -1.3597979545593262, 1.2111109495162964, -0.9783151149749756, -0.22651153802871704, -1.0213449001312256, 0.7829433679580688, 1.1866410970687866, 0.937470555305481, 1.8663630485534668, -1.324897289276123, 1.8240004777908325, -0.7056620121002197, -0.5534517168998718, 1.6029331684112549, -0.6086131930351257, -0.7880197763442993, 1.3977354764938354, -0.9718251824378967, -0.16080161929130554, -0.11690596491098404, -1.3087046146392822, -1.2794443368911743, -0.49726414680480957, -0.04742434248328209, 0.10122410207986832, 0.4672746956348419], [0.8276731371879578, 0.24716122448444366, 1.1235536336898804, 0.33677947521209717, -0.8720404505729675, -0.19455765187740326, -1.0246496200561523, -0.40990325808525085, 1.6788709163665771, 1.2414048910140991, 0.2810446619987488, -1.252286672592163, 1.3361419439315796, -0.8319437503814697, 0.5789255499839783, 1.059909701347351, 0.9832196235656738, -1.301632046699524, -0.1432127207517624, -1.9749236106872559, 0.6883288621902466, -0.9080274105072021, -1.0608199834823608, -0.9167473316192627, 0.4890604019165039, 0.5797243118286133, -0.35116270184516907, -0.9686539769172668, 0.29308265447616577, 0.8584117293357849, 0.3642832934856415, 2.6570703983306885, 0.2836296558380127, -0.164091557264328, 0.6137628555297852, -0.14166060090065002, -0.10449143499135971, 0.6962378621101379, -0.8543261289596558, 0.856461763381958, -0.9752600789070129, 0.5984376668930054, 0.608742892742157, -0.3295479118824005, 0.5844404101371765, 0.2758409082889557, -2.200873851776123, -1.7476879358291626, -0.6946383118629456, 1.6629256010055542, 2.066506862640381, 0.9491900205612183, -0.25446921586990356, -1.587724208831787, -0.6516521573066711, -0.39881086349487305, 0.8497114181518555, -0.5044963359832764, 0.6340702772140503, -0.8146295547485352, 0.23208913207054138, -0.5142080783843994, 1.0440024137496948, 0.7426922917366028, -0.4632530212402344, 0.32335665822029114, -0.6642186641693115, -0.9801421165466309, 1.2909562587738037, -0.3980710208415985, 1.4963384866714478, -1.1383901834487915, 0.9265434145927429, -0.375315397977829, 0.41692814230918884, -0.5386031270027161, 1.689894437789917, -0.587268054485321, -0.09704238176345825, -0.13500110805034637, 0.4489366412162781, -0.6121889352798462, 0.13767117261886597, 0.3965265154838562, -1.2773836851119995, -2.0101757049560547, -1.1601266860961914, 0.26867350935935974, -0.4179745316505432, -1.1126114130020142, -0.017682502046227455, -0.1992426961660385, -0.07630370557308197, -1.0490121841430664, -1.0607186555862427, 0.030437134206295013, 2.1698851585388184, 1.4981253147125244, -0.4548533856868744, 0.6303656101226807, 0.48000141978263855, 0.37628141045570374, 0.6592554450035095, 0.6432217955589294, -1.1294671297073364, 0.46815815567970276, 0.5478822588920593, -1.1683197021484375, -1.1475574970245361, 0.9374096989631653, -0.3249884843826294, -1.8473173379898071, 1.909854531288147, -0.20073406398296356, 0.6396579742431641, -1.5270782709121704, -0.6363463997840881, 0.12188466638326645, 0.047979600727558136, 0.09212026745080948, -0.3335564434528351, 1.081843376159668, 0.8246628642082214, -1.7903568744659424, 1.4851917028427124, -1.5340731143951416, -0.4708099663257599, 0.46396851539611816, 0.08122396469116211], [0.045781344175338745, 0.11507359892129898, 2.378305673599243, -1.5554393529891968, 0.6524691581726074, -0.034275028854608536, -1.0487068891525269, 1.1057575941085815, 1.1223547458648682, 1.247340202331543, 1.5067590475082397, 1.6961027383804321, -0.6645743250846863, 0.10978946089744568, 0.3911501467227936, 0.6234167218208313, -0.07825608551502228, -0.16955502331256866, -1.3236936330795288, 0.2030608057975769, 2.2617273330688477, -0.604504406452179, 1.854030728340149, -0.6436981558799744, -0.390781044960022, -0.8720136880874634, -0.23144342005252838, -0.5386784672737122, 4.283139705657959, 1.186793327331543, -1.1394246816635132, 1.426557183265686, -0.10480289161205292, 0.6522165536880493, -0.18755307793617249, -0.6962733864784241, 0.4704838991165161, 1.0877981185913086, 1.0698736906051636, 0.43962815403938293, -0.9619007110595703, -0.7721461057662964, 0.258865624666214, 0.11045068502426147, 1.0479298830032349, 1.3932392597198486, -1.382098913192749, 0.26770278811454773, -1.4971179962158203, 0.4560895562171936, -0.1378057301044464, 0.0327715128660202, -1.1895442008972168, 0.1547704041004181, -0.007651053369045258, -0.8932693004608154, -0.31959232687950134, 0.1249714270234108, -0.02378254570066929, -0.9226585030555725, 1.7867567539215088, -0.049772173166275024, 0.4561803340911865, -0.48380082845687866, 1.8123773336410522, 0.13351605832576752, -1.3856428861618042, -0.12271428108215332, 0.5781266689300537, -0.560441255569458, -0.8717561364173889, 2.029930830001831, 1.267817497253418, -1.3338618278503418, 0.38919147849082947, -0.3357909023761749, 0.34881338477134705, 0.23616129159927368, -1.179458498954773, 0.19557280838489532, -1.0802223682403564, 0.8716282248497009, 1.1243027448654175, 2.777583599090576, 0.1451641023159027, 0.0009271950111724436, 0.6922136545181274, -0.5865631103515625, 0.14202654361724854, 0.32701656222343445, -0.21645012497901917, -1.432462453842163, -1.0668286085128784, 0.463914692401886, -0.20979581773281097, 1.2858338356018066, 0.12733685970306396, 0.8468735218048096, -0.2265751212835312, -0.04998958483338356, 0.6073748469352722, -1.4812709093093872, 0.41658899188041687, 0.42328694462776184, -0.5525925755500793, 1.005753755569458, -0.10580722987651825, -0.3240494132041931, 1.8370264768600464, -2.124253034591675, 0.679225504398346, -0.38647428154945374, 0.08283857256174088, -0.7271595001220703, -1.1717084646224976, 0.2836402356624603, -0.8585915565490723, 0.5918582677841187, -2.060448169708252, 0.7096009254455566, -0.2645750045776367, 1.3454796075820923, 0.0798557847738266, 0.5941817760467529, -1.0720561742782593, -0.18623314797878265, -0.6656016707420349, 1.6279021501541138, -0.03098948299884796], [-0.2991284132003784, 1.4506206512451172, 0.5895501971244812, -0.6754104495048523, -0.6332891583442688, -0.4616537392139435, 0.4220661222934723, 0.8518792986869812, 0.0645175576210022, 0.6282517910003662, -1.194151759147644, 1.2160892486572266, 1.1865038871765137, 0.7580776810646057, -0.5782538056373596, 0.6696408987045288, 0.7775058746337891, 0.6445018649101257, -0.035176489502191544, -0.23206861317157745, -1.278496503829956, -1.4230632781982422, -0.23373602330684662, -0.4622659385204315, -0.06827143579721451, 0.8014046549797058, 0.6633652448654175, -1.1892766952514648, -1.2490556240081787, -0.06216626614332199, -0.21985742449760437, 0.395860493183136, 0.8296765685081482, -0.02966887317597866, 0.7309855222702026, 0.17117968201637268, 2.735419750213623, 0.9920737147331238, 0.7938752174377441, -0.1515820473432541, -3.1444547176361084, -1.1638621091842651, -1.8932627439498901, 0.5554162263870239, 0.9923501014709473, -0.7335296273231506, 0.25763168931007385, -1.1689655780792236, 0.5944332480430603, 0.08347628265619278, 0.6088220477104187, -0.3723168671131134, -1.4727839231491089, 0.37284454703330994, -0.6347825527191162, 1.3948135375976562, 0.44903576374053955, 1.0734930038452148, -0.4512459635734558, 0.7313823699951172, -0.15675877034664154, 0.5931271314620972, 1.6828337907791138, 0.1952250897884369, 0.07011842727661133, -1.78251051902771, 0.6939129829406738, 0.11497442424297333, 1.3719658851623535, -0.5305641293525696, -0.8949492573738098, 1.153095006942749, -1.2148770093917847, -0.7954733967781067, -0.6597902178764343, 0.16885998845100403, 0.19939342141151428, -0.6946283578872681, -0.761425793170929, 1.445203185081482, -0.5517101883888245, -0.3971233069896698, 1.3149691820144653, 1.501299262046814, -1.2844817638397217, 0.01701052486896515, 0.1276337206363678, -0.855800986289978, 0.9579392075538635, -0.002456626622006297, 1.1870346069335938, 0.4463578164577484, -1.995487928390503, -0.4087070822715759, -0.6507318615913391, -0.09109780192375183, 1.3417859077453613, 2.0859265327453613, -1.7249047756195068, 0.6108285188674927, -1.3200793266296387, -0.3657469153404236, 0.904079258441925, 0.9884911179542542, 0.9465804696083069, -1.2227290868759155, -1.2399649620056152, -1.4588090181350708, 1.203050136566162, 0.6620927453041077, 0.30540359020233154, -0.1267034113407135, 0.8248343467712402, 0.15062539279460907, -0.2129250317811966, 1.006680965423584, 0.7160508036613464, 0.0057575153186917305, 1.1686171293258667, 1.4454737901687622, -1.4745289087295532, 0.7668517827987671, -0.15192092955112457, -1.1318303346633911, 1.4321593046188354, -1.5660324096679688, 1.6664799451828003, 0.5326480269432068, -1.0225261449813843], [-0.836780309677124, -1.1464545726776123, 0.9000306129455566, -1.4796208143234253, 0.5192549824714661, -0.7091504335403442, 0.47394663095474243, 0.39011693000793457, -1.3216646909713745, 0.934650182723999, -1.2012794017791748, 0.13712802529335022, -1.1289771795272827, -1.641858696937561, -0.7585641741752625, 1.3172276020050049, -0.3456208407878876, -0.6187769174575806, -0.45118093490600586, 0.6781858205795288, -1.9784271717071533, 1.065833330154419, -0.22004537284374237, 1.5142251253128052, 0.9021490216255188, 0.4422333240509033, -2.5539145469665527, -1.8610870838165283, -0.6459376215934753, 2.411564350128174, 0.6881508231163025, -1.2341963052749634, -1.0938059091567993, -0.2778061330318451, -0.21092310547828674, 0.43612122535705566, -0.6354042887687683, -1.3950883150100708, 1.2753043174743652, -0.6246567964553833, -0.8969926238059998, -0.7369175553321838, -0.22689557075500488, 0.050321418792009354, -0.4236268997192383, -1.410693645477295, -0.2217027097940445, 0.7112061381340027, -0.44414466619491577, 0.4884811043739319, -1.3801472187042236, -0.24217303097248077, 1.2051787376403809, 1.0386143922805786, -0.10401380807161331, 0.8845704793930054, -0.3252241015434265, -1.178844690322876, -0.83222895860672, 0.9653145670890808, 1.8250956535339355, -0.9923655390739441, 0.8955398201942444, -1.6043628454208374, 0.37795329093933105, -0.2650150954723358, -0.7370190620422363, -0.48180457949638367, -1.6527856588363647, -1.497481346130371, -0.837498664855957, 0.8594023585319519, -0.2853480279445648, -0.10611558705568314, -1.4559781551361084, 0.05795852094888687, -1.7039785385131836, 0.5010120272636414, 2.8861186504364014, -1.6285761594772339, -0.029240205883979797, -1.3150966167449951, -1.380555272102356, 1.6758320331573486, -2.337430000305176, -1.149237036705017, -0.802902340888977, -0.20983527600765228, -2.0473573207855225, 1.226266860961914, 0.34317266941070557, 1.9417691230773926, 0.5003153085708618, 0.33338040113449097, 0.6165953874588013, -0.8157153129577637, -1.3339966535568237, -0.17425720393657684, -1.4853075742721558, -0.9443811774253845, -1.7101298570632935, -0.5657123327255249, -0.7178240418434143, 0.11767502874135971, 0.06756081432104111, -1.4590203762054443, 2.3237335681915283, -1.1356709003448486, 0.8242945671081543, -1.8451306819915771, 0.32646939158439636, -0.7877475619316101, -0.5439939498901367, 0.05202211067080498, -0.6521685123443604, 3.12093186378479, 1.4257296323776245, -0.6735590696334839, -0.7815006971359253, 0.4793437719345093, -1.2843589782714844, -0.5751673579216003, -0.7755566239356995, -0.6703309416770935, 1.0844365358352661, -0.7500394582748413, 0.7620336413383484, -0.3099297881126404, -1.380825161933899], [-2.8947789669036865, -0.5559414625167847, 0.7539800405502319, -0.004347710870206356, 0.6845636367797852, 0.31769827008247375, -1.638371467590332, 0.10820139199495316, 0.8699194192886353, 0.24813593924045563, 1.7710843086242676, -0.1513996422290802, -1.099660873413086, 0.9513896107673645, -0.4059293866157532, -0.26842591166496277, -0.8020601868629456, 2.340208053588867, -0.119794100522995, -0.8001558780670166, 0.7740110754966736, -1.5490609407424927, 1.2269468307495117, 0.28484126925468445, -0.06366097927093506, -0.5783755779266357, -0.2450455278158188, 0.012075171805918217, 1.3076667785644531, 0.20521636307239532, -0.4563406705856323, 1.8673936128616333, -0.1689540147781372, -0.7157506942749023, 0.37952741980552673, -1.445277452468872, 0.32627347111701965, 0.3764730393886566, -1.1405876874923706, -0.5996868014335632, -1.3948825597763062, -2.1659247875213623, 0.32584619522094727, -0.25697842240333557, 0.5307860970497131, -0.5245250463485718, -0.4045840799808502, 0.3836497962474823, -0.9187211990356445, 0.6470299959182739, 1.294486165046692, -1.4010282754898071, 1.4841828346252441, 0.19304557144641876, -1.1356704235076904, 0.7947949171066284, 0.6187572479248047, -1.0564672946929932, -0.0794479250907898, 1.6952803134918213, 0.5245218873023987, 1.0606149435043335, 0.9486951231956482, 1.991235375404358, 1.1173804998397827, -0.35079753398895264, 0.7050848603248596, -0.15821263194084167, -0.609025776386261, 0.021959828212857246, 0.7287552952766418, -0.6943463087081909, 0.5904937982559204, -1.5860323905944824, -0.2796615958213806, -1.303888201713562, -0.7467705607414246, -0.979954719543457, -1.9642446041107178, 0.5376551151275635, -2.372159004211426, -0.7915968298912048, -0.7616147398948669, 0.4239785075187683, 0.5698071122169495, 0.8099409341812134, -0.8131139278411865, -1.0144954919815063, -1.4159395694732666, -0.39795053005218506, -0.41695624589920044, -1.822249412536621, -0.6732149720191956, -0.3771708011627197, 0.46462100744247437, -0.7468640804290771, -0.6752190589904785, 1.5112422704696655, -2.4055352210998535, 0.6638026237487793, 0.29357144236564636, -0.3729461431503296, -0.6317509412765503, 1.2939512729644775, -1.8388447761535645, -0.13224853575229645, 1.5967568159103394, -2.099874258041382, 1.2332156896591187, -0.21056027710437775, 0.31768766045570374, -0.7890720963478088, 1.3005144596099854, -0.17087656259536743, -2.0124378204345703, -1.5535271167755127, 1.3051855564117432, -0.32640334963798523, 0.6980986595153809, 1.1359751224517822, 0.5242843627929688, 1.4141781330108643, 1.319770336151123, 0.08762189745903015, 0.36644446849823, -0.3748285472393036, 0.7911403179168701, 0.5977195501327515, 1.205541729927063], [1.4800992012023926, 0.6190136075019836, -0.7713578939437866, -0.3432159423828125, -0.9261890053749084, -1.9548594951629639, -0.8288925886154175, -2.015188694000244, 0.09934406727552414, 1.533442497253418, 0.7135912179946899, 1.182149887084961, -0.533240795135498, 0.28068020939826965, 0.34557095170021057, -1.0039584636688232, -1.0080161094665527, 1.760682225227356, 0.032478779554367065, 0.593448281288147, 0.3612949550151825, -0.674837052822113, 1.1830977201461792, 1.5828148126602173, -0.3186156451702118, -0.7467490434646606, -0.2211441546678543, -0.14425784349441528, -0.5440880656242371, -0.3921143710613251, 0.6141649484634399, 1.038623332977295, -1.0518513917922974, -0.3262075185775757, -2.695807456970215, -0.18373751640319824, 0.8117376565933228, 0.005116660613566637, 1.589854121208191, -0.5992497205734253, -1.2391453981399536, 1.7273942232131958, 1.4215240478515625, -0.19757702946662903, -0.7969064712524414, -0.8532021641731262, -0.6262814402580261, 0.9830537438392639, -1.5404094457626343, 0.19517992436885834, 0.40698906779289246, -0.9001387357711792, 1.5810447931289673, 1.7143769264221191, 2.12930965423584, -0.6112801432609558, 0.18966491520404816, 2.4553329944610596, -0.6295913457870483, 0.38515210151672363, -0.3987375497817993, -0.4187193214893341, 0.24753418564796448, 0.19673125445842743, -0.13189123570919037, 0.2812575399875641, -0.5796188116073608, -0.2686462700366974, -0.3729362189769745, 0.34620383381843567, -0.33238133788108826, -1.070244312286377, -0.6466277837753296, -1.2003016471862793, 2.36921763420105, 0.21425294876098633, 0.31942471861839294, 0.04848957061767578, -0.048749588429927826, 0.3738700747489929, -0.9937962293624878, 0.7283920049667358, -1.3911641836166382, -0.47177353501319885, -0.8994579911231995, -0.4043150544166565, -0.5633283257484436, 0.23001141846179962, 0.02495104819536209, 0.17822566628456116, -0.7142682075500488, -0.8413059711456299, 0.03193971887230873, -0.5715006589889526, -0.04471980780363083, 0.19380320608615875, 0.2353425920009613, 1.1309441328048706, 0.2635398507118225, 0.010769922286272049, -0.32804879546165466, 0.0358782559633255, -0.05699486657977104, 0.18663522601127625, 0.43415752053260803, 0.01893235556781292, 1.1635770797729492, 1.3297892808914185, 1.3251749277114868, 0.7736498713493347, -0.2968754470348358, -0.8204681873321533, -1.067134141921997, 0.5593025088310242, 0.1507519781589508, 0.3529489040374756, 0.5469583868980408, 0.5814309120178223, -0.6427814364433289, 2.3262810707092285, 0.651987612247467, -0.5823842287063599, 0.9318011403083801, 0.9728841781616211, -0.31201720237731934, -1.1685141324996948, 1.1501458883285522, -1.3080358505249023, 0.6181831359863281], [-0.8692688345909119, 1.600826382637024, -1.5612549781799316, 1.7276620864868164, -0.7692229151725769, -0.13334542512893677, 1.3065348863601685, -2.3895740509033203, 0.08275308459997177, -0.4645337164402008, 1.507689118385315, 0.43804749846458435, 0.2635571360588074, 0.23445378243923187, 0.8856140971183777, 0.6961292624473572, -1.1581214666366577, -1.6601190567016602, -0.36176812648773193, -2.812382459640503, 0.7934011220932007, 0.5892390012741089, -0.46152785420417786, -0.4946519732475281, -0.04649822786450386, 0.3502652049064636, 1.6092628240585327, -0.8578208088874817, -0.16802096366882324, 1.4906585216522217, -0.3826126754283905, 0.3661342263221741, 0.178253173828125, -1.3648942708969116, -0.44382336735725403, -0.09773887693881989, -0.41232961416244507, -0.3222696781158447, 0.25169798731803894, 0.6148048043251038, -0.6973423361778259, -1.2339897155761719, -0.5228679776191711, 0.47529083490371704, 0.23790836334228516, -0.698726236820221, 0.5874361395835876, -0.1957731693983078, 0.13356353342533112, -1.0306001901626587, -0.7367638349533081, 0.7280294299125671, -0.4135638475418091, 0.4406358003616333, -0.6716969013214111, -0.7257817387580872, -0.2700251340866089, -0.6323421001434326, -0.06689076870679855, -0.08101892471313477, -0.5865014791488647, -0.14864349365234375, -0.9940614104270935, 0.5754300951957703, -2.1099202632904053, 1.5598732233047485, 1.2069045305252075, 1.382783055305481, 0.08940503746271133, 0.14797428250312805, -0.19441357254981995, -1.7136565446853638, 0.7418305277824402, -1.6391751766204834, -1.9674030542373657, 0.5636671185493469, -0.11863674223423004, -0.5320054888725281, 1.4914066791534424, -1.2996225357055664, 0.713487982749939, 0.27718299627304077, -0.22736512124538422, 0.24720492959022522, -0.4926333725452423, 1.3948148488998413, -1.8659271001815796, -0.08158963918685913, 0.7533187866210938, 0.5176599621772766, 1.2689372301101685, 0.15639543533325195, 0.5352746844291687, -0.19450365006923676, -0.6641377806663513, 0.744469165802002, -1.462381362915039, 0.9567151665687561, -1.4842724800109863, -0.27655401825904846, -0.06707388907670975, -0.5192714929580688, -0.31632286310195923, -0.002002737019211054, 1.5475598573684692, 1.2053438425064087, 0.8734192252159119, -0.10797140747308731, 0.9206647276878357, 2.0494885444641113, 0.45357874035835266, -0.22877797484397888, 0.9999321103096008, -0.8559762835502625, 0.8356307744979858, 1.653410792350769, 0.6960586905479431, -0.7315136790275574, -1.5394068956375122, -0.3300460875034332, 0.993106484413147, 0.7675679922103882, -0.3846704661846161, -0.20523524284362793, -0.1702641248703003, -0.07833845913410187, -2.272989511489868, 1.4400019645690918, -0.7347733378410339], [-0.0048125009052455425, 0.07556711882352829, 0.0973869189620018, -0.512015163898468, -1.22236168384552, 0.11693793535232544, 0.6948752999305725, 0.166010782122612, -0.13871553540229797, 1.1104800701141357, -0.7660242915153503, 0.4517478942871094, 0.4956286549568176, -0.05676943063735962, 1.0683590173721313, -0.17830166220664978, 0.114400714635849, -2.062864303588867, 0.18197305500507355, 0.6712445616722107, 0.5912058353424072, 0.6497718691825867, -1.135690450668335, 0.5212795734405518, -1.6104457378387451, 0.1667117476463318, 0.8959135413169861, -0.21619005501270294, 0.6577754616737366, 0.5241814851760864, 0.9734128713607788, -0.4992416799068451, 1.6953446865081787, -1.0982064008712769, 0.5927171111106873, -0.7054668068885803, -1.8263072967529297, 1.0975255966186523, -0.6257142424583435, -0.3622470498085022, -0.8470122218132019, 0.14625097811222076, -1.356034278869629, -0.9916017055511475, -0.8991044759750366, -0.24528701603412628, -1.577386736869812, 0.5581476092338562, -0.28173816204071045, -0.14145523309707642, -0.264041930437088, -0.7687057256698608, -0.47182369232177734, 0.0818968340754509, 0.12522542476654053, 0.9440218210220337, 0.3024570345878601, 0.237389475107193, 0.5814758539199829, -0.9180085062980652, 1.1544969081878662, -0.8960742950439453, 0.6677932739257812, 0.2979535162448883, 0.6012861132621765, 0.11902900040149689, -0.09921742230653763, -0.15638728439807892, 1.652880072593689, 0.5470194816589355, -0.3625192940235138, 0.7049476504325867, 0.1085188090801239, -0.8065037131309509, 0.795431911945343, 1.3204371929168701, 0.3931591212749481, 1.8161165714263916, -2.032404661178589, 0.3325119614601135, 0.23910444974899292, -2.366785764694214, -1.0142382383346558, -0.4127017557621002, -0.9457041025161743, 0.18488159775733948, -0.9028394222259521, -0.4342136085033417, 1.827101469039917, -1.227941870689392, -0.12407820671796799, 0.09784047305583954, -0.060726966708898544, 1.2200695276260376, 1.042854905128479, -0.4865516424179077, 1.0568596124649048, -0.9968260526657104, 0.23636308312416077, -0.6353885531425476, 1.698270320892334, 0.9104617238044739, -0.1477256715297699, -1.0220500230789185, -0.9874783158302307, -1.9864306449890137, -1.2343870401382446, -0.4120299220085144, -0.098456971347332, 0.5534773468971252, -0.15600702166557312, -1.172855019569397, -0.08607940375804901, 1.204749584197998, -0.1717914491891861, 1.1405888795852661, -0.7991037368774414, 0.9355995655059814, -0.07442919909954071, 0.9655428528785706, 0.10546346753835678, 1.9069403409957886, 0.035348836332559586, -0.6843109726905823, 0.7948895692825317, 1.35235595703125, 1.173816204071045, -0.4050593674182892, 0.28841954469680786], [-0.7787356972694397, -0.22070887684822083, -0.25316914916038513, 0.26114341616630554, -1.1182997226715088, 1.8572584390640259, -0.6853229999542236, -0.5683351159095764, 1.9959887266159058, -0.0514470711350441, -0.34100401401519775, 0.15323783457279205, -0.926520586013794, -0.23491275310516357, -0.08451720327138901, 0.3609482944011688, -0.020840955898165703, 1.356113076210022, -1.175916314125061, 2.185669422149658, 0.009514633566141129, -0.5951753258705139, 1.11568284034729, -0.30898424983024597, 0.14711813628673553, -0.238278329372406, -0.4742757976055145, 0.6099306344985962, 0.950163722038269, -1.1528578996658325, -0.017991164699196815, 0.4432661235332489, 1.4588170051574707, -1.4795763492584229, 0.08385153114795685, 0.21814106404781342, 0.7208346128463745, 0.3580102026462555, 0.3219877779483795, -0.5817541480064392, -1.5058027505874634, 1.8527480363845825, -0.2939670979976654, -0.3106711506843567, -0.5760008692741394, 0.4440785050392151, -0.4291532337665558, -1.8513027429580688, 0.6788067817687988, -1.128512978553772, -0.6509092450141907, 0.24319668114185333, -0.053107500076293945, -1.2808364629745483, -1.7212092876434326, -0.8487206697463989, 0.703447699546814, -1.709455966949463, 0.09673924744129181, 8.239481394411996e-05, -0.2838026285171509, -2.9677281379699707, 0.5771338939666748, 0.27246493101119995, -1.7127035856246948, 0.3469955325126648, -0.19397686421871185, -0.26974764466285706, 0.7407943606376648, -0.36191946268081665, -0.8808761835098267, 1.0187898874282837, 1.4087715148925781, -0.15569435060024261, 1.5013154745101929, 0.2267865538597107, -0.903498649597168, 1.9923303127288818, 1.0837063789367676, -0.95695561170578, -0.702185869216919, -0.0315258726477623, -0.8159059286117554, -0.7483146786689758, -0.5257203578948975, -0.696002185344696, -0.6529102921485901, -0.05713135004043579, 0.5728856921195984, -0.5974207520484924, 0.9683653712272644, 0.012958514504134655, 0.2393430471420288, -0.013435591012239456, -1.3354774713516235, 0.7893151044845581, 0.517801821231842, -0.11234649270772934, -0.8887063264846802, -0.11222534626722336, 1.798087239265442, -1.6043000221252441, 0.26105040311813354, -0.6915856599807739, 0.9081900119781494, 0.9679252505302429, -0.21730679273605347, 0.08136551827192307, -0.34369486570358276, -0.1058676466345787, 0.9450258016586304, 1.3219095468521118, 0.37562668323516846, -0.6883153319358826, 0.10507122427225113, 0.13654033839702606, 0.5933209657669067, 0.35180753469467163, -0.44447916746139526, 0.4186716079711914, 0.7881859540939331, -1.4775785207748413, -0.22133633494377136, -0.16236576437950134, 1.8112657070159912, 0.6997225284576416, -0.8667531609535217, 0.8488288521766663, -0.7357262372970581], [-3.1733624935150146, 0.4659605324268341, -1.338633418083191, 0.7564161419868469, 1.2847713232040405, -0.6258681416511536, 0.5780045986175537, 0.33652859926223755, -0.4349449574947357, -0.05468863248825073, 0.2169434130191803, -0.674761176109314, 0.5021323561668396, 0.03240194544196129, 0.3421500325202942, 0.6569948792457581, -1.021504282951355, 0.22482411563396454, 0.4166359007358551, 2.435539960861206, 0.13130046427249908, 0.3883567750453949, -0.32416659593582153, -0.25685837864875793, 0.26485633850097656, 0.3102184534072876, -1.6846908330917358, -0.6333684921264648, 0.06075834855437279, -1.0587724447250366, -1.5253645181655884, -0.3056415021419525, 0.03190601244568825, -0.3689819574356079, -1.0818378925323486, 1.2543317079544067, 0.8495586514472961, -2.1879618167877197, -1.8071463108062744, 0.5832143425941467, 1.2574267387390137, -0.1866627186536789, -1.6141642332077026, -0.36195340752601624, 1.598070502281189, -1.459479808807373, 1.159420132637024, -1.80583655834198, 0.7331826686859131, -0.019152073189616203, -0.01836916245520115, 0.09876053780317307, -0.7856926321983337, -1.0420336723327637, -0.634175717830658, 0.5041379928588867, -0.3209972083568573, -1.2269186973571777, 1.0664721727371216, 0.07347316294908524, -0.8931877017021179, -1.2451857328414917, 0.6711677312850952, -0.1556314080953598, 1.4421932697296143, -0.43030306696891785, 0.9549657106399536, -1.130955696105957, -1.2597459554672241, 0.5405592918395996, -1.2624098062515259, 0.28389713168144226, 0.1601048707962036, 1.4929503202438354, -0.021559834480285645, 0.3655116856098175, 0.6790482401847839, 0.14924348890781403, -0.4945661425590515, 1.769684076309204, -0.796439528465271, 0.9466543197631836, -0.6076924800872803, 1.2856310606002808, 1.5605133771896362, -0.9420831799507141, -1.2359074354171753, 0.8809362053871155, 1.1590685844421387, 1.3905398845672607, 0.04081735759973526, -1.7685507535934448, -0.7145822048187256, -0.16384781897068024, 1.1868782043457031, -1.5412529706954956, -0.5672019720077515, 0.9889546632766724, 0.9882158041000366, 1.2873233556747437, -2.6027414798736572, -0.5466569066047668, 0.25269392132759094, -0.32367372512817383, 0.19023071229457855, -0.5626451969146729, 1.0596239566802979, 0.08863155543804169, -0.12000735849142075, 0.06285151094198227, -1.0088071823120117, 0.553728461265564, -0.08248215913772583, -0.029261238873004913, -1.4234603643417358, -0.3701184093952179, 1.042521595954895, 1.6190375089645386, 0.5621731281280518, 0.8099270462989807, -0.5550020337104797, -0.846035897731781, -0.22579728066921234, 0.2999711334705353, -0.06625524908304214, 1.8245062828063965, 2.1069259643554688, 0.194742351770401, -0.8985216617584229], [2.142524480819702, -0.48096349835395813, 0.00550342071801424, 0.2624395191669464, 0.9594380855560303, 0.5421860218048096, 2.0114634037017822, 0.44827261567115784, 1.2065401077270508, -0.7854411602020264, -0.6792120933532715, 1.3712208271026611, -1.270321011543274, 0.3137185871601105, 1.8472810983657837, 1.929255723953247, -0.16873343288898468, 1.2430514097213745, 0.031287651509046555, -0.21821123361587524, -1.1412882804870605, -1.0523535013198853, 0.16769371926784515, 1.0276604890823364, 0.14332084357738495, 1.9486496448516846, -0.6108643412590027, -0.151557058095932, 1.4476981163024902, 0.7810658812522888, -1.0458332300186157, 0.3368534445762634, 1.2572541236877441, 1.6489994525909424, 0.25449419021606445, 0.5007224678993225, 1.3167321681976318, 0.5651761889457703, 0.21343067288398743, -1.4378854036331177, -0.23284123837947845, -0.4140615463256836, 0.339478462934494, -0.701828122138977, -0.3720238208770752, -1.7312790155410767, -0.3534921109676361, -0.13249410688877106, -0.45211490988731384, 0.31291651725769043, 0.7667603492736816, 1.584009051322937, 0.916655957698822, 0.686989426612854, 0.7552613019943237, -0.5052321553230286, 0.2349439412355423, -0.48205840587615967, 1.927839756011963, 0.4054153263568878, 0.508970320224762, -0.003936359193176031, -0.10199680924415588, -0.3790886402130127, -0.8667876124382019, 1.5418074131011963, -0.6880902051925659, -1.8481210470199585, -0.7541072964668274, 0.12558996677398682, 0.7421361804008484, -0.7224255204200745, -0.9173439741134644, 0.8217197060585022, -0.37958288192749023, -0.19763749837875366, 0.17074626684188843, 0.05831464007496834, 0.09410752356052399, 1.5705991983413696, -0.5319857597351074, 0.3468497097492218, -0.2982349395751953, -0.7842873930931091, 1.1347254514694214, 0.3538900911808014, -1.5767072439193726, 2.701526403427124, -1.466343641281128, -0.24158638715744019, -0.048779163509607315, 1.2411898374557495, -1.5296350717544556, -1.3441811800003052, 0.3915279507637024, -0.40528973937034607, 0.1628967821598053, -0.4313381314277649, 1.704242467880249, 1.623049259185791, -1.2892683744430542, 0.4795369803905487, -0.7080873847007751, 0.273794561624527, 0.7054579854011536, 0.08708157390356064, 1.2593882083892822, 0.7364474534988403, 1.1308730840682983, -1.9261972904205322, 0.8721824288368225, -0.09508861601352692, 0.4748854637145996, -0.35515326261520386, 0.5609540939331055, 0.9450653195381165, 0.7296094298362732, -0.34436148405075073, 1.334375023841858, -0.5557534098625183, -0.1156277135014534, 0.03245638683438301, -1.627495288848877, 1.2932875156402588, 0.8762586116790771, 0.7288188934326172, 2.2119362354278564, 1.5976213216781616, 1.3140026330947876], [1.3236892223358154, 1.8340404033660889, 0.4907555878162384, -0.9809063673019409, -0.02991817519068718, -0.6514052748680115, 1.0420576333999634, -0.8556850552558899, -0.8942161798477173, 0.06318690627813339, 0.5261989235877991, 1.376542091369629, -2.423978090286255, -1.3257629871368408, 2.435511350631714, 0.5640466213226318, -0.1420743465423584, -0.45120084285736084, -0.20173722505569458, -0.3526349663734436, -0.2439044713973999, -0.21525326371192932, 0.14771483838558197, -0.41227638721466064, 1.3406442403793335, 0.526056170463562, 0.8471265435218811, -0.6116958856582642, 0.12773707509040833, -0.10088163614273071, 0.816733181476593, -1.1047738790512085, -2.7479400634765625, -0.18101200461387634, -0.3468036353588104, -1.2616711854934692, -1.5581467151641846, 0.8234389424324036, -1.2171398401260376, 1.247298240661621, -1.9461638927459717, -0.12879601120948792, -0.6876318454742432, 0.3956579267978668, 1.398029088973999, -0.1774754673242569, -1.5996050834655762, -0.47020435333251953, 0.15134631097316742, -1.0808006525039673, 0.22307351231575012, -0.5512017011642456, -0.38613730669021606, 1.6085584163665771, -0.8814036846160889, 1.3955594301223755, 0.4342168867588043, -1.4220623970031738, -1.0701156854629517, -0.39201924204826355, -0.8868411183357239, 0.04849446564912796, 2.1420886516571045, 0.6866120100021362, -0.6625041365623474, -0.3182770907878876, 1.321253776550293, 0.30716270208358765, -0.978566586971283, 0.7152738571166992, -0.5913214087486267, -1.2629401683807373, -0.14289957284927368, 1.3183369636535645, -1.1750526428222656, 0.24631085991859436, 0.43829089403152466, 2.159510612487793, -0.11973317712545395, 0.46493715047836304, -1.8754466772079468, -0.9131036996841431, 0.41656944155693054, -0.447501540184021, -0.27278438210487366, -0.8950828909873962, 2.5668063163757324, -0.1308230757713318, 0.6972514390945435, 1.5463647842407227, -0.6800267696380615, 1.8643499612808228, 0.49122440814971924, -0.7128912806510925, -0.8461739420890808, 1.652574062347412, 0.260530024766922, 1.1376968622207642, -1.0175209045410156, 2.7709033489227295, 1.1122022867202759, -1.018839955329895, 1.7882834672927856, -0.7899294495582581, -0.37754833698272705, -1.5566736459732056, -0.05517848581075668, -2.609532356262207, 0.3662189245223999, 1.454917311668396, 0.8273367285728455, -0.4808294177055359, 2.476543664932251, -0.36848506331443787, 0.5573000311851501, 0.30918431282043457, 0.05311671644449234, 0.4101274311542511, 1.0329248905181885, -0.14665290713310242, -0.829127848148346, -1.1596006155014038, 0.17137879133224487, -0.33888909220695496, 0.14700253307819366, 1.08138906955719, -1.3521029949188232, 0.9505621790885925, 1.0026012659072876], [0.2901080250740051, -1.3225715160369873, 0.3093036413192749, -0.9981583952903748, -0.3085278570652008, 0.07030634582042694, -0.7284106612205505, -0.6320052146911621, -0.7806100845336914, 0.6967220306396484, 0.3584426939487457, -0.14512011408805847, 0.4542735517024994, -0.3306400775909424, 0.31121590733528137, 0.2916325628757477, -0.5763698816299438, -0.07618465274572372, 1.5770916938781738, 1.6228910684585571, 0.2402658611536026, 0.6527953743934631, -1.245507836341858, -1.6572203636169434, -2.346454381942749, 0.4789882004261017, 1.4226287603378296, 0.29094234108924866, 0.3907702565193176, 0.34753715991973877, -0.024449873715639114, -0.8818788528442383, 0.2812251150608063, 0.7780835032463074, 0.20192407071590424, -0.47701048851013184, -1.506509780883789, 1.3468797206878662, 1.8236849308013916, 1.8661518096923828, 0.17215296626091003, -0.10522337257862091, 0.9874067306518555, 0.15260812640190125, -0.8430406451225281, -1.1609734296798706, -0.44284579157829285, -0.4006524682044983, 1.567304253578186, 1.5038182735443115, -0.17508459091186523, -0.05666368827223778, 1.538637399673462, -1.152098298072815, -1.7843425273895264, -0.14510977268218994, -0.8468296527862549, -0.5044321417808533, -0.09468048065900803, 0.7411181330680847, -0.8611307740211487, 2.261910915374756, -1.255509853363037, 1.2079297304153442, -0.6776848435401917, -0.7149619460105896, 0.052011728286743164, 0.2596742808818817, 0.4247555434703827, -0.303399920463562, -0.7038031816482544, -1.1254186630249023, 1.0241564512252808, -0.5495923161506653, -1.2839089632034302, -1.4251646995544434, -0.32735124230384827, -0.3269193768501282, -1.2264575958251953, -1.0843230485916138, 0.6795291304588318, 0.6075193285942078, 0.28153273463249207, -0.5701786279678345, -0.22429387271404266, 0.7297071218490601, -0.6977996826171875, 1.1565896272659302, 0.49128273129463196, 0.2324177771806717, -0.2552816569805145, 1.1187419891357422, 0.7444955110549927, 0.5211254954338074, 0.21404294669628143, -0.17700991034507751, -1.2000292539596558, 0.22472235560417175, 0.5683847665786743, -0.16012455523014069, -1.8141635656356812, 1.0288841724395752, 0.15553997457027435, 0.906559944152832, 0.4815748631954193, -1.4849950075149536, 1.0034695863723755, -1.7615795135498047, -0.42162707448005676, -1.5112318992614746, -0.6823629140853882, -1.6296533346176147, -0.8580275774002075, -1.1182663440704346, 0.1152576431632042, -0.03289360925555229, -0.13392168283462524, -0.5613548159599304, -0.28799939155578613, -1.6075880527496338, 0.7297075390815735, 1.1431522369384766, -0.07695537060499191, -0.3707941472530365, 2.1309385299682617, 0.6173644661903381, -0.7831075191497803, -0.24848885834217072, -0.5162410736083984], [0.6362335681915283, 0.5556060671806335, 1.7939448356628418, -0.21966339647769928, 0.8198395371437073, 0.05492154136300087, 0.19968047738075256, -0.5247945189476013, -0.6289024353027344, 1.113268494606018, 0.7090612053871155, -1.8922938108444214, 1.0099284648895264, 0.21630795300006866, -0.3168649971485138, -1.2864997386932373, 2.09816575050354, -0.6089726090431213, -0.5811517834663391, 0.47601333260536194, 0.5268794298171997, -2.3227128982543945, -1.4574482440948486, -0.15767307579517365, -0.7093528509140015, -0.623838484287262, -1.121968150138855, 0.11978565901517868, 1.0052907466888428, 0.9774938821792603, -0.09405314177274704, -1.9691541194915771, -2.224701166152954, -2.090092420578003, -1.113645076751709, 0.5753540396690369, -0.03633657097816467, -0.16299615800380707, -0.653884768486023, -1.331662654876709, 0.3882271945476532, -0.568976879119873, 0.41247010231018066, -0.6544037461280823, -0.1634664237499237, -2.4615561962127686, 0.6658938527107239, -1.1272281408309937, 1.7261985540390015, -0.8323111534118652, -0.3355467915534973, -0.22517892718315125, -0.32305824756622314, 0.7072487473487854, 1.274553656578064, -0.5692400336265564, 0.11050742119550705, -0.5647128224372864, 1.2039133310317993, -1.662773847579956, -1.585387110710144, 0.04300058260560036, -0.1956312209367752, -0.20281606912612915, 0.8723397254943848, 0.21342140436172485, 0.47998496890068054, -0.5629456043243408, 0.8422868847846985, -1.1181919574737549, -1.538896918296814, 1.244767665863037, 0.005780735053122044, 0.1506504863500595, 1.60440993309021, 1.0439809560775757, -0.015625666826963425, 0.3769415318965912, 0.9557234048843384, 0.1707107573747635, 1.049683690071106, 0.18814407289028168, 0.5972834825515747, -0.09655258804559708, 2.4923934936523438, -0.12156149744987488, 0.536566436290741, -0.051126878708601, 0.8622932434082031, -0.24066339433193207, 0.05445032939314842, -0.6927604079246521, 0.6905726194381714, 1.5035693645477295, 0.15975835919380188, 0.001520268153399229, 1.1450332403182983, 1.0913852453231812, -1.5876343250274658, -0.6333879232406616, -0.7479184865951538, -0.9515209197998047, -0.25963178277015686, 1.5063010454177856, 0.6374945044517517, 1.1904339790344238, 0.5235466361045837, -0.8045426607131958, 0.4586840271949768, -1.0152623653411865, -1.8367515802383423, -1.5986143350601196, 0.7339638471603394, -0.9486635327339172, -0.10358607023954391, 2.0265092849731445, -0.6318140029907227, 0.569721519947052, -0.35586413741111755, 1.056620717048645, 0.13461440801620483, -0.7837617993354797, 0.17570582032203674, 0.6733056902885437, -0.7190686464309692, -0.40798234939575195, 0.9429550170898438, 0.6141462922096252, -2.1095385551452637], [1.0215671062469482, -0.5828931331634521, -0.5201539397239685, 0.0308767631649971, -0.3247712552547455, 1.278899908065796, -1.0709683895111084, -0.26661691069602966, 1.1282427310943604, 0.34752964973449707, 1.8058815002441406, 0.07871302217245102, 0.8653793334960938, -0.0673564001917839, 0.36629992723464966, -0.3467042148113251, 0.7977443933486938, 1.364195704460144, 0.10457077622413635, 0.04742134362459183, 1.1095737218856812, 0.5752177834510803, -0.39896339178085327, -0.87786465883255, 1.1282458305358887, -1.7970753908157349, 0.07753129303455353, -0.4324410557746887, -0.49828261137008667, -1.0224571228027344, 1.6842122077941895, -0.7420973777770996, 0.6279891133308411, -0.08613843470811844, 0.3008069396018982, 0.11021753400564194, -0.34278935194015503, 0.19755586981773376, 0.7127171754837036, -1.4475117921829224, -0.1293497532606125, -1.9572218656539917, -0.43760448694229126, -1.6126762628555298, -0.39111945033073425, -0.8975681066513062, -0.9204815626144409, -0.7750251293182373, 0.3978382647037506, 1.0398982763290405, 0.01878098025918007, 1.3742625713348389, -0.15942545235157013, -0.977669358253479, 1.884368658065796, -0.2536803185939789, 0.9601215720176697, -0.43325355648994446, 0.10659908503293991, 1.8601033687591553, 1.9875128269195557, -0.05885491147637367, 0.6761244535446167, 1.173096776008606, 0.5143926739692688, 0.810496985912323, -1.958088755607605, 1.2632933855056763, 0.8153319358825684, 0.49593111872673035, -0.39626389741897583, -1.4939943552017212, -1.1845422983169556, 2.0743565559387207, 0.11387378722429276, 1.5524420738220215, -0.4873746633529663, 0.6291619539260864, 0.8124887943267822, 0.4190819561481476, 1.4962316751480103, 0.08256836235523224, 0.8393498659133911, 2.628943681716919, 0.11189473420381546, -0.7799556255340576, -1.1641916036605835, -0.6583524942398071, -2.721520185470581, -1.440450668334961, -0.010740133002400398, 0.14926289021968842, -1.3997608423233032, -1.0000181198120117, -0.3493894934654236, -0.4026288390159607, 0.9869698286056519, -0.6527521014213562, 0.44048044085502625, -0.0031046601943671703, 0.1813768446445465, -1.068074107170105, -0.4161854386329651, 1.038633108139038, -1.6612045764923096, -0.38773247599601746, 0.4139697849750519, -0.21436791121959686, -0.48990094661712646, -0.008420728147029877, 2.456726312637329, -0.37167298793792725, -2.06691312789917, 0.6219619512557983, 0.14944225549697876, 1.8438456058502197, 1.6064544916152954, -0.5697038173675537, -0.02442125417292118, 2.2829720973968506, -0.6968370676040649, -1.3552417755126953, 0.898928165435791, 0.0744912177324295, 1.653701663017273, 0.7842070460319519, 0.7990500330924988, -0.6428442597389221, 0.8593292236328125], [0.41782036423683167, -0.15025761723518372, 1.1805059909820557, 0.768883466720581, 0.8130726218223572, -0.10654432326555252, 1.9418872594833374, -0.7748186588287354, -2.3794877529144287, 0.41429606080055237, -0.4415176212787628, 0.14769363403320312, 0.8177051544189453, -0.3569832146167755, -1.4963557720184326, 1.3556506633758545, -1.2785753011703491, -0.3018350601196289, 0.20838536322116852, 2.5459303855895996, 0.9121865034103394, -1.4768900871276855, 0.5964329838752747, 1.0736409425735474, 1.4506601095199585, -0.1213664561510086, 0.5030561685562134, -0.14181041717529297, -0.8822121620178223, -1.6253488063812256, -0.6135810613632202, -1.3272731304168701, 0.08212056756019592, 0.41566967964172363, 0.03013196960091591, -0.31287142634391785, 0.7498694658279419, -1.2156145572662354, -0.22761285305023193, -0.1629621833562851, 0.21850906312465668, 0.49910202622413635, -0.9894221425056458, 0.6027058362960815, 1.7532809972763062, 0.6124544143676758, -0.27758118510246277, -0.9951600432395935, -0.5395757555961609, 0.5816071629524231, -0.6782641410827637, 2.5674502849578857, 0.11652176082134247, 0.7211275100708008, -0.6960243582725525, 0.48344793915748596, 0.5318186283111572, -0.8539784550666809, 1.988301396369934, -0.90622878074646, -0.18339627981185913, -0.9526548385620117, 1.9690392017364502, 0.6545661091804504, -1.2659308910369873, -0.7828000783920288, -2.62650203704834, -0.9834199547767639, 1.514272689819336, -0.5962434411048889, 0.6757237315177917, 0.9822320342063904, -0.27683526277542114, -1.226968765258789, -0.865183413028717, 0.25246474146842957, -0.4748442769050598, 0.6980760097503662, 0.9615264534950256, -0.9689377546310425, -0.15317341685295105, 0.43465253710746765, 0.03243952617049217, 0.26916709542274475, 1.24784255027771, 2.44112491607666, -0.7026314735412598, -0.17476752400398254, 0.37940025329589844, -0.2649981677532196, -1.3514268398284912, 0.13101573288440704, -1.5384703874588013, -1.117448329925537, -0.500970184803009, 0.16421282291412354, -0.6479650139808655, 0.2361510545015335, -1.1415578126907349, 0.8107641935348511, -0.46860358119010925, -1.034205436706543, 0.6841105222702026, -1.1224701404571533, 0.33741816878318787, 0.05628439038991928, 0.9826987385749817, 0.42101940512657166, -0.40898796916007996, 1.0193511247634888, -0.22729860246181488, -0.24087007343769073, -0.7394981384277344, 0.2855895757675171, -1.9244372844696045, -0.5623243451118469, -0.15249527990818024, 0.530335009098053, 0.48514503240585327, 0.048343997448682785, 1.8975374698638916, -1.8210612535476685, 0.6809784770011902, 1.342366337776184, -1.9911952018737793, 0.13228243589401245, -0.5319203734397888, -0.7217175960540771, 0.18093062937259674], [-0.9164384603500366, -0.5745624899864197, -2.392605781555176, 0.536605954170227, -0.6273113489151001, 0.08243981748819351, -0.7966631650924683, 0.2770564556121826, -0.9088327288627625, -1.8569048643112183, 0.6862818002700806, -1.3249636888504028, -1.5485085248947144, 0.010212195105850697, -1.3651599884033203, 1.0697660446166992, -0.9474296569824219, 1.450102686882019, 1.2274261713027954, 0.30065596103668213, -1.1050500869750977, 0.1879889816045761, 0.8241652846336365, 0.11522723734378815, -1.2632949352264404, 0.6044777631759644, -1.065274715423584, 2.271944761276245, 0.896432101726532, 0.6624699831008911, -0.684482753276825, -2.4326095581054688, 1.1256685256958008, -1.6610713005065918, -0.41679874062538147, 1.5095930099487305, 0.7490254044532776, -0.8884343504905701, 0.6507130265235901, 0.3089485466480255, 1.5311003923416138, 0.1911787986755371, 1.4126442670822144, -0.5721531510353088, -0.7871204018592834, 0.32469701766967773, -3.1955628395080566, 1.3943884372711182, -1.5119961500167847, -0.10197590291500092, -1.2468127012252808, -1.4139610528945923, -0.22673875093460083, -0.6360665559768677, 0.725295901298523, 1.7397375106811523, -2.5467216968536377, 0.8399766087532043, 1.1296184062957764, 0.5044996738433838, -0.9227414727210999, 0.7896206378936768, 1.05710768699646, 0.2110324203968048, -0.628266453742981, -0.43360766768455505, 1.4372785091400146, 0.3693285584449768, -1.0673222541809082, -0.6176851391792297, -0.8607640266418457, -2.601377487182617, 0.5224255919456482, 0.6251294612884521, -2.096290349960327, 0.6481648087501526, -0.11624351143836975, 0.9782682061195374, -0.07783590257167816, -0.14978010952472687, -0.7618150115013123, -1.1421853303909302, -1.0257554054260254, 0.5842311382293701, -0.4286399483680725, 0.24967962503433228, 0.8798432350158691, 0.48173248767852783, -0.9381702542304993, -2.480966806411743, -1.629262924194336, 0.6529186964035034, 1.0590906143188477, -0.743204653263092, -0.008167736232280731, -1.1677213907241821, -0.8826320767402649, 0.29150035977363586, -0.0010336999548599124, -0.9163383841514587, -1.4719256162643433, 1.0562015771865845, -1.7655935287475586, 0.7250503897666931, -0.444132000207901, 0.2899888753890991, -0.7221717834472656, 0.8240106105804443, -2.038133144378662, 0.47804445028305054, 1.8540012836456299, -0.7060512900352478, -1.5868496894836426, 0.8538178205490112, -0.1842249631881714, -0.9949825406074524, -0.24842381477355957, 0.3109948933124542, -0.9952545166015625, -1.4460773468017578, -0.16716639697551727, 0.051917191594839096, 1.1180733442306519, -0.645746111869812, -0.8076349496841431, -0.3422519564628601, -0.3242065906524658, -1.8442391157150269, 0.665514349937439], [1.768340826034546, 0.06368878483772278, 0.3427239954471588, -1.1497015953063965, -0.7088816165924072, 0.9297003746032715, 0.9343710541725159, -0.21953442692756653, -0.1125921681523323, 0.7021365761756897, -0.29264751076698303, 0.9078977704048157, 1.2514888048171997, 1.336828351020813, -0.39170390367507935, -0.05872361734509468, 0.26865464448928833, 1.3574509620666504, 0.6599198579788208, 0.16110414266586304, -1.0652369260787964, 1.7361284494400024, 0.6653078198432922, -1.0391662120819092, 2.512280225753784, -0.11655281484127045, -0.8218103051185608, -2.771202802658081, 0.0839034914970398, 0.013324065133929253, -0.7953284978866577, 0.2301565259695053, -1.1786961555480957, 1.3411813974380493, 0.08191405236721039, -0.344549298286438, -0.23671863973140717, 0.1424793303012848, -0.6012848019599915, -0.6212847828865051, 0.7440680265426636, 0.7126022577285767, 0.12446200847625732, 0.26836881041526794, 1.554226279258728, -3.6104018688201904, -0.17832930386066437, 0.6829771399497986, -0.9493353366851807, 0.6969062089920044, 0.9433870911598206, -0.7070347666740417, -0.9400116205215454, -0.8745729327201843, -0.3057137131690979, 1.4776108264923096, 0.14627760648727417, -0.5871888995170593, -0.44052886962890625, 1.6609718799591064, 0.31283000111579895, -0.18313029408454895, -0.8845687508583069, -0.23863132297992706, 0.8890230059623718, 0.5647794604301453, -0.3431461453437805, 0.3754824101924896, -0.9099766612052917, 0.15890461206436157, 1.4394378662109375, -1.539222240447998, 0.7634227871894836, -0.2991471588611603, -1.6517529487609863, -0.10630016028881073, -1.7743440866470337, 0.2786315381526947, 0.271010160446167, -0.8401474952697754, 0.5687798857688904, -0.7253222465515137, 0.6101966500282288, 0.01673547737300396, 0.1594182252883911, -0.5996204614639282, 0.7107044458389282, -0.011587535962462425, -1.4411836862564087, -0.13501639664173126, -0.9234970211982727, -1.1258033514022827, -0.6917172074317932, -0.056293386965990067, -1.8063617944717407, -1.4774730205535889, 1.4743174314498901, -0.22971664369106293, -0.9687507748603821, 0.3635469079017639, -1.0270819664001465, 0.34209147095680237, -1.3246856927871704, -0.6715902090072632, 0.27062490582466125, 0.05620478093624115, 0.12832440435886383, -1.4298423528671265, 0.9120498299598694, 0.3484729528427124, -0.32299959659576416, 1.5893868207931519, -0.7067492008209229, -0.8148762583732605, -0.06652723997831345, 1.1090550422668457, -0.6137968897819519, 1.017696738243103, 1.0861531496047974, -1.0781525373458862, 0.5361365079879761, 1.4565460681915283, -0.6682038307189941, 1.7913771867752075, 1.2632651329040527, 0.690018892288208, 1.5274782180786133, 1.3478847742080688, -0.017275243997573853], [0.9351662397384644, 2.6494245529174805, -0.660222589969635, -0.021943237632513046, -0.812468945980072, -1.9349886178970337, 0.3734210431575775, 1.1320068836212158, -0.48093685507774353, -0.15582355856895447, -0.045881979167461395, 0.050006378442049026, 0.11816175282001495, -0.3215041160583496, 0.8619288206100464, 0.3504825532436371, 0.5011704564094543, 2.0418102741241455, -0.233751580119133, -0.44725897908210754, -1.385072946548462, -1.2955083847045898, 1.5390636920928955, 0.13353800773620605, -1.7695131301879883, 0.002181168645620346, 1.0243843793869019, 0.6156736016273499, 0.6260637044906616, -0.05567753687500954, -1.7762014865875244, -1.0318284034729004, 0.8503369688987732, -0.37454232573509216, 0.022842582315206528, -1.0946451425552368, 0.736090362071991, 3.065762996673584, 0.5701425671577454, 0.5637856125831604, -0.1838710904121399, -0.3239375948905945, -1.1014114618301392, 1.5076663494110107, 1.0219920873641968, 0.23406310379505157, -0.7075572609901428, 2.159217596054077, -0.5964059233665466, -0.08222782611846924, -0.9816493988037109, -0.6186828017234802, 1.2718695402145386, -1.1081526279449463, 1.9738181829452515, 0.38444578647613525, -0.5401260852813721, 0.3896358013153076, 0.2941003441810608, -1.2532320022583008, -1.1311094760894775, 0.9280825853347778, 0.6497708559036255, 0.052015967667102814, 0.8131851553916931, 0.9109431505203247, 0.5691210031509399, -2.0552639961242676, -1.1734960079193115, -0.38315799832344055, 0.9364412426948547, 2.003032684326172, 0.015182300470769405, -1.055108666419983, 0.7272108793258667, 0.7221706509590149, -0.34380483627319336, 0.9856283068656921, -1.202531099319458, 0.160002201795578, 1.5315436124801636, -0.797831118106842, -0.32968300580978394, -0.2090078443288803, -1.5146098136901855, -0.9134178161621094, -2.2732677459716797, -0.712500810623169, 0.4285746216773987, 0.5641263723373413, -0.12478093057870865, 1.3280653953552246, -0.17843028903007507, -0.0036683958023786545, 2.024559736251831, -0.8743436932563782, -0.4933460056781769, -0.1596030294895172, 0.7293304204940796, -0.49035876989364624, 1.9660075902938843, -0.24465268850326538, 0.1987144500017166, 1.0505846738815308, 2.216513156890869, 0.2346414476633072, 0.04243970289826393, 0.6938872933387756, -0.8774299025535583, 0.9033805727958679, -0.6250728368759155, 0.17996054887771606, 0.7108825445175171, 0.584444522857666, 0.5782365202903748, -1.1677606105804443, -1.2868622541427612, -0.7663718461990356, -0.769783079624176, 0.94886314868927, 0.6030462384223938, 0.5065628886222839, -0.7044646739959717, -0.24651576578617096, -1.9970380067825317, 0.7458423972129822, 1.1105118989944458, 0.6283413767814636, -0.6101975440979004]], "dtype": "float32"}, {"__nd__": [[-2.156517267227173, 0.3241230845451355, -0.8824878931045532, -1.0449661016464233, -0.5935729146003723, -0.9571112394332886, -0.9503592848777771, 1.0013079643249512, -0.2629750072956085, 1.0072745084762573, 0.2977716624736786, 0.2605355381965637, 0.9779057502746582, 0.05008907988667488, -1.710054636001587, -0.9467192888259888, -0.9206634759902954, 0.6113236546516418, 0.8251509666442871, 0.8741735816001892, -0.5453457236289978, 0.6374669671058655, -0.09698300063610077, 0.9362103343009949, 1.119520664215088, -0.76059490442276, -0.7236656546592712, -2.0690808296203613, 0.3247920274734497, -0.7483091950416565, 0.00016326540207955986, 1.5378211736679077, 1.3709477186203003, 0.8211492896080017, -1.4871565103530884, -0.1986607015132904, 0.41705122590065, -0.35354647040367126, -1.4558343887329102, 1.285374402999878, -1.206912636756897, 1.1984213590621948, 0.33766090869903564, -1.443129062652588, 1.157934308052063, -0.41801634430885315, 1.4338167905807495, -0.023035243153572083, 2.4787914752960205, -0.28694161772727966, 0.5680321455001831, 0.7824990749359131, -0.6938170194625854, -1.3254073858261108, 0.6584542989730835, 0.45569920539855957, 0.6132671236991882, -1.4811205863952637, -0.34784090518951416, 1.0117319822311401, -0.1874658167362213, 1.094260334968567, 1.052512288093567, 2.573157548904419, 1.471413493156433, 0.6850629448890686, -0.08410250395536423, 0.9391445517539978, 1.017259120941162, 0.07774077355861664, 1.614912986755371, 1.4769679307937622, -0.4476088881492615, 0.43135204911231995, 0.9030011296272278, 0.49404728412628174, -1.2590669393539429, -1.139180302619934, 0.4488352835178375, -0.7002373337745667, -0.24387601017951965, -0.4183296263217926, -1.2076457738876343, -0.030577346682548523, -1.5321978330612183, 1.990964412689209, -1.5015579462051392, 0.4904072880744934, 0.2606671452522278, 0.36782512068748474, -0.9077161550521851, 1.8308314085006714, -0.28977352380752563, 0.4155714213848114, -1.9733003377914429, 1.3925834894180298, -0.4303765296936035, 0.5506808757781982, 1.0426840782165527, -0.5924205780029297, 0.5692756772041321, -0.6353093385696411, 0.22725734114646912, 1.0146167278289795, -2.137427806854248, 0.6494203805923462, -2.7122042179107666, -1.1821054220199585, 0.14599551260471344, -0.04337020963430405, 0.28781720995903015, -1.1290258169174194, 0.36472490429878235, 0.12494295090436935, -1.5955219268798828, -0.3399372398853302, 0.01770421676337719, -0.902285099029541, 1.4905787706375122, -1.7372039556503296, 0.359496533870697, 0.8003766536712646, 0.7669192552566528, 0.25300678610801697, -0.5779799222946167, -0.8026642203330994, -0.6199278831481934, 0.19622723758220673, 0.8322557210922241], [0.32187581062316895, -0.6943149566650391, -1.966619849205017, 0.30747780203819275, 0.09179948270320892, -1.819883942604065, 0.09437263011932373, -0.3103765547275543, -0.9717881679534912, -0.7989584803581238, -0.029602214694023132, -0.920173168182373, 1.147204875946045, 0.9523250460624695, 0.027015002444386482, 0.7005879282951355, -0.11520202457904816, 1.0990079641342163, 2.003899574279785, -0.08342122286558151, 0.2711433470249176, -0.5265628099441528, -1.7568047046661377, -0.24964125454425812, 1.511162519454956, 1.8047279119491577, -1.7566241025924683, 0.9233986139297485, 1.6701445579528809, -0.19114571809768677, 0.31914207339286804, -1.3811582326889038, 0.6945887804031372, -0.7574515342712402, 1.164732575416565, 0.17342790961265564, -1.1538046598434448, -0.5203425884246826, 0.28570225834846497, -1.1546621322631836, -0.7352486252784729, 0.8385207653045654, -0.6345287561416626, 1.0967960357666016, 1.162827968597412, -0.6228598952293396, 0.4076719880104065, -0.9891573786735535, -0.6257714629173279, 1.7790305614471436, 1.509360432624817, -0.2352457493543625, -0.6393462419509888, -0.3087100386619568, -0.09559712558984756, -0.7476407885551453, -0.19610156118869781, -0.4665113091468811, 0.28788337111473083, 0.7424249053001404, -0.1536848396062851, -1.7764278650283813, 0.723503053188324, -0.35645219683647156, -0.5662477612495422, -0.35907480120658875, -0.749799370765686, 0.45900022983551025, -1.8743855953216553, -0.4700090289115906, -0.065679632127285, 0.08829089254140854, 1.088212251663208, -1.5297445058822632, 1.948277235031128, -0.6483039259910583, -2.3588874340057373, 0.9503591656684875, 0.95123291015625, 0.6541855335235596, -1.2895455360412598, -1.5359147787094116, -0.2502487897872925, 0.42857077717781067, -0.09751814603805542, -1.3492932319641113, 0.4457424283027649, -0.12894266843795776, -0.7798683047294617, 0.4697830080986023, -1.154404878616333, -0.46440085768699646, -0.5884549617767334, 0.24841651320457458, 0.3753826320171356, -1.3793489933013916, -0.30788102746009827, -0.9867569804191589, -0.21833856403827667, 1.3138275146484375, 1.3354319334030151, 1.8134618997573853, 0.539960503578186, 0.38082394003868103, 0.7322282195091248, 1.558822751045227, -0.12716767191886902, -0.710443377494812, -0.54145348072052, 0.2885018587112427, 1.7673898935317993, -0.18158458173274994, 0.744057834148407, 2.0258045196533203, 1.2729625701904297, 0.5018383264541626, 2.2698240280151367, 0.026964545249938965, 0.16906926035881042, -0.46343016624450684, 0.34622722864151, 1.2430959939956665, 0.20708657801151276, 0.4371429979801178, -0.9718941450119019, 1.2202558517456055, -0.5262240767478943, 1.485227346420288, -0.8327706456184387], [-3.1079328060150146, -0.48734405636787415, 0.32086288928985596, -1.909243106842041, -1.1367042064666748, -0.2560507357120514, -1.4140263795852661, 0.564578115940094, 0.594038188457489, -1.114349126815796, 0.3605228066444397, 0.19851671159267426, -0.3949253261089325, 1.8598711490631104, 1.4823137521743774, -0.7251547574996948, -0.2789684534072876, 0.4607708156108856, 0.25204646587371826, 0.2826366126537323, 1.5959851741790771, 0.43821802735328674, 0.905861496925354, 1.002277135848999, -1.0437268018722534, 1.140201210975647, -0.04510319605469704, -0.2970755994319916, -1.2701388597488403, 3.3162293434143066, 0.7760102152824402, -1.0096529722213745, -0.14099998772144318, -0.5127200484275818, 1.370312213897705, -0.7037163972854614, 1.8301063776016235, -0.032780807465314865, 0.5962830185890198, -2.3733630180358887, -1.3376623392105103, 0.5868184566497803, 0.2502919137477875, 0.028613068163394928, 0.3434198200702667, -0.27833235263824463, -0.0939905121922493, 0.26384809613227844, -0.5011888742446899, -1.0534868240356445, -0.5520250201225281, -0.05873127281665802, -1.6471003293991089, -0.1543394774198532, 1.1376004219055176, -0.4480225741863251, 0.046910353004932404, 0.6102766990661621, -0.49856576323509216, 0.2909780442714691, -1.3907614946365356, 0.19735878705978394, -0.7140306830406189, -0.6668018102645874, -0.4795452952384949, -0.5186668038368225, 1.2342585325241089, 0.39453235268592834, -0.6339144110679626, 1.106930136680603, 0.5265787839889526, -0.13649486005306244, -0.6956413984298706, -0.5697960257530212, -1.166617751121521, 0.31312820315361023, 0.6427229642868042, -0.6114596128463745, 0.535487949848175, 0.5608883500099182, 0.5429816842079163, 1.438517689704895, 0.4709705710411072, 0.7129175066947937, 0.008255431428551674, -1.5115983486175537, 0.8266699910163879, -0.0681874230504036, 1.0975592136383057, -1.396496295928955, 1.1800838708877563, -0.46706846356391907, 0.31791508197784424, 0.43294957280158997, 1.8303943872451782, 1.5037826299667358, 1.578896403312683, -0.19831392168998718, -0.5397521257400513, 1.5386254787445068, 1.2193665504455566, -0.28117114305496216, 1.4292817115783691, 0.5903369188308716, 0.8852159976959229, -1.2678236961364746, -0.6443372368812561, 1.4040827751159668, 2.6673786640167236, -0.5621508955955505, -0.7780575752258301, -1.1452971696853638, -0.06548479199409485, -0.5132293105125427, 0.8953320384025574, -0.4965926706790924, 0.6451930403709412, -0.3333309292793274, 0.005662835668772459, 1.7818024158477783, 0.7336599230766296, 0.6270222067832947, -1.4539110660552979, -1.188853144645691, 0.9187856912612915, -0.11106599867343903, -0.03592275455594063, -0.07974221557378769, 0.9721066355705261], [1.5527198314666748, 2.917391538619995, 0.18349096179008484, 0.9396950602531433, -0.3261815309524536, 0.7773311138153076, 1.3605207204818726, 0.30615267157554626, 1.371312141418457, 0.537704586982727, 1.373274803161621, -0.39474332332611084, 0.6806076169013977, -0.4972878098487854, -0.42960548400878906, 0.17151156067848206, -1.3495951890945435, -0.09224171191453934, -0.36266082525253296, -0.44554445147514343, -0.18384458124637604, -1.4707317352294922, -0.3851175904273987, 0.2861502170562744, -0.9302260279655457, -0.34238627552986145, -0.48402032256126404, 0.07749976962804794, 2.4626824855804443, 0.24592696130275726, -0.03311428427696228, 0.0037410908844321966, -1.240366816520691, -1.822493553161621, 1.031906008720398, -0.8064255714416504, -1.7606539726257324, -0.6055612564086914, -0.5122976899147034, 0.01462998054921627, -1.0295052528381348, -0.10075878351926804, 1.0356383323669434, -1.9631403684616089, -1.5556447505950928, 0.6333372592926025, -1.8800791501998901, 0.603318452835083, -1.3440032005310059, 1.5968246459960938, 0.006096153985708952, 0.14490199089050293, -0.754791796207428, 2.027749538421631, -1.5197616815567017, 0.8179447054862976, 0.21588002145290375, -1.8889094591140747, 0.4749407172203064, 1.1186763048171997, -1.1070412397384644, 0.10764039307832718, 0.8198376297950745, -0.41665852069854736, -1.1231071949005127, 0.019961804151535034, -0.5121226906776428, 0.2690865099430084, -1.8796113729476929, 0.3096696138381958, 0.6217873096466064, -0.6580482125282288, 0.1705167144536972, -1.0912044048309326, -0.5858761072158813, -0.24834685027599335, -0.8436959385871887, -0.666362464427948, 0.0051344409584999084, -1.3839366436004639, 0.43877896666526794, 0.8061054348945618, 0.855334460735321, 0.6187517642974854, 0.8930249214172363, 2.3353271484375, 2.2811501026153564, -1.5304380655288696, 0.33377981185913086, 1.6714668273925781, -0.3251066505908966, -0.04843112453818321, 0.5003566145896912, 1.6500999927520752, 1.5847700834274292, -0.5799267292022705, 2.1401162147521973, -1.0284844636917114, -0.3630318343639374, -1.6226825714111328, -0.26808738708496094, 1.6731699705123901, 0.8898245096206665, -0.9106622338294983, -0.1398528814315796, 1.7171037197113037, 1.1626670360565186, -0.13843312859535217, 0.4405575394630432, -0.27632060647010803, -0.15481774508953094, -1.3242218494415283, -0.4282546043395996, 0.7414487600326538, 0.6995221376419067, 1.1365740299224854, 0.31210553646087646, 0.6540395617485046, 1.5650204420089722, -1.1529216766357422, 0.7097703814506531, -0.2560294568538666, 0.014531495049595833, 0.8513532280921936, 0.4910765290260315, 0.1037459596991539, 1.078261375427246, -0.8871005773544312, -0.868274450302124], [0.4113881587982178, 1.244510531425476, 1.3530129194259644, 2.306750774383545, -1.0749567747116089, 0.297035813331604, 0.2505899965763092, 1.5203192234039307, -1.161811113357544, 0.020793510600924492, 2.2136895656585693, 0.21497179567813873, 0.8717581033706665, 0.8702593445777893, 0.7719327211380005, 0.9394018650054932, -0.9272505640983582, -0.5258294939994812, 0.8830236792564392, 1.6598440408706665, 1.8712178468704224, -1.1795296669006348, -0.3397279381752014, -1.5387479066848755, 0.15070289373397827, 0.7232537865638733, 0.8028202056884766, -1.2010102272033691, 0.1881610006093979, -0.2205607146024704, 0.045836761593818665, -0.990081250667572, -0.080754853785038, -0.5606083273887634, 0.3284159004688263, 0.4442996680736542, 0.09067743271589279, 0.20268955826759338, 0.4989232122898102, -0.5120779275894165, -1.0756603479385376, -0.09570018202066422, 0.5999301075935364, 0.8465111255645752, 1.1435980796813965, 0.3338474631309509, 0.4004364609718323, -2.2877984046936035, -1.9924927949905396, -1.4742326736450195, -1.4457502365112305, 0.800873875617981, -0.8860876560211182, -0.9868930578231812, 1.8371374607086182, 0.121918223798275, 0.6424650549888611, -1.4584180116653442, -0.8388797640800476, -0.6027052998542786, -0.8011304140090942, 0.959787905216217, 0.43920648097991943, 0.3971061110496521, 0.7743354439735413, 0.2092025727033615, 0.14678427577018738, 1.3035686016082764, 1.276749610900879, -0.17956231534481049, -0.3514092266559601, 1.3049213886260986, 0.4848801791667938, 0.2114647626876831, 0.23760415613651276, -1.5813555717468262, 1.602563738822937, -1.8191181421279907, -1.3041231632232666, -0.5058580040931702, -0.7003851532936096, 0.8242469429969788, 0.9482978582382202, -1.2640522718429565, 0.11347003281116486, 1.1465275287628174, 0.7036855816841125, 0.8558096289634705, 0.8909125328063965, -0.03372696787118912, 0.008853599429130554, 0.14979645609855652, -0.6178175806999207, -0.7523664832115173, 1.8008300065994263, 1.0632410049438477, 1.3734465837478638, 0.6089164614677429, 1.140488624572754, -0.48223528265953064, 0.277805894613266, 0.0027849681209772825, -1.938084602355957, -0.7077419757843018, 0.215805321931839, 0.4204227328300476, 1.981723666191101, -0.2808585464954376, 0.631615936756134, -1.2123526334762573, -0.6347520351409912, -1.1961008310317993, 0.4888184070587158, -0.6712307333946228, -1.22274911403656, -0.9067338705062866, -1.5521918535232544, -0.81240314245224, 0.4524356722831726, 0.40854430198669434, 0.09174779802560806, -1.0026061534881592, 1.1338969469070435, 0.41559791564941406, 0.776532769203186, -0.7349911332130432, -1.320426106452942, 0.7108857035636902, -2.1155500411987305], [-0.9736403226852417, 0.11757408827543259, -0.32051724195480347, 0.8318692445755005, 0.008506247773766518, 0.904114305973053, 1.1282691955566406, -1.7025748491287231, 2.120049476623535, -1.873397707939148, -2.6558265686035156, -0.5662961006164551, 0.11975839734077454, -0.6771590113639832, 0.3863142728805542, -0.6273524165153503, 0.8842067718505859, -0.8602122068405151, -0.5484941005706787, -0.7495026588439941, -0.6140947341918945, 0.8061962723731995, -0.251338928937912, 0.5063872933387756, 0.4911234378814697, -0.2383178472518921, 1.4235193729400635, 1.1102969646453857, -2.8610892295837402, 0.8258870244026184, 0.3335627019405365, 0.7223119139671326, -0.23059475421905518, -0.48335784673690796, 1.2252851724624634, 0.8960891366004944, 0.6948330402374268, 2.413896322250366, 0.7957429885864258, -0.0028466125950217247, -1.1750469207763672, -0.8024425506591797, -0.3458448648452759, -0.42860838770866394, 1.1176128387451172, 0.3171752691268921, -0.34002381563186646, 1.0776689052581787, 1.0530227422714233, -0.11803432554006577, -0.40466755628585815, -0.5009482502937317, -1.566401481628418, -0.30272045731544495, -1.4545620679855347, -1.4869394302368164, 0.5292728543281555, 0.26315638422966003, 0.6296302676200867, -1.1812940835952759, 1.269707202911377, 0.20265568792819977, -0.49558398127555847, 0.5513110160827637, -0.6173921823501587, 0.23173457384109497, 0.6893539428710938, -1.2409605979919434, -0.24296312034130096, 0.7638493776321411, 1.1258097887039185, -1.137572169303894, 2.104078769683838, 0.22486677765846252, 1.8071211576461792, 1.1004304885864258, -0.8041210770606995, 0.3530798852443695, -0.022227870300412178, 0.17262667417526245, 0.22517773509025574, -1.7121155261993408, 1.3187761306762695, -2.0709922313690186, 1.0214569568634033, 0.5337787866592407, 0.7727752923965454, -0.1727391928434372, 0.6319887042045593, 1.2843852043151855, -1.1112492084503174, -2.0983872413635254, 1.4828606843948364, 0.06209301948547363, -1.3589473962783813, 2.547032594680786, -0.11626102030277252, -0.038107845932245255, -0.1134750097990036, -0.8922857046127319, -0.5211841464042664, -1.0990251302719116, 2.2064571380615234, -1.7873022556304932, 0.33753702044487, -1.3674222230911255, 0.6878218650817871, -0.8808339834213257, 1.2896569967269897, -2.2113399505615234, -1.1995720863342285, -0.6565348505973816, 0.5812756419181824, -0.7029352188110352, 1.4895607233047485, -1.1491262912750244, 0.37422260642051697, -0.09523345530033112, 0.18325835466384888, 2.09433913230896, 0.5394462943077087, -0.3584773540496826, 0.14413900673389435, -0.5034589171409607, -1.258548378944397, -0.7508449554443359, -0.16767169535160065, -0.9037880897521973, 0.07213862240314484], [-0.8687374591827393, 1.024886131286621, -0.980078399181366, -0.39935728907585144, 0.6902446746826172, 0.5697624087333679, -1.1259413957595825, 0.2866190969944, 1.1931228637695312, -1.261877179145813, -1.1881362199783325, 0.5839736461639404, -0.12242651730775833, 1.607139229774475, 1.2205817699432373, -1.0284658670425415, 1.2911182641983032, -0.26865267753601074, 1.3768482208251953, 0.3012779653072357, 1.7776539325714111, 0.5223000645637512, -0.2577041983604431, 0.9145663976669312, -0.41196244955062866, 0.26095086336135864, 1.513932466506958, 0.25674793124198914, 0.4475034177303314, 1.0611642599105835, -0.6960613131523132, -0.08607003092765808, -0.3832985460758209, -1.3636143207550049, 0.5128402709960938, -0.8559954166412354, 1.533261775970459, 0.23596352338790894, -1.6440359354019165, -1.3013888597488403, 1.6614629030227661, 1.3172080516815186, -1.6191328763961792, -0.2602270543575287, -0.879420816898346, -0.7341009974479675, 0.6841968297958374, 0.7403461337089539, -1.6415677070617676, -0.3300321400165558, -0.950890302658081, -0.21485312283039093, 0.4303237497806549, 1.1348538398742676, 0.18189731240272522, 1.9451470375061035, 1.2937365770339966, -0.3068552017211914, -0.25833359360694885, -0.9758636355400085, -0.7940244078636169, 0.527328610420227, 1.1489495038986206, -0.5112765431404114, -0.3605284094810486, 1.0632274150848389, -0.2566727101802826, -2.5437355041503906, 2.0900537967681885, -0.29900747537612915, -0.4957587420940399, 0.5918807983398438, 0.04944109171628952, -1.0880383253097534, 0.6419545412063599, 0.3165299594402313, 0.6809157133102417, 0.16410934925079346, -1.2752753496170044, 0.7566460371017456, 0.21136246621608734, -0.17009522020816803, 0.8525427579879761, 1.3976082801818848, -0.6447171568870544, -0.20222903788089752, 0.9852817058563232, -0.7712266445159912, -0.24538882076740265, 0.44113561511039734, 0.4550435245037079, 0.7852855324745178, -1.0565987825393677, -0.5871151089668274, -0.0707867294549942, -0.958567202091217, 0.6769282221794128, 1.1959067583084106, 0.9971272945404053, 1.215451955795288, -1.5737160444259644, -1.0837346315383911, -0.419307142496109, -0.576736569404602, -1.4137051105499268, -0.380951464176178, -0.4587177336215973, -0.35839512944221497, 2.2638046741485596, 1.6381347179412842, 0.6281175017356873, -0.02754657343029976, -0.1957879215478897, 1.3541008234024048, -0.6598033308982849, 0.9447857737541199, -0.14258016645908356, -0.22677910327911377, -0.41164150834083557, 1.0564407110214233, 0.9443035125732422, 1.3182569742202759, -1.5990427732467651, -1.817553162574768, -0.34819692373275757, -0.5042816996574402, -1.1018083095550537, -0.9648870825767517, 0.45704740285873413], [-0.4282017946243286, 0.2255481481552124, 0.6279487609863281, -0.5900020599365234, 0.0895419716835022, 1.1149067878723145, -0.5710171461105347, -1.5531829595565796, -0.001493290881626308, -1.7282084226608276, -1.0965880155563354, 3.064035177230835, 1.913156270980835, -0.14039263129234314, -1.2494685649871826, -1.8893165588378906, -1.8318092823028564, -1.0252583026885986, 0.3727453947067261, 1.2059825658798218, 0.3074088990688324, -0.09298571944236755, -0.7371255159378052, 0.43753355741500854, -1.2014472484588623, 1.2359528541564941, -0.48406174778938293, 0.11014451086521149, -0.5657786130905151, -0.1713535487651825, -0.2599743604660034, -0.5736974477767944, 0.14068126678466797, -0.8964188098907471, 0.03708915784955025, 0.009483530186116695, -0.6193124055862427, 1.512822151184082, -0.6091604232788086, 2.021442174911499, 0.008196592330932617, 1.0752232074737549, 0.26404833793640137, -0.1882251501083374, 0.5295013189315796, 1.1208192110061646, -0.3429640233516693, -1.1472707986831665, 0.7821429371833801, -0.10132286697626114, -1.4533040523529053, -1.582688331604004, -1.207015037536621, 0.7591037154197693, 0.24103541672229767, -0.46746671199798584, 0.28040045499801636, 0.21134118735790253, -0.4774039089679718, 0.8470893502235413, 1.0335389375686646, -0.522559642791748, -0.6177982091903687, 0.16498811542987823, 0.2898934483528137, -0.7677755951881409, 2.7092530727386475, -1.0742946863174438, -1.0631743669509888, 1.4494853019714355, -0.219199538230896, -1.1547234058380127, -0.8064871430397034, -0.7020229697227478, -0.5672829151153564, -1.3928053379058838, -1.523992896080017, -1.5014113187789917, -0.5938550233840942, -0.6838884353637695, -1.327751874923706, -0.5187649726867676, -0.5973935127258301, 0.4232906103134155, -1.3712719678878784, 1.0912120342254639, 0.5165106058120728, -0.5057746171951294, -0.7492393851280212, 1.2138769626617432, -0.6689346432685852, 0.3497919738292694, 0.7448425889015198, 0.22035598754882812, 0.8310138583183289, 0.99033522605896, -1.7400637865066528, 0.031046250835061073, -0.24273613095283508, -0.481954425573349, 0.09609818458557129, 1.6689997911453247, -0.12735314667224884, -1.6428756713867188, 0.27198854088783264, 0.43349868059158325, 0.9369868636131287, -1.9995396137237549, -0.8480945229530334, 0.9880991578102112, 0.19088679552078247, -1.8926753997802734, -0.7645321488380432, 0.2849487364292145, -0.037864189594984055, -0.4223235547542572, 0.48815831542015076, 0.27032366394996643, 0.5539883375167847, -1.0446867942810059, 1.3950583934783936, 1.473717451095581, 1.144047498703003, -1.448974847793579, 0.03432139754295349, 0.7330610156059265, 1.4040613174438477, 0.4320148229598999, 0.3232211470603943], [-1.2188379764556885, 0.015689950436353683, -0.5583189725875854, 0.10449293255805969, -0.5424097180366516, 1.51772940158844, 0.8739154934883118, -0.711879312992096, 1.7189271450042725, -0.8330339789390564, -0.10268665105104446, 0.2661206126213074, -1.3928040266036987, 0.7000150084495544, -0.6171177625656128, 1.5830528736114502, -0.6964181065559387, 0.23479188978672028, 1.8046259880065918, -0.8696709871292114, -0.8983903527259827, -1.1067010164260864, 0.7372325658798218, 0.5798996686935425, -1.6369596719741821, 0.3078010082244873, 1.2760509252548218, -0.3133314549922943, -1.7622997760772705, 0.3024289309978485, -0.7933565378189087, -1.4091997146606445, -0.4696517288684845, 1.079756259918213, 0.46451297402381897, -0.6074386239051819, 1.1963813304901123, 1.0669127702713013, 0.4104260206222534, 1.074846863746643, -0.6578472852706909, 0.8370275497436523, 0.12942729890346527, -1.300298810005188, -0.09181808680295944, -0.01623435690999031, 0.018878912553191185, -0.9838102459907532, 0.7804219722747803, 1.7805733680725098, 1.513224482536316, -1.4065693616867065, 0.9597275853157043, 1.012242078781128, 0.24332410097122192, -1.3636603355407715, -0.20717062056064606, 0.30548837780952454, -1.1714568138122559, -0.8711674213409424, -1.38009512424469, -0.305218368768692, 0.5140596628189087, -0.6128576993942261, -1.126423954963684, 0.5210897326469421, 0.030077144503593445, 0.8222514986991882, 1.4173774719238281, 2.4024229049682617, 0.23761963844299316, -1.6469582319259644, 0.10952939838171005, -0.3306736648082733, -0.429475873708725, -0.6712717413902283, -0.9635369777679443, -1.4143882989883423, -0.16876955330371857, 0.9844585061073303, -0.5835234522819519, 0.689658522605896, 0.09913530200719833, 0.7305840849876404, 2.5771484375, 0.4478469491004944, 0.06115652620792389, -0.5159522891044617, -0.30587348341941833, -1.3904321193695068, 1.5982829332351685, -0.4742320775985718, 0.5952174663543701, -0.10032425075769424, 0.4080059230327606, 1.9441852569580078, 0.5696533918380737, -0.6547813415527344, 0.681268572807312, 1.6739187240600586, -0.8818644285202026, -0.3771912753582001, 0.01224528532475233, -1.3561079502105713, 0.19270579516887665, -0.8992674946784973, 0.7611068487167358, -1.0708457231521606, -1.3790109157562256, 0.7989926934242249, -1.342532992362976, -0.6084902286529541, -1.6485669612884521, 1.1851170063018799, -0.5513442158699036, 0.7141819596290588, 0.3239167332649231, 0.35126301646232605, 0.9409881830215454, -0.7608264684677124, 0.0867820456624031, -0.5060021877288818, -1.3306832313537598, -0.710273802280426, 0.42800241708755493, -0.79813152551651, -0.26479974389076233, -1.3152170181274414, 1.0142813920974731], [0.37087106704711914, -0.6729502081871033, -2.168933153152466, -1.0498782396316528, 2.0699141025543213, -0.48689836263656616, 0.7043556571006775, 0.6705584526062012, 0.25509893894195557, -0.09409257769584656, 0.2155863493680954, 0.21884025633335114, -0.22340281307697296, -0.49581554532051086, -0.0013446984812617302, -0.5265392661094666, -2.8732738494873047, -2.2116897106170654, -0.5012702941894531, -1.754686951637268, 1.1212427616119385, -0.16278699040412903, -0.059948235750198364, 1.5278362035751343, 0.898442804813385, -0.6873694658279419, -1.370012879371643, 0.783309280872345, -0.21271498501300812, 0.22963638603687286, -1.8647253513336182, 0.6495600342750549, -0.0368160605430603, -1.8161839246749878, -1.5914580821990967, -1.3709369897842407, 1.6070337295532227, 0.9124236106872559, -1.3329088687896729, 0.5307177901268005, -0.12728740274906158, 0.2833351790904999, 0.1724693477153778, 1.1364210844039917, -0.06498876214027405, 0.8933969140052795, -0.6025539040565491, -0.29245343804359436, -0.898786187171936, 1.0749711990356445, 0.8123754262924194, 0.9260343313217163, 1.4675414562225342, -0.6637889742851257, 0.1934630423784256, -1.009036898612976, -0.08529866486787796, 1.0439563989639282, -0.5133758187294006, 0.6295558214187622, -0.19036626815795898, -0.2303849160671234, 0.4755576252937317, 0.6657066941261292, 1.6337782144546509, 0.2388940006494522, -1.661637544631958, 0.04559221491217613, 1.058205246925354, 1.074198603630066, 0.39698508381843567, -0.25334933400154114, -0.7358131408691406, -0.7554529309272766, 0.32007095217704773, -1.020870327949524, 0.5201053619384766, 1.3077702522277832, -0.6736489534378052, -0.6185364723205566, -0.07969372719526291, -1.0512892007827759, 0.03149257227778435, 0.10592186450958252, 0.9097214937210083, -0.3956049382686615, 2.431680202484131, 1.5693793296813965, 0.2880512773990631, -0.28888434171676636, -0.2687240242958069, 0.0639609843492508, -0.4339592456817627, 1.1971147060394287, -1.215738296508789, 0.2851282060146332, -0.05900413170456886, -0.9785856008529663, 0.7503978610038757, -0.19694383442401886, 0.19928885996341705, -0.7499065399169922, -0.7902566194534302, 0.04516255855560303, -1.0596098899841309, 0.41799741983413696, 2.4685468673706055, -0.6355991363525391, 0.9382503628730774, -1.0859410762786865, -0.8429909348487854, -0.10212522745132446, -1.0987399816513062, 0.34254276752471924, 1.2213404178619385, 0.6718332767486572, 0.5633020997047424, 0.43870124220848083, -0.5624257922172546, 0.18442702293395996, 1.404678225517273, -1.9730937480926514, 0.3161926865577698, 0.409282922744751, 0.5634496212005615, 1.6601589918136597, 1.3619353771209717, -0.28368648886680603, 0.35975897312164307], [0.054575592279434204, -0.30222246050834656, -0.29280248284339905, -0.6485093235969543, 0.6316627264022827, 0.4070770740509033, -0.09742945432662964, -1.005630612373352, 2.9656691551208496, -0.6131207346916199, 1.0799648761749268, -0.2663458287715912, 0.32620692253112793, -0.13331232964992523, 0.6542133688926697, 0.540291428565979, -0.20553244650363922, -2.2459349632263184, -0.02396957390010357, -0.10074197500944138, -1.1546293497085571, -0.8773103952407837, 2.000300407409668, 1.0933281183242798, 0.5914047360420227, 0.1626698225736618, 0.25805315375328064, 2.924976348876953, 0.05708956718444824, -1.0026615858078003, -0.13836507499217987, 0.302714079618454, -0.12422675639390945, 0.7043444514274597, 0.5753403902053833, -0.41663530468940735, -0.35526910424232483, -0.8321823477745056, -0.19426968693733215, 0.01696450635790825, -0.1871221959590912, -1.0494540929794312, 0.16442497074604034, -0.9371510744094849, 0.6420233845710754, -0.4725976586341858, -1.2189284563064575, 2.2841761112213135, 0.7743765711784363, -0.5156829357147217, 2.5190906524658203, 0.24724946916103363, 1.649758219718933, 0.027476122602820396, -0.49671462178230286, 0.8675252199172974, -1.054923176765442, 0.13477979600429535, 0.8316441178321838, 0.4717860817909241, -1.7133877277374268, -0.010836968198418617, 0.2377023547887802, 0.36797022819519043, -0.47330525517463684, -0.9470863342285156, -0.22477294504642487, -0.516212522983551, -1.1671998500823975, 0.9942528009414673, -0.09489061683416367, -0.7104149460792542, 1.129942536354065, -0.36670932173728943, -1.243700623512268, -0.3250564932823181, -0.5728310346603394, -0.917963445186615, 0.6145042777061462, 1.2360031604766846, -1.768496036529541, 2.0483555793762207, -1.1478291749954224, 0.27636516094207764, 0.520774245262146, -0.831571638584137, 0.8165079951286316, -1.031194806098938, -1.2869259119033813, -0.24128107726573944, 0.1785343438386917, 0.6694069504737854, 1.1548104286193848, 2.6347336769104004, 0.5713765621185303, -1.4683568477630615, -0.532981276512146, 1.6928002834320068, -0.5237667560577393, -0.31592243909835815, 0.3725481331348419, 0.10467041283845901, 0.6992295384407043, 1.1797581911087036, -0.6820036172866821, -0.4230785369873047, 0.3875625431537628, -1.5494157075881958, 1.4478137493133545, 1.9488837718963623, -1.6402901411056519, -0.49963808059692383, -0.6577255129814148, 0.025903284549713135, -0.28098180890083313, -0.1924167275428772, 0.8734661340713501, 0.5311631560325623, -2.881791830062866, 0.798916757106781, 1.0291247367858887, -0.1363101750612259, 0.23171794414520264, -2.128709316253662, -0.28093597292900085, -2.0534658432006836, -0.1897587776184082, 0.6758662462234497, -1.0609958171844482], [0.07514484226703644, 1.2069975137710571, -0.1210283637046814, 2.5498838424682617, 0.5620315074920654, -0.042660005390644073, -1.153252124786377, -0.37785834074020386, 0.5589539408683777, 0.5455386638641357, -0.0972459465265274, -0.524170458316803, -0.4076578617095947, -0.7803750038146973, -0.2649117112159729, -0.3205823004245758, 1.3429551124572754, 0.19291000068187714, 0.33239302039146423, 0.8673895597457886, -1.5234171152114868, 0.8262324929237366, -0.4316098093986511, -0.19242267310619354, -0.7221585512161255, 1.2042065858840942, -0.9999517202377319, -3.0499720573425293, -0.349942147731781, -0.40265119075775146, -1.4303196668624878, 0.2556631863117218, 1.031967043876648, 0.6396427750587463, 0.10377857834100723, -0.3898102045059204, -0.45659247040748596, -0.061765242367982864, 0.0790061503648758, 0.6213250160217285, -0.874065101146698, 0.9342387318611145, -1.2640737295150757, -0.2356090396642685, -1.667682409286499, -1.8614072799682617, 1.938045620918274, -0.23228102922439575, 1.030189037322998, 1.0129714012145996, 1.0416669845581055, 0.6432034969329834, 0.5296685695648193, -1.2464354038238525, 0.5531037449836731, 0.2761072516441345, 1.5472941398620605, -0.04602441191673279, -0.0936611145734787, -0.1919160634279251, 0.4172861874103546, 0.8678568005561829, 0.2970387041568756, 0.7572968006134033, -0.6738859415054321, 0.5700437426567078, -0.670513927936554, -1.5166107416152954, -0.05938339978456497, 1.0242128372192383, -0.12769551575183868, 0.24459318816661835, 0.15004615485668182, -0.018797053024172783, -0.9998964667320251, 0.8158584833145142, 1.9382963180541992, -0.923535943031311, 0.4404395520687103, -0.5096195340156555, 1.0764902830123901, -2.256727457046509, -0.10897640883922577, -0.8448934555053711, 0.7425541877746582, 0.7688397765159607, -0.3987976312637329, 0.41081535816192627, 0.18939165771007538, -2.542933464050293, -0.4432153105735779, -0.5454328060150146, -1.8582736253738403, 0.18398141860961914, -1.7227672338485718, -0.29240214824676514, 0.11833268404006958, 1.3462194204330444, 2.5659890174865723, 0.11437836289405823, 1.79818594455719, -0.32948964834213257, -0.5718228220939636, 0.03996126726269722, -0.8901638984680176, -0.7116096019744873, 0.6335168480873108, -1.102806568145752, 0.4033889174461365, 0.8441601991653442, 0.297937273979187, -0.3097504675388336, -1.31842041015625, 0.3834908604621887, -0.25908610224723816, 0.6361479759216309, 0.3759036660194397, -0.4884094297885895, 1.1950454711914062, 1.5619033575057983, 1.2592655420303345, 0.1740756332874298, -0.41723495721817017, 0.5759596824645996, -0.3257133960723877, 1.378365159034729, 0.5535051822662354, 0.06684639304876328, 0.5479267239570618], [0.9090452194213867, 1.0934795141220093, -0.23184438049793243, 0.33094173669815063, -1.0759412050247192, -0.2806377112865448, 0.23386235535144806, -0.6406455039978027, 0.7717347741127014, -0.8005695343017578, -0.9049178957939148, 1.032926082611084, -0.9078253507614136, -0.3472520112991333, -0.85373455286026, -1.3835995197296143, -2.2809648513793945, 0.16977638006210327, 1.6804008483886719, 0.0942067801952362, -0.2586304247379303, 0.733795702457428, -0.6178492307662964, 0.7843219637870789, 1.9224642515182495, -1.8301523923873901, -0.286909282207489, 0.3131188750267029, 0.11047405749559402, -0.5619978308677673, 0.9162451028823853, -2.2533018589019775, 0.9714184999465942, -1.5256741046905518, 0.3101818859577179, -1.0365183353424072, 1.0146316289901733, 0.8363122344017029, 1.3566120862960815, 0.25020670890808105, 1.2344191074371338, -2.3417372703552246, -0.447441965341568, 0.44072985649108887, -1.8918157815933228, -1.8002779483795166, 0.9859838485717773, 0.2633783519268036, 1.555113434791565, 0.03466256707906723, -1.1040241718292236, -2.0036463737487793, -1.678852915763855, 0.013242976740002632, -0.31793349981307983, -0.23026296496391296, -0.7590839862823486, 0.46245694160461426, 0.049642737954854965, 0.3487764298915863, 2.6512699127197266, 0.770633339881897, 0.9290082454681396, -1.9764554500579834, 0.04022826626896858, -0.7226786613464355, -0.07024651765823364, 0.2810310125350952, 0.38769111037254333, 0.772051215171814, -0.6588464975357056, 0.15877197682857513, 3.1270954608917236, -0.41076400876045227, 0.4916669726371765, 0.6043855547904968, 1.7177690267562866, 1.7431176900863647, 0.32041898369789124, 0.3141127824783325, -1.018448829650879, -0.075521320104599, -0.9224987030029297, -1.4220657348632812, 0.2501193583011627, -0.39358797669410706, -1.1110100746154785, 0.10367940366268158, -0.4049074053764343, -1.153502345085144, -0.7390768527984619, 0.6429051756858826, 1.149198293685913, -0.6129258871078491, -0.70779949426651, -0.01918877102434635, 0.16708272695541382, 1.6977037191390991, 0.9210841059684753, 0.004372660536319017, -0.793578565120697, 0.06151590496301651, 0.08953916281461716, -0.8592808246612549, -0.49599120020866394, -0.4832306206226349, -0.8270964026451111, -1.009002923965454, 0.5944646596908569, 1.4388047456741333, -2.1163556575775146, -1.3899478912353516, -0.027691924944519997, -0.923033595085144, -0.9545403122901917, 0.699307382106781, -1.2079787254333496, -0.3475757837295532, -0.08501939475536346, 1.0176951885223389, 1.4682302474975586, -0.6893025040626526, -0.967330276966095, -0.2789061367511749, -1.0873602628707886, -0.01340173278003931, 0.666079044342041, 0.8640589118003845, -0.4042193293571472], [-2.881986141204834, -1.6383167505264282, -0.24863392114639282, -0.7647989988327026, 0.08468417078256607, -0.7644937634468079, -0.14156906306743622, -0.6763790249824524, 1.2291693687438965, -0.8417616486549377, 1.8572312593460083, -0.5385258793830872, 1.671617031097412, -0.2897157669067383, -0.2870916426181793, -0.20528048276901245, 0.6490207314491272, -1.0689818859100342, -0.3023701012134552, -1.3651231527328491, -0.44392454624176025, -0.5883918404579163, 0.22785750031471252, 0.2964690029621124, -0.9475438594818115, 1.2674440145492554, 1.2214125394821167, -0.39695751667022705, 0.7400360107421875, -0.8865698575973511, -2.1737258434295654, 0.10322010517120361, 0.13329046964645386, -0.38472780585289, -0.4046056568622589, -0.5876749157905579, 0.3794560730457306, 0.43487125635147095, 0.06816458702087402, -0.7162895202636719, 0.7151711583137512, 1.0987420082092285, 1.3430426120758057, 0.5177335739135742, -2.13713002204895, -0.6498488783836365, -0.44672247767448425, 0.5071960687637329, 1.6841787099838257, -2.3538177013397217, 1.1151930093765259, -1.008488655090332, 0.5343270301818848, -0.8237818479537964, -1.2397865056991577, -0.19303235411643982, 1.7835137844085693, -0.7901735901832581, -0.9061400294303894, -0.013867729343473911, -0.13905586302280426, 0.8010532259941101, 0.7518056631088257, 1.6571022272109985, 0.9819770455360413, -0.1401580423116684, 0.13628338277339935, -0.882656455039978, -0.6526525616645813, -1.2496871948242188, -1.331944465637207, 0.7768804430961609, 1.4135911464691162, 1.0828912258148193, -0.4957895874977112, -0.2893100082874298, 0.7583823800086975, -0.28682804107666016, -0.31399571895599365, -1.0268800258636475, -0.3617280423641205, -0.5671493411064148, -0.5937706232070923, 0.335921972990036, 1.65947425365448, -1.1253223419189453, 0.9689776301383972, -0.0868954285979271, 0.030197396874427795, 0.3635292053222656, -0.4493882954120636, -0.06697060167789459, -0.5541055202484131, 0.26287510991096497, 0.25432419776916504, -1.0438023805618286, -1.1311721801757812, -1.3342665433883667, -0.9046169519424438, 0.29969850182533264, -0.13664163649082184, 0.5614076256752014, -1.2428386211395264, 0.8540635704994202, 0.3620606064796448, -1.3068208694458008, -1.5304886102676392, -0.3128705620765686, 1.3165180683135986, 0.9935428500175476, 0.2982499301433563, -0.7137565612792969, 0.5843251347541809, 0.10565117001533508, 1.3685920238494873, -0.7828831076622009, -0.8719530701637268, -1.3745461702346802, 0.17360620200634003, -0.22506402432918549, 1.3959333896636963, 1.7790272235870361, -2.6308534145355225, 1.89381742477417, 0.014694390818476677, -0.9990740418434143, -0.21026115119457245, 1.049771785736084, -0.7763340473175049], [-0.7517252564430237, 0.7279517650604248, 0.20106332004070282, -0.847428023815155, 0.3494875431060791, -2.6206023693084717, -1.0122990608215332, 2.0339958667755127, 0.9425917863845825, -1.3597453832626343, -0.08708800375461578, 0.7269667387008667, 0.5510433912277222, -0.6469548940658569, -0.7040291428565979, -0.7093472480773926, -0.47188475728034973, 0.09360150247812271, -1.7664482593536377, -0.11055142432451248, -0.014805673621594906, 0.5055087804794312, 0.031934406608343124, 0.5082836151123047, -0.7877906560897827, -0.09505481272935867, -0.9457867741584778, -0.89974445104599, -0.3936413824558258, -0.2692415416240692, -0.09443949162960052, 0.04825519397854805, 0.6795767545700073, 1.3480744361877441, 0.3445606827735901, -0.32374221086502075, -1.1757820844650269, 0.14129750430583954, 0.8504018783569336, 1.0446574687957764, -0.5830832123756409, -0.11226633936166763, 0.10386548936367035, 2.1453096866607666, -1.1088606119155884, -0.6368634700775146, -1.0972157716751099, -0.17769987881183624, 1.1180857419967651, 1.0375348329544067, 1.106558918952942, -0.5113008618354797, -0.9274014830589294, 0.1828491985797882, 0.6550130844116211, -0.8103252649307251, -0.7374290227890015, -0.25245821475982666, 1.3351744413375854, 0.09751971811056137, 1.2107610702514648, -0.9548344612121582, -0.8787106871604919, -1.1668034791946411, 1.5596271753311157, 1.1393753290176392, -0.32633236050605774, -1.2803912162780762, 0.22341173887252808, -0.22502350807189941, 0.9371935725212097, 1.310134768486023, 0.3321896195411682, 0.9180436730384827, 0.0969168022274971, -1.231199026107788, 0.5410802960395813, -0.9864847660064697, -0.8383210897445679, -2.3098056316375732, 0.5950697064399719, 0.7994928956031799, 1.4465022087097168, 1.4252161979675293, 0.27608630061149597, 1.3201711177825928, -0.47483116388320923, 0.9077601432800293, -0.8406927585601807, -1.4200183153152466, -0.7630901336669922, 0.5298156142234802, 1.2611908912658691, 0.2829890251159668, -0.5939998626708984, -1.329792857170105, 0.8631241917610168, 0.6507166028022766, -0.17287078499794006, -1.4256545305252075, 1.3783583641052246, -0.15826275944709778, -0.17051033675670624, 0.07468511164188385, -0.35054248571395874, -1.2839744091033936, -1.6941676139831543, -1.8978583812713623, 0.601165771484375, -1.0835059881210327, -1.3784269094467163, 1.065293550491333, 0.05765398219227791, -0.7157923579216003, -0.40571627020835876, -0.4830982983112335, -0.8741912245750427, -0.18549659848213196, -0.19992756843566895, -0.7812617421150208, -0.5982707738876343, 0.6875019669532776, 1.0375056266784668, -0.8109840750694275, -0.572841227054596, 0.0911744087934494, -1.6753857135772705, 3.327608585357666, -0.0684860572218895], [-1.1144155263900757, -1.072535753250122, 0.7865869402885437, -1.4134091138839722, -0.10720104724168777, 0.3225320279598236, 0.19515082240104675, 1.4842135906219482, -1.2229186296463013, 1.0991019010543823, 1.281486988067627, -0.9416662454605103, -1.5739918947219849, -1.3920302391052246, 0.579276442527771, -0.11216381192207336, -2.1565396785736084, -0.5471675992012024, 0.41380321979522705, -0.5757440328598022, 1.3373686075210571, -0.402953177690506, 0.3560636639595032, 1.8182827234268188, 0.16787856817245483, -0.4763731360435486, 0.6910439133644104, -0.8479606509208679, -1.6745154857635498, -0.32221537828445435, -0.8513850569725037, -3.089038848876953, 0.45584407448768616, -0.5285400152206421, 0.24131371080875397, -0.18716464936733246, 0.5068629384040833, 0.6812617778778076, 1.573259949684143, -0.4631707966327667, -0.6830986738204956, 1.8840217590332031, 0.23926593363285065, 0.3290290832519531, 0.7333566546440125, -0.4883892238140106, 0.020808441564440727, -0.7161787748336792, 1.2794983386993408, 0.7977596521377563, -0.008121335878968239, 2.184295892715454, 1.4698125123977661, -0.8052933216094971, -0.8360196948051453, 0.26673951745033264, -0.01791401393711567, -1.131293535232544, -0.8774343132972717, -1.1595999002456665, 1.3757516145706177, 0.11438126116991043, 0.8439921140670776, -0.34955084323883057, -0.18690241873264313, -0.6692982912063599, 1.3377935886383057, -0.25161170959472656, -1.3150906562805176, 0.04876206815242767, -0.15554557740688324, 1.3315480947494507, 1.3963412046432495, -0.2098262459039688, -0.7328609824180603, 0.21344336867332458, 0.5061061382293701, -0.5167821049690247, 0.023391790688037872, -0.5356332063674927, -0.9284988045692444, 0.3942304849624634, 1.837330937385559, -0.6884238719940186, -0.5153529047966003, 0.03860478848218918, 0.575131356716156, -0.5112949013710022, -1.1232603788375854, -0.7150710225105286, -0.9506509900093079, 1.002603530883789, 0.3988150358200073, 0.38043835759162903, -1.223585844039917, 0.4319011867046356, 0.6688912510871887, 0.9728829264640808, -1.8420367240905762, 0.7145063877105713, 2.516456365585327, -0.5113314986228943, 1.9807878732681274, 1.7003849744796753, -0.10536358505487442, -0.6762736439704895, -0.6503314971923828, -1.3570712804794312, 0.1687408834695816, 0.6278958320617676, -1.1938544511795044, 0.4358074963092804, -0.46924591064453125, 0.6860896944999695, -0.7013247609138489, 1.2795872688293457, 0.2760756015777588, -0.06449899822473526, 0.20721618831157684, -1.0041766166687012, -0.6648761630058289, -0.39540475606918335, -0.3173460364341736, -0.6507009863853455, 1.0098581314086914, 0.17835506796836853, 0.3995874226093292, 0.5033992528915405, -0.7345324754714966], [0.5926069617271423, 0.3121807873249054, -0.7259349822998047, -0.7720817923545837, 0.17519806325435638, 0.8524401187896729, -0.9436677694320679, -0.5638315081596375, -0.40711238980293274, -1.1650996208190918, 1.9832922220230103, -0.047181639820337296, -1.1762794256210327, -0.17059195041656494, -1.0740063190460205, 0.39504826068878174, -0.7446949481964111, 1.034231424331665, -0.8201519846916199, 0.06848786026239395, -1.0678311586380005, -0.3930915892124176, -1.4361168146133423, -1.119744062423706, 1.0449846982955933, 1.249922275543213, -1.3223410844802856, -0.11491838097572327, 0.722675621509552, -1.518589973449707, 0.9570955634117126, 1.0056042671203613, -0.8121618032455444, 1.383446216583252, 0.5748643279075623, -1.4058880805969238, 0.4163893461227417, 0.4560684561729431, 0.601215124130249, 1.147455096244812, 0.6747405529022217, -1.165006399154663, -1.2498303651809692, 1.0675311088562012, 1.4608362913131714, -0.47480228543281555, 0.10483665764331818, -1.0902425050735474, 0.25041940808296204, 1.5537424087524414, 0.3692544996738434, 1.7454510927200317, 2.20768404006958, 1.1417608261108398, 2.5213961601257324, 0.4045184850692749, 0.059600334614515305, -0.18937192857265472, 1.1016063690185547, 0.3535424768924713, 0.0315120629966259, -0.07858388125896454, -0.6604402661323547, 0.6104598045349121, -0.7717638611793518, -2.020689010620117, 1.310333490371704, -1.2835602760314941, 0.2744995653629303, -0.08985144644975662, 0.35626426339149475, -0.39050087332725525, -0.37918078899383545, -0.5930094122886658, 0.6508784294128418, 0.23995473980903625, 2.2183334827423096, -1.0380672216415405, 0.36696070432662964, 2.2620670795440674, 0.6653435826301575, -0.800601065158844, 0.33796587586402893, 1.2273699045181274, 0.9395496845245361, 0.5539787411689758, -0.5820744037628174, 0.09315858036279678, -1.1244332790374756, 0.5764661431312561, -0.5214476585388184, 0.36794304847717285, 0.1626182198524475, 1.2198543548583984, -0.030750161036849022, 0.16975858807563782, 0.17559540271759033, 1.0134986639022827, -0.7803809642791748, -0.48760098218917847, 2.028285264968872, -0.7124192714691162, 0.39740419387817383, -0.6570137143135071, -2.4630491733551025, -0.23821137845516205, 0.2692735493183136, 0.320100337266922, -0.10523989796638489, -2.435238838195801, -0.3075476884841919, 0.20599399507045746, 1.9887850284576416, -0.7576282620429993, -0.6972269415855408, -0.6776939034461975, -0.6884689927101135, 1.0778675079345703, -0.06275703012943268, 1.2089296579360962, -0.7809333205223083, 0.2031794786453247, -1.1519083976745605, 0.10579464584589005, 1.2522238492965698, 0.7491181492805481, -0.7292565107345581, 0.6008142232894897, -0.08207657188177109], [0.542314887046814, -0.5372142791748047, -1.5665111541748047, -0.8503199219703674, -0.9962509870529175, -1.9539906978607178, -0.32557493448257446, -0.09620766341686249, -0.593575656414032, 2.171743869781494, -0.043546468019485474, 1.1108198165893555, -0.874207615852356, -0.6880699992179871, -1.2566235065460205, -0.9124287962913513, -0.24449855089187622, -0.3744870126247406, -0.8663106560707092, -1.086846113204956, 0.3440931737422943, -0.23153318464756012, 0.3875912129878998, 0.22422504425048828, 1.1093941926956177, -2.057814359664917, 0.31766870617866516, -2.005659580230713, 1.0011862516403198, 0.17214085161685944, -0.04660229757428169, -0.24689330160617828, -5.5310727475443855e-05, -2.2577402591705322, -0.571362316608429, -1.0676888227462769, 0.1658933013677597, -1.1137840747833252, -1.2660958766937256, 0.19031132757663727, 0.14240580797195435, -0.33656632900238037, -0.1868288218975067, 1.0356824398040771, -0.08170928061008453, -0.30803757905960083, 0.05422961339354515, 0.975607693195343, -0.863593578338623, 1.9356337785720825, 1.3530715703964233, -0.7419359683990479, 0.7029568552970886, 1.6508491039276123, 0.4028792381286621, -1.1266181468963623, -0.6325398683547974, -0.20534734427928925, 1.231459140777588, -1.2014375925064087, -0.4381791055202484, -1.5410277843475342, 0.2964284420013428, -1.2973408699035645, -1.0071277618408203, 1.4557894468307495, -1.495094895362854, 0.36890721321105957, 0.37174487113952637, -0.7554261088371277, -0.20246535539627075, 0.8321580290794373, -1.7552731037139893, -1.2509666681289673, -0.6770868897438049, 1.3495445251464844, -1.742817759513855, 0.21105770766735077, 0.7896289825439453, 1.4343559741973877, 0.1244620606303215, 0.46724697947502136, -0.935749888420105, -1.017617106437683, -0.031075678765773773, -0.854289710521698, -0.378640353679657, 0.8289779424667358, 0.44338926672935486, -1.5754793882369995, -0.43607020378112793, 1.4491487741470337, -0.4773509204387665, -0.09280239790678024, -0.11537652462720871, 1.3353078365325928, 1.4496160745620728, 0.01797804981470108, -1.00538969039917, 0.2374928742647171, 0.1414855271577835, 0.60734623670578, -0.8382676839828491, -0.5292163491249084, -2.088120460510254, -0.08143115043640137, -0.9508034586906433, -0.04746644198894501, -1.0615252256393433, 1.7954611778259277, -1.1899802684783936, 1.3355121612548828, 1.2662897109985352, 1.0338690280914307, 1.1095268726348877, 0.2737048268318176, -2.048241138458252, -0.24808216094970703, -0.042671848088502884, -1.412125587463379, 0.2728564441204071, 0.44889959692955017, -0.5583939552307129, -0.8260779976844788, 0.6015815138816833, 0.016600966453552246, -0.5801365375518799, -1.7010307312011719, -0.46496859192848206], [0.7530184388160706, -0.6100831031799316, 0.8001810312271118, 1.26004159450531, 0.21725736558437347, -0.09403838217258453, 0.24047651886940002, -0.1506700962781906, -1.121360182762146, 0.8222042918205261, -0.9960297346115112, 0.039289604872465134, 0.9165620803833008, -0.9469314813613892, 0.734704852104187, -0.4549757242202759, 0.32811087369918823, 0.9863025546073914, 0.9811514019966125, -1.918990969657898, -0.12412898987531662, 1.2383309602737427, 1.3841110467910767, 0.5010167360305786, -1.894745945930481, -0.27532559633255005, -0.27547603845596313, 1.2258388996124268, 0.2918708026409149, -0.041657254099845886, -0.7675550580024719, -0.6153954863548279, 1.1825143098831177, 0.04699769243597984, -1.1168159246444702, 0.43189454078674316, 0.32355740666389465, 1.1936053037643433, 2.40800404548645, -0.8732296228408813, 0.6046224236488342, 0.23513798415660858, 0.6812357902526855, 0.9793331623077393, -1.658300757408142, -1.1836128234863281, 0.47426608204841614, 0.73268061876297, 0.5104106068611145, 1.6728389263153076, 2.3543174266815186, 0.4563975930213928, -0.480689138174057, 0.22138726711273193, 2.57719349861145, 0.1801898181438446, 0.8858010768890381, -1.0457478761672974, -0.9915032386779785, 0.5765697956085205, -0.8077366948127747, 0.6768606901168823, -0.589575469493866, -0.13462859392166138, -0.5119658708572388, 0.8457328677177429, 0.926536500453949, -0.48473021388053894, -1.1196322441101074, 0.22307036817073822, -0.2586044669151306, 0.5048598647117615, 0.5255449414253235, 0.3868962824344635, 0.8535416722297668, -0.40748462080955505, 0.13136477768421173, -0.3549728989601135, -2.640108823776245, -1.1112791299819946, 0.25784873962402344, -0.4253411591053009, 0.5087918639183044, 0.30041179060935974, -0.7348734736442566, -0.4313015639781952, 0.33464640378952026, -1.379712462425232, 0.16544611752033234, -0.6829439401626587, -1.9449366331100464, -0.5226191282272339, -2.7363510131835938, 1.240143060684204, -0.7838375568389893, -0.04207706078886986, -1.1094685792922974, -1.0641669034957886, -0.8518111109733582, 0.8781653642654419, 0.3060753047466278, -1.227205753326416, -0.692853569984436, 1.223091721534729, 1.376397967338562, -0.22454315423965454, -0.6408720016479492, -0.42290106415748596, 1.258371353149414, 0.6388294100761414, -1.3000690937042236, -0.012973609380424023, 0.1616736799478531, -1.164563775062561, 1.0596156120300293, 0.8336548209190369, -0.9268574118614197, 0.8894901275634766, 0.4711061120033264, -0.02713097631931305, -0.07902049273252487, -0.8539511561393738, 1.1019631624221802, 0.9214746952056885, 1.8300511837005615, -0.7061392068862915, -0.39604800939559937, 0.5211657285690308, -1.1916948556900024], [-0.8924266695976257, -1.1221551895141602, -0.06853058934211731, 0.15681859850883484, -1.6600106954574585, -0.1400201916694641, 0.756102979183197, -0.04234425351023674, 0.4145405888557434, 1.47166109085083, -0.214015394449234, 0.44544145464897156, -0.9565653204917908, 1.164562702178955, 0.9383373856544495, -0.31730300188064575, -1.1166622638702393, -0.27153339982032776, -0.2457341104745865, -0.1464410126209259, 0.6707804799079895, 1.0234843492507935, -0.4080880582332611, -1.4006856679916382, -1.3080416917800903, 1.2559099197387695, 0.676505982875824, 0.47634193301200867, -1.8675700426101685, -0.7717651724815369, -0.14357247948646545, -0.5011718273162842, 0.46742668747901917, 0.6017895936965942, 0.10446421802043915, -0.8380954265594482, 1.3871105909347534, 0.2918691635131836, 1.3258355855941772, 1.570643424987793, 0.5632184743881226, 1.4488834142684937, -0.6652488708496094, -0.6764594912528992, 0.3986952304840088, -2.6146297454833984, -0.3965758681297302, -0.6618003845214844, -0.34784749150276184, 1.3082835674285889, 2.1690549850463867, 0.02427896298468113, -0.5137401223182678, 1.5284152030944824, 0.1560078263282776, 0.07198487967252731, 1.336439609527588, 0.3061467707157135, -1.6650038957595825, 1.7829627990722656, 1.2359143495559692, -0.3194787800312042, 0.5893479585647583, 1.075855016708374, -0.49791818857192993, -0.2770821154117584, -1.4247453212738037, 0.4394453465938568, 0.11936648190021515, 0.8509869575500488, 0.47385790944099426, -0.6437715888023376, 0.3304145038127899, 1.1074254512786865, 1.0813764333724976, -0.27915456891059875, -1.3643423318862915, -1.4009380340576172, -0.10168097168207169, 1.3566735982894897, -1.8459115028381348, -0.8837234377861023, -1.551746129989624, 0.3645722568035126, -0.197683185338974, 0.4100818634033203, 1.5725724697113037, -0.8369085192680359, -1.1816132068634033, 0.44211092591285706, 0.32854077219963074, 0.6251668930053711, -2.318068027496338, 0.09686174243688583, 0.291009783744812, 0.16680988669395447, -0.30898112058639526, -0.45849505066871643, 0.502567708492279, -2.3744332790374756, 1.5187718868255615, 0.26286259293556213, -0.10440336167812347, 0.44743669033050537, -0.47490572929382324, -0.3251113295555115, 0.5530855059623718, -0.5566989183425903, -0.2031228393316269, 1.9296213388442993, 0.4447445869445801, -0.8656349182128906, -0.31203892827033997, 0.11950729787349701, 1.074941873550415, 1.4878253936767578, 0.7714885473251343, -0.3406578600406647, 1.211084246635437, 1.3512870073318481, -1.34439218044281, 0.06882656365633011, 1.093296766281128, -1.503724217414856, 2.632059335708618, -0.18453086912631989, -1.18230140209198, -1.187018871307373, 0.3205445110797882], [0.37012937664985657, 0.06680498272180557, 0.654876708984375, 1.2108211517333984, 0.9358898401260376, -1.038813591003418, -0.6055070161819458, 0.914861798286438, -0.7967644929885864, -1.5150649547576904, -0.5161476731300354, -0.7490425705909729, 1.1868994235992432, 0.35296711325645447, 0.5152835845947266, -0.7133122086524963, 1.3082334995269775, -1.2574042081832886, 0.5248556733131409, 1.1971904039382935, -1.7408581972122192, 1.6603745222091675, -0.6984677910804749, -1.1309921741485596, 2.656167507171631, 0.7285947799682617, -0.26656806468963623, 0.3833618462085724, -2.6776187419891357, -0.33857059478759766, -0.701543927192688, 0.5341687798500061, 0.5092465281486511, 0.22453995048999786, 0.07668248564004898, 0.5451772809028625, 1.026300311088562, 1.9508769512176514, -0.2765766978263855, -0.433361291885376, -0.16654692590236664, -0.1318707913160324, -0.04462634027004242, 0.1772356778383255, -1.0857927799224854, 1.659009337425232, 0.9708113074302673, 1.618772268295288, 1.2580934762954712, -1.4366990327835083, -0.07682538032531738, -0.7570343613624573, 2.1757843494415283, -1.0713706016540527, -1.1250163316726685, -0.42049092054367065, 0.9836571216583252, -1.7367116212844849, 0.3110578656196594, 0.174441859126091, -1.7249062061309814, -0.610392689704895, 0.4227302074432373, -0.46302032470703125, -0.2273426651954651, 0.8613951802253723, 1.7496380805969238, -1.492314338684082, 1.7391252517700195, -0.8579209446907043, 1.941450595855713, 2.9544730186462402, 0.24961024522781372, -0.281942754983902, -0.47105371952056885, -0.9819163680076599, -0.16117390990257263, 0.33903592824935913, 0.7792595624923706, 0.5774853229522705, -1.0501633882522583, 1.2882717847824097, -1.1247668266296387, -1.674336314201355, 1.3823100328445435, 0.09660225361585617, 2.3959498405456543, -0.8010118007659912, 0.4068206548690796, -0.375293105840683, 0.0040762415155768394, 1.106209635734558, 0.6237965226173401, 0.16834697127342224, 0.8404651284217834, 0.5232837200164795, 1.7327938079833984, 0.6222453713417053, 1.1184582710266113, 0.34996291995048523, -0.767707109451294, 1.5471059083938599, -0.7163934111595154, 0.3226485252380371, 1.2273318767547607, -1.6424251794815063, -0.5507648587226868, 0.03737149015069008, -1.2571821212768555, -0.9463492631912231, -0.38825058937072754, 0.5063816905021667, -0.8537675142288208, 0.3209668695926666, -1.0423359870910645, 0.6651694178581238, -0.7962581515312195, -0.5201054215431213, -0.8202752470970154, -1.9206573963165283, 0.08714475482702255, -1.5760657787322998, 0.3822488486766815, 1.8753341436386108, 0.5890440344810486, -1.4368177652359009, -0.9116652011871338, 1.6128066778182983, -0.17161712050437927], [0.9084305763244629, -1.5297188758850098, -0.6851379871368408, -0.23055586218833923, -0.4166603982448578, -0.7773804068565369, 0.005510266404598951, -0.41178715229034424, 0.3505573272705078, 0.8009724020957947, 2.277050018310547, -0.562971293926239, -0.4273586571216583, 0.4654087424278259, 0.6851465106010437, 1.6774585247039795, 0.5561332702636719, 1.2361239194869995, 1.5676637887954712, 1.5162098407745361, -0.5906841158866882, 1.5153218507766724, 0.552314281463623, 0.45204290747642517, -0.8455675840377808, -1.5339230298995972, 0.6026034355163574, -1.3319467306137085, -0.018827157095074654, 1.0618423223495483, -2.1783437728881836, -0.07746127247810364, -1.9382694959640503, 0.5011281967163086, -1.4905378818511963, -0.09237619489431381, 0.09916456788778305, -0.8819919228553772, -0.16727575659751892, -0.7976624965667725, -0.010112332180142403, 0.4057459533214569, -0.17589251697063446, -0.3538627326488495, -0.280989408493042, 0.5654094219207764, 0.9129354357719421, 2.043281078338623, -0.013299895450472832, 1.555273413658142, 0.8027685284614563, -0.811321496963501, 1.0029850006103516, 0.7631182074546814, -1.5877755880355835, -0.8411206007003784, -0.45713552832603455, 1.0974498987197876, -2.650689125061035, 0.7736981511116028, -1.0738427639007568, 0.3504229784011841, -0.7054805159568787, -2.2720272541046143, -1.6757560968399048, 0.06282614916563034, 0.7215890288352966, -2.390418767929077, -0.290898859500885, 0.08629124611616135, -0.7443960309028625, -0.3465462327003479, -0.6701162457466125, 1.061813473701477, -0.8281260132789612, -0.24394766986370087, -1.39667546749115, -0.09007137268781662, -0.5936373472213745, 0.6601059436798096, 2.0716919898986816, 1.0385231971740723, 1.1121268272399902, 0.4778861999511719, -0.776357114315033, -0.3038059175014496, 1.1159378290176392, -0.4961692988872528, -2.3153088092803955, -1.5863845348358154, 0.9582794308662415, 0.13643331825733185, 0.7204423546791077, -0.32812267541885376, -0.1143995001912117, -1.2130528688430786, -0.31572750210762024, -1.2353438138961792, -0.4383171498775482, -0.036173924803733826, 0.017578326165676117, -1.6921849250793457, -0.22971200942993164, 0.9713268280029297, -0.4928396940231323, 0.5191970467567444, 1.1239264011383057, -1.0655546188354492, 2.229093313217163, -0.04223662614822388, 0.7869810461997986, -1.8568155765533447, -0.5270424485206604, -0.28263187408447266, 0.6767579913139343, -0.09952380508184433, -0.5881354808807373, -1.678944706916809, 0.14036014676094055, 0.7417658567428589, 0.8413516283035278, 1.435144305229187, 0.5056982636451721, -1.6224240064620972, -0.03281313180923462, -1.9598286151885986, -0.35333457589149475, -0.5979478359222412, -1.3607391119003296], [0.7120583057403564, -0.6890051960945129, 0.9670141339302063, 1.055832028388977, 1.798317313194275, 1.7675179243087769, 0.23641939461231232, 0.01160700898617506, 0.5372139811515808, -1.3868598937988281, -0.568463146686554, -2.642045736312866, -1.4166661500930786, 0.6211346983909607, 0.9879042506217957, 0.10359128564596176, -1.5007051229476929, -0.708898663520813, 0.039833907037973404, -1.590052843093872, 1.5817443132400513, -0.8907316327095032, -0.0551513209939003, 1.195542573928833, 0.85208660364151, 1.0152233839035034, -0.572009801864624, -1.503790259361267, -0.6805421113967896, -0.8339005708694458, -1.4448167085647583, 2.0972347259521484, 0.8507954478263855, 0.49405843019485474, 0.4461808502674103, -0.32279372215270996, 1.1222251653671265, -0.5025701522827148, -0.22202089428901672, 0.8252254724502563, 1.4816603660583496, 0.5940775275230408, -0.3917281925678253, 1.5588021278381348, 0.42574089765548706, -0.1098841205239296, 2.2391388416290283, 0.6034783124923706, 0.7167980670928955, 0.43229442834854126, -2.258269786834717, 0.9870717525482178, 1.1493654251098633, 2.147951364517212, 0.4238527715206146, 0.7272197008132935, 1.227034330368042, -0.7472544312477112, 0.27878594398498535, 0.9935305118560791, -1.8610166311264038, -0.17019005119800568, -0.4858514666557312, 1.008618712425232, -0.6225913166999817, -0.106268011033535, -0.7103059887886047, -1.95002281665802, -1.710198998451233, 2.325651168823242, 1.8451685905456543, -1.653024673461914, -0.27449557185173035, 0.7160195112228394, 1.381691336631775, -1.529358983039856, 0.5830724835395813, -1.1149466037750244, -0.5501371026039124, 1.555416226387024, 1.716081976890564, -0.05332750082015991, 0.9311537146568298, 0.5666260719299316, -0.41521573066711426, 1.1169198751449585, -1.3551065921783447, -0.40160152316093445, 0.980552613735199, 0.20089419186115265, 0.1587124466896057, -0.7751474976539612, 1.688815951347351, -0.3332604765892029, 0.07871928811073303, -0.4495313763618469, 0.7303173542022705, 0.14548873901367188, -0.07565424591302872, 1.7302796840667725, 0.5398387312889099, 1.2637451887130737, 0.11178313940763474, -0.3621048331260681, -1.4716229438781738, -0.0042463080026209354, 0.4842737019062042, -0.5031543970108032, -1.1253111362457275, -1.3195117712020874, -0.13422119617462158, 0.47258269786834717, 2.051746368408203, -0.8636069893836975, -0.5073789954185486, 1.4755618572235107, -0.4540303647518158, -1.5318894386291504, 0.03170335292816162, -0.31344977021217346, 0.5647687911987305, 1.2190628051757812, -0.48822808265686035, 0.2141997218132019, 0.017570875585079193, -1.1944756507873535, 1.0261310338974, 0.09735482931137085, 1.1679069995880127], [0.3926873207092285, 0.5839087963104248, 0.10420600324869156, 0.2587304413318634, -1.780286192893982, 0.3088560700416565, -1.3870083093643188, 1.0720583200454712, 1.905747413635254, -0.18614761531352997, -0.2774949371814728, -0.17689450085163116, -0.16874946653842926, 0.9909961819648743, -0.4357610046863556, 0.8164599537849426, -0.28557637333869934, -0.45650753378868103, -0.0849733054637909, -0.3394569754600525, 0.6028665900230408, 0.6346773505210876, 1.2183016538619995, 0.542671263217926, 0.4736272692680359, -0.8243953585624695, -0.8517084717750549, -0.9193692803382874, -2.125657081604004, 1.1457940340042114, -0.15005552768707275, 3.0923702716827393, -0.01032340619713068, 0.42252376675605774, -2.2191760540008545, -0.7804712057113647, -1.3210192918777466, 0.5114709734916687, 0.3571103811264038, 0.29412969946861267, 0.7485209703445435, 1.14393949508667, 0.555341362953186, 0.6753587126731873, 0.7310970425605774, 0.6037958264350891, 0.7557176351547241, -0.7901250720024109, 1.692813515663147, 2.2168242931365967, -0.26657456159591675, 0.01101792510598898, -2.023271322250366, 0.759972333908081, 0.7509738206863403, 0.6250288486480713, 0.9359750151634216, -1.8207520246505737, -1.5029406547546387, 2.0606210231781006, 1.2691179513931274, 0.20747913420200348, 0.21544238924980164, -1.4157768487930298, -1.5646904706954956, -1.5843249559402466, 0.5115788578987122, 0.027768438681960106, -0.34088802337646484, 0.2239999771118164, -0.47959744930267334, -0.7296967506408691, -0.4750504791736603, -1.614809513092041, 0.22461800277233124, -0.24844147264957428, -0.8819774389266968, -0.06493809819221497, -1.219714879989624, 0.21000289916992188, -0.3088288903236389, 0.07407643646001816, -0.42965081334114075, 0.16426654160022736, 0.184035986661911, -0.3000186085700989, 0.6255393624305725, -0.647469699382782, -0.6846272945404053, -0.18557454645633698, -0.7923139333724976, 0.23499713838100433, 0.1156584620475769, -0.7011442184448242, -2.175657033920288, 0.41125085949897766, 0.047093626111745834, 0.5066226124763489, 0.09225358814001083, -1.1024737358093262, -0.4985737204551697, -0.8389485478401184, -0.918571412563324, 0.3942822515964508, -0.6428327560424805, 0.3363357186317444, 0.05165860801935196, -1.3498318195343018, 0.5430405139923096, 0.1765337735414505, -0.8359501361846924, -0.8804892897605896, 1.3967995643615723, -0.10694845765829086, 1.4716449975967407, 0.5252090096473694, -0.7683401107788086, -0.5013566613197327, -0.02333022840321064, 1.239363431930542, 1.0778599977493286, -2.3082115650177, -1.6134583950042725, 1.1493374109268188, 1.6471115350723267, -1.3736424446105957, -0.8827345967292786, -0.015574878081679344, 0.10841390490531921], [-0.6450870037078857, 0.5867599844932556, -0.44564464688301086, 0.052769940346479416, -0.41035377979278564, 0.77354896068573, 0.3850938677787781, -0.8464010953903198, 0.17094820737838745, 0.14303195476531982, 0.6355605125427246, 0.9541621208190918, 1.7311525344848633, 0.6787727475166321, -1.6169183254241943, -0.5296697020530701, -1.7612022161483765, -0.6216328740119934, -0.09486330300569534, -1.677069067955017, 0.7954614758491516, -0.3833586275577545, 0.677798867225647, 0.4863223433494568, -1.4446372985839844, 0.782038152217865, -0.07314632833003998, -0.48396173119544983, -1.535232424736023, -0.6684736013412476, 1.0629280805587769, -0.11945240199565887, -0.8529833555221558, -0.32929328083992004, -0.6561457514762878, -0.5518261194229126, -0.26161932945251465, 0.5975070595741272, 0.48016905784606934, -0.6754123568534851, 0.02720404975116253, 1.6963298320770264, -0.30111566185951233, -0.7631106376647949, -0.5401962995529175, 0.4951227903366089, -0.8276041150093079, 1.6616040468215942, -0.5157672762870789, -1.215295433998108, 0.011977317743003368, -1.145276427268982, 1.5482361316680908, 2.3332748413085938, -1.4306793212890625, 0.7314653992652893, -0.054035719484090805, 0.7166755795478821, 0.2897966206073761, -1.3498501777648926, -0.5917987823486328, -0.48126107454299927, -1.0588144063949585, 0.10982649028301239, -0.47815704345703125, -0.07866598665714264, -1.703127145767212, -0.3429347574710846, -1.164573073387146, 0.4444139003753662, 1.448850393295288, 2.0340354442596436, -0.07947168499231339, -0.8873275518417358, 0.13624513149261475, -0.995879054069519, -0.17873041331768036, 1.2982795238494873, 0.618187427520752, 0.7406063079833984, 0.5220574736595154, -0.8466966152191162, -0.3600173592567444, 0.5846930146217346, -1.6140515804290771, 0.22660711407661438, 1.6845628023147583, 0.2947220504283905, 0.3655865490436554, -0.2474283128976822, 0.014430662617087364, 1.5605826377868652, 0.7748687863349915, -1.7474210262298584, 1.0044975280761719, 1.7224671840667725, 0.3082939088344574, 0.21978744864463806, -1.24685537815094, -0.13084812462329865, 0.055151790380477905, -0.7070758938789368, -0.24814967811107635, 0.9954110980033875, 0.16713108122348785, -0.4833642542362213, 2.39467453956604, -1.1295005083084106, 0.4181680977344513, 1.3761314153671265, -1.8462982177734375, -0.1617017388343811, 0.18674565851688385, 2.2686355113983154, -1.2982800006866455, -0.10674408823251724, -0.6441376805305481, 2.48653507232666, 0.42349401116371155, 0.9070487022399902, -0.8628965616226196, 0.6245752573013306, -0.07248970121145248, 1.3704838752746582, 0.4906129837036133, -0.3373018503189087, 0.6850160956382751, -0.049758341163396835, -0.8325117230415344], [0.6084811091423035, 0.260712206363678, 1.1842942237854004, 0.3206396996974945, 0.4825594127178192, -0.48979178071022034, 0.5240645408630371, -1.8192452192306519, -0.4728478789329529, 0.45606017112731934, 1.4487286806106567, -0.025777805596590042, -0.5075761675834656, -0.5859774947166443, -0.747258722782135, 1.7418251037597656, -1.640278697013855, 0.5426511764526367, -0.222818523645401, -0.9364777207374573, 0.6854270696640015, 1.7643482685089111, 1.5345526933670044, 0.9448829889297485, -1.5224177837371826, -1.6427503824234009, 1.0873082876205444, 0.08334312587976456, -0.0020052797626703978, -0.05555949732661247, 0.8080576658248901, 0.33348339796066284, -0.8903776407241821, 1.413101077079773, 1.1908390522003174, 0.04797869175672531, -2.1772472858428955, -0.5679312944412231, 0.9533050656318665, -1.123082160949707, 0.24845659732818604, -0.20935888588428497, 0.13590803742408752, 0.9675335884094238, 1.6587339639663696, -0.48879116773605347, -0.6760464310646057, 0.8490653038024902, -1.0892521142959595, 0.7827162742614746, 0.4529725909233093, -1.3932623863220215, 0.637728214263916, -1.1788151264190674, -0.48663026094436646, -0.6286818981170654, -0.8645449280738831, 2.933405876159668, 0.6964942216873169, 1.1741948127746582, -1.431542158126831, -0.3514566123485565, -1.2521591186523438, -0.2431744933128357, 0.204849973320961, -0.7792720198631287, -2.5168683528900146, -0.6842600703239441, -0.10746928304433823, -2.3227593898773193, 0.30201256275177, 0.6425848007202148, 0.15238279104232788, 1.4375638961791992, -1.1820214986801147, 0.49938201904296875, 0.4289383888244629, 2.064053773880005, 1.4660093784332275, -0.618489146232605, 0.4295389950275421, -1.8932582139968872, -0.3004641830921173, 0.37188035249710083, 1.1358195543289185, -0.8004190325737, 1.3802917003631592, -0.19664661586284637, -0.2251329869031906, 0.37563222646713257, 0.5342443585395813, 0.40280860662460327, 1.1921874284744263, 0.7568092942237854, -0.7839438319206238, 0.08315752446651459, 1.5780503749847412, -0.20340391993522644, 1.210382103919983, 1.4096349477767944, 0.3385457992553711, -0.14193899929523468, 0.508836030960083, 0.46956729888916016, -2.6367998123168945, 1.3610719442367554, 0.5642130374908447, 1.1242115497589111, -0.13995757699012756, -0.14094102382659912, 0.3748084008693695, -2.074191093444824, 0.027166202664375305, -0.8327898979187012, -1.7739776372909546, -0.058206554502248764, -0.7792568206787109, 1.0755858421325684, -0.3399401605129242, 0.001459788647480309, -0.38525351881980896, -0.580780565738678, 0.3398505449295044, -0.3100396394729614, -0.8120523691177368, -0.5252805352210999, 0.48482760787010193, 0.019709231331944466, 0.30615803599357605], [0.32805612683296204, -0.28499898314476013, 0.40023720264434814, 0.3027036786079407, 0.5266357064247131, -1.0034376382827759, 1.4812346696853638, 0.08025466650724411, -0.4134441912174225, -0.1500965654850006, 0.795740008354187, -1.6323139667510986, 0.10178355127573013, 2.7765650749206543, 0.4951236844062805, -1.9117859601974487, -0.06820027530193329, 1.7335495948791504, -0.124163918197155, -0.3927372694015503, -1.1623204946517944, -1.6429246664047241, -0.7980098724365234, 1.2715094089508057, -0.3275781273841858, 0.5518996119499207, -0.24781258404254913, -0.9102721810340881, -0.13652454316616058, -1.1286392211914062, 0.33461838960647583, 0.02257472090423107, -0.052281588315963745, -0.1917143315076828, 0.9593584537506104, 1.0467020273208618, -3.137248992919922, -0.1763715147972107, 0.21841509640216827, 0.09170776605606079, 1.4769648313522339, 0.09073547273874283, -0.9971199035644531, 1.4586336612701416, -0.4599498510360718, -0.1817019134759903, -0.6056151390075684, 0.39716240763664246, 0.7480067610740662, -1.1896272897720337, 1.2206722497940063, -0.08978818356990814, 0.7361924648284912, 0.35252630710601807, -0.5792513489723206, 0.3033648729324341, 1.0993270874023438, -0.17760783433914185, 0.4135604202747345, 0.5105010867118835, 0.24327167868614197, -0.665967583656311, 0.10332495719194412, -0.351676344871521, -0.015355839394032955, 0.9355013966560364, 0.7886308431625366, -0.49386361241340637, -1.111687421798706, -2.955170154571533, -2.033202886581421, -0.5057730674743652, 1.5342342853546143, -1.2856454849243164, 0.5148114562034607, 0.007765308953821659, 1.4740710258483887, 0.9873613715171814, 0.8343004584312439, -0.772153913974762, 2.0974912643432617, -0.5304150581359863, -1.1950162649154663, 0.14659906923770905, -1.095291256904602, 0.47006064653396606, 2.20281720161438, -0.3200260400772095, -1.5889846086502075, 1.5378414392471313, 1.313685417175293, -0.0662727952003479, 0.3485083281993866, 1.665399193763733, 0.7655713558197021, 1.4874218702316284, 1.7939352989196777, -0.8425044417381287, 0.6890787482261658, 0.816871166229248, -0.0527520552277565, -0.7462019324302673, 0.5407915711402893, 0.4738106429576874, 1.5374194383621216, -1.500889778137207, 1.8334447145462036, -0.4219440519809723, -1.9740525484085083, 0.27976158261299133, -1.4340773820877075, -0.3609890639781952, -0.0778774693608284, 0.7565564513206482, 0.9749404788017273, 0.6925343871116638, -0.04913078621029854, 1.4773094654083252, 0.4447536766529083, 0.09234602749347687, 0.6021495461463928, -1.0567492246627808, -1.1464215517044067, 0.6947543025016785, 0.14841879904270172, -0.35543984174728394, 1.9886815547943115, 1.7185028791427612, 0.39602604508399963], [-0.9077795743942261, 1.0363634824752808, -0.043071065098047256, -0.022939685732126236, 1.261755347251892, 1.8672268390655518, 2.445261001586914, -1.6411938667297363, -0.5238761305809021, 0.6203499436378479, -1.2978200912475586, 1.4996898174285889, 0.08992837369441986, -0.8777678608894348, 0.7143540382385254, 0.6563438773155212, 0.24635392427444458, -0.8811219930648804, -0.3682001531124115, -0.8653848767280579, -1.6972455978393555, -0.5226543545722961, 0.10017119348049164, -1.6114174127578735, 1.0568783283233643, -0.7420665621757507, 1.0211173295974731, -2.2629659175872803, -0.962092399597168, 0.17358709871768951, -1.1825519800186157, -0.09283126145601273, 0.6074524521827698, 0.20059345662593842, -0.28787100315093994, 1.0178248882293701, 0.8396470546722412, 1.1583733558654785, -0.4571595788002014, 0.6602648496627808, 0.6694723963737488, 0.9816243052482605, 0.09636291116476059, -2.7842183113098145, 0.49563828110694885, -1.0563194751739502, 2.111790657043457, 0.8380056619644165, 0.2924138307571411, 1.8282756805419922, -0.13740544021129608, -2.2108066082000732, 1.7822562456130981, -0.28197696805000305, -0.8076116442680359, 0.060140352696180344, 1.1149940490722656, 0.5469598174095154, -0.6753407716751099, 0.21148145198822021, 0.43981799483299255, -1.5954314470291138, -0.9413878321647644, 0.6299461126327515, 0.07625113427639008, 0.32546040415763855, 0.18645192682743073, 0.0843149870634079, -1.1291362047195435, -0.07054468989372253, -0.07015680521726608, -0.7601151466369629, -0.13694122433662415, -2.163252115249634, 0.12865105271339417, -0.9267727732658386, -0.227517768740654, -0.5331641435623169, -0.5357517004013062, 0.6822854280471802, -2.2730674743652344, -1.7270464897155762, -0.28166356682777405, 0.7844887375831604, -0.27626484632492065, 0.21682888269424438, -0.7161704301834106, 0.057798001915216446, 0.404506653547287, -0.4916979670524597, 1.4505465030670166, -0.43908727169036865, 0.22915706038475037, -1.6221505403518677, -0.10615799576044083, -1.0340644121170044, -1.2371406555175781, -1.1796975135803223, 1.5600942373275757, 0.9938865303993225, 0.5782370567321777, 0.05318978801369667, -1.0766077041625977, -0.9559682011604309, 1.13578200340271, -2.2610158920288086, -1.3724418878555298, 0.1575707346200943, 0.8256933093070984, 0.7231996655464172, -0.7894673943519592, -2.5089211463928223, -0.3817882835865021, -0.009148440323770046, 1.072997808456421, -0.37289541959762573, 0.1754394918680191, 0.9576283097267151, 0.5843384861946106, -0.07322310656309128, -0.19428852200508118, 2.2565102577209473, 1.523963451385498, -1.3427599668502808, -0.8386646509170532, -0.809289276599884, -0.7619946002960205, 0.042462170124053955, 0.6155184507369995], [1.6890660524368286, -2.441030263900757, 0.4906272292137146, 0.5202760696411133, -1.9372247457504272, 1.6663017272949219, 1.5868593454360962, -2.299044370651245, 0.1774480640888214, 0.41205424070358276, -1.391756296157837, 0.28280946612358093, -0.7068145275115967, 0.013241227716207504, -0.4392731785774231, 0.713306188583374, 0.4277876913547516, 0.1419275403022766, -0.23709990084171295, -2.1636083126068115, -1.5679266452789307, 0.15390558540821075, 0.4855124056339264, -0.4907158315181732, -0.3380967974662781, -0.49019569158554077, -1.2055060863494873, -0.8736307621002197, 0.9199965596199036, -0.788087785243988, 0.8483463525772095, 0.0017600746359676123, -0.44528263807296753, 1.379319667816162, 0.8728911876678467, -2.37335205078125, 0.11632033437490463, -0.7897360324859619, 0.3403732478618622, -0.7978134751319885, 2.328782320022583, 1.1929384469985962, -1.8003047704696655, 1.632712483406067, 0.44958260655403137, -1.6733964681625366, 0.7220125794410706, -0.3225141763687134, 1.545538067817688, -1.1689684391021729, -1.257808804512024, -1.5981148481369019, -2.537454605102539, -0.7099922299385071, -0.7165841460227966, 2.1367034912109375, -1.2022252082824707, 1.586696743965149, -0.2814614176750183, -1.6198139190673828, -0.9792835116386414, -0.2689378559589386, -0.3240942060947418, 0.8023476004600525, -0.30312737822532654, 0.5129252076148987, -1.0847645998001099, -0.26928940415382385, 0.36810141801834106, -0.7819792628288269, 0.7281532287597656, 0.40007534623146057, -1.318767786026001, -0.1285458207130432, -0.4016774594783783, -0.6178434491157532, -0.07946426421403885, 0.5884218215942383, 0.7618680000305176, 2.0339298248291016, 0.3150952458381653, 1.0131393671035767, -1.50083589553833, 0.7444180846214294, -1.3979066610336304, 1.0658193826675415, -0.948517382144928, -1.0549670457839966, 0.23912259936332703, 1.4027959108352661, -0.05251738429069519, 0.44718271493911743, -0.49552276730537415, -2.0090017318725586, 0.9503505825996399, -0.08391246944665909, 1.6959559917449951, -0.7716342806816101, -0.12086531519889832, 2.18144154548645, 0.28860899806022644, -2.0873141288757324, 0.2734006345272064, -0.16005398333072662, -0.6969890594482422, -0.004312852863222361, -1.8623093366622925, -1.0916577577590942, -1.595534324645996, -0.4411662518978119, -0.4968640208244324, 0.7927015423774719, -0.18959487974643707, 1.1546286344528198, -0.7569733262062073, 1.1621149778366089, 0.085994191467762, -0.5501235127449036, -1.1736786365509033, 1.4903547763824463, -0.8169520497322083, 0.050968654453754425, -1.0414097309112549, 0.553919792175293, -0.16732704639434814, 0.7759972810745239, -0.9582551121711731, -0.6913223266601562, 0.39988306164741516], [0.417163223028183, 1.1546900272369385, 0.44474321603775024, 1.8275336027145386, -1.7790509462356567, 0.24716345965862274, -0.5967617034912109, 1.5334280729293823, -0.3199866712093353, -0.8710981607437134, -0.8751880526542664, -0.0004006164090242237, 1.0071077346801758, 1.471400499343872, -1.2413420677185059, -0.5984821319580078, 0.7606077790260315, -1.1950963735580444, -1.949109673500061, 1.2686327695846558, 0.9200855493545532, 0.298954576253891, 0.07891587913036346, 1.0371503829956055, 0.7128003835678101, -0.4005894958972931, 0.4807455539703369, 1.7646366357803345, -2.0390632152557373, 0.8976375460624695, 1.184329867362976, -0.9380052089691162, 0.3920353055000305, -1.1353437900543213, -0.11082407087087631, 0.8572306036949158, 1.4314815998077393, -1.238407015800476, -1.2976316213607788, -1.9207472801208496, 1.3283871412277222, 0.1175195500254631, 0.6597407460212708, -1.0077664852142334, 0.36668428778648376, 0.19569748640060425, 0.8240057229995728, 0.973770260810852, -0.6639930009841919, 0.6785438060760498, 1.53513765335083, 2.0890398025512695, -1.4121898412704468, 1.3186161518096924, 0.6040998101234436, 0.7843623757362366, 1.6736431121826172, 0.7319294214248657, -0.19327600300312042, -0.34360387921333313, -0.587443470954895, 0.05312719941139221, 0.5042164325714111, -0.23108473420143127, 0.8762825727462769, 1.364155650138855, 0.011480376124382019, 1.3967008590698242, -0.39912864565849304, -0.3063032627105713, -1.0084054470062256, 0.9231467247009277, 0.9569639563560486, -0.0013569262810051441, -0.01035676896572113, -0.5032296180725098, 1.4693020582199097, 1.5778354406356812, -0.5372467637062073, -0.06884030997753143, -0.7672979831695557, -0.5268498659133911, -0.9459252953529358, 0.08866126090288162, -1.0208603143692017, -0.8987991213798523, -0.39000949263572693, 0.8607544898986816, 1.0400768518447876, -1.6571369171142578, -1.3511360883712769, -1.561748743057251, -2.580933094024658, 1.9880446195602417, -0.24401715397834778, 0.27305254340171814, -0.260272353887558, -0.5365188717842102, -0.8757514357566833, 0.3863389492034912, -0.9947708249092102, 0.753818929195404, 0.7887452840805054, -2.6830222606658936, -0.44631102681159973, -0.7742611169815063, -0.23568867146968842, -0.8915685415267944, -0.42238494753837585, 0.39610597491264343, -0.16054777801036835, 1.1477750539779663, 0.06825409829616547, 1.8811016082763672, -0.2871173620223999, -0.8484461307525635, 0.6817778944969177, 0.5098876357078552, -0.01741614378988743, 0.2571462094783783, -1.0514788627624512, 0.3395627737045288, 0.8951215744018555, 1.2519216537475586, -0.6525216102600098, -1.136867642402649, 1.385650873184204, 0.6008183360099792, 0.013244984671473503], [-1.558227777481079, 0.8309270143508911, 1.8671817779541016, 0.14286933839321136, 2.014826774597168, -0.7898015379905701, -0.3310825824737549, 0.723141074180603, 0.3333321213722229, -0.372544527053833, 0.3862294554710388, -0.7242739200592041, -0.43344366550445557, 1.203698992729187, -0.22766852378845215, 1.3744109869003296, -1.7838819026947021, -0.3917417526245117, -1.3088432550430298, 2.0268542766571045, -0.7279937267303467, 0.5347300171852112, -0.12887291610240936, 0.040946781635284424, -0.018891198560595512, 0.26413288712501526, -0.19991004467010498, 2.274453639984131, 0.028478294610977173, -1.3657467365264893, -1.052979826927185, -0.13766050338745117, 0.9279517531394958, 0.017742225900292397, 1.1737293004989624, 0.41893255710601807, -1.1291452646255493, -0.47389355301856995, 0.31326571106910706, -1.5890212059020996, 2.2539074420928955, 0.25509366393089294, -0.39265769720077515, -2.0241031646728516, 1.5983575582504272, 0.7757209539413452, 1.1273763179779053, 1.0979294776916504, -0.7265409827232361, -0.4515564441680908, 0.5964837074279785, -1.202526569366455, 0.6450367569923401, 0.9290685653686523, 0.1435021162033081, 0.25592315196990967, 1.4941058158874512, -0.06126490607857704, -0.5153049826622009, -0.7167884111404419, 0.36684519052505493, -0.37972354888916016, 1.3935428857803345, 0.8808246850967407, 0.7922307848930359, 1.3536946773529053, 1.8728021383285522, 0.5141904354095459, 0.4948534667491913, -1.1988829374313354, -0.8043393492698669, -1.2616270780563354, 0.9787917733192444, 1.2799891233444214, -1.4129235744476318, -0.22865243256092072, 1.832457423210144, 0.5159581899642944, -1.4829275608062744, 1.2423511743545532, 0.3367621898651123, -0.7580826282501221, 0.6505480408668518, 0.14711466431617737, -1.284400463104248, -0.28731411695480347, -1.0873664617538452, -1.9135191440582275, -1.68346107006073, 0.2084386944770813, -0.4435953199863434, -0.764866054058075, 0.22700323164463043, 1.720174789428711, 1.6700347661972046, -0.4507502019405365, -0.38700947165489197, -1.9930082559585571, 0.06970009952783585, -0.04149186238646507, 0.7197856903076172, -1.274479866027832, -0.31294283270835876, -0.04341061785817146, 2.1380763053894043, -0.5620442032814026, 0.15964452922344208, 0.35980936884880066, -2.1240978240966797, -0.7078944444656372, 1.2705163955688477, -0.03872569277882576, 0.925963819026947, 0.14937426149845123, -0.45236414670944214, 0.8765819072723389, 0.6051050424575806, 1.4895670413970947, -0.5587692260742188, 1.5552892684936523, 0.0706620067358017, -1.8913813829421997, 1.450853943824768, -1.0312901735305786, -0.4061877727508545, 0.5866528153419495, 0.9486744999885559, -0.9706781506538391, -1.3182086944580078], [1.5411895513534546, -0.18416941165924072, -0.003283069236204028, 0.6714054942131042, 1.1088708639144897, 0.1665552705526352, 0.12630246579647064, -0.3704599440097809, 0.08793730288743973, -1.128170132637024, -0.6136549115180969, -0.6558625102043152, -1.286685824394226, -0.8864767551422119, 0.15978117287158966, 0.7361796498298645, -1.3056806325912476, 0.6707003712654114, -0.5246396064758301, 0.78614741563797, -0.013968202285468578, -0.3334636390209198, 1.5454514026641846, 0.17165647447109222, 0.18920952081680298, -0.40257471799850464, -0.9731667637825012, 0.01320171169936657, 2.3928728103637695, 0.5711971521377563, 0.5238312482833862, -1.9375802278518677, -0.2565489113330841, 0.5195409655570984, -0.13467489182949066, 0.06197464466094971, 0.8486524820327759, -0.9032295346260071, 0.16874517500400543, 0.62285315990448, 0.9525579810142517, 0.05671599507331848, -0.37359943985939026, -1.4120069742202759, 0.20949074625968933, -1.3858493566513062, 0.37596312165260315, -2.1312735080718994, 1.3418512344360352, -0.2920769155025482, 1.1865339279174805, -1.0379582643508911, 1.028387427330017, 0.037674903869628906, -1.5134764909744263, -0.4844871461391449, 0.721554696559906, -0.7632016539573669, 0.11494042724370956, -1.1304045915603638, -0.2043258547782898, -1.817022442817688, -0.16619224846363068, 0.46288031339645386, -0.08228276669979095, 3.4453046321868896, -0.8585289716720581, 0.6042947769165039, 0.5951905846595764, 0.445965975522995, -0.32790035009384155, -0.9348786473274231, 0.7067997455596924, -1.123420238494873, -1.2026190757751465, 0.3050253391265869, 0.22120416164398193, -1.2983208894729614, 0.25166183710098267, 1.3323029279708862, 1.3432010412216187, -2.0323357582092285, 0.7500781416893005, -0.016186296939849854, 0.7327480912208557, -1.8356821537017822, 1.0357072353363037, 0.519802987575531, 0.6778032183647156, 0.32985493540763855, -0.9363986253738403, 0.4198814630508423, -1.032277226448059, -0.14783477783203125, 0.1665496677160263, -0.028643066063523293, 0.9838107228279114, 0.6052066087722778, 1.220037817955017, -1.363955020904541, -2.269968271255493, 0.24421505630016327, -0.3705817461013794, -0.49398067593574524, -1.4046199321746826, 1.5987403392791748, -0.6640890836715698, 0.7884701490402222, 0.07774811238050461, 0.5967400074005127, -1.0405112504959106, -0.20374974608421326, -2.12408185005188, 1.013883352279663, -0.18063877522945404, 0.7712559700012207, -0.19164347648620605, -2.132185697555542, 0.7956957817077637, -0.017272137105464935, -0.8720754981040955, 0.39256131649017334, 1.3585832118988037, -0.01899576000869274, -1.2724753618240356, -0.2189410924911499, -1.2740222215652466, -0.44698068499565125, -0.4386633336544037], [0.04591342434287071, -0.29919102787971497, 0.0499938540160656, 0.3224291205406189, 0.24473260343074799, -0.21301139891147614, 0.04225825145840645, -0.957768976688385, -1.0864442586898804, 0.5114546418190002, -0.34769877791404724, -0.7049168348312378, 0.022655116394162178, -0.6015366315841675, 0.05913171172142029, 2.034759283065796, 0.8001817464828491, -0.5734832286834717, 0.33302757143974304, -0.46493402123451233, 0.6851164698600769, 0.8032110929489136, 0.6420230269432068, 1.1935968399047852, -0.05242351442575455, 0.5561439990997314, -0.48340484499931335, -1.2907096147537231, 0.6005322337150574, -1.5776257514953613, 0.5980190634727478, 0.28745850920677185, 0.6367232799530029, -0.08204004913568497, -1.0490111112594604, -0.13974682986736298, 0.38418224453926086, -0.4770687222480774, 0.9914013743400574, 2.5359597206115723, 1.006583333015442, 1.1277369260787964, -1.6578717231750488, -0.09459955990314484, -1.9051841497421265, -0.8149508237838745, -1.507208228111267, 0.43363380432128906, -0.5492045283317566, -0.5486873984336853, -0.5497199296951294, -0.10915827751159668, -0.9610813856124878, 0.15396614372730255, -0.3031781017780304, -1.1236116886138916, 0.0972352921962738, -1.792950987815857, 0.4868568480014801, -0.8824810981750488, 1.3854748010635376, -1.574942708015442, 0.7158597111701965, 0.3723656237125397, 0.40815675258636475, -1.080729603767395, -0.23997607827186584, -0.36663544178009033, -0.8732462525367737, -0.4164050817489624, 0.10493894666433334, -0.0628872811794281, -0.2901233434677124, -0.72662353515625, -2.0962586402893066, -0.5813159346580505, 0.3525117039680481, -1.1927766799926758, -0.5283107161521912, -0.6674439907073975, 0.5564191341400146, 0.6909075379371643, -0.11184053122997284, -0.9353960156440735, 1.1954292058944702, 0.28641918301582336, -2.640230655670166, 0.10613316297531128, -0.5959703922271729, -1.2741049528121948, -0.5379965305328369, 0.0802617222070694, 0.5046094059944153, -0.3584003448486328, 0.4338911771774292, -1.5056061744689941, -0.43487781286239624, 1.323346734046936, -1.4999173879623413, -0.6160538196563721, 0.8940170407295227, -0.9852682948112488, 0.6602362990379333, -0.4461130201816559, 0.7603142261505127, 1.0935871601104736, 0.06372993439435959, -0.05210617929697037, -0.30025166273117065, -0.6437197923660278, -0.01854764111340046, -0.915307343006134, 0.9103650450706482, 0.25424909591674805, -0.6081242561340332, -0.9363258481025696, 1.764510989189148, 1.6586079597473145, -0.25751182436943054, 0.8044400215148926, -0.23266933858394623, 0.035080038011074066, 0.848257839679718, -0.7880819439888, -0.8374828100204468, 1.2374600172042847, -1.7598379850387573, 0.17314040660858154, 0.3772675395011902], [-0.6948173642158508, 1.6099507808685303, 1.0900437831878662, 2.6623730659484863, 1.083057165145874, 2.141129732131958, -1.0913058519363403, 0.7867059707641602, 0.6504517197608948, 1.701665997505188, 0.4716476500034332, 1.4506770372390747, 0.6664711833000183, 1.4647462368011475, -0.8669087290763855, -0.4095061719417572, -1.0508289337158203, -1.2273906469345093, -0.48100724816322327, 0.6385209560394287, -0.0025733942165970802, -0.4362235963344574, -0.5171697735786438, -0.31685924530029297, -0.19049061834812164, 0.57171231508255, 0.6239434480667114, -1.821093201637268, 0.743844211101532, 0.6616280674934387, 0.35763993859291077, 0.21215209364891052, 1.186914324760437, 0.047641195356845856, -0.3858683407306671, -0.7868550419807434, -0.39909783005714417, 0.37322431802749634, 0.12082241475582123, -0.42691850662231445, -1.13547945022583, 0.5917463898658752, 0.873073160648346, 0.2800210416316986, -1.8757888078689575, 0.16233551502227783, 0.8796949982643127, -0.7410998940467834, -1.045203685760498, 0.5411837100982666, 0.12334214150905609, 1.4138330221176147, 1.2020082473754883, 0.061065949499607086, 0.7622612714767456, -0.5161820650100708, 1.4025522470474243, 0.0018890734063461423, -0.3515259921550751, 1.230963110923767, -0.5648446083068848, -2.061338424682617, 0.21231698989868164, -0.3089890778064728, -1.0692731142044067, -0.007642804645001888, 1.2303723096847534, 1.564205288887024, -0.6656485795974731, 0.624883770942688, -0.11435295641422272, -1.1404592990875244, -1.4988889694213867, 0.5229070782661438, -1.3162792921066284, 0.4127068817615509, 1.5765056610107422, 0.7376530766487122, 0.4453137516975403, 1.3324054479599, -0.4917283356189728, 0.07334506511688232, 0.6730576157569885, -0.043655987828969955, 0.8720688819885254, -0.8555366396903992, -1.3147464990615845, -0.13810114562511444, -0.7635704874992371, -0.5197858214378357, -0.4701870381832123, -1.0318890810012817, -0.35492974519729614, 0.8091706037521362, -1.4527912139892578, 0.577972412109375, -0.5151939988136292, 0.5516872406005859, 0.19745109975337982, 1.1119743585586548, 1.543761968612671, 0.6752822399139404, -0.7568218111991882, -1.1717307567596436, -1.9956642389297485, -0.5649347305297852, -1.0599792003631592, -0.6742156744003296, 1.2862401008605957, 1.8742167949676514, 0.3280217945575714, -0.22768300771713257, -0.5891578197479248, 2.4137251377105713, 1.6778664588928223, 1.1511731147766113, -0.2630975544452667, 1.9782261848449707, -1.2057628631591797, -1.1816133260726929, 0.8188117742538452, -0.1414930373430252, 0.05061876401305199, 0.16795268654823303, -0.4749941825866699, -0.6674383282661438, 1.042744517326355, 1.3785606622695923, 0.04645504802465439], [-1.3289695978164673, -0.11771277338266373, 1.345952033996582, 0.2496211975812912, 0.5729039311408997, 1.1900395154953003, -0.5079788565635681, 0.5273342728614807, -0.554390549659729, 1.0225496292114258, 1.051160216331482, -0.9821838736534119, 0.3530261516571045, 0.15990373492240906, -2.2006707191467285, -2.207493543624878, 1.2803480625152588, 0.05296271666884422, 1.479268193244934, -0.5811040997505188, -0.965171754360199, 0.11247718334197998, 0.03595582768321037, -1.0564191341400146, -1.2145845890045166, -0.6575677394866943, -2.4127159118652344, 1.278509497642517, 0.18934622406959534, -0.3753819763660431, 0.9227323532104492, 0.6977693438529968, -1.5037039518356323, 0.35740312933921814, -0.2762567698955536, 0.13690786063671112, 0.9875506162643433, -0.14270305633544922, -1.1384310722351074, -0.5053598284721375, 0.938266932964325, 2.4572947025299072, -0.22137606143951416, 1.120504379272461, -0.6658563613891602, -0.7574316263198853, 0.6723228693008423, -0.41364139318466187, -0.723405659198761, -2.1411495208740234, 0.07697160542011261, -0.7590716481208801, 0.526757538318634, -0.7463282942771912, -0.9356473684310913, 0.03611300513148308, -0.3514484167098999, -0.8430505990982056, 0.006812810432165861, 0.2045517861843109, 0.7765992283821106, 0.9326575398445129, -0.653567373752594, -0.5512444972991943, -0.4411216080188751, -2.2210371494293213, 0.8815943598747253, 1.0281271934509277, -0.16231931746006012, -0.2339959442615509, 0.7572911977767944, -0.4645102620124817, 0.646815299987793, 0.2626437544822693, 0.6554644703865051, -0.3281264007091522, 0.8585793972015381, -2.1888365745544434, -1.4104783535003662, 1.3187037706375122, -0.13269703090190887, -1.3059686422348022, -0.7407732009887695, 0.2810352146625519, -0.24316072463989258, 0.6126612424850464, -0.7807631492614746, 1.7762757539749146, -0.1942349672317505, -1.4764024019241333, -0.6232784390449524, 0.0842733383178711, 1.0822612047195435, 1.3242741823196411, -0.9213895201683044, 0.2664814889431, -1.1876866817474365, -0.17407114803791046, -0.8677811622619629, -1.2193766832351685, -1.2032800912857056, 0.8231016993522644, -0.47521719336509705, -0.5049197673797607, 0.7372084856033325, -0.04774288460612297, 1.59231436252594, -0.9833200573921204, 1.3776557445526123, -0.32222193479537964, -0.08687853813171387, -1.6321974992752075, -0.285410076379776, 0.12509304285049438, 0.415517657995224, -1.9686094522476196, 0.9870019555091858, -0.7482777237892151, -1.649908423423767, -0.1625310778617859, -1.7412506341934204, -2.109149932861328, 1.9718552827835083, 1.5846835374832153, 0.9330034255981445, 0.4703769385814667, 0.9855469465255737, -0.9800087809562683, 1.2185635566711426], [0.3128749430179596, -1.917518138885498, -1.3104007244110107, 1.3423115015029907, 1.5343161821365356, 0.4384959936141968, 1.0887911319732666, -0.05811760202050209, 0.4729607403278351, -0.07491077482700348, -0.46337130665779114, 0.44345778226852417, -0.07411106675863266, 2.258232831954956, 0.9407507181167603, 1.4458931684494019, -0.0417044572532177, 0.3495807647705078, 1.7279413938522339, 0.7537564635276794, -0.18040885031223297, 0.8365222215652466, 0.6198457479476929, 1.3022280931472778, -0.85978764295578, 0.5898705124855042, 0.3322010338306427, 0.3875376284122467, 0.3686481714248657, 0.951505720615387, 0.13770465552806854, 0.4274280369281769, 0.2853453457355499, -0.13874061405658722, 0.5581915378570557, 0.43229180574417114, 0.8100066184997559, -0.027995862066745758, 0.9802541732788086, -0.24101147055625916, 1.584452748298645, 1.1635282039642334, -1.0174038410186768, 1.5275354385375977, 0.3914763033390045, 0.7164400219917297, 0.417000412940979, -1.0159581899642944, 1.013410210609436, -1.5412184000015259, 1.3754831552505493, -0.16070128977298737, 0.8629677295684814, -0.06462851911783218, 0.6195520162582397, 0.8032408356666565, 1.3529255390167236, 0.03893917798995972, -0.6158055067062378, -0.276675283908844, 1.2230756282806396, 1.9711917638778687, 1.1132346391677856, -0.8994414210319519, -0.6168720126152039, -0.24976512789726257, 0.8541656136512756, -0.7524269223213196, 0.5390266180038452, 0.4187978208065033, -0.8243162035942078, -0.06522797793149948, 1.1751649379730225, -0.21874119341373444, 1.1987829208374023, -1.1195944547653198, -1.2271854877471924, -0.5161731243133545, 0.039231352508068085, -0.15232186019420624, -0.9958722591400146, 0.3605950176715851, 0.6640686988830566, 0.5908555388450623, -0.8171773552894592, -1.1494718790054321, 0.3702254295349121, 0.4071483612060547, -1.032036304473877, 0.4693776071071625, -0.3126567304134369, -0.30639591813087463, -1.4718040227890015, -2.120255947113037, -1.3985143899917603, 0.8712697625160217, 0.17100919783115387, -1.3070452213287354, 0.25829342007637024, 0.23738989233970642, 0.804478645324707, 0.861251711845398, -0.7103101015090942, -0.8258633613586426, -1.9065916538238525, -1.4938560724258423, 0.648598849773407, -0.4802834987640381, -0.804190993309021, -0.8973429799079895, 1.8440250158309937, 0.17842736840248108, -0.8183411359786987, -0.046430930495262146, -1.3170208930969238, 1.4019556045532227, -0.15359671413898468, -0.31702545285224915, 0.28210151195526123, 0.4451060891151428, -0.8927761316299438, 0.5263254642486572, 0.6045697927474976, 0.9684786200523376, 0.02235775999724865, 1.8146204948425293, -0.04126125946640968, -0.3259168863296509, -0.10055093467235565], [-0.11147541552782059, 0.3600318729877472, -1.556209921836853, 0.12479941546916962, 0.1859058439731598, 0.8009728789329529, -0.1303524374961853, -1.5440261363983154, -0.7041314840316772, -1.1760022640228271, 1.1509474515914917, 0.661158561706543, 0.7612143158912659, 1.6548820734024048, -0.445620596408844, 0.6026343703269958, 1.242458701133728, 0.8067712783813477, -0.04751969501376152, -1.4173389673233032, 0.3561207056045532, 0.3499631881713867, 0.6134455800056458, 0.5972705483436584, -1.4508416652679443, -0.26556694507598877, -0.08663404732942581, 1.3963419198989868, 1.657326102256775, 0.726594090461731, -0.4015270173549652, -0.43763506412506104, -1.5002992153167725, -0.2865084111690521, 1.1560899019241333, 1.9893829822540283, 1.4033055305480957, 0.1926317662000656, 0.3823440968990326, 0.9604846835136414, -0.27161139249801636, -0.3520050048828125, -0.11376705020666122, 0.12008804082870483, -1.4186807870864868, 0.3205435276031494, -1.248505711555481, 1.7856513261795044, -1.421934723854065, -0.9528136253356934, -0.9674298167228699, -0.35851848125457764, -0.8586064577102661, -0.20286984741687775, 0.8305059671401978, 0.22034689784049988, -1.8266403675079346, -0.5524357557296753, -0.7323105931282043, 0.09173285961151123, 0.6537556052207947, 0.8691383600234985, 1.1320972442626953, 0.10795791447162628, 1.2213916778564453, 1.092905879020691, 0.6790424585342407, -0.42851918935775757, -0.3311859965324402, -0.47691112756729126, 1.340056300163269, -2.2023234367370605, 2.1159420013427734, -0.6561180353164673, 1.2464972734451294, 0.5590091347694397, 0.8264118432998657, -0.9369728565216064, -0.3654675781726837, 0.8792586922645569, 0.9377157688140869, -0.6575941443443298, -1.8431315422058105, -0.73886638879776, -0.40990549325942993, 0.6767335534095764, 0.996159553527832, 1.9915295839309692, 0.4083476960659027, 1.2956305742263794, -1.6051361560821533, 1.3662630319595337, -0.36169740557670593, 0.06942244619131088, 0.6222514510154724, -1.1015878915786743, -0.8723343014717102, -0.8046585321426392, -1.1494954824447632, -0.3065349757671356, 0.35748863220214844, -1.5198066234588623, 1.1180719137191772, -0.2685939073562622, -0.12949667870998383, -0.37103739380836487, 0.4360775947570801, -0.2165716588497162, 1.4618971347808838, -1.301082730293274, 0.13666678965091705, 0.33667585253715515, 0.40302741527557373, -0.07544214278459549, 2.010939121246338, -0.5293413400650024, 1.243971824645996, -0.5886473655700684, 0.17200283706188202, 0.34284329414367676, 0.6134722232818604, -0.45773568749427795, 1.0177178382873535, -1.2166897058486938, -0.4774535298347473, -1.2244925498962402, -0.1834116131067276, 1.1354832649230957, -0.7109368443489075], [0.9030893445014954, 0.5741637349128723, 1.0563265085220337, -0.606073796749115, -0.42007315158843994, 0.49687638878822327, 1.4331737756729126, -0.7856830954551697, 1.2900946140289307, 0.04972715675830841, 1.3048027753829956, 0.4185645282268524, -1.80055570602417, 1.05337655544281, -0.2584543526172638, 0.185477614402771, 0.1634412258863449, 1.3833783864974976, -0.009444206953048706, 1.5581742525100708, -0.2128152698278427, 0.6336089968681335, -1.0819777250289917, -0.28962913155555725, -0.5851694941520691, 0.9774372577667236, 0.21110621094703674, -0.054554808884859085, -0.06754238158464432, 1.4496068954467773, -1.1480529308319092, 1.2553046941757202, -0.20360706746578217, 0.04236451908946037, 1.2850539684295654, 1.0231209993362427, -0.45014718174934387, 0.8206652402877808, -1.2896348237991333, -0.2606554627418518, -0.5164152979850769, -0.9319223761558533, -1.28652822971344, 0.004732575733214617, -0.11703057587146759, -0.7220395803451538, -1.636172890663147, 0.19497835636138916, 1.2037382125854492, 0.5085461735725403, 1.4839808940887451, -0.4701175391674042, 1.0516927242279053, 0.2909633219242096, 0.6767065525054932, 1.4124133586883545, 1.413212776184082, 0.09583299607038498, 0.3607580065727234, -0.4224745035171509, -1.1932505369186401, -1.4914926290512085, 1.0149480104446411, -1.0814963579177856, -2.968569755554199, 0.14525102078914642, 0.09299034625291824, -0.562462329864502, -1.0574387311935425, 0.8973944187164307, -1.2182246446609497, 0.1200561448931694, 1.3805207014083862, -0.11619317531585693, 0.7962165474891663, 0.014590209349989891, -1.8204022645950317, 2.228036880493164, -0.6350130438804626, -0.6974313855171204, 0.09019220620393753, 0.5112284421920776, -0.2412642538547516, 0.47535988688468933, 1.5315189361572266, 1.599692702293396, -0.7065688967704773, -1.23404860496521, -0.00784787442535162, -0.5690332055091858, 1.3285773992538452, -0.04589632526040077, 0.7942746877670288, -0.3471776843070984, 0.14095188677310944, 0.8425042033195496, -0.7645429372787476, 1.0434207916259766, -0.30786025524139404, 0.40503934025764465, 0.004295263905078173, 0.6144117712974548, -0.5032522082328796, 1.2219866514205933, -0.129856675863266, -0.8232839703559875, 2.3162941932678223, 0.2980951964855194, -0.718722403049469, -0.46485635638237, -0.3768216669559479, 0.48338010907173157, -0.17623168230056763, -0.2510415315628052, -0.8443188071250916, -0.42771098017692566, -1.942333698272705, 0.8524428606033325, -0.8287713527679443, 1.8196550607681274, 0.8078201413154602, 1.0180494785308838, 0.7116308808326721, 0.31598204374313354, -0.221885547041893, -0.7302193641662598, -1.795734167098999, 2.3115241527557373, -0.08846469223499298], [0.27639639377593994, -0.5328608751296997, -1.715726375579834, 0.6866796612739563, 0.6956767439842224, 1.3269716501235962, 1.189051628112793, 0.8563737273216248, 0.5893195867538452, -1.4790644645690918, 0.4958128333091736, -1.1169134378433228, 1.903361201286316, 0.056359417736530304, -0.04645023122429848, -1.232778549194336, -0.005651599261909723, 0.861637532711029, -0.5214516520500183, -1.2301819324493408, -0.8930661082267761, -1.05656099319458, 0.5964155197143555, 1.9045841693878174, 0.0414612852036953, -1.113384485244751, 1.1931967735290527, -1.4736151695251465, 1.1622474193572998, -0.40409332513809204, 0.10590572655200958, -1.2191959619522095, -1.6906774044036865, -0.6519822478294373, -1.353258728981018, -1.0088895559310913, -0.6826693415641785, 0.8123275637626648, 1.0106219053268433, 0.6851139664649963, -0.5291927456855774, -1.0869576930999756, -0.442476361989975, 2.127431631088257, 2.5411224365234375, 0.22252441942691803, -0.8156658411026001, 0.3330133557319641, 1.090645670890808, -1.5949130058288574, -0.677116334438324, 0.811586320400238, 0.6713244915008545, 0.08216606080532074, -0.29841700196266174, -1.9002286195755005, 1.1196208000183105, -1.15245521068573, 0.3999805450439453, -0.24836070835590363, -0.6667104363441467, 0.5814133286476135, 0.8907269239425659, 0.9397557973861694, 0.8932117223739624, 0.8801330924034119, -0.6401034593582153, 1.168591022491455, 0.7684162259101868, -0.022253375500440598, 0.31229469180107117, -1.0879688262939453, 0.504051685333252, 0.219722718000412, 1.3454138040542603, -1.838039755821228, 1.2326157093048096, 0.11207947880029678, -1.0829271078109741, -0.018723059445619583, -1.355839729309082, -0.08765503019094467, 1.0348538160324097, 1.2487876415252686, -0.02330373041331768, -1.6707143783569336, -0.4515567719936371, -0.4611128866672516, -1.8488361835479736, -0.8267947435379028, 0.27499768137931824, 0.6841758489608765, 0.8277673125267029, 0.8441667556762695, 0.17675261199474335, 0.3960607945919037, -0.41978326439857483, -0.27050915360450745, 0.33349865674972534, 0.019274741411209106, -0.7313527464866638, 0.16819876432418823, -1.4681427478790283, 0.2151975929737091, -0.24884475767612457, 0.32760024070739746, 1.0678352117538452, -0.3193848729133606, 0.2437179982662201, 0.2771061360836029, -0.5481646060943604, -1.0024031400680542, -0.8728296756744385, -1.0971381664276123, 0.16747696697711945, 1.7525908946990967, -1.7370866537094116, -0.6452914476394653, 0.2233237475156784, 0.13641735911369324, -1.2204475402832031, -0.941651463508606, -0.5206168293952942, -0.9653244018554688, 1.1178250312805176, -0.06678560376167297, -0.3825208246707916, -0.4866195619106293, -0.24422265589237213], [0.15752756595611572, 1.2392276525497437, -0.25792068243026733, -1.125260829925537, 1.6057945489883423, -0.16658610105514526, 0.31686288118362427, -0.5006212592124939, 0.401812344789505, 0.7569321393966675, -0.09552206099033356, -0.12194009870290756, -1.409441351890564, 0.22121796011924744, 0.9933170080184937, 1.2666360139846802, 1.5205663442611694, -0.3380281627178192, 0.9014230370521545, -0.9606561064720154, -1.2200145721435547, -1.3579593896865845, -0.5432639718055725, -1.6809191703796387, 0.8802577257156372, -0.12403629720211029, -0.4372788667678833, 1.0979793071746826, 0.2523024082183838, 0.0718483179807663, -0.14610892534255981, -1.0123881101608276, -2.9896979331970215, -1.5391513109207153, 1.0532535314559937, -0.3253447711467743, 1.5441617965698242, 0.04883062094449997, 0.5108921527862549, 1.3405910730361938, -0.024884777143597603, -0.5955836176872253, -0.1270465850830078, 1.1172118186950684, -0.17155592143535614, -0.8252575993537903, -1.2079534530639648, -0.5068541765213013, -0.6839678287506104, 0.4526309370994568, 0.7668209075927734, 0.2834968566894531, 1.4324240684509277, -0.32869717478752136, 1.7128677368164062, -0.25849947333335876, 0.9347107410430908, -1.8739030361175537, -0.3283156156539917, -0.5941660404205322, -0.6314040422439575, -0.03163440153002739, 0.48987308144569397, -2.159442186355591, 1.3345615863800049, 0.15300410985946655, -0.6957477331161499, 1.8739911317825317, 0.6241953372955322, 1.1223366260528564, -0.2652062773704529, 0.9758251905441284, -0.01240701973438263, -0.5593104362487793, 0.2585151791572571, -1.830245852470398, -0.9838109612464905, 1.1046648025512695, 1.0918561220169067, 1.3367542028427124, -0.6794192790985107, 0.7710015177726746, 0.7839746475219727, 0.1676623374223709, -0.7660102844238281, 0.23793672025203705, 0.5602668523788452, -1.297713279724121, -1.825671911239624, -0.9442660808563232, 0.49213096499443054, 1.6930383443832397, -0.3316698968410492, 2.253631114959717, -0.2395445853471756, -1.3787277936935425, 0.52839595079422, -1.9842690229415894, -0.5351819396018982, -0.6575943827629089, -0.3264774978160858, -2.6219589710235596, -0.1625434309244156, -0.9269726872444153, -0.15429948270320892, 0.17844097316265106, 1.1108263731002808, -0.32729464769363403, 1.370361566543579, 0.500523567199707, -0.36156800389289856, 0.7543001770973206, 1.1335750818252563, 1.1883857250213623, 0.0780821293592453, -0.4529629647731781, 0.2783409357070923, 0.5224170088768005, -1.747864007949829, 0.34428608417510986, 0.0866793543100357, 0.6166783571243286, 0.8414573669433594, -1.237453579902649, -1.8361732959747314, -1.0431873798370361, 0.22786922752857208, -2.8589134216308594, -0.03147827088832855], [-1.0634856224060059, 0.42883172631263733, 1.190932273864746, -0.1446925401687622, 0.10662984102964401, 1.6773654222488403, 1.5526169538497925, 1.9452040195465088, -1.8154114484786987, -0.29504165053367615, -1.2596486806869507, 1.2167232036590576, 0.08492466062307358, -0.7787351608276367, 0.5948851108551025, 0.7897440791130066, 1.7318726778030396, 1.2183609008789062, -0.20130136609077454, -0.577460527420044, 1.7324681282043457, -1.4711403846740723, -1.5154130458831787, -1.3046791553497314, 1.1915279626846313, 0.024021076038479805, -0.34099388122558594, 0.11192572861909866, 0.3456597626209259, -0.5011993050575256, -0.741807222366333, -1.2800476551055908, -0.05649589002132416, -0.045078009366989136, 0.5598872900009155, -0.20415718853473663, 0.11514493823051453, -0.9238176941871643, -1.088989496231079, 0.3401457667350769, -1.3723280429840088, 1.1750744581222534, 0.554823100566864, 0.01908102259039879, -1.1778779029846191, 0.9090544581413269, -0.3317842185497284, -0.6445196866989136, -1.792388916015625, 1.7696609497070312, 2.1859652996063232, 1.6863954067230225, -0.5067524313926697, 0.09299042820930481, 0.8629440665245056, 0.5750371813774109, -0.2854672968387604, 0.8746737837791443, 0.5468876957893372, -1.609818696975708, -0.16324418783187866, 0.09915164113044739, -0.2504507303237915, 0.21908104419708252, -0.44619032740592957, 0.42473527789115906, 1.9408293962478638, 0.4590253233909607, 0.6816633343696594, -2.586733341217041, 0.8309527635574341, 0.622861921787262, 0.25547611713409424, 0.4383518099784851, -1.228368878364563, 0.18977507948875427, 0.6705241203308105, -0.2512705326080322, 2.0898866653442383, 0.6240972876548767, 0.34316709637641907, 1.5750012397766113, -2.1270463466644287, 2.1200480461120605, -1.332327127456665, -0.45878174901008606, -0.23881962895393372, 1.0191844701766968, 0.35808682441711426, 1.2556877136230469, -0.45652881264686584, 0.07417565584182739, -0.8430954217910767, -0.00623107748106122, 1.2280253171920776, -0.0605221651494503, -0.08787666261196136, 0.7878373265266418, -1.63141667842865, -0.31030309200286865, 1.7032774686813354, -0.48622921109199524, 0.7486170530319214, 0.2001231610774994, 1.5979479551315308, 0.6924518346786499, 0.45108285546302795, -0.9234623908996582, -0.32694822549819946, 0.8075624704360962, 1.0990327596664429, 0.18670646846294403, 0.29215770959854126, -1.373550295829773, -0.04094824567437172, -0.11058925092220306, 1.714551568031311, 0.22237269580364227, 0.16779878735542297, 0.6751211285591125, 0.39847272634506226, -0.5392506122589111, -0.07526077330112457, 0.01589883863925934, 0.11804065108299255, -0.1680886298418045, 0.9321614503860474, -0.6810091733932495, -0.4973224401473999], [2.5182371139526367, -0.7142865657806396, -0.5276109576225281, 1.1577390432357788, -1.0876482725143433, 0.6149601936340332, -2.3946850299835205, -0.6938826441764832, -0.26895418763160706, 3.444477081298828, -0.3038139045238495, 0.09897685050964355, 0.5557535886764526, -0.29912668466567993, -0.3786091208457947, -0.759059488773346, 1.773621678352356, -0.6849048733711243, -0.4574310779571533, 2.414059638977051, -0.03744350001215935, 1.092158317565918, 1.862918734550476, -0.8226561546325684, -0.004720926284790039, 1.256304383277893, 0.12071564048528671, -0.014999926090240479, 0.2578790783882141, 1.4822503328323364, -0.7683005332946777, 0.07059473544359207, 0.24857833981513977, -1.2962100505828857, -0.5302674174308777, 0.7097295522689819, 0.6914330124855042, 0.3098710775375366, 0.9564694166183472, 0.03892079368233681, 0.3179854452610016, 0.8947535157203674, 0.6054344773292542, -0.9976561069488525, -0.9977819919586182, 0.11798358708620071, -0.7522361278533936, 0.9303486347198486, 0.7721294164657593, -0.5978691577911377, 1.236145257949829, -0.8685434460639954, 0.978291928768158, 0.10716031491756439, -0.3203505575656891, 1.1489323377609253, 0.1085161417722702, -0.04010453820228577, 1.499344825744629, -1.205357313156128, 0.2754787802696228, 1.0230712890625, 1.6557624340057373, -0.3721988797187805, 0.284629225730896, -0.5804932117462158, 0.23490557074546814, -2.0332608222961426, -1.2277486324310303, 1.0534982681274414, -0.13415688276290894, 0.34343138337135315, 1.472917914390564, -0.21519678831100464, 0.04963667318224907, 0.07504469901323318, 0.40038958191871643, 0.31991636753082275, -0.8532515168190002, -0.40001559257507324, -1.5649707317352295, 0.4259195029735565, 0.10617514699697495, -1.980699062347412, -1.3097165822982788, -0.6660029292106628, 0.4303056597709656, -0.03704136982560158, 0.14681974053382874, -1.0626851320266724, 2.900904893875122, 0.7881965041160583, -0.9600582122802734, -0.5728765726089478, -0.5804561972618103, -0.23302467167377472, 0.42249834537506104, 1.6976537704467773, 1.5144439935684204, -0.47583186626434326, -1.0156885385513306, 0.42443355917930603, -0.15795034170150757, 0.9406713843345642, 1.5240387916564941, 0.0143109867349267, 0.231913223862648, 0.3916481137275696, 0.45372095704078674, -1.3820489645004272, 1.4204999208450317, -0.4012340307235718, -0.5448099970817566, -1.0822479724884033, -0.2758327126502991, 1.1224805116653442, 1.5976473093032837, -1.0701658725738525, 1.4353766441345215, -0.32762810587882996, 0.10857633501291275, -0.7975982427597046, 0.4648340344429016, 0.5473597049713135, -0.2881559133529663, 0.2663203179836273, -1.3302956819534302, -0.020265066996216774, 1.063325047492981], [-0.07005789875984192, 0.7698960304260254, -0.47911158204078674, -0.9521915316581726, 0.3691045641899109, -0.5699640512466431, 0.32503947615623474, 0.6738888025283813, 1.0997943878173828, -1.1980284452438354, -1.2504212856292725, 0.8220961689949036, -0.44595029950141907, 0.38427549600601196, 0.1525001972913742, 1.8130871057510376, -0.36008813977241516, -0.10276180505752563, -0.7321010231971741, 0.7458969950675964, -0.9880917072296143, -0.8062997460365295, 0.5680783987045288, 0.45273736119270325, 0.09203305095434189, 0.7827513217926025, 1.069165825843811, -0.8593342900276184, 0.16151125729084015, 0.9004565477371216, 0.5283516049385071, -0.26082053780555725, -1.284157633781433, -0.013266398571431637, -0.594397246837616, -1.2024309635162354, 1.3526148796081543, -0.9820733666419983, -0.75457364320755, -0.5893279910087585, -0.2825321555137634, -1.600947380065918, -1.0583723783493042, -1.007556438446045, -0.2973531186580658, 1.252097487449646, 1.1999952793121338, 0.4556035101413727, -0.14218035340309143, -0.7554712891578674, -1.160606026649475, -0.5070129036903381, 1.0951604843139648, 1.7660305500030518, -0.15911753475666046, 0.5357192754745483, 0.019487250596284866, 0.6499073505401611, 0.6156895756721497, 0.9841321110725403, 1.3949307203292847, 0.24826684594154358, -0.875421941280365, -1.2222784757614136, 0.9819146394729614, -0.9710947275161743, -0.8722643852233887, -0.5566832423210144, -0.06848929077386856, 0.7958724498748779, -0.043433044105768204, 0.5767722129821777, 0.18501333892345428, -0.6865440607070923, 0.6401968598365784, 1.7311609983444214, -1.011468768119812, 0.8411297798156738, -1.0108845233917236, -1.1987158060073853, -0.34466758370399475, -1.2109938859939575, 0.29799309372901917, 0.9558854699134827, -0.7094707489013672, -0.5572123527526855, 0.42125141620635986, 1.0220547914505005, 1.312444806098938, -0.5875076651573181, -2.354245185852051, -0.24762296676635742, -1.3993229866027832, 0.2877443730831146, 0.9719923734664917, 0.5401706695556641, -0.9359092712402344, -1.8414686918258667, -1.4191042184829712, 0.23520420491695404, 1.7401347160339355, -0.8463889360427856, 0.09896401315927505, -1.6698743104934692, 1.3105729818344116, 1.0611134767532349, 1.0663807392120361, -0.668386697769165, -0.6685166954994202, 0.471851110458374, 0.7855756878852844, 1.317406415939331, 0.26345527172088623, -0.8038962483406067, 0.11445330828428268, 0.4279695451259613, -0.6217666268348694, 0.9677360653877258, 0.38623690605163574, -0.5370222330093384, 1.3128763437271118, 0.6129024624824524, -0.7554031610488892, -0.4622461795806885, 0.5037014484405518, 1.1136022806167603, -0.10158097743988037, 0.35475170612335205, -0.10103455185890198], [0.5813252329826355, -0.08592339605093002, 0.06960547715425491, 0.6702381372451782, 0.5059227347373962, -0.22901804745197296, -0.931624174118042, 0.9564000964164734, -1.3679981231689453, -0.6311606168746948, -1.9398618936538696, -1.7335139513015747, 0.7573232650756836, 0.5435883402824402, 0.8895587921142578, 0.03297305852174759, -0.08674673736095428, -1.2909314632415771, -0.15702606737613678, -0.4675361216068268, 0.31460821628570557, -0.30099552869796753, -0.1854768544435501, 0.9232392907142639, 0.6552798748016357, 0.6207098960876465, -0.1999407708644867, 0.5356776714324951, 0.3185736835002899, 1.3586583137512207, -1.047167181968689, -1.4024031162261963, 1.8475637435913086, -1.2560993432998657, -1.1471208333969116, 1.1026469469070435, -0.8686867356300354, -1.3226314783096313, 1.9334042072296143, -0.9340093731880188, 0.7545823454856873, 0.30657702684402466, 0.3305439352989197, 0.2511657774448395, 1.1738317012786865, 0.01783890090882778, -0.44204849004745483, 0.40684130787849426, -1.1337015628814697, 0.2370457798242569, 0.2724091410636902, -0.5467803478240967, -0.47415539622306824, 0.4093334972858429, -0.5865582227706909, -2.076733112335205, -0.030539190396666527, 0.14623712003231049, -0.6231617331504822, 1.8992705345153809, -0.5652016997337341, 0.8077355027198792, -1.4188178777694702, 1.4416790008544922, -1.8281018733978271, -2.0847413539886475, 0.026241779327392578, 0.14615152776241302, 0.03023584745824337, 0.201798677444458, 0.6405215263366699, -0.8408301472663879, 1.103568434715271, -0.5772706866264343, -1.4720531702041626, 0.03535912185907364, -0.648309051990509, 0.042243100702762604, 1.8361097574234009, 0.1626673936843872, -0.20535165071487427, 0.2217492014169693, 0.39999574422836304, -0.7284286022186279, -0.6553804278373718, 1.1171636581420898, 1.8832186460494995, -0.4595944583415985, -2.751096487045288, 0.1188843846321106, -0.25566545128822327, 1.095406413078308, 0.746088445186615, 1.198852777481079, -0.1529148519039154, -1.2699781656265259, -0.9465659856796265, -0.3588918149471283, 1.0310964584350586, -0.8994836211204529, 2.1142516136169434, 0.531039834022522, -2.2270143032073975, 2.126368999481201, 0.31470435857772827, 1.1415438652038574, -0.5509907007217407, -1.1775192022323608, -0.20207135379314423, 1.3568977117538452, -1.8214770555496216, -0.7865898609161377, -0.9750547409057617, 0.6646426320075989, 1.1167664527893066, -0.5607383847236633, 0.09673887491226196, -1.336731195449829, 0.8692544102668762, -1.9295246601104736, -0.7877047657966614, -0.2904115915298462, -0.018629757687449455, 2.2850866317749023, 0.5729732513427734, -0.32248353958129883, 1.3962135314941406, 1.2711818218231201, 0.2735220193862915], [-0.2615082859992981, 2.3806984424591064, -1.2777990102767944, 0.397055447101593, 0.051169559359550476, -0.3034655749797821, 0.012934639118611813, -0.8422983288764954, -0.6285302639007568, -0.1525776982307434, -1.0094724893569946, 0.759288489818573, 0.5372923016548157, -0.6607300043106079, -0.8838731050491333, -0.2959972321987152, -1.492148995399475, -0.44603174924850464, -0.615479052066803, 0.2724791467189789, 0.9077752232551575, 0.4678928554058075, 0.7826413512229919, -0.9338388442993164, 0.4424734115600586, -1.8134747743606567, -1.132521152496338, 0.7221938967704773, -0.5640661120414734, -1.0210188627243042, -1.1523947715759277, -0.6740959286689758, -0.08611779659986496, 1.4785484075546265, -0.9363295435905457, -0.8083496689796448, 0.7914553880691528, -0.35700559616088867, -1.3530244827270508, 0.4276334047317505, -2.2383861541748047, -0.09387209266424179, -0.6675966382026672, 1.5169706344604492, 1.3938143253326416, -0.43367859721183777, -1.42727530002594, 0.9012935757637024, -0.396246999502182, 0.03520405665040016, 0.09153100103139877, -1.5493093729019165, 0.9888647198677063, 1.4199413061141968, -1.216448187828064, -0.26446977257728577, -0.5823939442634583, 0.5630621910095215, -0.8638907074928284, 0.06769866496324539, -0.04575774073600769, 0.3120265007019043, 0.7099324464797974, -0.5957493782043457, -0.657578706741333, -0.6091023087501526, 0.3082607388496399, 1.621047019958496, -0.9499826431274414, 0.6349459290504456, 1.6919662952423096, -0.8126590251922607, -1.8657277822494507, 0.5640432834625244, 1.4483298063278198, -0.8214613795280457, 0.3302469849586487, 0.8329103589057922, -0.4717428386211395, -1.1443523168563843, 3.1248040199279785, -2.4996321201324463, 0.8244696259498596, -0.061828043311834335, -0.810824453830719, 0.15895923972129822, -0.8258675336837769, 0.18089638650417328, 0.7790003418922424, 1.1375453472137451, -0.6826906204223633, 0.8218093514442444, -0.6628629565238953, 0.05205494910478592, -1.2623871564865112, 0.006941807921975851, -1.7758371829986572, -0.26481837034225464, -0.9001861810684204, -0.0013829409144818783, -1.0583113431930542, 0.6648011207580566, 1.50875723361969, 1.179490327835083, -0.016455521807074547, -0.9553572535514832, 0.17425820231437683, -0.25634491443634033, 0.618462860584259, -0.8903295993804932, 1.3236920833587646, 2.47402024269104, -0.03270160034298897, 2.297572135925293, -0.0015353799099102616, -0.7572977542877197, 0.3291895091533661, 1.414857029914856, 1.7121939659118652, -2.830489158630371, -0.947766900062561, -1.1035466194152832, -1.717218041419983, -0.8834381103515625, -0.2960093319416046, -1.2337337732315063, -0.8418464064598083, -0.776721715927124, 1.2773300409317017], [0.0033464825246483088, -0.5631937384605408, -0.2175750583410263, 1.4722962379455566, 0.16736839711666107, 1.4782952070236206, 1.0020767450332642, -2.8815815448760986, -1.0951062440872192, -1.229405164718628, 2.044153928756714, -0.6749605536460876, 0.5720528364181519, -1.7394981384277344, 0.01168751996010542, 1.7604283094406128, -2.1708576679229736, 1.883684754371643, -0.6902134418487549, 0.13702939450740814, 0.7654929757118225, -1.2173130512237549, 1.3228609561920166, -1.6148629188537598, -0.8045116066932678, 0.5411661267280579, -0.8163039684295654, 2.164626121520996, -0.48860904574394226, -1.3738373517990112, -0.9007126092910767, 0.15660257637500763, -0.9366748332977295, -0.6394665837287903, 1.1631487607955933, -0.3457275927066803, -1.6426955461502075, -0.09968551993370056, 0.07742239534854889, -1.3648031949996948, -0.62168949842453, -1.9199588298797607, -0.47281041741371155, -1.9330333471298218, -0.19623440504074097, -2.1432385444641113, -2.073228359222412, -1.047939419746399, -0.18077032268047333, -0.5015615820884705, -0.009699380956590176, -0.5343611240386963, -0.35300058126449585, -0.40682142972946167, -0.21501575410366058, 0.1304391473531723, -1.1730867624282837, 0.6353477239608765, -0.43948841094970703, 0.7187333106994629, -1.577907919883728, -1.7169498205184937, -1.724543571472168, 1.6059013605117798, 2.044166326522827, -2.593914270401001, -0.6297363042831421, 0.07824311405420303, 1.00880765914917, 1.375372052192688, 0.10661479830741882, 0.4593317210674286, -0.851388692855835, 1.3402235507965088, -0.0697864443063736, 1.3344563245773315, -0.3405064046382904, 0.3172408640384674, -0.4818286597728729, 0.511067807674408, 0.5600489377975464, -0.6612839698791504, -1.2621793746948242, 0.14381735026836395, 0.7407193183898926, -0.3390181064605713, -1.777134656906128, -0.7888208031654358, 0.39040225744247437, 0.28349512815475464, -0.7804656028747559, 0.42993399500846863, 0.799399733543396, -0.37540897727012634, -0.6114467978477478, -0.3472190797328949, -0.2560271918773651, 1.8644382953643799, 0.3688834309577942, -0.4888153672218323, -0.8002795577049255, -0.574924111366272, 1.2222130298614502, 0.06065414473414421, 1.0827305316925049, 0.44807618856430054, -0.39294812083244324, -0.4750685393810272, 1.6778513193130493, -0.4034304618835449, 0.32049670815467834, -0.5986403822898865, 1.7999567985534668, 2.159802198410034, -0.4160090982913971, 1.4540841579437256, -1.536874532699585, 1.0314855575561523, 1.8806136846542358, -0.5231022238731384, 1.0117754936218262, -0.24998852610588074, 0.5312119126319885, 1.2363659143447876, 0.20161890983581543, 0.15782727301120758, -1.1895033121109009, 1.4109987020492554, 1.2507798671722412], [-0.2860787510871887, 0.20486310124397278, 0.16666261851787567, 0.9364650845527649, -0.8658771514892578, -0.6006719470024109, -1.2135707139968872, -0.5264151096343994, 0.3330397307872772, -1.2514948844909668, 1.9171559810638428, -0.7387568354606628, 1.0338507890701294, -0.18823526799678802, 1.4085789918899536, 1.0912482738494873, -0.1809089034795761, -0.1980370581150055, 0.8697702884674072, -0.09114297479391098, -0.48750409483909607, 0.3141413629055023, -0.09147267043590546, 1.2571461200714111, 1.0834424495697021, 1.0375468730926514, 0.3340778052806854, 0.08920951932668686, 0.5032764077186584, 0.09816376864910126, 0.5667292475700378, -1.0422570705413818, -1.4837538003921509, 0.3761807084083557, -1.1112804412841797, -0.8083281517028809, -0.4834916889667511, 1.5384163856506348, -0.5212512016296387, 0.17725780606269836, 0.16889260709285736, 0.06839912384748459, -0.9274346232414246, -0.4893016517162323, -1.468246579170227, -1.1076414585113525, -1.0949605703353882, 0.1976643204689026, -1.7501903772354126, 0.8973588943481445, 0.7302061319351196, -1.2835742235183716, 0.15160322189331055, -1.4457162618637085, 0.6430498957633972, 0.5898947715759277, 0.7080473303794861, -0.36942264437675476, 0.47302377223968506, -0.030783187597990036, 1.7909090518951416, 0.06056326627731323, -0.19046436250209808, -2.2149369716644287, 0.7606984972953796, -0.6909996271133423, 0.7169678807258606, -1.0033479928970337, -0.5295161008834839, 1.8239593505859375, -0.19345779716968536, 1.1852936744689941, -1.5767951011657715, -0.6890798211097717, 0.01343544665724039, -0.5837856531143188, 2.401554584503174, 0.5726255178451538, 0.14846567809581757, 0.6524786949157715, -0.479966402053833, -0.04368133470416069, 0.8480559587478638, -0.8910465240478516, 0.38734424114227295, 1.0804575681686401, 0.4476931393146515, -1.40159010887146, 0.9157044887542725, 0.522255003452301, 0.7481376528739929, -1.5352702140808105, -0.3621997833251953, -0.07230470329523087, 1.0128307342529297, 0.6707202792167664, -0.9866021871566772, -0.6188639998435974, 0.5165157914161682, 0.5063216090202332, -1.7186130285263062, 0.32596078515052795, 0.7992616295814514, 0.2712939977645874, -0.4863097071647644, -0.0464579202234745, 1.2451690435409546, 0.6235973834991455, 0.029884880408644676, -1.0481679439544678, -1.1289167404174805, -1.0569101572036743, -0.25693392753601074, 0.2540682256221771, -0.1858631819486618, -0.3970526456832886, 0.9822010397911072, 0.3097226917743683, -0.8930879831314087, -1.886121392250061, 1.0170971155166626, 0.11618158966302872, 1.413066029548645, 1.207295536994934, -0.03379611298441887, 1.4625484943389893, -2.5919055938720703, -0.5530864596366882, -1.3351466655731201], [-0.7000432014465332, 0.14679959416389465, 0.3769906163215637, 1.3246896266937256, -0.10880985110998154, 1.1623302698135376, -1.4513047933578491, 0.259015828371048, 0.7813035845756531, -0.20892317593097687, 0.12796567380428314, -0.2800278961658478, -1.3274059295654297, 1.0802863836288452, 0.04509587585926056, -1.263456106185913, -0.06253167986869812, 0.8542842864990234, -0.054037220776081085, -0.6979045867919922, -0.8512164950370789, -0.9472637176513672, 1.5065854787826538, -1.6376293897628784, 0.3741492033004761, 0.08215173333883286, 0.2515677511692047, -0.624580442905426, 1.0562822818756104, 1.428066611289978, -0.5866050720214844, 0.8984143733978271, 0.2567158639431, 0.7133962512016296, -0.43450021743774414, -0.094167560338974, -0.16312162578105927, -0.006785015109926462, 0.8340811133384705, -0.03800475597381592, 0.7107620239257812, 1.1013649702072144, 0.48508545756340027, -0.7189047932624817, -0.792452335357666, 0.14636464416980743, 0.37009701132774353, 0.4626776874065399, -0.18474341928958893, -0.7877804636955261, -0.03126638010144234, -0.4869987368583679, -0.7299582362174988, -1.5564414262771606, 0.03792918846011162, 2.067262649536133, 0.7305622100830078, 0.45313969254493713, -0.06016319617629051, -1.2016836404800415, 0.90696781873703, -2.2345359325408936, 0.968199610710144, -2.7967684268951416, 0.5793569684028625, -0.49507495760917664, 2.113591194152832, -1.012589693069458, -1.2089877128601074, -0.8624215722084045, 0.8672217130661011, 0.5864512920379639, -0.7818967700004578, 1.5573935508728027, -0.3224557936191559, -1.3097721338272095, -0.32565122842788696, 0.9777957201004028, -0.7640106081962585, 1.614643931388855, 0.6487805843353271, 0.08698813617229462, -0.4655916094779968, 0.991912305355072, 0.4204542636871338, 2.3389809131622314, 0.49782872200012207, -1.7382972240447998, -2.420588970184326, 0.1713424175977707, -0.0567794069647789, -0.4866231679916382, -2.6745214462280273, 0.07190391421318054, -1.45134699344635, -2.26113224029541, -1.2768069505691528, 0.5026724934577942, -0.3023523986339569, -0.310884952545166, -0.6718032360076904, -0.9178547263145447, 1.5070582628250122, 0.5940089225769043, -0.32294440269470215, -1.353957176208496, -1.0622648000717163, -0.7546291947364807, 0.7413449883460999, 1.4766284227371216, -0.8408656716346741, 0.5877517461776733, 0.18962445855140686, 0.363320529460907, 0.33206215500831604, -1.3612861633300781, 0.16022999584674835, -0.8830112218856812, 0.7634416818618774, 0.06637968868017197, -0.42779406905174255, 1.012847661972046, -1.0248459577560425, -0.9535858631134033, 0.5163282752037048, -0.5071669220924377, -2.2000083923339844, -0.817044734954834, 0.31932511925697327], [0.5385178923606873, 0.6201362609863281, -0.4667804539203644, -1.2817081212997437, -1.9188528060913086, -0.370241641998291, 0.640343189239502, 0.9097529053688049, -1.6672316789627075, 1.4083871841430664, 1.106276035308838, 0.10766847431659698, -0.22206798195838928, 0.8928525447845459, -0.5893561244010925, 1.6590559482574463, 0.5957750678062439, 1.1807972192764282, -0.785972535610199, 0.3867642283439636, 2.163701057434082, -1.3052855730056763, -0.053064003586769104, 1.4954649209976196, 0.421565443277359, 0.11905311793088913, -0.43880414962768555, 0.8554573655128479, 1.3282283544540405, 1.0283839702606201, 0.35929256677627563, 1.0138850212097168, -0.2869791090488434, 0.523819625377655, 0.17168885469436646, 3.1531269550323486, 1.522568941116333, -0.3152380585670471, 1.076653003692627, -0.4291533827781677, 1.242755651473999, 0.5971084237098694, 0.04052162915468216, 0.09644020348787308, 0.6688557863235474, 1.166678547859192, -1.5521608591079712, -1.531773567199707, -1.144630789756775, -0.6759580969810486, -0.8551107048988342, 0.770967960357666, -0.15384018421173096, 1.240389347076416, 0.5295807719230652, -1.038319706916809, -0.6648232936859131, -1.8242220878601074, -0.5230942368507385, -0.21752631664276123, 1.7220513820648193, 0.4927687644958496, 0.3421996831893921, -0.325973242521286, -0.03772062435746193, 0.07951978594064713, 0.7250493764877319, -0.19855239987373352, -0.3537452816963196, 0.9338488578796387, -0.2165507972240448, 1.282579779624939, 2.4328737258911133, 1.4140037298202515, 0.42932772636413574, 0.7102533578872681, -2.5347797870635986, -0.48786598443984985, -0.0677260011434555, 0.7238942384719849, -0.8480183482170105, -0.3536112308502197, 1.2186739444732666, 0.6498578786849976, 0.2603328227996826, -0.11545693874359131, 0.6117593050003052, 2.1204311847686768, -0.07325136661529541, 0.41534608602523804, 1.570361852645874, -0.2832898497581482, 0.030279437080025673, -0.410552054643631, -1.6817914247512817, 0.5329524874687195, 2.1061036586761475, -0.6932904720306396, 0.6081444025039673, 0.11654287576675415, -0.07922781258821487, 0.12374874949455261, 0.6209185719490051, -1.4175084829330444, 0.9177882075309753, 0.9399214386940002, 0.3569169342517853, 0.7087271213531494, -0.29304713010787964, 0.5343884229660034, -0.5114794969558716, 0.06312673538923264, -0.29792022705078125, -0.24886523187160492, -0.16439677774906158, -0.12809090316295624, 1.0327903032302856, -1.5778433084487915, -1.4453800916671753, 0.24991363286972046, 0.9886461496353149, 0.16963812708854675, -0.24225738644599915, 2.0685973167419434, 0.11364778131246567, -0.846315324306488, -0.1882336288690567, 0.8215429782867432, -0.03415318578481674], [-0.8471030592918396, 0.8964883089065552, 0.2746749222278595, 0.9729711413383484, -0.036938395351171494, -0.8312876224517822, 0.5062819123268127, 0.8516556024551392, 2.238999605178833, 1.807280421257019, 0.6147258281707764, -0.08985462784767151, -1.9422097206115723, 0.178201362490654, -0.2551771402359009, -0.06299687922000885, -0.6378463506698608, -0.8752030730247498, -1.8889743089675903, -0.3783353269100189, -0.6681358218193054, 0.8236522674560547, -0.3888546824455261, -0.4348013997077942, -0.31849610805511475, 0.16782858967781067, 0.1512298882007599, 1.0567549467086792, -0.5867646336555481, 2.30828857421875, -0.6353656053543091, -0.5502115488052368, 2.011373519897461, 0.5971640348434448, -0.9355816841125488, -0.44877535104751587, -0.036038633435964584, -0.09616349637508392, 0.7882033586502075, -0.6121507287025452, -0.1313474327325821, -0.36229386925697327, 0.46157512068748474, 1.036179780960083, 0.3384217321872711, -0.5686578750610352, -0.3210754096508026, 1.1056530475616455, 0.5562845468521118, -1.3940086364746094, -1.2145217657089233, -1.6273294687271118, 0.13047544658184052, -0.08328519761562347, -0.2363051176071167, -0.2441115528345108, 1.5200458765029907, -0.4322049021720886, 0.8104343414306641, 1.0534367561340332, 0.5278975367546082, 0.30912694334983826, -1.7313132286071777, -0.2336139976978302, 0.14430870115756989, 0.480355829000473, 0.05146603658795357, 1.405454397201538, -0.8691534996032715, -0.38707777857780457, -0.21046534180641174, 1.4434336423873901, -0.25313854217529297, 1.1520237922668457, 1.5404118299484253, 1.261737585067749, 0.7025651931762695, -0.2371118813753128, 0.043568119406700134, 0.5354511737823486, -0.8233653903007507, 1.1467759609222412, 0.21454370021820068, 1.504797339439392, -0.6731904149055481, -1.238521933555603, -1.2668297290802002, 0.04113331809639931, 1.0539097785949707, -2.9212124347686768, -0.5638141632080078, 1.0682861804962158, 1.3785552978515625, -0.6662797331809998, 0.2840593755245209, 0.5009784698486328, -1.1746708154678345, 0.5648992657661438, -2.394672155380249, 0.8765227794647217, 1.7971606254577637, 0.7950513362884521, 0.7322837114334106, 0.7040014863014221, -0.6674930453300476, -0.17097100615501404, 0.6677162051200867, 0.32995593547821045, 0.23268108069896698, -0.6725298166275024, 0.5719714164733887, -1.4194012880325317, 1.2920653820037842, -0.8843550682067871, 1.546871542930603, -0.4265304505825043, 0.8232659101486206, 0.07287846505641937, 1.2719645500183105, -0.9475404024124146, 1.4538275003433228, 0.3953913450241089, 0.02599378675222397, 0.3892381191253662, 0.2570573091506958, 1.3345569372177124, 1.2840142250061035, 0.4897717237472534, 0.9200771450996399], [-0.13046298921108246, 0.657052755355835, 1.9827088117599487, 0.4503490924835205, 0.45728304982185364, -0.30596724152565, -1.7298777103424072, -0.48860666155815125, 0.5851166248321533, -0.4117971658706665, 0.21795043349266052, 0.6685066819190979, -1.7788865566253662, -0.29066818952560425, 0.3956080377101898, 0.9190808534622192, 1.1722711324691772, 0.4974829852581024, 0.6159888505935669, 0.30405378341674805, 0.5466646552085876, 0.3181699514389038, 3.7327873706817627, 0.032730624079704285, -0.4084804654121399, 0.674221396446228, -0.7610941529273987, 0.6247406005859375, -0.6444233655929565, -1.805143117904663, -0.6021354794502258, 0.8619717955589294, 0.08455416560173035, 0.5759897232055664, -0.5018446445465088, 0.24854397773742676, -1.1040830612182617, -1.4310745000839233, -0.1467391550540924, -0.559147298336029, 1.2739872932434082, 0.324407696723938, 0.23278352618217468, -0.8235782980918884, 0.24324478209018707, 1.9699163436889648, 0.7601146697998047, 0.9334822297096252, -0.2407115399837494, 0.05649556592106819, 0.29031339287757874, -0.9512292146682739, -0.06220883131027222, -0.41437575221061707, -0.5423726439476013, 0.32961395382881165, -0.4757678210735321, -0.7669250965118408, 0.40578895807266235, 0.5139440298080444, 0.43586570024490356, 0.13386324048042297, -0.4808759391307831, -0.9443513751029968, 0.2464505285024643, -0.9499505162239075, 0.33958059549331665, -1.0637754201889038, -1.1867518424987793, 0.936621904373169, 1.1350651979446411, -1.45011305809021, -0.12849996984004974, -0.6073380708694458, -0.47575363516807556, 0.2508513927459717, 0.3896901607513428, -0.43220824003219604, 1.0175484418869019, 0.4209332764148712, -1.1830533742904663, 0.2273274064064026, 0.5523167252540588, -0.34172314405441284, -0.9416301250457764, -0.6808820366859436, -0.46075674891471863, -0.056017521768808365, -0.12659381330013275, -1.3963197469711304, 0.41904792189598083, -0.4969930946826935, -0.5644503235816956, -0.908336341381073, -0.0908130556344986, 0.5998706221580505, 1.336145043373108, 1.4288524389266968, 0.5603336691856384, -0.30857622623443604, 0.6126284599304199, -1.4928491115570068, -0.09501821547746658, 0.7435241341590881, 0.10830004513263702, 1.9055144786834717, 0.9219658374786377, 2.301786422729492, -0.1384396106004715, 0.36835116147994995, -0.3811039626598358, -2.3251290321350098, -1.6389394998550415, -0.2236948013305664, -0.9304395914077759, -1.1302077770233154, -1.217529058456421, -0.4133087396621704, 0.7669082880020142, 0.3474563956260681, 0.22451432049274445, -0.2111883908510208, 0.3788098990917206, 1.2172799110412598, -1.0872254371643066, 0.9956418871879578, 0.49527138471603394, -0.2470339983701706, -1.0164076089859009], [1.2274305820465088, 0.6727497577667236, 1.4825025796890259, -0.1356731802225113, -0.955254852771759, 0.39225906133651733, -0.09204939752817154, 0.13924282789230347, 0.5269713401794434, -0.5359015464782715, 0.43174171447753906, -0.624596893787384, 1.1885333061218262, 0.5466275811195374, 0.6473489999771118, -0.7541755437850952, -1.013979434967041, 1.9370365142822266, -0.773563027381897, 0.27129489183425903, -1.5231174230575562, -0.9090163707733154, 1.3648682832717896, -0.7084456086158752, -0.18092867732048035, 1.1562656164169312, 0.3595602512359619, 0.6432419419288635, 0.8667666912078857, 2.0781397819519043, -1.1926076412200928, -0.496480792760849, 0.22570845484733582, -1.077436089515686, -0.3268563747406006, 0.532008171081543, -0.14487715065479279, 1.4834216833114624, -0.9476934671401978, -1.048613429069519, 2.0962491035461426, 0.26037687063217163, -0.6004106402397156, 1.5640895366668701, -0.3233089745044708, 0.8139599561691284, 1.2300567626953125, -0.48458340764045715, -0.20867297053337097, -0.20330803096294403, 0.22276468575000763, -1.1869335174560547, -0.4645438492298126, -0.3832226097583771, -0.414447158575058, 1.132792592048645, 0.2175748646259308, 0.39094579219818115, 0.021397195756435394, 0.6544730067253113, 1.4166409969329834, -0.7122374176979065, -0.22365586459636688, -1.6661429405212402, -0.0037274814676493406, 0.418414443731308, 0.4654625952243805, 0.4779508411884308, 0.6636078357696533, 0.46268630027770996, 0.3581770360469818, -0.628738284111023, -0.4961220324039459, -0.47023311257362366, -0.02850317396223545, 0.8309411406517029, -2.2853856086730957, -1.027082920074463, -0.10722734034061432, 1.4806177616119385, 0.8732447624206543, 0.1269073784351349, 0.06502144783735275, 0.44001680612564087, -0.7104271650314331, -1.0000545978546143, -0.022027073428034782, 1.2360191345214844, -0.6012099981307983, 0.5201090574264526, 0.5080415606498718, 0.08601554483175278, -1.440962791442871, -1.7784487009048462, -0.6007879972457886, 1.008551836013794, 0.20455007255077362, -0.9627009034156799, 0.7099712491035461, -1.6056028604507446, 0.3032919764518738, -3.0514049530029297, -0.3129936754703522, -1.823659896850586, 0.34359559416770935, -1.2359650135040283, -0.7436859607696533, -0.18863186240196228, 0.6543818712234497, -0.13952648639678955, -1.240884780883789, -1.8644063472747803, -1.252251148223877, -2.5274081230163574, 0.8515108823776245, -0.17548160254955292, 0.45313891768455505, -0.3848915100097656, -0.317674458026886, 1.1346455812454224, -1.471432089805603, -0.02861032634973526, -1.957476019859314, 1.1324915885925293, 0.45244506001472473, 0.14528389275074005, -0.8413707613945007, 0.5563138723373413, -0.39213499426841736], [0.5775482058525085, 0.2212110459804535, -0.31566745042800903, 0.5881783962249756, -1.0553936958312988, 1.114741325378418, 0.2598433494567871, 0.23758548498153687, -0.6788444519042969, -1.5177916288375854, -0.505780041217804, 0.2660287022590637, -0.6609113216400146, 1.0935903787612915, -0.7736604809761047, -1.602616548538208, 0.46807509660720825, 0.08422620594501495, -0.1715819239616394, 1.2019256353378296, -0.7769722938537598, -1.7390247583389282, 0.2658097445964813, 0.05785456672310829, -0.8864296674728394, 1.4323992729187012, 0.9031625986099243, 1.238311529159546, -0.2621096670627594, 0.5894562602043152, -1.5607683658599854, -0.9936133027076721, 0.12088312208652496, -1.3076996803283691, 0.7400602698326111, 0.7335402965545654, 0.14736346900463104, 0.5608769655227661, 0.06919826567173004, -0.2545132637023926, -0.2363961786031723, -2.230926513671875, -0.46827659010887146, -0.9277986288070679, 0.7108281254768372, -0.3501523733139038, -0.39872169494628906, 1.0872600078582764, 0.28533661365509033, 0.4137832224369049, 0.8897410035133362, -0.3438112437725067, 1.6372640132904053, -1.044376254081726, 1.230958342552185, 1.56844162940979, 1.1257026195526123, 0.3245040476322174, -0.9970626831054688, -1.3793221712112427, 0.8042980432510376, 0.36735716462135315, 0.3698006272315979, -0.19969716668128967, -0.5887246131896973, -0.8967528343200684, 0.7390685081481934, 1.0842393636703491, 0.700160801410675, 0.7750034928321838, 2.0985968112945557, -0.45323246717453003, 0.5295158624649048, -1.4143575429916382, 0.8763048648834229, -0.7961184978485107, -1.5928504467010498, 0.12510307133197784, 1.2995647192001343, -1.3389276266098022, 0.36187806725502014, 0.02539725974202156, -1.4944953918457031, -1.367632508277893, 0.07833041250705719, 0.05395900830626488, -0.18898694217205048, 0.8994472622871399, -0.5136035084724426, 2.2700512409210205, 0.7541558742523193, -0.6785581707954407, -0.41286900639533997, 0.8243440389633179, -0.5182046294212341, -0.19446687400341034, -0.8899897336959839, 0.10986024141311646, 0.09874406456947327, 1.9963948726654053, -0.32104647159576416, -0.7884349822998047, 0.5120857357978821, 0.4980766475200653, 0.05028482526540756, 0.7206498980522156, -0.8664667010307312, -2.0490689277648926, -1.0572288036346436, 0.8345150351524353, 1.2873362302780151, -0.3297518491744995, -0.7022144198417664, -0.5333652496337891, -1.7333704233169556, -0.2144671380519867, -0.025106774643063545, 1.045941710472107, 0.7672477960586548, -1.5681602954864502, 1.0278362035751343, 0.35686129331588745, -1.18105947971344, 1.4218355417251587, 0.34528613090515137, 0.0732671469449997, 1.6177892684936523, -0.3608686625957489, -1.7096374034881592], [-0.09807304292917252, 2.67521595954895, 0.7000561356544495, -0.2519209682941437, 0.5497162938117981, -0.3820991814136505, 0.14811769127845764, -0.34510260820388794, 1.7778797149658203, -0.5464571118354797, 0.10726548731327057, -1.0121134519577026, 0.8960120677947998, 0.27474430203437805, 0.70415198802948, -1.4157880544662476, 0.42463698983192444, 0.18263044953346252, -0.5326933860778809, 0.449984073638916, -1.7754974365234375, 0.25864291191101074, 0.6558039784431458, -0.4687378406524658, -0.3829602599143982, 0.1788891851902008, 1.4185320138931274, -0.14603330194950104, -0.540733814239502, -1.7836552858352661, -0.6382701992988586, -0.46519580483436584, 0.8974133133888245, -1.2675307989120483, -2.526677370071411, -0.7956916689872742, 1.047182559967041, 0.8349223732948303, 0.8835248351097107, -2.590193510055542, 2.844255208969116, -0.18239302933216095, 2.240368127822876, -0.0711049884557724, 0.5852603316307068, 1.4401934146881104, -1.1473561525344849, -1.9528560638427734, -0.21721935272216797, 2.1878106594085693, -0.5957885980606079, 0.5649063587188721, 0.10796806961297989, -0.1403968781232834, 0.3607219159603119, -0.7059091329574585, -0.33285585045814514, -1.2581287622451782, -0.8866607546806335, -0.32730260491371155, -1.1630809307098389, 0.8198435306549072, 0.2915530800819397, 2.19094181060791, 1.8812017440795898, 0.8699930310249329, -1.0503523349761963, -1.600557804107666, 0.8510949611663818, 0.811532199382782, 0.7254241108894348, -0.34724298119544983, 1.8028751611709595, -1.0466833114624023, -1.0855848789215088, -1.5872122049331665, -0.052029579877853394, 1.1432194709777832, -1.5099396705627441, 0.5785167217254639, 1.3588764667510986, 0.36156556010246277, 0.849663496017456, 1.7811312675476074, 1.009253978729248, 0.8898274302482605, -0.20723773539066315, -0.7526453137397766, 1.6246274709701538, 0.5581047534942627, 0.6093834042549133, -0.9454091191291809, 1.2123796939849854, 0.30931130051612854, -0.9399529695510864, 0.42639368772506714, -0.7437564134597778, 0.18765559792518616, 0.6307178735733032, -1.36495041847229, 1.2031984329223633, 0.39185118675231934, -1.9818120002746582, -0.8148069977760315, -0.8475458025932312, 0.09456765651702881, -0.4441872835159302, 1.3859562873840332, -1.1029292345046997, -0.4895671308040619, -0.5259431004524231, -0.41546955704689026, -0.8811039328575134, 1.741229772567749, 0.18977251648902893, 0.3615849018096924, -0.3823099434375763, 0.2528970241546631, 1.1899981498718262, -0.5500190854072571, 2.6177635192871094, 0.2629666030406952, -1.4386188983917236, 0.22770148515701294, -0.7885740399360657, -0.15465635061264038, -0.15554693341255188, 0.04990164563059807, -0.3630620539188385], [0.38561853766441345, -1.571716547012329, 2.6527419090270996, -0.15430405735969543, -0.8732976317405701, 2.030242681503296, -1.3263859748840332, -0.8990983963012695, -1.7596712112426758, -1.6588884592056274, -0.6362119317054749, 0.17768937349319458, 2.370732307434082, -1.0050781965255737, -1.3318158388137817, -0.33660921454429626, -0.020365914329886436, -0.07317923754453659, -0.7839886546134949, 1.2001653909683228, -0.788987934589386, -0.17799688875675201, 0.3516457676887512, 0.2807021737098694, -0.04820294305682182, -1.5206018686294556, 1.0586013793945312, -0.43456152081489563, -1.7141026258468628, 0.5283244252204895, 0.8578503727912903, -1.0508147478103638, -0.2692734897136688, 0.935226321220398, -1.7142711877822876, -1.4945402145385742, 0.7196550369262695, 1.1967291831970215, -2.165278673171997, -0.5115025639533997, -0.32072800397872925, -0.2635733187198639, -0.3130619525909424, -0.3163185119628906, -0.08612927794456482, 0.2032238394021988, 0.673703670501709, 0.4997997283935547, -0.35107189416885376, -0.5288653373718262, -1.848486065864563, -0.2920180559158325, 0.6162479519844055, -2.1064531803131104, 0.47208186984062195, -0.6463915705680847, -0.9355316162109375, -0.8268775939941406, -0.024472175166010857, -0.19683651626110077, 0.5663823485374451, 0.7123971581459045, 0.6939722895622253, 1.0718525648117065, -0.5958309173583984, 0.25881636142730713, -0.12590806186199188, -1.348963975906372, -0.5382669568061829, 0.9247300624847412, -0.9089953899383545, -0.7688608169555664, -0.22330784797668457, 0.7516871690750122, 1.3418121337890625, -1.1120905876159668, -1.381115436553955, -0.42423132061958313, 0.5042186975479126, -0.5602500438690186, 0.34386199712753296, 0.6467024683952332, 0.580229640007019, 0.35181793570518494, -1.0918872356414795, 1.3328531980514526, -0.4798148572444916, -1.8299742937088013, -0.4852864444255829, 2.1442041397094727, -0.06562203168869019, -0.37406548857688904, 0.21577735245227814, 0.2100154012441635, -1.3969123363494873, 0.3991643786430359, -1.0097997188568115, 1.2154626846313477, 2.067474603652954, -0.2973945438861847, -1.2792173624038696, -0.783453643321991, 0.4784950017929077, -0.3677302300930023, 0.5287650227546692, -0.3512629568576813, 1.8473072052001953, -0.6398910284042358, -0.06707461178302765, -1.5971263647079468, -0.03995799273252487, 0.3769865334033966, -1.66616690158844, -0.10530086606740952, -0.047844916582107544, 1.5168861150741577, -0.16467106342315674, -0.9596930146217346, 0.3785553276538849, 0.4600856900215149, -2.265589714050293, 0.15985555946826935, 0.995136022567749, 1.698598861694336, -0.031293731182813644, 0.8484048247337341, -0.8611864447593689, -0.8501753211021423, 0.8269590735435486], [-1.565779209136963, -1.1783136129379272, 0.22154077887535095, -0.5482546091079712, -0.5444043874740601, -0.29394885897636414, 0.4305579364299774, -0.7737507224082947, -1.1466392278671265, -1.1674729585647583, 2.201669216156006, -0.3979857265949249, 2.594741106033325, -0.0862123966217041, -0.15619848668575287, 0.5943092703819275, -1.3267825841903687, -0.6037301421165466, 0.058182671666145325, -0.7451395392417908, 0.5534657835960388, 0.20625849068164825, 0.4811571538448334, -0.34369081258773804, 1.3338119983673096, -0.7665197253227234, 0.33977699279785156, 0.27600300312042236, -1.0345007181167603, -0.15495771169662476, -1.3399986028671265, -0.4791642725467682, -0.5091163516044617, -0.779617965221405, 0.1839597076177597, 1.5312854051589966, -0.974981963634491, 0.02926221489906311, 0.594927191734314, 0.21550306677818298, -0.5170934796333313, 0.6206931471824646, -0.4019950330257416, 1.141055941581726, -0.24117900431156158, 0.28222405910491943, 1.3279669284820557, -0.0020590464118868113, -0.7395277619361877, 0.6264445781707764, -0.6338452100753784, 0.4097333252429962, -1.6196051836013794, 0.29386019706726074, 0.2716005742549896, -1.6817742586135864, -2.0184366703033447, -0.9238874316215515, -1.2829346656799316, -1.220142126083374, -0.15910181403160095, -0.7298499345779419, -0.5417996048927307, 0.7468710541725159, 0.20772795379161835, -0.126667320728302, 0.016159633174538612, -0.8899363279342651, 0.0022121784277260303, 0.15717197954654694, -1.062833309173584, -0.40325871109962463, -2.060415267944336, 0.23513011634349823, -0.6010648012161255, 0.26493507623672485, 0.9147453904151917, 0.3717772960662842, 1.0446420907974243, -0.9578061699867249, 2.160618305206299, -0.6769711971282959, -0.8826379776000977, 0.42286527156829834, 0.528907060623169, -0.7211982011795044, 0.8805118203163147, 0.38982874155044556, 0.3476473093032837, -1.532841444015503, -1.2464927434921265, 1.2416582107543945, 0.8795276284217834, 0.6245694756507874, 0.059795498847961426, -0.4872902035713196, -1.7760518789291382, -1.4177762269973755, -1.619275689125061, 0.4587670564651489, 0.480855792760849, -0.9099544286727905, 0.14065033197402954, -0.46759670972824097, 0.3051745891571045, -0.1807025820016861, 1.0096101760864258, 0.8543354272842407, -0.4821227192878723, 2.2162320613861084, 0.2116609364748001, -0.8361308574676514, 0.08602617681026459, -0.1077323779463768, 0.08039754629135132, 1.0175641775131226, -0.6579705476760864, -0.729947030544281, -1.8667789697647095, -0.6221035122871399, -0.7572448253631592, 1.4304753541946411, 0.611527681350708, 1.6090606451034546, -0.41954270005226135, 1.9524757862091064, -1.3826000690460205, -0.18224820494651794, 0.6885858774185181], [-1.5757968425750732, -0.9483025670051575, 1.7855994701385498, 0.4188809394836426, -0.7803671360015869, 1.9576518535614014, -0.9649527072906494, 0.2651376724243164, -0.1841994673013687, -0.4739600718021393, -0.5255938768386841, 1.793285608291626, 2.781097173690796, -0.5640755295753479, 0.999125599861145, 2.0625603199005127, 0.8940207958221436, -0.2438693642616272, 0.04656275361776352, 0.07023310661315918, 2.2918941974639893, 0.781520426273346, -0.6642647385597229, -0.3015241026878357, -1.338379144668579, 1.9337234497070312, 0.7740962505340576, 1.0336248874664307, -1.0593246221542358, 1.5093770027160645, 0.4887794256210327, 0.5598680973052979, 1.6186333894729614, 0.1865377575159073, 1.0859543085098267, -0.5569280385971069, -0.12293266505002975, -1.8242404460906982, -0.5887791514396667, 0.4772317707538605, 0.6742611527442932, 1.3298397064208984, 0.5159062743186951, -0.6609204411506653, -0.4923844337463379, 0.3783765435218811, 0.17542870342731476, 1.1830300092697144, -1.6477231979370117, 1.6420944929122925, 2.147416830062866, 1.517455816268921, -0.6153064966201782, -0.17261166870594025, -1.072784662246704, -0.8190888166427612, -0.035645827651023865, 1.218429446220398, 1.6226344108581543, -2.1592793464660645, -1.926510214805603, -0.44081804156303406, -0.7651696801185608, 0.7335056662559509, 0.25411930680274963, 0.7712993621826172, -0.26120635867118835, 0.5426914095878601, -2.3702120780944824, -0.1313171237707138, -0.46035531163215637, 0.7816305160522461, 1.2639760971069336, 0.18455806374549866, -0.01749609038233757, 0.11448320746421814, -0.7250335812568665, -1.1578751802444458, 0.3093869388103485, 1.044891357421875, -1.4286149740219116, 0.6583721041679382, -1.1404691934585571, 0.35864996910095215, 1.8870149850845337, -0.5167183876037598, -1.0257956981658936, -0.8084532022476196, 0.23452050983905792, 0.393047034740448, 1.0248953104019165, -0.0488290898501873, 0.5430966019630432, 1.0937236547470093, 1.2817083597183228, 0.001150227733887732, -1.1446517705917358, 1.7959229946136475, -0.47178733348846436, -1.0540324449539185, 0.23336860537528992, -1.8271453380584717, 0.15524418652057648, -1.3389757871627808, 1.7248027324676514, 0.08550356328487396, -0.8034349679946899, -0.05483509227633476, -1.6614409685134888, -0.23718315362930298, -1.2487939596176147, -0.3076496422290802, 0.11503741890192032, 0.2506967782974243, -0.7665761709213257, 0.1941547393798828, -0.8576266765594482, -0.1087917760014534, -1.3945719003677368, 0.28062161803245544, 0.3262697160243988, 0.5128992795944214, 0.15294808149337769, 0.45585504174232483, 2.4496772289276123, 0.0692056268453598, 2.315910577774048, 1.3565407991409302, 1.03779137134552], [0.4545738995075226, 0.3264663815498352, -0.4402538537979126, 0.3812633752822876, -1.300882339477539, -1.4224921464920044, 1.1530163288116455, 1.156712293624878, 1.1369746923446655, 1.3742750883102417, -1.7516626119613647, -0.5320603847503662, 0.7647880911827087, -1.1495229005813599, 0.572031557559967, 2.080153226852417, 0.6613060235977173, -0.18757811188697815, -0.9872203469276428, 2.1834940910339355, -1.6782822608947754, 0.3382166624069214, 0.22628606855869293, -0.7414641380310059, -0.07001400738954544, 1.6105036735534668, 0.007114469539374113, -0.9171275496482849, -0.6192829608917236, 0.10298650711774826, -0.3024977445602417, -0.886052668094635, -1.2357853651046753, 1.254205346107483, -0.9707645773887634, -0.24985389411449432, 1.7167394161224365, 0.6054838299751282, 0.6301211714744568, 1.7717032432556152, 1.1260128021240234, 0.049472324550151825, 0.8070343732833862, -0.037789564579725266, -1.2120754718780518, -0.6839466691017151, -0.4851337671279907, 0.4726792573928833, -1.0819497108459473, -0.2745855450630188, -0.4037788212299347, 1.0793697834014893, -1.3048464059829712, 0.2180597186088562, -0.3964110314846039, 0.8914735913276672, 0.8184393048286438, 0.6960389018058777, 0.6603571772575378, -0.3611639142036438, -0.715580403804779, 1.335415244102478, -0.4870678186416626, -0.8189433813095093, 0.9484086036682129, -0.6068118214607239, 1.204972743988037, 0.20994646847248077, 1.14267897605896, 0.7096786499023438, -0.45484790205955505, 0.7183856964111328, 0.07235312461853027, 0.5983669757843018, -1.0385223627090454, 0.3941369652748108, 0.7830584049224854, 1.694645643234253, 0.4849361181259155, 1.3869664669036865, -0.03364778682589531, -0.8349056243896484, -1.2615253925323486, -0.32878443598747253, -0.5848193168640137, 1.2138774394989014, -1.483710765838623, -0.10198354721069336, 0.7398826479911804, -0.3227657973766327, 1.096247911453247, -1.786647915840149, 1.2145721912384033, -0.5379764437675476, -0.5064207315444946, 1.6080689430236816, -0.9886369109153748, 0.22156332433223724, -0.11798929423093796, -2.260780096054077, 0.9223982691764832, 0.9978687763214111, -1.070590853691101, -0.13997012376785278, -0.44120436906814575, -0.6152284741401672, -0.9774949550628662, 0.7687832713127136, -0.052868813276290894, 0.35795074701309204, -0.01388342585414648, 0.03190036118030548, -1.228157877922058, -0.4119523763656616, 0.8943304419517517, 1.792137861251831, 0.25315240025520325, 0.8210328221321106, -0.729544997215271, -0.3931432366371155, -0.29741060733795166, -1.1780481338500977, -0.23491692543029785, -0.5705150961875916, 0.5112734436988831, -0.24131496250629425, -1.1552950143814087, 0.22912488877773285, 0.19068747758865356], [0.9722330570220947, -0.34363746643066406, 0.9654684066772461, -1.001649022102356, 0.3979239761829376, 0.016458475962281227, -0.22389304637908936, -0.3061673641204834, 0.35137489438056946, -0.3051976263523102, -0.340403288602829, -2.3481085300445557, -0.20233659446239471, 1.3974908590316772, -0.8017293810844421, -0.2876795530319214, 0.20969730615615845, -0.15525734424591064, -0.21418093144893646, -0.10919587314128876, 0.5944752097129822, 1.4458445310592651, 0.6672908067703247, -0.11503348499536514, -0.44900229573249817, -0.5769344568252563, -1.518984079360962, 0.37378761172294617, 1.5141364336013794, -0.44090935587882996, -0.5342138409614563, 0.3685242831707001, 0.3266124129295349, 1.7237380743026733, 0.3124238848686218, -0.592054545879364, -1.5684077739715576, 0.6639103889465332, -0.06787184625864029, 0.12481730431318283, -1.8184070587158203, 1.277504324913025, -0.52768874168396, -0.4419725239276886, -0.11165478080511093, 0.7693049907684326, 0.2778858244419098, 1.5063226222991943, -0.08192028105258942, -0.6421787142753601, -0.8897007703781128, 0.08025995641946793, 1.5967432260513306, 1.9113235473632812, 0.8378317952156067, 0.2326749712228775, -0.24189841747283936, 0.9106642007827759, 1.5643253326416016, 0.7284323573112488, -2.067091226577759, -0.5920053720474243, -0.05510757118463516, -0.9381882548332214, -1.1885206699371338, -2.5957140922546387, 0.030667461454868317, 0.3049287796020508, -0.2468004822731018, 0.33811676502227783, 0.42772310972213745, 0.5812069773674011, 0.5899582505226135, 0.14664243161678314, -0.7373461127281189, -3.222627878189087, 0.2821398675441742, 1.5846854448318481, 1.1975027322769165, -0.17761273682117462, -1.1392836570739746, 1.461287021636963, 0.43281909823417664, -1.2544935941696167, -0.682090699672699, 2.014199733734131, 0.36269137263298035, 0.44124650955200195, -1.5362991094589233, -0.8340660929679871, 0.045047610998153687, -0.562381386756897, -0.909299373626709, 1.1035765409469604, -1.3197746276855469, -0.5735412240028381, -0.1779196858406067, 0.6678482890129089, -0.8961023092269897, 0.7982361912727356, 1.977626085281372, -0.07280448079109192, -0.6372153759002686, 0.7231701016426086, 0.10605818033218384, 0.4440441131591797, -0.653406023979187, 0.2613089680671692, 0.8944543600082397, 0.7603898644447327, -0.8113483190536499, -1.2436333894729614, 0.0785820484161377, 1.1574190855026245, -1.2152197360992432, -2.213275671005249, 0.6496027112007141, -0.9560527801513672, 0.6302884221076965, 0.14237898588180542, 3.1707851886749268, 1.9766860008239746, 0.3361552953720093, 0.8508545756340027, 0.6137301921844482, -0.9567351341247559, 1.12907075881958, -2.1893649101257324, 0.5072382092475891], [-0.2220240980386734, 0.7682973146438599, -2.0132789611816406, -0.5001307725906372, -1.546848177909851, -0.304146409034729, 0.29179197549819946, -0.22049733996391296, -0.9282592535018921, 0.4118710458278656, 0.7479506731033325, -0.29287609457969666, -1.0572435855865479, -0.39454278349876404, 0.2870333790779114, -0.7355875968933105, -0.7466052770614624, -0.15928663313388824, -0.5461909174919128, 0.47798171639442444, -1.0129523277282715, -1.2508559226989746, -0.2983115613460541, 0.2932780385017395, 0.45040786266326904, -0.4745081961154938, 0.8927980661392212, 3.734311580657959, 1.807472825050354, 1.8530290126800537, 0.13936197757720947, -0.449374258518219, 0.36603087186813354, 1.0842374563217163, -1.3734967708587646, -1.3225873708724976, -1.821604609489441, 0.6753437519073486, 0.1790095716714859, -0.3220842778682709, -0.5152406692504883, -1.0106573104858398, -0.03375418111681938, 1.9902499914169312, 2.0556015968322754, 0.3136548399925232, -0.716550886631012, 0.03435482829809189, -0.12291204929351807, 1.0522164106369019, 0.11310093849897385, -1.140752911567688, 0.4144572615623474, 0.8830258846282959, -0.918487012386322, 0.04627740755677223, -0.7966190576553345, 1.7203642129898071, 0.9727014303207397, -1.3192120790481567, -0.036677319556474686, 0.17507176101207733, -0.2771582305431366, 0.47842952609062195, -0.5923422574996948, -0.8888139724731445, 0.10135089606046677, 1.1767841577529907, -0.41827884316444397, 2.8984241485595703, 1.884251356124878, 1.0777349472045898, -0.3269227147102356, 0.02444593608379364, 0.9710964560508728, 0.5700227618217468, -2.0348305702209473, -1.6133149862289429, -0.4745515286922455, -0.3931733965873718, 0.2924049496650696, -1.2985107898712158, -1.4776740074157715, 0.21261544525623322, 0.5466089248657227, -2.4957032203674316, -0.3580663502216339, -1.7375195026397705, 1.1308001279830933, -0.5336602330207825, -0.5244839787483215, -0.28321513533592224, -0.6558825969696045, 1.3824090957641602, -0.8686010837554932, 0.17067988216876984, -1.064714789390564, -0.04225260391831398, -1.2978140115737915, -1.1316864490509033, -1.2604551315307617, 0.5014464855194092, -0.2345830500125885, -0.0507555715739727, 1.034631371498108, 0.22669221460819244, 0.08987210690975189, -0.659522533416748, -0.27536541223526, 0.4218539297580719, -0.7601203322410583, -1.0821096897125244, -0.8292617797851562, 0.6680372953414917, 1.4837547540664673, 1.0130329132080078, -0.6690264344215393, -0.16815334558486938, 0.2780115008354187, 0.9777588844299316, -1.5858018398284912, -0.02694273553788662, 1.3365719318389893, -1.514461874961853, -1.8304508924484253, -0.5385110974311829, 0.13951465487480164, -0.7950605154037476, -0.8285729289054871], [-0.5456640720367432, 2.1856815814971924, -1.9786343574523926, 0.38887351751327515, 0.3083818852901459, 0.27417024970054626, 0.6668263077735901, 1.1125054359436035, -1.6040945053100586, 0.8893752694129944, 0.33143797516822815, 0.2331620156764984, 0.1753673255443573, -0.15844981372356415, -0.999954342842102, 1.3079736232757568, -1.6816298961639404, -0.27147722244262695, -1.6248366832733154, 1.9574211835861206, 0.35586017370224, -1.3147668838500977, -0.06896695494651794, -0.7321488261222839, 0.2244112491607666, -1.406585454940796, -1.6770890951156616, 0.3914380967617035, 0.8514639139175415, -0.22869549691677094, 0.10133766382932663, -1.6581569910049438, -0.9401750564575195, 1.5377436876296997, 0.4037153422832489, -2.0603129863739014, -0.17309552431106567, -0.23686593770980835, 0.4948436915874481, 0.38197270035743713, 0.2757629454135895, -0.7103621363639832, 0.5838643312454224, 0.8439751863479614, -2.2762980461120605, 0.5453916788101196, -0.24227967858314514, 0.3663233518600464, 3.208688735961914, -1.0013821125030518, 0.175170436501503, -0.6131299734115601, -1.2816942930221558, -1.9322470426559448, -0.8999828696250916, 1.126304268836975, -0.5897802710533142, 2.08036732673645, 0.005464942194521427, -0.9253013730049133, -0.49484407901763916, -0.6486172080039978, 0.44086241722106934, -1.8067896366119385, -0.9502727389335632, 0.059572406113147736, -0.31553685665130615, 0.5693250298500061, -2.1953022480010986, -1.304954171180725, 0.14637987315654755, 1.3585668802261353, 0.7031323313713074, 0.48649871349334717, -1.5217170715332031, -2.0491366386413574, -0.5727605223655701, -1.674612283706665, 1.2382571697235107, 0.9851496815681458, -0.5679962038993835, 0.1343740075826645, -1.4580512046813965, 0.03192080929875374, 0.2056874781847, -0.16136956214904785, 0.33569636940956116, -0.37406623363494873, 0.637736976146698, -1.152618169784546, 0.008262206800282001, -0.9868348836898804, 1.367638349533081, 0.31691715121269226, 0.6171714067459106, 1.3283374309539795, -2.5347912311553955, -0.8200708627700806, -1.0757495164871216, -0.22069674730300903, 1.1587603092193604, -0.48552799224853516, 1.6929970979690552, -1.3165653944015503, 1.6225066184997559, 0.6215651035308838, -0.18502353131771088, 0.5889856815338135, 0.8998830914497375, 0.24296878278255463, -1.2927128076553345, 1.0790497064590454, 2.060844659805298, -0.6530823111534119, -0.038572922348976135, 1.032898187637329, 0.21445193886756897, -0.7295951843261719, -0.08875332772731781, 0.636794924736023, -0.934361457824707, -0.022850317880511284, 0.864223301410675, 0.5578513145446777, -0.9762464165687561, -0.7507240772247314, 0.8542271852493286, -0.20763105154037476, -0.15070517361164093], [0.6655021905899048, -0.74172043800354, 0.4952254295349121, -0.3628617525100708, -1.7364999055862427, 1.1268198490142822, 0.6540005207061768, 0.44584545493125916, -0.273329496383667, -0.7717252969741821, -0.14225658774375916, 1.3948311805725098, -0.9924122095108032, 0.7632006406784058, -1.4265697002410889, -1.2306607961654663, 0.3010282516479492, 0.45027777552604675, 1.6980342864990234, -0.36935630440711975, 0.27750518918037415, -0.007025467231869698, 0.44506168365478516, -1.377368688583374, -0.14121223986148834, -0.1236194521188736, 0.5406948328018188, 0.8455978035926819, -3.3174855709075928, -1.5367180109024048, -0.2424534261226654, 0.34627434611320496, -0.01074010506272316, -1.941823124885559, -0.045263197273015976, -0.7395036220550537, -0.49528878927230835, 0.09503664076328278, -1.3217089176177979, 1.1571917533874512, -0.17675325274467468, 0.9180455207824707, 1.1812735795974731, -0.03614295646548271, 0.20439371466636658, -1.0999990701675415, -0.5547590851783752, 0.1702526956796646, -1.9300377368927002, -1.7603328227996826, 1.3732415437698364, 0.5567552447319031, 0.07299970835447311, 0.3434132933616638, -1.5353642702102661, 0.3904569447040558, 0.08799949288368225, 0.008192146196961403, -0.1963692307472229, -0.8356619477272034, 0.07762695103883743, 0.6058520078659058, 1.259594202041626, -0.23808236420154572, -1.1512771844863892, 0.26978564262390137, -0.9567323327064514, -0.9565275311470032, 1.0717480182647705, -0.5034385323524475, 0.3797331154346466, 0.7872884273529053, 3.148192882537842, 0.3340611755847931, 0.9717918038368225, 1.7456355094909668, -0.12284994125366211, -0.22063523530960083, -0.13826203346252441, 0.6127286553382874, 0.3560720682144165, 0.059291351586580276, 1.449455738067627, 0.07972567528486252, 0.10577522963285446, -0.28400346636772156, -0.8230472803115845, -0.6281409859657288, -0.5684266090393066, 0.2616405487060547, 1.1985841989517212, -0.49015486240386963, -0.3943394422531128, -0.05601680651307106, -1.452507734298706, -0.6128690242767334, -1.273786187171936, -1.1983453035354614, -0.12359108775854111, 0.21251742541790009, -0.43567708134651184, -1.669573426246643, 0.845770001411438, 0.44993340969085693, 0.43157151341438293, 0.08794451504945755, -0.027234874665737152, 1.242286205291748, 0.17367948591709137, 0.00020125057199038565, -1.0769426822662354, -0.8582533597946167, 0.33964666724205017, 0.10325916111469269, -1.3865872621536255, 0.9857825636863708, -0.3129481077194214, -0.4244137108325958, -0.032253868877887726, 0.5379862189292908, -0.8953966498374939, -0.502253532409668, -0.6036065816879272, -0.6315259337425232, -0.860014796257019, 0.41382038593292236, 0.900712788105011, 0.1693560928106308, -0.16135893762111664], [0.12372224777936935, -2.265615463256836, -0.5901804566383362, 1.215263843536377, 1.034895420074463, 0.4372863173484802, -1.1086070537567139, 0.5708608031272888, -1.3135986328125, 0.12638962268829346, 0.024253804236650467, 0.9342517852783203, -0.23874664306640625, -0.2890828549861908, 2.3328118324279785, -0.5376541018486023, -0.9685268998146057, 1.2717539072036743, 1.9637809991836548, 0.026866892352700233, 1.3289543390274048, 1.6914565563201904, 0.8206564784049988, 0.632486879825592, 0.12521205842494965, 0.24698098003864288, -1.2973679304122925, 0.029541539028286934, 0.03268086165189743, -0.40824979543685913, 1.2536686658859253, -0.7120565176010132, -0.6301459670066833, 1.116752028465271, -0.597528874874115, -0.18009614944458008, -0.5827272534370422, 0.5434631109237671, 0.6397274732589722, 1.0229626893997192, -0.21115824580192566, -0.05001223832368851, 2.2527353763580322, -0.8982374668121338, -0.06039731204509735, 0.9541569948196411, -0.5436893701553345, 1.5697352886199951, 0.10189089179039001, -0.008646432310342789, -0.6029994487762451, 0.45122814178466797, -1.0142594575881958, -1.1836116313934326, -0.8058986663818359, 0.9803525805473328, 0.709588885307312, 0.9150990843772888, -0.14457128942012787, 1.0688331127166748, 0.24184328317642212, 1.841200828552246, -1.3340139389038086, 0.3646058738231659, -2.0110018253326416, 1.049395203590393, -0.8148713111877441, 1.1013739109039307, -0.5959663987159729, 0.043716784566640854, 0.0779060497879982, 0.6845682859420776, 0.17737217247486115, -0.6217758059501648, 0.18703050911426544, 0.42686960101127625, 0.48062101006507874, -0.139447420835495, -2.8254072666168213, 0.18114601075649261, -1.760661005973816, -0.15800757706165314, 0.19362348318099976, -1.7286033630371094, 0.26619571447372437, -0.6658391952514648, -1.0988762378692627, -2.200218677520752, -1.293668508529663, -0.17504127323627472, 0.4278682470321655, 0.4452853798866272, -1.1638965606689453, -0.43101271986961365, -0.5989052653312683, 0.294254869222641, 1.387626051902771, 0.36168789863586426, -0.8760834336280823, -0.824579656124115, 0.9555468559265137, -1.6258236169815063, -1.4916965961456299, 0.05856158584356308, -0.3330722451210022, -0.2911015748977661, -0.7732148766517639, -0.8991890549659729, 0.6321445107460022, 1.5079227685928345, -0.8607047200202942, 0.6005702018737793, 1.2109261751174927, 0.9201133251190186, -0.028788598254323006, -1.7135831117630005, -0.8345838189125061, -0.5264238119125366, 2.0820140838623047, -0.9059473872184753, 0.7369084358215332, -1.3226161003112793, 0.16094133257865906, -0.09408554434776306, -1.5337542295455933, -1.5943593978881836, -0.12501320242881775, -0.2660960555076599, 0.4256017208099365], [-1.209132194519043, -2.016134738922119, 0.782101571559906, 0.7366455793380737, 0.8787949681282043, -0.6286424398422241, -1.5313547849655151, -1.1384066343307495, 0.6378459334373474, 1.0753278732299805, 1.468200445175171, 0.43968963623046875, -1.1550359725952148, -0.46628284454345703, -0.38835960626602173, 0.11817112565040588, -0.11965221911668777, -0.03569699823856354, 0.09121190756559372, -0.65073561668396, -0.42368921637535095, 1.2621866464614868, 0.20073378086090088, -1.1845678091049194, -1.7780193090438843, -0.006627058610320091, -0.15201236307621002, 0.232963427901268, -2.0270185470581055, -1.111290454864502, -0.4015055298805237, -0.8270136117935181, -1.8787740468978882, 0.33439528942108154, -0.20020948350429535, -0.2891215682029724, -1.4357101917266846, 1.192859172821045, -0.48178830742836, 0.23211054503917694, 0.3456217050552368, 1.8590214252471924, 0.18734492361545563, 0.1830187737941742, -0.5189838409423828, -1.8884446620941162, -0.5497286915779114, -0.15326054394245148, -0.021928567439317703, -0.5405301451683044, 0.41888925433158875, -1.0801132917404175, -0.2883341908454895, 2.20415997505188, 0.23689337074756622, -0.5889936089515686, 0.09260313957929611, -0.9679644703865051, 0.9474629759788513, -0.9631991982460022, 1.9274436235427856, -1.3067203760147095, 0.30987122654914856, -0.617366373538971, 2.3936092853546143, -0.7303352355957031, 0.13495182991027832, -1.8637100458145142, 0.6538116931915283, 1.1085453033447266, -0.38950151205062866, -1.2339787483215332, -0.5134446024894714, -0.19531476497650146, 0.11943265795707703, 0.12136911600828171, 0.06843464076519012, -0.08109109103679657, -1.7161145210266113, -0.0799894705414772, 0.9024009108543396, 0.2759823501110077, 0.5435807704925537, -0.00022686485317535698, -1.1737309694290161, 0.3361489474773407, -0.7269335389137268, 1.0508956909179688, 0.6655034422874451, -1.5503250360488892, -0.3728685677051544, -1.216781735420227, 0.6499500870704651, 1.0933395624160767, -0.18636268377304077, 0.038581542670726776, -0.05597413331270218, 0.8714186549186707, -0.4710417687892914, 0.06657564640045166, 1.0629092454910278, -0.7481351494789124, -1.336693286895752, 0.6002965569496155, 1.070198893547058, -0.13935211300849915, -1.6657742261886597, -0.10507181286811829, 1.1068650484085083, -1.1344910860061646, -0.8673089742660522, -0.655877947807312, 0.7557281255722046, 0.23759229481220245, -0.05375339090824127, -0.24884004890918732, -2.0582680702209473, -0.3643658459186554, -0.6711638569831848, 0.15355485677719116, -0.5209009051322937, -1.1638137102127075, -0.38235849142074585, 1.4375531673431396, -0.08041559159755707, -1.2752387523651123, 0.7987083196640015, -1.7341328859329224, -0.2822716236114502], [-0.16483600437641144, 0.9611403346061707, 1.5664340257644653, 0.7248936295509338, 1.0806403160095215, 0.6986967325210571, -1.8419618606567383, 0.3018915355205536, -0.15232688188552856, -1.3960338830947876, 1.7490553855895996, 0.32429593801498413, 0.6555843353271484, 0.36833739280700684, 0.8515682816505432, 0.8833321332931519, -1.1519273519515991, 1.54717218875885, -0.08140455186367035, 0.20710091292858124, 0.9972220659255981, -0.8005289435386658, -0.11110471189022064, -0.8774795532226562, 0.6307583451271057, -0.16772158443927765, -0.3218330442905426, 0.11868996918201447, 1.2604399919509888, 0.9460816979408264, 0.42977195978164673, 0.36509326100349426, -0.4156893193721771, 1.0015238523483276, 1.8475122451782227, 0.4263990819454193, -0.11846250295639038, -0.5274484157562256, 0.3190804421901703, 1.168351411819458, -1.2580193281173706, 0.7298218011856079, -1.7090874910354614, -1.342402696609497, -3.2877321243286133, 2.612905740737915, -1.4574928283691406, -0.14834129810333252, 0.5569126009941101, 0.20724429190158844, -0.679292619228363, -0.2398196905851364, 1.2811661958694458, -1.528645396232605, 0.9734450578689575, 0.8468846082687378, -1.6228985786437988, -2.7152462005615234, 1.3616905212402344, -1.4338139295578003, 1.1363766193389893, 2.123141050338745, 1.0358912944793701, 0.5401310920715332, 0.4743160009384155, -0.691665530204773, 1.5418953895568848, -0.2969687879085541, -2.2470452785491943, -0.5338793396949768, -2.1468706130981445, 1.3119021654129028, 1.4877252578735352, 2.2711009979248047, -0.7822232246398926, -1.066455364227295, -0.7441766858100891, 0.7726432681083679, 1.7454246282577515, 1.4022952318191528, 1.2637919187545776, 1.2159262895584106, -0.8041543960571289, 0.42137256264686584, -0.8056101202964783, -1.749475359916687, 0.5839058756828308, -0.18468493223190308, 1.420168399810791, -1.2850627899169922, -0.6114014983177185, 0.814688503742218, 1.172868251800537, -1.5339223146438599, -0.1822965443134308, -0.9293220043182373, 0.12203609943389893, 1.5761528015136719, 0.5382421612739563, 2.0838165283203125, -0.17287002503871918, 2.690089702606201, -1.8912795782089233, 0.1311834454536438, 0.2544931471347809, 1.1590964794158936, 1.4549846649169922, 1.1946709156036377, -0.9426529407501221, -1.620892882347107, -0.3940999209880829, -0.5865610837936401, -0.7490731477737427, -0.0546342134475708, -0.3738355040550232, -1.6091721057891846, -0.663013219833374, 0.9175202250480652, 1.396286964416504, 0.8266110420227051, -1.5355607271194458, -0.9858046770095825, 0.33106115460395813, -1.7147537469863892, -0.4437122046947479, -0.21422652900218964, -2.0199928283691406, 0.40736839175224304, 0.941746711730957]], "dtype": "float32"}, {"__nd__": [5.192441463470459, -7.0809125900268555, 1.9400687217712402, 3.9710891246795654, 2.130520820617676, 10.483063697814941, 4.231761455535889, 4.51466178894043, 7.472269058227539, -1.896806240081787, 0.16442416608333588, -0.650545597076416, -2.597400188446045, -1.9148112535476685, -6.532105445861816, -0.5163020491600037, -2.6679930686950684, 14.891243934631348, 5.628937721252441, -5.643043518066406, -2.6966066360473633, -3.468355655670166, -4.478619575500488, -7.92092752456665, 7.616043567657471, 2.619222640991211, -3.8939156532287598, -1.555195927619934, 0.7629141807556152, -1.4741075038909912, -4.1676483154296875, 5.362317085266113, -1.6288775205612183, -3.483341932296753, -1.0018702745437622, -10.470732688903809, 2.760099172592163, -3.5941739082336426, -3.0812439918518066, 4.342715740203857, 3.619515895843506, 4.395451068878174, 1.4298605918884277, 0.07651165127754211, 2.1162562370300293, 3.276492118835449, -3.387038230895996, 4.660963535308838, 6.724948883056641, 3.0289978981018066, 1.33937406539917, 1.4587128162384033, -6.911224365234375, 5.709543704986572, 0.6084954142570496, -8.389754295349121, 1.1721665859222412, 3.8637664318084717, -6.215631484985352, 9.606956481933594, 0.5615240335464478, 1.3635427951812744, -1.4484024047851562, 0.04827028140425682, 0.7299385070800781], "dtype": "float32"}], "e": {"__nd__": [[0.0, 2.061572313308716, 2.61350679397583, 0.0, 0.0, 2.0915985107421875, 0.0, 0.0, 1.861409068107605, 0.0, 6.053838729858398, 0.0, 5.899139404296875, 0.0, 0.0, 0.0, 0.0, 4.097242832183838, 5.946306228637695, 0.0, 0.0, 1.394200325012207, 9.541029930114746, 0.0, 18.85604476928711, 19.580089569091797, 21.194658279418945, 0.0, 3.8128302097320557, 0.0, 9.510796546936035, 2.9395859241485596, 0.0, 0.0, 6.705208778381348, 0.0, 3.9725496768951416, 13.124053001403809, 0.0, 6.33996057510376, 0.0, 0.0, 12.928603172302246, 0.0, 0.0, 7.24370002746582, 10.201744079589844, 17.43812370300293, 9.171943664550781, 0.0, 16.92328453063965, 0.0, 0.0, 0.0, 4.446175575256348, 0.0, 0.0, 5.657500743865967, 1.2550143003463745, 17.103561401367188, 25.023303985595703, 0.0, 16.801034927368164, 5.461627960205078, 0.0], [2.773469924926758, 0.0, 28.90129280090332, 0.0, 0.0, 14.566299438476562, 2.4051666259765625, 22.75040626525879, 6.576039791107178, 19.084047317504883, 6.834965229034424, 0.0, 0.0, 12.215113639831543, 0.0, 10.20932388305664, 5.71466064453125, 13.462560653686523, 1.4797332286834717, 3.2241129875183105, 0.0, 0.0, 0.0, 0.0, 0.0, 5.443172454833984, 1.9175388813018799, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.9663825035095215, 0.0, 0.0, 0.0, 2.2435319423675537, 6.039586544036865, 1.602848768234253, 0.0, 0.0, 1.142412781715393, 0.0, 0.0, 8.689831733703613, 19.87379264831543, 0.0, 0.0, 5.227123737335205, 3.564976930618286, 0.0, 0.0, 0.0, 1.146979570388794, 0.0, 5.902472019195557, 0.0, 0.8489447832107544, 0.0, 0.0, 0.0], [1.0837056636810303, 0.0, 3.493756055831909, 22.05582046508789, 0.0, 26.57202911376953, 5.377662658691406, 1.3037655353546143, 0.0, 5.7428879737854, 17.4072265625, 0.0, 9.337008476257324, 0.0, 0.0, 0.0, 0.0, 13.525757789611816, 17.86403465270996, 0.0, 0.0, 0.0, 0.0, 2.7619378566741943, 11.234899520874023, 0.9451928734779358, 0.0, 0.0, 0.0, 17.88425636291504, 7.788264751434326, 1.4603843688964844, 0.0, 0.0, 2.004701614379883, 0.0, 6.274209976196289, 0.0, 0.0, 0.1413179337978363, 4.74514627456665, 6.0586838722229, 11.320905685424805, 7.83075475692749, 0.0, 27.742372512817383, 0.0, 0.0, 0.0, 2.325201988220215, 10.111579895019531, 0.0, 0.0, 0.0, 0.0, 0.0, 10.342911720275879, 7.310905933380127, 4.932701110839844, 37.50273895263672, 16.232690811157227, 12.440169334411621, 16.516035079956055, 11.864255905151367, 0.0], [8.991754531860352, 0.0, 0.0, 0.0, 13.971949577331543, 13.30398178100586, 4.877757549285889, 12.540122032165527, 13.383069038391113, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.944039821624756, 0.0, 12.010956764221191, 10.196656227111816, 0.0, 3.4695885181427, 0.0, 0.0, 0.0, 2.537008762359619, 9.098384857177734, 0.0, 0.0, 4.844517707824707, 27.180328369140625, 5.944480895996094, 2.099111795425415, 1.5956791639328003, 0.0, 0.0, 0.2949516475200653, 0.0, 0.0, 5.652162075042725, 0.0, 0.0, 0.0, 9.25787353515625, 2.4781405925750732, 2.418574571609497, 6.258180141448975, 0.0, 0.0, 0.0, 7.738225936889648, 0.0, 0.0, 0.0, 12.780887603759766, 18.827686309814453, 2.0642712116241455, 9.7807035446167, 0.0, 0.0, 0.0, 0.0, 11.887892723083496, 2.2083168029785156, 0.0, 7.733676433563232], [7.872655868530273, 0.0, 0.0, 7.191403865814209, 7.25911808013916, 0.0, 14.691628456115723, 0.0, 0.0, 0.0, 3.7369279861450195, 0.0, 0.0, 0.0, 0.0, 7.53316593170166, 3.5626256465911865, 16.228342056274414, 0.0, 0.0, 0.0, 1.1409062147140503, 0.0, 0.0, 5.8197340965271, 12.789637565612793, 0.0, 0.0, 27.857511520385742, 0.0, 0.0, 19.926054000854492, 14.281915664672852, 0.0, 18.22283935546875, 13.322145462036133, 16.78449058532715, 0.0, 0.0, 21.5550479888916, 1.0230106115341187, 0.0, 0.0, 0.0, 0.0, 0.03521690517663956, 26.9949893951416, 0.0, 31.612993240356445, 0.12826906144618988, 1.0901676416397095, 3.793123483657837, 0.0, 0.0, 0.0, 0.0, 0.0, 4.297482013702393, 9.309500694274902, 13.253439903259277, 19.804019927978516, 1.7866623401641846, 6.517953872680664, 11.011239051818848, 1.5150103569030762], [0.0, 0.0, 25.30242919921875, 0.0, 18.484617233276367, 0.0, 14.207353591918945, 0.0, 3.877609968185425, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 19.953275680541992, 19.340253829956055, 0.0, 3.6141350269317627, 13.247328758239746, 0.0, 0.0, 27.582351684570312, 23.814998626708984, 0.0, 0.0, 1.9750348329544067, 5.365331172943115, 10.6630859375, 0.0, 0.0, 0.0, 0.0, 0.6594305634498596, 5.079451084136963, 0.0, 0.9235436916351318, 0.32667022943496704, 0.0, 15.4296236038208, 3.244873285293579, 9.483003616333008, 1.9144879579544067, 6.7844319343566895, 0.0, 0.0, 0.0, 11.568102836608887, 0.0, 1.8474125862121582, 0.0, 0.0, 1.6950017213821411, 0.0, 15.635587692260742, 0.0, 0.0, 0.0, 7.1940436363220215, 3.9535410404205322, 12.523370742797852, 7.231124401092529, 0.0], [15.981589317321777, 0.6006298661231995, 0.0, 7.054457187652588, 0.0, 17.3720760345459, 0.0, 18.131311416625977, 14.420533180236816, 0.0, 1.169270634651184, 0.0, 0.0, 4.090272903442383, 0.0, 0.0, 0.0, 1.0726957321166992, 0.0, 0.0, 0.0, 0.0, 9.226186752319336, 0.0, 10.695555686950684, 7.998143672943115, 19.002674102783203, 0.0, 30.227008819580078, 0.0, 0.0, 7.1805524826049805, 0.0, 0.0, 0.0, 0.0, 1.154508113861084, 12.747900009155273, 7.661304950714111, 1.4347161054611206, 0.0, 0.0, 0.0, 0.0, 17.641498565673828, 2.834522247314453, 0.0, 10.848565101623535, 0.0, 6.352072715759277, 8.281610488891602, 12.866353034973145, 8.046613693237305, 2.592121124267578, 0.0, 0.0, 18.006858825683594, 4.513750076293945, 0.0, 24.74472999572754, 0.0, 0.9432196021080017, 7.194480895996094, 0.0, 0.0], [1.1565817594528198, 14.087129592895508, 1.8722156286239624, 12.836003303527832, 0.0, 4.857381343841553, 0.0, 19.759016036987305, 16.17186164855957, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.164457321166992, 0.0, 18.386022567749023, 14.538516998291016, 2.437077045440674, 0.0, 17.37726593017578, 5.069021224975586, 0.0, 0.0, 4.312417984008789, 0.0, 0.0, 0.0, 0.407478928565979, 0.0, 6.931844711303711, 0.0, 0.0, 0.0, 0.0, 0.0, 4.205132484436035, 0.0, 13.019432067871094, 13.800089836120605, 0.0, 11.586844444274902, 0.0, 21.462549209594727, 27.0323543548584, 0.0, 0.2019418627023697, 4.352804660797119, 13.432206153869629, 0.0, 0.0, 0.0, 5.983180999755859, 1.8625136613845825, 0.12306171655654907, 5.009980201721191, 0.0, 4.695154666900635, 23.119770050048828, 9.0462646484375, 0.0, 0.0, 6.1663103103637695, 12.76027774810791], [0.0, 0.0, 6.531115531921387, 15.942774772644043, 0.0, 17.917760848999023, 14.01974105834961, 22.332609176635742, 17.490209579467773, 7.475976943969727, 3.4362430572509766, 0.0, 0.0, 0.0, 0.0, 0.0, 7.5233564376831055, 0.0, 0.0, 0.0, 0.0, 2.153228282928467, 0.0, 0.0, 7.434800624847412, 6.540874004364014, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1646091639995575, 3.182914972305298, 0.0, 9.138258934020996, 0.0, 0.0, 18.69150161743164, 15.052911758422852, 0.664715051651001, 0.0, 8.02357006072998, 35.68968200683594, 7.779021263122559, 0.0, 0.0, 15.979799270629883, 0.0, 5.4070940017700195, 1.7838443517684937, 5.812135219573975, 5.552441120147705, 14.127327919006348, 0.0, 0.0, 0.0, 7.4874725341796875, 28.042593002319336, 0.0, 7.120553493499756, 0.0, 0.0, 3.381781816482544], [23.504138946533203, 0.0, 0.0, 3.4127421379089355, 0.0, 0.0, 0.0, 12.741564750671387, 10.96090030670166, 0.0, 0.0, 9.591910362243652, 0.0, 2.7141482830047607, 0.0, 0.241989865899086, 0.0, 12.801419258117676, 6.712325096130371, 0.0, 0.0, 0.8083069920539856, 0.0, 7.260581016540527, 0.0, 0.0, 6.71905517578125, 8.12222671508789, 0.0, 0.0, 0.0, 12.769107818603516, 2.896308422088623, 3.8571794033050537, 1.0552120208740234, 0.0, 0.0, 11.739974975585938, 0.0, 10.749747276306152, 1.2745306491851807, 0.11785949766635895, 20.474262237548828, 0.0, 0.0, 0.0, 0.0, 0.0, 9.818327903747559, 16.344036102294922, 9.66901969909668, 7.3486409187316895, 0.0, 11.09388256072998, 0.0, 0.0, 0.0, 0.0, 10.271161079406738, 0.0, 0.0, 6.849680423736572, 6.028445720672607, 0.0, 0.0], [0.0, 0.0, 20.20403289794922, 18.759153366088867, 8.425262451171875, 18.22199821472168, 0.0, 9.232177734375, 20.697969436645508, 0.0, 0.0, 0.0, 0.0, 9.115715980529785, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 15.865068435668945, 0.1897066831588745, 5.420656681060791, 0.0, 9.471543312072754, 0.0, 0.0, 5.006795883178711, 0.0, 0.0, 0.0, 0.0, 12.945857048034668, 0.0, 12.777673721313477, 10.897347450256348, 0.0, 25.062917709350586, 0.0, 0.0, 0.0, 11.793521881103516, 0.7428923845291138, 0.0, 1.9332941770553589, 2.622589588165283, 0.0, 4.293967247009277, 0.0, 23.50691795349121, 17.460691452026367, 0.0, 1.2836048603057861, 9.235921859741211, 8.17451286315918, 0.0, 32.465267181396484, 0.0, 0.0, 0.0, 25.31131935119629], [0.6284705400466919, 18.38231086730957, 0.0, 0.0, 0.0, 14.524508476257324, 10.313490867614746, 14.789108276367188, 11.64806842803955, 0.0, 0.0, 5.4139556884765625, 13.891255378723145, 0.0, 0.0, 0.0, 3.7634425163269043, 4.956428527832031, 2.1502554416656494, 0.0, 4.736049652099609, 0.0, 1.602297067642212, 0.0, 12.644779205322266, 0.0, 1.7403918504714966, 7.324120998382568, 9.162412643432617, 0.0, 0.0, 0.0904771015048027, 3.706569194793701, 0.0, 14.468592643737793, 0.0, 22.462080001831055, 0.0, 0.0, 0.0, 10.724778175354004, 13.778899192810059, 0.0, 7.818782806396484, 12.67831802368164, 27.636388778686523, 6.805524826049805, 3.0604774951934814, 27.38045883178711, 0.0, 2.982297897338867, 0.0, 2.5813686847686768, 12.573782920837402, 1.099212646484375, 0.0, 0.0, 8.954063415527344, 0.0, 0.0, 2.2091498374938965, 15.999458312988281, 8.508642196655273, 0.0, 0.0], [9.666775703430176, 0.0, 14.826150894165039, 0.0, 1.652062177658081, 0.0, 8.55396842956543, 0.0, 4.113151550292969, 19.199993133544922, 0.0, 0.0, 0.0, 2.929382085800171, 3.843635082244873, 0.0, 0.4103092551231384, 12.506800651550293, 3.7171201705932617, 7.842494964599609, 0.0, 16.92319107055664, 0.0, 0.0, 4.126672744750977, 11.43913459777832, 0.0, 17.716995239257812, 0.0, 0.8296526074409485, 0.0, 10.930330276489258, 0.0, 0.0, 0.0, 0.0, 0.0, 4.886873245239258, 0.0, 11.139711380004883, 5.626890182495117, 0.24411334097385406, 14.729326248168945, 0.0, 19.96725082397461, 8.6688232421875, 0.9045476317405701, 0.0, 6.870932579040527, 3.5928637981414795, 0.0, 0.0, 0.0, 14.361088752746582, 0.0, 2.0589940547943115, 0.0, 9.122153282165527, 0.0, 12.361533164978027, 1.802799940109253, 0.0, 0.0, 0.0, 5.647309303283691], [8.944174766540527, 10.737967491149902, 16.10096549987793, 0.892734169960022, 6.9120659828186035, 0.0, 23.285125732421875, 6.234783172607422, 0.0, 16.36953353881836, 0.0, 1.7591028213500977, 0.0, 5.972031116485596, 0.0, 7.352409839630127, 12.721064567565918, 8.498936653137207, 7.856603145599365, 0.0, 0.0, 3.6092374324798584, 0.0, 0.0, 28.691646575927734, 0.963675320148468, 0.0, 0.0, 0.0, 0.0, 0.0, 8.812678337097168, 0.0, 4.329267978668213, 20.12419891357422, 0.0, 11.756122589111328, 0.7237447500228882, 0.0, 37.16525650024414, 8.919407844543457, 0.0, 3.479088306427002, 0.0, 8.545060157775879, 28.06382942199707, 9.660367965698242, 6.6611127853393555, 10.518982887268066, 2.502791166305542, 0.0, 0.0, 0.0, 0.0, 0.0, 5.20974588394165, 0.0, 10.180980682373047, 0.0, 2.5111050605773926, 18.90997314453125, 0.131927028298378, 0.0, 0.0, 32.2551383972168], [18.457534790039062, 0.0, 3.853691816329956, 4.310341835021973, 4.954873561859131, 5.615508556365967, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 18.222946166992188, 0.0, 4.548092365264893, 18.69434928894043, 0.0, 2.993845224380493, 0.0, 0.0, 0.0, 0.29490238428115845, 0.0, 6.065721035003662, 5.939512729644775, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2787768840789795, 0.0, 22.779691696166992, 0.0, 14.566570281982422, 0.0, 0.0, 5.349245548248291, 31.713712692260742, 11.863384246826172, 0.0, 16.955974578857422, 0.0, 0.0, 0.6405744552612305, 0.0, 8.181087493896484, 0.0, 2.9881091117858887, 0.0, 0.0, 2.643496036529541, 11.887099266052246, 0.0, 24.385133743286133, 17.632421493530273, 11.991865158081055, 0.0, 25.52662467956543, 5.078761577606201, 0.0, 0.7496396899223328, 10.354190826416016], [20.36712646484375, 0.0, 3.7113523483276367, 0.0, 0.0, 0.0, 3.3019859790802, 1.011053204536438, 3.0130839347839355, 10.91733455657959, 0.35552552342414856, 4.034356594085693, 0.0, 6.605802059173584, 2.173095226287842, 0.0, 0.0, 13.28803539276123, 20.31267738342285, 0.0, 0.0, 0.0, 0.0, 8.918461799621582, 4.398437976837158, 1.5755923986434937, 2.4034903049468994, 0.0, 6.605911731719971, 0.17889487743377686, 0.0, 0.0, 4.248228073120117, 2.4059619903564453, 0.0, 0.11636734008789062, 19.895578384399414, 0.0, 0.0, 0.0, 17.574106216430664, 12.49317741394043, 7.6241044998168945, 1.417284607887268, 22.899547576904297, 1.3572947978973389, 0.0, 15.34880256652832, 6.3100175857543945, 7.321730613708496, 0.0, 14.926719665527344, 6.701154708862305, 0.0, 0.0, 0.0, 12.487175941467285, 5.80281400680542, 0.0, 8.593334197998047, 3.933201789855957, 0.4706604480743408, 16.201263427734375, 0.0, 6.753203868865967], [6.680969715118408, 0.0, 0.0, 16.235828399658203, 1.904956579208374, 0.0, 1.8758490085601807, 15.074277877807617, 24.898818969726562, 0.0, 0.0, 6.519101619720459, 2.2431068420410156, 0.0, 4.270153999328613, 0.3681742548942566, 3.2028417587280273, 20.785402297973633, 0.0, 0.0, 1.0609374046325684, 0.0, 0.0, 0.0, 16.598705291748047, 7.336300849914551, 9.15633773803711, 0.0, 20.292898178100586, 0.0, 0.0, 0.0, 3.141000986099243, 0.0, 14.192091941833496, 0.2958558201789856, 0.0, 13.392268180847168, 7.410467147827148, 7.55181884765625, 11.59999942779541, 21.76601791381836, 0.0, 0.0, 0.7596214413642883, 0.0, 22.608625411987305, 11.555540084838867, 21.288240432739258, 16.861759185791016, 0.0, 4.374950885772705, 3.7780582904815674, 0.0, 11.7283296585083, 0.0, 0.0, 19.856630325317383, 0.0, 13.13042163848877, 5.140769004821777, 0.0, 0.0, 0.0, 28.79539680480957], [0.025106538087129593, 0.0, 0.0, 20.899852752685547, 0.0, 0.0, 15.905367851257324, 2.6542506217956543, 16.78696060180664, 3.462252140045166, 15.109785079956055, 1.863294243812561, 0.0, 0.0, 0.0, 2.426959276199341, 0.0, 15.88792610168457, 7.221665859222412, 0.0, 7.3750152587890625, 19.20627784729004, 1.6591147184371948, 0.0, 7.626767158508301, 7.620816230773926, 0.0, 11.428854942321777, 0.0, 0.0, 0.0, 19.621139526367188, 0.0, 0.0, 0.0, 0.0, 17.678264617919922, 5.361883163452148, 19.453428268432617, 11.581276893615723, 0.0, 0.0, 0.0, 7.995516777038574, 26.928672790527344, 9.460039138793945, 7.041441917419434, 15.415181159973145, 0.0, 0.0, 0.0, 0.0, 12.234109878540039, 16.980009078979492, 15.547379493713379, 0.0, 0.0, 0.0, 3.760432004928589, 0.0, 0.0, 1.203561544418335, 1.7247881889343262, 0.3228054940700531, 0.0], [5.518486022949219, 0.7032095789909363, 0.0, 0.0, 2.1367552280426025, 0.0, 0.0, 1.2325371503829956, 6.877725124359131, 0.617559015750885, 0.0, 0.0, 3.231748104095459, 0.0, 0.0, 0.0, 0.0, 19.967491149902344, 2.759894371032715, 7.794782638549805, 0.0, 0.0, 0.0, 0.0, 0.0, 2.122666597366333, 5.365543365478516, 2.7617239952087402, 0.0, 0.0, 27.16176414489746, 16.80313491821289, 12.83079719543457, 0.0, 0.0, 0.0, 2.5132882595062256, 0.0, 5.952218055725098, 4.197847366333008, 8.08041000366211, 0.0, 0.0, 0.0, 2.576465129852295, 11.858668327331543, 0.0, 5.179126262664795, 16.99539566040039, 0.0, 7.30363130569458, 2.7042572498321533, 0.0, 10.614100456237793, 0.0, 0.0, 0.0, 0.0, 0.0, 13.642669677734375, 0.0, 0.0, 0.0, 0.0, 11.183428764343262], [12.50996208190918, 0.0, 35.220924377441406, 4.435034275054932, 6.414187908172607, 30.046894073486328, 17.071640014648438, 3.350031852722168, 16.763750076293945, 4.606095314025879, 0.0, 8.430081367492676, 7.730908393859863, 0.0, 0.0, 22.6121768951416, 7.593461990356445, 0.7071569561958313, 13.715547561645508, 11.430388450622559, 8.151870727539062, 0.0, 12.55856990814209, 10.805025100708008, 0.0, 0.0, 0.0, 5.8646674156188965, 0.6148300766944885, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.9990179538726807, 0.0, 6.365845680236816, 0.0, 10.92404842376709, 8.461064338684082, 6.049352169036865, 2.15771222114563, 7.464937686920166, 0.0, 7.58621072769165, 12.824125289916992, 19.145458221435547, 0.0, 0.0, 10.063350677490234, 19.383583068847656, 0.0, 0.0, 0.0, 10.606318473815918, 4.706711769104004, 14.827520370483398, 0.0, 0.8589929938316345, 9.388429641723633, 0.0, 0.0], [21.76722526550293, 0.0, 4.366804599761963, 0.0, 0.006312689743936062, 19.409404754638672, 0.6694746613502502, 0.0, 8.586409568786621, 2.3689143657684326, 9.874670028686523, 3.2596595287323, 0.0, 0.0, 0.0, 0.0, 5.571990489959717, 11.969564437866211, 4.81881856918335, 0.0, 0.0, 0.0, 14.946863174438477, 0.0, 0.0, 2.9205806255340576, 0.0, 0.0, 0.0, 10.889846801757812, 23.36688232421875, 0.8949149250984192, 0.11436211317777634, 7.106514930725098, 0.0, 0.0, 0.0, 0.0, 9.349915504455566, 8.317667007446289, 0.0, 0.0, 3.7201945781707764, 0.0, 0.0, 0.0, 0.0, 21.991437911987305, 10.267843246459961, 7.610140323638916, 6.669703960418701, 0.0, 0.0, 5.371822834014893, 8.095850944519043, 0.0, 0.0, 13.150655746459961, 4.322577953338623, 8.31022834777832, 10.037646293640137, 0.0, 0.0, 5.650837421417236, 11.201711654663086], [11.69169807434082, 0.0, 1.2168525457382202, 0.0, 0.0, 6.459079742431641, 7.992850303649902, 0.0, 6.656632423400879, 0.0, 3.012726068496704, 0.0, 0.0, 0.0, 0.0, 9.44875431060791, 3.3803799152374268, 27.126476287841797, 0.46637529134750366, 0.0, 0.0, 8.108738899230957, 0.0, 0.0, 9.907809257507324, 24.169404983520508, 0.0, 0.0, 12.59168529510498, 0.37973424792289734, 0.0, 0.0, 3.667794942855835, 0.0, 0.0, 0.0, 0.0, 8.672301292419434, 0.0, 5.244638919830322, 5.327882766723633, 16.22365379333496, 0.0, 1.5623339414596558, 5.825466156005859, 0.0, 0.0, 9.546506881713867, 17.157827377319336, 0.0, 0.0, 0.0, 0.0, 10.294615745544434, 5.619077682495117, 0.0, 15.541863441467285, 0.0, 13.13283920288086, 9.452000617980957, 1.8614574670791626, 1.0649770498275757, 13.278976440429688, 0.0, 0.0], [24.490610122680664, 0.0, 0.0, 18.629396438598633, 0.0, 0.0, 5.4125237464904785, 8.850747108459473, 0.2343551218509674, 1.6822946071624756, 7.718565464019775, 14.832682609558105, 0.0, 1.887839436531067, 0.0, 0.0, 22.537477493286133, 7.081846237182617, 0.08779405057430267, 0.0, 20.54224967956543, 0.0, 14.543647766113281, 9.486635208129883, 0.0, 0.0, 0.0, 10.05469799041748, 0.0, 0.0, 6.896459102630615, 0.0, 0.0, 20.13068199157715, 12.831340789794922, 0.0, 1.698988914489746, 0.0, 0.0, 18.111846923828125, 8.453692436218262, 14.641776084899902, 0.0, 0.0, 0.0, 8.148717880249023, 4.42364501953125, 12.58952522277832, 0.0, 1.3708845376968384, 3.9408457279205322, 1.220200777053833, 0.9624874591827393, 15.602416038513184, 0.0, 0.0, 0.0, 19.522579193115234, 14.787322998046875, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.9008623361587524, 5.496934413909912, 2.1751019954681396, 30.656734466552734, 0.0, 22.037803649902344, 4.295907974243164, 24.73657989501953, 17.45142364501953, 0.0, 0.0, 10.498013496398926, 0.0, 0.0, 0.0, 0.0, 7.539290428161621, 2.626960039138794, 9.801654815673828, 0.0, 0.0, 0.0, 0.0, 0.0, 1.297662377357483, 0.0, 9.277658462524414, 0.0, 0.0, 0.0, 0.0, 26.866348266601562, 7.912567138671875, 1.7308669090270996, 0.0, 0.0, 17.851957321166992, 0.4269745647907257, 0.0, 21.835973739624023, 8.913174629211426, 1.7018334865570068, 10.47851276397705, 0.0, 0.0, 0.0, 2.8454816341400146, 16.834692001342773, 20.873945236206055, 0.0, 0.0, 18.975122451782227, 17.080320358276367, 12.558873176574707, 4.249257564544678, 0.0, 0.0, 0.0, 0.0, 23.988813400268555, 0.0, 0.0, 0.0, 0.0, 21.414669036865234], [38.66322708129883, 10.626771926879883, 2.9519808292388916, 2.767085075378418, 7.824501991271973, 0.0, 24.33098030090332, 8.541794776916504, 12.073490142822266, 2.492396354675293, 6.741001129150391, 4.567751884460449, 0.0, 6.365921974182129, 0.44110581278800964, 13.417214393615723, 11.461915969848633, 21.374731063842773, 0.0, 4.045053482055664, 26.723302841186523, 3.740981340408325, 11.070747375488281, 0.0, 15.286789894104004, 0.0, 1.3156030178070068, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.695723533630371, 5.3682332038879395, 0.0, 0.0, 0.0, 11.828927993774414, 9.828866958618164, 0.0, 0.406390905380249, 11.348649024963379, 9.514842987060547, 13.112314224243164, 0.0, 22.649717330932617, 0.0, 4.2875542640686035, 16.55326271057129, 0.0, 17.415498733520508, 9.72220516204834, 0.0, 0.0, 0.0, 0.0, 28.99892234802246, 0.0, 0.0, 4.054312229156494, 17.06903076171875, 14.9052734375, 0.0], [17.95339584350586, 2.514504909515381, 0.0, 25.989229202270508, 21.38395881652832, 0.0, 0.0, 27.652402877807617, 22.93181610107422, 0.0, 0.0, 0.0, 0.3636694848537445, 0.0, 0.0, 1.4324859380722046, 0.0, 26.88923454284668, 0.8887240886688232, 0.0, 3.695589303970337, 0.0, 0.0, 0.0, 11.912753105163574, 0.0, 0.0, 0.0, 23.522092819213867, 0.06865917146205902, 0.0, 15.807564735412598, 2.136199474334717, 7.4100446701049805, 4.565275192260742, 0.0, 0.0, 5.212780952453613, 0.0, 8.888880729675293, 4.2833452224731445, 0.0, 3.7101025581359863, 0.0, 7.96066951751709, 8.9527006149292, 0.0, 0.0, 16.87981605529785, 0.0, 0.7235162258148193, 0.0, 0.0, 0.0, 32.31877517700195, 0.0, 0.0, 1.778249979019165, 0.0, 0.0, 22.8646183013916, 0.0, 0.0, 0.0, 5.08356237411499], [2.0029404163360596, 8.700376510620117, 0.0, 10.859773635864258, 0.0, 0.0, 1.2689011096954346, 10.082498550415039, 0.263011634349823, 0.0, 0.0, 0.0, 0.0, 0.6194679141044617, 0.0, 0.0, 0.0, 50.595947265625, 5.563473224639893, 0.0, 0.0, 3.875040292739868, 0.0, 0.0, 13.787333488464355, 6.30827522277832, 0.0, 13.455755233764648, 19.932649612426758, 13.479828834533691, 7.68717098236084, 33.681087493896484, 7.479728698730469, 14.175580978393555, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.8985122442245483, 1.3390673398971558, 0.0, 13.172287940979004, 14.003508567810059, 4.949347496032715, 4.927346229553223, 0.0, 14.693346977233887, 0.0, 0.0, 0.0, 0.0, 9.382905006408691, 1.554520845413208, 22.71044158935547, 0.0, 0.0, 9.457439422607422, 14.18781852722168, 0.0, 15.62633228302002, 2.443655490875244, 0.0], [0.864464282989502, 5.390331268310547, 0.0, 15.635635375976562, 0.0, 4.197887897491455, 16.386898040771484, 16.118444442749023, 3.0914323329925537, 18.517793655395508, 11.143975257873535, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 26.658035278320312, 0.0, 2.6124019622802734, 0.0, 7.041708946228027, 0.0, 0.0, 24.291118621826172, 16.930011749267578, 0.0, 0.0, 0.0, 1.7245023250579834, 0.0, 12.621285438537598, 0.0, 0.0, 6.724195957183838, 0.0, 2.377729654312134, 0.0, 0.305968314409256, 11.378935813903809, 10.115273475646973, 0.0, 5.6710286140441895, 0.0, 0.0, 24.401533126831055, 0.0, 0.0, 0.4310320019721985, 0.6257863640785217, 1.6733603477478027, 0.0, 0.0, 0.0, 6.033348083496094, 9.30294132232666, 15.879881858825684, 0.0, 0.0, 26.021543502807617, 0.0, 0.0, 0.0, 0.0, 14.247552871704102], [20.95321273803711, 0.0, 2.0445761680603027, 0.9451636672019958, 16.46552848815918, 7.719300746917725, 0.0, 2.7379467487335205, 0.0, 0.0, 0.0, 14.79482650756836, 2.521305561065674, 0.0, 0.0, 14.291507720947266, 0.59027498960495, 15.949019432067871, 13.05756664276123, 4.488799571990967, 0.0, 0.0, 0.0, 0.0, 25.299406051635742, 0.0, 0.0, 0.0, 11.471648216247559, 0.0, 0.0, 0.7055661082267761, 5.124263286590576, 0.0, 0.0, 0.0, 2.4129953384399414, 0.0, 0.0, 0.578978419303894, 21.334671020507812, 16.647621154785156, 10.175573348999023, 0.0, 6.405449390411377, 34.32637405395508, 0.0, 15.650263786315918, 0.9479021430015564, 0.4048219919204712, 0.0, 12.00723934173584, 0.0, 0.0, 0.0, 0.0, 0.0, 16.92683219909668, 0.0, 11.147055625915527, 19.933692932128906, 0.0, 3.785470485687256, 0.0, 2.6225483417510986], [3.5338919162750244, 4.898455619812012, 0.0, 4.221823215484619, 0.0, 15.117111206054688, 8.481405258178711, 1.773959994316101, 0.0, 0.0, 0.0, 3.129828691482544, 22.203237533569336, 1.8666348457336426, 0.0, 0.0, 0.0, 14.02841854095459, 21.738384246826172, 0.0, 1.1002802848815918, 3.643376588821411, 0.0, 0.0, 0.0, 10.408215522766113, 0.0, 0.0, 18.897403717041016, 0.0, 0.0, 0.7124332189559937, 1.372053623199463, 0.0, 9.238042831420898, 0.0, 16.032676696777344, 0.0, 0.0, 0.0, 14.13668155670166, 0.0, 2.240098714828491, 0.0, 0.0, 6.237249851226807, 0.0, 5.6577887535095215, 11.86292839050293, 0.0, 6.441309928894043, 12.054966926574707, 0.0, 2.4443790912628174, 2.9003851413726807, 0.0, 9.244572639465332, 0.0, 0.0, 6.722400188446045, 19.57232666015625, 0.0, 0.756004810333252, 0.0, 0.0], [15.745070457458496, 0.0, 5.732130527496338, 6.995194435119629, 0.0, 8.802629470825195, 17.225486755371094, 12.082746505737305, 28.74611473083496, 0.0, 6.61961555480957, 7.0824055671691895, 3.7855310440063477, 0.0, 10.184062004089355, 0.0, 0.0, 44.5517463684082, 0.0, 0.0, 10.312836647033691, 0.0, 0.0, 0.0, 15.51880168914795, 0.0, 0.038747191429138184, 0.0, 4.559244155883789, 9.844627380371094, 0.0, 18.000741958618164, 8.2903413772583, 3.609487295150757, 0.0, 0.0, 16.843547821044922, 10.940482139587402, 4.563331604003906, 37.17382049560547, 3.4393253326416016, 0.5553265810012817, 0.0, 0.0, 11.368724822998047, 0.0, 0.0, 12.103241920471191, 6.35558557510376, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.081241130828857, 0.0, 32.10575866699219, 9.412760734558105, 0.0, 0.0, 0.0, 0.0], [14.815671920776367, 0.0, 0.0, 5.739008903503418, 13.357791900634766, 18.565813064575195, 4.862656116485596, 0.0, 6.754418849945068, 0.0, 10.961143493652344, 13.154009819030762, 0.0, 0.0, 6.623732566833496, 1.9575573205947876, 12.245898246765137, 28.67898178100586, 1.6208069324493408, 4.522940158843994, 0.0, 0.0, 8.013848304748535, 11.393909454345703, 6.2219672203063965, 9.336706161499023, 0.0, 3.3887393474578857, 0.0, 0.0, 0.0, 5.699981212615967, 13.058076858520508, 7.832554340362549, 0.0, 0.0, 0.0, 1.9955533742904663, 2.4523134231567383, 5.762149333953857, 19.716920852661133, 10.630220413208008, 0.0, 0.0, 0.0, 0.711072564125061, 0.0, 6.177604675292969, 6.5227532386779785, 4.9313788414001465, 3.7569780349731445, 0.0, 0.0, 5.855235576629639, 0.0, 0.0, 15.638725280761719, 0.0, 0.0, 4.572990894317627, 0.0, 3.0245347023010254, 0.0, 1.1459566354751587, 16.797508239746094], [4.449400901794434, 0.0, 12.900726318359375, 12.10590934753418, 26.040069580078125, 0.0, 15.440126419067383, 0.0, 0.0, 11.014510154724121, 3.3002688884735107, 0.0, 0.0, 0.0, 23.857030868530273, 3.2317123413085938, 9.044553756713867, 9.773579597473145, 8.275857925415039, 0.0, 0.0, 0.0, 0.0, 0.0, 3.3998682498931885, 19.700035095214844, 0.0, 0.0, 0.0, 0.0, 0.0, 5.7899932861328125, 3.7718217372894287, 0.0, 0.0, 0.0, 9.770927429199219, 9.34213924407959, 1.1022813320159912, 15.804594993591309, 10.343486785888672, 9.784253120422363, 13.91499137878418, 0.0, 0.0, 0.0, 4.191564559936523, 15.05835247039795, 37.51007843017578, 10.709148406982422, 14.714812278747559, 12.922565460205078, 0.0, 13.163946151733398, 0.0, 0.0, 11.239633560180664, 0.0, 0.0, 11.339919090270996, 4.560539245605469, 0.0, 0.0, 6.98033332824707, 13.662784576416016], [17.954452514648438, 0.0, 27.443376541137695, 0.0, 10.653892517089844, 13.154044151306152, 26.657405853271484, 5.180745601654053, 16.239316940307617, 0.0, 0.0, 6.4214019775390625, 0.29862192273139954, 0.0, 0.0, 13.826553344726562, 2.665240526199341, 8.834171295166016, 13.386210441589355, 16.861291885375977, 0.0, 0.0, 0.0, 2.648000717163086, 14.234274864196777, 0.0, 0.0, 11.36109447479248, 12.409819602966309, 8.437623023986816, 0.0, 0.0, 0.0, 0.4890397787094116, 0.0, 0.0, 0.0, 0.0, 0.0, 9.002269744873047, 22.721229553222656, 8.078453063964844, 5.501399517059326, 0.0, 13.75015640258789, 3.8332815170288086, 0.0, 17.697973251342773, 2.7308597564697266, 2.0525717735290527, 0.0, 9.791232109069824, 0.0, 0.0, 0.0, 0.0, 0.0, 12.504071235656738, 0.0, 11.622101783752441, 0.0, 8.156065940856934, 0.0, 0.0, 0.0], [0.0, 4.36178731918335, 0.0, 8.838013648986816, 0.0, 0.0, 0.0, 8.717215538024902, 0.0, 0.0, 4.14424467086792, 0.0, 3.901789426803589, 0.0, 0.0, 22.959869384765625, 0.0, 20.82872200012207, 0.0, 1.8533821105957031, 0.0, 23.99268913269043, 0.0, 0.9747166037559509, 21.089393615722656, 11.426066398620605, 12.087488174438477, 1.7351011037826538, 5.896446228027344, 0.0, 3.8870699405670166, 6.48669958114624, 8.574698448181152, 0.0, 1.3951036930084229, 0.0, 2.2205190658569336, 0.0, 0.2169419527053833, 27.03018569946289, 15.23062515258789, 9.989315032958984, 16.206344604492188, 9.40927505493164, 7.548544406890869, 0.0, 0.0, 1.735679030418396, 6.525964736938477, 8.026677131652832, 0.0, 6.001040458679199, 0.0, 0.0, 22.547386169433594, 0.0, 0.0, 13.894405364990234, 0.0, 11.229422569274902, 12.590567588806152, 0.0, 0.0, 0.0, 1.5468511581420898], [28.920555114746094, 0.3911503851413727, 9.7298002243042, 13.925904273986816, 4.420712471008301, 4.210310459136963, 13.110742568969727, 17.81890296936035, 11.164058685302734, 5.2117791175842285, 32.0873908996582, 1.3139714002609253, 0.0, 18.964262008666992, 0.0, 4.773681163787842, 19.25164794921875, 3.356884479522705, 5.720201015472412, 6.714349269866943, 0.0, 6.375284194946289, 11.828402519226074, 1.2959039211273193, 14.38017463684082, 0.0, 0.0, 8.046730995178223, 0.0, 0.0, 5.575493335723877, 0.0, 5.133650302886963, 1.0708414316177368, 23.02743911743164, 0.0, 15.112759590148926, 14.987520217895508, 4.264736652374268, 8.283879280090332, 2.2551474571228027, 0.0, 2.904097557067871, 0.0, 0.0, 22.128549575805664, 0.6295778751373291, 25.750709533691406, 0.0, 16.400043487548828, 14.925603866577148, 0.0, 5.895560264587402, 2.9839084148406982, 0.0, 0.0, 3.3206942081451416, 0.0, 15.99482536315918, 15.002838134765625, 0.0, 0.0, 18.73002815246582, 4.970458030700684, 7.324294567108154], [0.0, 0.0, 13.600954055786133, 4.093467712402344, 0.0, 4.826993942260742, 9.770248413085938, 3.156514883041382, 12.911789894104004, 17.097694396972656, 10.243208885192871, 0.18905015289783478, 6.881564617156982, 0.0, 0.0, 0.0, 0.0, 30.649078369140625, 20.984834671020508, 12.050301551818848, 0.0, 15.024066925048828, 0.0, 4.055270671844482, 34.98785400390625, 14.521078109741211, 0.0, 3.2137272357940674, 0.0, 3.3760793209075928, 0.0, 1.9318807125091553, 0.0, 4.695318698883057, 4.5549726486206055, 0.0, 12.43921947479248, 0.10635210573673248, 0.0, 6.742682456970215, 0.0, 23.78492546081543, 0.0, 0.0, 5.250892162322998, 5.98539924621582, 1.1137093305587769, 0.0, 13.603057861328125, 3.1132564544677734, 17.7626953125, 0.0, 0.0, 6.41298770904541, 2.7896595001220703, 0.0, 0.0, 4.566796779632568, 9.145602226257324, 24.33074951171875, 0.0, 3.5681252479553223, 6.160118579864502, 16.994091033935547, 0.0], [0.0, 0.0, 8.889847755432129, 14.599815368652344, 9.163963317871094, 21.13825225830078, 3.52069091796875, 13.0557279586792, 13.063471794128418, 0.0, 11.636178016662598, 0.0, 3.3465893268585205, 6.684378623962402, 0.0, 0.0, 0.0, 21.43143653869629, 2.667733907699585, 0.0, 0.0, 0.0, 1.0729222297668457, 0.0, 11.843584060668945, 0.0, 0.0, 12.850687026977539, 0.0, 0.0, 0.0, 4.236412525177002, 15.184111595153809, 12.67113971710205, 0.0, 0.0, 17.442840576171875, 10.734414100646973, 5.821357727050781, 0.0, 9.166339874267578, 0.0, 2.021212577819824, 8.3653564453125, 3.8566606044769287, 15.200607299804688, 0.0, 0.0, 5.657320499420166, 14.51078987121582, 0.0, 1.9375535249710083, 0.0, 3.190124273300171, 0.0, 23.441022872924805, 6.855258941650391, 8.043901443481445, 0.0, 15.273200035095215, 7.36622428894043, 0.4450555741786957, 0.0, 0.0, 0.0], [4.885344982147217, 0.0, 9.06017017364502, 11.66372299194336, 0.0, 15.994415283203125, 27.181917190551758, 17.57403564453125, 0.003109602490440011, 0.0, 0.0, 15.176901817321777, 14.379399299621582, 0.0, 0.0, 0.0, 0.0, 31.539737701416016, 4.888746738433838, 8.604374885559082, 0.0, 0.0, 0.0, 0.0, 21.537282943725586, 0.8400091528892517, 0.0, 14.791668891906738, 7.259944915771484, 16.347185134887695, 6.597019672393799, 0.0, 1.9283478260040283, 10.203269004821777, 0.0, 0.0, 0.0, 3.0700583457946777, 0.0, 0.0, 0.0, 5.811620712280273, 15.755596160888672, 0.0, 0.0, 7.102152347564697, 0.1864306777715683, 0.0, 0.0, 13.057740211486816, 0.0, 0.0, 0.0, 8.166831016540527, 0.0, 0.0, 11.573602676391602, 16.90125846862793, 0.0, 23.377958297729492, 0.0, 2.6850831508636475, 0.0, 13.584280014038086, 0.0], [8.938477516174316, 5.78688383102417, 2.9774329662323, 0.0, 2.6632988452911377, 22.61107635498047, 0.0, 0.0, 4.170960903167725, 0.0, 0.0, 1.2565220594406128, 0.0, 0.0, 0.0, 12.61788558959961, 0.0, 13.834527969360352, 0.0, 0.0, 4.092703819274902, 0.0, 0.0, 2.7749717235565186, 3.732438087463379, 0.0, 0.0, 0.0, 4.819766044616699, 0.0, 0.2479037046432495, 6.507373809814453, 7.5200700759887695, 5.08988618850708, 0.0, 0.0, 0.0, 17.400562286376953, 0.0, 0.0, 10.506749153137207, 19.76557159423828, 10.234389305114746, 0.0, 0.0, 14.127336502075195, 0.0, 7.045300006866455, 24.331525802612305, 12.34101676940918, 4.622837543487549, 10.646567344665527, 0.0, 7.221879482269287, 0.0, 0.0, 7.202597618103027, 6.474781036376953, 0.0, 0.0, 8.162236213684082, 13.500020980834961, 0.0, 0.0, 7.088125705718994], [8.634317398071289, 5.407715797424316, 0.0, 0.0, 8.523046493530273, 0.0, 10.538579940795898, 4.670541763305664, 6.414116859436035, 3.4890880584716797, 0.6883991956710815, 10.786231994628906, 0.36919093132019043, 0.0, 0.0, 0.0, 4.447299480438232, 0.0, 0.0, 0.0, 0.0, 0.0, 3.8817665576934814, 0.0, 0.0, 0.0, 1.065139651298523, 3.1734225749969482, 0.0, 11.810687065124512, 24.103933334350586, 17.106555938720703, 10.873565673828125, 0.0, 8.686524391174316, 0.0, 0.0, 0.0, 0.0, 0.0, 10.166348457336426, 0.0, 0.0, 0.0, 0.0, 21.699556350708008, 7.462002277374268, 8.17747974395752, 7.282931804656982, 0.0, 7.752054214477539, 0.0, 0.0, 5.4856133460998535, 0.0, 0.0, 18.69322395324707, 4.429215431213379, 0.0, 0.0, 11.992433547973633, 0.0, 0.0, 0.00674403877928853, 11.95870590209961], [0.0, 3.943199396133423, 0.48281899094581604, 5.523468971252441, 0.0, 0.0013314821990206838, 5.0036396980285645, 0.0, 19.61202621459961, 8.666247367858887, 0.0, 19.2967529296875, 0.0, 0.0, 0.0, 0.0, 12.132416725158691, 4.566092014312744, 11.73883056640625, 0.0, 0.0, 0.0, 24.579233169555664, 0.0, 0.0, 0.0, 0.7614694237709045, 5.681921005249023, 3.9920833110809326, 0.0, 0.0, 18.785131454467773, 0.0, 3.437922239303589, 0.0, 10.575292587280273, 8.775084495544434, 9.020599365234375, 0.0, 0.0, 9.796945571899414, 18.531980514526367, 9.104860305786133, 3.9281904697418213, 0.0, 10.232858657836914, 0.0, 0.5595630407333374, 22.09681510925293, 14.200556755065918, 3.965473175048828, 14.822744369506836, 0.0, 0.0, 0.0, 0.0, 6.569973945617676, 4.101137638092041, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.658956050872803], [0.0, 0.0, 2.893733501434326, 16.606603622436523, 5.23116397857666, 23.867185592651367, 8.737690925598145, 14.885889053344727, 13.098142623901367, 0.0, 8.558024406433105, 0.0, 3.5712850093841553, 0.0, 0.0, 29.117874145507812, 9.623086929321289, 23.21961212158203, 0.0, 4.070765972137451, 0.0, 0.0, 0.0, 0.0, 18.129337310791016, 10.0557861328125, 5.341392993927002, 2.8171944618225098, 12.058115005493164, 0.0, 0.0, 0.5452508330345154, 0.0, 1.44412100315094, 0.0, 0.0, 6.426023483276367, 11.710298538208008, 5.755011558532715, 20.870851516723633, 13.196086883544922, 4.28672981262207, 14.190224647521973, 0.0, 27.039655685424805, 20.62999725341797, 1.535186529159546, 4.595964431762695, 2.433018207550049, 6.929224491119385, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.9125094413757324, 0.0, 14.692087173461914, 5.1476731300354, 2.256823778152466, 6.745388507843018, 0.0, 11.547347068786621], [8.156915664672852, 0.0, 0.0, 8.489431381225586, 14.047921180725098, 4.678881645202637, 0.0, 29.625701904296875, 22.7642765045166, 0.0, 5.047219753265381, 2.4973065853118896, 0.10714733600616455, 13.008232116699219, 2.099759340286255, 11.69640064239502, 0.0, 0.0, 19.090646743774414, 3.6001811027526855, 0.8919556736946106, 15.12647533416748, 0.0, 0.0, 0.0, 2.7408647537231445, 0.0, 3.6775224208831787, 0.0, 0.0, 0.0, 3.587270736694336, 0.24678689241409302, 0.0, 11.871536254882812, 0.0, 1.379153847694397, 3.7838525772094727, 1.8124313354492188, 0.0, 0.0, 21.19483757019043, 0.0, 11.419764518737793, 0.0, 8.897740364074707, 5.053954601287842, 0.0, 0.0, 0.4068281650543213, 0.0, 0.0, 1.9741573333740234, 11.607419967651367, 17.07466697692871, 0.0, 11.73971176147461, 8.049333572387695, 0.0, 11.356095314025879, 0.0, 0.0, 1.4536365270614624, 5.075296878814697, 11.64727783203125], [0.0, 0.0, 5.129516124725342, 22.649343490600586, 24.481645584106445, 13.815661430358887, 6.3442511558532715, 0.0, 0.0, 0.0, 2.467837333679199, 0.0, 7.836702346801758, 8.701454162597656, 0.0, 4.72934627532959, 1.3544687032699585, 18.506227493286133, 17.528305053710938, 0.0, 2.585340738296509, 0.0, 0.0, 0.0, 9.675971031188965, 1.7564443349838257, 21.454736709594727, 0.0, 0.0, 3.072727680206299, 0.0, 19.232208251953125, 0.0, 0.0, 12.696187973022461, 0.0, 1.2341219186782837, 9.40489673614502, 11.804386138916016, 23.31348419189453, 12.610713005065918, 8.415949821472168, 0.0, 0.0, 0.0, 18.97329330444336, 0.0, 0.0, 12.76551342010498, 4.237865924835205, 0.0, 4.870954513549805, 0.0, 13.34874439239502, 4.953420639038086, 0.0, 2.3489813804626465, 2.239805221557617, 0.0, 16.98512077331543, 9.230589866638184, 4.77257776260376, 0.0, 18.757585525512695, 0.0], [10.834914207458496, 0.0, 0.0, 0.0, 0.0, 0.0, 3.9275832176208496, 0.0, 6.336259841918945, 2.9499154090881348, 0.0, 21.2779598236084, 0.0, 0.0, 0.0, 0.0, 9.550079345703125, 8.552593231201172, 8.328481674194336, 26.317777633666992, 0.0, 4.622248649597168, 0.0, 0.0, 0.0, 11.684219360351562, 0.0, 0.0, 7.354422092437744, 0.0, 6.214780807495117, 12.741530418395996, 0.0, 12.588069915771484, 0.0, 2.2962918281555176, 0.0, 0.0, 16.295522689819336, 9.993324279785156, 4.977514266967773, 1.3578895330429077, 0.0, 0.0, 0.0, 17.76740074157715, 0.0, 1.3976597785949707, 7.873487949371338, 3.5216219425201416, 0.0, 7.74311637878418, 0.0, 6.0879998207092285, 20.75197982788086, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.383984088897705, 0.0, 0.0, 6.138342380523682], [15.5099458694458, 0.0, 0.0, 10.387157440185547, 23.202388763427734, 3.7387888431549072, 4.661347389221191, 0.0, 10.709291458129883, 7.765140056610107, 0.0, 5.5797319412231445, 0.0, 0.0, 0.0, 15.391439437866211, 0.0, 23.60372543334961, 0.0, 0.0, 2.4075992107391357, 0.0, 0.0, 0.0, 19.90049934387207, 13.192158699035645, 0.0, 0.0, 10.259686470031738, 14.687773704528809, 0.0, 12.95639419555664, 23.3023738861084, 0.0, 0.0, 0.0, 0.0, 5.389838695526123, 0.0, 5.543820858001709, 22.32851219177246, 0.0, 12.752673149108887, 0.0, 11.427913665771484, 17.734399795532227, 0.0, 0.0, 17.931489944458008, 0.0, 9.31575870513916, 0.0, 1.6014361381530762, 16.771873474121094, 15.024239540100098, 6.860930919647217, 0.4282355010509491, 2.591932773590088, 8.177005767822266, 3.436208963394165, 8.937743186950684, 15.915384292602539, 0.0, 0.0, 0.0], [0.0, 5.716425895690918, 0.0, 10.479752540588379, 0.0, 0.0, 1.609273910522461, 17.01311492919922, 14.11987018585205, 0.0, 15.256392478942871, 0.0, 8.350747108459473, 0.0, 0.0, 0.0, 0.0, 14.663505554199219, 0.0, 0.0, 0.0, 5.655884742736816, 0.0, 0.0, 0.0978504940867424, 14.854631423950195, 0.0, 0.0, 0.0, 0.6011865735054016, 14.354316711425781, 19.712276458740234, 7.38423490524292, 0.0, 17.576688766479492, 0.0, 33.93578338623047, 0.0, 0.0, 4.0307111740112305, 0.0, 16.36820411682129, 0.0, 28.66221809387207, 7.882450103759766, 15.115707397460938, 0.0, 15.181983947753906, 7.273604869842529, 3.5610783100128174, 15.807124137878418, 0.0, 0.04116995260119438, 9.954087257385254, 0.0, 4.8689093589782715, 0.0, 16.258358001708984, 0.5579438209533691, 25.78775405883789, 1.4239006042480469, 0.0, 6.979372024536133, 0.0, 35.24615478515625], [13.14199447631836, 10.72494888305664, 0.0, 15.545702934265137, 0.0, 0.0, 0.0, 4.335875511169434, 0.5358907580375671, 0.0, 0.0, 16.09061622619629, 2.160609483718872, 0.232035294175148, 1.4261282682418823, 0.0, 0.0, 21.129722595214844, 20.761211395263672, 1.439449429512024, 0.0, 10.985169410705566, 0.0, 10.669034957885742, 6.5442280769348145, 10.435954093933105, 6.416571617126465, 14.51124382019043, 18.93733024597168, 0.6927821636199951, 0.0, 11.629615783691406, 0.0, 4.017511367797852, 0.0, 0.0, 0.0, 1.4928799867630005, 0.0, 2.3383407592773438, 0.0, 3.2998781204223633, 7.46816349029541, 13.170754432678223, 11.010955810546875, 7.143276691436768, 0.0, 14.061702728271484, 9.208687782287598, 0.0, 0.0, 17.43492889404297, 6.151423931121826, 5.986724853515625, 5.904401779174805, 0.0, 0.0, 3.8737633228302, 0.0, 0.0, 0.0, 5.462471008300781, 2.4403023719787598, 0.23905673623085022, 0.0], [0.0, 0.0, 4.153062343597412, 0.0, 3.371269464492798, 5.263853073120117, 14.532963752746582, 0.0, 0.0, 0.0, 0.0, 3.3812344074249268, 1.679781436920166, 3.8579816818237305, 4.319050312042236, 0.0, 0.0, 20.39109992980957, 26.233346939086914, 0.0, 0.17144443094730377, 0.0, 0.0, 0.0, 17.99203109741211, 0.0, 8.990252494812012, 0.0, 4.9923834800720215, 0.0, 9.949357032775879, 0.0, 0.0, 0.0, 0.0, 0.0, 22.613834381103516, 7.514583587646484, 5.005416393280029, 17.605897903442383, 9.696247100830078, 7.357312202453613, 17.621152877807617, 0.0, 11.569345474243164, 1.3005549907684326, 0.0, 7.273815155029297, 3.4607112407684326, 15.302542686462402, 7.667205333709717, 15.890581130981445, 9.315712928771973, 28.988384246826172, 0.0, 0.0, 0.0, 11.321916580200195, 0.0, 37.047821044921875, 10.96989917755127, 5.653700828552246, 0.0, 0.0, 7.181581020355225]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.025238431990146637, -1.360439658164978, -0.13938191533088684, 2.5495245456695557, -0.035518914461135864, -0.7661659717559814, 0.9962978959083557, 0.288406640291214, -1.5731436014175415, -0.04872273653745651, 0.15916217863559723, 0.32777518033981323, -0.9418172240257263, -1.4158382415771484, -0.6987293362617493, 0.8343420028686523, -1.0693964958190918, -1.5729105472564697, 1.7227756977081299, 0.19075699150562286, -0.16273142397403717, -0.6845535635948181, 0.006309338845312595, 0.8848496079444885, 0.25177642703056335, -1.0956928730010986, 0.3317733407020569, -0.243084117770195, 2.069934129714966, -0.8921129703521729, 0.5905036926269531, -0.43192628026008606, -0.2077682614326477, 1.5919997692108154, 0.3842892348766327, 1.096774935722351, 0.15622659027576447, 0.37525373697280884, 1.3469949960708618, 2.2933056354522705, -0.18355123698711395, -0.9176009893417358, 0.02869381010532379, 0.9768891930580139, -0.2599581778049469, 1.3851757049560547, -0.4101031720638275, 0.07627930492162704, -0.45034313201904297, 2.192474603652954, 0.42702850699424744, -0.8428636193275452, -1.2642500400543213, -1.8621052503585815, -0.3902483880519867, 0.3149750828742981, 0.5779547095298767, -1.9835753440856934, -0.4082680940628052, -0.05754900723695755, 0.6930819153785706, 0.9147493839263916, -0.6527077555656433, -0.5852010846138, 0.21870660781860352, 0.04975193366408348, -0.029149392619729042, 0.5670362114906311, -1.7472196817398071, 0.34689420461654663, -0.5929839611053467, 0.4215821921825409, 0.2856946587562561, 1.3962547779083252, 0.44643229246139526, -0.2804291546344757, 0.5769556760787964, 0.3227791488170624, 0.6427629590034485, 2.2201600074768066, 0.617120623588562, 2.1743226051330566, 0.6073592901229858, -0.012748642824590206, -1.4541456699371338, 1.6414060592651367, -0.19055800139904022, -1.355961799621582, 1.0822093486785889, -0.7808677554130554, -1.3429651260375977, -0.37928885221481323, 1.5042474269866943, 1.3591337203979492, -1.6906906366348267, -1.2012425661087036, -0.8202974796295166, -1.492552638053894, -0.17617067694664001, -0.31945496797561646, -0.5323766469955444, 0.40438222885131836, -1.2815983295440674, -0.37134796380996704, -1.228061318397522, -1.5871502161026, -0.7153806686401367, 2.6415371894836426, 0.3387729227542877, 0.5139299035072327, -0.898438572883606, 2.0132739543914795, -1.3120614290237427, 1.237937569618225, -1.4541068077087402, 0.8433712124824524, 0.2785068452358246, 0.039334867149591446, 0.5585896372795105, -0.9533852934837341, 1.0068880319595337, 0.3016204833984375, -0.01981477253139019, -0.8655306696891785, 0.542512834072113, -1.4393199682235718, 0.7104336619377136, 0.36766383051872253, -0.2524307668209076, -0.0008254640852101147, -2.0694479942321777, 0.13141672313213348, 0.3346732556819916, 0.8583195805549622, 0.1777174323797226, 0.9727450013160706, -1.2302099466323853, 0.8386499285697937, 0.35194534063339233, -1.2352802753448486, -0.4914824664592743, -0.3717479705810547, 1.3104654550552368, -0.5116943120956421, 0.015405511483550072, 0.16593803465366364, 0.7532784938812256, -0.06927144527435303, -0.5401387810707092, 0.7236617803573608, 2.4027481079101562, -0.7327500581741333, 0.6278896331787109, 1.1503487825393677, 0.08886070549488068, 0.0689312294125557, 0.9977352023124695, -0.7667738199234009, 0.18919752538204193, 0.5693165063858032, -0.5122014880180359, 1.460096836090088, -0.7791997790336609, 0.2314807027578354, -0.5009071230888367, 0.2542722523212433, -1.9761021137237549, -0.26623719930648804, 0.6497364044189453, 0.3721742630004883, -0.5340364575386047, -1.6061992645263672, -0.2966950237751007, -1.0615363121032715, 0.055526990443468094, 1.689319372177124, 0.14520978927612305, -0.22380095720291138, -0.40670737624168396, 1.388833999633789, 1.4527829885482788, -0.9063109755516052, -0.485811710357666, 2.424490213394165, -1.4596270322799683, -0.10169064253568649, -1.2530529499053955, -0.36101898550987244, 1.0022125244140625, -0.5330556035041809, -0.2476702481508255, 0.3785788416862488, 1.7112971544265747, -1.5733015537261963, 0.22194112837314606, 0.4910101890563965, 1.3281877040863037, -2.606813430786133, -0.6302858591079712, 0.03189781308174133], [0.885981559753418, 0.8332473039627075, 0.5727887153625488, 2.1575262546539307, -0.22629643976688385, 2.03171968460083, 1.2318435907363892, -1.0861819982528687, -0.7105986475944519, 2.9113595485687256, -0.07100185751914978, -0.7274734973907471, 0.5013198256492615, -0.4131432771682739, -0.8166761994361877, -0.3808797001838684, -1.6257330179214478, 0.6904995441436768, -1.1191326379776, -0.3927161693572998, 0.6237028241157532, -0.6256237030029297, -1.2660757303237915, -0.10824821144342422, 0.4752029478549957, -0.06483448296785355, 1.308771014213562, -0.33612188696861267, 1.6612863540649414, 0.3060046434402466, 0.851146399974823, 0.3272576928138733, 0.5083940029144287, -2.1774492263793945, -0.40203362703323364, -2.1638641357421875, 1.0477333068847656, -1.4801868200302124, 0.28716161847114563, 0.1658257693052292, 0.5238640308380127, 0.8506746888160706, 0.1771673709154129, 1.1622813940048218, -1.2781386375427246, -1.167020320892334, -0.6011255383491516, 0.08606341481208801, -1.5440518856048584, -0.1606025993824005, 0.424822062253952, 1.0641469955444336, 0.25592201948165894, 0.20483222603797913, 0.43906980752944946, 0.035645369440317154, 1.707490086555481, -1.1855392456054688, 0.8042728304862976, -1.4794009923934937, -0.8752955198287964, -0.87587571144104, 0.30963608622550964, 1.2556676864624023, 1.613888144493103, -0.08731012791395187, 1.107177734375, -1.2326934337615967, -1.1477935314178467, 0.19638215005397797, -0.7424466609954834, -1.1648904085159302, -0.10205896943807602, -0.23261484503746033, 0.43617069721221924, 0.03372674062848091, 0.8420132994651794, -0.012417989782989025, -0.616932213306427, -1.2827059030532837, -0.9814592003822327, 0.007908794097602367, -1.0252200365066528, -2.2094192504882812, 1.2737654447555542, -0.05190829560160637, -0.4701325297355652, -0.4342386722564697, -1.8346362113952637, 0.29649943113327026, 0.08099792897701263, -1.8223129510879517, 0.2819608449935913, -0.9034425020217896, -2.7854437828063965, -0.22231197357177734, -0.2625698447227478, -0.015352330170571804, 1.1573622226715088, 1.9907619953155518, 1.818640112876892, -0.09012828767299652, 1.1007678508758545, -0.7554791569709778, -0.2514660954475403, 0.609616756439209, 0.414935439825058, 0.023115620017051697, -0.6278951168060303, 0.22565428912639618, 0.2761240005493164, -0.6018626093864441, 0.023663340136408806, -1.088500738143921, -1.2342205047607422, 0.6720802783966064, -1.1562083959579468, 0.9330840706825256, -1.2064905166625977, 1.3277512788772583, 0.292393296957016, 1.0083658695220947, 1.5767910480499268, 1.428914189338684, -1.4632622003555298, 0.5733927488327026, 0.9522556662559509, 0.5267859101295471, 1.9583408832550049, 1.414635181427002, 0.27679529786109924, -0.05313665047287941, -1.508491039276123, 0.7552184462547302, -0.4468116760253906, -1.1426821947097778, -0.19314292073249817, -1.4597461223602295, 2.5058443546295166, -0.27678510546684265, 0.3518706262111664, -1.3687520027160645, 0.6037032604217529, 1.0733435153961182, -1.1680306196212769, 1.0060046911239624, -1.029181957244873, 0.014844557270407677, -1.4195811748504639, -0.06817684322595596, -1.5264023542404175, 0.3016815185546875, -0.45363229513168335, 0.6999029517173767, 0.7318814396858215, -0.01666872948408127, -0.21282626688480377, -1.277820348739624, 0.42925822734832764, 1.5326777696609497, -0.20713455975055695, 0.5722222924232483, -1.7087626457214355, -0.7336273193359375, -0.6376134157180786, -0.1369515210390091, 0.022092103958129883, -1.5611754655838013, 1.2862423658370972, -0.5307862758636475, 2.0060718059539795, -0.2121383398771286, -0.6266475915908813, 1.6753489971160889, 0.6730570793151855, -0.11742828041315079, 0.430733323097229, 1.1548280715942383, 0.9859175086021423, -0.271901398897171, 0.547770082950592, -0.9972051382064819, 1.6882294416427612, 1.2790513038635254, 0.022018292918801308, 0.3843119442462921, 0.4215145409107208, -0.8644384741783142, 2.398927927017212, 0.8938584327697754, 0.1349899023771286, 1.8076903820037842, 0.1907660961151123, 0.48050418496131897, -0.3881710469722748, -0.46756309270858765, 0.5188650488853455, -0.21915273368358612, 0.6969136595726013, -1.318826675415039], [0.7120346426963806, -0.3430734872817993, -0.04597267508506775, -1.4321359395980835, -0.34316927194595337, 1.7092543840408325, -1.7860854864120483, -1.86650550365448, -0.5633783936500549, 0.5716531276702881, -0.1933593600988388, 0.42504817247390747, 0.5863088965415955, 0.18948248028755188, -0.5082364678382874, -0.1200251430273056, 0.023253070190548897, 1.1041499376296997, -0.40127891302108765, -0.3564559519290924, 0.09779462963342667, -0.7638740539550781, 2.434025526046753, -1.2713955640792847, -0.46533912420272827, 0.6876968741416931, -0.41101065278053284, -1.6845637559890747, 0.6214771270751953, 0.6899200677871704, 0.04751443862915039, 1.5176820755004883, 0.5124350190162659, -0.7926287651062012, -0.5757219791412354, -1.8875895738601685, -0.20338289439678192, 0.5202963948249817, 1.3038575649261475, -0.30127352476119995, -1.1653099060058594, 0.9350008368492126, 0.13564789295196533, 0.4027322232723236, -0.7770304679870605, 0.3509054481983185, 0.44300830364227295, 1.605810523033142, 0.19859381020069122, -0.12438683211803436, 0.8877213597297668, 0.597932755947113, 1.0185062885284424, 0.06703021377325058, 0.6414858102798462, -0.8037595152854919, -2.5418059825897217, -0.5715959668159485, 0.8505915999412537, 0.08805960416793823, -1.5761818885803223, 1.2720983028411865, 1.8668136596679688, -0.14929765462875366, 2.055516242980957, -0.5556661486625671, 1.3670676946640015, -1.5180742740631104, -2.087050676345825, -0.816994309425354, 0.48622044920921326, -0.14810416102409363, 1.122029423713684, 0.851740837097168, 1.2208260297775269, 0.542864978313446, 0.41153082251548767, -1.5312353372573853, 1.1938780546188354, 0.2963748276233673, -0.4279041588306427, -1.37880277633667, 1.5294322967529297, 1.1432722806930542, -0.6405683159828186, 1.0004549026489258, 1.0710561275482178, 0.5480204820632935, 1.7724546194076538, -1.522084355354309, -0.6404716968536377, 1.6375269889831543, 1.2567412853240967, -0.3700495660305023, 2.70015549659729, 0.6731770634651184, 1.4093669652938843, -0.6415572166442871, 1.194920301437378, -0.8456594944000244, -0.1349065899848938, 1.6012853384017944, 0.984489381313324, -1.4846071004867554, -0.37011954188346863, 0.7471247315406799, 0.2264620065689087, -0.550345778465271, 0.7633672952651978, 0.794241726398468, 0.23319284617900848, 0.26669687032699585, -2.976142168045044, 0.7594600319862366, -0.9677145481109619, -0.9557538032531738, -0.8437618017196655, 1.7362467050552368, -0.4019797742366791, 0.1455889791250229, 0.8480499386787415, 0.5067762136459351, -1.5326886177062988, 0.5470523238182068, 0.21668781340122223, 0.019837746396660805, -1.562986969947815, 1.0980347394943237, -1.236358642578125, -1.8520972728729248, -0.9723814129829407, 0.01826654002070427, 1.8927443027496338, -1.346765398979187, 0.2829146683216095, -0.5692530274391174, -0.739637017250061, -0.5383059978485107, -1.2312030792236328, -1.0841745138168335, 0.36931470036506653, -0.7559636831283569, -0.13714297115802765, -2.4603993892669678, -0.7614617943763733, 1.7708669900894165, 0.7420843243598938, 0.02669387497007847, 2.324185371398926, 0.7072463631629944, -1.1497547626495361, 1.541360855102539, 0.15344804525375366, 0.326583594083786, -0.22635141015052795, 0.7777580618858337, 1.7052030563354492, -0.5998749732971191, -1.3221242427825928, 0.14398615062236786, 0.5158758759498596, 0.26516950130462646, -0.21386590600013733, -0.684013843536377, 1.158064603805542, -0.7170793414115906, 0.12252682447433472, 1.4174377918243408, 2.266585111618042, 0.49213361740112305, 2.442002773284912, 1.4563196897506714, -1.6888483762741089, 0.3209362030029297, -0.041017796844244, -0.5302956104278564, -0.2613984942436218, -0.7882665991783142, -1.004074215888977, 1.0500487089157104, -0.61064213514328, 0.5565139651298523, -1.1359442472457886, -0.5304788947105408, 0.10025117546319962, 0.12246343493461609, 0.24301865696907043, 0.925392746925354, -0.7272665500640869, -1.2247427701950073, -0.1708262860774994, -0.9595757126808167, 0.23504236340522766, 1.5471984148025513, -0.5508679747581482, -0.22665733098983765, 0.4536665081977844, -1.165508508682251, -0.21321523189544678, 0.7002990245819092]], "dtype": "float32"}, {"__nd__": [[1.6086833477020264, 1.9212526082992554, -1.1604280471801758, -1.1427615880966187, 0.7787548303604126, -0.0021731939632445574, 0.1658935546875, 1.0016014575958252, 0.924046516418457, -0.1372116655111313, -0.2652357220649719, -0.08749137818813324, -0.233350932598114, 0.773235023021698, 1.08821702003479, -1.4995266199111938, -1.8700501918792725, 0.9556255340576172, -0.004215202759951353, 0.07552226632833481, -0.042571499943733215, -0.8843896985054016, -0.8625028729438782, -0.366554856300354, 1.8506453037261963, -0.06288151443004608, 0.44794315099716187, -0.7419226765632629, 0.30134326219558716, 1.668110966682434, 0.7574152946472168, 0.4610559344291687, -0.2585849165916443, -0.9726818203926086, -0.6189622282981873, 0.5545650720596313, 1.4471156597137451, 0.48770377039909363, -1.6729072332382202, -1.8573991060256958, 0.01052853837609291, -0.4681116044521332, 1.7680877447128296, 1.1659516096115112, -0.43303877115249634, -0.9721812009811401, -2.408795118331909, 0.1124330386519432, -0.8300583958625793, 0.25990772247314453, -0.850583553314209, 0.3536456525325775, -1.4757201671600342, 0.21479548513889313, 0.6108108758926392, -0.7496999502182007, -0.8730551600456238, -0.17843905091285706, -0.2754116654396057, 0.8089009523391724, 0.4856484532356262, 1.3108018636703491, 1.435401201248169, 0.1891636699438095, 0.8189316391944885, 0.7378788590431213, -0.52928626537323, 2.1576271057128906, -0.6133024096488953, -0.791641116142273, -0.7008727788925171, 0.29631948471069336, 0.23526796698570251, 0.8945233225822449, -0.4671141803264618, 1.182860255241394, 1.7817994356155396, -1.0614373683929443, 0.7504510879516602, 0.5014790892601013, -0.34750548005104065, 0.3405948579311371, -0.28412196040153503, -0.7773793935775757, 2.0835065841674805, 1.3113031387329102, 1.292378306388855, -0.24731309711933136, 1.3048279285430908, -0.5517585873603821, 0.05392445996403694, -1.0658928155899048, -1.4398399591445923, 0.5100630521774292, 0.43542128801345825, -1.2998645305633545, -0.15716217458248138, 0.03506721556186676, 1.7342294454574585, -1.7803078889846802, 1.0310475826263428, 1.05546236038208, 0.793929934501648, -1.3068726062774658, 1.4500558376312256, -0.5984058976173401, 0.3996071219444275, 0.4298895001411438, 0.850013256072998, -0.6010269522666931, 1.322322964668274, -1.3249905109405518, 0.02620726451277733, 0.40905898809432983, 0.9798924326896667, -1.0718635320663452, -1.3411134481430054, -1.9338364601135254, 0.7656363248825073, -1.1167778968811035, -0.9340167045593262, 0.5084075331687927, -1.0981460809707642, -0.09846091270446777, -0.36665746569633484, 0.07672358304262161, 2.314258098602295, 1.6122194528579712, 3.2240242958068848, -1.45670485496521, -0.2657683193683624, 0.3897431790828705, 0.17666667699813843, -0.681158721446991, -0.2095079869031906, -0.5376569628715515, -0.4056980609893799, -0.38385340571403503, -0.6713248491287231, -1.6855376958847046, -0.6983911395072937, 0.6602848768234253, -0.7452417612075806, -1.0533403158187866, 0.18100209534168243, -1.4278481006622314, 0.22398822009563446, 1.8330023288726807, 0.2664582431316376, -0.9291942119598389, -1.3344727754592896, 0.7436751127243042, 0.877109944820404, -0.397928923368454, -1.444869875907898, -1.2346551418304443, -0.7752485275268555, 2.8543503284454346, -0.5933991074562073, -0.7073542475700378, 1.1772873401641846, 1.4349256753921509, -1.5808261632919312, 1.0124062299728394, -0.8610033392906189, 0.13960038125514984, -1.0252753496170044, 1.6292859315872192, 0.30641433596611023, 0.9056716561317444, 0.38683947920799255, 1.0896011590957642, 0.35556697845458984, -0.5854403376579285, 1.5736589431762695, 2.092839479446411, 0.7232816219329834, -0.0069048041477799416, -1.7032432556152344, -0.38431552052497864, -1.8477519750595093, -0.835455060005188, 1.4682517051696777, -0.20640429854393005, -0.02807156927883625, -0.4636424481868744, -1.8568145036697388, 1.204032301902771, 1.1026479005813599, -1.3470656871795654, 0.15855777263641357, 0.8606600761413574, -2.1590681076049805, -1.3169734477996826, -2.3464949131011963, 0.4535408318042755, 0.4796040654182434, 1.1576071977615356, 0.26155537366867065, -1.255117416381836], [-1.0218123197555542, 1.7893078327178955, -0.30809706449508667, -1.1016089916229248, 0.4096851646900177, 0.2881597876548767, 0.14887294173240662, -0.038020677864551544, -1.4728584289550781, -0.05648733675479889, 0.8276897072792053, 0.712737500667572, -0.42991331219673157, -1.0593280792236328, -0.1299278885126114, -0.4276629090309143, 1.1564728021621704, -1.149490237236023, 0.3438809812068939, -1.5980547666549683, -1.3212155103683472, 0.5123422145843506, 0.39964014291763306, -0.3025488555431366, 0.8403865694999695, 0.7002208828926086, 0.20608581602573395, 0.9770810604095459, -0.6073038578033447, -1.82577645778656, -0.875329852104187, 1.0338917970657349, -1.6818982362747192, -0.1769295483827591, -0.3571142256259918, 1.8310236930847168, 1.248144507408142, -0.7960067391395569, -0.3965485095977783, -0.23890194296836853, 0.40102702379226685, -0.6235533356666565, 0.5202471613883972, -1.662300944328308, -0.9074188470840454, 0.6218237280845642, -2.4763803482055664, -2.5515353679656982, 0.4644871652126312, 0.05719991400837898, -0.028661120682954788, 0.3684740960597992, -0.8730634450912476, -0.4661593437194824, 1.1579911708831787, 0.6097193956375122, 1.2347965240478516, -1.0020010471343994, 0.39754465222358704, 1.7193107604980469, -0.9750208258628845, -0.34439846873283386, 0.7844658493995667, -0.39760878682136536, 0.3134937584400177, -0.8845159411430359, 0.16856294870376587, -1.3712164163589478, -2.1384263038635254, 0.691325306892395, -1.1144886016845703, -0.6492540836334229, 0.9218496680259705, -0.07321043312549591, -0.8702992796897888, 1.2618595361709595, -0.34527385234832764, 0.00772017752751708, 0.05566626787185669, -0.6702654361724854, -0.24027135968208313, 0.3757782578468323, -0.5238523483276367, 0.8607758283615112, 0.6065477728843689, 0.6416125893592834, 0.26987138390541077, -0.48602595925331116, -0.3663559854030609, 1.791311264038086, 0.5194519758224487, 0.08659782260656357, 0.6374673247337341, 0.22659087181091309, -0.9509881138801575, -0.17492541670799255, -0.1814013421535492, -0.7066459655761719, 1.4720337390899658, -1.96327543258667, 0.5299699306488037, -1.1626965999603271, 0.2345326542854309, 0.28062188625335693, -0.3631088137626648, 0.7587723731994629, -0.8914182782173157, -0.8324101567268372, -0.9651760458946228, 0.21375077962875366, -0.9023354649543762, 0.4170305132865906, -0.1569325476884842, 0.24814143776893616, 0.8256702423095703, 0.8686120510101318, -1.046534538269043, 0.6212689876556396, -0.999169111251831, -0.20593470335006714, -0.07590996474027634, -1.504838466644287, -0.5289708971977234, -0.9243137240409851, 0.7325423955917358, -0.9774731397628784, 0.32465335726737976, -1.633836269378662, -0.25191405415534973, 0.08522091060876846, -1.2597782611846924, -0.18795223534107208, -0.9572463035583496, 0.7495056986808777, 0.9055283069610596, 1.8128554821014404, -0.3312108516693115, 0.22025161981582642, -0.3340373933315277, -0.10393252968788147, 0.5890757441520691, -0.27475249767303467, 1.544327974319458, 0.5920146107673645, -0.271227091550827, 1.6724745035171509, -0.7396472692489624, 0.563584566116333, -0.5402878522872925, -0.22459512948989868, 0.22779330611228943, 2.2422165870666504, -0.4182080924510956, 0.7718889713287354, -0.7985478043556213, 0.29917824268341064, 0.8223832845687866, 0.5277311205863953, -0.5284940004348755, -1.8035920858383179, -2.349198579788208, 1.714625597000122, -0.39948585629463196, 0.491446852684021, -1.9508711099624634, -0.9753650426864624, 0.8671839237213135, -0.6343271136283875, 0.43605151772499084, -0.5829074382781982, -0.48769959807395935, -0.5831854939460754, 0.3777049481868744, 1.340015172958374, -0.08287935703992844, -1.4549474716186523, -0.7609409689903259, 0.6593601703643799, -0.3298896253108978, -0.5092597007751465, -1.6861183643341064, -1.998830795288086, -1.246056318283081, 0.2587583363056183, 0.4151318371295929, -0.7334540486335754, -0.1979021281003952, -1.0907059907913208, 1.102035403251648, -1.1817463636398315, 0.4109373688697815, -0.6422654390335083, 0.6334434747695923, 0.7340768575668335, -1.2295726537704468, -1.9635396003723145, -0.7633451819419861, 0.5920113921165466, -0.6581258177757263, 0.1489294469356537], [-0.8861358165740967, -1.8113919496536255, -1.0533339977264404, 0.45453694462776184, 0.8971174359321594, -0.2787685990333557, -0.4556724429130554, -0.4905245304107666, -0.5473738312721252, 1.4656071662902832, 0.14271554350852966, 1.2578866481781006, -0.5593692660331726, 0.41129663586616516, -0.36822977662086487, 0.4184614419937134, 1.22512686252594, 0.6366525888442993, -1.846124529838562, -0.8976517915725708, -0.5664171576499939, -0.017518579959869385, -2.148077964782715, 1.8228250741958618, -0.9627050161361694, -1.3606598377227783, 1.6057158708572388, 0.07895836979150772, -0.9301389455795288, 0.45022323727607727, 0.4425133168697357, -0.2185259908437729, 0.5038506388664246, 0.6635572910308838, 1.6589043140411377, -0.3398374319076538, 0.9196106195449829, 0.4146203398704529, -0.2717550992965698, 0.5892804861068726, 0.7120521068572998, 0.24663671851158142, 0.4557720422744751, 1.1588107347488403, 1.5374163389205933, -3.0448269844055176, 0.5035408735275269, -0.00808494258671999, 0.7967334389686584, -1.089642882347107, 0.15282635390758514, 0.7999117970466614, 1.2389286756515503, 0.5443529486656189, -1.8522562980651855, 0.23484958708286285, 0.5550982356071472, -1.562946081161499, -0.5967196822166443, 0.3857918679714203, 1.7104265689849854, 1.8303868770599365, 0.4821033477783203, -0.2640007734298706, -1.353133201599121, -1.18868887424469, 0.47952285408973694, -0.44976964592933655, -0.906348466873169, 0.06408581137657166, 0.4552782475948334, -0.7062271237373352, -0.39095041155815125, -0.19790273904800415, -0.7798565030097961, -1.0372366905212402, 2.0292611122131348, -0.9973171949386597, -1.041815161705017, -1.0852289199829102, 0.22967788577079773, 0.4394166171550751, 0.9545210599899292, -0.2776477038860321, -0.19176621735095978, -1.1628912687301636, -0.9449660778045654, 1.4280484914779663, -0.35206905007362366, -0.30605974793434143, 0.3621795177459717, 1.2215509414672852, 0.5522872805595398, -1.04334557056427, 0.8942391276359558, -1.192093014717102, 0.2635650932788849, -0.16173696517944336, -0.9469114542007446, -1.4517452716827393, -0.7686077356338501, 1.150350570678711, -1.2732222080230713, -0.25310319662094116, 0.3155234456062317, -0.4728511571884155, -0.4892995357513428, -1.7817808389663696, 0.11133518815040588, 1.5188093185424805, 1.6723151206970215, -0.9026568531990051, 0.6485424041748047, 0.9494026303291321, 0.2188553512096405, 1.0600131750106812, -0.0597548671066761, 0.5955281257629395, 0.17017598450183868, -0.19438758492469788, -0.43266353011131287, -0.571748673915863, -1.1809200048446655, -0.7588424682617188, -0.9845495223999023, -0.47126227617263794, 0.9977925419807434, -0.7604115009307861, 0.7944996953010559, 0.008722659200429916, -0.6231390237808228, 1.2594077587127686, 1.2104911804199219, -2.478593587875366, -0.33566537499427795, -0.2716483473777771, -1.542623519897461, -0.24868041276931763, 1.0499435663223267, 1.1720139980316162, -0.9554941654205322, 1.1681729555130005, -0.10710921883583069, 0.6558739542961121, -0.34271273016929626, -0.6793124079704285, -0.7796313166618347, -0.49953433871269226, -0.310017466545105, -1.368141770362854, 0.8914403319358826, 0.36978182196617126, -1.2744413614273071, 1.2312017679214478, -1.6281585693359375, 2.067676067352295, -1.2184232473373413, -0.175620436668396, -1.4521541595458984, -0.025485903024673462, 1.029317855834961, 0.6387655735015869, 1.5972551107406616, -0.6398416757583618, -1.539056420326233, -0.21722210943698883, -0.07373331487178802, -0.5437145233154297, 1.1059633493423462, -0.7100127339363098, -0.08925537765026093, 0.4213328957557678, 0.372425377368927, 0.11763517558574677, 1.5223296880722046, -0.08873133361339569, 0.39703884720802307, 1.3226333856582642, 0.09601655602455139, 1.6455849409103394, -0.32574501633644104, -0.7584512233734131, 0.6156116127967834, -0.003711429890245199, 1.4855883121490479, -0.26309698820114136, -0.48265141248703003, 1.0761762857437134, 1.6717114448547363, -1.58322274684906, 1.9021698236465454, -0.31215184926986694, -0.5776287317276001, 1.6940056085586548, 0.7797557711601257, 0.682060956954956, -0.34034934639930725, 0.7277601957321167, -0.3643094003200531, -0.4026317298412323], [-0.3494226634502411, -0.3571167588233948, -0.10827457159757614, -0.6739460229873657, 1.7831141948699951, -0.3254207670688629, 0.9508708715438843, 0.5676939487457275, -0.23561808466911316, 0.4011922776699066, -0.863110363483429, -0.12482154369354248, -0.3612586259841919, -0.6949905753135681, -1.0722239017486572, -0.48034417629241943, 0.576790988445282, 1.0917298793792725, 0.6552413702011108, 1.0819305181503296, 1.2411032915115356, -0.9557346701622009, -0.04291975125670433, 0.2988751232624054, 0.36269816756248474, -0.5509622097015381, 0.906957745552063, -0.2291288822889328, 0.7297534346580505, -1.736188530921936, -1.2250913381576538, 2.1915996074676514, -2.4762864112854004, -0.9117825627326965, 0.02563084289431572, 1.0606716871261597, -1.6535961627960205, 0.5682066679000854, -1.3339780569076538, -1.170379877090454, -0.7019837498664856, -0.028941094875335693, 0.2795763611793518, -0.8416568040847778, 0.4696589410305023, 0.8539183139801025, 1.9378727674484253, 0.4443984031677246, -0.08811287581920624, -0.24872617423534393, -0.13301628828048706, -0.33783796429634094, 0.7517496943473816, 0.033035583794116974, -0.7692557573318481, 1.833763837814331, 0.12120278924703598, -0.7745792865753174, -0.13154491782188416, 0.46771231293678284, -1.1849980354309082, 0.8431710600852966, -0.8396176099777222, -0.5580255389213562, -0.7682557106018066, 0.41878795623779297, -1.271700143814087, -0.9141175150871277, -2.1386537551879883, 0.48397159576416016, 0.30607154965400696, -1.7527295351028442, 0.3982335925102234, 0.20165272057056427, 0.06570012122392654, 0.19539226591587067, -0.3439277410507202, -0.7386521100997925, -0.36799004673957825, 0.8672525882720947, -0.262698233127594, 0.19873212277889252, 0.8643272519111633, -0.41636765003204346, 0.9404640793800354, 0.0303057748824358, -1.9986553192138672, 0.32637885212898254, -0.6893669962882996, 2.2243263721466064, 0.21532294154167175, 0.007834773510694504, -0.5193299651145935, 2.0166454315185547, 0.7903345823287964, 0.1179223582148552, 0.5981398224830627, -0.5244496464729309, 0.15060009062290192, -0.3612101376056671, 0.7297614216804504, -0.18272466957569122, 0.7153806686401367, 0.1901925802230835, -1.0143663883209229, -0.1498001217842102, 1.167288899421692, 0.4360393285751343, 0.8024657964706421, -1.4907110929489136, -0.16619648039340973, 2.3038084506988525, 1.247023105621338, 0.7431483268737793, 0.11313207447528839, 1.0613491535186768, 0.17027388513088226, 0.949924111366272, 1.1329141855239868, 0.5457238554954529, 1.923535704612732, -0.4555574953556061, 1.4926902055740356, 0.8199244141578674, 0.05245588347315788, -0.8914321660995483, 0.2239069640636444, 0.48992571234703064, -1.4602104425430298, 0.5194174647331238, -0.7766191363334656, -0.5728160738945007, 0.3645680844783783, 0.3945167064666748, -1.06125807762146, 1.7397783994674683, -0.8678891658782959, -0.7442497611045837, -1.7802447080612183, -1.4086817502975464, 0.7069446444511414, 0.01211099699139595, -0.7553892135620117, -0.5212967395782471, -1.3827248811721802, -1.5294643640518188, -0.3978566527366638, 0.5277986526489258, 1.1993268728256226, -0.10126250982284546, -0.12060455977916718, 0.48142966628074646, -1.7841891050338745, -1.2316683530807495, -0.11289800703525543, 0.36452582478523254, -1.167300820350647, 0.020498109981417656, -1.3023767471313477, -1.073513150215149, 0.06276822090148926, 1.0394055843353271, 1.6540418863296509, 0.07929357886314392, 0.886419951915741, 0.6265173554420471, 0.759479820728302, 0.6312975883483887, -0.40743526816368103, 0.8255751132965088, -0.37877553701400757, -2.125582218170166, -0.15775714814662933, 1.1843030452728271, -0.62119460105896, 0.5596099495887756, 0.5459465980529785, 1.6256860494613647, 0.3548099100589752, -1.7450878620147705, -1.1804438829421997, -0.5687986016273499, 0.5850796103477478, 0.20509599149227142, -0.9833623170852661, -0.16531120240688324, 0.5323781967163086, 0.005445851944386959, -0.970623791217804, -0.8080325722694397, 0.23394137620925903, -1.1207828521728516, -0.5223687291145325, -0.91454017162323, -0.511055052280426, -0.790951669216156, 0.2264559417963028, 0.6529604196548462, 0.8084143996238708, -0.651648223400116], [2.564833879470825, 0.7021024227142334, 0.38628050684928894, -1.1894290447235107, 0.17282278835773468, -1.8150277137756348, 0.32092154026031494, 0.13780751824378967, 1.392831802368164, 1.4834200143814087, -1.6465705633163452, 0.08152743428945541, 0.4107395112514496, 1.4634015560150146, 0.8046210408210754, -0.24267329275608063, 0.5383499264717102, -0.5078955292701721, -0.9913721680641174, 0.49277105927467346, -0.2180381864309311, -1.7466189861297607, -0.3344676196575165, 0.6706074476242065, 0.8645567893981934, 0.8537142276763916, -0.1086733266711235, 1.5391268730163574, -0.18163040280342102, 0.7474626898765564, 0.28700652718544006, 0.3699866533279419, -0.3758900463581085, -2.4306676387786865, 0.40802088379859924, 1.343721866607666, 0.6506784558296204, -0.6313824653625488, -0.32834121584892273, -0.28315454721450806, -0.25861018896102905, 1.4660789966583252, 0.09123941510915756, 2.0377821922302246, 0.33353644609451294, -0.5426188111305237, 1.2588549852371216, 1.586089015007019, 1.0616663694381714, -0.8670946359634399, 0.46537259221076965, 1.41974937915802, 2.083099365234375, -0.8726943731307983, 0.4495534300804138, 0.031410034745931625, -1.191676139831543, -0.42667967081069946, 0.13513419032096863, 1.424596905708313, -2.179835319519043, -0.6423241496086121, 1.3804378509521484, -0.32360556721687317, -2.2350261211395264, 0.4340926706790924, -0.14050361514091492, -0.14163225889205933, -0.24075725674629211, 1.3832135200500488, 0.8743966817855835, -0.42811471223831177, -0.622563362121582, 0.4118146300315857, 0.3348744809627533, 0.6707515716552734, 1.729212760925293, -0.9707502722740173, -1.935408353805542, 1.1774604320526123, 0.2104230672121048, 0.7590076923370361, 0.6132097244262695, -0.20957456529140472, 1.6533771753311157, -0.052791692316532135, 2.5005741119384766, 0.3101659417152405, 2.188117504119873, -0.41060107946395874, 0.35549259185791016, 1.9225842952728271, -1.6267943382263184, -2.080026388168335, 0.8208919167518616, -0.5540992617607117, 0.057119861245155334, -0.4745475947856903, 0.016271458938717842, 0.20530053973197937, -1.4245496988296509, -0.7004495859146118, 0.46911928057670593, 0.9209692478179932, -0.33373287320137024, -0.20144957304000854, -2.0421407222747803, -0.29030418395996094, -2.4993107318878174, -0.5757535696029663, -0.4226740300655365, 0.6260902881622314, 1.1496508121490479, 1.0775930881500244, -0.7122032642364502, -1.1606477499008179, 0.4871565103530884, -0.7183876037597656, 0.713203489780426, 1.423661708831787, 0.7873351573944092, -0.6317957639694214, -0.6079217195510864, -1.199907660484314, 0.7666050791740417, -1.624889612197876, -0.2684405744075775, -1.2180750370025635, 0.22597523033618927, -0.2386789470911026, 0.7681353092193604, 0.4277208149433136, -0.6329589486122131, -0.18720188736915588, -0.6685972809791565, 1.2409628629684448, 0.05672113224864006, 0.434247761964798, 1.2319493293762207, -0.09014300256967545, -0.6222325563430786, -0.5582644939422607, 0.07703980803489685, -1.0218818187713623, -0.02730899676680565, -1.2507673501968384, -1.0864709615707397, -1.3995901346206665, -0.3751074969768524, 1.6797986030578613, 0.6107761263847351, -2.25298810005188, -0.32550689578056335, 0.6468737721443176, -1.2111237049102783, 1.5742080211639404, -0.9605563282966614, 0.5651919841766357, 0.34850063920021057, -0.3207968771457672, 0.7266229391098022, 1.4743468761444092, 0.24907620251178741, 0.5749584436416626, -2.2385480403900146, -0.36856988072395325, 0.03857894986867905, 1.2954745292663574, -0.5903772711753845, 1.1162447929382324, 1.1396368741989136, -0.8445351719856262, -0.4991303086280823, 1.618964672088623, 2.450948476791382, 0.55535489320755, 0.28164809942245483, 1.433473825454712, 1.45362389087677, -0.07615718990564346, 0.5526416897773743, 1.4348108768463135, 0.013354417867958546, -2.0370724201202393, 0.21518689393997192, -0.47207945585250854, 0.6984548568725586, 1.1178404092788696, -1.1005909442901611, 1.5200430154800415, 0.598026692867279, 0.3831126093864441, -1.3913296461105347, -1.8851479291915894, 0.5242633819580078, 0.2702733874320984, -1.8089330196380615, -0.26137328147888184, -1.9033606052398682, 0.39992308616638184], [0.26765862107276917, -0.9894510507583618, -0.22094787657260895, 1.016103982925415, 0.4472908675670624, 0.2308063954114914, 1.30116605758667, 0.4362809956073761, -0.8594321608543396, 0.31617262959480286, -0.0383370965719223, 1.9837467670440674, -0.19017718732357025, 0.6437724232673645, 0.9959679841995239, -0.9129295945167542, 0.43020299077033997, -1.0518182516098022, 0.2737789750099182, 0.47143903374671936, 0.752651035785675, 1.4719665050506592, -0.1276644468307495, 0.7895816564559937, 0.5955208539962769, -0.0813644677400589, -1.1438134908676147, 1.2477116584777832, -0.6203933954238892, -0.5206884741783142, -0.5666234493255615, -1.189744234085083, 0.7344619631767273, -1.8033031225204468, -2.1799540519714355, 1.5322332382202148, -0.16251257061958313, -0.06041496619582176, -0.7824668288230896, -0.8529033064842224, 0.10179366171360016, 0.8007240295410156, -1.3766454458236694, 2.1895954608917236, -0.8743473887443542, -0.9383398294448853, 0.5758921504020691, 0.6025499701499939, 0.6713610887527466, 0.6525009274482727, -1.8160406351089478, 0.0005937818787060678, -0.04379934072494507, -1.3637582063674927, 1.5474311113357544, 0.12229809910058975, 0.9185243248939514, -0.5637457370758057, 0.6700819134712219, 0.14692118763923645, -1.0776368379592896, 0.7688950300216675, 0.7669368386268616, 0.3710581362247467, 0.06836692988872528, 1.805186152458191, -0.9575615525245667, -1.2749918699264526, -0.3616292476654053, -0.6746216416358948, 1.1553773880004883, -0.493327260017395, 0.2364533692598343, 0.0456809364259243, 1.8417214155197144, -0.8851675987243652, 0.3428887128829956, -0.657394528388977, -2.8625617027282715, 1.1105562448501587, -1.4472827911376953, 0.7549719214439392, -1.1795920133590698, 0.7176938652992249, -0.35314062237739563, -1.7352681159973145, -0.45189541578292847, -0.6940202116966248, -0.5069771409034729, -0.14634288847446442, 1.1579805612564087, 0.2273145467042923, 0.42225271463394165, -0.5630157589912415, -1.448482632637024, -0.22273662686347961, 1.051601767539978, -0.30836763978004456, -1.2728921175003052, 0.06552879512310028, 1.2451162338256836, -0.6520644426345825, -0.1612665057182312, -0.1911880224943161, -0.31056323647499084, 2.4959030151367188, 0.024702198803424835, -1.3327999114990234, -0.30573129653930664, 0.4514786899089813, 1.7174643278121948, 1.049059271812439, 1.0903993844985962, -0.8838202953338623, 1.4766948223114014, -1.785598635673523, -0.09045285731554031, -1.8832262754440308, -0.6780501008033752, -1.257805585861206, 0.06955374777317047, 0.9974350929260254, -0.710088312625885, 0.2329254299402237, 1.3883968591690063, -0.8561398386955261, 0.6148269772529602, 0.01797591522336006, -0.020383082330226898, -0.504145085811615, -0.0688471645116806, -0.5561946034431458, 1.491818904876709, -0.07171910256147385, 0.5802792906761169, -0.9638083577156067, 1.2097551822662354, -0.764607846736908, 0.4769197106361389, 0.7681393027305603, 1.3185186386108398, 1.263358235359192, -1.0420640707015991, -0.09027771651744843, -0.5813289284706116, -0.2131393700838089, -0.46496450901031494, -1.209574580192566, 0.3263518810272217, -0.6233649253845215, 0.1326281577348709, 0.37829744815826416, -0.3151426911354065, -0.012248432263731956, -0.09048616886138916, -1.063180923461914, 0.8164952397346497, -1.4229005575180054, -1.4849859476089478, 0.8822882771492004, -0.06194206699728966, -0.8764292597770691, -0.011036630719900131, -0.4933028221130371, 1.820249080657959, -0.016932565718889236, 0.2726638913154602, -0.32110846042633057, 0.21383485198020935, 0.7524491548538208, -1.3657593727111816, 0.8571702837944031, -0.6636713147163391, -2.362501382827759, -0.10034594684839249, -1.1846891641616821, -0.05386584252119064, 0.48555484414100647, -0.22454044222831726, 0.21380683779716492, 0.5040957927703857, -0.6564908623695374, 0.2184056043624878, 0.14870674908161163, -1.6376858949661255, -1.5718355178833008, 1.3348208665847778, -0.08459185063838959, 0.2262498438358307, -0.7769092917442322, -0.8192081451416016, 0.7340819835662842, -0.05966668203473091, 1.918732762336731, 1.8735926151275635, 0.2992960512638092, 1.173423409461975, -1.1080214977264404, -0.6086352467536926, -1.2704331874847412], [-0.3461049795150757, 1.1458097696304321, -1.8198931217193604, -0.1433398723602295, -1.9236390590667725, -0.30758416652679443, -1.1113277673721313, -1.0791195631027222, 0.47039392590522766, 0.30587711930274963, -1.5121418237686157, -1.0101076364517212, -0.45559099316596985, 1.5185682773590088, 0.33345088362693787, -1.3320516347885132, -1.2143124341964722, 0.3550586998462677, -0.642646312713623, 1.5336488485336304, 1.2172842025756836, 1.108069896697998, 1.1224149465560913, -1.3698582649230957, -1.8824840784072876, 1.067849040031433, 1.6817597150802612, 0.37988290190696716, 1.1973263025283813, 1.582176923751831, 0.2462630420923233, 0.43577319383621216, 1.5335259437561035, 0.778226375579834, -1.1811825037002563, 0.07888142019510269, 0.6183960437774658, 0.8855182528495789, -0.38810035586357117, -2.064659833908081, -1.475257158279419, 2.4107298851013184, 1.2098665237426758, 0.4061770737171173, -0.536464512348175, 1.3614591360092163, 0.37973976135253906, 1.1796196699142456, 0.1363363265991211, 1.1737029552459717, 0.6280949115753174, -1.2022725343704224, -0.9025251269340515, -0.397391140460968, 1.7048932313919067, 0.450919508934021, 0.5439799427986145, 0.46569132804870605, -0.9015263915061951, 0.49273619055747986, -0.050771474838256836, -0.29385054111480713, -0.8744572997093201, -0.12151554971933365, 0.7212425470352173, -0.9208996891975403, -0.9386200308799744, -0.7419849038124084, -0.7268466949462891, -1.2987456321716309, 1.7887784242630005, -1.286907434463501, -0.5948135852813721, 0.6207824945449829, 1.4129879474639893, -0.7801643013954163, -1.4271373748779297, -1.0760329961776733, 1.1500153541564941, 1.7082505226135254, -0.18473012745380402, 0.4216439127922058, 1.933641791343689, -1.1546846628189087, 0.880034863948822, -0.07066141813993454, 0.5021964311599731, -0.18624632060527802, -1.182018756866455, -1.5484713315963745, -0.8325330018997192, -0.028252216055989265, 0.5316193699836731, 1.3534908294677734, 0.11272257566452026, -0.8431145548820496, 1.7017788887023926, 0.251947820186615, 0.3693034052848816, -1.1246540546417236, 0.03611409664154053, -0.5201350450515747, 1.0352922677993774, -0.04381534829735756, -1.2520151138305664, -1.200868010520935, 0.9315851926803589, -0.20591917634010315, 1.017669677734375, -0.9206095933914185, -0.5063576102256775, 1.3357535600662231, -0.8277475833892822, -0.20447295904159546, -0.08555395901203156, -0.5774475932121277, -0.0029723632615059614, 0.7754716873168945, -0.9943976402282715, -1.0572943687438965, -1.2617131471633911, -0.6240372657775879, -0.7133688926696777, -0.7359583377838135, 0.8533086180686951, 1.6291872262954712, -0.5846474766731262, -0.8588542342185974, -1.1754872798919678, -0.048738691955804825, -0.891513466835022, 0.19375593960285187, 1.2562496662139893, -0.36143118143081665, -0.42878103256225586, -0.15713654458522797, -0.10559279471635818, 1.4170970916748047, 0.05449700728058815, 0.051906559616327286, -1.5591765642166138, -0.4909649193286896, -0.6599224805831909, -0.13811257481575012, -1.2946152687072754, -0.11722911894321442, 0.0509672649204731, -0.3445809483528137, 1.319954514503479, -0.6857975721359253, 1.754923701286316, 0.2906223237514496, -2.1812117099761963, -0.12607280910015106, 0.6944662928581238, -0.3701685667037964, 0.2748556137084961, -0.6034793257713318, -0.10820833593606949, -0.5276674628257751, -0.705426812171936, -1.29530930519104, -0.314688116312027, -0.882238507270813, 0.5089135766029358, -0.6380816698074341, 0.9918884634971619, -0.6540791392326355, -0.40762653946876526, 1.5148001909255981, 0.7261819243431091, -0.2709152400493622, -0.20193098485469818, -0.5287720561027527, -0.07596100121736526, 0.9670652747154236, -0.858445942401886, -0.5000946521759033, 1.683700680732727, -0.06425420939922333, 0.524439811706543, 1.0797334909439087, 0.7946501970291138, -0.40953800082206726, -0.3557073771953583, -0.19133643805980682, -0.26258960366249084, -0.7143270969390869, 0.07791701704263687, -1.9295097589492798, -1.6637682914733887, -0.9338099360466003, 0.7666003704071045, 1.3997902870178223, -1.1048035621643066, 0.4341731667518616, -1.5662120580673218, 0.22712716460227966, -1.469561219215393, 0.5615589618682861], [-0.8488346934318542, -0.02234059013426304, -1.3386917114257812, 0.9460660219192505, -0.9262266159057617, 1.7588716745376587, -0.09268695116043091, 0.2676324248313904, -1.0309948921203613, -0.5020724534988403, 1.7923367023468018, 0.7861068248748779, 0.7565597891807556, 0.06629011780023575, 0.4290938973426819, -0.3571264147758484, 0.12476707994937897, 1.0935516357421875, -0.1701522320508957, 0.9660054445266724, 0.866264283657074, -1.2003371715545654, -0.4919170141220093, -0.4792449176311493, 1.1799575090408325, -0.5633350014686584, 1.2511327266693115, 0.43065565824508667, -0.4999765455722809, -1.547649621963501, 0.9028269052505493, -0.822130024433136, 0.06863190978765488, 1.1181477308273315, 0.345319926738739, -0.2637709081172943, -0.09118713438510895, 0.5862453579902649, 0.26542431116104126, 0.7290846109390259, -1.3628264665603638, 0.3243720829486847, -1.7856143712997437, -0.2565874755382538, 0.08123660087585449, 2.403542995452881, 1.118926763534546, 0.7542662620544434, -0.21391747891902924, 0.5691147446632385, -0.9529028534889221, -1.2570693492889404, -1.8777673244476318, 1.3640114068984985, 0.13157974183559418, -0.14242559671401978, -0.8411431312561035, -0.7605977058410645, 0.8194396495819092, -1.2765573263168335, -1.1939679384231567, -0.37816449999809265, 0.01732828840613365, 1.0473885536193848, -0.8050350546836853, 1.4958361387252808, -0.004172623623162508, -0.028318272903561592, -1.0218006372451782, 0.46911218762397766, -0.8287022709846497, -0.3026773929595947, 0.2638246715068817, -1.0004059076309204, -1.2885898351669312, -0.6914809942245483, -0.6116517782211304, -0.272203654050827, -1.1303462982177734, 1.4527949094772339, 0.554685115814209, -0.8163192868232727, 0.3269244432449341, -2.708733558654785, -0.4244764745235443, 0.06901673972606659, 1.3259574174880981, 0.5041072368621826, 0.3692780137062073, 0.8720762729644775, -0.4202256500720978, 0.02058590203523636, -0.5915579795837402, -1.5142031908035278, 1.027330756187439, 1.22304105758667, -0.6017703413963318, 1.2009884119033813, -0.1969747096300125, 0.24483910202980042, -0.19147425889968872, 0.10380469262599945, 0.3226476013660431, -0.9404147863388062, 0.3961956799030304, 0.40917864441871643, -1.5840991735458374, 0.2661121189594269, -0.3716118335723877, -1.531622290611267, 1.4978570938110352, -0.17505528032779694, -2.3622875213623047, 0.0594734326004982, -1.5332103967666626, -2.3172762393951416, -0.7434071898460388, -0.9694796800613403, 0.24236804246902466, -0.46007615327835083, -2.287628412246704, 1.317320466041565, -0.7058114409446716, -0.17714650928974152, 1.46037757396698, 0.7770586013793945, -1.1282087564468384, 1.056731939315796, -0.046776894479990005, 0.5357797741889954, -0.21807807683944702, 0.2837097942829132, -0.5557640790939331, -0.16296258568763733, -0.4151344299316406, 0.018630588427186012, -1.5153000354766846, 0.6480426788330078, 1.6458187103271484, -0.42689523100852966, -1.830082893371582, -1.8590843677520752, -0.6056084036827087, 1.9944055080413818, -0.5442413687705994, 0.5142708420753479, 0.028917724266648293, 1.161596417427063, -0.6718412637710571, 1.8526155948638916, -0.2035169154405594, -0.5899820923805237, 0.48424360156059265, 1.1087121963500977, 0.9927148818969727, 0.11286763846874237, 1.2031259536743164, -0.19593770802021027, -1.0762325525283813, 0.5255125164985657, 0.872092068195343, -1.0634572505950928, -1.5585936307907104, -0.9128698110580444, 0.03142861649394035, 0.6724241375923157, 0.9242995381355286, -0.8665560483932495, -0.40083301067352295, 0.08997153490781784, 0.541360080242157, 1.1093617677688599, 0.785828709602356, 1.5739667415618896, 1.7144898176193237, 1.2054848670959473, 0.7600094079971313, -0.45480337738990784, 1.6743552684783936, 0.03363033011555672, -0.8745894432067871, -1.7849011421203613, 1.517337441444397, -0.8408986926078796, -0.8229363560676575, -0.9222224354743958, -1.0739688873291016, -0.42377129197120667, -0.9575070738792419, -1.0347379446029663, -0.04324471950531006, 0.9697749614715576, -0.4118216633796692, 0.4410700798034668, -0.21953976154327393, -0.6804576516151428, -0.8641474843025208, -2.318840742111206, 0.06957153230905533, -0.20010195672512054], [1.0996946096420288, -0.8291948437690735, 0.34981459379196167, -1.2887300252914429, 0.648872971534729, -0.9181575775146484, -0.3467284142971039, 0.33684298396110535, -0.3324723541736603, -1.3561354875564575, -0.6168169379234314, -0.1719103306531906, 0.1903030127286911, -0.32300564646720886, -1.3950426578521729, 2.2172787189483643, -0.9628725647926331, 0.18996255099773407, -0.38221514225006104, 0.1910216361284256, 0.6051210165023804, -0.23354679346084595, -0.025228288024663925, 0.1416625827550888, 0.8745619654655457, 0.12523525953292847, 0.3020099997520447, 0.19896306097507477, 0.877913236618042, -0.8605011701583862, -0.9310436248779297, 1.1049633026123047, -1.152840495109558, 0.014264929108321667, 0.2549213171005249, 0.805939257144928, -1.1543086767196655, -0.9869992733001709, -1.8590993881225586, -0.6078080534934998, 2.314344644546509, 1.3586713075637817, -1.2581191062927246, -1.3870750665664673, -0.07152120023965836, -1.2660396099090576, -0.4124436676502228, 0.3373410403728485, -1.4305964708328247, -0.21437454223632812, 0.19416949152946472, -0.03803626075387001, 0.7579007744789124, 1.1315518617630005, 1.6316839456558228, 0.018295492976903915, 0.2947483956813812, 1.0932259559631348, 2.014739751815796, -1.4823106527328491, -2.123497724533081, -0.15884238481521606, -0.25390657782554626, 1.0524640083312988, 0.6205006241798401, 1.3246283531188965, -1.1051034927368164, -1.0023761987686157, -0.8567284941673279, -2.270979642868042, 0.39600875973701477, 0.4284285008907318, 0.6016166806221008, 0.0039625088684260845, -1.3532955646514893, -0.6848618984222412, 1.942115306854248, -0.21322472393512726, -0.6732140779495239, -0.5003012418746948, 0.35176798701286316, 0.7999325394630432, 0.08753593266010284, 1.0862075090408325, -0.43116921186447144, 0.7488894462585449, 0.6965659260749817, -1.3536752462387085, -0.35554930567741394, -0.9855620861053467, 0.5463301539421082, 0.9846140146255493, 1.5205426216125488, 1.4062652587890625, 0.3819512128829956, -0.6515902876853943, -0.601649820804596, -2.51930832862854, -0.3834119439125061, -1.1542718410491943, -0.27741050720214844, 0.9279098510742188, -0.3310645818710327, -1.2184224128723145, 0.2907261252403259, -1.2994786500930786, 0.006162147969007492, -0.7245342135429382, -1.7220184803009033, 0.1767949014902115, 0.13735932111740112, -0.8517239689826965, 0.9056605696678162, -0.4363434612751007, 1.156494140625, 0.4204791188240051, -0.046513427048921585, -0.6204713582992554, -1.3341559171676636, -0.23194436728954315, 0.5492804050445557, -0.6850939393043518, -0.4362082779407501, 1.608862042427063, 2.0214500427246094, -1.4999384880065918, -0.4279329180717468, 1.2974704504013062, 1.5975226163864136, 0.7901045680046082, 0.1010301411151886, -0.18502531945705414, -0.20531712472438812, 1.4264248609542847, 1.5847041606903076, 1.369447112083435, -0.21628327667713165, 0.49798399209976196, -0.16050340235233307, -0.9396896362304688, 1.0624862909317017, 1.602577805519104, 0.43405747413635254, -0.31846505403518677, -0.04653928056359291, -0.6401187181472778, 0.4934806227684021, -1.39301335811615, -1.5985976457595825, -1.8031266927719116, 1.151080846786499, -0.7279723286628723, 0.1258702129125595, -0.26806244254112244, 1.4294319152832031, 0.8224327564239502, -0.25254619121551514, 0.4425087869167328, -0.3763048052787781, -0.42945846915245056, 1.140830397605896, -2.7152814865112305, 2.2010960578918457, 0.2984507977962494, -0.4441646337509155, -0.4469831585884094, 1.5303350687026978, 1.3235620260238647, 0.2795906662940979, -0.5258021354675293, 0.20086514949798584, 0.10063903778791428, -0.7759858965873718, 0.21148823201656342, -0.04004963859915733, -0.5117934942245483, -0.668210506439209, 0.15077921748161316, -0.03987343609333038, 0.22832033038139343, 0.26105189323425293, -0.7186533808708191, -0.6843944787979126, -1.760357141494751, 0.162482351064682, -1.433303952217102, -1.7819347381591797, 0.8777827620506287, 0.4962562322616577, -0.0705806314945221, 0.8656843304634094, -1.6189024448394775, -2.1101155281066895, 0.8986819386482239, 1.357883095741272, 2.205775737762451, -0.8570277690887451, 1.356183648109436, 1.2659975290298462, 0.16639457643032074], [0.884737491607666, -1.186130166053772, -0.05269039794802666, 0.3803006410598755, 0.9084920883178711, 0.9393558502197266, 0.3280847668647766, -0.6658441424369812, 1.0390874147415161, -0.04232250899076462, -0.3468034267425537, 0.5546606779098511, -0.4515206813812256, -0.37891122698783875, -0.7363810539245605, -0.28823795914649963, 0.9399053454399109, 0.1743038445711136, -0.5134719014167786, -1.6078922748565674, -1.1890000104904175, -1.3961282968521118, 0.5780193209648132, 0.28910839557647705, 1.0745848417282104, -0.11186366528272629, -0.40383216738700867, -1.8808223009109497, -2.1189706325531006, 1.4236249923706055, 0.13732343912124634, 0.4568057358264923, -1.2833905220031738, -0.4765734374523163, -0.34239643812179565, -0.1688269078731537, 0.5845651030540466, 1.2280473709106445, 0.6124618053436279, 0.8819164633750916, -0.01428072527050972, -0.7828026413917542, -0.1378306895494461, -0.9712815284729004, -0.32223737239837646, -0.23405997455120087, -0.21818169951438904, -0.1592927724123001, -0.13983823359012604, 0.22309136390686035, 0.5910197496414185, -1.6757978200912476, 0.28054556250572205, 0.944591224193573, -0.547252357006073, 0.6649683117866516, 0.9884494543075562, 4.209317207336426, 1.3352737426757812, 0.657765805721283, -1.555594801902771, 0.89058917760849, -1.4416651725769043, 0.5987546443939209, 0.43081337213516235, 0.5129380226135254, 1.0340354442596436, -1.611102819442749, 0.018487168475985527, -1.193405032157898, -0.6235948801040649, 0.7732588648796082, 0.1498268097639084, 0.4586421847343445, -0.9865579009056091, 1.554710865020752, 0.6691653728485107, 0.02973930723965168, -0.48287713527679443, 0.2469668686389923, 0.5991906523704529, -0.7496851086616516, 0.3676612079143524, -2.393928289413452, 1.237414002418518, 1.1982433795928955, -0.5739968419075012, -0.6213785409927368, -1.4906874895095825, -1.8500006198883057, 0.9685965776443481, -0.24138149619102478, -1.7449764013290405, 2.4730303287506104, -0.30039942264556885, 0.7071985006332397, 1.7669750452041626, 1.3178684711456299, 1.1767842769622803, -0.9034872055053711, -0.00805333536118269, -0.5292264223098755, -1.5418148040771484, -0.8245523571968079, -1.0319104194641113, -0.009095961228013039, 0.6251166462898254, -1.530779242515564, 2.009394407272339, -0.3424527645111084, -0.14079013466835022, 0.7255769968032837, 1.1857434511184692, -1.9124174118041992, 0.5992044806480408, -0.42978301644325256, -1.0375739336013794, -0.28909194469451904, -0.9912886023521423, -0.3773753046989441, -0.13719187676906586, -0.529248833656311, 1.4483822584152222, 0.6804251670837402, -0.41405007243156433, 0.044750142842531204, -0.3483445346355438, -0.21455514430999756, -0.39906197786331177, 0.6626558303833008, 0.43533334136009216, -0.8744640946388245, 0.46721190214157104, 0.4510402977466583, -1.5056874752044678, 1.2628153562545776, 0.16354142129421234, 1.9842970371246338, 0.6704503297805786, -1.9392433166503906, -0.420836865901947, -0.1406937837600708, 0.5362582206726074, 0.7512279152870178, -1.0005136728286743, -0.10429183393716812, 0.47033143043518066, 2.499209403991699, -2.2328503131866455, 0.5844539999961853, 1.5161702632904053, 1.730637788772583, -1.6382588148117065, 0.08719109743833542, 0.043725159019231796, -0.8403989672660828, 0.1879909187555313, 1.2473089694976807, -0.48170238733291626, 1.1845592260360718, -0.8187297582626343, 0.35378047823905945, -1.3841404914855957, -1.718517541885376, 2.493630886077881, -1.085800290107727, -1.6712578535079956, -0.6009604930877686, 0.17721110582351685, -0.8988989591598511, -1.0446960926055908, -0.21411660313606262, -0.359854131937027, 0.061175111681222916, 0.6779763698577881, 0.3577674329280853, 1.1525763273239136, 0.39333343505859375, 0.2755405902862549, -0.33037078380584717, 1.1888821125030518, -0.08281824737787247, 1.2498658895492554, -0.18665386736392975, -0.38861116766929626, -0.10455106943845749, 0.10287047922611237, 0.7134900093078613, 0.0763137936592102, -1.400262475013733, 1.0852570533752441, 0.5460083484649658, -0.9749855399131775, 1.2514926195144653, -0.9097166061401367, -0.5907580256462097, 1.9353917837142944, 0.6038768887519836, -0.14741237461566925, -0.7208372950553894], [0.06604255735874176, 1.5467947721481323, -1.2098374366760254, -0.9008775353431702, -0.5634005665779114, -1.5996042490005493, -1.03367280960083, 0.8590768575668335, 0.5291171669960022, -0.15983088314533234, 1.1244516372680664, -1.1760056018829346, -1.6869375705718994, 1.0960078239440918, 0.9822444915771484, 0.6576104760169983, 0.7385881543159485, -0.1582217961549759, -0.2383287250995636, 1.1288342475891113, -0.9857954978942871, -0.1414082795381546, -0.45428580045700073, 0.07985645532608032, -0.6378713250160217, 0.05038892850279808, 0.05618450790643692, -1.4191052913665771, 0.9062826037406921, -0.9792148470878601, -0.7965199947357178, -0.9435125589370728, -1.6735405921936035, -1.086405634880066, 1.9184869527816772, -0.7510956525802612, 0.0693068653345108, -0.610401451587677, -1.520027995109558, 2.2957704067230225, 1.5274442434310913, 0.4007812440395355, 0.7401823997497559, -1.5611642599105835, -0.6963387727737427, -0.8061781525611877, 0.9374504089355469, -1.1238837242126465, -1.0774016380310059, -0.24377362430095673, 0.04829650744795799, 0.8575100302696228, -0.1595716029405594, -0.8315937519073486, 1.6386263370513916, -1.4775551557540894, -0.2788299024105072, 0.6018656492233276, 0.8404253125190735, 0.45387986302375793, 1.261732578277588, -0.04631137102842331, -0.9966469407081604, 0.38316842913627625, -0.10883994400501251, 1.5422215461730957, -0.31862035393714905, -0.7508927583694458, -0.3253338038921356, -1.265049934387207, 0.8823931813240051, 1.6394060850143433, 0.2269953340291977, 0.7617118954658508, 0.7122297883033752, 0.7561116814613342, -0.26184824109077454, -0.482616662979126, 0.4916720688343048, -0.9905638694763184, 0.6328541040420532, 0.08758632093667984, 1.3763870000839233, 0.006790898274630308, 0.16496127843856812, 1.5381015539169312, 1.4721968173980713, 0.7942183017730713, -0.1571824550628662, -0.14978083968162537, 1.1965241432189941, 0.794552206993103, 0.48559918999671936, -1.153501033782959, -0.36434099078178406, 0.7071549892425537, 1.1135550737380981, -1.3242851495742798, -1.351637601852417, -0.15564315021038055, -0.49910297989845276, 2.008495569229126, -0.8697699904441833, -1.5622903108596802, -2.444455623626709, -0.7386932373046875, 0.5072863101959229, -0.05749434232711792, -0.06143023818731308, -0.1881498098373413, 0.34428277611732483, 0.3318271040916443, 2.3798580169677734, 0.6403017640113831, -0.023084502667188644, -2.518824815750122, 0.9760633707046509, -0.5918208360671997, -0.7626360654830933, -2.5496175289154053, 1.2084532976150513, 0.3199146091938019, -1.1520484685897827, 0.30419838428497314, 1.9470722675323486, 0.2619936466217041, -1.1955673694610596, 0.22711999714374542, -0.24150148034095764, -0.22703740000724792, -1.2167857885360718, 1.1583889722824097, 0.975937008857727, -0.5223679542541504, 0.397003710269928, 0.6850863695144653, -0.5523619651794434, -1.0440056324005127, -0.2551358938217163, -0.15923385322093964, -1.8078691959381104, 0.5502212047576904, 0.14717698097229004, -0.3023335635662079, -1.5956629514694214, 0.35084858536720276, 1.074017882347107, -0.8846672773361206, -0.5509583353996277, 0.6151874661445618, -0.3821987807750702, -0.693204402923584, -0.16105002164840698, -0.5048933625221252, 0.29308372735977173, 0.0245876032859087, 1.3783503770828247, 0.023705899715423584, -0.8255921602249146, -1.4095765352249146, 0.4514494836330414, -0.03793610632419586, 2.8667545318603516, 0.3398696482181549, 0.6621074676513672, 0.10266133397817612, -0.9707541465759277, 0.4619458019733429, 0.09534072130918503, -0.331450492143631, 0.21696990728378296, 0.05230671539902687, 0.4437568783760071, -0.34103062748908997, 0.36884987354278564, 1.912187099456787, -0.7229107618331909, 1.4302341938018799, 1.5238854885101318, 1.0990891456604004, 0.2305614948272705, -1.4941949844360352, 0.9185251593589783, -0.6887162923812866, -0.30232909321784973, -1.2113516330718994, -1.3097862005233765, 0.7487271428108215, 0.6224167346954346, -0.00575694115832448, 1.1045899391174316, 0.96654212474823, -1.3051252365112305, -1.6284338235855103, -2.6544878482818604, 0.16580936312675476, 0.7290443778038025, 0.7571847438812256, 0.7451446056365967, 0.32159125804901123], [-0.9670941829681396, 1.0627851486206055, 0.6641024947166443, 0.12412586063146591, -1.2585810422897339, -0.4801582992076874, -0.7358929514884949, -0.5853656530380249, 1.3016388416290283, -1.6908886432647705, -1.2190325260162354, -0.9721841812133789, 0.23926101624965668, 0.29110947251319885, 0.48554158210754395, -0.009483846835792065, 1.0785666704177856, 0.5153272151947021, 1.293211579322815, 0.8044077157974243, -0.043556682765483856, -0.2204175889492035, 1.0420618057250977, -0.7625018954277039, 0.6066253185272217, 0.5870547890663147, -2.2391085624694824, 0.6782195568084717, -0.3509734869003296, -0.6679380536079407, -0.6858057975769043, 0.7568003535270691, 0.29039716720581055, 0.022074680775403976, -0.22681628167629242, 0.07347725331783295, 0.6019521951675415, 0.0018018074333667755, 0.9617011547088623, -0.7451319098472595, 0.18360795080661774, 0.3000686466693878, -0.4573408365249634, -0.10488574951887131, -0.11536934971809387, 0.6246301531791687, -0.1961122304201126, 0.3508676290512085, 1.0250369310379028, 1.029308557510376, 0.019261635839939117, -0.9685360789299011, -0.8246254324913025, 0.4541257321834564, -1.2442047595977783, 0.11548197269439697, 0.6571923494338989, -0.7961336970329285, -0.5588277578353882, 0.2893211543560028, -0.27429571747779846, 0.3466933071613312, -0.6171853542327881, 1.0562965869903564, 0.454626202583313, 1.50184166431427, -0.3520125150680542, -0.6104169487953186, -1.385568618774414, -1.1655151844024658, 2.3239245414733887, -0.5229161381721497, -0.892547607421875, -1.0270264148712158, 1.184714674949646, 0.05699347332119942, 0.23369204998016357, -0.14719258248806, -0.3491099774837494, 0.040755536407232285, -1.9045857191085815, 0.9903572201728821, 1.3647128343582153, 1.6792515516281128, -0.15155799686908722, -0.13109427690505981, 0.08147841691970825, 1.3599227666854858, -0.5412306189537048, 0.27245354652404785, 0.4923543334007263, -1.0039249658584595, 2.171593189239502, 1.8893405199050903, -0.3637562096118927, -0.4832051694393158, 0.41970497369766235, -0.272469699382782, 1.1371498107910156, -0.9416727423667908, 0.707209587097168, 0.39004069566726685, 0.28629010915756226, -0.5892787575721741, 2.1663928031921387, -0.5765566825866699, 1.7862237691879272, 0.8345603942871094, 0.14367011189460754, 1.440409541130066, -0.6978548169136047, 0.4016570448875427, -0.324076384305954, 0.16892920434474945, -0.41709592938423157, -0.37246832251548767, -0.5875970125198364, 1.3182320594787598, 0.11440758407115936, -2.4475784301757812, -0.4080072343349457, 0.447813481092453, -2.0989463329315186, -0.7219104766845703, 0.055615976452827454, -0.5375044941902161, 1.213324785232544, 0.6859819889068604, 0.26976919174194336, -0.48731672763824463, 0.8598219156265259, -0.06983830034732819, 0.012538646347820759, -0.3123216927051544, -0.07363760471343994, -1.340248942375183, -0.7568758726119995, -0.4688918888568878, -1.8871080875396729, 0.13290663063526154, 1.2768057584762573, 0.5629048943519592, -0.33929067850112915, -1.1215887069702148, -1.2622569799423218, 0.40064066648483276, 1.0838979482650757, -0.8319450616836548, -0.6260061860084534, -0.7801974415779114, 0.3906511068344116, -0.32145988941192627, 0.8174497485160828, -0.783829391002655, -1.3663690090179443, -0.3579927682876587, 0.7359108924865723, 0.15975910425186157, -0.9412473440170288, 1.1504380702972412, -0.6975374221801758, -0.03441977500915527, -0.08113888651132584, 0.16580010950565338, -0.3857254385948181, -0.2907485067844391, -1.073103666305542, 1.986466407775879, 0.6159245371818542, 0.3307037949562073, -1.4442635774612427, -0.052454374730587006, -0.060766588896512985, -0.4628690183162689, -1.3842161893844604, -0.8990169763565063, -0.8642470836639404, 1.5234534740447998, 0.5184473991394043, 0.3865254819393158, -1.3537627458572388, -2.229779005050659, 0.8368567824363708, -1.9319051504135132, -1.2095681428909302, 1.1852220296859741, -0.16971270740032196, -1.4533181190490723, -0.8803071975708008, 0.9965195655822754, 1.294379472732544, -0.1293945461511612, -0.10947952419519424, 0.5500403046607971, 0.5905609130859375, 0.9330928921699524, -0.1468028873205185, -0.905185878276825, 1.4532710313796997, 0.25919875502586365], [-0.24255995452404022, -0.5386641025543213, 2.169260025024414, 0.6078184247016907, -0.8374767899513245, -0.5365028381347656, 0.8326071500778198, 0.6609046459197998, -0.00667785108089447, -0.12944820523262024, -2.1710641384124756, 0.7825199961662292, -0.5186936855316162, 1.5923336744308472, 1.300753116607666, 0.9262965321540833, -0.002778220223262906, 1.8628523349761963, 1.2861535549163818, 1.7505981922149658, -0.28196918964385986, 0.5858222246170044, -0.34797781705856323, -0.42540478706359863, 0.8371655344963074, 0.8177614212036133, 1.4050589799880981, 1.275254487991333, 1.330686330795288, -0.24244241416454315, 1.5822927951812744, -0.28487589955329895, 0.4912674129009247, 0.6307513117790222, -0.40841490030288696, 0.136128231883049, 1.2082366943359375, 1.1503546237945557, -0.658488392829895, 0.52983558177948, -0.37666407227516174, -2.261012077331543, -0.619701623916626, 0.7164246439933777, -0.9800047278404236, 0.62696772813797, -1.7262704372406006, 0.45618799328804016, -0.1714712530374527, 0.3760335147380829, 0.7775607109069824, 0.19891120493412018, -1.2882543802261353, -0.5028709173202515, 0.13210488855838776, 1.0710368156433105, -0.793872058391571, 0.6945046782493591, -0.8955508470535278, 0.11618024110794067, -0.15210683643817902, -0.7828951478004456, -0.33747968077659607, 0.5339815616607666, -0.4418734312057495, -1.3948734998703003, 0.49666211009025574, 2.513073205947876, -0.36535000801086426, 0.8138474822044373, 0.5286906361579895, 1.319783091545105, 0.5013681650161743, -0.9471124410629272, -1.1947643756866455, 2.433542490005493, -0.6987720131874084, -1.7020596265792847, 0.29789867997169495, 0.5015817880630493, -0.18503861129283905, -0.021319828927516937, 0.061405546963214874, -0.02372141368687153, 1.500356674194336, -1.9027997255325317, -0.5669063329696655, -0.42600467801094055, 1.533429503440857, 1.0191552639007568, 0.5622027516365051, 0.6622251868247986, 0.3609355688095093, -0.26227259635925293, 0.2099044770002365, 0.6801188588142395, -0.23946432769298553, -1.0788226127624512, -1.6043397188186646, -0.41640782356262207, 0.9259517192840576, 1.128788709640503, 0.2652812898159027, -0.1307825744152069, 1.6530274152755737, 0.14067812263965607, 0.6233183145523071, -0.36497414112091064, -0.5778665542602539, -0.6151710748672485, 2.0914061069488525, 0.13189788162708282, 0.2542766034603119, 0.2650579810142517, 0.7317510843276978, 1.3270407915115356, -1.1051008701324463, 0.7094761729240417, 1.714100956916809, 0.31476983428001404, 1.3391976356506348, -0.19647854566574097, -0.3330996036529541, -1.0849632024765015, -0.09380926936864853, 0.7673014402389526, -1.5651123523712158, -0.057620611041784286, -1.3689970970153809, 0.047526970505714417, -0.8600488901138306, -1.9772534370422363, 1.1777317523956299, -0.28402262926101685, -1.4373657703399658, -0.010454528965055943, 0.45735323429107666, -0.7712762355804443, 0.35132932662963867, -1.959567904472351, 1.1764098405838013, 0.6771297454833984, 0.27951115369796753, -1.2217885255813599, 1.058488368988037, 0.3089902102947235, 0.2822093367576599, -1.1318131685256958, -0.026082579046487808, -0.3944039046764374, 1.6066795587539673, 0.7314803004264832, 2.5570552349090576, 0.4446871876716614, -1.15936279296875, -0.3347015976905823, -2.1507537364959717, 0.8383564949035645, 1.1403932571411133, -0.015981607139110565, -0.4660731554031372, -1.514068365097046, -0.7371473908424377, 0.14309000968933105, -0.5544801354408264, 1.1870226860046387, 0.5228816270828247, -1.1510695219039917, -0.569266676902771, -0.8611390590667725, 0.6884640455245972, 0.20804910361766815, 0.9698262214660645, 0.7041831612586975, 0.7810696959495544, -0.6274409890174866, 1.3352644443511963, -0.49626976251602173, -0.4180988669395447, -2.4215950965881348, 1.118087887763977, 1.9731146097183228, -1.2785972356796265, 1.3194979429244995, 0.44238460063934326, -0.3373349905014038, -0.822601854801178, -2.187929153442383, 0.7824116349220276, -1.0522404909133911, -0.7866500616073608, -1.0135669708251953, -2.042844533920288, -0.15860538184642792, 1.0063481330871582, 1.6824641227722168, -0.5583868026733398, 0.7082309722900391, -1.5697784423828125, -0.052738405764102936], [-0.6052083373069763, -0.7108457684516907, -2.13179087638855, 1.1131482124328613, 0.48586541414260864, -0.6561989188194275, 0.4534146189689636, -1.2299531698226929, 0.07057162374258041, -0.25349345803260803, -1.005972981452942, -0.3915660083293915, -0.6480148434638977, 0.2303711473941803, 1.4408440589904785, 0.8665626645088196, -0.7306966781616211, -0.22189807891845703, -0.9695664048194885, 0.1144377589225769, 1.296732783317566, -0.8982036709785461, 0.07338669151067734, -0.36720457673072815, -0.806225061416626, 1.124462604522705, 0.3009604811668396, -0.4858357608318329, 1.4049776792526245, 0.7275574207305908, 0.37691184878349304, 0.7172619104385376, 3.0057036876678467, -1.5907059907913208, -1.0230743885040283, -0.7906032800674438, -0.7492615580558777, -0.6038610339164734, 0.5001242756843567, 0.856759786605835, -1.5905495882034302, 0.35579195618629456, 0.27086833119392395, 0.8793184757232666, 0.5835235118865967, 0.5095442533493042, -0.40151166915893555, -1.0154775381088257, 1.324026346206665, 0.8295316696166992, 0.5639753341674805, -0.12223171442747116, 0.2466103583574295, 1.3940768241882324, -0.29644742608070374, -0.21687020361423492, 0.575847327709198, 0.38652223348617554, -1.3002630472183228, 0.24708980321884155, 0.6432976126670837, -0.008031688630580902, -0.6657097935676575, 0.79386967420578, 0.15092867612838745, 2.2109951972961426, -0.1321016252040863, -0.15729095041751862, 1.5796347856521606, -1.1830874681472778, 0.36445727944374084, 2.065613269805908, 0.6811348795890808, 0.9247543811798096, -1.0340096950531006, 0.605190634727478, 1.1722642183303833, 1.7152411937713623, 0.006450360640883446, 0.8439985513687134, 0.08146156370639801, 1.353564977645874, -1.0370079278945923, -1.2252219915390015, -1.7579562664031982, 0.04966991767287254, -0.282418817281723, 0.7844822406768799, 0.33247652649879456, -1.3520314693450928, -1.1243603229522705, 0.44391724467277527, 0.6420623064041138, -0.4108663499355316, -0.80567467212677, -1.2914597988128662, 0.47546905279159546, 1.3829257488250732, 0.25466084480285645, -0.7867746353149414, -0.5401168465614319, -0.5388861894607544, -0.15084215998649597, -0.06847134232521057, -0.19345501065254211, -1.2730504274368286, 1.5042457580566406, -0.5942690372467041, -1.1987457275390625, -0.9992589950561523, -1.250667691230774, 0.40451210737228394, -0.06918808072805405, 0.41677987575531006, 0.6731181740760803, 1.0772358179092407, -0.87260901927948, 1.1325736045837402, 1.9457435607910156, 1.0937767028808594, 0.9385428428649902, -2.3824191093444824, 1.5035014152526855, 0.7026513814926147, -1.487450361251831, -0.12311393767595291, -0.3015609085559845, 0.014615223743021488, 0.9450079202651978, 0.5750282406806946, -0.4813831150531769, -1.1655828952789307, -0.05067615211009979, 1.12209153175354, -0.1433584988117218, -0.5203302502632141, -0.07260924577713013, -1.6242520809173584, -1.4692047834396362, -0.820915162563324, -0.503700315952301, -0.5035338997840881, 1.6772534847259521, -0.14123308658599854, -0.12926939129829407, -1.443998098373413, -0.36941036581993103, 0.8058152198791504, 0.45926031470298767, -1.7048343420028687, -0.40698373317718506, 0.3359912633895874, 0.8063629269599915, 1.211333155632019, 0.43927618861198425, -0.2952343225479126, -0.08624038100242615, 0.5762971043586731, -0.9599677324295044, 0.39492279291152954, -1.165992259979248, 0.2725472152233124, -0.037867773324251175, -0.5620715022087097, 1.0398222208023071, 1.0413914918899536, 1.257761836051941, 0.582118809223175, -0.3234151601791382, -0.14893418550491333, -0.3577946424484253, -0.2716251015663147, -0.3013850450515747, -0.5722442865371704, -1.67231023311615, 1.8496180772781372, 1.1222928762435913, -0.5844428539276123, 1.754417896270752, -0.025753458961844444, 3.610024929046631, -1.4101002216339111, -0.7222813963890076, -1.11603844165802, 1.3943965435028076, -1.2002067565917969, 1.5441522598266602, -0.19490690529346466, -1.5148663520812988, -0.2946476638317108, 0.42803633213043213, -0.4456785023212433, 0.1881464719772339, 0.803666889667511, 1.5360722541809082, -0.16633713245391846, 0.19850105047225952, -0.485971063375473, 0.07334180921316147, -0.99454265832901], [-0.9605661034584045, 0.5240731835365295, 0.08845657110214233, -0.5015486478805542, 0.4451830983161926, -0.31252068281173706, 0.4764578640460968, -2.133824348449707, -0.03267945721745491, 0.6964405179023743, 0.20875531435012817, 0.527846097946167, -1.0276873111724854, -1.0038979053497314, 0.7976827025413513, -0.19112348556518555, 0.5507709980010986, 0.5642619132995605, 1.1759616136550903, -0.6535171270370483, 0.5187190175056458, -0.9712457060813904, -1.7835547924041748, 0.35988059639930725, 1.4645342826843262, -0.9168621897697449, -1.214616060256958, -0.6772111058235168, -0.603542149066925, 1.0155595541000366, 0.37510958313941956, -1.4022952318191528, 0.6630632281303406, -0.6620073914527893, 1.938594102859497, -0.44123566150665283, 0.525854766368866, -2.7153732776641846, 0.5967075228691101, -0.05151509493589401, -0.44854846596717834, 0.2897852063179016, -0.552832305431366, 0.6313450932502747, 0.22816653549671173, -0.5595311522483826, -0.6423516869544983, -1.0901556015014648, -0.18098460137844086, 1.1204607486724854, 0.43813204765319824, 0.2895369529724121, 1.1783971786499023, 0.8993094563484192, -0.8147985935211182, 0.07054541260004044, 0.7358099222183228, -0.4061424732208252, 1.3648425340652466, 0.7831767797470093, -1.114827036857605, 0.486924409866333, 0.15660478174686432, 0.1060468927025795, -0.11444441229104996, -0.3813914656639099, 0.8639322519302368, -0.053379859775304794, 0.01798187755048275, -0.1177346259355545, 1.0355628728866577, 0.5541284680366516, -0.838816225528717, -1.7586146593093872, 1.1948063373565674, 0.08308132737874985, 1.105243444442749, -0.8229261040687561, -0.8591539263725281, 1.3690810203552246, -2.11458158493042, 0.3951933979988098, 0.09098253399133682, -1.9412883520126343, -0.4441477656364441, -0.021595405414700508, -0.12610627710819244, 1.1145986318588257, -0.0756896585226059, -1.5090386867523193, 1.0992136001586914, 0.2996026873588562, -0.6563534140586853, -0.76605623960495, 0.5923573970794678, -0.7638166546821594, 0.38037511706352234, 0.031315162777900696, -0.0776878073811531, -2.239102602005005, -0.5885683298110962, -1.2199041843414307, -1.920400619506836, 0.13843202590942383, -0.10828517377376556, 0.5646029114723206, 1.546940565109253, -0.13517238199710846, -0.8552164435386658, -0.8134464025497437, 0.2551935613155365, 0.40969032049179077, -0.2225407361984253, -2.1954689025878906, 1.1661115884780884, -0.0001030008279485628, -0.3507426977157593, 1.8493613004684448, 2.0437114238739014, -1.0076682567596436, 0.8111328482627869, 0.12118171155452728, 1.8296856880187988, -0.002715950831770897, 1.0566071271896362, 0.017251063138246536, 0.17219051718711853, -0.6488975286483765, -1.1392768621444702, -0.7270707488059998, 0.5654975771903992, 2.9176204204559326, 0.24642598628997803, 0.27442607283592224, 0.26138609647750854, -0.7607227563858032, -0.04440034553408623, 0.381411612033844, -0.2793503701686859, -0.21278445422649384, -1.2785718441009521, -0.7139455676078796, 0.9294466376304626, 0.13567401468753815, 1.3360507488250732, 0.5101274251937866, -0.7180814146995544, -1.2764936685562134, 1.8408467769622803, 0.8072469830513, 1.196520447731018, 0.4486805498600006, 0.2789458930492401, -0.9516139626502991, -1.4570800065994263, -0.3489823341369629, 2.0571448802948, -0.6574787497520447, 0.34686484932899475, -0.7307339310646057, 1.8785384893417358, 0.24834579229354858, 0.8379452228546143, -1.1981306076049805, 0.5124634504318237, 0.6433457136154175, -2.54974627494812, 0.6779617667198181, -0.16285178065299988, -0.8530735969543457, 2.581470251083374, -0.21674410998821259, -0.05234324559569359, 1.0965036153793335, -0.7559225559234619, -0.475974977016449, 0.5744047164916992, 0.4332706034183502, -0.6088111400604248, 1.4733095169067383, -1.7527053356170654, -0.4920935332775116, 1.1372884511947632, 1.427384376525879, 0.3542332351207733, 1.9364650249481201, 1.1415849924087524, 0.9722251296043396, -0.5398744344711304, -0.25219887495040894, 0.9324363470077515, 0.2939913272857666, 0.44963863492012024, 0.7117273211479187, 0.16245831549167633, 0.8632776141166687, 0.9626014232635498, -1.418092966079712, 1.2244259119033813, -0.5700680613517761], [0.7869926691055298, 0.15003198385238647, -0.5504595041275024, -0.2401646077632904, -1.3288698196411133, -1.3421257734298706, -0.09743023663759232, 0.26355525851249695, -0.09677989780902863, -0.4070855677127838, -0.45490357279777527, 0.48363015055656433, -0.8779377937316895, 1.645294427871704, 1.2876222133636475, 0.36752215027809143, 0.9924126267433167, -1.530846357345581, 0.39933398365974426, -0.7298768758773804, -1.494642734527588, -0.8074378371238708, -0.40917879343032837, 0.6691702604293823, 1.0747572183609009, -0.6641457676887512, -2.6080970764160156, 0.28966471552848816, 0.009549622423946857, -0.2978202998638153, 0.06834021210670471, 1.0516719818115234, -1.4132190942764282, 0.9207255244255066, 1.0153262615203857, 0.34493717551231384, 0.6407290697097778, 0.5603775978088379, -0.19702589511871338, 0.9293678402900696, -1.0424147844314575, -0.07391105592250824, -0.8988506197929382, -1.8576048612594604, 0.7298209071159363, 0.5461689829826355, -1.9498257637023926, -0.8102638721466064, -0.1571802943944931, 0.23061494529247284, -0.5190660357475281, -0.13761892914772034, 1.1296379566192627, 0.9749240875244141, 0.7358945608139038, 1.2690049409866333, -1.2591067552566528, -0.36748602986335754, -0.11522567272186279, -0.4143659174442291, 0.14213509857654572, -0.8693730235099792, -0.6029306054115295, -1.8669812679290771, 0.6381579041481018, 1.5564119815826416, 0.9309463500976562, -0.40007320046424866, -0.9501249194145203, -1.810361623764038, -1.2940173149108887, -0.4752027690410614, 1.1160080432891846, 1.2276458740234375, -1.1182231903076172, -0.21887530386447906, -0.08849245309829712, 0.24848221242427826, 0.3365873396396637, 0.7470182180404663, 0.07535070925951004, 1.5296169519424438, 0.7881621718406677, -0.2016042172908783, 2.7840332984924316, -0.430042564868927, -0.4970940351486206, -0.38524648547172546, -0.6280331611633301, 1.7673790454864502, -0.3625902533531189, 2.101306915283203, 0.9628850817680359, -1.4578914642333984, -2.5650522708892822, -1.2760915756225586, -1.412458062171936, 0.028959695249795914, -0.8511284589767456, -0.6090150475502014, -0.560149073600769, 0.2792062759399414, -0.9118202924728394, 0.5669366717338562, -0.4734030067920685, -0.592509925365448, -2.4856677055358887, 1.2103384733200073, -0.5067279934883118, -0.5457946062088013, 0.008133209310472012, -1.2632288932800293, 0.4321040213108063, -1.188353419303894, 0.11805686354637146, -0.48780182003974915, 0.029176097363233566, 0.5555542707443237, -1.1615009307861328, 0.8151099681854248, -1.0131971836090088, 0.849647045135498, -0.8549684286117554, 1.7624467611312866, -1.740179181098938, 0.9038717150688171, -1.0832661390304565, 0.24043935537338257, 0.6557618379592896, 1.9983835220336914, -2.564101219177246, -0.2929403781890869, 1.6669076681137085, 0.2873459756374359, -0.2637022137641907, 0.6587662100791931, 0.6619744300842285, -0.9237797260284424, 1.1084909439086914, 1.1181223392486572, 0.6823044419288635, -0.07863915711641312, 0.8809585571289062, -1.0646837949752808, 0.5775729417800903, 0.6481120586395264, -0.10803066939115524, -0.12354673445224762, -0.3420986533164978, 0.7798940539360046, 0.2938205897808075, -1.3591084480285645, 0.2504270374774933, -0.6293145418167114, 0.4943363070487976, -0.8482339978218079, 1.1327767372131348, -0.7757143974304199, 1.1933314800262451, -1.7465060949325562, -0.1621546596288681, 1.0850768089294434, -0.6866129040718079, -0.6608933806419373, 0.9116196036338806, 0.872780442237854, 1.0959497690200806, -0.15877054631710052, -1.7246333360671997, 0.17002901434898376, 3.165743112564087, 1.7825052738189697, -1.758736252784729, -0.5820451974868774, 0.29411253333091736, 1.0962169170379639, -0.6196848750114441, 0.019507519900798798, -0.18234975636005402, -0.28465956449508667, -0.866409182548523, -0.40933817625045776, 0.7460178732872009, 0.5346974730491638, 1.6487780809402466, -2.41813325881958, -0.3964264690876007, -0.8154042363166809, 0.15987233817577362, -0.5211031436920166, 1.2030681371688843, -2.282257318496704, 0.478934109210968, 0.3767729103565216, 0.4782271683216095, 0.7415373921394348, -0.015270729549229145, 0.25481003522872925, -0.37402620911598206, 0.4049999713897705], [-0.7919161319732666, -0.8292028903961182, 0.13691715896129608, -0.5771495699882507, 1.616182804107666, 1.2267377376556396, 0.49073708057403564, 1.5710140466690063, 0.2349158525466919, 0.3276464641094208, -1.580566167831421, 0.6564281582832336, -0.3204070031642914, -0.750900149345398, 0.5158808827400208, -0.5979701280593872, 0.43327391147613525, 0.5248029232025146, 1.0529412031173706, 0.07411077618598938, -0.7803138494491577, -0.4602242112159729, -0.21774053573608398, 0.10738088935613632, -0.12276274710893631, 0.7427884936332703, 0.03712960332632065, 1.207113265991211, 0.9685524106025696, -0.8898424506187439, -0.488730788230896, 0.21708650887012482, -0.4039423167705536, -0.984093427658081, 0.8573931455612183, -1.276045799255371, -0.7112903594970703, -0.3838688135147095, -0.9394077062606812, 0.14682264626026154, -0.009061304852366447, -0.7554379105567932, 0.09072015434503555, -1.2295773029327393, 1.0908915996551514, -0.21437974274158478, 0.05926067754626274, 0.3754824995994568, -0.9537288546562195, -1.1164757013320923, 0.5629749894142151, 0.38992220163345337, 0.4235099256038666, 0.08466836810112, 1.157396674156189, 0.7520942091941833, 1.6599117517471313, 0.5212143063545227, 0.024662816897034645, 0.7508958578109741, -3.1383919715881348, -1.0653536319732666, -0.5705615878105164, 0.6351096034049988, 0.9789206981658936, 0.30595359206199646, 0.5304391980171204, 0.8950827121734619, -0.12334958463907242, 0.05432405695319176, 1.7110151052474976, -0.08476655185222626, -0.6802476048469543, 0.02613231912255287, -0.016931379213929176, 0.4575141966342926, 0.44068947434425354, 0.2166450470685959, -1.7992298603057861, 0.8787657618522644, 0.9011538624763489, -1.1862856149673462, -0.48405569791793823, -0.1743796467781067, -2.2475433349609375, 0.8856386542320251, -0.20729006826877594, -0.303745299577713, -0.8037617206573486, 0.2656545639038086, -0.9786501526832581, -0.5615862011909485, 0.6414163708686829, -2.1900763511657715, -0.5917196273803711, -0.9182512760162354, 0.003177106147632003, 0.5480406880378723, -0.20419742166996002, -0.6342803835868835, -0.1823161095380783, 0.6077626943588257, -0.828949511051178, 1.3877187967300415, -0.8053148984909058, 0.2848268747329712, -1.8147581815719604, 1.3950834274291992, -0.733241856098175, 0.34880414605140686, -1.0797587633132935, -0.5951525568962097, 0.5814647078514099, -0.6766529083251953, -0.4118376672267914, -0.6356735825538635, 0.7121178507804871, -1.100651741027832, -0.2610510289669037, -0.5605215430259705, 0.14368779957294464, -0.8084789514541626, -1.1049491167068481, -0.9685887098312378, 0.07027189433574677, -0.45330914855003357, -1.4822547435760498, 1.5594185590744019, -1.905956506729126, 2.9921586513519287, 1.0591180324554443, -0.8559560775756836, 0.5173128247261047, 0.23512643575668335, 0.24330177903175354, 0.7187471389770508, -1.1894521713256836, 0.4757905602455139, 2.8801498413085938, 0.48183971643447876, -0.3970617651939392, -2.598414182662964, -0.34122568368911743, -1.7934972047805786, -1.102096676826477, 0.34754133224487305, -1.0242388248443604, 0.9575483202934265, 0.7784159183502197, 0.27137860655784607, -0.1120804026722908, -2.418795108795166, -0.1549898087978363, -0.5208984613418579, 0.6288676857948303, 1.939110279083252, 0.5930875539779663, 0.6736175417900085, 1.9013832807540894, 0.2091090977191925, 0.2292405366897583, -0.1779308319091797, 0.19169455766677856, 0.21499064564704895, -1.4818965196609497, -0.9130879640579224, -0.08188404142856598, -0.7079758048057556, 0.45875149965286255, 0.6429480314254761, 0.04422727972269058, 0.6169723868370056, 0.6672922372817993, -1.2212272882461548, 1.1870684623718262, 1.4806373119354248, 0.4620954394340515, -1.7197271585464478, -0.7926604151725769, 1.2997578382492065, 0.10850156098604202, 0.5168185830116272, 2.692516803741455, -0.5064525604248047, -2.042888641357422, 0.6290927529335022, -0.23206695914268494, -0.24486638605594635, -0.6798669695854187, -1.034874439239502, 0.23618334531784058, 0.21211177110671997, -0.051515232771635056, 0.13201001286506653, 0.19865567982196808, 0.09587085992097855, -0.6728425621986389, -0.848392903804779, -0.3729201853275299, 1.5571503639221191], [-0.38595348596572876, -0.3353543281555176, -1.5215224027633667, -1.1225621700286865, 1.6703184843063354, -1.0154759883880615, 1.1224455833435059, 1.3707252740859985, -0.9935302138328552, 0.2029937207698822, -1.1089904308319092, -0.0048375059850513935, 0.18883204460144043, 0.1592990905046463, -1.0394045114517212, 0.09154783189296722, -0.8192437887191772, -1.4825609922409058, -0.7941066026687622, 1.0242854356765747, -0.5780321955680847, 1.5412274599075317, 0.5324105024337769, 0.8116850852966309, -0.1594022810459137, -1.3560394048690796, -1.328839659690857, 0.3126258850097656, -0.09106422960758209, 1.6020022630691528, -0.37795504927635193, -0.4511260390281677, 0.53703773021698, 1.2971898317337036, -1.1755759716033936, 0.2599118649959564, 0.5825715661048889, -1.2536896467208862, 0.5831860899925232, 2.3387813568115234, 0.07920152693986893, 0.38682666420936584, -2.466549873352051, -2.268954038619995, 1.0695397853851318, -0.5387378334999084, 0.2059279829263687, 0.39220744371414185, -0.39288145303726196, 0.9364476203918457, 0.49394822120666504, 1.2147845029830933, 1.0955843925476074, -0.59381103515625, 0.03195986896753311, 0.0017363053048029542, -0.21358059346675873, -0.8418402075767517, -2.3246190547943115, -0.666399359703064, -1.1372126340866089, -0.7539330124855042, 1.266984462738037, 0.19104911386966705, 0.2500495910644531, 0.004092586226761341, 0.0038064743857830763, 0.49012795090675354, -1.6663278341293335, -0.37678849697113037, 1.8892676830291748, -0.7922005653381348, 1.016818642616272, -1.6247894763946533, -0.8647310733795166, -0.3631460666656494, 0.07925986498594284, -0.282135009765625, 0.6151134967803955, 0.41687506437301636, -0.4042961597442627, 0.3180956244468689, -1.1037274599075317, -0.009650669060647488, 0.9108879566192627, -0.8466917872428894, 0.11539249867200851, 0.38412657380104065, -0.47332972288131714, -0.505237877368927, 1.3762534856796265, -0.7807179689407349, 0.4802667796611786, -0.7738640904426575, 1.0988839864730835, 0.5620604753494263, -0.530145525932312, 0.8653494715690613, 0.7299988269805908, -0.09560377150774002, 0.25966355204582214, -0.653775155544281, -0.21862304210662842, 0.354341059923172, 0.2868753671646118, -0.3564654588699341, 1.6967430114746094, -1.0303738117218018, 0.05474844202399254, -1.3767670392990112, -0.735843300819397, -0.18764717876911163, 1.0151244401931763, -1.6966793537139893, -2.9760403633117676, 1.176426887512207, -0.7136681079864502, 0.4871865212917328, 0.5527119040489197, -0.8010905385017395, -0.6738070249557495, -1.2026488780975342, 0.9984928965568542, -0.7083967924118042, -0.3370901048183441, -0.6306346654891968, -0.14248773455619812, -1.0271817445755005, 0.11275308579206467, -0.3158228397369385, -0.4553513526916504, -0.7297019958496094, -0.647083044052124, 0.015096809715032578, -0.8106323480606079, 0.4273313283920288, -0.6827616691589355, -0.9743491411209106, -0.07619599252939224, 1.807850956916809, 0.46091362833976746, 2.796262264251709, 0.18001896142959595, -0.29557958245277405, -0.23513780534267426, 1.191072940826416, -0.0637575164437294, -1.0650025606155396, 0.21543438732624054, 1.3745616674423218, -1.4381564855575562, 1.2185708284378052, -0.9552629590034485, -2.0286922454833984, -0.7017473578453064, 1.2696166038513184, 1.4252488613128662, -0.6335329413414001, -0.4623553454875946, 0.5108128786087036, 0.2088436335325241, 1.3107057809829712, -0.42784610390663147, 0.3891681432723999, 0.5401840209960938, 0.7758826017379761, -1.0910732746124268, 0.34389883279800415, 0.24671408534049988, -1.5533024072647095, 0.7776539921760559, -0.9263426661491394, 0.7928926348686218, 2.3660969734191895, 1.0625730752944946, -1.7001547813415527, -1.0811936855316162, 2.2604541778564453, 0.5108473896980286, -0.3358307182788849, 0.7814280390739441, -0.1029236763715744, 0.20766976475715637, 0.7595387101173401, -0.3990591764450073, 0.2697044014930725, -1.3570014238357544, -0.7504868507385254, 0.9096065163612366, 0.0930393636226654, 0.5023688077926636, 0.3163563013076782, -0.7609046697616577, -0.5629782676696777, 0.05867486819624901, -0.046630214899778366, 0.49988895654678345, -0.3257448673248291, 0.09271108359098434, -1.5740994215011597], [1.4373135566711426, 1.519400715827942, 0.7645366191864014, -1.032849907875061, 1.4482131004333496, 0.05931764096021652, 2.3114991188049316, -0.5058387517929077, 0.9269833564758301, -2.030221939086914, 0.2000935673713684, -0.23654820024967194, -0.6977336406707764, -0.42755383253097534, -1.0365530252456665, -0.9532713890075684, 1.2509032487869263, 0.33579376339912415, 1.5955476760864258, 0.007985450327396393, -0.04208969324827194, 0.7168369293212891, 0.5314880013465881, -0.624103844165802, -0.08287670463323593, 0.19850963354110718, 0.6561801433563232, 1.8369139432907104, -0.8099502921104431, -0.32559722661972046, -0.778641402721405, 0.2511615753173828, 1.019914150238037, -1.1265629529953003, 1.7688016891479492, 1.0434526205062866, -0.027397828176617622, -1.4436529874801636, -1.3624542951583862, -0.7877295017242432, 0.8335636258125305, -1.1925028562545776, -0.419556587934494, -0.8794618844985962, -0.041023850440979004, -0.4433331787586212, -0.8744415640830994, 0.16614803671836853, 1.7965701818466187, 0.346810907125473, 0.8120803236961365, 0.2283884435892105, 0.021794913336634636, 1.4688284397125244, -0.1479814648628235, 0.5615612268447876, 0.49593034386634827, -0.021141622215509415, 2.213662624359131, -0.44130998849868774, 0.5054342150688171, -0.0055043441243469715, -1.553152322769165, -0.7835796475410461, 0.42382511496543884, 0.018182091414928436, 1.4585357904434204, -0.32483822107315063, -0.2183220088481903, 0.705025851726532, -0.4266054928302765, 0.8188425898551941, 0.5397970676422119, 0.9159263372421265, -1.260480284690857, -2.715954303741455, 0.07287134975194931, 0.14042915403842926, 2.2588346004486084, -0.6259549856185913, -2.732274055480957, 0.2870051860809326, 1.2068676948547363, -0.3649223744869232, 0.9217527508735657, 0.6308567523956299, -0.6336473226547241, -3.0777194499969482, -1.9964053630828857, 0.29292187094688416, 1.2750276327133179, -1.6961461305618286, 0.9927203059196472, -1.0630871057510376, 0.9954240322113037, 0.39198753237724304, 0.7705762982368469, -2.4748289585113525, -0.6686285734176636, -0.3390938341617584, 0.4947540760040283, -0.13572347164154053, -0.7149716019630432, -0.6084803342819214, 0.9116907119750977, -0.7759873867034912, -0.07074930518865585, 0.4029453992843628, 1.9294975996017456, -1.0389341115951538, -0.6129696369171143, -0.436957985162735, -0.6763803958892822, -0.6313753128051758, 1.1868234872817993, 1.6595836877822876, -1.385110855102539, -2.1789400577545166, 0.12675713002681732, 0.692540168762207, -1.2461827993392944, -0.8526830673217773, -0.19814135134220123, 0.5986260175704956, -0.9159817099571228, -0.2663419842720032, -0.5115289688110352, 0.009464522823691368, -1.1461989879608154, -0.6759935617446899, -0.15910184383392334, 0.5317476391792297, 0.8336543440818787, 0.7983784675598145, 0.03318021073937416, -0.6690990328788757, 1.7155306339263916, 1.4276551008224487, -0.747105598449707, 0.7098447680473328, 2.659080743789673, 0.6193059682846069, -1.4541993141174316, -0.22463367879390717, -2.6844124794006348, 0.19636063277721405, 0.5322690010070801, -1.6964422464370728, -1.1156340837478638, 0.19626212120056152, 0.6874523758888245, 2.125598192214966, -0.0065772985108196735, 1.7951078414916992, 0.8469064831733704, -0.12207353115081787, -0.6670691967010498, 0.7589500546455383, -0.31729063391685486, -1.2065225839614868, -0.2296617478132248, 1.222228765487671, -0.32610419392585754, 0.8343951106071472, 0.21640607714653015, 1.654854655265808, -0.5490847826004028, -0.19113194942474365, 0.3543601632118225, 0.7304086685180664, 0.8142318725585938, 0.26130929589271545, 0.3831018805503845, 0.1352476328611374, -1.0824122428894043, -0.527494490146637, -0.48594045639038086, -0.0013846082147210836, 1.028154969215393, -0.04222160577774048, 1.4004770517349243, 1.80044686794281, 2.72135329246521, -0.4428679347038269, 1.104971170425415, -1.104128122329712, -1.42361581325531, 0.5313354730606079, 1.6893106698989868, 0.0656449943780899, 0.2695677876472473, -0.7938690185546875, 0.3160668909549713, -0.23793138563632965, -0.2727980315685272, 0.5780562162399292, -0.2807822525501251, -0.4596046209335327, 1.9249341487884521, -0.11241958290338516], [-0.2228982001543045, 0.08304466307163239, 0.7377203702926636, -1.2163833379745483, 1.3330918550491333, 0.007059282623231411, -0.6953052282333374, -0.03474964573979378, -0.8450661301612854, 0.2504793703556061, 0.7510798573493958, 1.0845575332641602, 0.16943037509918213, 0.32048138976097107, 0.3717671036720276, -0.5356829762458801, 0.34341415762901306, -0.7665013670921326, 0.7234978675842285, -0.002524648793041706, -0.2078613042831421, -0.1976201832294464, 0.5626842379570007, 0.2556386888027191, 0.97046959400177, -0.16985724866390228, -0.9737588763237, -0.22079084813594818, -1.839798927307129, -0.24063876271247864, 0.18638044595718384, -0.9992990493774414, -0.6872300505638123, -0.1701633334159851, -0.7623957991600037, 0.8463753461837769, -1.1840895414352417, -0.653265655040741, 1.766927719116211, 0.7162079811096191, -0.15726064145565033, 0.21399571001529694, -0.237764373421669, -1.055680274963379, 2.3622546195983887, -0.39809608459472656, -1.1528072357177734, 1.8588515520095825, -0.38733237981796265, 0.2882063388824463, -0.0560261569917202, 0.15795989334583282, 0.46458882093429565, -0.5059572458267212, -0.2683202624320984, 0.18851393461227417, -2.1855862140655518, -0.09983369708061218, -1.152600884437561, -2.3143417835235596, 0.6047160625457764, 0.9384399652481079, 0.9823914170265198, -1.1040347814559937, 0.37123221158981323, 0.5216264724731445, 0.3282727003097534, -0.7237794399261475, -0.6351446509361267, 1.0056569576263428, 0.17357072234153748, -0.7379797101020813, 0.6177908778190613, 0.08939071744680405, -0.21782317757606506, 1.759081244468689, 0.9819121956825256, 0.1967194825410843, 0.034682173281908035, 1.223958134651184, -2.0407238006591797, -0.42396727204322815, 0.41113775968551636, -0.04858490452170372, 0.2901631295681, -0.49821335077285767, -2.2045886516571045, 0.10162762552499771, 1.187734842300415, -0.1942189335823059, 0.42055195569992065, -1.2887440919876099, 0.32964885234832764, 0.5425055027008057, 2.0275304317474365, -0.0030650540720671415, 1.3345938920974731, -0.9108430743217468, -1.1857904195785522, 0.7237167358398438, -0.6025802493095398, 0.22602403163909912, 0.5472871661186218, -0.9623444080352783, 1.2859379053115845, -0.19091741740703583, -1.6009620428085327, 0.8519406318664551, -0.29348108172416687, 1.1163684129714966, -1.2160214185714722, -0.7114741206169128, 1.4460780620574951, -1.3574891090393066, 0.48265567421913147, -0.056559473276138306, 2.047372341156006, 0.6558781862258911, 2.794922351837158, -0.48745012283325195, -0.794235110282898, -0.7234506607055664, -0.09762880206108093, -0.5509554147720337, 0.657009482383728, -1.3314112424850464, 0.0779617503285408, -1.6232746839523315, -2.5833845138549805, 0.4775460660457611, 0.0826302319765091, -1.7886277437210083, 1.2871313095092773, 0.32478606700897217, 1.046156406402588, -0.9334900975227356, 0.08439144492149353, 1.2033158540725708, -1.3658698797225952, -0.514970064163208, 1.569754958152771, 2.3377153873443604, -0.2669641971588135, 0.8031582236289978, -0.15170122683048248, 0.02719936892390251, -0.08907120674848557, -0.05907435715198517, -1.243221640586853, 0.19227860867977142, 0.5347587466239929, -2.3361175060272217, -0.8306373357772827, 1.2296078205108643, -0.088649220764637, -0.6367247700691223, 0.6296427249908447, -0.8562697768211365, 0.2633257806301117, 0.5630776286125183, -0.21970437467098236, -0.887763500213623, 0.7221822738647461, 1.7840783596038818, 1.5685487985610962, -0.4938093423843384, -0.7142428755760193, -1.6816076040267944, 1.4902902841567993, -1.372700810432434, -1.3079246282577515, -0.763455867767334, -0.7003822922706604, 1.1723151206970215, -1.0944699048995972, 1.906247615814209, 0.9583481550216675, -0.1271589696407318, 0.9093496203422546, 0.4220823347568512, -0.6198803186416626, 0.9927621483802795, -1.1034809350967407, -0.8550994396209717, 1.8050639629364014, 1.1473631858825684, -0.449205219745636, 0.17167574167251587, 0.5334048271179199, -1.2375526428222656, 0.0859687402844429, 1.017511248588562, 0.9890998601913452, -1.0212322473526, -0.5868651270866394, 0.13427580893039703, 1.7530102729797363, 1.7299349308013916, 3.6162161827087402, -0.9355217814445496], [1.1882814168930054, -0.4471902549266815, -1.0842199325561523, -1.0713694095611572, -0.7599751949310303, 0.8870729207992554, -1.7898151874542236, 0.21424375474452972, 0.3578717112541199, -3.0770370960235596, 0.2847434878349304, 0.6907876133918762, 1.2543208599090576, -0.31544628739356995, 0.6564976572990417, 0.06727465987205505, 0.6723672747612, 0.07674112915992737, -0.46738162636756897, -1.476893663406372, -2.689148426055908, 0.12617239356040955, 0.2378600835800171, 1.7032338380813599, -0.08040956407785416, 0.6929004788398743, -0.33221739530563354, 1.3876018524169922, 0.2869369685649872, 1.5989686250686646, -0.2010229527950287, 0.4549523591995239, -2.850393056869507, 1.207285761833191, 0.7438973188400269, 0.7533754110336304, -0.8036234974861145, -0.6422516107559204, 0.18392111361026764, 0.19609257578849792, -0.1329272836446762, 0.8563606142997742, -2.4695870876312256, -0.5117910504341125, 0.6832065582275391, 0.2057582437992096, 0.7646925449371338, 0.3960401713848114, -0.0496271550655365, 0.5860541462898254, -1.0191617012023926, -0.34563103318214417, -0.7964245676994324, -0.19737693667411804, -1.020542025566101, 0.3384369909763336, -1.3081742525100708, 0.85677170753479, 0.3124905228614807, -0.015014090575277805, -0.36006858944892883, -0.1387065052986145, 2.075951337814331, -0.6887328028678894, -0.02449950762093067, 0.1087135523557663, 1.3501355648040771, 0.4573536217212677, -0.25278761982917786, -0.16476880013942719, 0.0722692683339119, -1.105871319770813, -1.0514222383499146, -2.270007848739624, -1.8662701845169067, -0.6873136162757874, -1.1023684740066528, -0.7875643372535706, 0.5561338067054749, 1.0796737670898438, 2.011643409729004, -1.686877965927124, -0.012423344887793064, 0.8066160082817078, 1.7814052104949951, -0.08157113194465637, -0.3083963692188263, -0.05551110580563545, -1.6229678392410278, -1.202718734741211, 0.7817561030387878, 1.123071551322937, 0.17930427193641663, -1.235427737236023, -0.9555875658988953, -1.0674314498901367, -0.725361704826355, 0.08817098289728165, 0.9091468453407288, -0.325283020734787, 1.0741207599639893, -0.5048590302467346, 0.12315063923597336, 0.525639533996582, -0.236617311835289, 0.5513912439346313, -0.30438509583473206, -0.27445510029792786, -0.21475811302661896, 1.1618231534957886, 2.4560909271240234, 0.6784162521362305, 0.06903532892465591, 0.005581124685704708, 0.9431164264678955, 1.2843286991119385, 0.7226992249488831, -0.40446531772613525, 1.0296804904937744, 1.2022998332977295, -1.0146018266677856, 0.8494184017181396, 1.073047399520874, -0.012274268083274364, -2.2042930126190186, -0.7870140671730042, 0.5134553909301758, -0.4969092905521393, 1.7507812976837158, -0.6228545308113098, 0.4082722067832947, -0.9494502544403076, -0.13951359689235687, -0.9136136174201965, 0.640596866607666, -0.3474476635456085, -0.8828698396682739, -0.09387487173080444, -1.5918493270874023, 0.5052198171615601, 0.11330538988113403, -0.6397479176521301, 1.0881659984588623, -0.6195502281188965, -1.1675333976745605, -1.0497381687164307, -0.5010859370231628, 1.1156129837036133, 2.2056989669799805, -0.06924036890268326, -1.1679664850234985, -0.4974316358566284, 0.3908626139163971, 0.9118824005126953, -1.6572167873382568, 0.9657841324806213, 1.8704993724822998, -0.03354830667376518, 1.5159595012664795, -0.06311947107315063, -0.7594994306564331, -1.6884790658950806, 0.6989802718162537, -0.9998884201049805, 0.7491376399993896, -1.1330652236938477, 0.5961660742759705, -0.7392193675041199, -0.2324172556400299, -0.4263729155063629, 0.29085662961006165, -0.7299184203147888, -1.6391065120697021, -0.22614170610904694, 0.5825721025466919, -0.30099502205848694, -0.7977068424224854, -0.6901049017906189, -0.5605903267860413, -0.31149035692214966, 1.5345538854599, 0.2771227955818176, -0.40453454852104187, 0.3683193027973175, -1.8629016876220703, -1.1807572841644287, -1.511857271194458, 0.6499025821685791, 0.14181378483772278, 0.968084454536438, 1.2006025314331055, 1.828909158706665, -2.553147315979004, 0.9653498530387878, 1.4192078113555908, -0.37709152698516846, 0.3380204439163208, 1.9189530611038208, 0.5241336822509766, 0.13707207143306732], [0.05581198260188103, 0.23435214161872864, 0.6060084700584412, 0.37125420570373535, -0.06523691862821579, 0.678114652633667, 0.39498189091682434, -0.4962621033191681, -0.19753393530845642, -0.3677676022052765, 0.39536622166633606, 0.24185174703598022, -0.2934340536594391, 1.6910157203674316, -1.0203261375427246, 1.2094292640686035, 0.1156013086438179, -0.3830893039703369, -0.8242110013961792, 1.8412855863571167, -0.24748942255973816, 1.5624217987060547, -0.8604900240898132, 1.0247784852981567, 1.4410817623138428, 0.6665433645248413, -0.14326028525829315, 0.3012993335723877, 1.481374979019165, -0.13544148206710815, -0.376033753156662, -0.13241945207118988, 1.1776655912399292, -0.2829219400882721, 2.0047829151153564, 1.9239239692687988, 0.991908848285675, 0.40847641229629517, 0.4269576966762543, -1.4128022193908691, -2.2202091217041016, 0.3312003016471863, -0.17589238286018372, 0.30983999371528625, 0.8729785680770874, -0.8452699184417725, 1.3881874084472656, 0.8237642049789429, -0.939303994178772, 0.012149828486144543, -1.2941136360168457, 1.3803284168243408, -1.9399429559707642, -0.8109350800514221, 0.8272529244422913, 0.8987042307853699, -2.2208218574523926, 1.0997017621994019, -0.3278069794178009, 0.7934205532073975, -0.8826098442077637, 0.7750563621520996, -0.1367914080619812, -1.236816167831421, -0.5131733417510986, -0.39023831486701965, 0.4352114796638489, -0.019235417246818542, 0.04296372830867767, 0.28812047839164734, 0.029586905613541603, 2.532179117202759, 0.9266644716262817, 1.7285957336425781, -1.4111778736114502, 0.6539022326469421, 0.4007105231285095, -0.8408678770065308, -2.4421935449936427e-05, -0.5833511352539062, 0.4222802221775055, 1.473862886428833, -0.496982216835022, 1.2974498271942139, -1.4701330661773682, -0.8056163787841797, 0.27141422033309937, -1.2650606632232666, -0.7464770078659058, 0.24806566536426544, -1.248815894126892, 1.303370714187622, 0.9907205700874329, -0.655440092086792, 0.06622034311294556, -0.13657423853874207, 0.9794692397117615, 0.5080515146255493, 0.43916627764701843, 1.919924020767212, 1.217790126800537, 0.5410978198051453, 0.7403400540351868, 0.4927290380001068, 0.6494982242584229, -0.37515711784362793, 0.3526543378829956, 0.923662543296814, -1.5652996301651, 1.2900656461715698, 0.47872987389564514, -0.535957396030426, -0.7940758466720581, 1.0917268991470337, -0.5946486592292786, -1.3216454982757568, 1.0889729261398315, -0.1785532385110855, -1.8206590414047241, -0.5872020125389099, -0.6265546083450317, -1.1564842462539673, -1.0339797735214233, 0.5519211888313293, -0.07382914423942566, -0.01460485439747572, -0.3513472080230713, -0.11103041470050812, 0.16875848174095154, -0.6750057339668274, 0.03660043329000473, -0.6270560622215271, 0.09565190970897675, 0.7003216743469238, 0.8407387137413025, -0.5724340677261353, -0.16574132442474365, 0.9385244846343994, 0.8722321391105652, 0.7538360953330994, -0.8170629143714905, -0.683232307434082, 0.8102549910545349, 0.3909855782985687, 0.9241054058074951, 0.4475395679473877, 0.08757705241441727, 0.44571930170059204, -0.4822569489479065, 1.4363809823989868, -1.0411934852600098, -0.8969234824180603, -1.3019461631774902, -2.2971463203430176, 0.16822054982185364, -1.1406090259552002, 0.15527477860450745, -0.04921979084610939, -0.9235562086105347, -0.6704612374305725, 0.058228060603141785, -0.4799198508262634, -0.5736055374145508, -1.1644309759140015, 0.440992146730423, 0.729834258556366, 1.382425308227539, 1.9322865009307861, -1.0722436904907227, 1.143488883972168, -0.2468453198671341, -0.107783243060112, 0.9030281901359558, 0.5791154503822327, -0.26796457171440125, -0.5298972129821777, 0.05242610350251198, 0.9353499412536621, -0.7519800662994385, -0.3447924852371216, -0.25138017535209656, 0.3056212365627289, -0.11474420875310898, 1.3139495849609375, 0.5695923566818237, 0.5381223559379578, 0.4139571487903595, 0.4804629683494568, -0.6923561692237854, 0.5086060166358948, 0.4679647982120514, 0.9415428638458252, -0.5827703475952148, -0.05075531825423241, -0.2742719352245331, 0.7733436226844788, 0.4045582711696625, 1.1726690530776978, 1.3374030590057373, -0.9432510733604431], [-1.5661143064498901, -1.1161003112792969, 1.0015802383422852, -0.3002759516239166, -0.48865750432014465, -1.3946211338043213, -0.6277709603309631, 0.9940502047538757, 0.25235456228256226, 0.3805467486381531, 0.16260716319084167, -0.16792096197605133, 1.3959330320358276, 0.5785768032073975, 0.6558235287666321, -0.8104202151298523, 1.5765926837921143, 1.4866315126419067, 0.6583380699157715, -1.315163016319275, 0.9562814235687256, -0.6370617747306824, 1.5603313446044922, -2.264965534210205, 0.9016931653022766, -0.0020747289527207613, -0.21580074727535248, 1.170190453529358, -1.3751695156097412, -0.08525752276182175, 0.06781023740768433, 0.5270616412162781, 0.014994769357144833, 0.639722466468811, 0.8933913707733154, -1.0692899227142334, 1.034898281097412, 1.1579138040542603, -0.2061767727136612, 0.22446033358573914, -0.06657253950834274, 1.2298357486724854, -0.6863044500350952, 1.0410610437393188, 0.026632923632860184, -0.420046865940094, -1.02016019821167, -0.326347678899765, -0.16580472886562347, 1.5005900859832764, 0.19435158371925354, 0.9416269063949585, 0.5992090702056885, -0.5076846480369568, 0.24326637387275696, -0.032103877514600754, -1.1642327308654785, 0.11043266206979752, 0.5460385680198669, -0.9342308044433594, -0.7953266501426697, -0.5166817903518677, -2.3934075832366943, 0.7847462296485901, -0.04777519404888153, -0.15350855886936188, -0.6658071875572205, -1.2386504411697388, 0.16629323363304138, -0.923943042755127, 1.0669902563095093, 2.054753303527832, -0.23250612616539001, -1.9758284091949463, -0.21803413331508636, 0.5139837861061096, 1.5130115747451782, 0.14583607017993927, 2.486684799194336, 0.9637828469276428, 0.3874948024749756, 1.1778596639633179, 1.1999895572662354, -0.48354703187942505, -0.06403056532144547, -1.288047432899475, -0.11906083673238754, 0.4104078412055969, 0.7566801309585571, 0.5732635855674744, 1.5854308605194092, -0.034645356237888336, 0.45935961604118347, -0.4364890456199646, 0.16212330758571625, 0.04178684204816818, 0.945780336856842, 0.06610107421875, -0.3951345980167389, 0.05193759500980377, 0.5009440779685974, 1.6243188381195068, -0.27015507221221924, 0.5161216855049133, -0.19448943436145782, -0.7269854545593262, -1.1414328813552856, -1.3642319440841675, -1.1769180297851562, 0.6457824110984802, -0.0779595673084259, -0.824943482875824, -0.8055856823921204, -1.3545337915420532, -1.297005534172058, -1.0288736820220947, -0.10712260752916336, 0.1373901218175888, -0.21506793797016144, 1.04714035987854, -1.1186479330062866, -0.9293130040168762, 0.13399216532707214, 1.8582561016082764, 0.5590865015983582, -0.930237889289856, 0.4228406250476837, -0.9528596997261047, 0.7452864050865173, 0.2231505811214447, 0.03292665258049965, 0.6437105536460876, -0.6152567863464355, 1.2606278657913208, 0.03117343969643116, 0.4490194320678711, -0.06505607813596725, 0.651312530040741, 1.0774108171463013, 1.0311952829360962, 0.8460673093795776, -0.012106535024940968, -0.38473162055015564, 0.5676583647727966, -1.1820564270019531, 0.23813176155090332, -0.2728259861469269, -0.22984440624713898, 2.2925848960876465, -0.2648667097091675, 1.508998155593872, 0.0343969464302063, 0.43188583850860596, 1.3515074253082275, -0.273579865694046, 1.3816213607788086, 1.3790041208267212, -0.18502508103847504, 1.5892425775527954, -0.028829893097281456, 0.676356852054596, -0.5815354585647583, -1.1996943950653076, 0.3210970461368561, 0.9088824391365051, -0.577890157699585, 0.43968886137008667, -1.2948969602584839, -0.1845947802066803, -1.1187617778778076, 2.125556468963623, 0.6053994297981262, 0.6418700218200684, -1.823534607887268, -0.24091175198554993, -0.4029204845428467, -1.0082319974899292, -0.42234939336776733, -0.9090970754623413, -1.1661564111709595, -0.33206093311309814, 0.7483304738998413, -0.583109974861145, -1.2732124328613281, 0.8873906135559082, -0.15123426914215088, -0.6748461127281189, -0.2333248406648636, -0.13799908757209778, 0.8109222054481506, 0.4886826276779175, 0.23066185414791107, 0.3451474905014038, -1.355918288230896, -0.22511503100395203, -0.5544137358665466, 0.4482072591781616, -0.639503002166748, 0.5217232704162598, -0.6710047125816345], [0.5630105137825012, 0.5445424914360046, -0.14857207238674164, -1.5786869525909424, 0.48489707708358765, 0.16447418928146362, -1.939215064048767, -0.07436349987983704, 0.09076360613107681, -1.441922903060913, 1.034421443939209, -0.2303495705127716, -0.7604892253875732, 1.5700374841690063, -0.7863170504570007, 0.3224077522754669, -0.38064587116241455, -2.1254546642303467, -1.871087670326233, 0.6292425990104675, 1.2298191785812378, -0.3040497303009033, 0.42611563205718994, 1.9734865427017212, 1.1055186986923218, 0.08874638378620148, 0.04993976280093193, 0.28384754061698914, 0.14377380907535553, 1.2449040412902832, 0.5946951508522034, 1.0265939235687256, -0.651756227016449, 0.06716382503509521, 1.1048110723495483, 0.8564246892929077, -0.8616871237754822, 0.47330448031425476, 1.3315612077713013, 1.1778995990753174, -2.6391282081604004, -0.05950543284416199, -0.08236337453126907, -0.3587301969528198, -0.969568133354187, -0.295710027217865, 0.7985784411430359, 0.048330213874578476, -1.4573278427124023, 0.6005373001098633, 0.634898841381073, -0.7042911648750305, 0.673062801361084, 1.334692120552063, -0.18959757685661316, 0.3851647675037384, 0.805929958820343, 1.008373737335205, 0.04300568625330925, -2.2499732971191406, -0.6835799217224121, 0.3950138986110687, 0.34170424938201904, -0.05546196922659874, -1.340143084526062, 2.3620004653930664, -0.997249186038971, 1.6066818237304688, -0.1652565449476242, 0.2188410758972168, 1.7824914455413818, 0.6955103278160095, 0.5720458626747131, -1.5538647174835205, -0.6828829050064087, 0.24484997987747192, 0.2942524552345276, 0.8532189726829529, -0.5262569189071655, -0.5620093941688538, -0.16283699870109558, -1.2654900550842285, -1.2264059782028198, -0.4465726912021637, 0.6007504463195801, 0.3864343762397766, 0.9968247413635254, 0.4067907929420471, -0.11269860714673996, 0.5708280801773071, 0.27213871479034424, -1.4398919343948364, 1.3589318990707397, 0.010732726193964481, 1.451074242591858, 0.5882227420806885, 2.0282907485961914, 0.4481760263442993, 0.02907276526093483, 0.8987924456596375, 0.7016885876655579, -0.7150713205337524, 1.5127792358398438, -0.14687076210975647, -1.2081224918365479, -1.3764734268188477, -0.044804420322179794, -2.1167972087860107, -0.2824852168560028, 0.3582645356655121, 0.7163645625114441, -1.282875657081604, 0.2062869518995285, -0.7400619387626648, -0.17176759243011475, -1.2609525918960571, 1.7928310632705688, 0.9193307757377625, 0.38463059067726135, -0.7785516977310181, -0.5663806200027466, -0.5640177726745605, -1.1865254640579224, 0.29843899607658386, 0.4779280424118042, -1.9417933225631714, -0.2812958061695099, -1.5816651582717896, 0.3005618453025818, -0.5119747519493103, -0.46155065298080444, 0.5859283804893494, 0.19147063791751862, 0.33598172664642334, 0.2417132556438446, 0.260599821805954, -0.07032447308301926, -1.4237914085388184, 1.2463198900222778, -0.481392502784729, -0.7571073174476624, 0.25282904505729675, -0.026482481509447098, -1.1290370225906372, 0.195932999253273, -0.6100249290466309, 1.4027348756790161, -0.9580098986625671, 0.9020087122917175, -1.3528923988342285, 0.8091005086898804, 0.9875982403755188, -0.04884187504649162, 0.08242014050483704, -1.6874685287475586, 0.11029709875583649, -1.7519575357437134, 2.3143835067749023, 1.1148512363433838, -0.5063241124153137, 0.24616177380084991, 1.6738029718399048, 0.4980408847332001, 1.4577962160110474, 1.9590998888015747, 0.2828260660171509, 0.9146549701690674, 0.46767109632492065, -0.27967026829719543, -1.7231944799423218, -0.2690810263156891, -1.192399263381958, -0.35623013973236084, -0.3202597498893738, -0.4173475205898285, -0.22047151625156403, 1.167320966720581, 0.6906521916389465, 0.0897131934762001, 0.8639234900474548, -1.1605144739151, -0.788115918636322, 0.3881005644798279, 0.09979142248630524, -0.930600106716156, 0.874887228012085, 0.7465779185295105, -0.16714979708194733, 0.6534879207611084, 0.9982849359512329, 0.044863902032375336, 1.5699721574783325, 0.9281042814254761, 0.9908058047294617, 0.23548799753189087, 0.9898806810379028, 1.927865743637085, 0.3906538784503937, -0.9594051837921143, -0.30607813596725464], [0.17162935435771942, -0.7589899897575378, -2.6725690364837646, 0.24087455868721008, -2.6873598098754883, 2.0832996368408203, 0.23349669575691223, 0.618068277835846, -1.1943050622940063, 0.07878157496452332, -0.29842743277549744, -1.456048607826233, 0.6645748615264893, 0.37499094009399414, 0.7758086919784546, 0.5101266503334045, -1.8552613258361816, 0.8331440091133118, 0.7965617775917053, 0.37993791699409485, -0.15030145645141602, 0.757706344127655, 0.8367685079574585, 1.1086232662200928, 0.22928981482982635, 1.3169397115707397, 1.0457696914672852, 0.8123555779457092, 0.10379563271999359, 0.044694237411022186, -0.09569151699542999, -0.21056698262691498, 0.3321318030357361, -0.3274642527103424, 1.222553014755249, 0.4715568423271179, -0.762836217880249, -0.12965914607048035, -0.5485240817070007, -0.2049424648284912, -0.14425219595432281, -1.8141199350357056, 0.24856112897396088, -0.8681247234344482, -0.5556085109710693, 1.0072689056396484, -0.42976632714271545, -1.4345881938934326, -0.7310734987258911, 1.003758192062378, 0.11247023195028305, 2.0637691020965576, 0.5233795642852783, 0.8168600797653198, 0.4621502757072449, -1.3031855821609497, 1.2292585372924805, 0.5342342853546143, 0.6286501884460449, -0.42991551756858826, -0.14872489869594574, -0.10260158777236938, 0.17650340497493744, -1.1008470058441162, 0.08510593324899673, 0.47421231865882874, 1.103649616241455, -1.163686752319336, -0.11765957623720169, 2.2350966930389404, 0.45342397689819336, 1.0960711240768433, -0.48640862107276917, 1.7974023818969727, 1.4459455013275146, -0.8488847613334656, 0.07499848306179047, -0.11477041989564896, -2.883132219314575, -0.5276007652282715, -1.204719066619873, 0.09334518015384674, -0.24831436574459076, 0.7642739415168762, 0.19003336131572723, -1.5113310813903809, 0.07532552629709244, -1.3331598043441772, 0.9233697056770325, -0.8356254696846008, 0.20470194518566132, 1.7448631525039673, -0.35877835750579834, 0.8786231875419617, -0.2662511467933655, 1.024925708770752, 1.6315150260925293, -0.385748028755188, 1.9678544998168945, 1.474396824836731, -0.8230525255203247, -0.44180938601493835, -0.20288865268230438, -0.31556999683380127, 1.1038225889205933, 1.139417290687561, -0.45622071623802185, 1.028864860534668, -1.0054219961166382, -0.5475079417228699, 0.7296792268753052, 1.2690918445587158, 1.5536969900131226, 0.3298546075820923, 1.2366811037063599, -0.4763069152832031, 1.5628267526626587, 2.1278507709503174, 1.8842480182647705, 2.629857063293457, -1.4150665998458862, 1.8611701726913452, -0.019923269748687744, -0.17522764205932617, 3.430499792098999, -0.03546445444226265, 0.11002591997385025, 0.5705235600471497, 1.2103620767593384, 1.184571385383606, -0.7177157402038574, 0.40194129943847656, 0.21422693133354187, 0.3108954131603241, -1.505203127861023, -0.4227240979671478, 1.672615647315979, -0.8910243511199951, 0.5197705626487732, 2.324082374572754, 0.5047033429145813, -0.7537609338760376, 1.5130103826522827, 0.8999536633491516, 0.22246278822422028, 0.1643645465373993, -1.083682656288147, 0.22725003957748413, -1.24127197265625, 0.2092997133731842, -0.39487847685813904, 1.247298240661621, 0.35991427302360535, -0.601581335067749, 0.12108040601015091, -1.1803455352783203, 1.3028112649917603, 1.2283499240875244, -1.170028567314148, 0.24402442574501038, -0.8789345622062683, 0.2547468841075897, 0.5318621397018433, -0.6367947459220886, 0.05772697925567627, 0.632595956325531, -1.0803788900375366, -0.6970708966255188, 1.2157772779464722, -1.5702928304672241, -0.2854675352573395, 0.08560323715209961, 1.7039594650268555, -0.651518702507019, 0.46120980381965637, 0.5045028328895569, -1.114403247833252, -1.6546176671981812, 0.05637325346469879, 1.2456892728805542, -1.042602300643921, -0.9048227667808533, 0.09247621148824692, 0.035174448043107986, 0.6261816024780273, 0.7147993445396423, 0.5945878028869629, 0.21562352776527405, -0.8926656246185303, 1.408135175704956, -0.160784512758255, -1.2496278285980225, -0.2045305371284485, -1.1997195482254028, 0.1766263246536255, -0.6239994764328003, -1.8211182355880737, 0.14590582251548767, -1.5123543739318848, -0.5417708158493042], [0.057122550904750824, -0.5047056078910828, 0.5059153437614441, -1.731441617012024, -0.2418513149023056, -0.2181416004896164, -0.6955373287200928, 0.3980525732040405, 0.3592492938041687, -0.7344667911529541, -0.8737163543701172, -1.1192072629928589, -0.15786761045455933, 0.3922204077243805, -0.940062403678894, -0.7058423757553101, 0.13770818710327148, 0.8487342596054077, -1.0433348417282104, 1.4523510932922363, -0.5523167848587036, 0.5994287729263306, -2.0692150592803955, 0.3546988368034363, -1.9540660381317139, -0.009613614529371262, 0.39220625162124634, -1.0638532638549805, -0.7296521663665771, -1.4459035396575928, -1.1107289791107178, 1.7705250978469849, 1.832790732383728, -1.9384880065917969, -0.11191502958536148, 0.20801441371440887, 1.0831098556518555, -2.0964548587799072, 1.3494559526443481, -0.35787343978881836, 1.4429097175598145, -0.07240139693021774, 0.6578152775764465, 1.1253793239593506, 2.5583696365356445, -0.46592432260513306, -0.27471646666526794, 1.3542414903640747, -0.03693700581789017, -0.2136743813753128, 0.7643666863441467, -0.3390628695487976, 0.046762287616729736, 1.0657939910888672, -0.35092365741729736, -1.5438618659973145, -1.2053295373916626, -0.6667506098747253, 1.257538914680481, -1.9841499328613281, 0.48803141713142395, 0.9439075589179993, 1.8836983442306519, -0.24565033614635468, 0.6783905029296875, 3.789851188659668, -0.7120625972747803, 0.06601835042238235, -1.2308859825134277, -0.8553053736686707, 0.6185639500617981, -0.4036670923233032, -0.31203749775886536, 1.332531213760376, -0.2773958444595337, -0.845926821231842, 0.6727123260498047, -0.539207398891449, 1.1713966131210327, 0.3524903655052185, -0.9917481541633606, 1.10897696018219, 0.8263088464736938, -1.1593090295791626, -0.39359453320503235, 0.3655461072921753, 0.8274472951889038, -0.5656845569610596, 1.0815587043762207, -0.08112769573926926, -0.09267754852771759, -0.12378894537687302, -0.23354142904281616, -0.7904787063598633, -1.3346596956253052, -0.8507089018821716, 1.7875728607177734, 0.837883472442627, -1.1632282733917236, -0.5186308026313782, -0.6517695188522339, 1.54623281955719, 0.00694182887673378, 1.2820532321929932, 0.377696692943573, -0.06369546055793762, -1.6515693664550781, 0.35735300183296204, 0.7586559653282166, 0.7867211699485779, -0.015352035872638226, 2.0121467113494873, -0.4364311993122101, -0.20459868013858795, 2.000380516052246, -1.204710841178894, 0.8775849938392639, 0.9914942383766174, -0.2267661839723587, 1.5821950435638428, 1.2075293064117432, 1.0214807987213135, -1.5375940799713135, 1.549983024597168, 1.1798945665359497, 1.164572834968567, -0.08852702379226685, -2.345069408416748, -0.15238602459430695, 0.9537333846092224, 1.0489528179168701, 0.43373697996139526, -0.16119162738323212, 0.46028798818588257, 0.9812142252922058, -1.5438706874847412, 0.5392505526542664, 0.6871390342712402, -1.506279468536377, -0.26673954725265503, -0.8099212050437927, -0.30463653802871704, 0.5637927651405334, -1.1008199453353882, 0.4204167127609253, 0.35099607706069946, 1.289657473564148, 1.2382017374038696, 0.00924444105476141, 1.085543155670166, 0.42743009328842163, -0.7115018367767334, 0.09089595824480057, 0.5060623288154602, 0.5660229921340942, 0.7251073122024536, -0.5634729862213135, 0.12119714170694351, 2.4771335124969482, -1.4493517875671387, -0.2294529527425766, 0.38226765394210815, 0.5620644092559814, -0.3991498649120331, 1.156966209411621, -0.8211403489112854, -0.6300750970840454, 1.3437246084213257, -1.8753505945205688, 0.14968276023864746, 0.06240757182240486, -1.303656816482544, -1.246886968612671, -0.7072237730026245, -0.047836218029260635, 1.25509512424469, -0.37321028113365173, 0.19966569542884827, 1.2965366840362549, 0.00708985747769475, -1.999319314956665, -0.2792432904243469, 1.6317778825759888, -0.9343369603157043, 0.039803482592105865, -0.8365001678466797, -0.3608250021934509, -0.349955677986145, 0.3475389778614044, 2.440483331680298, -0.014163755811750889, 0.33945566415786743, -0.06188289821147919, 0.06773766130208969, 1.1531891822814941, -1.3041338920593262, -0.8881985545158386, 0.007495113182812929, -0.38843682408332825, -0.11213455349206924], [-0.48229655623435974, -0.1533420830965042, -0.4159984588623047, 1.1758733987808228, 0.08228860795497894, 0.8531702756881714, 0.4113178849220276, -0.1605858951807022, -0.6479622721672058, -1.3344658613204956, -0.2536143660545349, -0.7020889520645142, 0.45286107063293457, 0.5731627345085144, -3.328155755996704, -0.3053315281867981, -0.9365953803062439, 0.8284551501274109, 2.275071620941162, 0.28230100870132446, -0.0619843453168869, -0.18341699242591858, 0.001306807273067534, 0.5168825387954712, 1.0897868871688843, -0.8478126525878906, -0.8381150364875793, 0.92225182056427, -0.6857343912124634, 0.9776923656463623, 0.042939502745866776, 1.1580384969711304, -0.553673267364502, 0.7488820552825928, -0.05940064787864685, 0.149715855717659, 0.09779240936040878, 1.4085463285446167, 0.5380939245223999, -0.3535636365413666, 0.4284144341945648, 1.380502462387085, -0.7857028841972351, 0.8172035217285156, -0.32507652044296265, -0.9525699615478516, -0.019580315798521042, 0.5788400173187256, 1.057453989982605, -0.18398182094097137, 1.2299251556396484, -0.6660212874412537, 0.4670514464378357, 0.629505455493927, -0.6583905816078186, 0.049086328595876694, -1.4361777305603027, -0.6897207498550415, 0.05263090878725052, 0.3041338324546814, 0.5518391132354736, 0.704531729221344, -0.26564425230026245, -0.417775422334671, -2.0196714401245117, -1.1533981561660767, -0.11591410636901855, -0.6970248222351074, -0.6898881196975708, 0.4059509038925171, 1.657036542892456, -1.4740222692489624, 0.37498778104782104, 1.0489261150360107, -0.422793447971344, 1.1643022298812866, -0.7023345828056335, -0.0047491854056715965, -0.28551098704338074, 1.3675552606582642, 1.4382215738296509, 0.24377787113189697, -1.4019627571105957, 0.5676516890525818, 0.4258175492286682, -0.3576539158821106, 0.4454246461391449, -1.4116336107254028, -0.6423222422599792, -0.6679883003234863, -0.6658661365509033, 0.2390236109495163, 0.05383796989917755, 0.015071822330355644, -0.512431800365448, 0.5278096795082092, -0.08038854598999023, -1.1964001655578613, 0.22695806622505188, -1.4433114528656006, 0.38308656215667725, -1.634121298789978, -0.880001962184906, -0.5479418039321899, -0.15144626796245575, -1.7449883222579956, 0.5096175074577332, -1.0222790241241455, -0.059806790202856064, -0.9022996425628662, 0.21128802001476288, 0.8615143299102783, -0.6013482213020325, 0.017934955656528473, 0.9282034039497375, 0.8667994141578674, -0.846527636051178, -0.26619523763656616, 1.2453893423080444, -0.02020585536956787, 0.9573730826377869, 0.6904259920120239, 0.7249078154563904, 2.709200859069824, -1.8986045122146606, -1.2725070714950562, 0.5509693026542664, 0.22165726125240326, -0.391401469707489, 0.44947388768196106, -1.8655275106430054, 0.8448625206947327, 0.36783134937286377, 0.6185703873634338, 0.524246871471405, 0.3652971684932709, -1.783174753189087, 2.616168737411499, 2.647535562515259, -1.3964818716049194, -0.3051392436027527, -0.667098879814148, 0.3088671863079071, 0.2769630551338196, -0.11604778468608856, -0.5756082534790039, 0.7627583146095276, -0.813613772392273, 1.2534364461898804, -0.7024100422859192, 0.4017496109008789, 0.5229374170303345, 1.4013147354125977, 1.0856961011886597, -1.3329875469207764, 0.011936631053686142, -1.237294316291809, 2.1150710582733154, -0.5113672018051147, 0.35239139199256897, 0.5143023133277893, -1.8463214635849, -0.9559957385063171, 0.6232628226280212, 1.2285494804382324, -0.6685534119606018, 0.9037652611732483, -0.19122539460659027, 0.2893723249435425, -1.8513977527618408, -0.21728824079036713, 0.018876370042562485, -0.8318943977355957, -0.02016100101172924, -1.0872383117675781, 0.3947577476501465, -1.970229148864746, -1.3643728494644165, 0.37424880266189575, -1.1455110311508179, -0.5275218486785889, 1.232680082321167, 1.8149718046188354, -0.4759029746055603, 0.8460463881492615, 1.7498606443405151, -0.9376522302627563, -0.20982567965984344, -0.9196425676345825, -0.9071162939071655, 0.40821781754493713, -1.2079302072525024, 0.8441583514213562, 1.0561422109603882, -0.3974091112613678, 0.5559291243553162, 0.8280287981033325, -0.03991704806685448, -0.07981403172016144, -0.9833667278289795], [-1.2479151487350464, 0.367343932390213, -0.12894706428050995, -0.19171077013015747, -1.6500868797302246, 0.38204988837242126, 1.5632297992706299, -1.8736095428466797, -1.5021296739578247, -1.0892318487167358, 0.9681181907653809, 0.552266001701355, 0.9378408789634705, -0.31009024381637573, -0.6868944764137268, 0.21556499600410461, 0.952942967414856, 0.32634374499320984, 0.28848928213119507, 2.4575703144073486, 0.2997016906738281, -1.0452858209609985, -0.7202073335647583, -0.6032252907752991, -0.12708115577697754, 1.005411982536316, 1.3944367170333862, 0.8244500160217285, -0.6766689419746399, -1.263403296470642, 1.1027628183364868, -0.8217124938964844, -0.8268886804580688, -0.6829016208648682, -0.7109412550926208, -0.20067958533763885, 1.2491633892059326, -0.48827189207077026, 0.7120093107223511, 0.832999587059021, -0.09280245751142502, -0.6877448558807373, 1.3821383714675903, -0.6561850905418396, -0.23489001393318176, 0.9472507834434509, 0.948891818523407, -0.5033897757530212, 0.4397734999656677, -0.48941171169281006, 0.7498670816421509, 1.1025807857513428, 1.4003562927246094, 0.09478718787431717, 0.12025123089551926, -1.3103727102279663, 1.5059837102890015, 0.12581409513950348, 0.022904766723513603, -1.06284761428833, 0.21894481778144836, 1.4179699420928955, 0.39829522371292114, -0.3504728376865387, 1.673129677772522, 0.8808173537254333, 1.1068146228790283, 0.5719621777534485, 0.09928985685110092, 1.8241544961929321, -1.2503478527069092, 0.8227291703224182, -0.9900262355804443, 0.6208698153495789, -0.3863163888454437, -1.4875073432922363, 1.3724738359451294, 0.21004700660705566, -0.16763025522232056, -0.07558053731918335, 0.25084006786346436, -1.6565080881118774, -0.01705346815288067, -0.6231513619422913, 0.5292892456054688, -0.6612141728401184, -0.20387861132621765, 0.5468032956123352, -0.5002320408821106, -0.6933290958404541, -0.2842932343482971, 0.707576334476471, 0.9344325065612793, -0.24000410735607147, -2.3063857555389404, -0.890292227268219, 0.5434275269508362, 0.388654887676239, -1.0179754495620728, -0.856886625289917, 0.8791425228118896, -0.620513916015625, -0.3815676271915436, -1.3914321660995483, 2.4479048252105713, -0.771234929561615, -0.4093000888824463, 1.3600168228149414, 0.15196745097637177, -0.5017600059509277, -1.9364334344863892, -0.2139357179403305, -0.26427412033081055, 0.237200066447258, 0.20770962536334991, 0.41830044984817505, 1.121618390083313, 0.17953179776668549, -0.039314135909080505, -2.138176679611206, -0.26424285769462585, 1.1453427076339722, 1.0276061296463013, 0.6081624627113342, -0.36350318789482117, 1.7632148265838623, -1.6322461366653442, -1.3659051656723022, -0.18248052895069122, -1.044600486755371, -0.10168709605932236, -0.62257319688797, -0.6231263279914856, 0.006802602205425501, 0.013784026727080345, -0.22222688794136047, -0.11044242978096008, 0.08024242520332336, 0.22078858315944672, -0.23140276968479156, -0.21307072043418884, 0.2652105987071991, -2.794971466064453, -0.5873528718948364, 1.4888496398925781, 0.9402639865875244, -0.3428862988948822, 2.0218095779418945, -0.7729097008705139, 0.9880507588386536, 1.4930857419967651, -1.4979007244110107, -0.866316556930542, 0.6565622091293335, -0.1777523308992386, -0.7676018476486206, 1.9242830276489258, 0.5205840468406677, 0.7247603535652161, 0.23006466031074524, 0.5611146092414856, 0.672130823135376, 1.4865052700042725, 1.121536135673523, -0.9537337422370911, 0.56369948387146, 0.3541017770767212, -1.1426702737808228, 1.1378141641616821, -0.2182399481534958, -0.3681623339653015, -0.2975464165210724, 1.5715103149414062, -0.2266044169664383, 0.8592299222946167, 0.2168014496564865, -0.2526717483997345, -0.23782174289226532, 0.5005373954772949, -0.637944757938385, -0.3492279350757599, 0.24544845521450043, -0.38001906871795654, -0.7389022707939148, 0.5117396116256714, -0.6516039371490479, -0.20136034488677979, 1.132582426071167, -2.521716594696045, -0.22550490498542786, 0.7447454929351807, -0.24204035103321075, -0.19023744761943817, 0.37416359782218933, -0.24017252027988434, 1.0082396268844604, -1.4041478633880615, 1.4652526378631592, 1.2320219278335571, -0.3966290056705475], [-0.19045597314834595, 0.5944424867630005, 0.30050942301750183, -0.7823810577392578, -1.1027394533157349, -0.036304838955402374, 2.210402727127075, 1.5787509679794312, -0.9145994782447815, -0.08720378577709198, 0.028746142983436584, 0.38334548473358154, -1.0472338199615479, -0.9779481887817383, -0.08355339616537094, -0.2309839129447937, 0.5312425494194031, 0.3324625790119171, 1.5983080863952637, -0.4912552535533905, 0.9826105833053589, 1.6546766757965088, 1.2704704999923706, -1.837272047996521, -1.7533388137817383, 0.9309127330780029, 1.0414119958877563, -0.6489772200584412, -2.80226731300354, -0.41978105902671814, -0.252739280462265, -0.6443869471549988, -1.0657302141189575, 1.8714011907577515, 0.7126309871673584, 1.5404995679855347, 0.10893681645393372, 1.6351550817489624, 0.8143609166145325, -0.23229098320007324, -1.156798243522644, -1.2696205377578735, -0.10272518545389175, 0.09756257385015488, 0.5348036885261536, -0.11024679988622665, 0.8764837980270386, 1.4302302598953247, 0.1752900779247284, 0.7503931522369385, -0.9930492043495178, -0.05239590257406235, -0.8249815106391907, 0.11225356161594391, -1.9867572784423828, -1.0610352754592896, -0.7056016325950623, 0.7561553120613098, -0.17428751289844513, 0.323690801858902, -0.685201108455658, 0.7923384308815002, 0.10064666718244553, 0.31181666254997253, 0.2886321544647217, -1.36961030960083, -1.6386560201644897, 0.669060468673706, 1.0053107738494873, 1.8327380418777466, 0.22786656022071838, 1.5438909530639648, -0.6013839244842529, -0.6779469847679138, 2.1090943813323975, 0.020627310499548912, 0.5902937054634094, -0.8926486968994141, 0.45375123620033264, 0.9812785387039185, 0.5144461393356323, 0.5436543226242065, 1.4136407375335693, -0.3824550211429596, 1.2140415906906128, -0.8641042709350586, 0.026150882244110107, -0.05197998881340027, -0.07129455357789993, 1.1188404560089111, 0.4188406467437744, -0.5890131592750549, -2.271153211593628, 1.2893394231796265, 0.5922093987464905, -0.8837493062019348, -1.4326629638671875, -1.1141694784164429, -1.0530531406402588, 1.1329888105392456, 0.8806382417678833, 0.36966243386268616, -1.5364782810211182, 0.19628258049488068, -0.13981610536575317, 0.2499256581068039, -0.6136982440948486, 0.8404372930526733, 2.0818963050842285, 0.15285657346248627, 0.07390031218528748, 1.9845045804977417, 0.07545946538448334, -0.7719797492027283, -0.31764328479766846, -0.18443839251995087, -1.3402811288833618, -0.6180387139320374, -1.5552505254745483, 0.8194811344146729, -0.6291837096214294, -1.0469566583633423, -0.05336067080497742, -0.3110463619232178, 0.23551660776138306, -0.6740051507949829, 1.0696600675582886, -0.09923779219388962, 0.8817107081413269, -0.07072007656097412, 1.763049602508545, 0.9614899158477783, 0.02964613400399685, 0.8129695653915405, 0.01057923398911953, -0.03151273727416992, -1.4408237934112549, -0.14707641303539276, -1.2792876958847046, -0.45565780997276306, -0.4534224569797516, 0.32560229301452637, 1.209959626197815, 0.611786425113678, 0.8142752647399902, -1.1150790452957153, 1.1101633310317993, 0.49703583121299744, 0.028863979503512383, -0.5803003311157227, -0.1923549771308899, -0.15849611163139343, 1.4380900859832764, -1.1221193075180054, -0.6903393268585205, -0.6616120338439941, -0.027755131945014, 0.8094016313552856, -1.0786482095718384, 0.2614484131336212, 0.22989603877067566, -0.4444734752178192, -1.6677424907684326, 2.11342453956604, -1.1554557085037231, -0.6127146482467651, 0.5144297480583191, -1.258845329284668, 0.15981541574001312, 0.8486395478248596, -0.41579246520996094, 0.4207319915294647, -1.0999780893325806, -0.6051027774810791, 1.5546082258224487, -0.22071272134780884, -1.3502132892608643, 0.17807064950466156, 0.29254060983657837, 0.6478801965713501, -0.2148449867963791, -0.18419525027275085, -0.6512253880500793, -0.27937206625938416, -1.1474969387054443, -1.5254812240600586, 1.2378227710723877, 0.019592691212892532, -1.089188575744629, 0.4707317650318146, 0.8522705435752869, 2.1529555320739746, -0.8483829498291016, -2.666926860809326, -0.5122653841972351, 0.5081003308296204, 0.10100109130144119, 0.4929998219013214, 0.49839815497398376, -0.5090911984443665], [-0.595983624458313, 0.15717501938343048, 0.5232542753219604, -0.5505303144454956, -0.23262079060077667, -1.5082156658172607, -0.1457832306623459, -0.16587789356708527, 1.2946417331695557, 0.5468702912330627, -1.2542160749435425, 1.393057107925415, 0.8606529235839844, -0.20493324100971222, 1.3190656900405884, 0.02546694315969944, -0.5391459465026855, -0.1265999674797058, 0.5715222954750061, -1.6369484663009644, 2.490739583969116, -1.7055280208587646, 0.04595399647951126, -2.475426197052002, 0.1340479701757431, -2.0300252437591553, 1.2780208587646484, -1.388523817062378, -2.110084056854248, 0.307407021522522, -1.5182511806488037, -0.4626249074935913, 0.4025178551673889, 0.49213725328445435, 0.26322242617607117, -3.133993148803711, 2.041935682296753, -0.08506675809621811, -0.06211399286985397, 1.943308711051941, 1.5453258752822876, 0.08427909016609192, 0.2952466309070587, 1.0910428762435913, -0.9686015248298645, 0.4022575318813324, 0.20135299861431122, 0.9181859493255615, 0.4251967668533325, -0.894753634929657, 0.010113577358424664, 0.4550704061985016, 1.2968910932540894, 0.47652292251586914, -1.6015177965164185, 0.4301184117794037, 1.9154188632965088, -0.6323831677436829, -0.446350634098053, 0.04306424409151077, -0.35499224066734314, 1.3389368057250977, 0.864119291305542, -1.0683915615081787, 0.7519380450248718, 0.23160716891288757, 0.8164239525794983, 0.7855219841003418, 0.27724725008010864, 0.21650268137454987, -1.1706478595733643, 0.7395690083503723, 0.005125256255269051, -0.562801718711853, -1.3800185918807983, 0.2843824028968811, 1.758194088935852, -1.1414188146591187, -0.31806960701942444, 0.43431031703948975, 0.24332192540168762, -0.7365581393241882, 1.2325530052185059, -1.0466591119766235, -0.3887542486190796, 1.576569676399231, -1.771845817565918, 1.5804297924041748, 0.9742092490196228, -1.3827015161514282, -0.1011900007724762, 0.5444364547729492, 0.827613890171051, 0.2735178768634796, -0.4198819398880005, -0.5108761191368103, -1.8858426809310913, -1.2627222537994385, -1.504055142402649, -0.0036095043178647757, -0.9192633628845215, 0.23575802147388458, -0.7379693984985352, -1.1625128984451294, 1.1299047470092773, 1.2047325372695923, 0.6604742407798767, -0.20057106018066406, 1.8465923070907593, -0.999468207359314, 0.7701420187950134, 2.3064777851104736, 0.4908027946949005, -1.0487431287765503, -0.06943896412849426, -1.8737865686416626, 0.06708092242479324, -0.15065525472164154, 0.7135528326034546, -1.0414743423461914, -1.8660497665405273, 0.17114487290382385, 0.35519707202911377, 2.162116050720215, -0.19107328355312347, 0.7146406173706055, 0.37169963121414185, -0.6760377883911133, 1.2108150720596313, -0.6952121257781982, 0.6002942323684692, -0.08089554309844971, 1.0264066457748413, 0.7519773244857788, -0.7280750274658203, -1.2041139602661133, 0.5263428092002869, -0.6709293723106384, -0.5293275713920593, 0.9832891225814819, -1.6475180387496948, -0.05232062190771103, 1.137379765510559, 0.4816664457321167, -0.6981222629547119, -0.36324599385261536, -0.7906614542007446, 1.0746784210205078, 0.2978892922401428, 0.41642332077026367, 1.1923024654388428, -0.8765360116958618, 1.12535560131073, -1.7214186191558838, 1.1028882265090942, -0.32406380772590637, 1.1010403633117676, -0.40479984879493713, -0.13382843136787415, 1.0286542177200317, 0.033159561455249786, -0.9223579168319702, 0.15797853469848633, 0.32991763949394226, -0.3806084096431732, -0.07545255869626999, -0.6481768488883972, 1.392302393913269, -0.7319741249084473, -0.7784664034843445, 1.1091538667678833, 0.6511800289154053, 0.899838924407959, -1.036997675895691, 0.04805906489491463, -0.43035680055618286, -0.8745954036712646, -0.22539879381656647, -1.5238672494888306, 0.11610302329063416, 1.765452265739441, 1.157331943511963, 0.07272055000066757, 1.88943612575531, -0.7031594514846802, 1.6993862390518188, 0.12329385429620743, 1.0223408937454224, 0.4732441306114197, -1.0163724422454834, -0.5132645964622498, 0.6243756413459778, 0.051634352654218674, 0.24144189059734344, -1.0000122785568237, 1.0576978921890259, -0.623997688293457, -0.9443503022193909, 0.2796987295150757, -1.0286811590194702], [-0.9870737195014954, -0.195487380027771, 1.1864546537399292, -0.927604615688324, 1.4786272048950195, -0.7570749521255493, 0.262857586145401, 0.9701526761054993, 0.2873290181159973, -0.9853171706199646, -1.3998891115188599, 1.5527193546295166, 0.4131217896938324, 0.6693300008773804, 0.7934772372245789, 0.9928522109985352, 0.5151453614234924, 1.1392027139663696, 0.574694037437439, 0.4149433374404907, 0.7620033621788025, -0.09088362008333206, -1.8234037160873413, 0.5136573910713196, 0.32233983278274536, -0.7992208003997803, 1.7971572875976562, 0.924146294593811, -0.4017351269721985, -0.5552170872688293, -0.8029235601425171, 0.0879468321800232, 0.6092069745063782, 0.4407308101654053, -1.2008147239685059, -0.18462330102920532, -1.2072720527648926, 0.572277843952179, 1.839105248451233, -0.02331599034368992, 0.45700472593307495, 1.32326340675354, 0.2815597951412201, -0.8725865483283997, -1.573366641998291, 0.4158517122268677, -1.047186255455017, -1.140548825263977, -1.033449649810791, -1.2477129697799683, -0.023691026493906975, 1.4911229610443115, 1.962706208229065, 0.899135410785675, -1.9503370523452759, 1.4261369705200195, 1.7406964302062988, 0.1951356828212738, -0.8229095339775085, -0.5705161690711975, 0.20469434559345245, 0.5400682091712952, 0.6177017688751221, 0.26562151312828064, -0.5715360641479492, -0.500228226184845, -0.46971631050109863, -0.660443127155304, -0.15491288900375366, -0.909714937210083, 0.9274606108665466, 0.5856921672821045, 1.5840728282928467, -0.901849627494812, -1.4954746961593628, -1.8752195835113525, -1.2074211835861206, 1.7016819715499878, 0.629070520401001, 0.0013154932530596852, 0.32285723090171814, -1.326004147529602, -1.5741240978240967, 2.655480146408081, 1.3038212060928345, -0.36568307876586914, 1.846001386642456, 0.1741369664669037, 0.9009276032447815, -1.525139331817627, 0.2392147034406662, -0.9512795209884644, 0.13973024487495422, -0.06766334176063538, 1.0385754108428955, -0.13375811278820038, -0.495047390460968, 0.5351097583770752, 3.415768623352051, -1.472304344177246, -0.9681238532066345, 1.7168951034545898, -0.5929509401321411, 1.0449239015579224, 0.3174067437648773, -0.8583853244781494, 0.9671823382377625, -0.6196955442428589, 0.07605885714292526, 0.6605054140090942, -0.9126940369606018, -0.1257675737142563, -1.0443412065505981, 0.23088428378105164, -1.07802414894104, 1.1701619625091553, 0.7009028792381287, -0.09684909880161285, 0.6444815993309021, -0.08069463819265366, -1.7982807159423828, -0.15128788352012634, -0.24591252207756042, 1.3060797452926636, 1.2778974771499634, -1.961020588874817, 0.733768105506897, -0.34796130657196045, -0.11686196178197861, 0.8060683012008667, 0.649444580078125, -0.7262100577354431, 0.5374600887298584, 0.37029802799224854, -0.36833518743515015, 0.2694641649723053, 0.6995941996574402, -0.8080119490623474, 0.7803232073783875, 1.4633891582489014, -0.07427048683166504, 0.9187309145927429, 0.4423314332962036, 1.1214998960494995, 0.6639186143875122, -0.6315728425979614, 1.4547815322875977, -1.0817844867706299, 1.101312518119812, -0.9938690662384033, 0.24237099289894104, 0.4416091740131378, -2.086630344390869, 0.5762145519256592, -1.7652820348739624, 0.5058889985084534, 0.32889261841773987, -0.3889804184436798, 0.3995476961135864, -0.7859534621238708, 0.40086832642555237, 2.355867385864258, 0.5952266454696655, -2.1099982261657715, 0.13020390272140503, 0.32482412457466125, -0.2848142981529236, -0.028225768357515335, -0.04444335028529167, 1.482944369316101, -0.3337782919406891, -0.39919614791870117, -0.6052024960517883, 0.5840991139411926, 1.4628374576568604, -1.3864680528640747, -1.712123990058899, 0.5796077251434326, -0.41444718837738037, 1.250441551208496, -0.7216432690620422, -0.37516430020332336, 0.6790064573287964, 0.10073833912611008, -0.0006024466129019856, 0.43854111433029175, 0.7525613903999329, -0.02955162152647972, 0.08780407160520554, -1.3602499961853027, -0.4178686738014221, 0.5154935121536255, -1.8027055263519287, 0.33290621638298035, -0.5488074421882629, -1.7018846273422241, 0.005946178454905748, -1.107283115386963, 0.13714607059955597, 0.510475218296051], [0.4954787790775299, -1.9913796186447144, 1.1544384956359863, 1.3858891725540161, -0.4502450227737427, 0.9413271546363831, 1.4366573095321655, -0.008174863643944263, 1.1927684545516968, 0.8589476346969604, 0.23712791502475739, -1.5123891830444336, 1.5784660577774048, 0.34213411808013916, -0.5369249582290649, -2.351292371749878, 0.8443989157676697, -0.09106634557247162, 0.8818851113319397, -0.8572019934654236, -0.5620883703231812, -0.24105128645896912, 1.1059026718139648, -0.509040355682373, 1.7416307926177979, 0.5464263558387756, 1.1511317491531372, 1.1370514631271362, -0.06644352525472641, 0.061130572110414505, -0.3292606472969055, -0.263391375541687, 0.5558440685272217, -0.8405479788780212, -0.7024675607681274, 1.6351691484451294, -1.239572286605835, 0.24018336832523346, -1.2691494226455688, 0.16097351908683777, 0.2524226903915405, -1.1954405307769775, -0.013369991444051266, -0.5714809894561768, 1.2126221656799316, 1.7935606241226196, 1.1318440437316895, 1.410301923751831, 1.4249851703643799, -0.2809910774230957, -0.9160906672477722, -0.31689906120300293, 0.8391754031181335, 1.1466870307922363, -0.46897801756858826, 0.20396040380001068, 1.0923168659210205, -2.627871513366699, 0.8782575130462646, 0.24848228693008423, -0.2847515940666199, 0.35574302077293396, 0.36495107412338257, -1.415461778640747, 0.007939508184790611, -0.1558019369840622, -0.5065286755561829, -0.7484586834907532, -0.13381193578243256, 0.28519028425216675, 1.1548370122909546, -0.06430739909410477, -0.5042085647583008, 0.5911708474159241, 0.2448025345802307, -0.593757688999176, 1.5257316827774048, -0.005305036902427673, 1.5736271142959595, 0.21202205121517181, -0.3349035680294037, 0.06477927416563034, -0.5957467555999756, 1.374428629875183, 0.820591390132904, -1.6941996812820435, -0.2830798029899597, 0.6662600636482239, 0.1699928343296051, -0.5938711762428284, -1.8221782445907593, -0.6666242480278015, 1.5927850008010864, 0.8191378712654114, -0.8004653453826904, 0.5854586958885193, 0.28649166226387024, 0.2885436415672302, -0.7535033226013184, 0.12875965237617493, 0.8617756366729736, 1.1094577312469482, 0.6040619015693665, -0.92203688621521, 0.8365629315376282, -1.1086527109146118, -0.6533769369125366, -0.6296114325523376, 0.41986551880836487, -1.6492339372634888, 0.7510244846343994, -0.5684455037117004, -0.7027124166488647, -1.7633510828018188, -0.9064318537712097, -1.2939320802688599, 1.675515055656433, -0.12055972963571548, -0.7573467493057251, 0.29414016008377075, -2.911106586456299, -0.06760420650243759, 1.4882789850234985, 0.25350767374038696, -0.9585347771644592, 1.344925880432129, 0.43802571296691895, -1.630228877067566, -0.8563240170478821, -0.9022221565246582, -1.0924772024154663, 0.8823659420013428, 1.5495810508728027, 0.907517671585083, 2.463271379470825, -1.249464988708496, 0.694752037525177, -0.2177671194076538, 0.7632485032081604, -0.6042550206184387, -0.1823876053094864, -0.6860830187797546, 0.060294684022665024, 0.17472827434539795, 1.5074002742767334, -1.3887640237808228, -0.9641397595405579, -1.503602385520935, -1.121213436126709, 0.9499923586845398, 1.3925000429153442, -1.3887511491775513, 0.39516541361808777, -1.6675466299057007, 1.8648169040679932, 0.5615317821502686, 0.1759193241596222, 2.1560895442962646, -0.21319371461868286, 0.828665018081665, 0.012429720722138882, 0.9115946292877197, -0.4269726574420929, 0.8208656311035156, -0.6888611316680908, -1.922237753868103, -0.15800713002681732, -0.3123735189437866, -1.0645923614501953, 1.2108441591262817, -0.008078043349087238, -0.45015648007392883, 1.318466305732727, 0.613519549369812, 1.534072756767273, 0.8578407168388367, -1.4005440473556519, -2.0310566425323486, 0.5032748579978943, -2.0115389823913574, -1.0009387731552124, -1.17485773563385, 0.870198130607605, 1.3499561548233032, 1.1072810888290405, -0.6943138837814331, -0.7908591032028198, -0.2680165469646454, -0.05552726611495018, -0.007335355971008539, 1.1586010456085205, 1.2378607988357544, 0.027634147554636, -1.026858925819397, -1.2231501340866089, 0.5195249915122986, -0.890215277671814, -0.11575043946504593, 0.3778820335865021, -0.2678123116493225], [2.140468120574951, 0.2708500921726227, 0.795123279094696, -1.268196702003479, 0.3440156877040863, 1.929388165473938, -0.7920401692390442, 0.4232388138771057, -1.2193351984024048, -0.13000938296318054, -1.3415822982788086, -1.446937918663025, -0.12070248275995255, 0.7108332514762878, 1.150550365447998, -1.1618919372558594, 0.725192666053772, 0.5008419156074524, 1.15261709690094, 0.18662551045417786, 1.7767724990844727, 0.6575078368186951, -1.0569579601287842, -0.09606427699327469, 0.49937817454338074, -0.9149747490882874, 0.7627062201499939, 0.7944628000259399, -0.5393164753913879, -0.3978132903575897, -1.1717050075531006, 0.16213051974773407, 1.394662618637085, 0.23400792479515076, 1.3453420400619507, -1.1743533611297607, 1.444042682647705, -0.9640701413154602, -1.3625283241271973, 0.9414921402931213, 0.9681068658828735, 0.6127279996871948, 0.3870795667171478, -0.21282760798931122, 1.4824578762054443, -1.3659669160842896, 0.40137431025505066, -1.1600350141525269, 1.037805199623108, 1.0782206058502197, 0.5999947786331177, 0.2942758798599243, 0.3853980600833893, 0.9195410013198853, 0.7036564946174622, -0.0824311152100563, 1.1246060132980347, 0.8279240727424622, 0.46879273653030396, 0.4503488540649414, -0.27538958191871643, 0.6466323137283325, -0.5634719729423523, -0.11738356202840805, 0.5079494118690491, 0.5004146099090576, -2.265974283218384, 0.7355682253837585, 1.596943736076355, -0.3149227201938629, -1.7161005735397339, 0.20486587285995483, -0.8672031164169312, 0.2096855491399765, -1.6376428604125977, 0.9417319297790527, 0.4609915018081665, 0.35364314913749695, -0.9481574296951294, -0.15228527784347534, -1.1544967889785767, 0.49150705337524414, -0.45135146379470825, -0.7700177431106567, 0.17989923059940338, -0.35157328844070435, -1.4030531644821167, -0.9509020447731018, 2.1272637844085693, 1.1390162706375122, 1.349532961845398, -1.3836698532104492, -0.35045289993286133, 0.6030515432357788, -0.29934072494506836, -1.5298024415969849, -0.5045416355133057, -1.272615909576416, -0.11593781411647797, -0.24901452660560608, -2.0722737312316895, 0.19016528129577637, -0.009332926943898201, 0.5903037786483765, 0.1990446150302887, 0.00950157456099987, 0.3525089919567108, -0.6560222506523132, 0.9882405400276184, 1.31094491481781, 0.36557671427726746, -1.812804937362671, 0.6595552563667297, -1.0329421758651733, -1.104857087135315, 0.8971101641654968, -0.7299612164497375, 0.6049696803092957, 1.1834828853607178, -1.553305745124817, -0.9463446140289307, -0.37587934732437134, -0.609980583190918, 1.0444453954696655, -1.092339038848877, -1.0870728492736816, 0.14935001730918884, 0.8921463489532471, -0.26904159784317017, -0.40867799520492554, -1.7545387744903564, 0.512053370475769, 1.8429391384124756, 0.6098825335502625, -1.4732692241668701, -0.9999856948852539, -0.6043962240219116, 0.7722499370574951, 0.12197989970445633, 1.0270839929580688, -0.7457390427589417, 0.06804520636796951, -1.3210018873214722, -0.6278072595596313, -0.652794599533081, -1.3981430530548096, 1.1450024843215942, 0.14496628940105438, -0.6809421181678772, 0.02466932311654091, 0.13809587061405182, -0.5547271370887756, 0.3333829939365387, 1.2245484590530396, 0.8034272193908691, 1.500058889389038, -1.0600364208221436, 1.7589128017425537, -0.05998072028160095, 0.5866955518722534, 1.1806154251098633, -0.14199768006801605, 1.1421382427215576, -0.43727949261665344, -0.4304051101207733, -0.7680497169494629, -0.06125674024224281, -0.05145401135087013, 1.6188547611236572, 0.8023297190666199, 0.9843255877494812, 0.051929768174886703, -0.8435783982276917, 0.8382285833358765, -0.15894298255443573, 0.1409260630607605, -0.16513001918792725, 0.407269150018692, -0.5770502686500549, 1.1175124645233154, 0.48440441489219666, -0.2565408945083618, -0.60369473695755, 1.4636918306350708, -0.6939259171485901, -0.9544582962989807, -0.07605864852666855, -1.1371272802352905, 0.6279488205909729, -0.7045606970787048, 0.14756566286087036, 1.0847991704940796, -0.23776453733444214, -0.4450923800468445, 1.4204158782958984, 1.2798751592636108, 0.20569424331188202, 0.8622116446495056, 0.6500319242477417, 0.21667958796024323]], "dtype": "float32"}, {"__nd__": [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], "dtype": "float32"}], "e": {"__nd__": [[87.16477966308594, 109.94566345214844, 88.70384979248047, 104.00588989257812, 76.88445281982422, 92.02244567871094, 102.107177734375, 110.01353454589844, 83.70536041259766, 90.99000549316406, 108.65843963623047, 106.40003967285156, 103.34400177001953, 115.02662658691406, 104.01326751708984, 112.22177124023438, 112.20750427246094, 96.89473724365234, 90.00535583496094, 106.82880401611328, 73.5038070678711, 93.55962371826172, 86.79156494140625, 83.52312469482422, 94.12652587890625, 93.0823745727539, 118.90779113769531, 99.54322052001953, 113.33429718017578, 110.50110626220703, 88.24534606933594, 105.98946380615234, 98.8814697265625], [107.00837707519531, 106.5689697265625, 102.64592742919922, 83.47309875488281, 85.78367614746094, 107.33269500732422, 83.67387390136719, 122.29798889160156, 83.17741394042969, 118.99987030029297, 72.90612030029297, 75.57313537597656, 80.98014831542969, 106.72333526611328, 110.5665283203125, 97.98177337646484, 105.92890930175781, 119.34947204589844, 103.61314392089844, 67.7596435546875, 84.57601928710938, 90.97109985351562, 97.32290649414062, 88.14195251464844, 111.1552734375, 94.92393493652344, 91.83914184570312, 101.11450958251953, 68.3677749633789, 106.41056823730469, 86.65351867675781, 106.48091888427734, 107.27687072753906], [112.7595443725586, 92.59001922607422, 97.03695678710938, 97.43667602539062, 99.24166107177734, 93.92192840576172, 144.43096923828125, 111.10934448242188, 100.09269714355469, 96.25668334960938, 108.47074890136719, 124.4717025756836, 86.7420654296875, 92.65365600585938, 119.81373596191406, 93.56318664550781, 92.04547882080078, 91.78902435302734, 85.3802719116211, 102.50843811035156, 103.71549224853516, 102.64443969726562, 113.25724792480469, 98.66178894042969, 91.22439575195312, 112.49114990234375, 107.3923110961914, 88.63932037353516, 72.6679458618164, 111.94673156738281, 102.2943115234375, 80.75485229492188, 83.45340728759766]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(linear_relu, json.loads(_TESTS), 'triton', *[0.0001, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")